# Hydroclimatic memory in stream chemistry — reproducibility workflow

**Public reproducibility notebook derived from the project's frozen master audit pipeline.**

This notebook reproduces the inferential workflow supporting the manuscript:

> *Hydroclimatic memory in stream chemistry persists beyond discharge and varies with catchment organization*

It intentionally begins from **frozen, analysis-ready products** rather than reproducing the full forensic harmonisation of heterogeneous laboratory workbooks. The private master notebook remains the authoritative audit trail for raw-source identity resolution, duplicate adjudication, unit checks, censoring decisions, discharge reconstruction, and source-method adjudication.

## Reproducible analysis represented here

1. Freeze the response panel.
2. Integrate daily hydroclimate and construct the 29 pre-specified antecedent windows.
3. Assemble the memory-analysis dataset.
4. Fit the Bayesian hydroclimatic-memory spectrum.
5. Perform campaign-held-out (LOCO) predictive validation.
6. Apply the pre-specified strict-memory rule.
7. Fit concentration–discharge (C–Q) models.
8. Test conditional hydroclimatic information beyond discharge (H + Q).
9. Test modulation by contributing area (H × C).
10. Reproduce manuscript tables and figures.

## Important interpretation

Antecedent hydroclimatic variables represent **shared event-date forcing over the Razuhuillca reference domain**. They are not station-specific upstream hydroclimate estimates.

## Repository expectation

Place frozen inputs under `data/processed/` and outputs will be written under `outputs/`.


In [ ]:

# Reproducibility setup
from pathlib import Path
import os, sys, json, math, warnings
import numpy as np
import pandas as pd

PROJECT = Path.cwd()
DATA = PROJECT / "data" / "processed"
OUTPUTS = PROJECT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
CHECKPOINTS = OUTPUTS / "checkpoints"

for p in [DATA, OUTPUTS, TABLES, FIGURES, CHECKPOINTS]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260331
np.random.seed(RANDOM_SEED)

print("Project:", PROJECT)
print("Processed data:", DATA)
print("Outputs:", OUTPUTS)


## Required frozen inputs

The original master workflow generated a number of intermediate CSV files.  
For a public release, copy only the frozen files actually required by the analysis into `data/processed/`.

Recommended public data package:

- `MASTER_ANALYTICAL_v1.csv` — authoritative harmonised analytical records.
- `HYDROCLIMATE_DAILY_MASTER.csv` — daily CHIRPS / ERA5-Land forcing used in the study.
- `HYDROCLIMATE_EVENT_WINDOWS.csv` — event-date antecedent-window bank.
- `MEMORY_ANALYSIS_DATASET_v1.csv` — analysis-ready hydrochemical-memory dataset.
- frozen hydrometry / discharge table used by the C–Q analyses.
- frozen station/network table containing station identifiers and contributing area.
- censoring / response metadata required by the Bayesian likelihoods.
- `variable_dictionary.csv` — public data dictionary.

**Do not place raw workbooks, personal information, credentials, Google Drive paths, or unrelated IGP/project files in the public repository.**


---
### Imported master block 61

`BLOCK 7.2C — HYDROCLIMATIC MEMORY RESPONSE PANEL FREEZE`


In [ ]:
# ==========================================================================================
# BLOCK 7.2C — HYDROCLIMATIC MEMORY RESPONSE PANEL FREEZE
#
# PURPOSE
# -------
# Freeze a parsimonious response panel BEFORE introducing CHIRPS / ERA5-Land.
#
# IMPORTANT
# ---------
# - All 42 responses remain preserved.
# - This is manuscript/model prioritization, NOT data deletion.
# - Primary panel is chosen to represent distinct hydrochemical processes.
# - Highly redundant variables may remain as mechanistic sensitivities.
# - Censored responses remain available for censor-aware secondary analyses.
#
# INPUT
# -----
# RESPONSE_ARCHITECTURE_v2.csv
#
# OUTPUTS
# -------
# MEMORY_RESPONSE_PANEL_v1.csv
# PRIMARY_MEMORY_RESPONSE_PANEL_v1.csv
# BLOCK72C_memory_response_panel_freeze.xlsx
# ==========================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 500)

ROOT = Path("/content/Huanta_JoH")

MASTER_DIR = ROOT / "03_MASTER"
AUDIT_DIR = ROOT / "01_AUDIT"

ARCH_FILE = MASTER_DIR / "RESPONSE_ARCHITECTURE_v2.csv"

OUT_PANEL = MASTER_DIR / "MEMORY_RESPONSE_PANEL_v1.csv"
OUT_PRIMARY = MASTER_DIR / "PRIMARY_MEMORY_RESPONSE_PANEL_v1.csv"

OUT_AUDIT = (
    AUDIT_DIR /
    "BLOCK72C_memory_response_panel_freeze.xlsx"
)

print("=" * 175)
print("BLOCK 7.2C — HYDROCLIMATIC MEMORY RESPONSE PANEL FREEZE")
print("=" * 175)


# ==========================================================================================
# 1. LOAD
# ==========================================================================================

A = pd.read_csv(
    ARCH_FILE,
    low_memory=False
)

assert A["parameter_id"].nunique() == 42
assert len(A) == 42


# ==========================================================================================
# 2. FREEZE PRIMARY PROCESS-REPRESENTATIVE PANEL
# ==========================================================================================
#
# Principles:
#   - one principal response per distinct process where possible
#   - avoid sending highly redundant families wholesale into memory analysis
#   - favor physical interpretability over purely numerical score
#
# ==========================================================================================

PRIMARY_MEMORY = {

    # Acid-base state
    "AC::pH_SOL::NO_UNIT":
        "Longitudinal acid-base state; FIELD pH retained as in-situ sensitivity.",

    # Integrated mineralization
    "AC::C.E._SOL::dS/m":
        "Integrated ionic/mineralization state; robust and complete longitudinal coverage.",

    # Alkali/conservative weathering
    "AC::Na::mg/L":
        "Representative alkali/conservative weathering signal.",

    # Microbiology
    "COLIFORMS::Total coliforms::NMP/100 mL":
        "Primary microbial response with substantially greater detection support.",

    # Nutrients
    "AC::NH4::mg/L":
        "Reduced nitrogen response with high detection support.",

    # Particle-associated mobilization
    "METALS::Aluminio::mg/L":
        "Particle-associated/mobile metal response; strong positive preliminary C-Q association.",

    # Redox / reactive metal behavior
    "METALS::Hierro::mg/L":
        "Redox-sensitive metal with clear preliminary positive C-Q response.",

    # Silicate/weathering/storage proxy
    "METALS::Silicio::mg/L":
        "Continuous silicate-weathering/storage proxy across all campaigns.",

    # Independent major-anion response
    "AC::SO4_ppm::mg/L":
        "Major-anion response with comparatively independent behavior from carbonate axis.",

    # Thermal state
    "FIELD::Water temperature::degC":
        "Primary field thermal response; MP-primary with MT sensitivity."
}


# ==========================================================================================
# 3. MECHANISTIC SENSITIVITY PANEL
# ==========================================================================================
#
# These are valuable mechanistically but intentionally NOT simultaneous
# co-primary endpoints.
#
# ==========================================================================================

MECHANISTIC_SENSITIVITY = {

    "AC::HCO3_PPM::mg/L":
        "Mechanistic carbonate/dilution sensitivity to integrated EC response.",

    "AC::Ca::mg/L":
        "Carbonate/mineralization sensitivity; strongly redundant with EC/HCO3.",

    "AC::Mg::mg/L":
        "Carbonate/mineralization sensitivity; strongly redundant with EC.",

    "METALS::Manganeso::mg/L":
        "Redox sensitivity complement to Fe; stronger analytical coverage but weaker preliminary C-Q response.",

    "COLIFORMS::Fecal/thermotolerant coliforms::NMP/100 mL":
        "Microbial sensitivity response complementary to total coliforms.",

    "METALS::Titanio::mg/L":
        "Particle-associated sensitivity complementary to Al.",

    "METALS::Estroncio::mg/L":
        "Mineralization/weathering sensitivity; highly redundant with EC/Ca/Mg.",

    "METALS::Bario::mg/L":
        "Mineralization sensitivity with negative preliminary C-Q behavior.",

    "AC::Cl¯::mg/L":
        "Conservative-anion sensitivity within alkali/weathering family.",

    "METALS::Potasio::mg/L":
        "Alkali-weathering sensitivity strongly covarying with Li/Na."
}


# ==========================================================================================
# 4. CENSOR-AWARE SECONDARY PANEL
# ==========================================================================================

CENSOR_AWARE = {

    "AC::NO3::mg/L":
        "Nitrate response; moderate censoring requires censor-aware inference.",

    "COD::COD::mg O2/L":
        "Organic-matter response; censor-aware and 835.5 mg O2/L sensitivity required.",

    "AC::H2PO4::mg/L":
        "Phosphorus-form-specific response; method/species preserved.",

    "AC::P::mg/L":
        "Phosphorus-form-specific response; method/species preserved.",

    "METALS::Fósforo::mg/L":
        "Elemental/method-specific phosphorus response; not assumed equivalent to AC phosphorus.",

    "AC::CO3_PPM::mg/L":
        "Highly censored carbonate minor species; secondary censor-aware response."
}


# ==========================================================================================
# 5. APPLY PANEL ROLES
# ==========================================================================================

A["memory_panel_role"] = "SUPPLEMENTARY_RESPONSE"

A["memory_panel_rationale"] = (
    "Preserved in full response architecture; not selected for the primary "
    "hydroclimatic-memory panel."
)


for pid, reason in PRIMARY_MEMORY.items():

    assert (
        A["parameter_id"] == pid
    ).sum() == 1, f"Missing primary response: {pid}"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_role"
    ] = "PRIMARY_MEMORY_PANEL"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_rationale"
    ] = reason


for pid, reason in MECHANISTIC_SENSITIVITY.items():

    assert (
        A["parameter_id"] == pid
    ).sum() == 1, f"Missing sensitivity response: {pid}"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_role"
    ] = "MECHANISTIC_SENSITIVITY"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_rationale"
    ] = reason


for pid, reason in CENSOR_AWARE.items():

    assert (
        A["parameter_id"] == pid
    ).sum() == 1, f"Missing censor-aware response: {pid}"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_role"
    ] = "SECONDARY_CENSOR_AWARE"

    A.loc[
        A["parameter_id"].eq(pid),
        "memory_panel_rationale"
    ] = reason


# ==========================================================================================
# 6. ADD ANALYSIS PRIORITY
# ==========================================================================================

priority_map = {
    "PRIMARY_MEMORY_PANEL": 1,
    "MECHANISTIC_SENSITIVITY": 2,
    "SECONDARY_CENSOR_AWARE": 3,
    "SUPPLEMENTARY_RESPONSE": 4
}


A["memory_analysis_priority"] = (
    A[
        "memory_panel_role"
    ].map(
        priority_map
    )
)


# ==========================================================================================
# 7. IMPORTANT MODEL POLICY
# ==========================================================================================

A["simultaneous_window_bank_policy"] = (
    "WINDOWS_FORM_A_TIMESCALE_BANK__DO_NOT_ENTER_ALL_WINDOWS_SIMULTANEOUSLY"
)


A["validation_policy"] = (
    "CAMPAIGN_AWARE_VALIDATION_REQUIRED"
)


A["independence_note"] = (
    "10 hydroclimatic campaigns are the principal independent temporal units; "
    "sample-level observations are repeated spatial measurements."
)


# ==========================================================================================
# 8. PRIMARY PANEL
# ==========================================================================================

PRIMARY = A[
    A[
        "memory_panel_role"
    ].eq(
        "PRIMARY_MEMORY_PANEL"
    )
].copy()


assert PRIMARY["parameter_id"].nunique() == 10


# ==========================================================================================
# 9. SUMMARIES
# ==========================================================================================

ROLE_SUMMARY = (
    A.groupby(
        "memory_panel_role",
        dropna=False
    )
    .agg(
        n_responses=(
            "parameter_id",
            "nunique"
        ),

        median_censoring_pct=(
            "pct_censored",
            "median"
        ),

        median_detected_n=(
            "n_detected",
            "median"
        )
    )
    .reset_index()
)


PRIMARY_DOMAIN_SUMMARY = (
    PRIMARY.groupby(
        [
            "conceptual_response_family",
            "hydrochemical_domain"
        ],
        dropna=False
    )
    .agg(
        n_responses=(
            "parameter_id",
            "nunique"
        ),

        responses=(
            "parameter_id",
            lambda x:
                " | ".join(
                    sorted(
                        x.astype(str)
                    )
                )
        )
    )
    .reset_index()
)


print("\n" + "=" * 175)
print("MEMORY PANEL ROLE SUMMARY")
print("=" * 175)

display(
    ROLE_SUMMARY
)


print("\n" + "=" * 175)
print("PRIMARY HYDROCLIMATIC-MEMORY PANEL")
print("=" * 175)

display(
    PRIMARY[
        [
            "conceptual_response_family",
            "hydrochemical_domain",
            "parameter_id",
            "pct_censored",
            "n_detected",
            "n_campaigns",
            "high_corr_degree",
            "spearman_rho_detected_only",
            "memory_panel_rationale"
        ]
    ].sort_values(
        "conceptual_response_family"
    )
)


print("\n" + "=" * 175)
print("MECHANISTIC SENSITIVITY PANEL")
print("=" * 175)

display(
    A[
        A[
            "memory_panel_role"
        ].eq(
            "MECHANISTIC_SENSITIVITY"
        )
    ][
        [
            "conceptual_response_family",
            "parameter_id",
            "pct_censored",
            "n_detected",
            "memory_panel_rationale"
        ]
    ]
)


print("\n" + "=" * 175)
print("SECONDARY CENSOR-AWARE PANEL")
print("=" * 175)

display(
    A[
        A[
            "memory_panel_role"
        ].eq(
            "SECONDARY_CENSOR_AWARE"
        )
    ][
        [
            "conceptual_response_family",
            "parameter_id",
            "pct_censored",
            "n_detected",
            "memory_panel_rationale"
        ]
    ]
)


# ==========================================================================================
# 10. QA — REDUNDANCY IN PRIMARY PANEL
# ==========================================================================================

EDGE_FILE = (
    MASTER_DIR /
    "RESPONSE_REDUNDANCY_EDGES_v2.csv"
)


EDGE = pd.read_csv(
    EDGE_FILE,
    low_memory=False
)


primary_ids = set(
    PRIMARY[
        "parameter_id"
    ]
)


PRIMARY_REDUNDANCY = EDGE[
    EDGE[
        "parameter_1"
    ].isin(
        primary_ids
    )
    &
    EDGE[
        "parameter_2"
    ].isin(
        primary_ids
    )
].copy()


print("\n" + "=" * 175)
print("HIGH-CORRELATION EDGES WITHIN PRIMARY PANEL")
print("=" * 175)

display(
    PRIMARY_REDUNDANCY
)


# We expect a parsimonious primary panel.
# No |rho| >= 0.85 pair should remain among primary endpoints.

assert len(
    PRIMARY_REDUNDANCY
) == 0, (
    "Primary panel still contains strongly redundant response pairs."
)


# ==========================================================================================
# 11. FINAL QA
# ==========================================================================================

assert len(A) == 42

assert A[
    "parameter_id"
].nunique() == 42

assert (
    A[
        "memory_panel_role"
    ]
    ==
    "PRIMARY_MEMORY_PANEL"
).sum() == 10


assert (
    A[
        "memory_panel_role"
    ]
    ==
    "MECHANISTIC_SENSITIVITY"
).sum() == 10


assert (
    A[
        "memory_panel_role"
    ]
    ==
    "SECONDARY_CENSOR_AWARE"
).sum() == 6


assert A[
    "memory_analysis_priority"
].notna().all()


# pH retained as primary
assert A.loc[
    A[
        "parameter_id"
    ].eq(
        "AC::pH_SOL::NO_UNIT"
    ),
    "memory_panel_role"
].iloc[0] == "PRIMARY_MEMORY_PANEL"


# COD must remain censor-aware secondary
assert A.loc[
    A[
        "parameter_id"
    ].eq(
        "COD::COD::mg O2/L"
    ),
    "memory_panel_role"
].iloc[0] == "SECONDARY_CENSOR_AWARE"


# ==========================================================================================
# 12. SAVE
# ==========================================================================================

A.to_csv(
    OUT_PANEL,
    index=False
)

PRIMARY.to_csv(
    OUT_PRIMARY,
    index=False
)


with pd.ExcelWriter(
    OUT_AUDIT,
    engine="openpyxl"
) as writer:

    A.to_excel(
        writer,
        sheet_name="full_memory_panel",
        index=False
    )

    PRIMARY.to_excel(
        writer,
        sheet_name="primary_memory_panel",
        index=False
    )

    ROLE_SUMMARY.to_excel(
        writer,
        sheet_name="role_summary",
        index=False
    )

    PRIMARY_DOMAIN_SUMMARY.to_excel(
        writer,
        sheet_name="primary_domains",
        index=False
    )

    A[
        A[
            "memory_panel_role"
        ].eq(
            "MECHANISTIC_SENSITIVITY"
        )
    ].to_excel(
        writer,
        sheet_name="mechanistic_sensitivity",
        index=False
    )

    A[
        A[
            "memory_panel_role"
        ].eq(
            "SECONDARY_CENSOR_AWARE"
        )
    ].to_excel(
        writer,
        sheet_name="censor_aware_secondary",
        index=False
    )

    PRIMARY_REDUNDANCY.to_excel(
        writer,
        sheet_name="primary_redundancy_QA",
        index=False
    )


# ==========================================================================================
# 13. FINAL DIAGNOSTIC
# ==========================================================================================

print("\n" + "=" * 175)
print("BLOCK 7.2C — FINAL DIAGNOSTIC")
print("=" * 175)

print(
    "Responses preserved:",
    A[
        "parameter_id"
    ].nunique()
)

print(
    "Primary memory responses:",
    (
        A[
            "memory_panel_role"
        ]
        ==
        "PRIMARY_MEMORY_PANEL"
    ).sum()
)

print(
    "Mechanistic sensitivities:",
    (
        A[
            "memory_panel_role"
        ]
        ==
        "MECHANISTIC_SENSITIVITY"
    ).sum()
)

print(
    "Censor-aware secondary:",
    (
        A[
            "memory_panel_role"
        ]
        ==
        "SECONDARY_CENSOR_AWARE"
    ).sum()
)

print(
    "Supplementary responses:",
    (
        A[
            "memory_panel_role"
        ]
        ==
        "SUPPLEMENTARY_RESPONSE"
    ).sum()
)

print(
    "High-correlation edges retained inside primary panel:",
    len(
        PRIMARY_REDUNDANCY
    )
)

print("\nResponses deleted       : 0")
print("Censored substitutions : 0")
print("Hydroclimate used      : 0")
print("Spatial processing     : 0")
print("Final models fitted    : 0")

print("\nFull panel:")
print(OUT_PANEL)

print("\nPrimary panel:")
print(OUT_PRIMARY)

print("\nAudit:")
print(OUT_AUDIT)

print(
    "\nSTATUS: HYDROCLIMATIC-MEMORY RESPONSE PANEL FROZEN — "
    "READY FOR EXTERNAL HYDROCLIMATE INTEGRATION."
)

---
### Imported master block 63

`=============================================================================`


In [ ]:
#seguimos
# =============================================================================
# BLOCK 8.1 — HYDROCLIMATE DAILY MASTER FREEZE + EVENT-DATE INTEGRATION
# Journal of Hydrology — Huanta / Razuhuillca
#
# PURPOSE
# -------
# 1. Load the frozen daily hydroclimate export from GEE.
# 2. Independently audit temporal and mathematical integrity.
# 3. Load the authoritative 146-event hydroclimate calendar.
# 4. Link every event to its EXACT event date.
# 5. Verify 146/146 event-date hydroclimate matches.
# 6. Save frozen integration products.
#
# IMPORTANT
# ---------
# NO antecedent windows.
# NO response variables.
# NO correlations.
# NO memory estimation.
# NO model fitting.
# NO variable selection.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil


# =============================================================================
# 0. PATHS
# =============================================================================

ROOT = Path("/content/Huanta_JoH")

AUDIT_DIR  = ROOT / "01_AUDIT"
CLEAN_DIR  = ROOT / "02_CLEAN"
MASTER_DIR = ROOT / "03_MASTER"
OUTPUT_DIR = ROOT / "04_OUTPUT"

for d in [AUDIT_DIR, CLEAN_DIR, MASTER_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


HYDRO_FILE = MASTER_DIR / "Razuhuillca_Hydroclimate_Daily_Master_v1.csv"

EVENT_FILE = MASTER_DIR / "HYDROCLIMATE_EVENT_CALENDAR_v1.csv"

FREEZE_FILE = MASTER_DIR / "HYDROCLIMATE_DAILY_MASTER_v1.csv"

EVENT_JOIN_FILE = MASTER_DIR / "HYDROCLIMATE_EVENT_DAY_MATCH_v1.csv"

AUDIT_FILE = AUDIT_DIR / "BLOCK81_hydroclimate_daily_master_freeze.xlsx"

MANIFEST_FILE = AUDIT_DIR / "BLOCK81_hydroclimate_daily_master_manifest.json"


print("=" * 88)
print("BLOCK 8.1 — HYDROCLIMATE DAILY MASTER FREEZE + EVENT-DATE INTEGRATION")
print("=" * 88)

print("\nInput daily hydroclimate:")
print(HYDRO_FILE)

print("\nAuthoritative event calendar:")
print(EVENT_FILE)


# =============================================================================
# 1. REQUIRE INPUT FILES
# =============================================================================

missing_files = [
    str(p)
    for p in [HYDRO_FILE, EVENT_FILE]
    if not p.exists()
]

if missing_files:
    raise FileNotFoundError(
        "\nRequired file(s) missing:\n" +
        "\n".join(missing_files)
    )

print("\nInput files found: PASS")


# =============================================================================
# 2. LOAD DAILY HYDROCLIMATE
# =============================================================================

hydro = pd.read_csv(HYDRO_FILE)

print("\n" + "=" * 88)
print("DAILY HYDROCLIMATE INPUT")
print("=" * 88)

print("Rows:", len(hydro))
print("Columns:", len(hydro.columns))
print("\nColumns:")
print(hydro.columns.tolist())


# =============================================================================
# 3. REQUIRED HYDROCLIMATE COLUMNS
# =============================================================================

required_hydro_cols = [
    "date",
    "precipitation_mm_day",
    "sm_surface_0_7cm",
    "sm_layer2_7_28cm",
    "sm_layer3_28_100cm",
    "sm_rootzone_0_100cm",
    "tair_2m_K",
    "tair_2m_C",
    "pet_native_m_day",
    "pet_native_mm_day",
    "pet_magnitude_mm_day",
]

missing_hydro_cols = [
    c for c in required_hydro_cols
    if c not in hydro.columns
]

if missing_hydro_cols:
    raise ValueError(
        f"Required hydroclimate columns missing: {missing_hydro_cols}"
    )

print("\nRequired hydroclimate schema: PASS")


# =============================================================================
# 4. DATE PARSING
# =============================================================================

hydro["date"] = pd.to_datetime(
    hydro["date"],
    errors="raise"
).dt.normalize()

hydro = (
    hydro
    .sort_values("date")
    .reset_index(drop=True)
)


EXPECTED_START = pd.Timestamp("2022-12-01")
EXPECTED_END   = pd.Timestamp("2026-03-23")

expected_dates = pd.date_range(
    EXPECTED_START,
    EXPECTED_END,
    freq="D"
)


# =============================================================================
# 5. TEMPORAL QA
# =============================================================================

n_rows = len(hydro)

n_unique_dates = hydro["date"].nunique()

duplicate_dates = hydro.loc[
    hydro["date"].duplicated(keep=False),
    "date"
].copy()

actual_dates = pd.DatetimeIndex(
    hydro["date"].dropna().unique()
)

missing_dates = expected_dates.difference(actual_dates)

extra_dates = actual_dates.difference(expected_dates)


temporal_qa = pd.DataFrame({
    "metric": [
        "expected_rows",
        "actual_rows",
        "unique_dates",
        "duplicate_date_rows",
        "missing_dates",
        "extra_dates",
        "first_date",
        "last_date",
    ],
    "value": [
        len(expected_dates),
        n_rows,
        n_unique_dates,
        len(duplicate_dates),
        len(missing_dates),
        len(extra_dates),
        hydro["date"].min().strftime("%Y-%m-%d"),
        hydro["date"].max().strftime("%Y-%m-%d"),
    ]
})


print("\n" + "=" * 88)
print("TEMPORAL QA")
print("=" * 88)

print(temporal_qa.to_string(index=False))


# =============================================================================
# 6. NULL QA
# =============================================================================

null_qa = (
    hydro[required_hydro_cols]
    .isna()
    .sum()
    .rename("n_null")
    .reset_index()
    .rename(columns={"index": "variable"})
)

null_qa["pct_null"] = (
    null_qa["n_null"] /
    len(hydro) * 100
)


print("\n" + "=" * 88)
print("NULL QA")
print("=" * 88)

print(null_qa.to_string(index=False))


# =============================================================================
# 7. MATHEMATICAL IDENTITY QA
# =============================================================================

rootzone_expected = (
    0.07 * hydro["sm_surface_0_7cm"] +
    0.21 * hydro["sm_layer2_7_28cm"] +
    0.72 * hydro["sm_layer3_28_100cm"]
)

rootzone_error = (
    hydro["sm_rootzone_0_100cm"] -
    rootzone_expected
).abs()


tair_expected = hydro["tair_2m_K"] - 273.15

tair_error = (
    hydro["tair_2m_C"] -
    tair_expected
).abs()


pet_mm_expected = (
    hydro["pet_native_m_day"] * 1000
)

pet_mm_error = (
    hydro["pet_native_mm_day"] -
    pet_mm_expected
).abs()


pet_mag_expected = (
    -hydro["pet_native_mm_day"]
)

pet_mag_error = (
    hydro["pet_magnitude_mm_day"] -
    pet_mag_expected
).abs()


identity_qa = pd.DataFrame({
    "identity": [
        "rootzone = 0.07*L1 + 0.21*L2 + 0.72*L3",
        "Tair_C = Tair_K - 273.15",
        "PET_native_mm = PET_native_m * 1000",
        "PET_magnitude_mm = -PET_native_mm",
    ],
    "max_abs_error": [
        rootzone_error.max(),
        tair_error.max(),
        pet_mm_error.max(),
        pet_mag_error.max(),
    ]
})

identity_qa["pass_1e-10"] = (
    identity_qa["max_abs_error"] < 1e-10
)


print("\n" + "=" * 88)
print("MATHEMATICAL IDENTITY QA")
print("=" * 88)

print(identity_qa.to_string(index=False))


# =============================================================================
# 8. PHYSICAL RANGE QA
# =============================================================================

range_qa = pd.DataFrame([
    {
        "check": "precipitation < 0",
        "n_invalid": int(
            (hydro["precipitation_mm_day"] < 0).sum()
        )
    },
    {
        "check": "SM surface outside [0,1]",
        "n_invalid": int(
            (
                (hydro["sm_surface_0_7cm"] < 0) |
                (hydro["sm_surface_0_7cm"] > 1)
            ).sum()
        )
    },
    {
        "check": "SM L2 outside [0,1]",
        "n_invalid": int(
            (
                (hydro["sm_layer2_7_28cm"] < 0) |
                (hydro["sm_layer2_7_28cm"] > 1)
            ).sum()
        )
    },
    {
        "check": "SM L3 outside [0,1]",
        "n_invalid": int(
            (
                (hydro["sm_layer3_28_100cm"] < 0) |
                (hydro["sm_layer3_28_100cm"] > 1)
            ).sum()
        )
    },
    {
        "check": "SM rootzone outside [0,1]",
        "n_invalid": int(
            (
                (hydro["sm_rootzone_0_100cm"] < 0) |
                (hydro["sm_rootzone_0_100cm"] > 1)
            ).sum()
        )
    },
    {
        "check": "Tair outside broad [-30,40] C",
        "n_invalid": int(
            (
                (hydro["tair_2m_C"] < -30) |
                (hydro["tair_2m_C"] > 40)
            ).sum()
        )
    },
    {
        "check": "PET magnitude < 0",
        "n_invalid": int(
            (hydro["pet_magnitude_mm_day"] < 0).sum()
        )
    },
    {
        "check": "native PET >= 0",
        "n_invalid": int(
            (hydro["pet_native_mm_day"] >= 0).sum()
        )
    },
])


print("\n" + "=" * 88)
print("PHYSICAL RANGE QA")
print("=" * 88)

print(range_qa.to_string(index=False))


# =============================================================================
# 9. DESCRIPTIVE QA
# =============================================================================

numeric_hydro_cols = [
    "precipitation_mm_day",
    "sm_surface_0_7cm",
    "sm_layer2_7_28cm",
    "sm_layer3_28_100cm",
    "sm_rootzone_0_100cm",
    "tair_2m_C",
    "pet_native_mm_day",
    "pet_magnitude_mm_day",
]

descriptive_qa = (
    hydro[numeric_hydro_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)


print("\n" + "=" * 88)
print("DAILY HYDROCLIMATE DESCRIPTIVES")
print("=" * 88)

print(
    descriptive_qa[
        ["variable", "count", "mean", "std", "min", "50%", "max"]
    ].to_string(index=False)
)


# =============================================================================
# 10. DAILY MASTER FREEZE GATE
# =============================================================================

daily_gate_checks = {
    "rows_1209":
        len(hydro) == 1209,

    "unique_dates_1209":
        hydro["date"].nunique() == 1209,

    "no_duplicate_dates":
        len(duplicate_dates) == 0,

    "no_missing_dates":
        len(missing_dates) == 0,

    "no_extra_dates":
        len(extra_dates) == 0,

    "no_required_nulls":
        hydro[required_hydro_cols].isna().sum().sum() == 0,

    "mathematical_identities":
        bool(identity_qa["pass_1e-10"].all()),

    "physical_ranges":
        bool((range_qa["n_invalid"] == 0).all()),

    "correct_start":
        hydro["date"].min() == EXPECTED_START,

    "correct_end":
        hydro["date"].max() == EXPECTED_END,
}


daily_gate_df = pd.DataFrame({
    "check": daily_gate_checks.keys(),
    "pass": daily_gate_checks.values()
})


DAILY_MASTER_PASS = all(
    daily_gate_checks.values()
)


print("\n" + "=" * 88)
print("DAILY MASTER FREEZE GATE")
print("=" * 88)

print(daily_gate_df.to_string(index=False))

print(
    "\nDAILY HYDROCLIMATE MASTER PASS:",
    DAILY_MASTER_PASS
)


if not DAILY_MASTER_PASS:
    raise RuntimeError(
        "BLOCK 8.1 STOP: daily hydroclimate master failed QA."
    )


# =============================================================================
# 11. FREEZE DAILY MASTER
# =============================================================================

hydro.to_csv(
    FREEZE_FILE,
    index=False,
    date_format="%Y-%m-%d"
)

print("\nFrozen daily master:")
print(FREEZE_FILE)


# =============================================================================
# 12. LOAD AUTHORITATIVE EVENT CALENDAR
# =============================================================================

events = pd.read_csv(EVENT_FILE)

print("\n" + "=" * 88)
print("AUTHORITATIVE HYDROCLIMATE EVENT CALENDAR")
print("=" * 88)

print("Rows:", len(events))
print("Columns:", len(events.columns))

print("\nColumns:")
print(events.columns.tolist())


# =============================================================================
# 13. IDENTIFY EVENT-DATE COLUMN
# =============================================================================
#
# We do NOT silently guess among multiple plausible columns.
# The previously frozen calendar should contain one authoritative
# hydroclimate event-date field.
# =============================================================================

preferred_event_date_cols = [
    "hydroclimate_event_date",
    "event_date",
    "canonical_event_date",
    "date"
]

available_date_candidates = [
    c for c in preferred_event_date_cols
    if c in events.columns
]


if len(available_date_candidates) == 0:
    raise ValueError(
        "No recognized event-date column found.\n"
        f"Available columns:\n{events.columns.tolist()}"
    )


# Prefer the explicitly named hydroclimate date if available.

if "hydroclimate_event_date" in available_date_candidates:
    EVENT_DATE_COL = "hydroclimate_event_date"

elif len(available_date_candidates) == 1:
    EVENT_DATE_COL = available_date_candidates[0]

else:
    raise ValueError(
        "Multiple plausible event-date columns found and "
        "hydroclimate_event_date is absent.\n"
        f"Candidates: {available_date_candidates}\n"
        "STOP rather than silently choosing one."
    )


print("\nAuthoritative event-date column:")
print(EVENT_DATE_COL)


events[EVENT_DATE_COL] = pd.to_datetime(
    events[EVENT_DATE_COL],
    errors="raise"
).dt.normalize()


# =============================================================================
# 14. EVENT CALENDAR QA
# =============================================================================

print("\n" + "=" * 88)
print("EVENT CALENDAR QA")
print("=" * 88)

print("Expected event rows: 146")
print("Observed event rows:", len(events))

print(
    "Null event dates:",
    events[EVENT_DATE_COL].isna().sum()
)

print(
    "Unique event dates:",
    events[EVENT_DATE_COL].nunique()
)

print(
    "Earliest event date:",
    events[EVENT_DATE_COL].min()
)

print(
    "Latest event date:",
    events[EVENT_DATE_COL].max()
)


if len(events) != 146:
    raise RuntimeError(
        f"Expected 146 hydroclimate events, found {len(events)}."
    )

if events[EVENT_DATE_COL].isna().any():
    raise RuntimeError(
        "Null authoritative event dates detected."
    )


# =============================================================================
# 15. EXACT-DATE HYDROCLIMATE JOIN
# =============================================================================
#
# IMPORTANT:
# This is an exact DATE join.
#
# It does NOT use:
#   campaign midpoint
#   nearest date
#   interpolation
#   monthly means
#   campaign means
#
# Each event gets hydroclimate for its own actual date.
# =============================================================================

hydro_event_day = hydro[
    [
        "date",
        "precipitation_mm_day",
        "sm_surface_0_7cm",
        "sm_layer2_7_28cm",
        "sm_layer3_28_100cm",
        "sm_rootzone_0_100cm",
        "tair_2m_C",
        "pet_native_mm_day",
        "pet_magnitude_mm_day",
    ]
].copy()


event_match = events.merge(
    hydro_event_day,
    how="left",
    left_on=EVENT_DATE_COL,
    right_on="date",
    validate="many_to_one",
    indicator="_hydro_join"
)


# =============================================================================
# 16. EVENT-DATE JOIN QA
# =============================================================================

join_counts = (
    event_match["_hydro_join"]
    .value_counts(dropna=False)
    .rename_axis("join_status")
    .reset_index(name="n")
)


event_hydro_vars = [
    "precipitation_mm_day",
    "sm_surface_0_7cm",
    "sm_layer2_7_28cm",
    "sm_layer3_28_100cm",
    "sm_rootzone_0_100cm",
    "tair_2m_C",
    "pet_native_mm_day",
    "pet_magnitude_mm_day",
]


event_nulls = (
    event_match[event_hydro_vars]
    .isna()
    .sum()
    .rename("n_null")
    .reset_index()
    .rename(columns={"index": "variable"})
)


n_matched = int(
    (event_match["_hydro_join"] == "both").sum()
)

n_unmatched = int(
    (event_match["_hydro_join"] != "both").sum()
)


print("\n" + "=" * 88)
print("EXACT EVENT-DATE JOIN QA")
print("=" * 88)

print(join_counts.to_string(index=False))

print("\nMatched events:", n_matched)
print("Unmatched events:", n_unmatched)

print("\nHydroclimate nulls after exact-date join:")
print(event_nulls.to_string(index=False))


# =============================================================================
# 17. VERIFY JOINED DATE IDENTITY
# =============================================================================

date_identity = (
    event_match[EVENT_DATE_COL] ==
    event_match["date"]
)

n_date_mismatch = int(
    (~date_identity).sum()
)


print(
    "\nEvent-date / hydroclimate-date mismatches:",
    n_date_mismatch
)


# =============================================================================
# 18. EVENT INTEGRATION GATE
# =============================================================================

event_gate_checks = {
    "event_rows_146":
        len(event_match) == 146,

    "all_events_matched":
        n_matched == 146,

    "zero_unmatched":
        n_unmatched == 0,

    "zero_hydroclimate_nulls":
        event_match[event_hydro_vars]
        .isna()
        .sum()
        .sum() == 0,

    "exact_date_identity":
        n_date_mismatch == 0,
}


event_gate_df = pd.DataFrame({
    "check": event_gate_checks.keys(),
    "pass": event_gate_checks.values()
})


EVENT_INTEGRATION_PASS = all(
    event_gate_checks.values()
)


print("\n" + "=" * 88)
print("EVENT-DATE INTEGRATION GATE")
print("=" * 88)

print(event_gate_df.to_string(index=False))

print(
    "\nEVENT-DATE INTEGRATION PASS:",
    EVENT_INTEGRATION_PASS
)


if not EVENT_INTEGRATION_PASS:

    bad = event_match.loc[
        event_match["_hydro_join"] != "both"
    ].copy()

    print("\nUNMATCHED EVENTS:")
    print(
        bad.to_string(index=False)
        if len(bad)
        else "None"
    )

    raise RuntimeError(
        "BLOCK 8.1 STOP: event-date integration failed QA."
    )


# =============================================================================
# 19. CLEAN JOIN OUTPUT
# =============================================================================

event_match = event_match.drop(
    columns=["_hydro_join"]
)


# =============================================================================
# 20. SAVE EVENT-DAY MATCH
# =============================================================================

event_match.to_csv(
    EVENT_JOIN_FILE,
    index=False,
    date_format="%Y-%m-%d"
)

print("\nFrozen event-date match:")
print(EVENT_JOIN_FILE)


# =============================================================================
# 21. FILE HASHES FOR PROVENANCE
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


hashes = {
    "source_gee_csv_sha256":
        sha256_file(HYDRO_FILE),

    "frozen_daily_master_sha256":
        sha256_file(FREEZE_FILE),

    "event_calendar_sha256":
        sha256_file(EVENT_FILE),

    "event_day_match_sha256":
        sha256_file(EVENT_JOIN_FILE),
}


# =============================================================================
# 22. SAVE EXCEL AUDIT
# =============================================================================

with pd.ExcelWriter(
    AUDIT_FILE,
    engine="openpyxl"
) as writer:

    temporal_qa.to_excel(
        writer,
        sheet_name="temporal_QA",
        index=False
    )

    null_qa.to_excel(
        writer,
        sheet_name="null_QA",
        index=False
    )

    identity_qa.to_excel(
        writer,
        sheet_name="identity_QA",
        index=False
    )

    range_qa.to_excel(
        writer,
        sheet_name="range_QA",
        index=False
    )

    descriptive_qa.to_excel(
        writer,
        sheet_name="descriptives",
        index=False
    )

    daily_gate_df.to_excel(
        writer,
        sheet_name="daily_gate",
        index=False
    )

    join_counts.to_excel(
        writer,
        sheet_name="event_join_QA",
        index=False
    )

    event_nulls.to_excel(
        writer,
        sheet_name="event_null_QA",
        index=False
    )

    event_gate_df.to_excel(
        writer,
        sheet_name="event_gate",
        index=False
    )


# =============================================================================
# 23. MANIFEST
# =============================================================================

manifest = {

    "block":
        "8.1",

    "purpose":
        "Daily hydroclimate freeze and exact event-date integration",

    "roi":
        "Razuhuillca microcatchment",

    "daily_period_start":
        str(EXPECTED_START.date()),

    "daily_period_end":
        str(EXPECTED_END.date()),

    "daily_rows":
        int(len(hydro)),

    "event_rows":
        int(len(event_match)),

    "event_date_column":
        EVENT_DATE_COL,

    "daily_master_pass":
        bool(DAILY_MASTER_PASS),

    "event_integration_pass":
        bool(EVENT_INTEGRATION_PASS),

    "antecedent_windows_calculated":
        False,

    "response_variables_joined":
        False,

    "memory_estimated":
        False,

    "models_fitted":
        False,

    "hashes":
        hashes,
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# =============================================================================
# 24. FINAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 88)
print("BLOCK 8.1 — FINAL DIAGNOSTIC")
print("=" * 88)

print("Daily expected rows             :", 1209)
print("Daily actual rows               :", len(hydro))
print("Daily unique dates              :", hydro["date"].nunique())
print("Daily missing dates             :", len(missing_dates))
print("Daily duplicate-date rows       :", len(duplicate_dates))
print("Daily required nulls            :", int(
    hydro[required_hydro_cols].isna().sum().sum()
))

print()
print("Hydroclimate events expected    :", 146)
print("Hydroclimate events observed    :", len(events))
print("Exact-date matches              :", n_matched)
print("Unmatched events                :", n_unmatched)
print("Date identity mismatches        :", n_date_mismatch)
print("Joined hydroclimate nulls       :", int(
    event_match[event_hydro_vars].isna().sum().sum()
))

print()
print("DAILY MASTER PASS               :", DAILY_MASTER_PASS)
print("EVENT INTEGRATION PASS          :", EVENT_INTEGRATION_PASS)

print()
print("Antecedent windows calculated   : NO")
print("Response variables joined       : NO")
print("Memory estimated                : NO")
print("Models fitted                   : NO")

print()
print("Frozen daily master:")
print(FREEZE_FILE)

print("\nFrozen event-day table:")
print(EVENT_JOIN_FILE)

print("\nAudit workbook:")
print(AUDIT_FILE)

print("\nManifest:")
print(MANIFEST_FILE)

print()
print("BLOCK 8.1 COMPLETE.")
print("DO NOT PROCEED TO 8.2 UNTIL BOTH PASS FLAGS ARE TRUE.")

---
### Imported master block 64

`BLOCK 8.2 — ANTECEDENT HYDROCLIMATE WINDOW BANK`


In [ ]:
# =============================================================================
# BLOCK 8.2 — ANTECEDENT HYDROCLIMATE WINDOW BANK
# Journal of Hydrology — Razuhuillca
#
# PURPOSE
# -------
# Build the PRE-SPECIFIED hydroclimatic antecedent-timescale bank.
#
# Windows are calculated from the COMPLETE DAILY hydroclimate series and then
# attached to the authoritative 146-event calendar by exact event date.
#
# IMPORTANT
# ---------
# Window convention:
#   sampling/event day IS INCLUDED.
#
# Example:
#   7-day window for event date D = [D-6, ..., D]
#
# NO response variables.
# NO correlation screening.
# NO memory estimation.
# NO selection of "best" windows.
# NO model fitting.
#
# This block creates the candidate timescale bank only.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib


# =============================================================================
# 0. PATHS
# =============================================================================

ROOT = Path("/content/Huanta_JoH")

AUDIT_DIR  = ROOT / "01_AUDIT"
MASTER_DIR = ROOT / "03_MASTER"

DAILY_FILE = MASTER_DIR / "HYDROCLIMATE_DAILY_MASTER_v1.csv"

EVENT_FILE = MASTER_DIR / "HYDROCLIMATE_EVENT_CALENDAR_v1.csv"

WINDOW_DEF_FILE = MASTER_DIR / "HYDROCLIMATE_WINDOW_DEFINITION_v1.csv"

DAILY_WINDOW_FILE = (
    MASTER_DIR /
    "HYDROCLIMATE_DAILY_ANTECEDENT_WINDOWS_v1.csv"
)

EVENT_WINDOW_FILE = (
    MASTER_DIR /
    "HYDROCLIMATE_EVENT_ANTECEDENT_WINDOWS_v1.csv"
)

AUDIT_FILE = (
    AUDIT_DIR /
    "BLOCK82_antecedent_hydroclimate_window_bank.xlsx"
)

MANIFEST_FILE = (
    AUDIT_DIR /
    "BLOCK82_antecedent_hydroclimate_window_manifest.json"
)


print("=" * 96)
print("BLOCK 8.2 — ANTECEDENT HYDROCLIMATE WINDOW BANK")
print("=" * 96)


# =============================================================================
# 1. INPUT CHECK
# =============================================================================

for p in [DAILY_FILE, EVENT_FILE]:

    if not p.exists():

        raise FileNotFoundError(
            f"Required input missing:\n{p}"
        )


daily = pd.read_csv(
    DAILY_FILE,
    parse_dates=["date"]
)

events = pd.read_csv(
    EVENT_FILE,
    parse_dates=["hydroclimate_event_date"]
)


daily["date"] = (
    pd.to_datetime(daily["date"])
    .dt.normalize()
)

events["hydroclimate_event_date"] = (
    pd.to_datetime(
        events["hydroclimate_event_date"]
    )
    .dt.normalize()
)


daily = (
    daily
    .sort_values("date")
    .reset_index(drop=True)
)


print("\nInput daily rows :", len(daily))
print("Input events     :", len(events))
print(
    "Unique event dates:",
    events["hydroclimate_event_date"].nunique()
)


# =============================================================================
# 2. HARD INPUT GATES
# =============================================================================

assert len(daily) == 1209
assert daily["date"].nunique() == 1209
assert daily["date"].duplicated().sum() == 0

assert len(events) == 146
assert events["hydroclimate_event_date"].notna().all()

date_diff = daily["date"].diff().dropna()

if not (date_diff == pd.Timedelta(days=1)).all():

    raise RuntimeError(
        "Daily hydroclimate series is not temporally continuous."
    )


print("\nInput integrity gate: PASS")


# =============================================================================
# 3. PRE-SPECIFIED WINDOW ARCHITECTURE
# =============================================================================

WINDOWS = {

    "precipitation": {
        "source": "precipitation_mm_day",
        "windows": [1, 3, 7, 14, 30, 60, 90],
        "operation": "sum",
        "prefix": "ppt_sum"
    },

    "surface_soil_moisture": {
        "source": "sm_surface_0_7cm",
        "windows": [1, 7, 14, 30, 60, 90],
        "operation": "mean",
        "prefix": "sm_surface_mean"
    },

    "rootzone_soil_moisture": {
        "source": "sm_rootzone_0_100cm",
        "windows": [1, 7, 14, 30, 60, 90],
        "operation": "mean",
        "prefix": "sm_rootzone_mean"
    },

    "air_temperature": {
        "source": "tair_2m_C",
        "windows": [7, 14, 30, 60, 90],
        "operation": "mean",
        "prefix": "tair_mean"
    },

    "potential_evaporation": {
        "source": "pet_magnitude_mm_day",
        "windows": [7, 14, 30, 60, 90],
        "operation": "sum",
        "prefix": "pet_sum"
    }
}


# =============================================================================
# 4. WINDOW DEFINITION TABLE
# =============================================================================

window_definition_rows = []


for family, spec in WINDOWS.items():

    for w in spec["windows"]:

        output_name = (
            f"{spec['prefix']}_{w}d"
        )

        window_definition_rows.append({

            "hydroclimate_family":
                family,

            "source_variable":
                spec["source"],

            "window_days":
                w,

            "aggregation":
                spec["operation"],

            "event_day_included":
                True,

            "window_start_relative_day":
                -(w - 1),

            "window_end_relative_day":
                0,

            "output_variable":
                output_name
        })


window_def = pd.DataFrame(
    window_definition_rows
)


print("\n" + "=" * 96)
print("PRE-SPECIFIED WINDOW ARCHITECTURE")
print("=" * 96)

print(window_def.to_string(index=False))

print(
    "\nTotal candidate antecedent metrics:",
    len(window_def)
)


if len(window_def) != 29:

    raise RuntimeError(
        f"Expected 29 antecedent metrics, "
        f"found {len(window_def)}."
    )


# =============================================================================
# 5. SAVE / VERIFY FROZEN WINDOW DEFINITION
# =============================================================================
#
# If the prior temporal-architecture block already created
# HYDROCLIMATE_WINDOW_DEFINITION_v1.csv, we do NOT silently overwrite a
# conflicting definition.
# =============================================================================

if WINDOW_DEF_FILE.exists():

    old_def = pd.read_csv(
        WINDOW_DEF_FILE
    )

    compare_cols = [
        "hydroclimate_family",
        "source_variable",
        "window_days",
        "aggregation",
        "event_day_included",
        "window_start_relative_day",
        "window_end_relative_day",
        "output_variable"
    ]

    # Compare only if the old file contains the same explicit schema.
    if all(c in old_def.columns for c in compare_cols):

        old_cmp = (
            old_def[compare_cols]
            .astype(str)
            .sort_values(compare_cols)
            .reset_index(drop=True)
        )

        new_cmp = (
            window_def[compare_cols]
            .astype(str)
            .sort_values(compare_cols)
            .reset_index(drop=True)
        )

        if not old_cmp.equals(new_cmp):

            raise RuntimeError(
                "STOP: existing HYDROCLIMATE_WINDOW_DEFINITION_v1.csv "
                "conflicts with the current pre-specified architecture."
            )

        print(
            "\nExisting window definition matches current architecture: PASS"
        )

    else:

        print(
            "\nNOTE: Existing window-definition file uses an older schema."
        )

        print(
            "It will NOT be overwritten. Current architecture will be "
            "audited independently in BLOCK 8.2."
        )

else:

    window_def.to_csv(
        WINDOW_DEF_FILE,
        index=False
    )

    print(
        "\nWindow definition frozen:",
        WINDOW_DEF_FILE
    )


# =============================================================================
# 6. CALCULATE DAILY ANTECEDENT WINDOWS
# =============================================================================
#
# min_periods = window length.
#
# Therefore a 90-day value exists ONLY if all 90 calendar days are
# available in the daily series.
#
# We never calculate a "90-day" mean from 70 available days.
# =============================================================================

daily_windows = daily[
    [
        "date",
        "precipitation_mm_day",
        "sm_surface_0_7cm",
        "sm_rootzone_0_100cm",
        "tair_2m_C",
        "pet_magnitude_mm_day"
    ]
].copy()


window_columns = []


for family, spec in WINDOWS.items():

    source = spec["source"]

    for w in spec["windows"]:

        col = f"{spec['prefix']}_{w}d"

        window_columns.append(col)

        roller = daily[source].rolling(
            window=w,
            min_periods=w
        )

        if spec["operation"] == "sum":

            daily_windows[col] = roller.sum()

        elif spec["operation"] == "mean":

            daily_windows[col] = roller.mean()

        else:

            raise ValueError(
                f"Unknown operation: {spec['operation']}"
            )


print("\nCalculated window columns:", len(window_columns))


# =============================================================================
# 7. EXPECTED EDGE NA COUNTS
# =============================================================================
#
# Because daily extraction begins 2022-12-01:
#
# 1d  -> 0 leading NA
# 3d  -> 2
# 7d  -> 6
# ...
# 90d -> 89
#
# These are EXPECTED at the beginning of the DAILY window table.
#
# They must NOT affect study events because first sampling occurs
# 2023-03-14, >90 days after extraction began.
# =============================================================================

edge_qa_rows = []


for _, r in window_def.iterrows():

    col = r["output_variable"]

    w = int(
        r["window_days"]
    )

    observed_na = int(
        daily_windows[col].isna().sum()
    )

    expected_na = w - 1

    edge_qa_rows.append({

        "variable":
            col,

        "window_days":
            w,

        "expected_leading_na":
            expected_na,

        "observed_total_na":
            observed_na,

        "pass":
            observed_na == expected_na
    })


edge_qa = pd.DataFrame(
    edge_qa_rows
)


print("\n" + "=" * 96)
print("DAILY WINDOW EDGE QA")
print("=" * 96)

print(edge_qa.to_string(index=False))


# =============================================================================
# 8. MANUAL WINDOW IDENTITY CHECKS
# =============================================================================
#
# Independently reconstruct selected windows using explicit date slicing.
# This guards against off-by-one errors.
# =============================================================================

check_dates = [

    pd.Timestamp("2023-03-14"),

    pd.Timestamp("2024-06-29"),

    pd.Timestamp("2025-11-25"),

    pd.Timestamp("2026-03-23")
]


manual_checks = []


def explicit_window(
    df,
    end_date,
    source,
    days,
    operation
):

    start_date = (
        end_date -
        pd.Timedelta(days=days - 1)
    )

    x = df.loc[
        df["date"].between(
            start_date,
            end_date
        ),
        source
    ]

    if len(x) != days:

        return np.nan, len(x)

    if operation == "sum":

        return x.sum(), len(x)

    elif operation == "mean":

        return x.mean(), len(x)

    else:

        raise ValueError(operation)


test_specs = [

    (
        "precipitation_mm_day",
        7,
        "sum",
        "ppt_sum_7d"
    ),

    (
        "precipitation_mm_day",
        90,
        "sum",
        "ppt_sum_90d"
    ),

    (
        "sm_surface_0_7cm",
        30,
        "mean",
        "sm_surface_mean_30d"
    ),

    (
        "sm_rootzone_0_100cm",
        90,
        "mean",
        "sm_rootzone_mean_90d"
    ),

    (
        "tair_2m_C",
        30,
        "mean",
        "tair_mean_30d"
    ),

    (
        "pet_magnitude_mm_day",
        60,
        "sum",
        "pet_sum_60d"
    ),
]


daily_lookup = (
    daily_windows
    .set_index("date")
)


for d in check_dates:

    if d not in daily_lookup.index:
        continue

    for source, days, op, output_col in test_specs:

        explicit_value, n_days = explicit_window(
            daily,
            d,
            source,
            days,
            op
        )

        rolling_value = daily_lookup.loc[
            d,
            output_col
        ]

        abs_error = abs(
            explicit_value -
            rolling_value
        )

        manual_checks.append({

            "date":
                d,

            "source":
                source,

            "window_days":
                days,

            "operation":
                op,

            "output_variable":
                output_col,

            "n_explicit_days":
                n_days,

            "explicit_value":
                explicit_value,

            "rolling_value":
                rolling_value,

            "abs_error":
                abs_error,

            "pass":
                (
                    n_days == days and
                    np.isfinite(abs_error) and
                    abs_error < 1e-10
                )
        })


manual_qa = pd.DataFrame(
    manual_checks
)


print("\n" + "=" * 96)
print("MANUAL OFF-BY-ONE / WINDOW IDENTITY QA")
print("=" * 96)

print(
    manual_qa[
        [
            "date",
            "output_variable",
            "n_explicit_days",
            "abs_error",
            "pass"
        ]
    ].to_string(index=False)
)


# =============================================================================
# 9. JOIN WINDOW BANK TO 146 EVENTS
# =============================================================================

event_windows = events.merge(
    daily_windows[
        ["date"] + window_columns
    ],
    how="left",
    left_on="hydroclimate_event_date",
    right_on="date",
    validate="many_to_one",
    indicator="_window_join"
)


# =============================================================================
# 10. EVENT WINDOW QA
# =============================================================================

event_join_counts = (
    event_windows["_window_join"]
    .value_counts(dropna=False)
    .rename_axis("join_status")
    .reset_index(name="n")
)


event_window_nulls = (
    event_windows[window_columns]
    .isna()
    .sum()
    .rename("n_null")
    .reset_index()
    .rename(columns={"index": "variable"})
)


n_event_matches = int(
    (event_windows["_window_join"] == "both").sum()
)

n_event_unmatched = int(
    (event_windows["_window_join"] != "both").sum()
)

total_event_window_nulls = int(
    event_windows[window_columns]
    .isna()
    .sum()
    .sum()
)


print("\n" + "=" * 96)
print("EVENT WINDOW INTEGRATION QA")
print("=" * 96)

print(event_join_counts.to_string(index=False))

print(
    "\nEvent matches:",
    n_event_matches
)

print(
    "Event unmatched:",
    n_event_unmatched
)

print(
    "Total event-window nulls:",
    total_event_window_nulls
)

print("\nNulls by candidate metric:")
print(
    event_window_nulls.to_string(index=False)
)


# =============================================================================
# 11. EXACT DATE IDENTITY
# =============================================================================

event_date_mismatch = int(
    (
        event_windows["hydroclimate_event_date"] !=
        event_windows["date"]
    ).sum()
)


print(
    "\nEvent-date/window-date mismatches:",
    event_date_mismatch
)


# =============================================================================
# 12. DUPLICATED EVENT-DATE CONSISTENCY
# =============================================================================
#
# Multiple samples collected on the same calendar date MUST receive
# exactly the same catchment-scale antecedent hydroclimate values.
# =============================================================================

same_date_consistency_rows = []


for d, g in event_windows.groupby(
    "hydroclimate_event_date"
):

    if len(g) <= 1:
        continue

    max_unique = (
        g[window_columns]
        .nunique(dropna=False)
        .max()
    )

    same_date_consistency_rows.append({

        "hydroclimate_event_date":
            d,

        "n_events":
            len(g),

        "max_unique_values_any_window":
            int(max_unique),

        "pass":
            int(max_unique) == 1
    })


same_date_qa = pd.DataFrame(
    same_date_consistency_rows
)


if len(same_date_qa):

    same_date_pass = bool(
        same_date_qa["pass"].all()
    )

else:

    same_date_pass = True


print("\n" + "=" * 96)
print("SAME-DATE REPLICATE CONSISTENCY")
print("=" * 96)

print(
    "Unique event dates:",
    events["hydroclimate_event_date"].nunique()
)

print(
    "Dates containing >1 event:",
    len(same_date_qa)
)

print(
    "All same-date events receive identical windows:",
    same_date_pass
)


# =============================================================================
# 13. WINDOW NESTING SANITY CHECK — PRECIPITATION
# =============================================================================
#
# Because precipitation is non-negative and windows are nested:
#
# PPT_1 <= PPT_3 <= PPT_7 <= ... <= PPT_90
#
# for every event.
#
# This is a useful independent sanity check.
# =============================================================================

ppt_cols = [
    "ppt_sum_1d",
    "ppt_sum_3d",
    "ppt_sum_7d",
    "ppt_sum_14d",
    "ppt_sum_30d",
    "ppt_sum_60d",
    "ppt_sum_90d"
]


ppt_nesting_fail = np.zeros(
    len(event_windows),
    dtype=bool
)


for left, right in zip(
    ppt_cols[:-1],
    ppt_cols[1:]
):

    ppt_nesting_fail |= (
        event_windows[left] >
        event_windows[right] + 1e-10
    ).to_numpy()


n_ppt_nesting_fail = int(
    ppt_nesting_fail.sum()
)


print("\n" + "=" * 96)
print("PRECIPITATION WINDOW NESTING QA")
print("=" * 96)

print(
    "Events violating nested precipitation sums:",
    n_ppt_nesting_fail
)


# =============================================================================
# 14. DESCRIPTIVE EVENT-WINDOW TABLE
# =============================================================================

event_window_descriptives = (
    event_windows[window_columns]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)


print("\n" + "=" * 96)
print("EVENT WINDOW DESCRIPTIVES — PREVIEW")
print("=" * 96)

print(
    event_window_descriptives[
        [
            "variable",
            "count",
            "mean",
            "std",
            "min",
            "50%",
            "max"
        ]
    ]
    .head(29)
    .to_string(index=False)
)


# =============================================================================
# 15. FINAL GATE
# =============================================================================

gate_checks = {

    "daily_rows_1209":
        len(daily) == 1209,

    "event_rows_146":
        len(events) == 146,

    "window_metrics_29":
        len(window_columns) == 29,

    "daily_edge_na_pattern":
        bool(edge_qa["pass"].all()),

    "manual_window_identity":
        bool(manual_qa["pass"].all()),

    "all_events_joined":
        n_event_matches == 146,

    "zero_unmatched_events":
        n_event_unmatched == 0,

    "zero_event_window_nulls":
        total_event_window_nulls == 0,

    "exact_event_date_identity":
        event_date_mismatch == 0,

    "same_date_consistency":
        same_date_pass,

    "precipitation_nesting":
        n_ppt_nesting_fail == 0
}


gate_df = pd.DataFrame({
    "check":
        gate_checks.keys(),

    "pass":
        gate_checks.values()
})


BLOCK82_PASS = all(
    gate_checks.values()
)


print("\n" + "=" * 96)
print("BLOCK 8.2 FINAL GATE")
print("=" * 96)

print(
    gate_df.to_string(index=False)
)

print(
    "\nBLOCK 8.2 PASS:",
    BLOCK82_PASS
)


if not BLOCK82_PASS:

    raise RuntimeError(
        "BLOCK 8.2 STOP — antecedent-window bank failed QA."
    )


# =============================================================================
# 16. CLEAN + SAVE
# =============================================================================

event_windows = event_windows.drop(
    columns=["_window_join"]
)


daily_windows.to_csv(
    DAILY_WINDOW_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


event_windows.to_csv(
    EVENT_WINDOW_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("\nDaily antecedent-window bank:")
print(DAILY_WINDOW_FILE)

print("\n146-event antecedent-window bank:")
print(EVENT_WINDOW_FILE)


# =============================================================================
# 17. AUDIT WORKBOOK
# =============================================================================

with pd.ExcelWriter(
    AUDIT_FILE,
    engine="openpyxl"
) as writer:

    window_def.to_excel(
        writer,
        sheet_name="window_definition",
        index=False
    )

    edge_qa.to_excel(
        writer,
        sheet_name="daily_edge_QA",
        index=False
    )

    manual_qa.to_excel(
        writer,
        sheet_name="manual_window_QA",
        index=False
    )

    event_join_counts.to_excel(
        writer,
        sheet_name="event_join_QA",
        index=False
    )

    event_window_nulls.to_excel(
        writer,
        sheet_name="event_null_QA",
        index=False
    )

    same_date_qa.to_excel(
        writer,
        sheet_name="same_date_QA",
        index=False
    )

    event_window_descriptives.to_excel(
        writer,
        sheet_name="event_descriptives",
        index=False
    )

    gate_df.to_excel(
        writer,
        sheet_name="final_gate",
        index=False
    )


# =============================================================================
# 18. HASHES / MANIFEST
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {

    "block":
        "8.2",

    "title":
        "Antecedent hydroclimate window bank",

    "event_day_included":
        True,

    "daily_rows":
        int(len(daily)),

    "event_rows":
        int(len(event_windows)),

    "unique_event_dates":
        int(
            events[
                "hydroclimate_event_date"
            ].nunique()
        ),

    "candidate_metrics":
        int(len(window_columns)),

    "precipitation_windows_days":
        [1, 3, 7, 14, 30, 60, 90],

    "soil_moisture_windows_days":
        [1, 7, 14, 30, 60, 90],

    "temperature_windows_days":
        [7, 14, 30, 60, 90],

    "pet_windows_days":
        [7, 14, 30, 60, 90],

    "response_variables_used":
        False,

    "window_selection_performed":
        False,

    "memory_estimated":
        False,

    "models_fitted":
        False,

    "block_pass":
        bool(BLOCK82_PASS),

    "hashes": {

        "daily_master":
            sha256_file(DAILY_FILE),

        "event_calendar":
            sha256_file(EVENT_FILE),

        "daily_window_bank":
            sha256_file(DAILY_WINDOW_FILE),

        "event_window_bank":
            sha256_file(EVENT_WINDOW_FILE)
    }
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# =============================================================================
# 19. FINAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 96)
print("BLOCK 8.2 — FINAL DIAGNOSTIC")
print("=" * 96)

print(
    "Daily hydroclimate rows             :",
    len(daily)
)

print(
    "Hydroclimate events                 :",
    len(event_windows)
)

print(
    "Unique event dates                  :",
    events["hydroclimate_event_date"].nunique()
)

print(
    "Candidate antecedent metrics        :",
    len(window_columns)
)

print(
    "Event-window nulls                  :",
    total_event_window_nulls
)

print(
    "Unmatched events                    :",
    n_event_unmatched
)

print(
    "Event/window date mismatches        :",
    event_date_mismatch
)

print(
    "Same-date consistency PASS          :",
    same_date_pass
)

print(
    "Precipitation nesting failures      :",
    n_ppt_nesting_fail
)

print(
    "Manual identity checks PASS         :",
    bool(manual_qa["pass"].all())
)

print()
print(
    "BLOCK 8.2 PASS                      :",
    BLOCK82_PASS
)

print()
print(
    "Response variables used             : NO"
)

print(
    "Best windows selected               : NO"
)

print(
    "Memory estimated                    : NO"
)

print(
    "Models fitted                       : NO"
)

print()
print(
    "BLOCK 8.2 COMPLETE — WINDOW BANK FROZEN."
)

---
### Imported master block 69

`BLOCK 8.4A-R — MEMORY ANALYSIS DATASET ASSEMBLY & ELIGIBILITY FREEZE`


In [ ]:
# =============================================================================
# BLOCK 8.4A-R — MEMORY ANALYSIS DATASET ASSEMBLY & ELIGIBILITY FREEZE
# REVISED CANONICAL VERSION
#
# Journal of Hydrology — Razuhuillca
#
# PURPOSE
# -------
# Build the authoritative primary memory-analysis dataset using:
#
#   LAB RESPONSES:
#       exact MASTER_ANALYTICAL_v1_1.parameter_id
#
#   FIELD WATER TEMPERATURE:
#       water_temperature_primary_C
#       deduplicated to one physical FIELD response per sample/event identity
#
# Then attach the 29 pre-specified antecedent hydroclimate metrics by exact
# sample_id.
#
# IMPORTANT
# ---------
# - 9 laboratory responses use exact canonical parameter_id.
# - FIELD temperature is NOT treated as an analytical parameter row.
# - No fuzzy matching.
# - No LOD/2.
# - No censor substitution.
# - No averaging of analytical subsamples.
# - No collapsing distinct analytical subsamples.
# - No hydroclimate-informed response selection.
# - No best-window selection.
# - No memory estimation.
# - No statistical models.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib


# =============================================================================
# 0. PATHS
# =============================================================================

ROOT = Path("/content/Huanta_JoH")

MASTER_DIR = ROOT / "03_MASTER"
AUDIT_DIR  = ROOT / "01_AUDIT"

ANALYTICAL_FILE = (
    MASTER_DIR /
    "MASTER_ANALYTICAL_v1_1.csv"
)

PANEL_FILE = (
    MASTER_DIR /
    "PRIMARY_MEMORY_RESPONSE_PANEL_v1.csv"
)

HYDRO_FILE = (
    MASTER_DIR /
    "HYDROCLIMATE_EVENT_ANTECEDENT_WINDOWS_v1.csv"
)

OUTPUT_FILE = (
    MASTER_DIR /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

RESPONSE_MAP_FILE = (
    MASTER_DIR /
    "MEMORY_RESPONSE_MAPPING_v1.csv"
)

ELIGIBILITY_FILE = (
    MASTER_DIR /
    "MEMORY_RESPONSE_ELIGIBILITY_v1.csv"
)

AUDIT_FILE = (
    AUDIT_DIR /
    "BLOCK84A_R_memory_analysis_dataset_assembly.xlsx"
)

MANIFEST_FILE = (
    AUDIT_DIR /
    "BLOCK84A_R_memory_analysis_dataset_manifest.json"
)


print("=" * 112)
print("BLOCK 8.4A-R — MEMORY ANALYSIS DATASET ASSEMBLY & ELIGIBILITY FREEZE")
print("=" * 112)


# =============================================================================
# 1. INPUT GATE
# =============================================================================

required_files = [
    ANALYTICAL_FILE,
    PANEL_FILE,
    HYDRO_FILE
]

missing = [
    str(p)
    for p in required_files
    if not p.exists()
]

if missing:

    raise FileNotFoundError(
        "Missing required files:\n" +
        "\n".join(missing)
    )


analytical = pd.read_csv(
    ANALYTICAL_FILE,
    low_memory=False
)

panel = pd.read_csv(
    PANEL_FILE,
    low_memory=False
)

hydro = pd.read_csv(
    HYDRO_FILE,
    low_memory=False
)


print("\nINPUT DIMENSIONS")
print("-" * 112)

print(
    "Analytical master:",
    analytical.shape
)

print(
    "Frozen primary panel:",
    panel.shape
)

print(
    "Hydroclimate event windows:",
    hydro.shape
)


# =============================================================================
# 2. EXACT FROZEN RESPONSE PANEL
# =============================================================================

LAB_KEYS = [

    "AC::pH_SOL::NO_UNIT",
    "AC::Na::mg/L",
    "AC::C.E._SOL::dS/m",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "AC::NH4::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
    "AC::SO4_ppm::mg/L"
]

TEMP_KEY = (
    "FIELD::Water temperature::degC"
)

EXPECTED_KEYS = (
    LAB_KEYS +
    [TEMP_KEY]
)


if "response_key" not in panel.columns:

    raise RuntimeError(
        "Frozen panel lacks response_key."
    )


panel_keys = (
    panel["response_key"]
    .astype(str)
    .tolist()
)


panel_gate = (

    len(panel) == 10 and
    panel["response_key"].nunique() == 10 and
    set(panel_keys) == set(EXPECTED_KEYS)
)


if not panel_gate:

    raise RuntimeError(
        "Frozen primary panel does not match "
        "the 10-response BLOCK 7.2C freeze."
    )


print("\nFrozen 10-response panel: PASS")


# =============================================================================
# 3. AUTHORITATIVE ANALYTICAL DICTIONARY GATE
# =============================================================================

required_analytical_cols = [

    "parameter_id",
    "value_raw",
    "value_numeric",
    "censored",
    "censor_type",
    "censor_limit",

    "sample_id",
    "field_linked_sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",

    "parameter_std",
    "unit_final",
    "eligible_primary_response",

    "water_temperature_primary_C",
    "water_temperature_primary_source_final"
]


missing_cols = [
    c
    for c in required_analytical_cols
    if c not in analytical.columns
]


if missing_cols:

    raise RuntimeError(
        "Required analytical columns missing:\n" +
        "\n".join(missing_cols)
    )


# Exact dictionary mapping.

dict_rows = (
    analytical.loc[
        analytical["parameter_id"]
        .isin(LAB_KEYS),
        [
            "parameter_id",
            "dataset",
            "domain",
            "parameter_std",
            "unit_final",
            "eligible_primary_response",
            "analysis_role",
            "final_model_input_status"
        ]
    ]
    .drop_duplicates()
)


dict_counts = (
    dict_rows[
        "parameter_id"
    ]
    .value_counts()
)


dictionary_exact = (

    set(
        dict_rows["parameter_id"]
    ) == set(LAB_KEYS)

    and

    all(
        dict_counts.get(k, 0) == 1
        for k in LAB_KEYS
    )
)


print("\n" + "=" * 112)
print("EXACT CANONICAL LAB RESPONSE MAPPING")
print("=" * 112)

print(
    dict_rows
    .sort_values("parameter_id")
    .to_string(index=False)
)


if not dictionary_exact:

    raise RuntimeError(
        "STOP — one or more frozen laboratory responses "
        "do not map uniquely by parameter_id."
    )


if not (
    dict_rows[
        "eligible_primary_response"
    ]
    .fillna(False)
    .astype(bool)
    .all()
):

    raise RuntimeError(
        "STOP — one or more frozen laboratory responses "
        "are not flagged eligible_primary_response."
    )


print("\nExact canonical parameter_id mapping: PASS")


# =============================================================================
# 4. EXTRACT NINE LABORATORY RESPONSES
# =============================================================================
#
# IMPORTANT:
# Each analytical row remains an analytical row.
#
# We do NOT aggregate same-sample subsamples.
# =============================================================================

lab = analytical.loc[
    analytical[
        "parameter_id"
    ].isin(LAB_KEYS)
].copy()


lab["response_key"] = (
    lab["parameter_id"]
)


lab["response_origin"] = (
    "LAB_ANALYTICAL"
)


lab["response_value_raw"] = (
    lab["value_raw"]
)


lab["response_value_numeric"] = (
    lab["value_numeric"]
)


lab["response_censored"] = (
    lab["censored"]
    .fillna(False)
    .astype(bool)
)


lab["response_censor_type"] = (
    lab["censor_type"]
)


lab["response_censor_limit"] = (
    lab["censor_limit"]
)


lab["temperature_primary_source"] = (
    np.nan
)


lab["temperature_identity_policy"] = (
    np.nan
)


print("\n" + "=" * 112)
print("LABORATORY PRIMARY RESPONSES")
print("=" * 112)

lab_summary = (
    lab
    .groupby(
        "response_key",
        dropna=False
    )
    .agg(
        n_rows=("sample_id", "size"),
        n_samples=("sample_id", "nunique"),
        n_analytical_sample_id=(
            "analytical_sample_id",
            "nunique"
        ),
        n_campaigns=(
            "campaign_id",
            "nunique"
        ),
        n_numeric=(
            "response_value_numeric",
            lambda x: x.notna().sum()
        ),
        n_censored=(
            "response_censored",
            "sum"
        )
    )
    .reset_index()
)


print(
    lab_summary.to_string(
        index=False
    )
)


# =============================================================================
# 5. FIELD WATER TEMPERATURE — BUILD PHYSICAL RESPONSE TABLE
# =============================================================================
#
# water_temperature_primary_C is repeated across analytical rows because the
# FIELD metadata were joined onto the analytical master.
#
# Therefore:
#   DO NOT use all master rows.
#
# We first reduce to the unique identity/value combinations carrying the
# physical FIELD response.
# =============================================================================

temp_source_cols = [

    "sample_id",
    "field_linked_sample_id",
    "field_observation_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",
    "field_sampling_date",

    "water_temperature_primary_C",
    "water_temperature_primary_source_final",
    "water_temperature_MT_sensitivity_C"
]


temp_source_cols = [
    c
    for c in temp_source_cols
    if c in analytical.columns
]


temp0 = (
    analytical[
        temp_source_cols
    ]
    .drop_duplicates()
)


temp0 = temp0.loc[
    temp0[
        "water_temperature_primary_C"
    ].notna()
].copy()


print("\n" + "=" * 112)
print("FIELD WATER TEMPERATURE IDENTITY AUDIT")
print("=" * 112)

print(
    "Metadata-deduplicated rows with primary temperature:",
    len(temp0)
)

print(
    "Unique sample_id:",
    temp0["sample_id"].nunique()
)

print(
    "Unique FIELD observations:",
    temp0["field_observation_id"].nunique(
        dropna=True
    )
)


# =============================================================================
# 6. CHECK WHETHER ANY sample_id HAS CONFLICTING PRIMARY TEMPERATURE VALUES
# =============================================================================

temp_conflict = (
    temp0
    .groupby(
        "sample_id",
        dropna=False
    )
    .agg(
        n_rows=(
            "water_temperature_primary_C",
            "size"
        ),
        n_values=(
            "water_temperature_primary_C",
            "nunique"
        ),
        n_field_observations=(
            "field_observation_id",
            "nunique"
        )
    )
    .reset_index()
)


temp_value_conflicts = temp_conflict.loc[
    temp_conflict[
        "n_values"
    ] > 1
].copy()


print(
    "sample_id with conflicting temperature values:",
    len(temp_value_conflicts)
)


if len(temp_value_conflicts):

    print(
        temp_value_conflicts.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — same sample_id carries conflicting "
        "primary FIELD temperature values."
    )


# =============================================================================
# 7. CHECK FIELD-OBSERVATION → TEMPERATURE UNIQUENESS
# =============================================================================

field_temp_conflict = (
    temp0.loc[
        temp0[
            "field_observation_id"
        ].notna()
    ]
    .groupby(
        "field_observation_id"
    )
    .agg(
        n_values=(
            "water_temperature_primary_C",
            "nunique"
        ),
        n_samples=(
            "sample_id",
            "nunique"
        )
    )
    .reset_index()
)


field_value_conflicts = (
    field_temp_conflict.loc[
        field_temp_conflict[
            "n_values"
        ] > 1
    ]
)


if len(field_value_conflicts):

    print(
        field_value_conflicts.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — same FIELD observation carries "
        "conflicting primary temperature values."
    )


# =============================================================================
# 8. COLLAPSE ONLY METADATA DUPLICATION OF THE SAME PHYSICAL TEMPERATURE
# =============================================================================
#
# This is NOT analytical averaging.
#
# If the same sample_id appears repeatedly because multiple analytical rows
# inherited the same FIELD temperature, keep one physical temperature record.
# =============================================================================

temp = (
    temp0
    .sort_values(
        [
            "sample_id",
            "field_observation_id"
        ],
        na_position="last"
    )
    .drop_duplicates(
        subset=[
            "sample_id"
        ],
        keep="first"
    )
    .copy()
)


if temp[
    "sample_id"
].duplicated().any():

    raise RuntimeError(
        "Temperature sample_id remains duplicated."
    )


temp["response_key"] = (
    TEMP_KEY
)

temp["response_origin"] = (
    "FIELD_PHYSICAL"
)

temp["response_value_raw"] = (
    temp[
        "water_temperature_primary_C"
    ]
)

temp["response_value_numeric"] = (
    temp[
        "water_temperature_primary_C"
    ]
)

temp["response_censored"] = False

temp["response_censor_type"] = np.nan

temp["response_censor_limit"] = np.nan

temp["analytical_sample_id"] = np.nan

temp["parameter_id"] = TEMP_KEY

temp["parameter_std"] = (
    "Water temperature"
)

temp["unit_final"] = (
    "degC"
)

temp["temperature_primary_source"] = (
    temp[
        "water_temperature_primary_source_final"
    ]
)

temp["temperature_identity_policy"] = (
    "ONE_PHYSICAL_RESPONSE_PER_SAMPLE_ID__NO_AVERAGING"
)


print(
    "Final physical temperature response rows:",
    len(temp)
)

print(
    "Final unique temperature sample_id:",
    temp["sample_id"].nunique()
)

print(
    "Final unique FIELD observations:",
    temp["field_observation_id"].nunique(
        dropna=True
    )
)


# =============================================================================
# 9. COMMON RESPONSE SCHEMA
# =============================================================================

common_cols = [

    "response_key",
    "response_origin",

    "sample_id",
    "field_linked_sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",

    "parameter_id",
    "parameter_std",
    "unit_final",

    "response_value_raw",
    "response_value_numeric",

    "response_censored",
    "response_censor_type",
    "response_censor_limit",

    "temperature_primary_source",
    "temperature_identity_policy"
]


# Add absent columns without inventing scientific values.

for c in common_cols:

    if c not in lab.columns:
        lab[c] = np.nan

    if c not in temp.columns:
        temp[c] = np.nan


lab_common = (
    lab[
        common_cols
    ].copy()
)

temp_common = (
    temp[
        common_cols
    ].copy()
)


responses = pd.concat(
    [
        lab_common,
        temp_common
    ],
    ignore_index=True
)


# =============================================================================
# 10. ATTACH FROZEN PANEL METADATA
# =============================================================================

panel_meta_cols = [
    c
    for c in [
        "response_key",
        "response_family",
        "memory_panel_role",
        "scientific_role",
        "panel_version",
        "freeze_status",
        "restoration_status",
        "response_informed_by_hydroclimate",
        "automatic_selection"
    ]
    if c in panel.columns
]


responses = responses.merge(

    panel[
        panel_meta_cols
    ],

    how="left",

    on="response_key",

    validate="many_to_one"
)


if responses[
    "memory_panel_role"
].isna().any():

    raise RuntimeError(
        "Frozen response metadata failed to attach."
    )


# =============================================================================
# 11. RESPONSE IDENTITY AUDIT BEFORE HYDROCLIMATE
# =============================================================================

response_counts = (
    responses
    .groupby(
        "response_key",
        dropna=False
    )
    .agg(
        n_rows=(
            "sample_id",
            "size"
        ),
        n_unique_samples=(
            "sample_id",
            "nunique"
        ),
        n_field_observations=(
            "field_observation_id",
            "nunique"
        ),
        n_analytical_subsamples=(
            "analytical_sample_id",
            "nunique"
        ),
        n_campaigns=(
            "campaign_id",
            "nunique"
        ),
        n_numeric=(
            "response_value_numeric",
            lambda x: x.notna().sum()
        ),
        n_censored=(
            "response_censored",
            "sum"
        )
    )
    .reset_index()
)


print("\n" + "=" * 112)
print("COMBINED PRIMARY RESPONSE COVERAGE")
print("=" * 112)

print(
    response_counts.to_string(
        index=False
    )
)


if set(
    response_counts[
        "response_key"
    ]
) != set(EXPECTED_KEYS):

    raise RuntimeError(
        "Not all 10 frozen primary responses are represented."
    )


# =============================================================================
# 12. HYDROCLIMATE BANK GATE
# =============================================================================

hydro_window_cols = [

    "ppt_sum_1d",
    "ppt_sum_3d",
    "ppt_sum_7d",
    "ppt_sum_14d",
    "ppt_sum_30d",
    "ppt_sum_60d",
    "ppt_sum_90d",

    "sm_surface_mean_1d",
    "sm_surface_mean_7d",
    "sm_surface_mean_14d",
    "sm_surface_mean_30d",
    "sm_surface_mean_60d",
    "sm_surface_mean_90d",

    "sm_rootzone_mean_1d",
    "sm_rootzone_mean_7d",
    "sm_rootzone_mean_14d",
    "sm_rootzone_mean_30d",
    "sm_rootzone_mean_60d",
    "sm_rootzone_mean_90d",

    "tair_mean_7d",
    "tair_mean_14d",
    "tair_mean_30d",
    "tair_mean_60d",
    "tair_mean_90d",

    "pet_sum_7d",
    "pet_sum_14d",
    "pet_sum_30d",
    "pet_sum_60d",
    "pet_sum_90d"
]


missing_windows = [
    c
    for c in hydro_window_cols
    if c not in hydro.columns
]


if missing_windows:

    raise RuntimeError(
        "Missing hydroclimate metrics:\n" +
        "\n".join(missing_windows)
    )


if len(hydro) != 146:

    raise RuntimeError(
        f"Expected 146 hydroclimate event rows; found {len(hydro)}."
    )


if hydro[
    "sample_id"
].duplicated().any():

    dups = (
        hydro.loc[
            hydro[
                "sample_id"
            ].duplicated(
                keep=False
            )
        ]
        .sort_values(
            "sample_id"
        )
    )

    print(
        dups[
            [
                c
                for c in [
                    "sample_id",
                    "field_linked_sample_id",
                    "field_observation_id",
                    "hydroclimate_event_date"
                ]
                if c in dups.columns
            ]
        ].to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — hydroclimate sample_id is not unique."
    )


if hydro[
    hydro_window_cols
].isna().any().any():

    raise RuntimeError(
        "Hydroclimate window bank contains nulls."
    )


print("\nHydroclimate bank: 146 rows × 29 windows — PASS")


# =============================================================================
# 13. HYDROCLIMATE MERGE FIELDS
# =============================================================================

hydro_meta_cols = [

    "sample_id",
    "hydroclimate_event_date",
    "hydroclimate_date_source",
    "hydroclimate_temporal_role",
    "hydroclimate_integration_eligible"
]


hydro_keep = [
    c
    for c in (
        hydro_meta_cols +
        hydro_window_cols
    )
    if c in hydro.columns
]


hydro_for_merge = (
    hydro[
        hydro_keep
    ]
    .copy()
)


# =============================================================================
# 14. EXACT sample_id JOIN
# =============================================================================

memory = responses.merge(

    hydro_for_merge,

    how="left",

    on="sample_id",

    validate="many_to_one",

    indicator="_hydro_join"
)


join_summary = (
    memory[
        "_hydro_join"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "join_status"
    )
    .reset_index(
        name="n_rows"
    )
)


print("\n" + "=" * 112)
print("RESPONSE × HYDROCLIMATE EXACT JOIN")
print("=" * 112)

print(
    join_summary.to_string(
        index=False
    )
)


n_unmatched = int(
    (
        memory[
            "_hydro_join"
        ] != "both"
    ).sum()
)


unmatched = (
    memory.loc[
        memory[
            "_hydro_join"
        ] != "both",
        [
            "response_key",
            "response_origin",
            "sample_id",
            "field_linked_sample_id",
            "field_observation_id",
            "analytical_sample_id"
        ]
    ]
    .drop_duplicates()
)


print(
    "\nUnmatched response rows:",
    n_unmatched
)


if len(unmatched):

    print("\nUnmatched identities:")
    print(
        unmatched.to_string(
            index=False
        )
    )


# =============================================================================
# 15. HYDROCLIMATE COMPLETENESS
# =============================================================================

total_hydro_nulls = int(
    memory[
        hydro_window_cols
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Total hydroclimate-window nulls:",
    total_hydro_nulls
)


# =============================================================================
# 16. RESPONSE × EVENT-DATE COVERAGE
# =============================================================================

memory[
    "hydroclimate_event_date"
] = pd.to_datetime(
    memory[
        "hydroclimate_event_date"
    ],
    errors="coerce"
)


eligibility = (
    memory
    .groupby(
        "response_key",
        dropna=False
    )
    .agg(
        response_origin=(
            "response_origin",
            "first"
        ),

        n_rows=(
            "sample_id",
            "size"
        ),

        n_unique_samples=(
            "sample_id",
            "nunique"
        ),

        n_unique_field_observations=(
            "field_observation_id",
            "nunique"
        ),

        n_analytical_subsamples=(
            "analytical_sample_id",
            "nunique"
        ),

        n_campaigns=(
            "campaign_id",
            "nunique"
        ),

        n_unique_hydroclimate_dates=(
            "hydroclimate_event_date",
            "nunique"
        ),

        n_numeric=(
            "response_value_numeric",
            lambda x: x.notna().sum()
        ),

        n_censored=(
            "response_censored",
            "sum"
        ),

        n_hydro_matched=(
            "_hydro_join",
            lambda x:
                (x == "both").sum()
        )
    )
    .reset_index()
)


eligibility[
    "censoring_pct"
] = (
    100 *
    eligibility[
        "n_censored"
    ] /
    eligibility[
        "n_rows"
    ]
)


eligibility[
    "memory_analysis_role"
] = np.where(

    eligibility[
        "n_censored"
    ] > 0,

    "PRIMARY__CENSOR_AWARE_REQUIRED",

    "PRIMARY__DIRECT_CONTINUOUS"
)


eligibility[
    "automatic_exclusion"
] = False


eligibility[
    "response_informed_window_selection"
] = False


eligibility[
    "memory_estimated"
] = False


print("\n" + "=" * 112)
print("PRIMARY MEMORY RESPONSE ELIGIBILITY")
print("=" * 112)

print(
    eligibility.to_string(
        index=False
    )
)


# =============================================================================
# 17. ANALYTICAL SUBSAMPLE DUPLICATION AUDIT
# =============================================================================
#
# This specifically checks same sample_id × response_key.
#
# We preserve them.
# =============================================================================

sample_response_counts = (
    memory
    .groupby(
        [
            "sample_id",
            "response_key"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="n_rows"
    )
)


repeated_sample_response = (
    sample_response_counts.loc[
        sample_response_counts[
            "n_rows"
        ] > 1
    ]
    .copy()
)


print("\n" + "=" * 112)
print("SAMPLE × RESPONSE REPLICATION AUDIT")
print("=" * 112)

print(
    "Repeated sample×response combinations:",
    len(
        repeated_sample_response
    )
)


if len(repeated_sample_response):

    print(
        repeated_sample_response
        .to_string(
            index=False
        )
    )


# =============================================================================
# 18. TEMPERATURE SPECIAL QA
# =============================================================================

temp_memory = memory.loc[
    memory[
        "response_key"
    ] == TEMP_KEY
].copy()


temperature_gate = (

    len(temp_memory) ==
    temp_memory[
        "sample_id"
    ].nunique()

    and

    not temp_memory[
        "sample_id"
    ].duplicated().any()

    and

    (
        temp_memory[
            "response_censored"
        ] == False
    ).all()

    and

    temp_memory[
        "response_value_numeric"
    ].notna().all()
)


print("\n" + "=" * 112)
print("TEMPERATURE PRIMARY RESPONSE QA")
print("=" * 112)

print(
    "Rows:",
    len(temp_memory)
)

print(
    "Unique sample_id:",
    temp_memory[
        "sample_id"
    ].nunique()
)

print(
    "Unique FIELD observations:",
    temp_memory[
        "field_observation_id"
    ].nunique(
        dropna=True
    )
)

print(
    "Temperature identity gate:",
    temperature_gate
)


# =============================================================================
# 19. RESPONSE MAPPING TABLE
# =============================================================================

mapping_lab = (
    dict_rows
    .rename(
        columns={
            "parameter_id":
                "response_key"
        }
    )
    .copy()
)


mapping_lab[
    "mapping_method"
] = (
    "EXACT_PARAMETER_ID"
)


mapping_lab[
    "response_origin"
] = (
    "LAB_ANALYTICAL"
)


mapping_temp = pd.DataFrame(
    [
        {
            "response_key":
                TEMP_KEY,

            "dataset":
                "FIELD",

            "domain":
                "FIELD",

            "parameter_std":
                "Water temperature",

            "unit_final":
                "degC",

            "eligible_primary_response":
                True,

            "analysis_role":
                "STANDARD_CONTINUOUS",

            "final_model_input_status":
                "PRIMARY_FIELD_RESPONSE",

            "mapping_method":
                "WATER_TEMPERATURE_PRIMARY_C",

            "response_origin":
                "FIELD_PHYSICAL"
        }
    ]
)


response_mapping = pd.concat(
    [
        mapping_lab,
        mapping_temp
    ],
    ignore_index=True,
    sort=False
)


# =============================================================================
# 20. FINAL GATE
# =============================================================================

gate_checks = {

    "frozen_primary_panel_10":
        panel_gate,

    "lab_primary_responses_9":
        lab[
            "response_key"
        ].nunique() == 9,

    "exact_parameter_id_mapping":
        dictionary_exact,

    "temperature_primary_response_present":
        len(temp_memory) > 0,

    "temperature_identity_unique":
        temperature_gate,

    "all_10_responses_represented":
        memory[
            "response_key"
        ].nunique() == 10,

    "hydro_event_rows_146":
        len(hydro) == 146,

    "hydro_sample_id_unique":
        not hydro[
            "sample_id"
        ].duplicated().any(),

    "antecedent_metrics_29":
        len(
            hydro_window_cols
        ) == 29,

    "all_response_rows_hydro_matched":
        n_unmatched == 0,

    "zero_hydro_window_nulls":
        total_hydro_nulls == 0,

    "no_lod_half_substitution":
        True,

    "no_censor_substitution":
        True,

    "no_analytical_subsample_collapsing":
        True,

    "no_response_guided_window_selection":
        True,

    "no_memory_estimation":
        True,

    "no_models_fitted":
        True
}


gate_df = pd.DataFrame(
    {
        "check":
            gate_checks.keys(),

        "pass":
            gate_checks.values()
    }
)


BLOCK84AR_PASS = all(
    gate_checks.values()
)


print("\n" + "=" * 112)
print("BLOCK 8.4A-R FINAL GATE")
print("=" * 112)

print(
    gate_df.to_string(
        index=False
    )
)

print(
    "\nBLOCK 8.4A-R PASS:",
    BLOCK84AR_PASS
)


if not BLOCK84AR_PASS:

    raise RuntimeError(
        "BLOCK 8.4A-R STOP — "
        "do not proceed to memory-spectrum analysis."
    )


# =============================================================================
# 21. CLEAN INTERNAL MERGE COLUMN
# =============================================================================

memory = memory.drop(
    columns=[
        "_hydro_join"
    ]
)


# =============================================================================
# 22. FREEZE OUTPUTS
# =============================================================================

memory.to_csv(
    OUTPUT_FILE,
    index=False
)


response_mapping.to_csv(
    RESPONSE_MAP_FILE,
    index=False
)


eligibility.to_csv(
    ELIGIBILITY_FILE,
    index=False
)


# =============================================================================
# 23. AUDIT WORKBOOK
# =============================================================================

with pd.ExcelWriter(
    AUDIT_FILE,
    engine="openpyxl"
) as writer:

    panel.to_excel(
        writer,
        sheet_name="frozen_panel",
        index=False
    )

    dict_rows.to_excel(
        writer,
        sheet_name="lab_mapping",
        index=False
    )

    lab_summary.to_excel(
        writer,
        sheet_name="lab_coverage",
        index=False
    )

    temp_conflict.to_excel(
        writer,
        sheet_name="temperature_identity",
        index=False
    )

    response_counts.to_excel(
        writer,
        sheet_name="response_coverage",
        index=False
    )

    eligibility.to_excel(
        writer,
        sheet_name="eligibility",
        index=False
    )

    repeated_sample_response.to_excel(
        writer,
        sheet_name="repeated_sample_response",
        index=False
    )

    unmatched.to_excel(
        writer,
        sheet_name="unmatched_hydro",
        index=False
    )

    response_mapping.to_excel(
        writer,
        sheet_name="response_mapping",
        index=False
    )

    gate_df.to_excel(
        writer,
        sheet_name="final_gate",
        index=False
    )


# =============================================================================
# 24. HASH FUNCTION
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 25. MANIFEST
# =============================================================================

manifest = {

    "block":
        "8.4A-R",

    "title":
        "Memory analysis dataset assembly and eligibility freeze",

    "mapping_architecture": {

        "laboratory_responses":
            "exact parameter_id",

        "field_temperature":
            "water_temperature_primary_C",

        "hydroclimate_join":
            "exact sample_id"
    },

    "n_primary_responses":
        int(
            memory[
                "response_key"
            ].nunique()
        ),

    "n_lab_primary_responses":
        9,

    "n_field_primary_responses":
        1,

    "n_memory_rows":
        int(
            len(memory)
        ),

    "n_unique_samples":
        int(
            memory[
                "sample_id"
            ].nunique()
        ),

    "n_hydroclimate_metrics":
        29,

    "unmatched_hydro_rows":
        int(
            n_unmatched
        ),

    "hydroclimate_window_nulls":
        int(
            total_hydro_nulls
        ),

    "repeated_sample_response_combinations":
        int(
            len(
                repeated_sample_response
            )
        ),

    "temperature_identity_policy":
        "one physical response per sample_id; no averaging",

    "lod_half_substitution":
        False,

    "censored_value_substitution":
        False,

    "analytical_subsample_collapsing":
        False,

    "hydroclimate_response_screening":
        False,

    "best_window_selected":
        False,

    "memory_estimated":
        False,

    "models_fitted":
        False,

    "block_pass":
        bool(
            BLOCK84AR_PASS
        ),

    "hashes": {

        "analytical_master":
            sha256_file(
                ANALYTICAL_FILE
            ),

        "frozen_primary_panel":
            sha256_file(
                PANEL_FILE
            ),

        "hydroclimate_windows":
            sha256_file(
                HYDRO_FILE
            ),

        "memory_dataset":
            sha256_file(
                OUTPUT_FILE
            ),

        "response_mapping":
            sha256_file(
                RESPONSE_MAP_FILE
            ),

        "eligibility":
            sha256_file(
                ELIGIBILITY_FILE
            )
    }
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# =============================================================================
# 26. FINAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 112)
print("BLOCK 8.4A-R — FINAL DIAGNOSTIC")
print("=" * 112)

print(
    "Frozen primary responses                :",
    memory[
        "response_key"
    ].nunique()
)

print(
    "Laboratory primary responses            :",
    lab[
        "response_key"
    ].nunique()
)

print(
    "FIELD primary responses                 :",
    1
)

print(
    "Total response rows                     :",
    len(memory)
)

print(
    "Unique sample_id                        :",
    memory[
        "sample_id"
    ].nunique()
)

print(
    "Hydroclimate event table rows           :",
    len(hydro)
)

print(
    "Antecedent hydroclimate metrics         :",
    len(
        hydro_window_cols
    )
)

print(
    "Unmatched response rows                 :",
    n_unmatched
)

print(
    "Hydroclimate-window nulls               :",
    total_hydro_nulls
)

print(
    "Repeated sample×response combinations   :",
    len(
        repeated_sample_response
    )
)

print(
    "Temperature physical response rows      :",
    len(
        temp_memory
    )
)

print(
    "Temperature unique FIELD observations   :",
    temp_memory[
        "field_observation_id"
    ].nunique(
        dropna=True
    )
)

print()
print(
    "Laboratory mapping                      : EXACT parameter_id"
)

print(
    "Temperature mapping                     : water_temperature_primary_C"
)

print(
    "Hydroclimate linkage                    : EXACT sample_id"
)

print()
print(
    "LOD/2 substitution                      : NO"
)

print(
    "Censored-value substitution             : NO"
)

print(
    "Analytical subsamples collapsed         : NO"
)

print(
    "Response-guided window selection        : NO"
)

print(
    "Best window selected                    : NO"
)

print(
    "Memory estimated                        : NO"
)

print(
    "Models fitted                           : NO"
)

print()
print(
    "Frozen dataset:"
)

print(
    OUTPUT_FILE
)

print()
print(
    "Audit workbook:"
)

print(
    AUDIT_FILE
)

print()
print(
    "Manifest:"
)

print(
    MANIFEST_FILE
)

print()
print(
    "BLOCK 8.4A-R PASS                       :",
    BLOCK84AR_PASS
)

print()
print(
    "BLOCK 8.4A-R COMPLETE — "
    "MEMORY ANALYSIS DATASET FROZEN."
)

---
### Imported master block 72

`BLOCK 8.4B — RESPONSE TRANSFORMATION, CENSORING &`


In [ ]:
# =============================================================================
# BLOCK 8.4B — RESPONSE TRANSFORMATION, CENSORING &
#              MEMORY-INFERENCE ARCHITECTURE FREEZE
#
# Journal of Hydrology — Razuhuillca
#
# PURPOSE
# -------
# Freeze the response-side statistical architecture BEFORE examining any
# response × hydroclimate association.
#
# THIS BLOCK:
#   1. audits the 10 frozen primary responses;
#   2. audits censoring without substitution;
#   3. characterizes response distributions using DETECTED observations only
#      for transformation diagnostics;
#   4. assigns pre-hydroclimate transformation candidates;
#   5. freezes censor-aware vs direct-continuous pathways;
#   6. freezes the later memory-spectrum inference rules.
#
# THIS BLOCK DOES NOT:
#   - correlate responses with hydroclimate;
#   - inspect which window performs best;
#   - select a memory timescale;
#   - fit hydroclimate-response models;
#   - replace censored observations;
#   - use LOD/2;
#   - collapse analytical subsamples.
#
# IMPORTANT
# ---------
# Distributional diagnostics for censored responses are descriptive only.
# They do NOT turn censored responses into detected-only inferential datasets.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import warnings

from scipy.stats import skew, kurtosis

warnings.filterwarnings("ignore")


# =============================================================================
# 0. PATHS
# =============================================================================

ROOT = Path("/content/Huanta_JoH")

MASTER_DIR = ROOT / "03_MASTER"
AUDIT_DIR  = ROOT / "01_AUDIT"

INPUT_FILE = (
    MASTER_DIR /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

PANEL_FILE = (
    MASTER_DIR /
    "PRIMARY_MEMORY_RESPONSE_PANEL_v1.csv"
)

ELIGIBILITY_INPUT = (
    MASTER_DIR /
    "MEMORY_RESPONSE_ELIGIBILITY_v1.csv"
)

ARCHITECTURE_FILE = (
    MASTER_DIR /
    "MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"
)

CENSOR_FILE = (
    MASTER_DIR /
    "MEMORY_RESPONSE_CENSORING_ARCHITECTURE_v1.csv"
)

DISTRIBUTION_FILE = (
    MASTER_DIR /
    "MEMORY_RESPONSE_DISTRIBUTION_AUDIT_v1.csv"
)

INFERENCE_RULES_FILE = (
    MASTER_DIR /
    "MEMORY_INFERENCE_RULES_v1.csv"
)

AUDIT_FILE = (
    AUDIT_DIR /
    "BLOCK84B_response_statistical_architecture.xlsx"
)

MANIFEST_FILE = (
    AUDIT_DIR /
    "BLOCK84B_response_statistical_architecture_manifest.json"
)


print("=" * 116)
print(
    "BLOCK 8.4B — RESPONSE TRANSFORMATION, CENSORING & "
    "MEMORY-INFERENCE ARCHITECTURE FREEZE"
)
print("=" * 116)


# =============================================================================
# 1. INPUT GATE
# =============================================================================

required_files = [
    INPUT_FILE,
    PANEL_FILE,
    ELIGIBILITY_INPUT
]

missing = [
    str(p)
    for p in required_files
    if not p.exists()
]

if missing:

    raise FileNotFoundError(
        "Required file(s) missing:\n" +
        "\n".join(missing)
    )


df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

panel = pd.read_csv(
    PANEL_FILE,
    low_memory=False
)

eligibility_previous = pd.read_csv(
    ELIGIBILITY_INPUT,
    low_memory=False
)


print("\nINPUT DIMENSIONS")
print("-" * 116)

print(
    "Memory analysis dataset:",
    df.shape
)

print(
    "Frozen primary panel:",
    panel.shape
)

print(
    "Previous eligibility:",
    eligibility_previous.shape
)


# =============================================================================
# 2. EXPECTED PRIMARY PANEL
# =============================================================================

EXPECTED_KEYS = [

    "AC::pH_SOL::NO_UNIT",
    "AC::Na::mg/L",
    "AC::C.E._SOL::dS/m",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "AC::NH4::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
    "AC::SO4_ppm::mg/L",
    "FIELD::Water temperature::degC"
]


if (
    len(panel) != 10
    or panel["response_key"].nunique() != 10
    or set(panel["response_key"]) != set(EXPECTED_KEYS)
):

    raise RuntimeError(
        "Frozen 10-response panel does not match BLOCK 7.2C."
    )


if set(df["response_key"].unique()) != set(EXPECTED_KEYS):

    raise RuntimeError(
        "Memory dataset does not contain exactly the "
        "10 frozen primary responses."
    )


print("\nFrozen primary response identity: PASS")


# =============================================================================
# 3. VERIFY THAT HYDROCLIMATE VARIABLES WILL NOT BE USED
# =============================================================================

hydro_prefixes = (
    "ppt_sum_",
    "sm_surface_mean_",
    "sm_rootzone_mean_",
    "tair_mean_",
    "pet_sum_"
)

hydro_cols_present = [
    c
    for c in df.columns
    if c.startswith(hydro_prefixes)
]


print(
    "Hydroclimate columns physically present in input:",
    len(hydro_cols_present)
)

if len(hydro_cols_present) != 29:

    raise RuntimeError(
        "Expected the frozen 29-variable hydroclimate bank "
        "to be present in MEMORY_ANALYSIS_DATASET_v1."
    )


# Create response-only working table.
# Hydroclimate columns are explicitly removed from the working object.

response_only_cols = [
    c
    for c in df.columns
    if c not in hydro_cols_present
]

rdat = df[
    response_only_cols
].copy()


if any(
    c.startswith(hydro_prefixes)
    for c in rdat.columns
):

    raise RuntimeError(
        "Hydroclimate leakage into response-only diagnostic table."
    )


print(
    "Hydroclimate excluded from transformation decisions: PASS"
)


# =============================================================================
# 4. REQUIRED RESPONSE FIELDS
# =============================================================================

required_cols = [

    "response_key",
    "response_origin",

    "sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "campaign_id",

    "response_value_raw",
    "response_value_numeric",

    "response_censored",
    "response_censor_type",
    "response_censor_limit"
]


missing_cols = [
    c
    for c in required_cols
    if c not in rdat.columns
]


if missing_cols:

    raise RuntimeError(
        "Required response fields missing:\n" +
        "\n".join(missing_cols)
    )


# Robust boolean normalization.

def to_bool_series(x):

    if pd.api.types.is_bool_dtype(x):
        return x.fillna(False)

    return (
        x.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
            "nan": False,
            "none": False,
            "": False
        })
        .fillna(False)
        .astype(bool)
    )


rdat["response_censored"] = to_bool_series(
    rdat["response_censored"]
)


rdat["response_value_numeric"] = pd.to_numeric(
    rdat["response_value_numeric"],
    errors="coerce"
)


rdat["response_censor_limit"] = pd.to_numeric(
    rdat["response_censor_limit"],
    errors="coerce"
)


# =============================================================================
# 5. CENSORING ARCHITECTURE
# =============================================================================

censor_rows = []


for key in EXPECTED_KEYS:

    g = rdat.loc[
        rdat["response_key"] == key
    ].copy()

    n = len(g)

    cens = g[
        "response_censored"
    ]

    detected = ~cens

    limits = g.loc[
        cens,
        "response_censor_limit"
    ].dropna()

    censor_types = (
        g.loc[
            cens,
            "response_censor_type"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    censor_rows.append({

        "response_key":
            key,

        "n_rows":
            n,

        "n_detected":
            int(detected.sum()),

        "n_censored":
            int(cens.sum()),

        "censoring_pct":
            (
                100 * cens.sum() / n
                if n > 0
                else np.nan
            ),

        "n_censored_with_limit":
            int(
                g.loc[
                    cens,
                    "response_censor_limit"
                ]
                .notna()
                .sum()
            ),

        "n_unique_censor_limits":
            int(
                limits.nunique()
            ),

        "min_censor_limit":
            (
                float(limits.min())
                if len(limits)
                else np.nan
            ),

        "median_censor_limit":
            (
                float(limits.median())
                if len(limits)
                else np.nan
            ),

        "max_censor_limit":
            (
                float(limits.max())
                if len(limits)
                else np.nan
            ),

        "censor_types":
            (
                " | ".join(
                    sorted(censor_types)
                )
                if censor_types
                else ""
            ),

        "sample_specific_limits_present":
            bool(
                limits.nunique() > 1
            ),

        "lod_half_substitution_allowed":
            False,

        "detected_only_final_inference_allowed":
            False if cens.sum() > 0 else True,

        "censor_aware_model_required":
            bool(
                cens.sum() > 0
            )
    })


censor_arch = pd.DataFrame(
    censor_rows
)


print("\n" + "=" * 116)
print("CENSORING ARCHITECTURE")
print("=" * 116)

print(
    censor_arch.to_string(
        index=False
    )
)


# =============================================================================
# 6. HARD CENSORING CONSISTENCY QA
# =============================================================================
#
# For a censored record we require a censor limit for later censor-aware
# inference. We do NOT require value_numeric to contain a substituted value.
# =============================================================================

censored_rows = rdat.loc[
    rdat["response_censored"]
].copy()


n_censored_total = len(
    censored_rows
)


n_censored_missing_limit = int(
    censored_rows[
        "response_censor_limit"
    ]
    .isna()
    .sum()
)


print("\nTotal censored primary-response rows:",
      n_censored_total)

print(
    "Censored rows missing censor limit:",
    n_censored_missing_limit
)


if n_censored_missing_limit > 0:

    bad = censored_rows.loc[
        censored_rows[
            "response_censor_limit"
        ].isna(),
        [
            "response_key",
            "sample_id",
            "analytical_sample_id",
            "response_value_raw",
            "response_censor_type",
            "response_censor_limit"
        ]
    ]

    print("\nCensored rows without explicit limit:")
    print(
        bad.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — censored primary observations without "
        "sample-specific censor limit."
    )


# =============================================================================
# 7. RESPONSE DISTRIBUTION AUDIT
# =============================================================================
#
# Transformation diagnostics use detected observations only.
#
# This is legitimate for describing the observed distribution, but NOT for
# final inference on censored responses.
#
# No hydroclimate variable is accessed here.
# =============================================================================

def safe_quantile(x, q):

    if len(x) == 0:
        return np.nan

    return float(
        np.quantile(
            x,
            q
        )
    )


distribution_rows = []


for key in EXPECTED_KEYS:

    g = rdat.loc[
        rdat["response_key"] == key
    ].copy()

    detected_values = (
        g.loc[
            ~g["response_censored"],
            "response_value_numeric"
        ]
        .dropna()
        .astype(float)
    )


    n_detected_numeric = len(
        detected_values
    )


    if n_detected_numeric:

        minimum = float(
            detected_values.min()
        )

        maximum = float(
            detected_values.max()
        )

        mean_value = float(
            detected_values.mean()
        )

        median_value = float(
            detected_values.median()
        )

        sd_value = float(
            detected_values.std(
                ddof=1
            )
        ) if n_detected_numeric > 1 else np.nan

        q05 = safe_quantile(
            detected_values,
            0.05
        )

        q25 = safe_quantile(
            detected_values,
            0.25
        )

        q75 = safe_quantile(
            detected_values,
            0.75
        )

        q95 = safe_quantile(
            detected_values,
            0.95
        )

        n_zero = int(
            (detected_values == 0).sum()
        )

        n_negative = int(
            (detected_values < 0).sum()
        )

        all_positive = bool(
            (detected_values > 0).all()
        )

        skew_raw = (
            float(
                skew(
                    detected_values,
                    bias=False
                )
            )
            if n_detected_numeric >= 3
            else np.nan
        )

        kurt_raw = (
            float(
                kurtosis(
                    detected_values,
                    fisher=True,
                    bias=False
                )
            )
            if n_detected_numeric >= 4
            else np.nan
        )


        # ---------------------------------------------------------
        # log10 diagnostic only where mathematically valid.
        # ---------------------------------------------------------

        if all_positive:

            log_values = np.log10(
                detected_values
            )

            skew_log10 = (
                float(
                    skew(
                        log_values,
                        bias=False
                    )
                )
                if len(log_values) >= 3
                else np.nan
            )

            kurt_log10 = (
                float(
                    kurtosis(
                        log_values,
                        fisher=True,
                        bias=False
                    )
                )
                if len(log_values) >= 4
                else np.nan
            )

        else:

            skew_log10 = np.nan
            kurt_log10 = np.nan


        # ---------------------------------------------------------
        # log1p diagnostic only for non-negative values.
        # ---------------------------------------------------------

        if (
            detected_values >= 0
        ).all():

            log1p_values = np.log1p(
                detected_values
            )

            skew_log1p = (
                float(
                    skew(
                        log1p_values,
                        bias=False
                    )
                )
                if len(log1p_values) >= 3
                else np.nan
            )

        else:

            skew_log1p = np.nan


    else:

        minimum = np.nan
        maximum = np.nan
        mean_value = np.nan
        median_value = np.nan
        sd_value = np.nan

        q05 = np.nan
        q25 = np.nan
        q75 = np.nan
        q95 = np.nan

        n_zero = 0
        n_negative = 0
        all_positive = False

        skew_raw = np.nan
        kurt_raw = np.nan
        skew_log10 = np.nan
        kurt_log10 = np.nan
        skew_log1p = np.nan


    distribution_rows.append({

        "response_key":
            key,

        "n_total_rows":
            len(g),

        "n_detected_numeric":
            n_detected_numeric,

        "n_censored":
            int(
                g[
                    "response_censored"
                ].sum()
            ),

        "minimum_detected":
            minimum,

        "q05_detected":
            q05,

        "q25_detected":
            q25,

        "median_detected":
            median_value,

        "mean_detected":
            mean_value,

        "q75_detected":
            q75,

        "q95_detected":
            q95,

        "maximum_detected":
            maximum,

        "sd_detected":
            sd_value,

        "n_zero_detected":
            n_zero,

        "n_negative_detected":
            n_negative,

        "all_detected_positive":
            all_positive,

        "skew_raw_detected":
            skew_raw,

        "excess_kurtosis_raw_detected":
            kurt_raw,

        "skew_log10_detected":
            skew_log10,

        "excess_kurtosis_log10_detected":
            kurt_log10,

        "skew_log1p_detected":
            skew_log1p
    })


distribution = pd.DataFrame(
    distribution_rows
)


print("\n" + "=" * 116)
print("RESPONSE DISTRIBUTION AUDIT — DETECTED VALUES ONLY")
print("=" * 116)

print(
    distribution.to_string(
        index=False
    )
)


# =============================================================================
# 8. FREEZE TRANSFORMATION ARCHITECTURE
# =============================================================================
#
# IMPORTANT:
#
# We are NOT choosing transformations based on hydroclimate fit.
#
# Rules:
#
# pH:
#   preserve original scale because it is already logarithmic chemically.
#
# EC:
#   preserve original scale as integrated mineralization indicator unless
#   extreme response-only skew requires later sensitivity.
#
# Water temperature:
#   preserve °C scale.
#
# Positive concentration / microbial responses:
#   response-only skewness determines whether log10 is the primary candidate.
#
# Conservative rule:
#
#   |skew_raw| <= 1.0  -> ORIGINAL
#
#   |skew_raw| > 1.0 and all detected > 0 and log10 improves absolute skew
#       -> LOG10
#
#   non-negative with zero values and log1p improves skew
#       -> LOG1P
#
# Censored responses:
#   transformation refers to the latent response scale for later censor-aware
#   modeling; censored observations are NOT transformed into substituted
#   pseudo-values here.
# =============================================================================

FORCED_ORIGINAL = {
    "AC::pH_SOL::NO_UNIT":
        "pH is already a logarithmic chemical scale",

    "FIELD::Water temperature::degC":
        "temperature retained on physically interpretable Celsius scale"
}


architecture_rows = []


for _, row in distribution.iterrows():

    key = row[
        "response_key"
    ]

    raw_skew = row[
        "skew_raw_detected"
    ]

    log_skew = row[
        "skew_log10_detected"
    ]

    log1p_skew = row[
        "skew_log1p_detected"
    ]

    all_positive = bool(
        row[
            "all_detected_positive"
        ]
    )

    n_zero = int(
        row[
            "n_zero_detected"
        ]
    )

    n_censored = int(
        row[
            "n_censored"
        ]
    )


    # ---------------------------------------------------------
    # Forced physical/chemical scale
    # ---------------------------------------------------------

    if key in FORCED_ORIGINAL:

        transform = "ORIGINAL"

        transform_reason = (
            FORCED_ORIGINAL[key]
        )


    # ---------------------------------------------------------
    # EC: retain physical scale unless severely skewed.
    # ---------------------------------------------------------

    elif key == "AC::C.E._SOL::dS/m":

        if (
            pd.notna(raw_skew)
            and abs(raw_skew) <= 2.0
        ):

            transform = "ORIGINAL"

            transform_reason = (
                "integrated mineralization indicator; "
                "response-only skew not severe"
            )

        elif (
            all_positive
            and pd.notna(log_skew)
            and pd.notna(raw_skew)
            and abs(log_skew) < abs(raw_skew)
        ):

            transform = "LOG10"

            transform_reason = (
                "severe positive response-only skew; "
                "log10 improves symmetry"
            )

        else:

            transform = "ORIGINAL"

            transform_reason = (
                "retain interpretable EC scale; "
                "log transformation not clearly superior"
            )


    # ---------------------------------------------------------
    # Other responses
    # ---------------------------------------------------------

    else:

        if pd.isna(raw_skew):

            transform = "UNRESOLVED"

            transform_reason = (
                "insufficient detected numeric observations"
            )


        elif abs(raw_skew) <= 1.0:

            transform = "ORIGINAL"

            transform_reason = (
                "|response-only skewness| <= 1"
            )


        elif (
            all_positive
            and pd.notna(log_skew)
            and abs(log_skew) < abs(raw_skew)
        ):

            transform = "LOG10"

            transform_reason = (
                "positive response with |skew| > 1; "
                "log10 reduces absolute skewness"
            )


        elif (
            n_zero > 0
            and pd.notna(log1p_skew)
            and abs(log1p_skew) < abs(raw_skew)
        ):

            transform = "LOG1P"

            transform_reason = (
                "non-negative response containing zero; "
                "log1p reduces absolute skewness"
            )


        else:

            transform = "ORIGINAL"

            transform_reason = (
                "alternative transformation does not "
                "clearly improve response-only symmetry"
            )


    # ---------------------------------------------------------
    # Statistical pathway
    # ---------------------------------------------------------

    if n_censored > 0:

        pathway = (
            "CENSOR_AWARE_CONTINUOUS"
        )

        censor_policy = (
            "USE_OBSERVATION_SPECIFIC_CENSOR_LIMITS"
        )

        detected_only_role = (
            "DESCRIPTIVE_SENSITIVITY_ONLY"
        )

    else:

        pathway = (
            "DIRECT_CONTINUOUS"
        )

        censor_policy = (
            "NOT_APPLICABLE"
        )

        detected_only_role = (
            "PRIMARY_VALUES"
        )


    architecture_rows.append({

        "response_key":
            key,

        "primary_transformation":
            transform,

        "transformation_reason":
            transform_reason,

        "statistical_pathway":
            pathway,

        "censoring_policy":
            censor_policy,

        "detected_only_role":
            detected_only_role,

        "hydroclimate_used_for_transformation":
            False,

        "window_used_for_transformation":
            False,

        "automatic_response_exclusion":
            False
    })


architecture = pd.DataFrame(
    architecture_rows
)


# Attach response metadata.

panel_meta = panel[
    [
        c
        for c in [
            "response_key",
            "response_family",
            "scientific_role",
            "memory_panel_role",
            "freeze_status"
        ]
        if c in panel.columns
    ]
].copy()


architecture = architecture.merge(
    panel_meta,
    how="left",
    on="response_key",
    validate="one_to_one"
)


architecture = architecture.merge(
    distribution,
    how="left",
    on="response_key",
    validate="one_to_one"
)


architecture = architecture.merge(
    censor_arch[
        [
            "response_key",
            "n_detected",
            "n_censored",
            "censoring_pct",
            "n_unique_censor_limits",
            "sample_specific_limits_present"
        ]
    ],
    how="left",
    on="response_key",
    validate="one_to_one",
    suffixes=(
        "_distribution",
        "_censor"
    )
)


print("\n" + "=" * 116)
print("FROZEN RESPONSE STATISTICAL ARCHITECTURE")
print("=" * 116)

show_cols = [

    "response_key",
    "response_family",
    "primary_transformation",
    "statistical_pathway",
    "censoring_pct",
    "transformation_reason"
]

print(
    architecture[
        show_cols
    ].to_string(
        index=False
    )
)


# =============================================================================
# 9. TRANSFORMATION VALIDITY QA
# =============================================================================

invalid_log10 = architecture.loc[
    (
        architecture[
            "primary_transformation"
        ] == "LOG10"
    )
    &
    (
        architecture[
            "all_detected_positive"
        ] != True
    )
]


if len(invalid_log10):

    print(
        invalid_log10[
            [
                "response_key",
                "minimum_detected",
                "n_zero_detected",
                "n_negative_detected"
            ]
        ].to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — LOG10 assigned to response with "
        "non-positive detected values."
    )


if (
    architecture[
        "primary_transformation"
    ] == "UNRESOLVED"
).any():

    unresolved = architecture.loc[
        architecture[
            "primary_transformation"
        ] == "UNRESOLVED"
    ]

    print(
        unresolved[
            [
                "response_key",
                "n_detected_numeric"
            ]
        ].to_string(
            index=False
        )
    )

    raise RuntimeError(
        "STOP — response transformation architecture unresolved."
    )


# =============================================================================
# 10. MEMORY-INFERENCE RULES — PRE-SPECIFIED
# =============================================================================
#
# These rules are frozen BEFORE inspecting response × hydroclimate results.
# =============================================================================

rules = [

    {
        "rule_id": "M01",
        "category": "UNIT_OF_ANALYSIS",
        "rule":
            "Hydroclimate is event-date specific; repeated sites sharing a date "
            "do not constitute independent hydroclimatic states."
    },

    {
        "rule_id": "M02",
        "category": "TIMESCALE_BANK",
        "rule":
            "All 29 antecedent metrics remain pre-specified candidates; "
            "correlation among nested windows does not trigger deletion."
    },

    {
        "rule_id": "M03",
        "category": "WITHIN_FAMILY",
        "rule":
            "Windows are compared within hydroclimatic driver family; nested "
            "windows from the same family are not entered simultaneously "
            "as competing predictors in the primary memory model."
    },

    {
        "rule_id": "M04",
        "category": "COMMON_SAMPLE",
        "rule":
            "For a given response × hydroclimate family, all candidate windows "
            "must be compared using an identical response observation set."
    },

    {
        "rule_id": "M05",
        "category": "CENSORING",
        "rule":
            "Responses with censored observations use observation-specific "
            "censor limits in primary inference; no global LOD/2 substitution."
    },

    {
        "rule_id": "M06",
        "category": "CENSORING",
        "rule":
            "Detected-only associations for censored responses are descriptive "
            "or sensitivity diagnostics and cannot define the primary "
            "memory timescale."
    },

    {
        "rule_id": "M07",
        "category": "TRANSFORMATION",
        "rule":
            "Response transformation is frozen without reference to "
            "hydroclimate-window performance."
    },

    {
        "rule_id": "M08",
        "category": "MEMORY_DEFINITION",
        "rule":
            "The window with maximum absolute bivariate association is not, "
            "by itself, interpreted as the hydrological memory timescale."
    },

    {
        "rule_id": "M09",
        "category": "MEMORY_DEFINITION",
        "rule":
            "Memory is represented as an ordered response-strength/performance "
            "curve across antecedent windows within each driver family."
    },

    {
        "rule_id": "M10",
        "category": "MEMORY_DEFINITION",
        "rule":
            "When neighboring windows have comparable support, report a "
            "supported timescale range or plateau rather than false exact-day "
            "precision."
    },

    {
        "rule_id": "M11",
        "category": "VALIDATION",
        "rule":
            "Final timescale support must include campaign-aware validation; "
            "random row-wise train/test splitting is prohibited."
    },

    {
        "rule_id": "M12",
        "category": "VALIDATION",
        "rule":
            "Leave-one-campaign-out evaluation is the primary predictive "
            "validation architecture because only 10 campaigns are independent "
            "sampling campaigns."
    },

    {
        "rule_id": "M13",
        "category": "REPEATED_MEASURES",
        "rule":
            "Repeated observations across sites and analytical subsamples must "
            "be represented explicitly; analytical subsamples are not silently "
            "averaged."
    },

    {
        "rule_id": "M14",
        "category": "MULTIPLE_TESTING",
        "rule":
            "Nested-window p-values are not treated as 29 independent "
            "hypothesis tests; inference emphasizes effect trajectories, "
            "uncertainty and validation stability."
    },

    {
        "rule_id": "M15",
        "category": "DRIVER_FAMILIES",
        "rule":
            "Precipitation, surface soil moisture, root-zone soil moisture, "
            "air temperature and potential evaporation remain physically "
            "distinct driver families despite cross-family covariance."
    },

    {
        "rule_id": "M16",
        "category": "THERMAL_RESPONSE",
        "rule":
            "FIELD water temperature is retained as a thermal response but "
            "air-temperature coupling is interpreted separately from "
            "hydrochemical memory evidence."
    },

    {
        "rule_id": "M17",
        "category": "PCA",
        "rule":
            "Hydroclimate PCA from BLOCK 8.3 is diagnostic only and does not "
            "replace physically interpretable driver families or timescales."
    },

    {
        "rule_id": "M18",
        "category": "SPATIAL_CONNECTIVITY",
        "rule":
            "Catchment-connectivity effects are not inferred until explicit "
            "network/spatial covariates are available."
    },

    {
        "rule_id": "M19",
        "category": "INTERPRETATION",
        "rule":
            "Different response domains are not compared as though they had "
            "identical temporal coverage; coverage and censoring are reported "
            "with each memory spectrum."
    },

    {
        "rule_id": "M20",
        "category": "SELECTION",
        "rule":
            "No response or hydroclimate window may be added to the primary "
            "panel because it produces a stronger association after inspection."
    }
]


rules_df = pd.DataFrame(
    rules
)


print("\n" + "=" * 116)
print("FROZEN MEMORY-INFERENCE RULES")
print("=" * 116)

print(
    rules_df.to_string(
        index=False
    )
)


# =============================================================================
# 11. RESPONSE COVERAGE / REPLICATION ARCHITECTURE
# =============================================================================

coverage_rows = []


for key in EXPECTED_KEYS:

    g = rdat.loc[
        rdat[
            "response_key"
        ] == key
    ].copy()


    sample_rep = (
        g.groupby(
            "sample_id"
        )
        .size()
    )


    coverage_rows.append({

        "response_key":
            key,

        "n_rows":
            len(g),

        "n_unique_samples":
            g[
                "sample_id"
            ].nunique(),

        "n_unique_field_observations":
            g[
                "field_observation_id"
            ].nunique(
                dropna=True
            ),

        "n_analytical_subsamples":
            g[
                "analytical_sample_id"
            ].nunique(
                dropna=True
            ),

        "n_campaigns":
            g[
                "campaign_id"
            ].nunique(
                dropna=True
            ),

        "n_sample_ids_with_replication":
            int(
                (
                    sample_rep > 1
                ).sum()
            ),

        "maximum_rows_per_sample_id":
            int(
                sample_rep.max()
            )
            if len(sample_rep)
            else 0
    })


coverage = pd.DataFrame(
    coverage_rows
)


print("\n" + "=" * 116)
print("RESPONSE REPLICATION ARCHITECTURE")
print("=" * 116)

print(
    coverage.to_string(
        index=False
    )
)


# =============================================================================
# 12. FINAL GATE
# =============================================================================

n_censor_aware = int(
    (
        architecture[
            "statistical_pathway"
        ] ==
        "CENSOR_AWARE_CONTINUOUS"
    ).sum()
)


n_direct = int(
    (
        architecture[
            "statistical_pathway"
        ] ==
        "DIRECT_CONTINUOUS"
    ).sum()
)


n_log10 = int(
    (
        architecture[
            "primary_transformation"
        ] ==
        "LOG10"
    ).sum()
)


n_log1p = int(
    (
        architecture[
            "primary_transformation"
        ] ==
        "LOG1P"
    ).sum()
)


n_original = int(
    (
        architecture[
            "primary_transformation"
        ] ==
        "ORIGINAL"
    ).sum()
)


gate_checks = {

    "input_primary_responses_10":
        rdat[
            "response_key"
        ].nunique() == 10,

    "frozen_panel_exact":
        set(
            rdat[
                "response_key"
            ].unique()
        ) == set(EXPECTED_KEYS),

    "hydroclimate_not_used_for_transformation":
        (
            architecture[
                "hydroclimate_used_for_transformation"
            ] == False
        ).all(),

    "window_not_used_for_transformation":
        (
            architecture[
                "window_used_for_transformation"
            ] == False
        ).all(),

    "all_transformations_resolved":
        (
            architecture[
                "primary_transformation"
            ] != "UNRESOLVED"
        ).all(),

    "log10_only_positive_detected_values":
        len(
            invalid_log10
        ) == 0,

    "all_censored_rows_have_limits":
        n_censored_missing_limit == 0,

    "no_lod_half_substitution":
        (
            censor_arch[
                "lod_half_substitution_allowed"
            ] == False
        ).all(),

    "censored_responses_flagged_censor_aware":
        (
            architecture.loc[
                architecture[
                    "n_censored_censor"
                ] > 0,
                "statistical_pathway"
            ] ==
            "CENSOR_AWARE_CONTINUOUS"
        ).all(),

    "no_automatic_response_exclusion":
        (
            architecture[
                "automatic_response_exclusion"
            ] == False
        ).all(),

    "memory_rules_20":
        len(
            rules_df
        ) == 20,

    "no_memory_timescale_selected":
        True,

    "no_hydroclimate_response_models_fitted":
        True
}


gate_df = pd.DataFrame({

    "check":
        gate_checks.keys(),

    "pass":
        gate_checks.values()
})


BLOCK84B_PASS = bool(
    all(
        gate_checks.values()
    )
)


print("\n" + "=" * 116)
print("BLOCK 8.4B FINAL GATE")
print("=" * 116)

print(
    gate_df.to_string(
        index=False
    )
)

print(
    "\nBLOCK 8.4B PASS:",
    BLOCK84B_PASS
)


if not BLOCK84B_PASS:

    raise RuntimeError(
        "BLOCK 8.4B STOP — "
        "do not proceed to hydroclimatic memory-spectrum analysis."
    )


# =============================================================================
# 13. SAVE FROZEN OUTPUTS
# =============================================================================

architecture.to_csv(
    ARCHITECTURE_FILE,
    index=False
)


censor_arch.to_csv(
    CENSOR_FILE,
    index=False
)


distribution.to_csv(
    DISTRIBUTION_FILE,
    index=False
)


rules_df.to_csv(
    INFERENCE_RULES_FILE,
    index=False
)


# =============================================================================
# 14. AUDIT WORKBOOK
# =============================================================================

with pd.ExcelWriter(
    AUDIT_FILE,
    engine="openpyxl"
) as writer:

    architecture.to_excel(
        writer,
        sheet_name="response_architecture",
        index=False
    )

    distribution.to_excel(
        writer,
        sheet_name="distribution_audit",
        index=False
    )

    censor_arch.to_excel(
        writer,
        sheet_name="censoring_architecture",
        index=False
    )

    coverage.to_excel(
        writer,
        sheet_name="replication_coverage",
        index=False
    )

    rules_df.to_excel(
        writer,
        sheet_name="memory_rules",
        index=False
    )

    gate_df.to_excel(
        writer,
        sheet_name="final_gate",
        index=False
    )


# =============================================================================
# 15. HASH FUNCTION
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# =============================================================================
# 16. MANIFEST
# =============================================================================

manifest = {

    "block":
        "8.4B",

    "title":
        "Response transformation, censoring and memory-inference architecture freeze",

    "n_primary_responses":
        10,

    "n_direct_continuous":
        n_direct,

    "n_censor_aware":
        n_censor_aware,

    "n_original_scale":
        n_original,

    "n_log10":
        n_log10,

    "n_log1p":
        n_log1p,

    "n_censored_observations":
        int(
            n_censored_total
        ),

    "n_censored_missing_limit":
        int(
            n_censored_missing_limit
        ),

    "hydroclimate_columns_present_in_input":
        29,

    "hydroclimate_used_for_transformation":
        False,

    "response_window_associations_computed":
        False,

    "best_window_selected":
        False,

    "memory_timescale_estimated":
        False,

    "lod_half_substitution":
        False,

    "detected_only_primary_inference_for_censored_responses":
        False,

    "random_rowwise_validation_allowed":
        False,

    "campaign_aware_validation_frozen":
        True,

    "memory_rules":
        int(
            len(
                rules_df
            )
        ),

    "block_pass":
        BLOCK84B_PASS,

    "hashes": {

        "memory_analysis_dataset":
            sha256_file(
                INPUT_FILE
            ),

        "primary_panel":
            sha256_file(
                PANEL_FILE
            ),

        "statistical_architecture":
            sha256_file(
                ARCHITECTURE_FILE
            ),

        "censoring_architecture":
            sha256_file(
                CENSOR_FILE
            ),

        "distribution_audit":
            sha256_file(
                DISTRIBUTION_FILE
            ),

        "memory_inference_rules":
            sha256_file(
                INFERENCE_RULES_FILE
            )
    }
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# =============================================================================
# 17. FINAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 116)
print("BLOCK 8.4B — FINAL DIAGNOSTIC")
print("=" * 116)

print(
    "Frozen primary responses                 :",
    10
)

print(
    "Direct-continuous responses              :",
    n_direct
)

print(
    "Censor-aware responses                   :",
    n_censor_aware
)

print()
print(
    "ORIGINAL-scale responses                 :",
    n_original
)

print(
    "LOG10 responses                          :",
    n_log10
)

print(
    "LOG1P responses                          :",
    n_log1p
)

print()
print(
    "Total censored observations              :",
    n_censored_total
)

print(
    "Censored observations missing limit      :",
    n_censored_missing_limit
)

print()
print(
    "Hydroclimate metrics present             :",
    len(
        hydro_cols_present
    )
)

print(
    "Hydroclimate used for transformation     : NO"
)

print(
    "Response × window associations computed  : NO"
)

print(
    "Best hydroclimate window selected        : NO"
)

print(
    "Memory timescale estimated               : NO"
)

print()
print(
    "LOD/2 substitution                       : NO"
)

print(
    "Detected-only primary inference "
    "for censored responses                  : NO"
)

print(
    "Random row-wise validation               : PROHIBITED"
)

print(
    "Campaign-aware validation                : FROZEN"
)

print(
    "Memory-inference rules                   :",
    len(
        rules_df
    )
)

print()
print(
    "Statistical architecture:"
)
print(
    ARCHITECTURE_FILE
)

print()
print(
    "Censoring architecture:"
)
print(
    CENSOR_FILE
)

print()
print(
    "Distribution audit:"
)
print(
    DISTRIBUTION_FILE
)

print()
print(
    "Memory-inference rules:"
)
print(
    INFERENCE_RULES_FILE
)

print()
print(
    "Audit workbook:"
)
print(
    AUDIT_FILE
)

print()
print(
    "BLOCK 8.4B PASS                          :",
    BLOCK84B_PASS
)

print()
print(
    "BLOCK 8.4B COMPLETE — "
    "RESPONSE-SIDE MEMORY ARCHITECTURE FROZEN."
)

---
### Imported master block 74

`BLOCK 8.5B-D0 — INFERENTIAL ENGINE AVAILABILITY DIAGNOSTIC`


In [ ]:
# =============================================================================
# BLOCK 8.5B-D0 — INFERENTIAL ENGINE AVAILABILITY DIAGNOSTIC
# =============================================================================

import sys
import importlib.util
import statsmodels
import scipy
import sklearn
import pandas as pd
import numpy as np

print("=" * 100)
print("BLOCK 8.5B-D0 — INFERENTIAL ENGINE AVAILABILITY DIAGNOSTIC")
print("=" * 100)

print("\nPython       :", sys.version.split()[0])
print("pandas       :", pd.__version__)
print("numpy        :", np.__version__)
print("scipy        :", scipy.__version__)
print("statsmodels  :", statsmodels.__version__)
print("scikit-learn :", sklearn.__version__)

packages = [
    "lifelines",
    "pymc",
    "bambi",
    "arviz"
]

print("\nOPTIONAL PACKAGES")
print("-" * 100)

for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    print(f"{pkg:12s}: {'AVAILABLE' if spec is not None else 'NOT AVAILABLE'}")

print("\nSTATSMODELS CAPABILITIES")
print("-" * 100)

# MixedLM
try:
    from statsmodels.regression.mixed_linear_model import MixedLM
    print("MixedLM                         : AVAILABLE")
except Exception as e:
    print("MixedLM                         : NOT AVAILABLE", repr(e))

# GEE
try:
    from statsmodels.genmod.generalized_estimating_equations import GEE
    print("GEE                             : AVAILABLE")
except Exception as e:
    print("GEE                             : NOT AVAILABLE", repr(e))

# Censored/Tobit-related modules
candidate_modules = [
    "statsmodels.duration",
    "statsmodels.duration.hazard_regression",
    "statsmodels.miscmodels",
    "statsmodels.othermod"
]

for mod in candidate_modules:
    try:
        m = __import__(mod, fromlist=["*"])
        names = [
            x for x in dir(m)
            if any(k in x.lower() for k in [
                "censor", "tobit", "survival", "phreg"
            ])
        ]
        print(f"{mod:40s}: {names if names else 'no obvious censored-regression class'}")
    except Exception as e:
        print(f"{mod:40s}: ERROR {repr(e)}")

print("\nDATA STRUCTURE READ-BACK")
print("-" * 100)

f = "/content/Huanta_JoH/03_MASTER/MEMORY_ANALYSIS_DATASET_v1.csv"
df = pd.read_csv(f, low_memory=False)

cols = [
    "response_key",
    "sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",
    "hydroclimate_event_date",
    "response_censored",
    "response_censor_limit",
    "response_value_numeric"
]

for c in cols:
    print(f"{c:32s}: {'YES' if c in df.columns else 'NO'}")

print("\nREPLICATION / CLUSTER COUNTS")
print("-" * 100)

for c in [
    "station_id",
    "physical_station_id_v1",
    "campaign_id",
    "hydroclimate_event_date",
    "field_observation_id",
    "analytical_sample_id"
]:
    if c in df.columns:
        print(f"{c:32s}: {df[c].nunique(dropna=True)} unique")

print("\n" + "=" * 100)
print("BLOCK 8.5B-D0 COMPLETE — DIAGNOSTIC ONLY")
print("=" * 100)

print("""
No model fitted.
No p-value calculated.
No hydroclimate window selected.
No memory timescale selected.
No package installed.
""")

---
### Imported master block 75

`BLOCK 8.5B — BAYESIAN MEMORY-INFERENCE ENGINE VALIDATION`


In [ ]:
# =============================================================================
# BLOCK 8.5B — BAYESIAN MEMORY-INFERENCE ENGINE VALIDATION
# =============================================================================
#
# PURPOSE
# -------
# Validate the inferential engine BEFORE scaling to all response × timescale
# combinations.
#
# TWO COMPUTATIONAL TEST CASES:
#
# 1) FIELD water temperature × Tair 7 d
#       -> DIRECT_CONTINUOUS pathway
#
# 2) Al × precipitation 14 d
#       -> CENSOR_AWARE_CONTINUOUS pathway
#
# IMPORTANT
# ---------
# These windows are NOT selected as memory timescales.
# They are computational validation cases only.
#
# Model:
#
#   y_i ~ Normal(mu_i, sigma)
#
#   mu_i = alpha
#        + beta_hydro * z(hydroclimate_i)
#        + station_random_intercept
#
# For censored responses:
#
#   observed y is represented using pm.Censored()
#   with observation-specific upper censoring boundary after transformation.
#
# Analytical subsamples remain separate rows.
#
# No campaign effect is included here because this block validates the
# engine only. Campaign-aware predictive validation is introduced later
# through leave-one-campaign-out refitting.
#
# =============================================================================

from pathlib import Path
import warnings
import json
import hashlib

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("/content/Huanta_JoH")
MASTER = ROOT / "03_MASTER"
AUDIT = ROOT / "01_AUDIT"

DATA_FILE = MASTER / "MEMORY_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = MASTER / "MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"

OUT_SUMMARY = MASTER / "MEMORY_INFERENCE_ENGINE_VALIDATION_v1.csv"
OUT_AUDIT = AUDIT / "BLOCK85B_inference_engine_validation.xlsx"
OUT_MANIFEST = AUDIT / "BLOCK85B_inference_engine_validation_manifest.json"

print("=" * 118)
print("BLOCK 8.5B — BAYESIAN MEMORY-INFERENCE ENGINE VALIDATION")
print("=" * 118)

# =============================================================================
# 1. LOAD
# =============================================================================

if not DATA_FILE.exists():
    raise FileNotFoundError(DATA_FILE)

if not ARCH_FILE.exists():
    raise FileNotFoundError(ARCH_FILE)

df = pd.read_csv(DATA_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE, low_memory=False)

print("\nINPUT")
print("-" * 118)
print("Memory dataset:", df.shape)
print("Architecture:", arch.shape)
print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)

# =============================================================================
# 2. HELPERS
# =============================================================================

def to_bool(x):
    if pd.api.types.is_bool_dtype(x):
        return x.fillna(False)

    return (
        x.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
        .fillna(False)
        .astype(bool)
    )


df["response_censored"] = to_bool(df["response_censored"])

df["response_value_numeric"] = pd.to_numeric(
    df["response_value_numeric"],
    errors="coerce"
)

df["response_censor_limit"] = pd.to_numeric(
    df["response_censor_limit"],
    errors="coerce"
)

arch_lookup = (
    arch
    .drop_duplicates("response_key")
    .set_index("response_key")
)


def transform_values(x, transformation):
    x = np.asarray(x, dtype=float)

    if transformation == "ORIGINAL":
        return x

    elif transformation == "LOG10":
        if np.any(x <= 0):
            raise RuntimeError(
                "LOG10 transformation received non-positive value."
            )
        return np.log10(x)

    elif transformation == "LOG1P":
        if np.any(x < 0):
            raise RuntimeError(
                "LOG1P transformation received negative value."
            )
        return np.log1p(x)

    else:
        raise RuntimeError(
            f"Unknown transformation: {transformation}"
        )


# =============================================================================
# 3. COMPUTATIONAL VALIDATION CASES
# =============================================================================

cases = [
    {
        "case_id": "DIRECT_ENGINE_TEST",
        "response_key": "FIELD::Water temperature::degC",
        "hydro_variable": "tair_mean_7d",
        "hydro_family": "air_temperature",
        "window_days": 7,
        "expected_pathway": "DIRECT_CONTINUOUS"
    },

    {
        "case_id": "CENSORED_ENGINE_TEST",
        "response_key": "METALS::Aluminio::mg/L",
        "hydro_variable": "ppt_sum_14d",
        "hydro_family": "precipitation",
        "window_days": 14,
        "expected_pathway": "CENSOR_AWARE_CONTINUOUS"
    }
]

print("\n" + "=" * 118)
print("COMPUTATIONAL VALIDATION CASES — NOT MEMORY SELECTION")
print("=" * 118)

for case in cases:
    print(case)

# =============================================================================
# 4. MODEL FITTER
# =============================================================================

def fit_validation_model(case):

    response_key = case["response_key"]
    hydro_var = case["hydro_variable"]

    if hydro_var not in df.columns:
        raise RuntimeError(
            f"Hydroclimate variable missing: {hydro_var}"
        )

    if response_key not in arch_lookup.index:
        raise RuntimeError(
            f"Response missing from frozen architecture: {response_key}"
        )

    transformation = arch_lookup.loc[
        response_key,
        "primary_transformation"
    ]

    pathway = arch_lookup.loc[
        response_key,
        "statistical_pathway"
    ]

    if pathway != case["expected_pathway"]:
        raise RuntimeError(
            f"Pathway mismatch for {response_key}: "
            f"{pathway} != {case['expected_pathway']}"
        )

    g = df.loc[
        df["response_key"] == response_key
    ].copy()

    # -------------------------------------------------------------
    # Basic structural checks
    # -------------------------------------------------------------

    required = [
        hydro_var,
        "response_value_numeric",
        "response_censored",
        "response_censor_limit",
        "physical_station_id_v1",
        "campaign_id",
        "hydroclimate_event_date"
    ]

    missing = [c for c in required if c not in g.columns]

    if missing:
        raise RuntimeError(
            f"Missing required columns: {missing}"
        )

    g[hydro_var] = pd.to_numeric(
        g[hydro_var],
        errors="coerce"
    )

    if g[hydro_var].isna().any():
        raise RuntimeError(
            f"Hydroclimate nulls found in {hydro_var}"
        )

    # -------------------------------------------------------------
    # Response preparation
    # -------------------------------------------------------------

    detected = ~g["response_censored"]

    if pathway == "DIRECT_CONTINUOUS":

        if g["response_value_numeric"].isna().any():
            raise RuntimeError(
                f"Unexpected missing response values in {response_key}"
            )

        y_original = g["response_value_numeric"].to_numpy(float)

        y_analysis = transform_values(
            y_original,
            transformation
        )

        censor_limit_analysis = np.full(
            len(g),
            np.nan
        )

        observed_for_model = y_analysis.copy()

    else:

        # Detected observations need measured values.
        if g.loc[
            detected,
            "response_value_numeric"
        ].isna().any():
            raise RuntimeError(
                "Detected censored-pathway observations contain missing values."
            )

        # Every censored observation needs its own limit.
        if g.loc[
            ~detected,
            "response_censor_limit"
        ].isna().any():
            raise RuntimeError(
                "Censored observations missing censor limits."
            )

        # Transform detected measurements.
        y_analysis = np.full(
            len(g),
            np.nan,
            dtype=float
        )

        y_analysis[detected.to_numpy()] = transform_values(
            g.loc[
                detected,
                "response_value_numeric"
            ].to_numpy(float),
            transformation
        )

        # Transform censor limits.
        censor_limit_analysis = np.full(
            len(g),
            np.nan,
            dtype=float
        )

        censor_limit_analysis[
            (~detected).to_numpy()
        ] = transform_values(
            g.loc[
                ~detected,
                "response_censor_limit"
            ].to_numpy(float),
            transformation
        )

        # ---------------------------------------------------------
        # pm.Censored convention:
        #
        # Left-censored observation y <= L:
        #   upper = L
        #
        # For the observed array, use the censoring boundary for
        # censored rows and actual measurement for detected rows.
        # ---------------------------------------------------------

        observed_for_model = y_analysis.copy()

        observed_for_model[
            (~detected).to_numpy()
        ] = censor_limit_analysis[
            (~detected).to_numpy()
        ]

    # -------------------------------------------------------------
    # Standardize response for numerically stable common priors.
    #
    # IMPORTANT:
    # Scaling parameters are estimated ONLY from detected response
    # values. Censored observations are not substituted.
    # -------------------------------------------------------------

    detected_analysis_values = y_analysis[
        detected.to_numpy()
    ]

    y_center = float(
        np.mean(detected_analysis_values)
    )

    y_scale = float(
        np.std(
            detected_analysis_values,
            ddof=1
        )
    )

    if not np.isfinite(y_scale) or y_scale <= 0:
        raise RuntimeError(
            f"Invalid response scale for {response_key}"
        )

    observed_z = (
        observed_for_model - y_center
    ) / y_scale

    if pathway == "CENSOR_AWARE_CONTINUOUS":

        upper_z = np.full(
            len(g),
            np.inf,
            dtype=float
        )

        upper_z[
            (~detected).to_numpy()
        ] = (
            censor_limit_analysis[
                (~detected).to_numpy()
            ] - y_center
        ) / y_scale

    else:

        upper_z = np.full(
            len(g),
            np.inf,
            dtype=float
        )

    # -------------------------------------------------------------
    # Standardize hydroclimate predictor.
    # -------------------------------------------------------------

    x = g[hydro_var].to_numpy(float)

    x_mean = float(np.mean(x))
    x_sd = float(np.std(x, ddof=1))

    if not np.isfinite(x_sd) or x_sd <= 0:
        raise RuntimeError(
            f"Invalid predictor SD for {hydro_var}"
        )

    x_z = (x - x_mean) / x_sd

    # -------------------------------------------------------------
    # Station random intercept.
    # -------------------------------------------------------------

    station = (
        g["physical_station_id_v1"]
        .astype(str)
        .fillna("UNRESOLVED")
    )

    station_codes, station_levels = pd.factorize(
        station,
        sort=True
    )

    n_station = len(station_levels)

    # -------------------------------------------------------------
    # Model
    # -------------------------------------------------------------

    coords = {
        "obs_id": np.arange(len(g)),
        "station": station_levels.astype(str)
    }

    with pm.Model(coords=coords) as model:

        x_data = pm.Data(
            "x_z",
            x_z,
            dims="obs_id"
        )

        station_idx = pm.Data(
            "station_idx",
            station_codes,
            dims="obs_id"
        )

        # Weakly regularizing priors on standardized response scale.
        alpha = pm.Normal(
            "alpha",
            mu=0.0,
            sigma=1.0
        )

        beta_hydro = pm.Normal(
            "beta_hydro",
            mu=0.0,
            sigma=1.0
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1.0
        )

        station_offset = pm.Normal(
            "station_offset",
            mu=0.0,
            sigma=1.0,
            dims="station"
        )

        station_effect = pm.Deterministic(
            "station_effect",
            station_offset * sigma_station,
            dims="station"
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1.0
        )

        mu = (
            alpha
            + beta_hydro * x_data
            + station_effect[station_idx]
        )

        base_dist = pm.Normal.dist(
            mu=mu,
            sigma=sigma
        )

        if pathway == "DIRECT_CONTINUOUS":

            y_obs = pm.Normal(
                "y_obs",
                mu=mu,
                sigma=sigma,
                observed=observed_z,
                dims="obs_id"
            )

        else:

            y_obs = pm.Censored(
                "y_obs",
                base_dist,
                lower=None,
                upper=upper_z,
                observed=observed_z,
                dims="obs_id"
            )

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=20260914,
            return_inferencedata=True,
            progressbar=True
        )

    # -------------------------------------------------------------
    # Posterior diagnostic
    # -------------------------------------------------------------

    posterior_summary = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_hydro",
            "sigma",
            "sigma_station"
        ],
        hdi_prob=0.95,
        round_to=6
    )

    beta_samples = (
        idata.posterior["beta_hydro"]
        .values
        .reshape(-1)
    )

    beta_mean = float(
        np.mean(beta_samples)
    )

    beta_median = float(
        np.median(beta_samples)
    )

    hdi = az.hdi(
        beta_samples,
        hdi_prob=0.95
    )

    beta_hdi_low = float(hdi[0])
    beta_hdi_high = float(hdi[1])

    prob_positive = float(
        np.mean(beta_samples > 0)
    )

    prob_negative = float(
        np.mean(beta_samples < 0)
    )

    beta_row = posterior_summary.loc[
        "beta_hydro"
    ]

    rhat = float(
        beta_row["r_hat"]
    )

    ess_bulk = float(
        beta_row["ess_bulk"]
    )

    ess_tail = float(
        beta_row["ess_tail"]
    )

    divergences = int(
        idata.sample_stats["diverging"]
        .sum()
        .values
    )

    max_rhat = float(
        posterior_summary["r_hat"].max()
    )

    min_ess_bulk = float(
        posterior_summary["ess_bulk"].min()
    )

    print("\n" + "=" * 118)
    print(case["case_id"])
    print("=" * 118)

    print("Response             :", response_key)
    print("Hydroclimate         :", hydro_var)
    print("Pathway              :", pathway)
    print("Transformation       :", transformation)
    print("Rows                 :", len(g))
    print("Detected             :", int(detected.sum()))
    print("Censored             :", int((~detected).sum()))
    print("Stations             :", n_station)
    print("Campaigns            :", g["campaign_id"].nunique())
    print(
        "Hydroclimate dates   :",
        g["hydroclimate_event_date"].nunique()
    )

    if pathway == "CENSOR_AWARE_CONTINUOUS":
        print(
            "Unique censor limits :",
            g.loc[
                ~detected,
                "response_censor_limit"
            ].nunique()
        )

    print("\nPOSTERIOR")
    print(posterior_summary.to_string())

    print("\nHydroclimate beta")
    print("Mean                 :", beta_mean)
    print("Median               :", beta_median)
    print(
        "95% HDI              :",
        beta_hdi_low,
        "to",
        beta_hdi_high
    )
    print("P(beta > 0)          :", prob_positive)
    print("P(beta < 0)          :", prob_negative)

    print("\nSAMPLING DIAGNOSTICS")
    print("Divergences          :", divergences)
    print("beta R-hat           :", rhat)
    print("Max R-hat            :", max_rhat)
    print("beta ESS bulk        :", ess_bulk)
    print("beta ESS tail        :", ess_tail)
    print("Min ESS bulk         :", min_ess_bulk)

    result = {
        "case_id": case["case_id"],
        "response_key": response_key,
        "hydroclimate_family": case["hydro_family"],
        "hydroclimate_variable": hydro_var,
        "window_days": case["window_days"],

        "computational_validation_only": True,
        "memory_timescale_selected": False,

        "statistical_pathway": pathway,
        "transformation": transformation,

        "n_rows": len(g),
        "n_detected": int(detected.sum()),
        "n_censored": int((~detected).sum()),
        "n_stations": n_station,
        "n_campaigns": int(
            g["campaign_id"].nunique()
        ),
        "n_hydroclimate_dates": int(
            g["hydroclimate_event_date"].nunique()
        ),

        "beta_mean": beta_mean,
        "beta_median": beta_median,
        "beta_hdi_2_5": beta_hdi_low,
        "beta_hdi_97_5": beta_hdi_high,
        "prob_beta_positive": prob_positive,
        "prob_beta_negative": prob_negative,

        "beta_rhat": rhat,
        "max_rhat": max_rhat,
        "beta_ess_bulk": ess_bulk,
        "beta_ess_tail": ess_tail,
        "min_ess_bulk": min_ess_bulk,
        "divergences": divergences,

        "engine_convergence_pass": bool(
            divergences == 0
            and max_rhat <= 1.01
            and min_ess_bulk >= 400
        )
    }

    return result, posterior_summary


# =============================================================================
# 5. RUN TWO VALIDATION CASES
# =============================================================================

results = []
posterior_tables = {}

for case in cases:

    result, posterior_summary = fit_validation_model(case)

    results.append(result)

    posterior_tables[
        case["case_id"]
    ] = posterior_summary.reset_index().rename(
        columns={"index": "parameter"}
    )

results_df = pd.DataFrame(results)

# =============================================================================
# 6. FINAL ENGINE GATE
# =============================================================================

direct = results_df.loc[
    results_df["case_id"] ==
    "DIRECT_ENGINE_TEST"
]

censored = results_df.loc[
    results_df["case_id"] ==
    "CENSORED_ENGINE_TEST"
]

gate = {
    "validation_cases_2":
        len(results_df) == 2,

    "direct_engine_test_present":
        len(direct) == 1,

    "censored_engine_test_present":
        len(censored) == 1,

    "direct_engine_converged":
        bool(
            direct[
                "engine_convergence_pass"
            ].iloc[0]
        ),

    "censored_engine_converged":
        bool(
            censored[
                "engine_convergence_pass"
            ].iloc[0]
        ),

    "censored_model_retains_censored_rows":
        int(
            censored["n_censored"].iloc[0]
        ) > 0,

    "station_random_intercept_used":
        True,

    "analytical_subsamples_not_collapsed":
        True,

    "campaigns_not_randomly_split":
        True,

    "no_lod_half_substitution":
        True,

    "no_censor_substitution":
        True,

    "no_memory_timescale_selected":
        (
            results_df[
                "memory_timescale_selected"
            ] == False
        ).all(),

    "validation_cases_not_called_memory":
        (
            results_df[
                "computational_validation_only"
            ] == True
        ).all()
}

gate_df = pd.DataFrame({
    "check": list(gate.keys()),
    "pass": list(gate.values())
})

BLOCK85B_PASS = bool(
    all(gate.values())
)

print("\n" + "=" * 118)
print("BLOCK 8.5B FINAL GATE")
print("=" * 118)

print(
    gate_df.to_string(
        index=False
    )
)

print(
    "\nBLOCK 8.5B PASS:",
    BLOCK85B_PASS
)

if not BLOCK85B_PASS:
    raise RuntimeError(
        "BLOCK 8.5B STOP — inferential engine validation failed."
    )

# =============================================================================
# 7. SAVE
# =============================================================================

results_df.to_csv(
    OUT_SUMMARY,
    index=False
)

with pd.ExcelWriter(
    OUT_AUDIT,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="engine_validation",
        index=False
    )

    gate_df.to_excel(
        writer,
        sheet_name="final_gate",
        index=False
    )

    for name, table in posterior_tables.items():

        sheet = name[:31]

        table.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

# =============================================================================
# 8. MANIFEST
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "block": "8.5B",
    "title":
        "Bayesian memory-inference engine validation",

    "engine":
        "PyMC",

    "model_family_direct":
        "Gaussian hierarchical regression",

    "model_family_censored":
        "Gaussian hierarchical left-censored regression",

    "random_effect":
        "physical_station_id_v1",

    "hydroclimate_predictor":
        "one standardized antecedent metric per model",

    "validation_cases":
        2,

    "computational_validation_only":
        True,

    "memory_timescale_selected":
        False,

    "campaign_aware_validation_performed":
        False,

    "campaign_aware_validation_reserved_for":
        "subsequent block",

    "block_pass":
        BLOCK85B_PASS,

    "hashes": {
        "input_dataset":
            sha256_file(DATA_FILE),

        "response_architecture":
            sha256_file(ARCH_FILE),

        "output_summary":
            sha256_file(OUT_SUMMARY)
    }
}

with open(
    OUT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# =============================================================================
# 9. FINAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 118)
print("BLOCK 8.5B — FINAL DIAGNOSTIC")
print("=" * 118)

print(
    results_df[
        [
            "case_id",
            "response_key",
            "hydroclimate_variable",
            "statistical_pathway",
            "n_rows",
            "n_censored",
            "beta_mean",
            "beta_hdi_2_5",
            "beta_hdi_97_5",
            "prob_beta_positive",
            "prob_beta_negative",
            "max_rhat",
            "min_ess_bulk",
            "divergences",
            "engine_convergence_pass"
        ]
    ].to_string(index=False)
)

print()
print("Censor substitution                  : NO")
print("LOD/2                                : NO")
print("Station random intercept             : YES")
print("Analytical subsamples collapsed      : NO")
print("Campaign-aware CV                    : NOT YET")
print("Memory timescale selected            : NO")
print("Validation windows called memory     : NO")

print()
print("Output:")
print(OUT_SUMMARY)

print()
print("Audit:")
print(OUT_AUDIT)

print()
print("BLOCK 8.5B PASS                      :", BLOCK85B_PASS)

print()
print(
    "BLOCK 8.5B COMPLETE — "
    "INFERENTIAL ENGINE VALIDATED; "
    "NO MEMORY TIMESCALE SELECTED."
)

---
### Imported master block 76

`BLOCK 8.5C — FULL BAYESIAN MEMORY EFFECT-SPECTRUM ESTIMATION`


In [ ]:
# ================================================================================================================
# BLOCK 8.5C — FULL BAYESIAN MEMORY EFFECT-SPECTRUM ESTIMATION
# ================================================================================================================
#
# PURPOSE
# -------
# Estimate the complete Bayesian hydroclimatic effect spectrum for:
#
#     10 frozen primary responses × 29 pre-specified antecedent metrics = 290 models
#
# IMPORTANT
# ---------
# This block DOES NOT:
#   - select a best window
#   - estimate a final memory timescale
#   - perform LOCO validation
#   - rank responses
#   - use LOD/2 substitution
#   - replace censored observations
#   - collapse analytical subsamples
#
# Each model contains:
#
#   response ~ beta_hydro * standardized_hydroclimate
#              + station random intercept
#
# For censored responses:
#   observation-specific censor limits are retained in the likelihood.
#
# Response transformations were frozen in BLOCK 8.4B.
#
# ================================================================================================================

import os
import json
import hashlib
import warnings
import traceback
import time
import gc

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

from scipy.stats import rankdata

warnings.filterwarnings("ignore")

print("=" * 112)
print("BLOCK 8.5C — FULL BAYESIAN MEMORY EFFECT-SPECTRUM ESTIMATION")
print("=" * 112)

# ----------------------------------------------------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------------------------------------------------

BASE = "/content/Huanta_JoH"

MEMORY_FILE = f"{BASE}/03_MASTER/MEMORY_ANALYSIS_DATASET_v1.csv"
ARCH_FILE   = f"{BASE}/03_MASTER/MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"

OUT_SPECTRUM = f"{BASE}/03_MASTER/MEMORY_BAYESIAN_EFFECT_SPECTRUM_v1.csv"
OUT_SUMMARY  = f"{BASE}/03_MASTER/MEMORY_BAYESIAN_EFFECT_SPECTRUM_SUMMARY_v1.csv"
OUT_STATION  = f"{BASE}/03_MASTER/MEMORY_STATION_IDENTITY_AUDIT_v1.csv"

AUDIT_XLSX = f"{BASE}/01_AUDIT/BLOCK85C_bayesian_memory_effect_spectrum.xlsx"
MANIFEST   = f"{BASE}/01_AUDIT/BLOCK85C_bayesian_memory_effect_spectrum_manifest.json"

CHECKPOINT = f"{BASE}/04_OUTPUT/BLOCK85C_model_checkpoint.csv"

os.makedirs(f"{BASE}/03_MASTER", exist_ok=True)
os.makedirs(f"{BASE}/01_AUDIT", exist_ok=True)
os.makedirs(f"{BASE}/04_OUTPUT", exist_ok=True)

for f in [MEMORY_FILE, ARCH_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Required file missing: {f}")

# ----------------------------------------------------------------------------------------------------------------
# LOAD
# ----------------------------------------------------------------------------------------------------------------

df = pd.read_csv(MEMORY_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE, low_memory=False)

print("\nINPUT")
print("-" * 112)
print("Memory dataset:", df.shape)
print("Architecture :", arch.shape)
print("PyMC         :", pm.__version__)
print("ArviZ        :", az.__version__)

# ----------------------------------------------------------------------------------------------------------------
# REQUIRED RESPONSE KEYS
# ----------------------------------------------------------------------------------------------------------------

PRIMARY_KEYS = [
    "AC::pH_SOL::NO_UNIT",
    "AC::Na::mg/L",
    "AC::C.E._SOL::dS/m",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "AC::NH4::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
    "AC::SO4_ppm::mg/L",
    "FIELD::Water temperature::degC",
]

if set(df["response_key"].dropna().unique()) != set(PRIMARY_KEYS):
    raise RuntimeError("Primary response identity mismatch.")

# ----------------------------------------------------------------------------------------------------------------
# HYDROCLIMATE ARCHITECTURE — EXACTLY 29 PRE-SPECIFIED METRICS
# ----------------------------------------------------------------------------------------------------------------

HYDRO_FAMILIES = {
    "precipitation": {
        1: "ppt_sum_1d",
        3: "ppt_sum_3d",
        7: "ppt_sum_7d",
        14: "ppt_sum_14d",
        30: "ppt_sum_30d",
        60: "ppt_sum_60d",
        90: "ppt_sum_90d",
    },

    "surface_soil_moisture": {
        1: "sm_surface_mean_1d",
        7: "sm_surface_mean_7d",
        14: "sm_surface_mean_14d",
        30: "sm_surface_mean_30d",
        60: "sm_surface_mean_60d",
        90: "sm_surface_mean_90d",
    },

    "rootzone_soil_moisture": {
        1: "sm_rootzone_mean_1d",
        7: "sm_rootzone_mean_7d",
        14: "sm_rootzone_mean_14d",
        30: "sm_rootzone_mean_30d",
        60: "sm_rootzone_mean_60d",
        90: "sm_rootzone_mean_90d",
    },

    "air_temperature": {
        7: "tair_mean_7d",
        14: "tair_mean_14d",
        30: "tair_mean_30d",
        60: "tair_mean_60d",
        90: "tair_mean_90d",
    },

    "potential_evaporation": {
        7: "pet_sum_7d",
        14: "pet_sum_14d",
        30: "pet_sum_30d",
        60: "pet_sum_60d",
        90: "pet_sum_90d",
    },
}

HYDRO_ROWS = []

for fam, mapping in HYDRO_FAMILIES.items():
    for w, var in mapping.items():
        HYDRO_ROWS.append({
            "hydroclimate_family": fam,
            "window_days": int(w),
            "hydroclimate_variable": var,
        })

hydro_arch = pd.DataFrame(HYDRO_ROWS)

if len(hydro_arch) != 29:
    raise RuntimeError("Hydroclimate architecture is not 29 metrics.")

missing_hydro = [
    x for x in hydro_arch["hydroclimate_variable"]
    if x not in df.columns
]

if missing_hydro:
    raise RuntimeError(f"Missing hydroclimate variables: {missing_hydro}")

print("\nHydroclimate metrics:", len(hydro_arch), "— PASS")

# ================================================================================================================
# PART 1 — STATION IDENTITY AUDIT / FIX
# ================================================================================================================

print("\n" + "=" * 112)
print("STATION IDENTITY AUDIT")
print("=" * 112)

# IMPORTANT:
# Do NOT use astype(str) before resolving missing values.
# Otherwise np.nan becomes literal "nan".

def clean_identifier(series):
    s = series.copy()

    # Preserve actual NA first
    s = s.where(s.notna(), np.nan)

    # Convert only non-null values
    s = s.map(lambda x: str(x).strip() if pd.notna(x) else np.nan)

    # Strings that semantically mean missing
    bad = {"", "nan", "none", "null", "na", "<na>"}

    s = s.map(
        lambda x: np.nan
        if pd.isna(x) or str(x).strip().lower() in bad
        else str(x).strip()
    )

    return s


if "physical_station_id_v1" not in df.columns:
    raise RuntimeError("physical_station_id_v1 missing.")

if "station_id" not in df.columns:
    raise RuntimeError("station_id missing.")

physical_clean = clean_identifier(df["physical_station_id_v1"])
station_clean  = clean_identifier(df["station_id"])

df["_station_physical_clean"] = physical_clean
df["_station_secondary_clean"] = station_clean

# Primary identity = physical_station_id_v1
# Fallback = station_id
df["_station_model_id"] = df["_station_physical_clean"]

fallback_mask = (
    df["_station_model_id"].isna() &
    df["_station_secondary_clean"].notna()
)

df.loc[fallback_mask, "_station_model_id"] = \
    df.loc[fallback_mask, "_station_secondary_clean"]

# If neither identifier exists, DO NOT pretend this is a known station.
# Use explicit unresolved groups at sample/event level so unrelated missing
# observations are not silently pooled into one fictitious station.

both_missing = df["_station_model_id"].isna()

def unresolved_station_label(row):
    for col in [
        "field_observation_id",
        "sample_id",
        "analytical_sample_id"
    ]:
        if col in row.index:
            val = row[col]
            if pd.notna(val):
                val = str(val).strip()
                if val.lower() not in {"", "nan", "none", "null", "na", "<na>"}:
                    return f"UNRESOLVED::{col}::{val}"

    return f"UNRESOLVED::ROW::{row.name}"


if both_missing.any():
    df.loc[both_missing, "_station_model_id"] = df.loc[both_missing].apply(
        unresolved_station_label,
        axis=1
    )

df["_station_resolution"] = np.select(
    [
        df["_station_physical_clean"].notna(),
        df["_station_physical_clean"].isna() &
        df["_station_secondary_clean"].notna(),
    ],
    [
        "PHYSICAL_STATION_ID_V1",
        "STATION_ID_FALLBACK",
    ],
    default="UNRESOLVED_EXPLICIT_GROUP"
)

station_audit = (
    df.groupby("_station_resolution", dropna=False)
      .agg(
          n_rows=("response_key", "size"),
          n_model_station_ids=("_station_model_id", "nunique")
      )
      .reset_index()
)

print(station_audit.to_string(index=False))

print("\nDistinct identifiers")
print("-" * 112)
print("Raw physical_station_id_v1:",
      df["physical_station_id_v1"].nunique(dropna=True))
print("Raw station_id:",
      df["station_id"].nunique(dropna=True))
print("Resolved model station IDs:",
      df["_station_model_id"].nunique(dropna=True))
print("Rows requiring station_id fallback:",
      int(fallback_mask.sum()))
print("Rows with both station identifiers missing:",
      int(both_missing.sum()))

# Explicitly detect the old "nan" problem
literal_nan_station = (
    df["_station_model_id"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("nan")
      .sum()
)

print('Literal "nan" station IDs after resolution:',
      int(literal_nan_station))

if literal_nan_station != 0:
    raise RuntimeError('Station identity failure: literal "nan" remains.')

station_audit.to_csv(OUT_STATION, index=False)

# ================================================================================================================
# PART 2 — RESPONSE ARCHITECTURE
# ================================================================================================================

required_arch_cols = [
    "response_key",
    "primary_transformation",
    "statistical_pathway",
]

missing_arch_cols = [x for x in required_arch_cols if x not in arch.columns]

if missing_arch_cols:
    raise RuntimeError(
        f"Architecture missing columns: {missing_arch_cols}"
    )

arch_primary = (
    arch[arch["response_key"].isin(PRIMARY_KEYS)]
    .drop_duplicates("response_key")
    .copy()
)

if len(arch_primary) != 10:
    raise RuntimeError(
        f"Expected 10 response architecture rows, got {len(arch_primary)}"
    )

ARCH_MAP = arch_primary.set_index("response_key").to_dict("index")

print("\n" + "=" * 112)
print("FROZEN RESPONSE ARCHITECTURE")
print("=" * 112)

print(
    arch_primary[
        [
            "response_key",
            "primary_transformation",
            "statistical_pathway"
        ]
    ].to_string(index=False)
)

# ================================================================================================================
# PART 3 — TRANSFORMATION FUNCTIONS
# ================================================================================================================

def transform_detected(values, transformation):
    values = np.asarray(values, dtype=float)

    if transformation == "ORIGINAL":
        return values

    elif transformation == "LOG10":
        if np.any(values <= 0):
            raise ValueError(
                "LOG10 requested with non-positive detected values."
            )
        return np.log10(values)

    elif transformation == "LOG1P":
        if np.any(values <= -1):
            raise ValueError(
                "LOG1P requested with values <= -1."
            )
        return np.log1p(values)

    else:
        raise ValueError(
            f"Unknown transformation: {transformation}"
        )


def transform_limits(values, transformation):
    values = np.asarray(values, dtype=float)

    if transformation == "ORIGINAL":
        return values

    elif transformation == "LOG10":
        if np.any(values <= 0):
            raise ValueError(
                "LOG10 censor limits must be > 0."
            )
        return np.log10(values)

    elif transformation == "LOG1P":
        if np.any(values <= -1):
            raise ValueError(
                "LOG1P censor limits must be > -1."
            )
        return np.log1p(values)

    else:
        raise ValueError(
            f"Unknown transformation: {transformation}"
        )

# ================================================================================================================
# PART 4 — DATA PREPARATION
# ================================================================================================================

def prepare_model_data(response_key, hydro_variable):

    d = df.loc[
        df["response_key"].eq(response_key)
    ].copy()

    pathway = ARCH_MAP[response_key]["statistical_pathway"]
    transformation = ARCH_MAP[response_key]["primary_transformation"]

    # ------------------------------------------------------------
    # Hydroclimate must exist
    # ------------------------------------------------------------

    d[hydro_variable] = pd.to_numeric(
        d[hydro_variable],
        errors="coerce"
    )

    d["response_value_numeric"] = pd.to_numeric(
        d["response_value_numeric"],
        errors="coerce"
    )

    d["response_censor_limit"] = pd.to_numeric(
        d["response_censor_limit"],
        errors="coerce"
    )

    cens = (
        d["response_censored"]
        .fillna(False)
        .astype(bool)
        .values
    )

    # ------------------------------------------------------------
    # Required response information
    # ------------------------------------------------------------

    valid = d[hydro_variable].notna()

    # detected rows require observed numeric response
    valid &= (
        (~pd.Series(cens, index=d.index)) &
        d["response_value_numeric"].notna()
    ) | (
        pd.Series(cens, index=d.index) &
        d["response_censor_limit"].notna()
    )

    d = d.loc[valid].copy()

    cens = (
        d["response_censored"]
        .fillna(False)
        .astype(bool)
        .values
    )

    # ------------------------------------------------------------
    # Hydroclimate standardization
    # ------------------------------------------------------------

    x_raw = d[hydro_variable].astype(float).values

    x_mean = np.mean(x_raw)
    x_sd = np.std(x_raw, ddof=1)

    if (not np.isfinite(x_sd)) or x_sd <= 0:
        raise ValueError(
            f"Hydroclimate predictor has invalid SD: {hydro_variable}"
        )

    x = (x_raw - x_mean) / x_sd

    # ------------------------------------------------------------
    # Response transformation
    # ------------------------------------------------------------

    y_raw = d["response_value_numeric"].astype(float).values

    y_trans = np.full(len(d), np.nan, dtype=float)

    det_mask = ~cens

    y_trans[det_mask] = transform_detected(
        y_raw[det_mask],
        transformation
    )

    # ------------------------------------------------------------
    # Censor limits
    # ------------------------------------------------------------

    censor_limit_trans = np.full(len(d), np.nan, dtype=float)

    if cens.any():

        raw_limits = d.loc[
            cens,
            "response_censor_limit"
        ].astype(float).values

        censor_limit_trans[cens] = transform_limits(
            raw_limits,
            transformation
        )

    # ------------------------------------------------------------
    # Response standardization
    #
    # For censor-aware responses:
    # location/scale are based on DETECTED transformed values only.
    # This is a deterministic scale transformation, NOT censor substitution.
    # ------------------------------------------------------------

    y_reference = y_trans[det_mask]

    y_mean = np.mean(y_reference)
    y_sd = np.std(y_reference, ddof=1)

    if (not np.isfinite(y_sd)) or y_sd <= 0:
        raise ValueError(
            f"Response has invalid SD: {response_key}"
        )

    y_std = np.full(len(d), np.nan, dtype=float)

    y_std[det_mask] = (
        y_trans[det_mask] - y_mean
    ) / y_sd

    censor_limit_std = np.full(len(d), np.nan, dtype=float)

    if cens.any():
        censor_limit_std[cens] = (
            censor_limit_trans[cens] - y_mean
        ) / y_sd

    # ------------------------------------------------------------
    # Station coding
    # ------------------------------------------------------------

    station_labels = d["_station_model_id"].astype(str).values

    station_codes, station_uniques = pd.factorize(
        station_labels,
        sort=True
    )

    if np.any(station_codes < 0):
        raise RuntimeError("Station factorization produced missing codes.")

    # ------------------------------------------------------------
    # Campaign/date QA
    # ------------------------------------------------------------

    n_campaigns = (
        d["campaign_id"].nunique(dropna=True)
        if "campaign_id" in d.columns else np.nan
    )

    n_dates = (
        d["hydroclimate_event_date"].nunique(dropna=True)
        if "hydroclimate_event_date" in d.columns else np.nan
    )

    return {
        "data": d,
        "pathway": pathway,
        "transformation": transformation,

        "x": x,
        "x_raw": x_raw,
        "x_mean": x_mean,
        "x_sd": x_sd,

        "y_std": y_std,
        "y_mean": y_mean,
        "y_sd": y_sd,

        "censored": cens,
        "censor_limit_std": censor_limit_std,

        "station_codes": station_codes.astype("int32"),
        "station_uniques": station_uniques,

        "n_rows": len(d),
        "n_detected": int((~cens).sum()),
        "n_censored": int(cens.sum()),
        "n_stations": len(station_uniques),
        "n_campaigns": n_campaigns,
        "n_dates": n_dates,
    }

# ================================================================================================================
# PART 5 — BAYESIAN MODEL
# ================================================================================================================

def fit_bayesian_model(
    prepared,
    draws=500,
    tune=500,
    chains=4,
    target_accept=0.90,
    random_seed=20260914,
    progressbar=False,
):

    x = prepared["x"]
    y_std = prepared["y_std"]
    cens = prepared["censored"]
    limit_std = prepared["censor_limit_std"]
    station_idx = prepared["station_codes"]

    n_station = prepared["n_stations"]

    with pm.Model() as model:

        # Weakly informative standardized-scale priors
        alpha = pm.Normal(
            "alpha",
            mu=0.0,
            sigma=1.0
        )

        beta_hydro = pm.Normal(
            "beta_hydro",
            mu=0.0,
            sigma=1.0
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1.0
        )

        z_station = pm.Normal(
            "z_station",
            mu=0.0,
            sigma=1.0,
            shape=n_station
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1.0
        )

        mu = (
            alpha
            + beta_hydro * x
            + station_effect[station_idx]
        )

        # --------------------------------------------------------
        # DIRECT CONTINUOUS
        # --------------------------------------------------------

        if not cens.any():

            pm.Normal(
                "y_obs",
                mu=mu,
                sigma=sigma,
                observed=y_std
            )

        # --------------------------------------------------------
        # CENSOR-AWARE CONTINUOUS
        # --------------------------------------------------------

        else:

            det_idx = np.where(~cens)[0]
            cen_idx = np.where(cens)[0]

            # Detected observations
            if len(det_idx) > 0:
                pm.Normal(
                    "y_detected",
                    mu=mu[det_idx],
                    sigma=sigma,
                    observed=y_std[det_idx]
                )

            # Left-censored observations:
            # contribution = log P(Y <= censor_limit)
            if len(cen_idx) > 0:

                base_dist = pm.Normal.dist(
                    mu=mu[cen_idx],
                    sigma=sigma
                )

                logcdf = pm.logcdf(
                    base_dist,
                    limit_std[cen_idx]
                )

                pm.Potential(
                    "left_censor_likelihood",
                    logcdf.sum()
                )

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            cores=min(chains, 4),
            target_accept=target_accept,
            random_seed=random_seed,
            progressbar=progressbar,
            return_inferencedata=True,
            compute_convergence_checks=True,
        )

    return idata

# ================================================================================================================
# PART 6 — MODEL SUMMARY
# ================================================================================================================

def summarize_fit(idata):

    summary = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_hydro",
            "sigma",
            "sigma_station"
        ],
        hdi_prob=0.95,
        round_to=None
    )

    beta_draws = (
        idata.posterior["beta_hydro"]
        .values
        .reshape(-1)
    )

    beta_row = summary.loc["beta_hydro"]

    divergences = int(
        idata.sample_stats["diverging"]
        .values
        .sum()
    )

    max_rhat = float(
        np.nanmax(
            summary["r_hat"].values
        )
    )

    min_ess_bulk = float(
        np.nanmin(
            summary["ess_bulk"].values
        )
    )

    min_ess_tail = float(
        np.nanmin(
            summary["ess_tail"].values
        )
    )

    beta_rhat = float(beta_row["r_hat"])
    beta_ess_bulk = float(beta_row["ess_bulk"])
    beta_ess_tail = float(beta_row["ess_tail"])

    convergence_pass = (
        divergences == 0
        and np.isfinite(max_rhat)
        and max_rhat <= 1.05
        and np.isfinite(min_ess_bulk)
        and min_ess_bulk >= 200
        and np.isfinite(beta_rhat)
        and beta_rhat <= 1.05
        and beta_ess_bulk >= 400
    )

    return {
        "beta_mean": float(np.mean(beta_draws)),
        "beta_median": float(np.median(beta_draws)),
        "beta_sd": float(np.std(beta_draws, ddof=1)),

        "beta_hdi_2_5": float(beta_row["hdi_2.5%"]),
        "beta_hdi_97_5": float(beta_row["hdi_97.5%"]),

        "prob_beta_positive": float(
            np.mean(beta_draws > 0)
        ),

        "prob_beta_negative": float(
            np.mean(beta_draws < 0)
        ),

        "prob_beta_same_sign": float(
            max(
                np.mean(beta_draws > 0),
                np.mean(beta_draws < 0)
            )
        ),

        "beta_rhat": beta_rhat,
        "beta_ess_bulk": beta_ess_bulk,
        "beta_ess_tail": beta_ess_tail,

        "max_rhat": max_rhat,
        "min_ess_bulk": min_ess_bulk,
        "min_ess_tail": min_ess_tail,

        "divergences": divergences,

        "sigma_mean": float(
            idata.posterior["sigma"]
            .values
            .mean()
        ),

        "sigma_station_mean": float(
            idata.posterior["sigma_station"]
            .values
            .mean()
        ),

        "engine_convergence_pass": bool(
            convergence_pass
        ),
    }

# ================================================================================================================
# PART 7 — MODEL GRID
# ================================================================================================================

grid = []

for response_key in PRIMARY_KEYS:

    for _, h in hydro_arch.iterrows():

        grid.append({
            "response_key": response_key,
            "hydroclimate_family": h["hydroclimate_family"],
            "window_days": int(h["window_days"]),
            "hydroclimate_variable": h["hydroclimate_variable"],
        })

grid = pd.DataFrame(grid)

assert len(grid) == 290

grid["model_id"] = [
    f"M{i:03d}"
    for i in range(1, len(grid) + 1)
]

# ================================================================================================================
# PART 8 — CHECKPOINT RECOVERY
# ================================================================================================================

if os.path.exists(CHECKPOINT):

    old = pd.read_csv(CHECKPOINT)

    completed_ids = set(
        old.loc[
            old["model_status"].eq("SUCCESS"),
            "model_id"
        ].astype(str)
    )

    results = old.to_dict("records")

    print("\nCheckpoint found.")
    print("Successful models already present:", len(completed_ids))

else:

    completed_ids = set()
    results = []

    print("\nNo previous checkpoint found.")

# ================================================================================================================
# PART 9 — FULL EFFECT-SPECTRUM ESTIMATION
# ================================================================================================================

print("\n" + "=" * 112)
print("FULL BAYESIAN EFFECT-SPECTRUM")
print("=" * 112)

print("Planned models:", len(grid))
print("Already completed:", len(completed_ids))
print("Remaining:", len(grid) - len(completed_ids))

run_start = time.time()

for i, row in grid.iterrows():

    model_id = row["model_id"]

    if model_id in completed_ids:
        continue

    response_key = row["response_key"]
    family = row["hydroclimate_family"]
    window = int(row["window_days"])
    hydro_var = row["hydroclimate_variable"]

    print(
        f"\n[{i+1:03d}/290] {model_id} | "
        f"{response_key} | {family} | {window} d"
    )

    t0 = time.time()

    try:

        prepared = prepare_model_data(
            response_key=response_key,
            hydro_variable=hydro_var
        )

        # --------------------------------------------------------
        # INITIAL FIT
        # --------------------------------------------------------

        idata = fit_bayesian_model(
            prepared,
            draws=500,
            tune=500,
            chains=4,
            target_accept=0.90,
            random_seed=20260914 + i,
            progressbar=False,
        )

        fit_summary = summarize_fit(idata)

        refit_performed = False
        fit_stage = "INITIAL_500_500"

        # --------------------------------------------------------
        # AUTOMATIC REFIT ONLY IF NEEDED
        # --------------------------------------------------------

        if not fit_summary["engine_convergence_pass"]:

            print("  Initial convergence gate failed → refitting...")

            del idata
            gc.collect()

            idata = fit_bayesian_model(
                prepared,
                draws=1000,
                tune=1000,
                chains=4,
                target_accept=0.97,
                random_seed=30260914 + i,
                progressbar=False,
            )

            fit_summary = summarize_fit(idata)

            refit_performed = True
            fit_stage = "REFIT_1000_1000"

        elapsed = time.time() - t0

        record = {
            "model_id": model_id,

            "response_key": response_key,
            "response_family":
                ARCH_MAP[response_key].get(
                    "response_family",
                    np.nan
                ),

            "primary_transformation":
                prepared["transformation"],

            "statistical_pathway":
                prepared["pathway"],

            "hydroclimate_family": family,
            "window_days": window,
            "hydroclimate_variable": hydro_var,

            "n_rows": prepared["n_rows"],
            "n_detected": prepared["n_detected"],
            "n_censored": prepared["n_censored"],
            "n_stations": prepared["n_stations"],
            "n_campaigns": prepared["n_campaigns"],
            "n_unique_hydroclimate_dates":
                prepared["n_dates"],

            "hydro_mean_raw":
                prepared["x_mean"],

            "hydro_sd_raw":
                prepared["x_sd"],

            "response_detected_transformed_mean":
                prepared["y_mean"],

            "response_detected_transformed_sd":
                prepared["y_sd"],

            **fit_summary,

            "refit_performed":
                refit_performed,

            "fit_stage":
                fit_stage,

            "elapsed_seconds":
                elapsed,

            "model_status":
                "SUCCESS",

            "memory_timescale_selected":
                False,

            "diagnostic_peak_used_for_selection":
                False,

            "loco_validation_performed":
                False,
        }

        results.append(record)

        print(
            "  beta = "
            f"{record['beta_mean']:.3f} "
            f"[{record['beta_hdi_2_5']:.3f}, "
            f"{record['beta_hdi_97_5']:.3f}] | "
            f"P(sign)={record['prob_beta_same_sign']:.3f} | "
            f"Rhat={record['max_rhat']:.3f} | "
            f"div={record['divergences']} | "
            f"{elapsed:.1f}s"
        )

        del idata
        gc.collect()

    except Exception as e:

        elapsed = time.time() - t0

        print("  MODEL FAILED:")
        print(" ", repr(e))

        record = {
            "model_id": model_id,
            "response_key": response_key,
            "hydroclimate_family": family,
            "window_days": window,
            "hydroclimate_variable": hydro_var,

            "model_status": "FAILED",
            "error_type": type(e).__name__,
            "error_message": str(e),

            "elapsed_seconds": elapsed,

            "memory_timescale_selected": False,
            "diagnostic_peak_used_for_selection": False,
            "loco_validation_performed": False,
        }

        results.append(record)

    # ------------------------------------------------------------
    # CHECKPOINT AFTER EVERY MODEL
    # ------------------------------------------------------------

    result_df = pd.DataFrame(results)

    # Keep only latest record per model_id in case of rerun
    result_df = (
        result_df
        .drop_duplicates(
            subset=["model_id"],
            keep="last"
        )
        .sort_values("model_id")
    )

    result_df.to_csv(
        CHECKPOINT,
        index=False
    )

# ================================================================================================================
# PART 10 — FINAL MODEL TABLE
# ================================================================================================================

spectrum = pd.DataFrame(results)

spectrum = (
    spectrum
    .drop_duplicates(
        subset=["model_id"],
        keep="last"
    )
    .sort_values("model_id")
    .reset_index(drop=True)
)

spectrum.to_csv(
    OUT_SPECTRUM,
    index=False
)

# ================================================================================================================
# PART 11 — FINAL QA
# ================================================================================================================

success = spectrum[
    spectrum["model_status"].eq("SUCCESS")
].copy()

failed = spectrum[
    ~spectrum["model_status"].eq("SUCCESS")
].copy()

print("\n" + "=" * 112)
print("MODEL COMPLETION QA")
print("=" * 112)

print("Expected models     :", 290)
print("Successful models   :", len(success))
print("Failed models       :", len(failed))

if len(success):

    print("Convergence PASS    :",
          int(success["engine_convergence_pass"].fillna(False).sum()))

    print("Convergence FAIL    :",
          int((~success["engine_convergence_pass"].fillna(False)).sum()))

    print("Automatic refits    :",
          int(success["refit_performed"].fillna(False).sum()))

    print("Total divergences   :",
          int(success["divergences"].fillna(0).sum()))

    print("Maximum R-hat       :",
          success["max_rhat"].max())

    print("Minimum ESS bulk    :",
          success["min_ess_bulk"].min())

# ================================================================================================================
# PART 12 — RESPONSE × DRIVER-FAMILY EFFECT CURVE SUMMARY
# ================================================================================================================

if len(success) == 290:

    curve_summary = (
        success
        .groupby(
            [
                "response_key",
                "hydroclimate_family"
            ],
            as_index=False
        )
        .agg(
            n_windows=("window_days", "size"),

            min_beta=("beta_mean", "min"),
            max_beta=("beta_mean", "max"),

            min_prob_same_sign=(
                "prob_beta_same_sign",
                "min"
            ),

            max_prob_same_sign=(
                "prob_beta_same_sign",
                "max"
            ),

            n_hdi_excluding_zero=(
                "beta_hdi_2_5",
                lambda x: 0
            ),
        )
    )

    # Correct HDI-exclusion count explicitly
    hdi_counts = (
        success.assign(
            hdi_excludes_zero=(
                (success["beta_hdi_2_5"] > 0) |
                (success["beta_hdi_97_5"] < 0)
            )
        )
        .groupby(
            [
                "response_key",
                "hydroclimate_family"
            ]
        )["hdi_excludes_zero"]
        .sum()
        .reset_index(
            name="n_hdi_excluding_zero"
        )
    )

    curve_summary = (
        curve_summary
        .drop(
            columns=["n_hdi_excluding_zero"]
        )
        .merge(
            hdi_counts,
            on=[
                "response_key",
                "hydroclimate_family"
            ],
            how="left"
        )
    )

    # Direction consistency from posterior means
    direction = (
        success.assign(
            beta_direction=np.where(
                success["beta_mean"] > 0,
                "POSITIVE",
                np.where(
                    success["beta_mean"] < 0,
                    "NEGATIVE",
                    "ZERO"
                )
            )
        )
        .groupby(
            [
                "response_key",
                "hydroclimate_family"
            ]
        )["beta_direction"]
        .agg(
            lambda s:
            "CONSISTENT_POSITIVE"
            if set(s) == {"POSITIVE"}
            else (
                "CONSISTENT_NEGATIVE"
                if set(s) == {"NEGATIVE"}
                else "SIGN_CHANGE_ACROSS_WINDOWS"
            )
        )
        .reset_index(
            name="posterior_mean_direction_pattern"
        )
    )

    curve_summary = curve_summary.merge(
        direction,
        on=[
            "response_key",
            "hydroclimate_family"
        ],
        how="left"
    )

else:

    curve_summary = pd.DataFrame()

curve_summary.to_csv(
    OUT_SUMMARY,
    index=False
)

# ================================================================================================================
# PART 13 — PRINT EFFECT CURVES
# ================================================================================================================

if len(success):

    print("\n" + "=" * 112)
    print("BAYESIAN EFFECT SPECTRUM — ALL SUCCESSFUL MODELS")
    print("=" * 112)

    show_cols = [
        "response_key",
        "hydroclimate_family",
        "window_days",
        "beta_mean",
        "beta_hdi_2_5",
        "beta_hdi_97_5",
        "prob_beta_positive",
        "prob_beta_negative",
        "prob_beta_same_sign",
        "divergences",
        "max_rhat",
        "min_ess_bulk",
        "refit_performed",
    ]

    print(
        success[show_cols]
        .sort_values(
            [
                "response_key",
                "hydroclimate_family",
                "window_days"
            ]
        )
        .to_string(index=False)
    )

# ================================================================================================================
# PART 14 — STRONGEST POSTERIOR EFFECTS
#
# IMPORTANT:
# This is descriptive read-back only.
# It MUST NOT be interpreted as memory-window selection.
# ================================================================================================================

if len(success):

    strongest = (
        success
        .assign(
            abs_beta=lambda x:
                x["beta_mean"].abs(),

            hdi_excludes_zero=lambda x:
                (x["beta_hdi_2_5"] > 0) |
                (x["beta_hdi_97_5"] < 0)
        )
        .sort_values(
            "abs_beta",
            ascending=False
        )
        .head(40)
    )

    print("\n" + "=" * 112)
    print("TOP 40 POSTERIOR EFFECT MAGNITUDES — DESCRIPTIVE ONLY; NOT MEMORY SELECTION")
    print("=" * 112)

    print(
        strongest[
            [
                "response_key",
                "hydroclimate_family",
                "window_days",
                "beta_mean",
                "beta_hdi_2_5",
                "beta_hdi_97_5",
                "prob_beta_same_sign",
                "hdi_excludes_zero",
            ]
        ].to_string(index=False)
    )

# ================================================================================================================
# PART 15 — CURVE SUMMARY
# ================================================================================================================

if len(curve_summary):

    print("\n" + "=" * 112)
    print("RESPONSE × DRIVER-FAMILY BAYESIAN CURVE SUMMARY")
    print("=" * 112)

    print(
        curve_summary
        .sort_values(
            [
                "response_key",
                "hydroclimate_family"
            ]
        )
        .to_string(index=False)
    )

# ================================================================================================================
# PART 16 — FINAL GATE
# ================================================================================================================

checks = []

def add_check(name, condition):
    checks.append({
        "check": name,
        "pass": bool(condition)
    })

add_check(
    "planned_models_290",
    len(grid) == 290
)

add_check(
    "successful_models_290",
    len(success) == 290
)

add_check(
    "failed_models_zero",
    len(failed) == 0
)

add_check(
    "all_models_converged",
    (
        len(success) == 290 and
        success["engine_convergence_pass"]
        .fillna(False)
        .all()
    )
)

add_check(
    "zero_total_divergences",
    (
        len(success) == 290 and
        success["divergences"]
        .fillna(0)
        .sum() == 0
    )
)

add_check(
    "no_literal_nan_station",
    literal_nan_station == 0
)

add_check(
    "response_panel_10",
    success["response_key"].nunique() == 10
    if len(success) else False
)

add_check(
    "hydroclimate_metrics_29",
    success["hydroclimate_variable"].nunique() == 29
    if len(success) else False
)

add_check(
    "response_family_curves_50",
    (
        success[
            [
                "response_key",
                "hydroclimate_family"
            ]
        ]
        .drop_duplicates()
        .shape[0] == 50
    )
    if len(success) else False
)

add_check(
    "no_lod_half_substitution",
    True
)

add_check(
    "no_censor_substitution",
    True
)

add_check(
    "analytical_subsamples_not_collapsed",
    True
)

add_check(
    "no_memory_timescale_selected",
    (
        len(success) == 290 and
        (~success["memory_timescale_selected"])
        .all()
    )
)

add_check(
    "no_loco_validation_yet",
    (
        len(success) == 290 and
        (~success["loco_validation_performed"])
        .all()
    )
)

gate = pd.DataFrame(checks)

print("\n" + "=" * 112)
print("BLOCK 8.5C FINAL GATE")
print("=" * 112)

print(gate.to_string(index=False))

BLOCK_PASS = bool(gate["pass"].all())

print("\nBLOCK 8.5C PASS:", BLOCK_PASS)

# ================================================================================================================
# PART 17 — AUDIT WORKBOOK
# ================================================================================================================

with pd.ExcelWriter(
    AUDIT_XLSX,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    station_audit.to_excel(
        writer,
        sheet_name="STATION_IDENTITY",
        index=False
    )

    hydro_arch.to_excel(
        writer,
        sheet_name="HYDRO_ARCHITECTURE",
        index=False
    )

    arch_primary.to_excel(
        writer,
        sheet_name="RESPONSE_ARCHITECTURE",
        index=False
    )

    grid.to_excel(
        writer,
        sheet_name="MODEL_GRID",
        index=False
    )

    spectrum.to_excel(
        writer,
        sheet_name="MODEL_RESULTS",
        index=False
    )

    if len(curve_summary):
        curve_summary.to_excel(
            writer,
            sheet_name="CURVE_SUMMARY",
            index=False
        )

    if len(failed):
        failed.to_excel(
            writer,
            sheet_name="FAILED_MODELS",
            index=False
        )

# ================================================================================================================
# PART 18 — MANIFEST
# ================================================================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "block": "8.5C",
    "title": "Full Bayesian Memory Effect-Spectrum Estimation",

    "input_memory_dataset": MEMORY_FILE,
    "input_response_architecture": ARCH_FILE,

    "n_primary_responses": 10,
    "n_hydroclimate_metrics": 29,
    "n_planned_models": 290,

    "n_successful_models": int(len(success)),
    "n_failed_models": int(len(failed)),

    "n_response_driver_curves":
        int(
            success[
                [
                    "response_key",
                    "hydroclimate_family"
                ]
            ]
            .drop_duplicates()
            .shape[0]
        )
        if len(success) else 0,

    "station_identity_policy": (
        "physical_station_id_v1 primary; "
        "station_id fallback; "
        "explicit unresolved event/sample grouping if both missing"
    ),

    "response_transformations":
        "Frozen in BLOCK 8.4B",

    "censoring_policy": (
        "Observation-specific left-censor likelihood; "
        "no substitution"
    ),

    "predictor_policy": (
        "One standardized antecedent hydroclimate predictor "
        "per model; no simultaneous nested windows"
    ),

    "random_effect":
        "station random intercept",

    "initial_sampling":
        "4 chains; 500 tune; 500 posterior draws; target_accept=0.90",

    "refit_policy":
        "Failed convergence gate refit with 4 chains; "
        "1000 tune; 1000 posterior draws; target_accept=0.97",

    "convergence_gate":
        {
            "divergences": 0,
            "max_rhat": "<=1.05",
            "min_ess_bulk": ">=200",
            "beta_rhat": "<=1.05",
            "beta_ess_bulk": ">=400",
        },

    "memory_timescale_selected": False,
    "loco_validation_performed": False,

    "block_pass": BLOCK_PASS,

    "output_spectrum": OUT_SPECTRUM,
    "output_curve_summary": OUT_SUMMARY,
    "output_station_audit": OUT_STATION,
    "audit_workbook": AUDIT_XLSX,
}

if os.path.exists(OUT_SPECTRUM):
    manifest["spectrum_sha256"] = sha256_file(
        OUT_SPECTRUM
    )

with open(
    MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================================================================
# FINAL DIAGNOSTIC
# ================================================================================================================

elapsed_total = time.time() - run_start

print("\n" + "=" * 112)
print("BLOCK 8.5C — FINAL DIAGNOSTIC")
print("=" * 112)

print("Primary responses                    :", 10)
print("Hydroclimate metrics                 :", 29)
print("Planned Bayesian models              :", 290)
print("Successful Bayesian models           :", len(success))
print("Failed Bayesian models               :", len(failed))

if len(success):
    print(
        "Models requiring automatic refit     :",
        int(
            success["refit_performed"]
            .fillna(False)
            .sum()
        )
    )

    print(
        "Total divergences                    :",
        int(
            success["divergences"]
            .fillna(0)
            .sum()
        )
    )

    print(
        "Maximum R-hat                        :",
        success["max_rhat"].max()
    )

    print(
        "Minimum ESS bulk                     :",
        success["min_ess_bulk"].min()
    )

print(
    "Rows with station_id fallback        :",
    int(fallback_mask.sum())
)

print(
    "Rows with both station IDs missing   :",
    int(both_missing.sum())
)

print(
    'Literal "nan" station IDs             :',
    int(literal_nan_station)
)

print("\nCensor substitution                  : NO")
print("LOD/2                                : NO")
print("Analytical subsamples collapsed      : NO")
print("Nested windows entered simultaneously: NO")
print("Memory timescale selected            : NO")
print("LOCO campaign validation             : NOT YET")

print(
    "\nElapsed total minutes                 :",
    round(elapsed_total / 60, 2)
)

print("\nEffect spectrum:")
print(OUT_SPECTRUM)

print("\nCurve summary:")
print(OUT_SUMMARY)

print("\nStation identity audit:")
print(OUT_STATION)

print("\nAudit workbook:")
print(AUDIT_XLSX)

print("\nManifest:")
print(MANIFEST)

print("\nBLOCK 8.5C PASS                      :", BLOCK_PASS)

if BLOCK_PASS:
    print(
        "\nBLOCK 8.5C COMPLETE — "
        "FULL BAYESIAN EFFECT SPECTRUM ESTIMATED; "
        "NO MEMORY TIMESCALE SELECTED."
    )
else:
    print(
        "\nBLOCK 8.5C NOT FROZEN — "
        "REVIEW FAILED OR NON-CONVERGED MODELS BEFORE PROCEEDING."
    )

---
### Imported master block 79

`BLOCK 8.5D-AUDIT — CAMPAIGN-AWARE LOCO READINESS AUDIT`


In [ ]:
# ================================================================================================================
# BLOCK 8.5D-AUDIT — CAMPAIGN-AWARE LOCO READINESS AUDIT
# ================================================================================================================
#
# PURPOSE
# -------
# Audit the exact data structure required BEFORE leave-one-campaign-out (LOCO)
# predictive validation of the hydroclimatic memory spectrum.
#
# QUESTIONS
# ---------
# 1. How many observations does each response contribute to each campaign?
# 2. How many stations recur across campaigns?
# 3. In each LOCO fold, which test stations are already represented in training?
# 4. Which test observations belong to genuinely unseen stations?
# 5. How is censoring distributed across train/test folds?
# 6. Does every response retain adequate training support in all 10 folds?
# 7. Are the 29 hydroclimate windows complete in every fold?
#
# IMPORTANT
# ---------
# - NO model fitting.
# - NO memory-window selection.
# - NO predictive scoring.
# - NO censor substitution.
# - NO LOD/2.
# - NO averaging/collapsing analytical subsamples.
# - NO modification of frozen BLOCK 8.5C products.
#
# ================================================================================================================

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 120)
print("BLOCK 8.5D-AUDIT — CAMPAIGN-AWARE LOCO READINESS AUDIT")
print("=" * 120)

# ------------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------------

BASE = Path("/content/Huanta_JoH")

MEMORY_FILE = BASE / "03_MASTER" / "MEMORY_ANALYSIS_DATASET_v1.csv"
ARCH_FILE   = BASE / "03_MASTER" / "MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"

OUT_STATION_RECURRENCE = (
    BASE / "03_MASTER" /
    "LOCO_STATION_RECURRENCE_AUDIT_v1.csv"
)

OUT_CAMPAIGN_RESPONSE = (
    BASE / "03_MASTER" /
    "LOCO_CAMPAIGN_RESPONSE_COVERAGE_v1.csv"
)

OUT_FOLDS = (
    BASE / "03_MASTER" /
    "LOCO_FOLD_READINESS_v1.csv"
)

OUT_CENSOR = (
    BASE / "03_MASTER" /
    "LOCO_CENSORING_BY_CAMPAIGN_v1.csv"
)

OUT_HYDRO = (
    BASE / "03_MASTER" /
    "LOCO_HYDROCLIMATE_FOLD_AUDIT_v1.csv"
)

AUDIT_XLSX = (
    BASE / "01_AUDIT" /
    "BLOCK85D_AUDIT_loco_readiness.xlsx"
)

MANIFEST = (
    BASE / "01_AUDIT" /
    "BLOCK85D_AUDIT_loco_readiness_manifest.json"
)

for p in [
    BASE / "03_MASTER",
    BASE / "01_AUDIT"
]:
    p.mkdir(parents=True, exist_ok=True)

for f in [MEMORY_FILE, ARCH_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"Required file missing: {f}")

# ------------------------------------------------------------------------------------------------
# LOAD
# ------------------------------------------------------------------------------------------------

df = pd.read_csv(MEMORY_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE, low_memory=False)

print("\nINPUT")
print("-" * 120)
print("Memory dataset :", df.shape)
print("Architecture   :", arch.shape)

# ------------------------------------------------------------------------------------------------
# FROZEN RESPONSE PANEL
# ------------------------------------------------------------------------------------------------

PRIMARY_KEYS = [
    "AC::pH_SOL::NO_UNIT",
    "AC::Na::mg/L",
    "AC::C.E._SOL::dS/m",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "AC::NH4::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
    "AC::SO4_ppm::mg/L",
    "FIELD::Water temperature::degC",
]

observed_keys = set(df["response_key"].dropna().unique())

if observed_keys != set(PRIMARY_KEYS):
    raise RuntimeError(
        "Primary response identity mismatch.\n"
        f"Observed: {sorted(observed_keys)}"
    )

# ------------------------------------------------------------------------------------------------
# EXACT HYDROCLIMATE ARCHITECTURE
# ------------------------------------------------------------------------------------------------

HYDRO_FAMILIES = {
    "precipitation": {
        1: "ppt_sum_1d",
        3: "ppt_sum_3d",
        7: "ppt_sum_7d",
        14: "ppt_sum_14d",
        30: "ppt_sum_30d",
        60: "ppt_sum_60d",
        90: "ppt_sum_90d",
    },

    "surface_soil_moisture": {
        1: "sm_surface_mean_1d",
        7: "sm_surface_mean_7d",
        14: "sm_surface_mean_14d",
        30: "sm_surface_mean_30d",
        60: "sm_surface_mean_60d",
        90: "sm_surface_mean_90d",
    },

    "rootzone_soil_moisture": {
        1: "sm_rootzone_mean_1d",
        7: "sm_rootzone_mean_7d",
        14: "sm_rootzone_mean_14d",
        30: "sm_rootzone_mean_30d",
        60: "sm_rootzone_mean_60d",
        90: "sm_rootzone_mean_90d",
    },

    "air_temperature": {
        7: "tair_mean_7d",
        14: "tair_mean_14d",
        30: "tair_mean_30d",
        60: "tair_mean_60d",
        90: "tair_mean_90d",
    },

    "potential_evaporation": {
        7: "pet_sum_7d",
        14: "pet_sum_14d",
        30: "pet_sum_30d",
        60: "pet_sum_60d",
        90: "pet_sum_90d",
    },
}

hydro_rows = []

for family, windows in HYDRO_FAMILIES.items():
    for window, variable in windows.items():
        hydro_rows.append({
            "hydroclimate_family": family,
            "window_days": int(window),
            "hydroclimate_variable": variable,
        })

hydro_arch = pd.DataFrame(hydro_rows)

assert len(hydro_arch) == 29

missing_hydro = [
    c for c in hydro_arch["hydroclimate_variable"]
    if c not in df.columns
]

if missing_hydro:
    raise RuntimeError(
        f"Missing hydroclimate columns: {missing_hydro}"
    )

# ------------------------------------------------------------------------------------------------
# CAMPAIGN ID QA
# ------------------------------------------------------------------------------------------------

if "campaign_id" not in df.columns:
    raise RuntimeError("campaign_id missing.")

campaigns = sorted(
    df["campaign_id"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nCAMPAIGNS")
print("-" * 120)
print("Campaigns:", campaigns)
print("N campaigns:", len(campaigns))

if len(campaigns) != 10:
    raise RuntimeError(
        f"Expected exactly 10 campaigns, found {len(campaigns)}."
    )

# ================================================================================================================
# PART 1 — STATION IDENTITY
# ================================================================================================================

print("\n" + "=" * 120)
print("STATION IDENTITY RECONSTRUCTION")
print("=" * 120)

def clean_identifier(series):

    s = series.copy()

    s = s.where(
        s.notna(),
        np.nan
    )

    s = s.map(
        lambda x:
        str(x).strip()
        if pd.notna(x)
        else np.nan
    )

    bad = {
        "",
        "nan",
        "none",
        "null",
        "na",
        "<na>"
    }

    s = s.map(
        lambda x:
        np.nan
        if pd.isna(x)
        or str(x).strip().lower() in bad
        else str(x).strip()
    )

    return s


physical = clean_identifier(
    df["physical_station_id_v1"]
)

secondary = clean_identifier(
    df["station_id"]
)

df["_physical_station_clean"] = physical
df["_secondary_station_clean"] = secondary

df["_station_model_id"] = physical.copy()

fallback_mask = (
    df["_station_model_id"].isna()
    &
    secondary.notna()
)

df.loc[
    fallback_mask,
    "_station_model_id"
] = secondary.loc[fallback_mask]


def unresolved_label(row):

    for col in [
        "field_observation_id",
        "sample_id",
        "analytical_sample_id"
    ]:

        if col not in row.index:
            continue

        value = row[col]

        if pd.notna(value):

            value = str(value).strip()

            if value.lower() not in {
                "",
                "nan",
                "none",
                "null",
                "na",
                "<na>"
            }:

                return (
                    f"UNRESOLVED::{col}::{value}"
                )

    return f"UNRESOLVED::ROW::{row.name}"


both_missing = df["_station_model_id"].isna()

if both_missing.any():

    df.loc[
        both_missing,
        "_station_model_id"
    ] = df.loc[
        both_missing
    ].apply(
        unresolved_label,
        axis=1
    )

df["_station_resolution"] = np.select(
    [
        physical.notna(),

        physical.isna()
        & secondary.notna(),
    ],
    [
        "PHYSICAL_STATION_ID_V1",
        "STATION_ID_FALLBACK",
    ],
    default="UNRESOLVED_EXPLICIT_GROUP"
)

literal_nan = (
    df["_station_model_id"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("nan")
    .sum()
)

print("Physical station IDs       :", physical.nunique(dropna=True))
print("Secondary station IDs      :", secondary.nunique(dropna=True))
print("Rows using fallback        :", int(fallback_mask.sum()))
print("Rows both IDs missing      :", int(both_missing.sum()))
print("Model grouping IDs         :", df["_station_model_id"].nunique())
print('Literal "nan" IDs          :', int(literal_nan))

if literal_nan:
    raise RuntimeError('Literal "nan" station ID detected.')

# ================================================================================================================
# PART 2 — STATION RECURRENCE ACROSS CAMPAIGNS
# ================================================================================================================

station_recurrence = (
    df[
        [
            "_station_model_id",
            "_station_resolution",
            "campaign_id",
            "response_key"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "_station_model_id",
            "_station_resolution"
        ],
        as_index=False
    )
    .agg(
        n_campaigns=("campaign_id", "nunique"),
        n_responses=("response_key", "nunique")
    )
)

station_recurrence["recurrence_class"] = pd.cut(
    station_recurrence["n_campaigns"],
    bins=[0, 1, 2, 4, 7, np.inf],
    labels=[
        "ONE_CAMPAIGN_ONLY",
        "TWO_CAMPAIGNS",
        "THREE_TO_FOUR",
        "FIVE_TO_SEVEN",
        "EIGHT_TO_TEN"
    ]
)

station_recurrence.to_csv(
    OUT_STATION_RECURRENCE,
    index=False
)

print("\nSTATION RECURRENCE")
print("-" * 120)

print(
    station_recurrence[
        "recurrence_class"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nCampaign recurrence distribution:")
print(
    station_recurrence["n_campaigns"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ================================================================================================================
# PART 3 — RESPONSE × CAMPAIGN COVERAGE
# ================================================================================================================

campaign_response_rows = []

for response in PRIMARY_KEYS:

    dr = df[
        df["response_key"].eq(response)
    ].copy()

    for campaign in campaigns:

        dc = dr[
            dr["campaign_id"]
            .astype(str)
            .eq(campaign)
        ].copy()

        cens = (
            dc["response_censored"]
            .fillna(False)
            .astype(bool)
        )

        campaign_response_rows.append({
            "response_key": response,
            "campaign_id": campaign,

            "n_rows": len(dc),

            "n_detected":
                int((~cens).sum()),

            "n_censored":
                int(cens.sum()),

            "censor_fraction":
                float(cens.mean())
                if len(dc)
                else np.nan,

            "n_station_groups":
                dc["_station_model_id"]
                .nunique(),

            "n_physical_station_ids":
                dc["_physical_station_clean"]
                .nunique(dropna=True),

            "n_unique_hydroclimate_dates":
                dc["hydroclimate_event_date"]
                .nunique(dropna=True),

            "n_analytical_subsamples":
                dc["analytical_sample_id"]
                .nunique(dropna=True)
                if "analytical_sample_id" in dc.columns
                else np.nan,
        })

campaign_response = pd.DataFrame(
    campaign_response_rows
)

campaign_response.to_csv(
    OUT_CAMPAIGN_RESPONSE,
    index=False
)

print("\n" + "=" * 120)
print("RESPONSE × CAMPAIGN COVERAGE")
print("=" * 120)

coverage_pivot = campaign_response.pivot(
    index="response_key",
    columns="campaign_id",
    values="n_rows"
)

print("\nRows:")
print(coverage_pivot.to_string())

print("\nMinimum campaign rows by response:")
print(
    campaign_response
    .groupby("response_key")["n_rows"]
    .min()
    .sort_values()
    .to_string()
)

# ================================================================================================================
# PART 4 — CENSORING BY CAMPAIGN
# ================================================================================================================

censor_audit = (
    campaign_response[
        [
            "response_key",
            "campaign_id",
            "n_rows",
            "n_detected",
            "n_censored",
            "censor_fraction"
        ]
    ]
    .copy()
)

censor_audit.to_csv(
    OUT_CENSOR,
    index=False
)

print("\n" + "=" * 120)
print("CENSORING BY CAMPAIGN")
print("=" * 120)

censored_only = censor_audit[
    censor_audit["n_censored"] > 0
].copy()

if len(censored_only):

    print(
        censored_only
        .sort_values(
            [
                "response_key",
                "campaign_id"
            ]
        )
        .to_string(index=False)
    )

else:

    print("No censored observations detected.")

# ================================================================================================================
# PART 5 — EXACT LOCO FOLD ARCHITECTURE
# ================================================================================================================

fold_rows = []

for response in PRIMARY_KEYS:

    dr = df[
        df["response_key"].eq(response)
    ].copy()

    for held_out in campaigns:

        train = dr[
            ~dr["campaign_id"]
            .astype(str)
            .eq(held_out)
        ].copy()

        test = dr[
            dr["campaign_id"]
            .astype(str)
            .eq(held_out)
        ].copy()

        train_stations = set(
            train["_station_model_id"]
            .dropna()
            .astype(str)
        )

        test_stations = set(
            test["_station_model_id"]
            .dropna()
            .astype(str)
        )

        seen_test_stations = (
            test_stations
            & train_stations
        )

        unseen_test_stations = (
            test_stations
            - train_stations
        )

        test_seen_mask = (
            test["_station_model_id"]
            .astype(str)
            .isin(seen_test_stations)
        )

        test_unseen_mask = (
            test["_station_model_id"]
            .astype(str)
            .isin(unseen_test_stations)
        )

        train_cens = (
            train["response_censored"]
            .fillna(False)
            .astype(bool)
        )

        test_cens = (
            test["response_censored"]
            .fillna(False)
            .astype(bool)
        )

        # --------------------------------------------------------
        # Censor-limit support
        # --------------------------------------------------------

        train_limits = set(
            pd.to_numeric(
                train.loc[
                    train_cens,
                    "response_censor_limit"
                ],
                errors="coerce"
            )
            .dropna()
            .round(12)
            .tolist()
        )

        test_limits = set(
            pd.to_numeric(
                test.loc[
                    test_cens,
                    "response_censor_limit"
                ],
                errors="coerce"
            )
            .dropna()
            .round(12)
            .tolist()
        )

        unseen_test_limits = (
            test_limits - train_limits
        )

        # --------------------------------------------------------
        # Response support
        # --------------------------------------------------------

        train_detected = int(
            (~train_cens).sum()
        )

        test_detected = int(
            (~test_cens).sum()
        )

        # Conservative readiness checks.
        #
        # These are NOT inferential thresholds.
        # They only catch structurally unusable folds.
        # --------------------------------------------------------

        train_support_pass = (
            len(train) >= 30
            and train_detected >= 20
            and train["_station_model_id"].nunique() >= 5
            and train["campaign_id"].nunique() == 9
        )

        test_support_pass = (
            len(test) >= 1
        )

        fold_rows.append({

            "response_key": response,
            "held_out_campaign": held_out,

            "n_train": len(train),
            "n_test": len(test),

            "n_train_detected": train_detected,
            "n_train_censored":
                int(train_cens.sum()),

            "n_test_detected": test_detected,
            "n_test_censored":
                int(test_cens.sum()),

            "n_train_campaigns":
                train["campaign_id"].nunique(),

            "n_train_station_groups":
                train["_station_model_id"].nunique(),

            "n_test_station_groups":
                len(test_stations),

            "n_test_seen_station_groups":
                len(seen_test_stations),

            "n_test_unseen_station_groups":
                len(unseen_test_stations),

            "n_test_rows_seen_station":
                int(test_seen_mask.sum()),

            "n_test_rows_unseen_station":
                int(test_unseen_mask.sum()),

            "fraction_test_rows_seen_station":
                float(test_seen_mask.mean())
                if len(test)
                else np.nan,

            "fraction_test_rows_unseen_station":
                float(test_unseen_mask.mean())
                if len(test)
                else np.nan,

            "n_train_hydro_dates":
                train[
                    "hydroclimate_event_date"
                ].nunique(dropna=True),

            "n_test_hydro_dates":
                test[
                    "hydroclimate_event_date"
                ].nunique(dropna=True),

            "n_train_censor_limits":
                len(train_limits),

            "n_test_censor_limits":
                len(test_limits),

            "n_test_censor_limits_unseen_in_train":
                len(unseen_test_limits),

            "train_support_pass":
                train_support_pass,

            "test_support_pass":
                test_support_pass,
        })

folds = pd.DataFrame(fold_rows)

folds.to_csv(
    OUT_FOLDS,
    index=False
)

print("\n" + "=" * 120)
print("LOCO FOLD READINESS")
print("=" * 120)

print("Expected folds       :", 10 * 10)
print("Observed folds       :", len(folds))

print(
    "Train support PASS   :",
    int(folds["train_support_pass"].sum()),
    "/",
    len(folds)
)

print(
    "Test support PASS    :",
    int(folds["test_support_pass"].sum()),
    "/",
    len(folds)
)

print(
    "Folds with unseen test stations:",
    int(
        (
            folds[
                "n_test_unseen_station_groups"
            ] > 0
        ).sum()
    )
)

print(
    "Total test rows assigned to unseen stations:",
    int(
        folds[
            "n_test_rows_unseen_station"
        ].sum()
    )
)

print(
    "Minimum train observations:",
    int(folds["n_train"].min())
)

print(
    "Minimum train detected:",
    int(folds["n_train_detected"].min())
)

print(
    "Minimum train station groups:",
    int(folds["n_train_station_groups"].min())
)

# ------------------------------------------------------------------------------------------------
# Folds containing unseen station groups
# ------------------------------------------------------------------------------------------------

unseen_folds = folds[
    folds["n_test_unseen_station_groups"] > 0
].copy()

print("\nFOLDS WITH UNSEEN TEST STATIONS")
print("-" * 120)

if len(unseen_folds):

    print(
        unseen_folds[
            [
                "response_key",
                "held_out_campaign",
                "n_test",
                "n_test_station_groups",
                "n_test_seen_station_groups",
                "n_test_unseen_station_groups",
                "n_test_rows_seen_station",
                "n_test_rows_unseen_station",
                "fraction_test_rows_unseen_station",
            ]
        ]
        .sort_values(
            [
                "response_key",
                "held_out_campaign"
            ]
        )
        .to_string(index=False)
    )

else:

    print("NONE")

# ================================================================================================================
# PART 6 — HYDROCLIMATE COMPLETENESS IN EACH LOCO FOLD
# ================================================================================================================

hydro_fold_rows = []

for held_out in campaigns:

    train = df[
        ~df["campaign_id"]
        .astype(str)
        .eq(held_out)
    ].copy()

    test = df[
        df["campaign_id"]
        .astype(str)
        .eq(held_out)
    ].copy()

    for _, h in hydro_arch.iterrows():

        var = h["hydroclimate_variable"]

        train_values = pd.to_numeric(
            train[var],
            errors="coerce"
        )

        test_values = pd.to_numeric(
            test[var],
            errors="coerce"
        )

        hydro_fold_rows.append({

            "held_out_campaign": held_out,

            "hydroclimate_family":
                h["hydroclimate_family"],

            "window_days":
                int(h["window_days"]),

            "hydroclimate_variable":
                var,

            "n_train_rows":
                len(train),

            "n_test_rows":
                len(test),

            "n_train_missing":
                int(train_values.isna().sum()),

            "n_test_missing":
                int(test_values.isna().sum()),

            "train_sd":
                float(train_values.std(ddof=1)),

            "test_sd":
                float(test_values.std(ddof=1))
                if test_values.notna().sum() > 1
                else np.nan,

            "train_variation_pass":
                bool(
                    train_values.notna().sum() > 1
                    and
                    train_values.std(ddof=1) > 0
                ),

            "hydro_complete_pass":
                bool(
                    train_values.notna().all()
                    and
                    test_values.notna().all()
                ),
        })

hydro_folds = pd.DataFrame(
    hydro_fold_rows
)

hydro_folds.to_csv(
    OUT_HYDRO,
    index=False
)

print("\n" + "=" * 120)
print("HYDROCLIMATE LOCO COMPLETENESS")
print("=" * 120)

print("Expected campaign × metric checks:", 10 * 29)
print("Observed checks                  :", len(hydro_folds))

print(
    "Complete train/test checks       :",
    int(
        hydro_folds[
            "hydro_complete_pass"
        ].sum()
    ),
    "/",
    len(hydro_folds)
)

print(
    "Training variation checks PASS   :",
    int(
        hydro_folds[
            "train_variation_pass"
        ].sum()
    ),
    "/",
    len(hydro_folds)
)

# ================================================================================================================
# PART 7 — RESPONSE-SPECIFIC LOCO SUMMARY
# ================================================================================================================

response_loco_summary = (
    folds
    .groupby(
        "response_key",
        as_index=False
    )
    .agg(
        n_folds=(
            "held_out_campaign",
            "size"
        ),

        min_train_n=(
            "n_train",
            "min"
        ),

        min_test_n=(
            "n_test",
            "min"
        ),

        min_train_detected=(
            "n_train_detected",
            "min"
        ),

        max_test_censored=(
            "n_test_censored",
            "max"
        ),

        min_train_station_groups=(
            "n_train_station_groups",
            "min"
        ),

        folds_with_unseen_test_stations=(
            "n_test_unseen_station_groups",
            lambda s: int((s > 0).sum())
        ),

        total_test_rows_unseen_station=(
            "n_test_rows_unseen_station",
            "sum"
        ),

        min_fraction_test_rows_seen_station=(
            "fraction_test_rows_seen_station",
            "min"
        ),

        all_train_support_pass=(
            "train_support_pass",
            "all"
        ),

        all_test_support_pass=(
            "test_support_pass",
            "all"
        ),
    )
)

print("\n" + "=" * 120)
print("RESPONSE-SPECIFIC LOCO SUMMARY")
print("=" * 120)

print(
    response_loco_summary
    .to_string(index=False)
)

# ================================================================================================================
# PART 8 — GLOBAL INTERPRETIVE COUNTS
# ================================================================================================================

n_fold_unseen = int(
    (
        folds[
            "n_test_unseen_station_groups"
        ] > 0
    ).sum()
)

n_rows_unseen = int(
    folds[
        "n_test_rows_unseen_station"
    ].sum()
)

n_total_test_rows_across_folds = int(
    folds["n_test"].sum()
)

fraction_unseen_global = (
    n_rows_unseen /
    n_total_test_rows_across_folds
    if n_total_test_rows_across_folds
    else np.nan
)

censor_limit_novel_folds = int(
    (
        folds[
            "n_test_censor_limits_unseen_in_train"
        ] > 0
    ).sum()
)

print("\n" + "=" * 120)
print("GLOBAL LOCO STRUCTURE")
print("=" * 120)

print(
    "Response × campaign folds                 :",
    len(folds)
)

print(
    "Folds containing unseen station groups    :",
    n_fold_unseen
)

print(
    "Test rows from unseen station groups       :",
    n_rows_unseen
)

print(
    "Total test rows across all response folds  :",
    n_total_test_rows_across_folds
)

print(
    "Fraction of test rows from unseen stations :",
    round(fraction_unseen_global, 4)
)

print(
    "Folds with censor limit absent from train  :",
    censor_limit_novel_folds
)

# ================================================================================================================
# PART 9 — FINAL GATE
# ================================================================================================================

checks = []

def add_check(name, condition):

    checks.append({
        "check": name,
        "pass": bool(condition)
    })


add_check(
    "primary_responses_10",
    len(PRIMARY_KEYS) == 10
)

add_check(
    "campaigns_10",
    len(campaigns) == 10
)

add_check(
    "hydroclimate_metrics_29",
    len(hydro_arch) == 29
)

add_check(
    "response_campaign_cells_100",
    len(campaign_response) == 100
)

add_check(
    "loco_response_folds_100",
    len(folds) == 100
)

add_check(
    "all_train_sets_have_9_campaigns",
    (
        folds[
            "n_train_campaigns"
        ] == 9
    ).all()
)

add_check(
    "all_train_support_pass",
    folds[
        "train_support_pass"
    ].all()
)

add_check(
    "all_test_support_pass",
    folds[
        "test_support_pass"
    ].all()
)

add_check(
    "hydro_fold_checks_290",
    len(hydro_folds) == 290
)

add_check(
    "all_hydro_complete",
    hydro_folds[
        "hydro_complete_pass"
    ].all()
)

add_check(
    "all_hydro_train_variation_pass",
    hydro_folds[
        "train_variation_pass"
    ].all()
)

add_check(
    "no_literal_nan_station",
    literal_nan == 0
)

add_check(
    "no_models_fitted",
    True
)

add_check(
    "no_memory_timescale_selected",
    True
)

add_check(
    "no_censor_substitution",
    True
)

add_check(
    "no_lod_half",
    True
)

add_check(
    "analytical_subsamples_not_collapsed",
    True
)

gate = pd.DataFrame(checks)

BLOCK_PASS = bool(
    gate["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 8.5D-AUDIT FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print(
    "\nBLOCK 8.5D-AUDIT PASS:",
    BLOCK_PASS
)

# ================================================================================================================
# PART 10 — WRITE AUDIT WORKBOOK
# ================================================================================================================

with pd.ExcelWriter(
    AUDIT_XLSX,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    station_recurrence.to_excel(
        writer,
        sheet_name="STATION_RECURRENCE",
        index=False
    )

    campaign_response.to_excel(
        writer,
        sheet_name="CAMPAIGN_RESPONSE",
        index=False
    )

    censor_audit.to_excel(
        writer,
        sheet_name="CENSORING",
        index=False
    )

    folds.to_excel(
        writer,
        sheet_name="LOCO_FOLDS",
        index=False
    )

    hydro_folds.to_excel(
        writer,
        sheet_name="HYDRO_FOLDS",
        index=False
    )

    response_loco_summary.to_excel(
        writer,
        sheet_name="RESPONSE_SUMMARY",
        index=False
    )

# ================================================================================================================
# PART 11 — MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda:
            f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {

    "block":
        "8.5D-AUDIT",

    "title":
        "Campaign-aware LOCO readiness audit",

    "input_memory_dataset":
        str(MEMORY_FILE),

    "n_primary_responses":
        10,

    "n_campaigns":
        10,

    "n_hydroclimate_metrics":
        29,

    "n_response_campaign_cells":
        int(len(campaign_response)),

    "n_response_loco_folds":
        int(len(folds)),

    "n_hydroclimate_fold_checks":
        int(len(hydro_folds)),

    "station_identity_policy":
        (
            "physical_station_id_v1 primary; "
            "station_id fallback; "
            "explicit unresolved event/sample grouping "
            "when both identifiers are absent"
        ),

    "fold_definition":
        (
            "Leave exactly one complete campaign out; "
            "train on remaining nine campaigns."
        ),

    "n_folds_with_unseen_test_station_groups":
        n_fold_unseen,

    "n_test_rows_from_unseen_station_groups":
        n_rows_unseen,

    "fraction_test_rows_from_unseen_station_groups":
        (
            None
            if not np.isfinite(fraction_unseen_global)
            else float(fraction_unseen_global)
        ),

    "n_folds_with_test_censor_limit_absent_from_train":
        censor_limit_novel_folds,

    "models_fitted":
        False,

    "memory_timescale_selected":
        False,

    "censor_substitution":
        False,

    "lod_half_substitution":
        False,

    "analytical_subsamples_collapsed":
        False,

    "block_pass":
        BLOCK_PASS,

    "outputs": {
        "station_recurrence":
            str(OUT_STATION_RECURRENCE),

        "campaign_response_coverage":
            str(OUT_CAMPAIGN_RESPONSE),

        "fold_readiness":
            str(OUT_FOLDS),

        "censoring_by_campaign":
            str(OUT_CENSOR),

        "hydroclimate_fold_audit":
            str(OUT_HYDRO),

        "audit_workbook":
            str(AUDIT_XLSX),
    }
}

for key, path in [
    ("station_recurrence_sha256",
     OUT_STATION_RECURRENCE),

    ("campaign_response_sha256",
     OUT_CAMPAIGN_RESPONSE),

    ("fold_readiness_sha256",
     OUT_FOLDS),

    ("censoring_sha256",
     OUT_CENSOR),

    ("hydro_fold_sha256",
     OUT_HYDRO),

    ("audit_workbook_sha256",
     AUDIT_XLSX),
]:

    if path.exists():
        manifest[key] = sha256_file(path)


with open(
    MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================================================================
# FINAL DIAGNOSTIC
# ================================================================================================================

print("\n" + "=" * 120)
print("BLOCK 8.5D-AUDIT — FINAL DIAGNOSTIC")
print("=" * 120)

print(
    "Primary responses                       :",
    10
)

print(
    "Campaigns                               :",
    len(campaigns)
)

print(
    "Hydroclimate metrics                    :",
    29
)

print(
    "Response × campaign cells               :",
    len(campaign_response)
)

print(
    "LOCO response folds                     :",
    len(folds)
)

print(
    "Hydroclimate fold checks                :",
    len(hydro_folds)
)

print(
    "Folds with unseen test stations         :",
    n_fold_unseen
)

print(
    "Test rows from unseen stations          :",
    n_rows_unseen
)

print(
    "Fraction test rows unseen station       :",
    round(fraction_unseen_global, 4)
)

print(
    "Novel test censor-limit folds           :",
    censor_limit_novel_folds
)

print(
    "Models fitted                           : NO"
)

print(
    "Memory timescale selected               : NO"
)

print(
    "Censor substitution                     : NO"
)

print(
    "LOD/2                                   : NO"
)

print(
    "Analytical subsamples collapsed         : NO"
)

print("\nOutputs:")
print(OUT_STATION_RECURRENCE)
print(OUT_CAMPAIGN_RESPONSE)
print(OUT_FOLDS)
print(OUT_CENSOR)
print(OUT_HYDRO)

print("\nAudit:")
print(AUDIT_XLSX)

print("\nManifest:")
print(MANIFEST)

print(
    "\nBLOCK 8.5D-AUDIT PASS                   :",
    BLOCK_PASS
)

if BLOCK_PASS:

    print(
        "\nBLOCK 8.5D-AUDIT COMPLETE — "
        "LOCO DATA ARCHITECTURE VERIFIED; "
        "NO PREDICTIVE MODEL FITTED."
    )

else:

    print(
        "\nBLOCK 8.5D-AUDIT NOT FROZEN — "
        "REVIEW STRUCTURAL FAILURES BEFORE LOCO."
    )

---
### Imported master block 81

`BLOCK 8.5D-V-R — LOCO PREDICTIVE ENGINE VALIDATION — SCHEMA-CORRECTED`


In [ ]:
# ================================================================================================================
# BLOCK 8.5D-V-R — LOCO PREDICTIVE ENGINE VALIDATION — SCHEMA-CORRECTED
# ================================================================================================================
#
# PURPOSE
# -------
# Validate the exact campaign-held-out predictive engine before scaling to the
# complete 10 responses × 29 hydroclimate metrics × 10 campaigns architecture.
#
# VALIDATION CASES
# ----------------
# V1 DIRECT:
#   FIELD water temperature × Tair 7 d | hold C10
#
# V2 CENSOR-AWARE:
#   Al × precipitation 14 d | hold C09
#
# V3 ALL-CENSORED TEST CAMPAIGN:
#   Fe × root-zone SM 60 d | hold C06
#
# V4 UNSEEN-STATION TEST:
#   Na × root-zone SM 60 d | hold C10
#
# FROZEN RULES
# ------------
# - response transformations inherited EXACTLY from BLOCK 8.4B
# - primary_transformation is authoritative
# - statistical_pathway is authoritative
# - hydroclimate was NOT used to choose transformations
# - one whole campaign held out
# - TRAIN-only standardization
# - explicit left-censor likelihood
# - sample-specific censor limits
# - no LOD/2
# - no censor substitution
# - no analytical-subsample collapse
# - seen station: use posterior trained station effect
# - unseen station: mean random effect = 0, predictive variance includes sigma_station²
# - censor-aware predictive scoring
# - NO memory-timescale selection
#
# ================================================================================================================

from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

from scipy.special import log_ndtr

warnings.filterwarnings("ignore", category=FutureWarning)

print("=" * 120)
print("BLOCK 8.5D-V-R — LOCO PREDICTIVE ENGINE VALIDATION")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

BASE = Path("/content/Huanta_JoH")

DATA_FILE = (
    BASE / "03_MASTER" /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

ARCH_FILE = (
    BASE / "03_MASTER" /
    "MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"
)

FOLD_AUDIT_FILE = (
    BASE / "03_MASTER" /
    "LOCO_FOLD_READINESS_v1.csv"
)

OUT_RESULTS = (
    BASE / "03_MASTER" /
    "LOCO_ENGINE_VALIDATION_v1.csv"
)

OUT_PREDICTIONS = (
    BASE / "03_MASTER" /
    "LOCO_ENGINE_VALIDATION_PREDICTIONS_v1.csv"
)

AUDIT_FILE = (
    BASE / "01_AUDIT" /
    "BLOCK85D_V_R_loco_predictive_engine_validation.xlsx"
)

MANIFEST_FILE = (
    BASE / "01_AUDIT" /
    "BLOCK85D_V_R_loco_predictive_engine_validation_manifest.json"
)

for f in [
    DATA_FILE,
    ARCH_FILE,
    FOLD_AUDIT_FILE
]:
    assert f.exists(), f"STOP — missing required file: {f}"

# ================================================================================================================
# 2. LOAD
# ================================================================================================================

df = pd.read_csv(DATA_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE, low_memory=False)
fold_audit = pd.read_csv(FOLD_AUDIT_FILE, low_memory=False)

print("\nINPUT")
print("-" * 120)

print("Memory dataset :", df.shape)
print("Architecture   :", arch.shape)
print("Fold audit     :", fold_audit.shape)
print("PyMC           :", pm.__version__)
print("ArviZ          :", az.__version__)

# ================================================================================================================
# 3. EXACT SCHEMA GATE
# ================================================================================================================

REQUIRED_ARCH_COLUMNS = [
    "response_key",
    "primary_transformation",
    "statistical_pathway",
    "censoring_policy",
    "hydroclimate_used_for_transformation",
    "window_used_for_transformation",
    "freeze_status",
]

missing_arch = [
    c for c in REQUIRED_ARCH_COLUMNS
    if c not in arch.columns
]

assert not missing_arch, (
    f"STOP — architecture columns missing: {missing_arch}"
)

assert len(arch) == 10, (
    f"STOP — expected 10 frozen responses, got {len(arch)}"
)

# ----------------------------------------------------------------
# Verify response transformations were frozen independently of hydro
# ----------------------------------------------------------------

hydro_transform_flag = (
    arch["hydroclimate_used_for_transformation"]
    .fillna(False)
    .astype(bool)
)

window_transform_flag = (
    arch["window_used_for_transformation"]
    .fillna(False)
    .astype(bool)
)

assert not hydro_transform_flag.any(), (
    "STOP — hydroclimate was used for at least one transformation."
)

assert not window_transform_flag.any(), (
    "STOP — hydroclimate window was used for at least one transformation."
)

valid_transformations = {
    "ORIGINAL",
    "LOG10"
}

observed_transformations = set(
    arch["primary_transformation"]
    .astype(str)
    .str.upper()
    .unique()
)

assert observed_transformations.issubset(
    valid_transformations
), (
    "STOP — unsupported frozen transformation(s): "
    f"{observed_transformations - valid_transformations}"
)

valid_pathways = {
    "DIRECT_CONTINUOUS",
    "CENSOR_AWARE_CONTINUOUS"
}

observed_pathways = set(
    arch["statistical_pathway"]
    .astype(str)
    .str.upper()
    .unique()
)

assert observed_pathways.issubset(
    valid_pathways
), (
    "STOP — unsupported statistical pathway(s): "
    f"{observed_pathways - valid_pathways}"
)

print("\nARCHITECTURE GATE")
print("-" * 120)

print("Responses                         :", len(arch))
print("Transformations                   :", sorted(observed_transformations))
print("Statistical pathways              :", sorted(observed_pathways))
print("Hydro used for transformation     :", hydro_transform_flag.any())
print("Window used for transformation    :", window_transform_flag.any())
print("Architecture schema               : PASS")

# ================================================================================================================
# 4. ROBUST STATION IDENTITY
# ================================================================================================================

def clean_identifier(series):

    s = series.copy()

    s = s.where(
        s.notna(),
        np.nan
    )

    s = s.map(
        lambda x:
        str(x).strip()
        if pd.notna(x)
        else np.nan
    )

    bad = {
        "",
        "nan",
        "none",
        "null",
        "na",
        "<na>"
    }

    return s.map(
        lambda x:
        np.nan
        if pd.isna(x)
        or str(x).strip().lower() in bad
        else str(x).strip()
    )


physical = clean_identifier(
    df["physical_station_id_v1"]
)

secondary = clean_identifier(
    df["station_id"]
)

df["_station_model_id"] = physical.copy()

fallback = (
    df["_station_model_id"].isna()
    &
    secondary.notna()
)

df.loc[
    fallback,
    "_station_model_id"
] = secondary.loc[fallback]


def unresolved_label(row):

    for col in [
        "field_observation_id",
        "sample_id",
        "analytical_sample_id"
    ]:

        if col in row.index:

            value = row[col]

            if pd.notna(value):

                value = str(value).strip()

                if value.lower() not in {
                    "",
                    "nan",
                    "none",
                    "null",
                    "na",
                    "<na>"
                }:

                    return (
                        f"UNRESOLVED::{col}::{value}"
                    )

    return (
        f"UNRESOLVED::ROW::{row.name}"
    )


missing_station = (
    df["_station_model_id"].isna()
)

if missing_station.any():

    df.loc[
        missing_station,
        "_station_model_id"
    ] = (
        df.loc[
            missing_station
        ]
        .apply(
            unresolved_label,
            axis=1
        )
    )

literal_nan = (
    df["_station_model_id"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("nan")
    .sum()
)

assert literal_nan == 0

print("\nSTATION IDENTITY")
print("-" * 120)

print("Physical IDs          :", physical.nunique(dropna=True))
print("Fallback rows         :", int(fallback.sum()))
print("Both IDs missing      :", int(missing_station.sum()))
print("Model grouping IDs    :", df["_station_model_id"].nunique())
print('Literal "nan" IDs     :', int(literal_nan))

# ================================================================================================================
# 5. ARCHITECTURE LOOKUP
# ================================================================================================================

def architecture_for(response_key):

    row = arch.loc[
        arch["response_key"].eq(response_key)
    ]

    assert len(row) == 1, (
        f"Architecture lookup failed for {response_key}"
    )

    return row.iloc[0]

# ================================================================================================================
# 6. FROZEN TRANSFORMATION
# ================================================================================================================

def transform_values(values, transformation):

    values = np.asarray(
        values,
        dtype=float
    )

    transformation = (
        str(transformation)
        .strip()
        .upper()
    )

    if transformation == "ORIGINAL":
        return values

    if transformation == "LOG10":

        finite = np.isfinite(values)

        if np.any(
            finite
            &
            (values <= 0)
        ):
            raise ValueError(
                "LOG10 requested with non-positive value."
            )

        return np.log10(values)

    raise ValueError(
        f"Unsupported frozen transformation: {transformation}"
    )

# ================================================================================================================
# 7. NUMERIC UTILITIES
# ================================================================================================================

def normal_logpdf(y, mu, sigma):

    return (
        -0.5 * np.log(2.0 * np.pi)
        - np.log(sigma)
        - 0.5 * ((y - mu) / sigma) ** 2
    )


def logmeanexp(x):

    x = np.asarray(
        x,
        dtype=float
    )

    m = np.max(x)

    return float(
        m
        +
        np.log(
            np.mean(
                np.exp(x - m)
            )
        )
    )

# ================================================================================================================
# 8. VALIDATION CASES
# ================================================================================================================

CASES = [

    {
        "case_id":
            "V1_DIRECT_TEMP_TAIR7_C10",

        "response_key":
            "FIELD::Water temperature::degC",

        "hydro_var":
            "tair_mean_7d",

        "hydro_family":
            "air_temperature",

        "window_days":
            7,

        "held_out_campaign":
            "C10",
    },

    {
        "case_id":
            "V2_CENSORED_AL_P14_C09",

        "response_key":
            "METALS::Aluminio::mg/L",

        "hydro_var":
            "ppt_sum_14d",

        "hydro_family":
            "precipitation",

        "window_days":
            14,

        "held_out_campaign":
            "C09",
    },

    {
        "case_id":
            "V3_FE_ROOT60_C06_ALL_CENSORED",

        "response_key":
            "METALS::Hierro::mg/L",

        "hydro_var":
            "sm_rootzone_mean_60d",

        "hydro_family":
            "rootzone_soil_moisture",

        "window_days":
            60,

        "held_out_campaign":
            "C06",
    },

    {
        "case_id":
            "V4_NA_ROOT60_C10_UNSEEN_STATIONS",

        "response_key":
            "AC::Na::mg/L",

        "hydro_var":
            "sm_rootzone_mean_60d",

        "hydro_family":
            "rootzone_soil_moisture",

        "window_days":
            60,

        "held_out_campaign":
            "C10",
    },
]

# ================================================================================================================
# 9. FIT ONE CAMPAIGN-HELD-OUT MODEL
# ================================================================================================================

def fit_loco_case(case):

    response_key = case["response_key"]
    hydro_var = case["hydro_var"]
    held_out = case["held_out_campaign"]

    print("\n" + "=" * 120)
    print(case["case_id"])
    print("=" * 120)

    # ------------------------------------------------------------------------------------------------
    # Frozen response architecture
    # ------------------------------------------------------------------------------------------------

    a = architecture_for(
        response_key
    )

    transformation = (
        str(
            a["primary_transformation"]
        )
        .strip()
        .upper()
    )

    pathway = (
        str(
            a["statistical_pathway"]
        )
        .strip()
        .upper()
    )

    censoring_policy = (
        str(
            a["censoring_policy"]
        )
        .strip()
    )

    censored_pathway = (
        pathway ==
        "CENSOR_AWARE_CONTINUOUS"
    )

    # ------------------------------------------------------------------------------------------------
    # Response data
    # ------------------------------------------------------------------------------------------------

    d = (
        df.loc[
            df["response_key"].eq(
                response_key
            )
        ]
        .copy()
        .reset_index(drop=False)
        .rename(
            columns={
                "index":
                    "_source_row_index"
            }
        )
    )

    assert hydro_var in d.columns, (
        f"Hydro variable missing: {hydro_var}"
    )

    # ------------------------------------------------------------------------------------------------
    # Exact campaign split
    # ------------------------------------------------------------------------------------------------

    train = (
        d.loc[
            ~d["campaign_id"]
            .astype(str)
            .eq(held_out)
        ]
        .copy()
        .reset_index(drop=True)
    )

    test = (
        d.loc[
            d["campaign_id"]
            .astype(str)
            .eq(held_out)
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(train) > 0
    assert len(test) > 0

    assert (
        train["campaign_id"]
        .astype(str)
        .nunique()
        == 9
    )

    assert (
        test["campaign_id"]
        .astype(str)
        .nunique()
        == 1
    )

    assert (
        test["campaign_id"]
        .astype(str)
        .iloc[0]
        == held_out
    )

    # ------------------------------------------------------------------------------------------------
    # Response and hydro numeric values
    # ------------------------------------------------------------------------------------------------

    train_y_raw = pd.to_numeric(
        train["response_value_numeric"],
        errors="coerce"
    ).to_numpy(float)

    test_y_raw = pd.to_numeric(
        test["response_value_numeric"],
        errors="coerce"
    ).to_numpy(float)

    train_x_raw = pd.to_numeric(
        train[hydro_var],
        errors="coerce"
    ).to_numpy(float)

    test_x_raw = pd.to_numeric(
        test[hydro_var],
        errors="coerce"
    ).to_numpy(float)

    assert np.isfinite(train_x_raw).all()
    assert np.isfinite(test_x_raw).all()

    # ------------------------------------------------------------------------------------------------
    # Censoring flags and limits
    # ------------------------------------------------------------------------------------------------

    train_cens = (
        train["response_censored"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    test_cens = (
        test["response_censored"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    train_detect = ~train_cens
    test_detect = ~test_cens

    train_limit_raw = pd.to_numeric(
        train["response_censor_limit"],
        errors="coerce"
    ).to_numpy(float)

    test_limit_raw = pd.to_numeric(
        test["response_censor_limit"],
        errors="coerce"
    ).to_numpy(float)

    # Direct pathway must not contain censoring.
    if not censored_pathway:

        assert not train_cens.any(), (
            f"Direct pathway has censored TRAIN rows: {response_key}"
        )

        assert not test_cens.any(), (
            f"Direct pathway has censored TEST rows: {response_key}"
        )

    # Censor-aware pathway must retain valid limits where censored.
    if train_cens.any():

        assert np.isfinite(
            train_limit_raw[
                train_cens
            ]
        ).all()

    if test_cens.any():

        assert np.isfinite(
            test_limit_raw[
                test_cens
            ]
        ).all()

    # ------------------------------------------------------------------------------------------------
    # Transform response values and censor limits
    # ------------------------------------------------------------------------------------------------

    train_y_t = np.full(
        len(train),
        np.nan
    )

    train_y_t[
        train_detect
    ] = transform_values(
        train_y_raw[
            train_detect
        ],
        transformation
    )

    test_y_t = np.full(
        len(test),
        np.nan
    )

    if test_detect.any():

        test_y_t[
            test_detect
        ] = transform_values(
            test_y_raw[
                test_detect
            ],
            transformation
        )

    train_limit_t = np.full(
        len(train),
        np.nan
    )

    if train_cens.any():

        train_limit_t[
            train_cens
        ] = transform_values(
            train_limit_raw[
                train_cens
            ],
            transformation
        )

    test_limit_t = np.full(
        len(test),
        np.nan
    )

    if test_cens.any():

        test_limit_t[
            test_cens
        ] = transform_values(
            test_limit_raw[
                test_cens
            ],
            transformation
        )

    # ------------------------------------------------------------------------------------------------
    # TRAIN-ONLY standardization
    #
    # RESPONSE:
    # detected TRAIN values only.
    #
    # HYDRO:
    # TRAIN rows only.
    #
    # TEST NEVER contributes to means or SDs.
    # ------------------------------------------------------------------------------------------------

    y_train_detected = (
        train_y_t[
            train_detect
        ]
    )

    assert len(
        y_train_detected
    ) >= 2

    assert np.isfinite(
        y_train_detected
    ).all()

    y_mean = float(
        np.mean(
            y_train_detected
        )
    )

    y_sd = float(
        np.std(
            y_train_detected,
            ddof=1
        )
    )

    x_mean = float(
        np.mean(
            train_x_raw
        )
    )

    x_sd = float(
        np.std(
            train_x_raw,
            ddof=1
        )
    )

    assert (
        np.isfinite(y_sd)
        and
        y_sd > 0
    )

    assert (
        np.isfinite(x_sd)
        and
        x_sd > 0
    )

    x_train_z = (
        train_x_raw
        -
        x_mean
    ) / x_sd

    x_test_z = (
        test_x_raw
        -
        x_mean
    ) / x_sd

    y_train_z = np.full(
        len(train),
        np.nan
    )

    y_train_z[
        train_detect
    ] = (
        train_y_t[
            train_detect
        ]
        -
        y_mean
    ) / y_sd

    y_test_z = np.full(
        len(test),
        np.nan
    )

    if test_detect.any():

        y_test_z[
            test_detect
        ] = (
            test_y_t[
                test_detect
            ]
            -
            y_mean
        ) / y_sd

    train_limit_z = np.full(
        len(train),
        np.nan
    )

    if train_cens.any():

        train_limit_z[
            train_cens
        ] = (
            train_limit_t[
                train_cens
            ]
            -
            y_mean
        ) / y_sd

    test_limit_z = np.full(
        len(test),
        np.nan
    )

    if test_cens.any():

        test_limit_z[
            test_cens
        ] = (
            test_limit_t[
                test_cens
            ]
            -
            y_mean
        ) / y_sd

    # ------------------------------------------------------------------------------------------------
    # TRAIN-only station encoding
    # ------------------------------------------------------------------------------------------------

    train_station_strings = (
        train[
            "_station_model_id"
        ]
        .astype(str)
        .to_numpy()
    )

    test_station_strings = (
        test[
            "_station_model_id"
        ]
        .astype(str)
        .to_numpy()
    )

    train_station_levels = pd.Index(
        pd.unique(
            train_station_strings
        )
    )

    train_station_lookup = {
        station: idx
        for idx, station
        in enumerate(
            train_station_levels
        )
    }

    train_station_idx = np.array(
        [
            train_station_lookup[
                station
            ]
            for station
            in train_station_strings
        ],
        dtype=int
    )

    test_seen = np.array(
        [
            station
            in train_station_lookup
            for station
            in test_station_strings
        ],
        dtype=bool
    )

    # ------------------------------------------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------------------------------------------

    coords = {
        "station":
            train_station_levels.tolist()
    }

    with pm.Model(
        coords=coords
    ) as model:

        alpha = pm.Normal(
            "alpha",
            mu=0.0,
            sigma=1.0
        )

        beta = pm.Normal(
            "beta_hydro",
            mu=0.0,
            sigma=1.0
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1.0
        )

        station_offset = pm.Normal(
            "station_offset",
            mu=0.0,
            sigma=1.0,
            dims="station"
        )

        station_effect = pm.Deterministic(
            "station_effect",
            station_offset
            * sigma_station,
            dims="station"
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1.0
        )

        mu = (
            alpha
            +
            beta
            * x_train_z
            +
            station_effect[
                train_station_idx
            ]
        )

        # --------------------------------------------------------------------------------------------
        # Detected observations
        # --------------------------------------------------------------------------------------------

        if train_detect.any():

            pm.Normal(
                "y_detected",
                mu=mu[
                    train_detect
                ],
                sigma=sigma,
                observed=y_train_z[
                    train_detect
                ]
            )

        # --------------------------------------------------------------------------------------------
        # Explicit LEFT censoring
        #
        # log P(Y <= L)
        # --------------------------------------------------------------------------------------------

        if train_cens.any():

            cens_dist = pm.Normal.dist(
                mu=mu[
                    train_cens
                ],
                sigma=sigma
            )

            logcdf = pm.logcdf(
                cens_dist,
                train_limit_z[
                    train_cens
                ]
            )

            pm.Potential(
                "left_censor_loglik",
                pm.math.sum(
                    logcdf
                )
            )

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.97,
            random_seed=20260915,
            progressbar=False,
            return_inferencedata=True
        )

    # ============================================================================================================
    # CONVERGENCE
    # ============================================================================================================

    summary = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_hydro",
            "sigma_station",
            "sigma"
        ],
        round_to=None
    )

    max_rhat = float(
        summary["r_hat"].max()
    )

    min_ess = float(
        summary["ess_bulk"].min()
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ]
        .values
        .sum()
    )

    convergence_pass = bool(
        divergences == 0
        and
        max_rhat <= 1.05
        and
        min_ess >= 200
    )

    # ============================================================================================================
    # POSTERIOR ARRAYS
    # ============================================================================================================

    posterior = idata.posterior

    alpha_draw = (
        posterior["alpha"]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    beta_draw = (
        posterior["beta_hydro"]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    sigma_draw = (
        posterior["sigma"]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    sigma_station_draw = (
        posterior["sigma_station"]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    station_effect_draw = (
        posterior[
            "station_effect"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .transpose(
            "station",
            "sample"
        )
        .values
    )

    # ============================================================================================================
    # TEST PREDICTION
    # ============================================================================================================

    pred_rows = []

    total_lpd = 0.0

    detected_sq_errors = []
    detected_abs_errors = []

    n_seen_rows = 0
    n_unseen_rows = 0

    for j in range(
        len(test)
    ):

        station = (
            test_station_strings[j]
        )

        seen = bool(
            test_seen[j]
        )

        base_mu = (
            alpha_draw
            +
            beta_draw
            * x_test_z[j]
        )

        # --------------------------------------------------------------------------------------------
        # Seen station
        # --------------------------------------------------------------------------------------------

        if seen:

            station_idx = (
                train_station_lookup[
                    station
                ]
            )

            mu_draw = (
                base_mu
                +
                station_effect_draw[
                    station_idx,
                    :
                ]
            )

            pred_sd_draw = (
                sigma_draw
            )

            prediction_station_policy = (
                "SEEN_STATION_POSTERIOR_EFFECT"
            )

            n_seen_rows += 1

        # --------------------------------------------------------------------------------------------
        # Unseen station
        #
        # E[b_new] = 0
        #
        # Marginal predictive variance:
        #
        # sigma_pred² = sigma² + sigma_station²
        # --------------------------------------------------------------------------------------------

        else:

            mu_draw = (
                base_mu
            )

            pred_sd_draw = np.sqrt(
                sigma_draw**2
                +
                sigma_station_draw**2
            )

            prediction_station_policy = (
                "UNSEEN_STATION_POPULATION_MARGINAL"
            )

            n_unseen_rows += 1

        # --------------------------------------------------------------------------------------------
        # Censored TEST observation
        # --------------------------------------------------------------------------------------------

        if test_cens[j]:

            limit_z = (
                test_limit_z[j]
            )

            z_draw = (
                limit_z
                -
                mu_draw
            ) / pred_sd_draw

            logprob_draw = (
                log_ndtr(
                    z_draw
                )
            )

            row_lpd = (
                logmeanexp(
                    logprob_draw
                )
            )

            observed_type = (
                "LEFT_CENSORED"
            )

            y_obs_z = np.nan

            pred_mean_z = float(
                np.mean(
                    mu_draw
                )
            )

            pred_mean_original_scale = np.nan

        # --------------------------------------------------------------------------------------------
        # Detected TEST observation
        # --------------------------------------------------------------------------------------------

        else:

            y_obs_z = float(
                y_test_z[j]
            )

            logpdf_draw = (
                normal_logpdf(
                    y_obs_z,
                    mu_draw,
                    pred_sd_draw
                )
            )

            row_lpd = (
                logmeanexp(
                    logpdf_draw
                )
            )

            observed_type = (
                "DETECTED"
            )

            pred_mean_z = float(
                np.mean(
                    mu_draw
                )
            )

            detected_sq_errors.append(
                (
                    y_obs_z
                    -
                    pred_mean_z
                ) ** 2
            )

            detected_abs_errors.append(
                abs(
                    y_obs_z
                    -
                    pred_mean_z
                )
            )

            pred_transformed = (
                pred_mean_z
                * y_sd
                +
                y_mean
            )

            if transformation == "ORIGINAL":

                pred_mean_original_scale = float(
                    pred_transformed
                )

            elif transformation == "LOG10":

                # NOTE:
                # This is the back-transform of the posterior mean
                # on transformed scale, NOT the arithmetic expectation
                # of a lognormal predictive distribution.
                pred_mean_original_scale = float(
                    10.0 **
                    pred_transformed
                )

            else:

                pred_mean_original_scale = np.nan

        total_lpd += (
            row_lpd
        )

        pred_rows.append({

            "case_id":
                case["case_id"],

            "response_key":
                response_key,

            "held_out_campaign":
                held_out,

            "hydroclimate_family":
                case["hydro_family"],

            "window_days":
                case["window_days"],

            "hydroclimate_variable":
                hydro_var,

            "source_row_index":
                test.loc[
                    j,
                    "_source_row_index"
                ],

            "sample_id":
                test.loc[
                    j,
                    "sample_id"
                ],

            "station_model_id":
                station,

            "station_seen_in_train":
                seen,

            "station_prediction_policy":
                prediction_station_policy,

            "observed_type":
                observed_type,

            "observed_value_raw":
                (
                    test_y_raw[j]
                    if np.isfinite(
                        test_y_raw[j]
                    )
                    else np.nan
                ),

            "censor_limit_raw":
                (
                    test_limit_raw[j]
                    if test_cens[j]
                    else np.nan
                ),

            "hydro_raw":
                test_x_raw[j],

            "hydro_z_train_scaled":
                x_test_z[j],

            "observed_response_z":
                y_obs_z,

            "posterior_predictive_mean_z":
                pred_mean_z,

            "posterior_predictive_mean_original_scale":
                pred_mean_original_scale,

            "row_log_predictive_density":
                row_lpd,
        })

    # ============================================================================================================
    # PREDICTIVE SUMMARY
    # ============================================================================================================

    if detected_sq_errors:

        rmse_detected = float(
            np.sqrt(
                np.mean(
                    detected_sq_errors
                )
            )
        )

        mae_detected = float(
            np.mean(
                detected_abs_errors
            )
        )

    else:

        rmse_detected = np.nan
        mae_detected = np.nan

    mean_lpd = float(
        total_lpd
        /
        len(test)
    )

    beta_mean = float(
        np.mean(
            beta_draw
        )
    )

    beta_low = float(
        np.quantile(
            beta_draw,
            0.025
        )
    )

    beta_high = float(
        np.quantile(
            beta_draw,
            0.975
        )
    )

    prob_pos = float(
        np.mean(
            beta_draw > 0
        )
    )

    prob_neg = float(
        np.mean(
            beta_draw < 0
        )
    )

    # ============================================================================================================
    # PRINT
    # ============================================================================================================

    print("Response             :", response_key)
    print("Held-out campaign    :", held_out)
    print("Transformation       :", transformation)
    print("Statistical pathway  :", pathway)
    print("Censoring policy     :", censoring_policy)

    print(
        "Train / test         :",
        len(train),
        "/",
        len(test)
    )

    print(
        "Train detected/cens. :",
        int(train_detect.sum()),
        "/",
        int(train_cens.sum())
    )

    print(
        "Test detected/cens.  :",
        int(test_detect.sum()),
        "/",
        int(test_cens.sum())
    )

    print(
        "Seen/unseen test rows:",
        n_seen_rows,
        "/",
        n_unseen_rows
    )

    print(
        "Train-only x mean/sd :",
        round(x_mean, 6),
        "/",
        round(x_sd, 6)
    )

    print(
        "Train-only y mean/sd :",
        round(y_mean, 6),
        "/",
        round(y_sd, 6)
    )

    print(
        "beta                 :",
        round(beta_mean, 4),
        "[",
        round(beta_low, 4),
        ",",
        round(beta_high, 4),
        "]"
    )

    print(
        "P(beta>0) / P(beta<0):",
        round(prob_pos, 4),
        "/",
        round(prob_neg, 4)
    )

    print(
        "Mean test LPD        :",
        round(mean_lpd, 4)
    )

    print(
        "Detected RMSE (z)    :",
        (
            round(
                rmse_detected,
                4
            )
            if np.isfinite(
                rmse_detected
            )
            else "NA"
        )
    )

    print(
        "Detected MAE (z)     :",
        (
            round(
                mae_detected,
                4
            )
            if np.isfinite(
                mae_detected
            )
            else "NA"
        )
    )

    print(
        "Rhat / ESS / div     :",
        round(max_rhat, 4),
        "/",
        round(min_ess, 1),
        "/",
        divergences
    )

    print(
        "Convergence PASS     :",
        convergence_pass
    )

    # ============================================================================================================
    # RETURN
    # ============================================================================================================

    result = {

        "case_id":
            case["case_id"],

        "response_key":
            response_key,

        "held_out_campaign":
            held_out,

        "hydroclimate_family":
            case["hydro_family"],

        "window_days":
            case["window_days"],

        "hydroclimate_variable":
            hydro_var,

        "primary_transformation":
            transformation,

        "statistical_pathway":
            pathway,

        "censoring_policy":
            censoring_policy,

        "n_train":
            len(train),

        "n_test":
            len(test),

        "n_train_detected":
            int(
                train_detect.sum()
            ),

        "n_train_censored":
            int(
                train_cens.sum()
            ),

        "n_test_detected":
            int(
                test_detect.sum()
            ),

        "n_test_censored":
            int(
                test_cens.sum()
            ),

        "n_train_station_groups":
            len(
                train_station_levels
            ),

        "n_test_seen_station_rows":
            n_seen_rows,

        "n_test_unseen_station_rows":
            n_unseen_rows,

        "x_train_mean":
            x_mean,

        "x_train_sd":
            x_sd,

        "y_train_detected_mean_transformed":
            y_mean,

        "y_train_detected_sd_transformed":
            y_sd,

        "beta_mean":
            beta_mean,

        "beta_hdi_2_5":
            beta_low,

        "beta_hdi_97_5":
            beta_high,

        "prob_beta_positive":
            prob_pos,

        "prob_beta_negative":
            prob_neg,

        "test_total_log_predictive_density":
            float(
                total_lpd
            ),

        "test_mean_log_predictive_density":
            mean_lpd,

        "detected_test_rmse_z":
            rmse_detected,

        "detected_test_mae_z":
            mae_detected,

        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess,

        "convergence_pass":
            convergence_pass,

        "preprocessing_train_only":
            True,

        "new_station_population_prediction":
            True,

        "new_station_uncertainty_integrated":
            True,

        "censor_aware_test_scoring":
            True,

        "memory_timescale_selected":
            False,
    }

    return (
        result,
        pd.DataFrame(
            pred_rows
        )
    )

# ================================================================================================================
# 10. RUN VALIDATION CASES
# ================================================================================================================

results = []
predictions = []

for case in CASES:

    result, pred = fit_loco_case(
        case
    )

    results.append(
        result
    )

    predictions.append(
        pred
    )

results_df = pd.DataFrame(
    results
)

predictions_df = pd.concat(
    predictions,
    ignore_index=True
)

# ================================================================================================================
# 11. CASE-SPECIFIC VALIDATION
# ================================================================================================================

print("\n" + "=" * 120)
print("CASE-SPECIFIC VALIDATION")
print("=" * 120)

v1 = (
    results_df.loc[
        results_df[
            "case_id"
        ].eq(
            "V1_DIRECT_TEMP_TAIR7_C10"
        )
    ]
    .iloc[0]
)

v1_pass = bool(
    v1["n_test_censored"] == 0
    and
    np.isfinite(
        v1[
            "detected_test_rmse_z"
        ]
    )
)

v2 = (
    results_df.loc[
        results_df[
            "case_id"
        ].eq(
            "V2_CENSORED_AL_P14_C09"
        )
    ]
    .iloc[0]
)

v2_pass = bool(
    v2["n_train_censored"] > 0
    and
    v2["n_test_censored"] > 0
    and
    np.isfinite(
        v2[
            "test_mean_log_predictive_density"
        ]
    )
)

v3 = (
    results_df.loc[
        results_df[
            "case_id"
        ].eq(
            "V3_FE_ROOT60_C06_ALL_CENSORED"
        )
    ]
    .iloc[0]
)

v3_pass = bool(
    v3["n_test"] > 0
    and
    v3["n_test_detected"] == 0
    and
    v3["n_test_censored"] == v3["n_test"]
    and
    np.isfinite(
        v3[
            "test_mean_log_predictive_density"
        ]
    )
    and
    pd.isna(
        v3[
            "detected_test_rmse_z"
        ]
    )
)

v4 = (
    results_df.loc[
        results_df[
            "case_id"
        ].eq(
            "V4_NA_ROOT60_C10_UNSEEN_STATIONS"
        )
    ]
    .iloc[0]
)

v4_pass = bool(
    v4[
        "n_test_unseen_station_rows"
    ] > 0
    and
    bool(
        v4[
            "new_station_uncertainty_integrated"
        ]
    )
)

case_checks = pd.DataFrame(
    [
        [
            "V1_direct_response",
            v1_pass
        ],
        [
            "V2_censored_response",
            v2_pass
        ],
        [
            "V3_all_censored_test_campaign",
            v3_pass
        ],
        [
            "V4_unseen_station_prediction",
            v4_pass
        ],
    ],
    columns=[
        "check",
        "pass"
    ]
)

print(
    case_checks.to_string(
        index=False
    )
)

# ================================================================================================================
# 12. GLOBAL VALIDATION GATE
# ================================================================================================================

global_checks = pd.DataFrame(
    [
        [
            "architecture_schema_verified",
            True
        ],

        [
            "response_transformations_hydro_independent",
            (
                not hydro_transform_flag.any()
                and
                not window_transform_flag.any()
            )
        ],

        [
            "validation_cases_4",
            len(results_df) == 4
        ],

        [
            "all_models_converged",
            bool(
                results_df[
                    "convergence_pass"
                ].all()
            )
        ],

        [
            "zero_total_divergences",
            int(
                results_df[
                    "divergences"
                ].sum()
            ) == 0
        ],

        [
            "all_preprocessing_train_only",
            bool(
                results_df[
                    "preprocessing_train_only"
                ].all()
            )
        ],

        [
            "all_censor_aware_scoring",
            bool(
                results_df[
                    "censor_aware_test_scoring"
                ].all()
            )
        ],

        [
            "new_station_uncertainty_integrated",
            bool(
                results_df[
                    "new_station_uncertainty_integrated"
                ].all()
            )
        ],

        [
            "all_row_lpd_finite",
            bool(
                np.isfinite(
                    predictions_df[
                        "row_log_predictive_density"
                    ]
                ).all()
            )
        ],

        [
            "no_memory_timescale_selected",
            bool(
                (
                    ~results_df[
                        "memory_timescale_selected"
                    ].astype(bool)
                ).all()
            )
        ],

        [
            "no_lod_half",
            True
        ],

        [
            "no_censor_substitution",
            True
        ],

        [
            "analytical_subsamples_not_collapsed",
            True
        ],
    ],
    columns=[
        "check",
        "pass"
    ]
)

gate = pd.concat(
    [
        case_checks,
        global_checks
    ],
    ignore_index=True
)

BLOCK_PASS = bool(
    gate["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 8.5D-V-R FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print(
    "\nBLOCK 8.5D-V-R PASS:",
    BLOCK_PASS
)

# ================================================================================================================
# 13. SAVE
# ================================================================================================================

results_df.to_csv(
    OUT_RESULTS,
    index=False
)

predictions_df.to_csv(
    OUT_PREDICTIONS,
    index=False
)

with pd.ExcelWriter(
    AUDIT_FILE,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    results_df.to_excel(
        writer,
        sheet_name="CASE_RESULTS",
        index=False
    )

    predictions_df.to_excel(
        writer,
        sheet_name="TEST_PREDICTIONS",
        index=False
    )

# ================================================================================================================
# 14. MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
            f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


manifest = {

    "block":
        "8.5D-V-R",

    "title":
        "LOCO predictive engine validation — schema corrected",

    "architecture_fields": {
        "transformation":
            "primary_transformation",

        "pathway":
            "statistical_pathway",

        "censoring":
            "censoring_policy"
    },

    "n_validation_cases":
        4,

    "validation_cases":
        CASES,

    "fold_unit":
        "campaign",

    "training_campaigns_per_fold":
        9,

    "held_out_campaigns_per_fold":
        1,

    "response_preprocessing":
        (
            "Frozen primary_transformation from BLOCK 8.4B; "
            "response mean and SD estimated exclusively from "
            "detected TRAIN observations."
        ),

    "hydroclimate_preprocessing":
        (
            "Mean and SD estimated exclusively from TRAIN rows."
        ),

    "station_prediction_policy":
        (
            "Seen station uses posterior station effect learned "
            "from training. Unseen station uses population mean "
            "random effect zero and marginal predictive variance "
            "sigma^2 + sigma_station^2."
        ),

    "censoring_policy":
        (
            "Explicit left-censor likelihood in training and "
            "sample-specific censor-aware predictive probability "
            "in held-out test observations."
        ),

    "lod_half":
        False,

    "censor_substitution":
        False,

    "analytical_subsamples_collapsed":
        False,

    "memory_timescale_selected":
        False,

    "block_pass":
        BLOCK_PASS,

    "outputs": {
        "results":
            str(OUT_RESULTS),

        "predictions":
            str(OUT_PREDICTIONS),

        "audit":
            str(AUDIT_FILE),
    }
}

for key, path in [

    (
        "results_sha256",
        OUT_RESULTS
    ),

    (
        "predictions_sha256",
        OUT_PREDICTIONS
    ),

    (
        "audit_sha256",
        AUDIT_FILE
    ),

]:

    if path.exists():

        manifest[key] = (
            sha256_file(
                path
            )
        )


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================================================================
# 15. FINAL DIAGNOSTIC
# ================================================================================================================

print("\n" + "=" * 120)
print("VALIDATION CASE RESULTS")
print("=" * 120)

display_cols = [

    "case_id",
    "response_key",
    "held_out_campaign",

    "primary_transformation",
    "statistical_pathway",

    "n_train",
    "n_test",

    "n_test_detected",
    "n_test_censored",

    "n_test_seen_station_rows",
    "n_test_unseen_station_rows",

    "beta_mean",
    "beta_hdi_2_5",
    "beta_hdi_97_5",

    "test_mean_log_predictive_density",
    "detected_test_rmse_z",

    "max_rhat",
    "min_ess_bulk",
    "divergences",
    "convergence_pass",
]

print(
    results_df[
        display_cols
    ].to_string(
        index=False
    )
)

print("\nOutputs:")
print(OUT_RESULTS)
print(OUT_PREDICTIONS)

print("\nAudit:")
print(AUDIT_FILE)

print("\nManifest:")
print(MANIFEST_FILE)

print(
    "\nBLOCK 8.5D-V-R PASS:",
    BLOCK_PASS
)

if BLOCK_PASS:

    print(
        "\nBLOCK 8.5D-V-R COMPLETE — "
        "CAMPAIGN-HELD-OUT PREDICTIVE ENGINE VALIDATED; "
        "NO MEMORY TIMESCALE SELECTED."
    )

else:

    print(
        "\nBLOCK 8.5D-V-R NOT FROZEN — "
        "DO NOT SCALE TO FULL LOCO."
    )

---
### Imported master block 83

`BLOCK 8.5D-FULL — RESUMABLE FULL CAMPAIGN-HELD-OUT PREDICTIVE SPECTRUM`


In [ ]:
# ================================================================================================================
# BLOCK 8.5D-FULL — RESUMABLE FULL CAMPAIGN-HELD-OUT PREDICTIVE SPECTRUM
# ================================================================================================================
#
# 10 responses × 29 hydroclimate metrics × 10 campaigns = 2,900 LOCO fits
#
# PURPOSE
# -------
# Estimate OUT-OF-CAMPAIGN predictive performance for every frozen
# response × hydroclimate-window combination.
#
# THIS BLOCK DOES NOT SELECT A MEMORY TIMESCALE.
#
# PERSISTENCE
# -----------
# Every completed fit is appended immediately to Google Drive.
# Runtime interruption is safe.
# Rerunning this block resumes from the next unfinished job.
#
# PRIMARY PREDICTIVE SCORE
# ------------------------
# Censor-aware row log predictive density (LPD).
#
# Detected:
#   log p(y_i | training)
#
# Left-censored:
#   log P(Y_i <= L_i | training)
#
# Station policy:
#   seen station   -> posterior trained station effect
#   unseen station -> E[b_new]=0 and predictive variance sigma²+sigma_station²
#
# PREPROCESSING:
#   TRAIN ONLY in every fold.
#
# NO:
#   LOD/2
#   censor substitution
#   random row splitting
#   global preprocessing
#   analytical-subsample collapse
#   memory-window selection
#
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from scipy.special import log_ndtr
import json
import hashlib
import time
import warnings
import os

warnings.filterwarnings("ignore", category=FutureWarning)

print("=" * 120)
print("BLOCK 8.5D-FULL — RESUMABLE FULL CAMPAIGN-HELD-OUT PREDICTIVE SPECTRUM")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

BASE = Path("/content/Huanta_JoH")

DATA_FILE = (
    BASE / "03_MASTER" /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

ARCH_FILE = (
    BASE / "03_MASTER" /
    "MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/FULL_LOCO"
)

CHECKPOINT_DIR = (
    DRIVE_ROOT /
    "CHECKPOINTS"
)

RESULTS_DIR = (
    DRIVE_ROOT /
    "RESULTS"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_FILE = (
    CHECKPOINT_DIR /
    "BLOCK85D_FULL_fit_checkpoint.csv"
)

PREDICTION_FILE = (
    CHECKPOINT_DIR /
    "BLOCK85D_FULL_prediction_checkpoint.csv"
)

RUN_STATE_FILE = (
    CHECKPOINT_DIR /
    "BLOCK85D_FULL_run_state.json"
)

FINAL_FOLD_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_FOLD_RESULTS_v1.csv"
)

FINAL_PRED_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_TEST_PREDICTIONS_v1.csv"
)

FINAL_CURVE_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_PREDICTIVE_CURVES_v1.csv"
)

FINAL_AUDIT_FILE = (
    RESULTS_DIR /
    "BLOCK85D_FULL_loco_predictive_spectrum_audit.xlsx"
)

FINAL_MANIFEST_FILE = (
    RESULTS_DIR /
    "BLOCK85D_FULL_loco_predictive_spectrum_manifest.json"
)

assert DATA_FILE.exists()
assert ARCH_FILE.exists()

# ================================================================================================================
# 2. LOAD
# ================================================================================================================

df = pd.read_csv(
    DATA_FILE,
    low_memory=False
)

arch = pd.read_csv(
    ARCH_FILE,
    low_memory=False
)

print("\nINPUT")
print("-" * 120)

print("Memory dataset :", df.shape)
print("Architecture   :", arch.shape)
print("PyMC           :", pm.__version__)
print("ArviZ          :", az.__version__)

# ================================================================================================================
# 3. FROZEN RESPONSE PANEL
# ================================================================================================================

PRIMARY_RESPONSES = [

    "AC::pH_SOL::NO_UNIT",
    "AC::Na::mg/L",
    "AC::C.E._SOL::dS/m",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "AC::NH4::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
    "AC::SO4_ppm::mg/L",
    "FIELD::Water temperature::degC",
]

assert set(
    PRIMARY_RESPONSES
) == set(
    arch["response_key"]
)

# ================================================================================================================
# 4. FROZEN HYDROCLIMATE BANK
# ================================================================================================================

HYDRO_BANK = []

for w in [1, 3, 7, 14, 30, 60, 90]:

    HYDRO_BANK.append({
        "hydro_family": "precipitation",
        "window_days": w,
        "hydro_var": f"ppt_sum_{w}d",
    })

for w in [1, 7, 14, 30, 60, 90]:

    HYDRO_BANK.append({
        "hydro_family": "surface_soil_moisture",
        "window_days": w,
        "hydro_var": f"sm_surface_mean_{w}d",
    })

for w in [1, 7, 14, 30, 60, 90]:

    HYDRO_BANK.append({
        "hydro_family": "rootzone_soil_moisture",
        "window_days": w,
        "hydro_var": f"sm_rootzone_mean_{w}d",
    })

for w in [7, 14, 30, 60, 90]:

    HYDRO_BANK.append({
        "hydro_family": "air_temperature",
        "window_days": w,
        "hydro_var": f"tair_mean_{w}d",
    })

for w in [7, 14, 30, 60, 90]:

    HYDRO_BANK.append({
        "hydro_family": "pet",
        "window_days": w,
        "hydro_var": f"pet_sum_{w}d",
    })

assert len(HYDRO_BANK) == 29

for h in HYDRO_BANK:
    assert h["hydro_var"] in df.columns, h

CAMPAIGNS = [
    "C01", "C02", "C03", "C04", "C05",
    "C06", "C07", "C08", "C09", "C10"
]

# ================================================================================================================
# 5. EXACT JOB TABLE
# ================================================================================================================

jobs = []

job_counter = 0

for response_key in PRIMARY_RESPONSES:

    for hydro in HYDRO_BANK:

        for campaign in CAMPAIGNS:

            job_counter += 1

            jobs.append({

                "job_id":
                    f"J{job_counter:04d}",

                "response_key":
                    response_key,

                "hydro_family":
                    hydro[
                        "hydro_family"
                    ],

                "window_days":
                    hydro[
                        "window_days"
                    ],

                "hydro_var":
                    hydro[
                        "hydro_var"
                    ],

                "held_out_campaign":
                    campaign,
            })

jobs = pd.DataFrame(
    jobs
)

assert len(jobs) == 2900

assert (
    jobs[
        [
            "response_key",
            "hydro_var",
            "held_out_campaign"
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 2900
)

print("\nJOB ARCHITECTURE")
print("-" * 120)

print("Responses         :", len(PRIMARY_RESPONSES))
print("Hydro metrics     :", len(HYDRO_BANK))
print("Campaigns         :", len(CAMPAIGNS))
print("Planned LOCO fits :", len(jobs))

# ================================================================================================================
# 6. ARCHITECTURE GATE
# ================================================================================================================

required_arch = [
    "response_key",
    "primary_transformation",
    "statistical_pathway",
    "censoring_policy",
    "hydroclimate_used_for_transformation",
    "window_used_for_transformation",
]

missing = [
    c for c in required_arch
    if c not in arch.columns
]

assert not missing

assert not (
    arch[
        "hydroclimate_used_for_transformation"
    ]
    .fillna(False)
    .astype(bool)
    .any()
)

assert not (
    arch[
        "window_used_for_transformation"
    ]
    .fillna(False)
    .astype(bool)
    .any()
)

# ================================================================================================================
# 7. STATION IDENTITY — SAME POLICY AS 8.5D-V-R
# ================================================================================================================

def clean_identifier(series):

    s = series.where(
        series.notna(),
        np.nan
    )

    s = s.map(
        lambda x:
        str(x).strip()
        if pd.notna(x)
        else np.nan
    )

    bad = {
        "",
        "nan",
        "none",
        "null",
        "na",
        "<na>"
    }

    return s.map(
        lambda x:
        np.nan
        if pd.isna(x)
        or str(x).lower() in bad
        else str(x)
    )


physical = clean_identifier(
    df[
        "physical_station_id_v1"
    ]
)

secondary = clean_identifier(
    df[
        "station_id"
    ]
)

df[
    "_station_model_id"
] = physical.copy()

fallback = (
    df[
        "_station_model_id"
    ].isna()
    &
    secondary.notna()
)

df.loc[
    fallback,
    "_station_model_id"
] = secondary.loc[
    fallback
]


def unresolved_label(row):

    for col in [
        "field_observation_id",
        "sample_id",
        "analytical_sample_id"
    ]:

        if col in row.index:

            value = row[col]

            if pd.notna(value):

                value = str(
                    value
                ).strip()

                if value.lower() not in {
                    "",
                    "nan",
                    "none",
                    "null",
                    "na",
                    "<na>"
                }:

                    return (
                        f"UNRESOLVED::{col}::{value}"
                    )

    return (
        f"UNRESOLVED::ROW::{row.name}"
    )


missing_station = (
    df[
        "_station_model_id"
    ].isna()
)

if missing_station.any():

    df.loc[
        missing_station,
        "_station_model_id"
    ] = (
        df.loc[
            missing_station
        ]
        .apply(
            unresolved_label,
            axis=1
        )
    )

assert (
    df[
        "_station_model_id"
    ]
    .nunique()
    == 43
)

# ================================================================================================================
# 8. HELPERS
# ================================================================================================================

def architecture_for(
    response_key
):

    x = arch.loc[
        arch[
            "response_key"
        ].eq(
            response_key
        )
    ]

    assert len(x) == 1

    return x.iloc[0]


def transform_values(
    values,
    transformation
):

    values = np.asarray(
        values,
        dtype=float
    )

    transformation = (
        str(
            transformation
        )
        .strip()
        .upper()
    )

    if transformation == "ORIGINAL":

        return values

    if transformation == "LOG10":

        finite = np.isfinite(
            values
        )

        if np.any(
            finite
            &
            (
                values <= 0
            )
        ):

            raise ValueError(
                "LOG10 with non-positive value."
            )

        return np.log10(
            values
        )

    raise ValueError(
        transformation
    )


def normal_logpdf(
    y,
    mu,
    sigma
):

    return (
        -0.5
        * np.log(
            2.0
            * np.pi
        )
        -
        np.log(
            sigma
        )
        -
        0.5
        * (
            (
                y
                -
                mu
            )
            /
            sigma
        ) ** 2
    )


def logmeanexp(
    x
):

    x = np.asarray(
        x,
        dtype=float
    )

    m = np.max(
        x
    )

    return float(
        m
        +
        np.log(
            np.mean(
                np.exp(
                    x
                    -
                    m
                )
            )
        )
    )

# ================================================================================================================
# 9. CHECKPOINT HELPERS
# ================================================================================================================

CHECKPOINT_COLUMNS = [

    "job_id",
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "held_out_campaign",

    "primary_transformation",
    "statistical_pathway",

    "n_train",
    "n_test",
    "n_train_detected",
    "n_train_censored",
    "n_test_detected",
    "n_test_censored",

    "n_train_station_groups",
    "n_test_seen_station_rows",
    "n_test_unseen_station_rows",

    "beta_mean",

    "test_total_lpd",
    "test_mean_lpd",

    "detected_rmse_z",
    "detected_mae_z",

    "sampling_stage",
    "draws",
    "tune",
    "target_accept",

    "divergences",
    "max_rhat",
    "min_ess_bulk",

    "convergence_pass",
    "fit_status",

    "elapsed_seconds",
]


def append_csv_row(
    path,
    row_dict,
    columns
):

    row_df = pd.DataFrame(
        [
            row_dict
        ]
    )

    row_df = row_df.reindex(
        columns=columns
    )

    exists = (
        path.exists()
        and
        path.stat().st_size > 0
    )

    row_df.to_csv(
        path,
        mode="a",
        header=not exists,
        index=False
    )


def load_completed_jobs():

    if not CHECKPOINT_FILE.exists():

        return set()

    old = pd.read_csv(
        CHECKPOINT_FILE,
        low_memory=False
    )

    if "fit_status" not in old.columns:

        return set()

    completed = old.loc[
        old[
            "fit_status"
        ].isin(
            [
                "SUCCESS_INITIAL",
                "SUCCESS_REFIT"
            ]
        ),
        "job_id"
    ]

    return set(
        completed.astype(str)
    )

# ================================================================================================================
# 10. FIT FUNCTION
# ================================================================================================================

def run_job(
    job,
    draws,
    tune,
    target_accept,
    seed
):

    start_time = time.time()

    response_key = (
        job[
            "response_key"
        ]
    )

    hydro_var = (
        job[
            "hydro_var"
        ]
    )

    held_out = (
        job[
            "held_out_campaign"
        ]
    )

    a = architecture_for(
        response_key
    )

    transformation = (
        str(
            a[
                "primary_transformation"
            ]
        )
        .strip()
        .upper()
    )

    pathway = (
        str(
            a[
                "statistical_pathway"
            ]
        )
        .strip()
        .upper()
    )

    d = (
        df.loc[
            df[
                "response_key"
            ].eq(
                response_key
            )
        ]
        .copy()
        .reset_index(
            drop=False
        )
        .rename(
            columns={
                "index":
                    "_source_row_index"
            }
        )
    )

    train = (
        d.loc[
            ~d[
                "campaign_id"
            ]
            .astype(str)
            .eq(
                held_out
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    test = (
        d.loc[
            d[
                "campaign_id"
            ]
            .astype(str)
            .eq(
                held_out
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    assert (
        train[
            "campaign_id"
        ]
        .nunique()
        == 9
    )

    assert len(
        test
    ) > 0

    # ------------------------------------------------------------------------------------------------------------
    # Raw response/hydro
    # ------------------------------------------------------------------------------------------------------------

    y_train_raw = pd.to_numeric(
        train[
            "response_value_numeric"
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    y_test_raw = pd.to_numeric(
        test[
            "response_value_numeric"
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    x_train_raw = pd.to_numeric(
        train[
            hydro_var
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    x_test_raw = pd.to_numeric(
        test[
            hydro_var
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    assert np.isfinite(
        x_train_raw
    ).all()

    assert np.isfinite(
        x_test_raw
    ).all()

    # ------------------------------------------------------------------------------------------------------------
    # Censoring
    # ------------------------------------------------------------------------------------------------------------

    train_cens = (
        train[
            "response_censored"
        ]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    test_cens = (
        test[
            "response_censored"
        ]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    train_det = (
        ~train_cens
    )

    test_det = (
        ~test_cens
    )

    limit_train_raw = pd.to_numeric(
        train[
            "response_censor_limit"
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    limit_test_raw = pd.to_numeric(
        test[
            "response_censor_limit"
        ],
        errors="coerce"
    ).to_numpy(
        float
    )

    if train_cens.any():

        assert np.isfinite(
            limit_train_raw[
                train_cens
            ]
        ).all()

    if test_cens.any():

        assert np.isfinite(
            limit_test_raw[
                test_cens
            ]
        ).all()

    # ------------------------------------------------------------------------------------------------------------
    # Transform
    # ------------------------------------------------------------------------------------------------------------

    y_train_t = np.full(
        len(train),
        np.nan
    )

    y_train_t[
        train_det
    ] = transform_values(
        y_train_raw[
            train_det
        ],
        transformation
    )

    y_test_t = np.full(
        len(test),
        np.nan
    )

    if test_det.any():

        y_test_t[
            test_det
        ] = transform_values(
            y_test_raw[
                test_det
            ],
            transformation
        )

    limit_train_t = np.full(
        len(train),
        np.nan
    )

    if train_cens.any():

        limit_train_t[
            train_cens
        ] = transform_values(
            limit_train_raw[
                train_cens
            ],
            transformation
        )

    limit_test_t = np.full(
        len(test),
        np.nan
    )

    if test_cens.any():

        limit_test_t[
            test_cens
        ] = transform_values(
            limit_test_raw[
                test_cens
            ],
            transformation
        )

    # ------------------------------------------------------------------------------------------------------------
    # TRAIN-ONLY scaling
    # ------------------------------------------------------------------------------------------------------------

    y_mean = float(
        np.mean(
            y_train_t[
                train_det
            ]
        )
    )

    y_sd = float(
        np.std(
            y_train_t[
                train_det
            ],
            ddof=1
        )
    )

    x_mean = float(
        np.mean(
            x_train_raw
        )
    )

    x_sd = float(
        np.std(
            x_train_raw,
            ddof=1
        )
    )

    assert (
        np.isfinite(
            y_sd
        )
        and
        y_sd > 0
    )

    assert (
        np.isfinite(
            x_sd
        )
        and
        x_sd > 0
    )

    x_train_z = (
        x_train_raw
        -
        x_mean
    ) / x_sd

    x_test_z = (
        x_test_raw
        -
        x_mean
    ) / x_sd

    y_train_z = np.full(
        len(train),
        np.nan
    )

    y_train_z[
        train_det
    ] = (
        y_train_t[
            train_det
        ]
        -
        y_mean
    ) / y_sd

    y_test_z = np.full(
        len(test),
        np.nan
    )

    if test_det.any():

        y_test_z[
            test_det
        ] = (
            y_test_t[
                test_det
            ]
            -
            y_mean
        ) / y_sd

    limit_train_z = np.full(
        len(train),
        np.nan
    )

    if train_cens.any():

        limit_train_z[
            train_cens
        ] = (
            limit_train_t[
                train_cens
            ]
            -
            y_mean
        ) / y_sd

    limit_test_z = np.full(
        len(test),
        np.nan
    )

    if test_cens.any():

        limit_test_z[
            test_cens
        ] = (
            limit_test_t[
                test_cens
            ]
            -
            y_mean
        ) / y_sd

    # ------------------------------------------------------------------------------------------------------------
    # TRAIN station structure
    # ------------------------------------------------------------------------------------------------------------

    train_station = (
        train[
            "_station_model_id"
        ]
        .astype(str)
        .to_numpy()
    )

    test_station = (
        test[
            "_station_model_id"
        ]
        .astype(str)
        .to_numpy()
    )

    station_levels = pd.Index(
        pd.unique(
            train_station
        )
    )

    station_lookup = {
        s: i
        for i, s
        in enumerate(
            station_levels
        )
    }

    station_idx = np.array(
        [
            station_lookup[
                s
            ]
            for s in train_station
        ],
        dtype=int
    )

    test_seen = np.array(
        [
            s in station_lookup
            for s in test_station
        ],
        dtype=bool
    )

    # ------------------------------------------------------------------------------------------------------------
    # PyMC model
    # ------------------------------------------------------------------------------------------------------------

    coords = {
        "station":
            station_levels.tolist()
    }

    with pm.Model(
        coords=coords
    ) as model:

        alpha = pm.Normal(
            "alpha",
            0,
            1
        )

        beta = pm.Normal(
            "beta_hydro",
            0,
            1
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            1
        )

        station_offset = pm.Normal(
            "station_offset",
            0,
            1,
            dims="station"
        )

        station_effect = pm.Deterministic(
            "station_effect",
            station_offset
            * sigma_station,
            dims="station"
        )

        sigma = pm.HalfNormal(
            "sigma",
            1
        )

        mu = (
            alpha
            +
            beta
            * x_train_z
            +
            station_effect[
                station_idx
            ]
        )

        if train_det.any():

            pm.Normal(
                "y_detected",
                mu=mu[
                    train_det
                ],
                sigma=sigma,
                observed=y_train_z[
                    train_det
                ]
            )

        if train_cens.any():

            dist = pm.Normal.dist(
                mu=mu[
                    train_cens
                ],
                sigma=sigma
            )

            pm.Potential(
                "left_censor_loglik",
                pm.math.sum(
                    pm.logcdf(
                        dist,
                        limit_train_z[
                            train_cens
                        ]
                    )
                )
            )

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=4,
            cores=4,
            target_accept=target_accept,
            random_seed=seed,
            progressbar=False,
            return_inferencedata=True,
            compute_convergence_checks=False
        )

    # ------------------------------------------------------------------------------------------------------------
    # Convergence
    # ------------------------------------------------------------------------------------------------------------

    summ = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_hydro",
            "sigma_station",
            "sigma"
        ],
        round_to=None
    )

    max_rhat = float(
        summ[
            "r_hat"
        ].max()
    )

    min_ess = float(
        summ[
            "ess_bulk"
        ].min()
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ]
        .values
        .sum()
    )

    convergence_pass = bool(
        divergences == 0
        and
        max_rhat <= 1.05
        and
        min_ess >= 200
    )

    # ------------------------------------------------------------------------------------------------------------
    # Posterior
    # ------------------------------------------------------------------------------------------------------------

    post = (
        idata.posterior
    )

    alpha_draw = (
        post[
            "alpha"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    beta_draw = (
        post[
            "beta_hydro"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    sigma_draw = (
        post[
            "sigma"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    sigma_station_draw = (
        post[
            "sigma_station"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .values
        .reshape(-1)
    )

    station_effect_draw = (
        post[
            "station_effect"
        ]
        .stack(
            sample=(
                "chain",
                "draw"
            )
        )
        .transpose(
            "station",
            "sample"
        )
        .values
    )

    # ------------------------------------------------------------------------------------------------------------
    # Test scoring
    # ------------------------------------------------------------------------------------------------------------

    prediction_rows = []

    total_lpd = 0.0

    sq_errors = []
    abs_errors = []

    for j in range(
        len(test)
    ):

        base_mu = (
            alpha_draw
            +
            beta_draw
            * x_test_z[j]
        )

        if test_seen[j]:

            idx = (
                station_lookup[
                    test_station[j]
                ]
            )

            mu_draw = (
                base_mu
                +
                station_effect_draw[
                    idx,
                    :
                ]
            )

            pred_sd = (
                sigma_draw
            )

            station_policy = (
                "SEEN_STATION_POSTERIOR_EFFECT"
            )

        else:

            mu_draw = (
                base_mu
            )

            pred_sd = np.sqrt(
                sigma_draw**2
                +
                sigma_station_draw**2
            )

            station_policy = (
                "UNSEEN_STATION_POPULATION_MARGINAL"
            )

        if test_cens[j]:

            z = (
                limit_test_z[j]
                -
                mu_draw
            ) / pred_sd

            row_lpd = logmeanexp(
                log_ndtr(
                    z
                )
            )

            observed_type = (
                "LEFT_CENSORED"
            )

            observed_z = np.nan

        else:

            observed_z = float(
                y_test_z[j]
            )

            row_lpd = logmeanexp(
                normal_logpdf(
                    observed_z,
                    mu_draw,
                    pred_sd
                )
            )

            pred_mean = float(
                np.mean(
                    mu_draw
                )
            )

            sq_errors.append(
                (
                    observed_z
                    -
                    pred_mean
                ) ** 2
            )

            abs_errors.append(
                abs(
                    observed_z
                    -
                    pred_mean
                )
            )

            observed_type = (
                "DETECTED"
            )

        total_lpd += (
            row_lpd
        )

        prediction_rows.append({

            "job_id":
                job[
                    "job_id"
                ],

            "response_key":
                response_key,

            "hydro_family":
                job[
                    "hydro_family"
                ],

            "window_days":
                job[
                    "window_days"
                ],

            "hydro_var":
                hydro_var,

            "held_out_campaign":
                held_out,

            "sample_id":
                test.loc[
                    j,
                    "sample_id"
                ],

            "station_model_id":
                test_station[j],

            "station_seen_in_train":
                bool(
                    test_seen[j]
                ),

            "station_prediction_policy":
                station_policy,

            "observed_type":
                observed_type,

            "row_lpd":
                row_lpd,
        })

    rmse = (
        float(
            np.sqrt(
                np.mean(
                    sq_errors
                )
            )
        )
        if sq_errors
        else np.nan
    )

    mae = (
        float(
            np.mean(
                abs_errors
            )
        )
        if abs_errors
        else np.nan
    )

    elapsed = (
        time.time()
        -
        start_time
    )

    result = {

        "job_id":
            job[
                "job_id"
            ],

        "response_key":
            response_key,

        "hydro_family":
            job[
                "hydro_family"
            ],

        "window_days":
            job[
                "window_days"
            ],

        "hydro_var":
            hydro_var,

        "held_out_campaign":
            held_out,

        "primary_transformation":
            transformation,

        "statistical_pathway":
            pathway,

        "n_train":
            len(train),

        "n_test":
            len(test),

        "n_train_detected":
            int(
                train_det.sum()
            ),

        "n_train_censored":
            int(
                train_cens.sum()
            ),

        "n_test_detected":
            int(
                test_det.sum()
            ),

        "n_test_censored":
            int(
                test_cens.sum()
            ),

        "n_train_station_groups":
            len(
                station_levels
            ),

        "n_test_seen_station_rows":
            int(
                test_seen.sum()
            ),

        "n_test_unseen_station_rows":
            int(
                (
                    ~test_seen
                ).sum()
            ),

        "beta_mean":
            float(
                np.mean(
                    beta_draw
                )
            ),

        "test_total_lpd":
            float(
                total_lpd
            ),

        "test_mean_lpd":
            float(
                total_lpd
                /
                len(test)
            ),

        "detected_rmse_z":
            rmse,

        "detected_mae_z":
            mae,

        "sampling_stage":
            "INITIAL",

        "draws":
            draws,

        "tune":
            tune,

        "target_accept":
            target_accept,

        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess,

        "convergence_pass":
            convergence_pass,

        "fit_status":
            (
                "SUCCESS_INITIAL"
                if convergence_pass
                else "NEEDS_REFIT"
            ),

        "elapsed_seconds":
            elapsed,
    }

    return (
        result,
        pd.DataFrame(
            prediction_rows
        )
    )

# ================================================================================================================
# 11. RESUME STATE
# ================================================================================================================

completed_jobs = (
    load_completed_jobs()
)

remaining_jobs = (
    jobs.loc[
        ~jobs[
            "job_id"
        ]
        .isin(
            completed_jobs
        )
    ]
    .copy()
)

print("\nRESUME STATE")
print("-" * 120)

print(
    "Completed successful fits :",
    len(
        completed_jobs
    )
)

print(
    "Remaining fits            :",
    len(
        remaining_jobs
    )
)

# ================================================================================================================
# 12. FULL RUN
# ================================================================================================================

session_start = (
    time.time()
)

processed_this_session = 0

for _, job in remaining_jobs.iterrows():

    processed_this_session += 1

    job_number = int(
        job[
            "job_id"
        ][1:]
    )

    print(
        f"\n[{job_number:04d}/2900] "
        f"{job['job_id']} | "
        f"{job['response_key']} | "
        f"{job['hydro_var']} | "
        f"hold {job['held_out_campaign']}"
    )

    # ------------------------------------------------------------------------------------------------------------
    # Initial fit
    # ------------------------------------------------------------------------------------------------------------

    try:

        result, pred = run_job(
            job,
            draws=300,
            tune=300,
            target_accept=0.92,
            seed=20260915 + job_number
        )

        # --------------------------------------------------------------------------------------------------------
        # Automatic refit
        # --------------------------------------------------------------------------------------------------------

        if not result[
            "convergence_pass"
        ]:

            print(
                "  Initial convergence gate failed -> REFIT"
            )

            result, pred = run_job(
                job,
                draws=700,
                tune=700,
                target_accept=0.97,
                seed=30260915 + job_number
            )

            result[
                "sampling_stage"
            ] = "REFIT"

            result[
                "fit_status"
            ] = (
                "SUCCESS_REFIT"
                if result[
                    "convergence_pass"
                ]
                else "FAILED_CONVERGENCE"
            )

        # --------------------------------------------------------------------------------------------------------
        # Persistent write
        # --------------------------------------------------------------------------------------------------------

        append_csv_row(
            CHECKPOINT_FILE,
            result,
            CHECKPOINT_COLUMNS
        )

        # Predictions only retained for successful fits.
        if result[
            "fit_status"
        ] in {
            "SUCCESS_INITIAL",
            "SUCCESS_REFIT"
        }:

            pred.to_csv(
                PREDICTION_FILE,
                mode="a",
                header=(
                    not PREDICTION_FILE.exists()
                    or
                    PREDICTION_FILE.stat().st_size == 0
                ),
                index=False
            )

        print(
            "  status:",
            result[
                "fit_status"
            ],
            "| LPD:",
            round(
                result[
                    "test_mean_lpd"
                ],
                4
            ),
            "| Rhat:",
            round(
                result[
                    "max_rhat"
                ],
                3
            ),
            "| ESS:",
            round(
                result[
                    "min_ess_bulk"
                ],
                1
            ),
            "| div:",
            result[
                "divergences"
            ],
            "| sec:",
            round(
                result[
                    "elapsed_seconds"
                ],
                1
            )
        )

    except Exception as e:

        failure = {

            c: np.nan
            for c in CHECKPOINT_COLUMNS
        }

        failure.update({

            "job_id":
                job[
                    "job_id"
                ],

            "response_key":
                job[
                    "response_key"
                ],

            "hydro_family":
                job[
                    "hydro_family"
                ],

            "window_days":
                job[
                    "window_days"
                ],

            "hydro_var":
                job[
                    "hydro_var"
                ],

            "held_out_campaign":
                job[
                    "held_out_campaign"
                ],

            "fit_status":
                "ERROR",

            "convergence_pass":
                False,
        })

        append_csv_row(
            CHECKPOINT_FILE,
            failure,
            CHECKPOINT_COLUMNS
        )

        print(
            "  ERROR:",
            repr(
                e
            )
        )

    # ------------------------------------------------------------------------------------------------------------
    # Persistent run state
    # ------------------------------------------------------------------------------------------------------------

    current = pd.read_csv(
        CHECKPOINT_FILE,
        low_memory=False
    )

    success_now = current[
        "fit_status"
    ].isin(
        [
            "SUCCESS_INITIAL",
            "SUCCESS_REFIT"
        ]
    ).sum()

    state = {

        "updated_at":
            datetime.now().isoformat(),

        "planned_jobs":
            2900,

        "successful_jobs":
            int(
                success_now
            ),

        "processed_rows_in_checkpoint":
            int(
                len(
                    current
                )
            ),

        "last_job_id":
            str(
                job[
                    "job_id"
                ]
            ),

        "memory_timescale_selected":
            False,
    }

    with open(
        RUN_STATE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            indent=2
        )

# ================================================================================================================
# 13. FINAL CHECKPOINT LOAD
# ================================================================================================================

fold_results = pd.read_csv(
    CHECKPOINT_FILE,
    low_memory=False
)

# If a job has more than one historical record from an interrupted/restarted attempt,
# keep the LAST successful record if available; otherwise the last record.

status_rank = {
    "ERROR": 0,
    "FAILED_CONVERGENCE": 1,
    "NEEDS_REFIT": 2,
    "SUCCESS_INITIAL": 3,
    "SUCCESS_REFIT": 4,
}

fold_results[
    "_status_rank"
] = (
    fold_results[
        "fit_status"
    ]
    .map(
        status_rank
    )
    .fillna(-1)
)

fold_results[
    "_row_order"
] = np.arange(
    len(
        fold_results
    )
)

fold_results = (
    fold_results
    .sort_values(
        [
            "job_id",
            "_status_rank",
            "_row_order"
        ]
    )
    .groupby(
        "job_id",
        as_index=False
    )
    .tail(1)
    .drop(
        columns=[
            "_status_rank",
            "_row_order"
        ]
    )
    .sort_values(
        "job_id"
    )
    .reset_index(
        drop=True
    )
)

# ================================================================================================================
# 14. SUCCESSFUL FITS
# ================================================================================================================

successful = fold_results.loc[
    fold_results[
        "fit_status"
    ].isin(
        [
            "SUCCESS_INITIAL",
            "SUCCESS_REFIT"
        ]
    )
].copy()

# ================================================================================================================
# 15. CAMPAIGN-EQUAL-WEIGHTED PREDICTIVE CURVES
# ================================================================================================================

# Every held-out campaign contributes ONE mean-LPD value.
# Therefore campaign sizes do not determine the scientific weight.

curve_rows = []

group_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
]

for keys, g in successful.groupby(
    group_cols,
    dropna=False
):

    (
        response_key,
        hydro_family,
        window_days,
        hydro_var
    ) = keys

    campaign_scores = (
        g[
            [
                "held_out_campaign",
                "test_mean_lpd"
            ]
        ]
        .dropna()
    )

    n_campaigns = (
        campaign_scores[
            "held_out_campaign"
        ]
        .nunique()
    )

    values = (
        campaign_scores[
            "test_mean_lpd"
        ]
        .to_numpy(
            float
        )
    )

    mean_lpd = (
        float(
            np.mean(
                values
            )
        )
        if len(
            values
        )
        else np.nan
    )

    sd_lpd = (
        float(
            np.std(
                values,
                ddof=1
            )
        )
        if len(
            values
        ) >= 2
        else np.nan
    )

    se_lpd = (
        float(
            sd_lpd
            /
            np.sqrt(
                len(
                    values
                )
            )
        )
        if len(
            values
        ) >= 2
        else np.nan
    )

    curve_rows.append({

        "response_key":
            response_key,

        "hydro_family":
            hydro_family,

        "window_days":
            window_days,

        "hydro_var":
            hydro_var,

        "n_loco_campaigns":
            n_campaigns,

        "mean_campaign_lpd":
            mean_lpd,

        "sd_campaign_lpd":
            sd_lpd,

        "se_campaign_lpd":
            se_lpd,

        "all_10_campaigns_present":
            n_campaigns == 10,

        "memory_timescale_selected":
            False,
    })

curves = pd.DataFrame(
    curve_rows
)

# ================================================================================================================
# 16. FINAL QA
# ================================================================================================================

n_success = len(
    successful
)

n_failed = int(
    (
        ~fold_results[
            "fit_status"
        ].isin(
            [
                "SUCCESS_INITIAL",
                "SUCCESS_REFIT"
            ]
        )
    ).sum()
)

n_refits = int(
    successful[
        "fit_status"
    ]
    .eq(
        "SUCCESS_REFIT"
    )
    .sum()
)

expected_curves = (
    10
    * 5
)

observed_curves = (
    curves[
        [
            "response_key",
            "hydro_family"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

all_2900_success = (
    n_success == 2900
)

all_curves_complete = (
    len(
        curves
    ) == 290
    and
    curves[
        "all_10_campaigns_present"
    ].all()
)

gate = pd.DataFrame(
    [

        [
            "planned_jobs_2900",
            len(jobs) == 2900
        ],

        [
            "unique_job_results_2900",
            len(fold_results) == 2900
        ],

        [
            "successful_jobs_2900",
            all_2900_success
        ],

        [
            "failed_jobs_zero",
            n_failed == 0
        ],

        [
            "predictive_window_curves_290",
            len(curves) == 290
        ],

        [
            "response_family_curves_50",
            observed_curves == expected_curves
        ],

        [
            "all_curves_have_10_campaigns",
            bool(
                curves[
                    "all_10_campaigns_present"
                ].all()
            )
            if len(curves)
            else False
        ],

        [
            "zero_successful_fit_divergences",
            int(
                successful[
                    "divergences"
                ].sum()
            ) == 0
        ],

        [
            "all_successful_rhat_pass",
            bool(
                (
                    successful[
                        "max_rhat"
                    ]
                    <= 1.05
                ).all()
            )
        ],

        [
            "all_successful_ess_pass",
            bool(
                (
                    successful[
                        "min_ess_bulk"
                    ]
                    >= 200
                ).all()
            )
        ],

        [
            "no_memory_timescale_selected",
            True
        ],

        [
            "no_lod_half",
            True
        ],

        [
            "no_censor_substitution",
            True
        ],

        [
            "train_only_preprocessing",
            True
        ],

        [
            "campaign_equal_weighting",
            True
        ],

        [
            "analytical_subsamples_not_collapsed",
            True
        ],
    ],

    columns=[
        "check",
        "pass"
    ]
)

BLOCK_PASS = bool(
    gate[
        "pass"
    ].all()
)

# ================================================================================================================
# 17. SAVE FINAL RESULTS TO DRIVE
# ================================================================================================================

fold_results.to_csv(
    FINAL_FOLD_FILE,
    index=False
)

if PREDICTION_FILE.exists():

    predictions = pd.read_csv(
        PREDICTION_FILE,
        low_memory=False
    )

    predictions.to_csv(
        FINAL_PRED_FILE,
        index=False
    )

else:

    predictions = pd.DataFrame()

curves.to_csv(
    FINAL_CURVE_FILE,
    index=False
)

with pd.ExcelWriter(
    FINAL_AUDIT_FILE,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    fold_results.to_excel(
        writer,
        sheet_name="LOCO_FOLDS",
        index=False
    )

    curves.to_excel(
        writer,
        sheet_name="PREDICTIVE_CURVES",
        index=False
    )

# ================================================================================================================
# 18. MANIFEST
# ================================================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
            f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


manifest = {

    "block":
        "8.5D-FULL",

    "planned_fits":
        2900,

    "successful_fits":
        int(
            n_success
        ),

    "failed_fits":
        int(
            n_failed
        ),

    "automatic_refits":
        int(
            n_refits
        ),

    "predictive_window_curves":
        int(
            len(
                curves
            )
        ),

    "response_family_curves":
        int(
            observed_curves
        ),

    "primary_predictive_score":
        "campaign-equal-weighted mean held-out log predictive density",

    "detected_test_scoring":
        "posterior predictive density",

    "left_censored_test_scoring":
        "posterior predictive probability Y <= observation-specific censor limit",

    "station_policy_seen":
        "posterior station effect estimated from training",

    "station_policy_unseen":
        "population random-effect mean zero; variance includes sigma_station squared",

    "preprocessing":
        "training only",

    "initial_sampling":
        {
            "chains": 4,
            "draws": 300,
            "tune": 300,
            "target_accept": 0.92,
        },

    "refit_sampling":
        {
            "chains": 4,
            "draws": 700,
            "tune": 700,
            "target_accept": 0.97,
        },

    "convergence_gate":
        {
            "divergences": 0,
            "max_rhat": 1.05,
            "min_ess_bulk": 200,
        },

    "lod_half":
        False,

    "censor_substitution":
        False,

    "memory_timescale_selected":
        False,

    "block_pass":
        BLOCK_PASS,
}

for name, path in [

    (
        "fold_results",
        FINAL_FOLD_FILE
    ),

    (
        "predictive_curves",
        FINAL_CURVE_FILE
    ),

    (
        "audit",
        FINAL_AUDIT_FILE
    ),
]:

    manifest[
        f"{name}_sha256"
    ] = sha256_file(
        path
    )

with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 19. FINAL REPORT
# ================================================================================================================

elapsed_total = (
    time.time()
    -
    session_start
)

print("\n" + "=" * 120)
print("BLOCK 8.5D-FULL FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print("\nSUMMARY")
print("-" * 120)

print(
    "Planned fits              :",
    2900
)

print(
    "Successful fits           :",
    n_success
)

print(
    "Failed fits               :",
    n_failed
)

print(
    "Automatic successful refits:",
    n_refits
)

print(
    "Predictive window curves  :",
    len(
        curves
    ),
    "/ 290"
)

print(
    "Response-family curves    :",
    observed_curves,
    "/ 50"
)

print(
    "Session elapsed minutes   :",
    round(
        elapsed_total
        /
        60,
        2
    )
)

print(
    "\nPersistent checkpoint:"
)

print(
    CHECKPOINT_FILE
)

print(
    "\nFinal fold results:"
)

print(
    FINAL_FOLD_FILE
)

print(
    "\nPredictive curves:"
)

print(
    FINAL_CURVE_FILE
)

print(
    "\nAudit:"
)

print(
    FINAL_AUDIT_FILE
)

print(
    "\nManifest:"
)

print(
    FINAL_MANIFEST_FILE
)

print(
    "\nBLOCK 8.5D-FULL PASS:",
    BLOCK_PASS
)

if BLOCK_PASS:

    print(
        "\nBLOCK 8.5D-FULL COMPLETE — "
        "FULL CAMPAIGN-HELD-OUT PREDICTIVE SPECTRUM ESTIMATED; "
        "NO MEMORY TIMESCALE SELECTED."
    )

else:

    print(
        "\nBLOCK 8.5D-FULL INCOMPLETE OR FAILED — "
        "SAFE TO RERUN; CHECKPOINT/RESUME IS ACTIVE."
    )

---
### Imported master block 87

`BLOCK 8.5D-FULL — FINAL FIT + PREDICTION INTEGRITY AUDIT`


In [ ]:
# ================================================================================================================
# BLOCK 8.5D-FULL — FINAL FIT + PREDICTION INTEGRITY AUDIT
# READ ONLY
# NO MODEL FITTING
# NO FILE MODIFICATION
# IMPORTANT: does NOT use variable name `df`
# ================================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 120)
print("BLOCK 8.5D-FULL — FINAL FIT + PREDICTION INTEGRITY AUDIT")
print("=" * 120)

ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/FULL_LOCO"
)

FIT_FILE = (
    ROOT / "CHECKPOINTS" /
    "BLOCK85D_FULL_fit_checkpoint.csv"
)

PRED_FILE = (
    ROOT / "CHECKPOINTS" /
    "BLOCK85D_FULL_prediction_checkpoint.csv"
)

assert FIT_FILE.exists(), FIT_FILE
assert PRED_FILE.exists(), PRED_FILE

# ------------------------------------------------------------------------------------------------
# 1. LOAD — NEVER USE GLOBAL NAME `df`
# ------------------------------------------------------------------------------------------------

fit_raw = pd.read_csv(
    FIT_FILE,
    low_memory=False
)

pred_raw = pd.read_csv(
    PRED_FILE,
    low_memory=False
)

print("\nRAW FILES")
print("-" * 120)
print("Fit checkpoint rows       :", len(fit_raw))
print("Fit unique job IDs        :", fit_raw["job_id"].nunique())
print("Prediction checkpoint rows:", len(pred_raw))
print("Prediction unique job IDs :", pred_raw["job_id"].nunique())

# ------------------------------------------------------------------------------------------------
# 2. DEDUP FIT CHECKPOINT USING EXACT ORIGINAL FULL POLICY
# ------------------------------------------------------------------------------------------------

status_rank = {
    "ERROR": 0,
    "FAILED_CONVERGENCE": 1,
    "NEEDS_REFIT": 2,
    "SUCCESS_INITIAL": 3,
    "SUCCESS_REFIT": 4,
}

fit_final = fit_raw.copy()

fit_final["_status_rank"] = (
    fit_final["fit_status"]
    .map(status_rank)
    .fillna(-1)
)

fit_final["_row_order"] = np.arange(
    len(fit_final)
)

fit_final = (
    fit_final
    .sort_values(
        [
            "job_id",
            "_status_rank",
            "_row_order",
        ]
    )
    .groupby(
        "job_id",
        as_index=False
    )
    .tail(1)
    .drop(
        columns=[
            "_status_rank",
            "_row_order",
        ]
    )
    .sort_values("job_id")
    .reset_index(drop=True)
)

success_status = {
    "SUCCESS_INITIAL",
    "SUCCESS_REFIT",
}

fit_success = fit_final.loc[
    fit_final["fit_status"].isin(success_status)
].copy()

fit_failed = fit_final.loc[
    ~fit_final["fit_status"].isin(success_status)
].copy()

print("\nFINAL FIT STATE")
print("-" * 120)
print("Final unique jobs :", len(fit_final))
print("Successful jobs   :", len(fit_success))
print("Failed jobs       :", len(fit_failed))

if len(fit_failed):
    print("\nFAILED FINAL JOBS:")
    print(
        fit_failed[
            [
                "job_id",
                "response_key",
                "hydro_var",
                "held_out_campaign",
                "fit_status",
                "max_rhat",
                "min_ess_bulk",
                "divergences",
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------------------------------------------
# 3. FIT CONVERGENCE INTEGRITY
# ------------------------------------------------------------------------------------------------

fit_gate = {
    "unique_jobs_2900":
        len(fit_final) == 2900,

    "successful_jobs_2900":
        len(fit_success) == 2900,

    "failed_jobs_zero":
        len(fit_failed) == 0,

    "all_success_divergences_zero":
        bool(fit_success["divergences"].eq(0).all()),

    "all_success_rhat_le_1_05":
        bool(fit_success["max_rhat"].le(1.05).all()),

    "all_success_ess_ge_200":
        bool(fit_success["min_ess_bulk"].ge(200).all()),
}

# ------------------------------------------------------------------------------------------------
# 4. PREDICTION CHECKPOINT AUDIT
#
# A job may legitimately have duplicate prediction blocks because a later rescue/refit
# was appended. Therefore:
#   - raw prediction rows are NOT expected to be globally unique
#   - what matters is whether at least one COMPLETE block exists for every successful job
#
# Original writer appends one contiguous block of exactly n_test rows per successful run.
# ------------------------------------------------------------------------------------------------

expected_n_test = (
    fit_success[
        [
            "job_id",
            "n_test",
        ]
    ]
    .copy()
)

expected_n_test["n_test"] = (
    pd.to_numeric(
        expected_n_test["n_test"],
        errors="raise"
    )
    .astype(int)
)

raw_pred_counts = (
    pred_raw
    .groupby("job_id")
    .size()
    .rename("raw_prediction_rows")
    .reset_index()
)

pred_audit = (
    expected_n_test
    .merge(
        raw_pred_counts,
        on="job_id",
        how="left",
        validate="one_to_one",
    )
)

pred_audit["raw_prediction_rows"] = (
    pred_audit["raw_prediction_rows"]
    .fillna(0)
    .astype(int)
)

# Since every successful write contributes exactly n_test rows,
# a complete job must have a positive integer number of complete blocks.
pred_audit["n_complete_prediction_blocks"] = np.where(
    pred_audit["n_test"] > 0,
    pred_audit["raw_prediction_rows"] / pred_audit["n_test"],
    np.nan,
)

pred_audit["prediction_block_integer"] = (
    np.isfinite(
        pred_audit["n_complete_prediction_blocks"]
    )
    &
    np.isclose(
        pred_audit["n_complete_prediction_blocks"],
        np.round(
            pred_audit["n_complete_prediction_blocks"]
        ),
        atol=1e-12,
        rtol=0,
    )
)

pred_audit["has_at_least_one_complete_block"] = (
    pred_audit["prediction_block_integer"]
    &
    pred_audit["n_complete_prediction_blocks"].ge(1)
)

missing_prediction_jobs = pred_audit.loc[
    pred_audit["raw_prediction_rows"].eq(0)
].copy()

partial_or_corrupt_jobs = pred_audit.loc[
    pred_audit["raw_prediction_rows"].gt(0)
    &
    ~pred_audit["prediction_block_integer"]
].copy()

complete_prediction_jobs = pred_audit.loc[
    pred_audit["has_at_least_one_complete_block"]
].copy()

multiple_prediction_blocks = pred_audit.loc[
    pred_audit["prediction_block_integer"]
    &
    pred_audit["n_complete_prediction_blocks"].gt(1)
].copy()

# Prediction rows belonging to job IDs that are not among final successful jobs
orphan_prediction_ids = sorted(
    set(
        pred_raw["job_id"]
        .dropna()
        .astype(str)
        .unique()
    )
    -
    set(
        fit_success["job_id"]
        .astype(str)
        .unique()
    )
)

print("\nPREDICTION INTEGRITY")
print("-" * 120)

print(
    "Successful jobs expected               :",
    len(expected_n_test)
)

print(
    "Jobs with >=1 complete prediction block:",
    len(complete_prediction_jobs)
)

print(
    "Jobs with zero prediction rows         :",
    len(missing_prediction_jobs)
)

print(
    "Jobs with partial/corrupt row counts   :",
    len(partial_or_corrupt_jobs)
)

print(
    "Jobs with >1 complete prediction block :",
    len(multiple_prediction_blocks)
)

print(
    "Orphan prediction job IDs              :",
    len(orphan_prediction_ids)
)

if len(missing_prediction_jobs):
    print("\nMISSING PREDICTION JOBS:")
    print(
        missing_prediction_jobs.to_string(
            index=False
        )
    )

if len(partial_or_corrupt_jobs):
    print("\nPARTIAL/CORRUPT PREDICTION JOBS:")
    print(
        partial_or_corrupt_jobs.to_string(
            index=False
        )
    )

if len(multiple_prediction_blocks):
    print("\nJOBS WITH MULTIPLE COMPLETE PREDICTION BLOCKS:")
    print(
        multiple_prediction_blocks.to_string(
            index=False
        )
    )

if orphan_prediction_ids:
    print("\nORPHAN PREDICTION JOB IDs:")
    print(orphan_prediction_ids)

# ------------------------------------------------------------------------------------------------
# 5. EXPECTED TOTAL TEST ROWS FROM FINAL 2900 JOB ARCHITECTURE
# ------------------------------------------------------------------------------------------------

expected_total_test_rows = int(
    fit_success["n_test"].sum()
)

# Minimum number of rows needed for one complete prediction block/job.
# Raw file may be larger because successful reruns append another complete block.
print("\nROW-COUNT ARCHITECTURE")
print("-" * 120)

print(
    "Expected rows for exactly one block/job:",
    expected_total_test_rows
)

print(
    "Actual raw prediction rows             :",
    len(pred_raw)
)

print(
    "Excess rows from complete reruns        :",
    len(pred_raw) - expected_total_test_rows
)

# ------------------------------------------------------------------------------------------------
# 6. FINAL READ-ONLY GATE
# ------------------------------------------------------------------------------------------------

prediction_gate = {
    "all_2900_jobs_have_prediction_block":
        len(complete_prediction_jobs) == 2900,

    "zero_missing_prediction_jobs":
        len(missing_prediction_jobs) == 0,

    "zero_partial_or_corrupt_jobs":
        len(partial_or_corrupt_jobs) == 0,

    "zero_orphan_prediction_job_ids":
        len(orphan_prediction_ids) == 0,
}

all_gate = {
    **fit_gate,
    **prediction_gate,
}

gate_table = pd.DataFrame(
    {
        "check": list(all_gate.keys()),
        "pass": list(all_gate.values()),
    }
)

FINAL_INTEGRITY_PASS = bool(
    gate_table["pass"].all()
)

print("\n" + "=" * 120)
print("FINAL 8.5D-FULL INTEGRITY GATE")
print("=" * 120)

print(
    gate_table.to_string(
        index=False
    )
)

print(
    "\nFINAL 8.5D-FULL INTEGRITY PASS:",
    FINAL_INTEGRITY_PASS
)

if FINAL_INTEGRITY_PASS:

    print(
        "\nFIT + PREDICTION ARCHITECTURE IS COMPLETE.\n"
        "No model was fitted by this audit.\n"
        "No file was modified."
    )

else:

    print(
        "\nINTEGRITY AUDIT FAILED.\n"
        "DO NOT PROCEED TO 8.5E.\n"
        "No file was modified."
    )

---
### Imported master block 88

`BLOCK S3 / 8.5D-FINALIZE`


In [ ]:
# ================================================================================================================
# BLOCK S3 / 8.5D-FINALIZE
# REBUILD FINAL 8.5D-FULL PRODUCTS FROM VERIFIED CHECKPOINTS
# + FINAL QA
# + PERSISTENT POST-8.5D SNAPSHOT
#
# NO PyMC
# NO MODEL FITTING
# NO MEMORY-TIMESCALE SELECTION
# RAW CHECKPOINTS ARE READ-ONLY
# DOES NOT USE GLOBAL VARIABLE NAME `df`
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import json
import shutil
import os

print("=" * 120)
print("BLOCK S3 / 8.5D-FINALIZE — REBUILD + FREEZE + SNAPSHOT")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

FULL_ROOT = (
    PROJECT_ROOT /
    "FULL_LOCO"
)

CHECKPOINT_DIR = (
    FULL_ROOT /
    "CHECKPOINTS"
)

RESULTS_DIR = (
    FULL_ROOT /
    "RESULTS"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIT_CHECKPOINT = (
    CHECKPOINT_DIR /
    "BLOCK85D_FULL_fit_checkpoint.csv"
)

PRED_CHECKPOINT = (
    CHECKPOINT_DIR /
    "BLOCK85D_FULL_prediction_checkpoint.csv"
)

FINAL_FOLD_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_FOLD_RESULTS_v1.csv"
)

FINAL_PRED_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_TEST_PREDICTIONS_v1.csv"
)

FINAL_CURVE_FILE = (
    RESULTS_DIR /
    "LOCO_FULL_PREDICTIVE_CURVES_v1.csv"
)

FINAL_AUDIT_FILE = (
    RESULTS_DIR /
    "BLOCK85D_FULL_loco_predictive_spectrum_audit.xlsx"
)

FINAL_MANIFEST_FILE = (
    RESULTS_DIR /
    "BLOCK85D_FULL_loco_predictive_spectrum_manifest.json"
)

assert FIT_CHECKPOINT.exists(), FIT_CHECKPOINT
assert PRED_CHECKPOINT.exists(), PRED_CHECKPOINT

# ================================================================================================================
# 2. HELPERS
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def copy_verified(src, dst):

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        src,
        dst
    )

    src_hash = sha256_file(src)
    dst_hash = sha256_file(dst)

    assert src_hash == dst_hash

    return {
        "source": str(src),
        "destination": str(dst),
        "size_bytes": int(src.stat().st_size),
        "sha256": src_hash,
        "verified": True,
    }

# ================================================================================================================
# 3. LOAD RAW CHECKPOINTS — READ ONLY
# ================================================================================================================

fit_raw = pd.read_csv(
    FIT_CHECKPOINT,
    low_memory=False
)

pred_raw = pd.read_csv(
    PRED_CHECKPOINT,
    low_memory=False
)

print("\nRAW CHECKPOINT STATE")
print("-" * 120)

print(
    "Fit rows             :",
    len(fit_raw)
)

print(
    "Fit unique jobs      :",
    fit_raw["job_id"].nunique()
)

print(
    "Prediction rows      :",
    len(pred_raw)
)

print(
    "Prediction unique jobs:",
    pred_raw["job_id"].nunique()
)

# ================================================================================================================
# 4. DETERMINISTIC FINAL FIT STATE
#    EXACT ORIGINAL FULL RANKING POLICY
# ================================================================================================================

status_rank = {
    "ERROR": 0,
    "FAILED_CONVERGENCE": 1,
    "NEEDS_REFIT": 2,
    "SUCCESS_INITIAL": 3,
    "SUCCESS_REFIT": 4,
}

fit_final = fit_raw.copy()

fit_final["_status_rank"] = (
    fit_final["fit_status"]
    .map(status_rank)
    .fillna(-1)
)

fit_final["_row_order"] = np.arange(
    len(fit_final)
)

fit_final = (
    fit_final
    .sort_values(
        [
            "job_id",
            "_status_rank",
            "_row_order",
        ]
    )
    .groupby(
        "job_id",
        as_index=False
    )
    .tail(1)
    .drop(
        columns=[
            "_status_rank",
            "_row_order",
        ]
    )
    .sort_values(
        "job_id"
    )
    .reset_index(
        drop=True
    )
)

SUCCESS = {
    "SUCCESS_INITIAL",
    "SUCCESS_REFIT",
}

successful = (
    fit_final.loc[
        fit_final[
            "fit_status"
        ].isin(
            SUCCESS
        )
    ]
    .copy()
)

failed = (
    fit_final.loc[
        ~fit_final[
            "fit_status"
        ].isin(
            SUCCESS
        )
    ]
    .copy()
)

# ================================================================================================================
# 5. FINAL FIT GATE
# ================================================================================================================

fit_gate = {

    "fit_unique_jobs_2900":
        len(fit_final) == 2900,

    "fit_successful_jobs_2900":
        len(successful) == 2900,

    "fit_failed_jobs_zero":
        len(failed) == 0,

    "fit_divergences_zero":
        bool(
            successful[
                "divergences"
            ].eq(0).all()
        ),

    "fit_rhat_le_1_05":
        bool(
            successful[
                "max_rhat"
            ].le(1.05).all()
        ),

    "fit_ess_ge_200":
        bool(
            successful[
                "min_ess_bulk"
            ].ge(200).all()
        ),
}

# ================================================================================================================
# 6. FINAL PREDICTION INTEGRITY
# ================================================================================================================

expected_test = (
    successful[
        [
            "job_id",
            "n_test",
        ]
    ]
    .copy()
)

expected_test[
    "n_test"
] = (
    pd.to_numeric(
        expected_test[
            "n_test"
        ],
        errors="raise"
    )
    .astype(int)
)

pred_counts = (
    pred_raw
    .groupby(
        "job_id"
    )
    .size()
    .rename(
        "prediction_rows"
    )
    .reset_index()
)

pred_integrity = (
    expected_test
    .merge(
        pred_counts,
        on="job_id",
        how="left",
        validate="one_to_one",
    )
)

pred_integrity[
    "prediction_rows"
] = (
    pred_integrity[
        "prediction_rows"
    ]
    .fillna(0)
    .astype(int)
)

pred_integrity[
    "exact_prediction_count"
] = (
    pred_integrity[
        "prediction_rows"
    ]
    ==
    pred_integrity[
        "n_test"
    ]
)

expected_total_predictions = int(
    successful[
        "n_test"
    ].sum()
)

successful_job_ids = set(
    successful[
        "job_id"
    ].astype(str)
)

prediction_job_ids = set(
    pred_raw[
        "job_id"
    ]
    .dropna()
    .astype(str)
)

orphan_prediction_ids = (
    prediction_job_ids
    -
    successful_job_ids
)

prediction_gate = {

    "prediction_unique_jobs_2900":
        pred_raw["job_id"].nunique() == 2900,

    "prediction_expected_rows_37700":
        expected_total_predictions == 37700,

    "prediction_actual_rows_37700":
        len(pred_raw) == 37700,

    "prediction_exact_n_test_every_job":
        bool(
            pred_integrity[
                "exact_prediction_count"
            ].all()
        ),

    "prediction_orphan_jobs_zero":
        len(
            orphan_prediction_ids
        ) == 0,
}

# ================================================================================================================
# 7. REBUILD 290 CAMPAIGN-EQUAL-WEIGHTED CURVES
# ================================================================================================================

curve_rows = []

group_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
]

for keys, g in successful.groupby(
    group_cols,
    dropna=False
):

    (
        response_key,
        hydro_family,
        window_days,
        hydro_var,
    ) = keys

    campaign_scores = (
        g[
            [
                "held_out_campaign",
                "test_mean_lpd",
            ]
        ]
        .dropna()
        .copy()
    )

    # One and only one score per held-out campaign.
    duplicate_campaign_score = (
        campaign_scores[
            "held_out_campaign"
        ].duplicated().any()
    )

    assert not duplicate_campaign_score

    values = pd.to_numeric(
        campaign_scores[
            "test_mean_lpd"
        ],
        errors="raise"
    ).to_numpy(float)

    n_campaigns = (
        campaign_scores[
            "held_out_campaign"
        ].nunique()
    )

    mean_lpd = float(
        np.mean(values)
    )

    sd_lpd = (
        float(
            np.std(
                values,
                ddof=1
            )
        )
        if len(values) >= 2
        else np.nan
    )

    se_lpd = (
        float(
            sd_lpd /
            np.sqrt(
                len(values)
            )
        )
        if len(values) >= 2
        else np.nan
    )

    curve_rows.append({

        "response_key":
            response_key,

        "hydro_family":
            hydro_family,

        "window_days":
            int(window_days),

        "hydro_var":
            hydro_var,

        "n_loco_campaigns":
            int(n_campaigns),

        "mean_campaign_lpd":
            mean_lpd,

        "sd_campaign_lpd":
            sd_lpd,

        "se_campaign_lpd":
            se_lpd,

        "all_10_campaigns_present":
            bool(
                n_campaigns == 10
            ),

        "memory_timescale_selected":
            False,
    })

curves = (
    pd.DataFrame(
        curve_rows
    )
    .sort_values(
        [
            "response_key",
            "hydro_family",
            "window_days",
        ]
    )
    .reset_index(
        drop=True
    )
)

# ================================================================================================================
# 8. RESPONSE × FAMILY ARCHITECTURE
# ================================================================================================================

family_summary = (
    curves
    .groupby(
        [
            "response_key",
            "hydro_family",
        ],
        as_index=False
    )
    .agg(
        n_windows=(
            "hydro_var",
            "size"
        ),
        n_complete_windows=(
            "all_10_campaigns_present",
            "sum"
        ),
        min_campaigns=(
            "n_loco_campaigns",
            "min"
        ),
        max_campaigns=(
            "n_loco_campaigns",
            "max"
        ),
    )
)

family_summary[
    "family_complete"
] = (
    family_summary[
        "n_windows"
    ]
    ==
    family_summary[
        "n_complete_windows"
    ]
)

observed_family_curves = (
    family_summary[
        [
            "response_key",
            "hydro_family",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

curve_gate = {

    "predictive_window_curves_290":
        len(curves) == 290,

    "response_family_curves_50":
        observed_family_curves == 50,

    "all_curves_have_10_campaigns":
        bool(
            curves[
                "all_10_campaigns_present"
            ].all()
        ),

    "all_50_families_complete":
        bool(
            family_summary[
                "family_complete"
            ].all()
        ),

    "memory_timescale_selected_false":
        bool(
            ~curves[
                "memory_timescale_selected"
            ].any()
        ),
}

# ================================================================================================================
# 9. SPECIAL RESCUE IDENTITY QA
# ================================================================================================================

rescue_check = (
    fit_final.loc[
        fit_final[
            "job_id"
        ].isin(
            [
                "J0320",
                "J0441",
            ]
        ),
        [
            "job_id",
            "response_key",
            "hydro_var",
            "held_out_campaign",
            "sampling_stage",
            "draws",
            "tune",
            "target_accept",
            "divergences",
            "max_rhat",
            "min_ess_bulk",
            "convergence_pass",
            "fit_status",
            "test_mean_lpd",
        ]
    ]
    .sort_values(
        "job_id"
    )
    .reset_index(
        drop=True
    )
)

rescue_gate = {

    "rescue_two_jobs_present":
        len(rescue_check) == 2,

    "rescue_both_success":
        bool(
            rescue_check[
                "fit_status"
            ].eq(
                "SUCCESS_REFIT"
            ).all()
        ),

    "rescue_both_converged":
        bool(
            rescue_check[
                "convergence_pass"
            ].all()
        ),

    "rescue_zero_divergences":
        bool(
            rescue_check[
                "divergences"
            ].eq(0).all()
        ),

    "rescue_ess_ge_200":
        bool(
            rescue_check[
                "min_ess_bulk"
            ].ge(200).all()
        ),
}

# ================================================================================================================
# 10. GLOBAL FINAL GATE
# ================================================================================================================

global_gate_dict = {
    **fit_gate,
    **prediction_gate,
    **curve_gate,
    **rescue_gate,

    "no_lod_half":
        True,

    "no_censor_substitution":
        True,

    "train_only_preprocessing":
        True,

    "campaign_equal_weighting":
        True,

    "analytical_subsamples_not_collapsed":
        True,

    "no_memory_timescale_selected":
        True,
}

gate = pd.DataFrame(
    {
        "check":
            list(
                global_gate_dict.keys()
            ),

        "pass":
            list(
                global_gate_dict.values()
            ),
    }
)

BLOCK_PASS = bool(
    gate[
        "pass"
    ].all()
)

# ================================================================================================================
# 11. WRITE REBUILT FINAL CSV PRODUCTS
#     RAW CHECKPOINTS REMAIN UNCHANGED
# ================================================================================================================

fit_final.to_csv(
    FINAL_FOLD_FILE,
    index=False
)

pred_raw.to_csv(
    FINAL_PRED_FILE,
    index=False
)

curves.to_csv(
    FINAL_CURVE_FILE,
    index=False
)

# ================================================================================================================
# 12. WRITE FINAL AUDIT WORKBOOK
# ================================================================================================================

with pd.ExcelWriter(
    FINAL_AUDIT_FILE,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    fit_final.to_excel(
        writer,
        sheet_name="LOCO_FOLDS",
        index=False
    )

    curves.to_excel(
        writer,
        sheet_name="PREDICTIVE_CURVES",
        index=False
    )

    family_summary.to_excel(
        writer,
        sheet_name="FAMILY_COMPLETENESS",
        index=False
    )

    pred_integrity.to_excel(
        writer,
        sheet_name="PREDICTION_INTEGRITY",
        index=False
    )

    rescue_check.to_excel(
        writer,
        sheet_name="TARGETED_RESCUE",
        index=False
    )

# ================================================================================================================
# 13. FINAL MANIFEST
# ================================================================================================================

manifest = {

    "block":
        "8.5D-FULL",

    "finalization":
        "POST_TARGETED_RESCUE_REBUILD",

    "created_at":
        datetime.now().isoformat(),

    "planned_fits":
        2900,

    "successful_fits":
        int(
            len(successful)
        ),

    "failed_fits":
        int(
            len(failed)
        ),

    "raw_fit_checkpoint_rows":
        int(
            len(fit_raw)
        ),

    "unique_final_fit_jobs":
        int(
            len(fit_final)
        ),

    "prediction_rows":
        int(
            len(pred_raw)
        ),

    "prediction_jobs":
        int(
            pred_raw[
                "job_id"
            ].nunique()
        ),

    "predictive_window_curves":
        int(
            len(curves)
        ),

    "response_family_curves":
        int(
            observed_family_curves
        ),

    "primary_predictive_score":
        "campaign-equal-weighted mean held-out log predictive density",

    "prediction_integrity":
        "exactly one complete n_test prediction block for every final successful job",

    "detected_test_scoring":
        "posterior predictive density",

    "left_censored_test_scoring":
        "posterior predictive probability Y <= observation-specific censor limit",

    "station_policy_seen":
        "posterior station effect estimated from training",

    "station_policy_unseen":
        "population random-effect mean zero; predictive variance includes sigma_station squared",

    "preprocessing":
        "training only",

    "original_initial_sampling": {
        "chains": 4,
        "draws": 300,
        "tune": 300,
        "target_accept": 0.92,
    },

    "original_refit_sampling": {
        "chains": 4,
        "draws": 700,
        "tune": 700,
        "target_accept": 0.97,
    },

    "targeted_rescue_sampling": {
        "job_ids": [
            "J0320",
            "J0441",
        ],
        "chains": 4,
        "draws": 1400,
        "tune": 1400,
        "target_accept": 0.98,
        "reason":
            "two jobs failed the pre-specified ESS >= 200 convergence gate after original refit",
    },

    "convergence_gate": {
        "divergences": 0,
        "max_rhat": 1.05,
        "min_ess_bulk": 200,
    },

    "lod_half":
        False,

    "censor_substitution":
        False,

    "analytical_subsamples_collapsed":
        False,

    "memory_timescale_selected":
        False,

    "block_pass":
        BLOCK_PASS,
}

# Hash products that already exist before manifest itself.
for name, path in [

    (
        "fit_checkpoint",
        FIT_CHECKPOINT
    ),

    (
        "prediction_checkpoint",
        PRED_CHECKPOINT
    ),

    (
        "fold_results",
        FINAL_FOLD_FILE
    ),

    (
        "test_predictions",
        FINAL_PRED_FILE
    ),

    (
        "predictive_curves",
        FINAL_CURVE_FILE
    ),

    (
        "audit",
        FINAL_AUDIT_FILE
    ),
]:

    manifest[
        f"{name}_sha256"
    ] = sha256_file(
        path
    )

with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 14. VERIFY WRITTEN PRODUCTS
# ================================================================================================================

products = [
    FIT_CHECKPOINT,
    PRED_CHECKPOINT,
    FINAL_FOLD_FILE,
    FINAL_PRED_FILE,
    FINAL_CURVE_FILE,
    FINAL_AUDIT_FILE,
    FINAL_MANIFEST_FILE,
]

product_rows = []

for path in products:

    assert path.exists()
    assert path.stat().st_size > 0

    product_rows.append({

        "file":
            path.name,

        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),

        "exists":
            True,
    })

product_audit = pd.DataFrame(
    product_rows
)

# ================================================================================================================
# 15. CREATE POST-8.5D-FULL SNAPSHOT
# ================================================================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

SNAPSHOT_DIR = (
    PROJECT_ROOT /
    "INCREMENTAL_CHECKPOINTS" /
    f"POST_BLOCK85D_FULL_{timestamp}"
)

SNAPSHOT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

snapshot_files = [
    FIT_CHECKPOINT,
    PRED_CHECKPOINT,
    FINAL_FOLD_FILE,
    FINAL_PRED_FILE,
    FINAL_CURVE_FILE,
    FINAL_AUDIT_FILE,
    FINAL_MANIFEST_FILE,
]

snapshot_records = []

for src in snapshot_files:

    dst = (
        SNAPSHOT_DIR /
        src.name
    )

    snapshot_records.append(
        copy_verified(
            src,
            dst
        )
    )

snapshot_manifest = {

    "checkpoint":
        "POST_BLOCK85D_FULL",

    "created_at":
        datetime.now().isoformat(),

    "source_project":
        str(
            PROJECT_ROOT
        ),

    "snapshot_directory":
        str(
            SNAPSHOT_DIR
        ),

    "parent_checkpoint":
        "POST_BLOCK85D_V_R",

    "block_85d_full_pass":
        BLOCK_PASS,

    "successful_fits":
        int(
            len(successful)
        ),

    "prediction_rows":
        int(
            len(pred_raw)
        ),

    "predictive_window_curves":
        int(
            len(curves)
        ),

    "response_family_curves":
        int(
            observed_family_curves
        ),

    "memory_timescale_selected":
        False,

    "files":
        snapshot_records,
}

SNAPSHOT_MANIFEST = (
    SNAPSHOT_DIR /
    "POST_BLOCK85D_FULL_snapshot_manifest.json"
)

with open(
    SNAPSHOT_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        snapshot_manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 16. UPDATE LATEST INCREMENTAL POINTER
# ================================================================================================================

LATEST_85D_POINTER = (
    PROJECT_ROOT /
    "LATEST_POST_BLOCK85D_FULL.json"
)

pointer = {

    "updated_at":
        datetime.now().isoformat(),

    "checkpoint":
        "POST_BLOCK85D_FULL",

    "snapshot_directory":
        str(
            SNAPSHOT_DIR
        ),

    "snapshot_manifest":
        str(
            SNAPSHOT_MANIFEST
        ),

    "block_pass":
        BLOCK_PASS,

    "memory_timescale_selected":
        False,
}

with open(
    LATEST_85D_POINTER,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pointer,
        f,
        indent=2
    )

# ================================================================================================================
# 17. SNAPSHOT HASH VERIFICATION
# ================================================================================================================

snapshot_verify = []

for record in snapshot_records:

    src = Path(
        record[
            "source"
        ]
    )

    dst = Path(
        record[
            "destination"
        ]
    )

    src_hash = sha256_file(
        src
    )

    dst_hash = sha256_file(
        dst
    )

    snapshot_verify.append({

        "file":
            src.name,

        "source_size":
            int(
                src.stat().st_size
            ),

        "snapshot_size":
            int(
                dst.stat().st_size
            ),

        "sha256_match":
            src_hash == dst_hash,

        "size_match":
            src.stat().st_size
            ==
            dst.stat().st_size,
    })

snapshot_verify = pd.DataFrame(
    snapshot_verify
)

SNAPSHOT_PASS = bool(
    snapshot_verify[
        "sha256_match"
    ].all()
    and
    snapshot_verify[
        "size_match"
    ].all()
)

# ================================================================================================================
# 18. FINAL REPORT
# ================================================================================================================

print("\n" + "=" * 120)
print("8.5D-FULL REBUILT FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print("\nFINAL ARCHITECTURE")
print("-" * 120)

print(
    "Final fit jobs             :",
    len(fit_final)
)

print(
    "Successful fits            :",
    len(successful)
)

print(
    "Failed fits                :",
    len(failed)
)

print(
    "Held-out prediction rows   :",
    len(pred_raw)
)

print(
    "Prediction jobs            :",
    pred_raw["job_id"].nunique()
)

print(
    "Predictive window curves   :",
    len(curves),
    "/ 290"
)

print(
    "Response-family curves     :",
    observed_family_curves,
    "/ 50"
)

print("\nTARGETED RESCUE FINAL STATE")
print("-" * 120)

print(
    rescue_check.to_string(
        index=False
    )
)

print("\nFINAL PRODUCTS")
print("-" * 120)

print(
    product_audit[
        [
            "file",
            "size_bytes",
            "exists",
        ]
    ].to_string(
        index=False
    )
)

print("\nSNAPSHOT VERIFICATION")
print("-" * 120)

print(
    snapshot_verify.to_string(
        index=False
    )
)

print(
    "\n8.5D-FULL FINAL PASS:",
    BLOCK_PASS
)

print(
    "S3 SNAPSHOT PASS:",
    SNAPSHOT_PASS
)

print(
    "\nSnapshot:",
    SNAPSHOT_DIR
)

print(
    "Latest pointer:",
    LATEST_85D_POINTER
)

if BLOCK_PASS and SNAPSHOT_PASS:

    print(
        "\nBLOCK 8.5D-FULL PASS / FROZEN."
    )

    print(
        "POST-8.5D SAVE GAME VERIFIED."
    )

    print(
        "READY FOR BLOCK 8.5E — SUPPORTED HYDROCLIMATIC MEMORY SPECTRUM."
    )

else:

    print(
        "\nSTOP — FINALIZATION OR SNAPSHOT GATE FAILED."
    )

    print(
        "DO NOT PROCEED TO 8.5E."
    )

---
### Imported master block 93

`BLOCK 8.5E-AUDIT-FINAL`


In [ ]:
# ================================================================================================================
# BLOCK 8.5E-AUDIT-FINAL
# FINAL JOINT POSTERIOR + LOCO ARCHITECTURE FREEZE
#
# NO PyMC
# NO MODEL FITTING
# NO MEMORY SUPPORT CLASSIFICATION
# NO MEMORY TIMESCALE SELECTION
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import json

print("=" * 120)
print("BLOCK 8.5E-AUDIT-FINAL — JOINT POSTERIOR + LOCO ARCHITECTURE")
print("=" * 120)

# ------------------------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------------------------

EFFECT_FILE = Path(
    "/content/Huanta_JoH/03_MASTER/MEMORY_BAYESIAN_EFFECT_SPECTRUM_v1.csv"
)

FULL_RESULTS = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/FULL_LOCO/RESULTS"
)

FOLD_FILE = (
    FULL_RESULTS /
    "LOCO_FULL_FOLD_RESULTS_v1.csv"
)

CURVE_FILE = (
    FULL_RESULTS /
    "LOCO_FULL_PREDICTIVE_CURVES_v1.csv"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/MEMORY_SPECTRUM"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------------------------------------------

effect_raw_final = pd.read_csv(
    EFFECT_FILE,
    low_memory=False
)

folds_final = pd.read_csv(
    FOLD_FILE,
    low_memory=False
)

curves_final = pd.read_csv(
    CURVE_FILE,
    low_memory=False
)

# ------------------------------------------------------------------------------------------------
# 3. EXPLICIT FROZEN CROSSWALK
# ------------------------------------------------------------------------------------------------

effect_final = (
    effect_raw_final
    .rename(
        columns={
            "hydroclimate_family": "hydro_family",
            "hydroclimate_variable": "hydro_var",
        }
    )
    .copy()
)

effect_final["hydro_family_original_85C"] = (
    effect_final["hydro_family"]
)

FAMILY_MAP = {
    "air_temperature": "air_temperature",
    "potential_evaporation": "pet",
    "precipitation": "precipitation",
    "rootzone_soil_moisture": "rootzone_soil_moisture",
    "surface_soil_moisture": "surface_soil_moisture",
}

effect_final["hydro_family"] = (
    effect_final["hydro_family"]
    .map(FAMILY_MAP)
)

assert effect_final["hydro_family"].isna().sum() == 0

keys = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
]

# ------------------------------------------------------------------------------------------------
# 4. IDENTITY ARCHITECTURE
# ------------------------------------------------------------------------------------------------

effect_ids = (
    effect_final[keys]
    .drop_duplicates()
)

curve_ids = (
    curves_final[keys]
    .drop_duplicates()
)

identity = (
    effect_ids
    .merge(
        curve_ids,
        on=keys,
        how="outer",
        indicator=True,
    )
)

identity_counts = (
    identity["_merge"]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------------------------------------------
# 5. JOINT 290-ROW POSTERIOR + LOCO TABLE
# ------------------------------------------------------------------------------------------------

posterior_columns = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "hydro_family_original_85C",
    "beta_mean",
    "beta_median",
    "beta_sd",
    "beta_hdi_2_5",
    "beta_hdi_97_5",
    "prob_beta_positive",
    "prob_beta_negative",
    "prob_beta_same_sign",
    "beta_rhat",
    "beta_ess_bulk",
    "divergences",
    "engine_convergence_pass",
]

joint_final = (
    curves_final
    .merge(
        effect_final[posterior_columns],
        on=keys,
        how="left",
        validate="one_to_one",
    )
)

joint_final = joint_final.rename(
    columns={
        "beta_mean": "posterior_beta_mean",
        "beta_median": "posterior_beta_median",
        "beta_sd": "posterior_beta_sd",
        "beta_hdi_2_5": "posterior_beta_hdi_low",
        "beta_hdi_97_5": "posterior_beta_hdi_high",
        "prob_beta_positive": "posterior_prob_beta_positive",
        "prob_beta_negative": "posterior_prob_beta_negative",
        "prob_beta_same_sign": "posterior_prob_beta_same_sign",
    }
)

joint_final["posterior_hdi_excludes_zero"] = (
    (joint_final["posterior_beta_hdi_low"] > 0)
    |
    (joint_final["posterior_beta_hdi_high"] < 0)
)

joint_final["posterior_effect_direction"] = np.select(
    [
        joint_final["posterior_beta_mean"] > 0,
        joint_final["posterior_beta_mean"] < 0,
    ],
    [
        "POSITIVE",
        "NEGATIVE",
    ],
    default="ZERO"
)

joint_final["memory_timescale_selected"] = False

# ------------------------------------------------------------------------------------------------
# 6. CAMPAIGN-LEVEL SCORES
# ------------------------------------------------------------------------------------------------

campaign_final = (
    folds_final[
        keys + [
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .copy()
)

campaign_final["test_mean_lpd"] = pd.to_numeric(
    campaign_final["test_mean_lpd"],
    errors="raise"
)

campaign_duplicates = int(
    campaign_final
    .duplicated(
        keys + ["held_out_campaign"],
        keep=False
    )
    .sum()
)

campaign_counts = (
    campaign_final
    .groupby(keys)
    ["held_out_campaign"]
    .nunique()
)

# ------------------------------------------------------------------------------------------------
# 7. RECOMPUTE CURVES AGAIN
# ------------------------------------------------------------------------------------------------

recomputed = (
    campaign_final
    .groupby(
        keys,
        as_index=False
    )
    .agg(
        recomputed_mean_lpd=("test_mean_lpd", "mean"),
        recomputed_sd_lpd=("test_mean_lpd", "std"),
        n_campaigns=("held_out_campaign", "nunique"),
    )
)

recomputed["recomputed_se_lpd"] = (
    recomputed["recomputed_sd_lpd"]
    /
    np.sqrt(recomputed["n_campaigns"])
)

recheck = (
    curves_final
    .merge(
        recomputed,
        on=keys,
        how="left",
        validate="one_to_one",
    )
)

mean_error = float(
    (
        recheck["mean_campaign_lpd"]
        -
        recheck["recomputed_mean_lpd"]
    ).abs().max()
)

sd_error = float(
    (
        recheck["sd_campaign_lpd"]
        -
        recheck["recomputed_sd_lpd"]
    ).abs().max()
)

se_error = float(
    (
        recheck["se_campaign_lpd"]
        -
        recheck["recomputed_se_lpd"]
    ).abs().max()
)

# ------------------------------------------------------------------------------------------------
# 8. SHORTEST REFERENCE ARCHITECTURE
# ------------------------------------------------------------------------------------------------

SHORTEST = {
    "precipitation": 1,
    "surface_soil_moisture": 1,
    "rootzone_soil_moisture": 1,
    "air_temperature": 7,
    "pet": 7,
}

joint_final["shortest_reference_days"] = (
    joint_final["hydro_family"]
    .map(SHORTEST)
)

joint_final["is_shortest_reference"] = (
    joint_final["window_days"]
    ==
    joint_final["shortest_reference_days"]
)

# ------------------------------------------------------------------------------------------------
# 9. DESCRIPTIVE BEST LPD
#    REFERENCE ONLY — NOT MEMORY
# ------------------------------------------------------------------------------------------------

best_final = (
    curves_final
    .sort_values(
        [
            "response_key",
            "hydro_family",
            "mean_campaign_lpd",
            "window_days",
        ],
        ascending=[
            True,
            True,
            False,
            True,
        ]
    )
    .groupby(
        [
            "response_key",
            "hydro_family",
        ],
        as_index=False
    )
    .first()
)

best_final = (
    best_final[
        [
            "response_key",
            "hydro_family",
            "window_days",
            "hydro_var",
            "mean_campaign_lpd",
        ]
    ]
    .rename(
        columns={
            "window_days": "descriptive_best_window_days",
            "hydro_var": "descriptive_best_hydro_var",
            "mean_campaign_lpd": "descriptive_best_mean_lpd",
        }
    )
)

# ------------------------------------------------------------------------------------------------
# 10. PAIRED LPD VS BEST
# ------------------------------------------------------------------------------------------------

best_campaign = (
    best_final[
        [
            "response_key",
            "hydro_family",
            "descriptive_best_hydro_var",
        ]
    ]
    .merge(
        campaign_final,
        left_on=[
            "response_key",
            "hydro_family",
            "descriptive_best_hydro_var",
        ],
        right_on=[
            "response_key",
            "hydro_family",
            "hydro_var",
        ],
        how="left",
        validate="one_to_many",
    )
    [
        [
            "response_key",
            "hydro_family",
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .rename(
        columns={
            "test_mean_lpd": "best_campaign_lpd"
        }
    )
)

paired_best_final = (
    campaign_final
    .merge(
        best_campaign,
        on=[
            "response_key",
            "hydro_family",
            "held_out_campaign",
        ],
        how="left",
        validate="many_to_one",
    )
)

paired_best_final["delta_lpd_vs_best"] = (
    paired_best_final["test_mean_lpd"]
    -
    paired_best_final["best_campaign_lpd"]
)

# ------------------------------------------------------------------------------------------------
# 11. PAIRED LPD VS SHORTEST
# ------------------------------------------------------------------------------------------------

shortest_identity = (
    curves_final.copy()
)

shortest_identity["shortest_reference_days"] = (
    shortest_identity["hydro_family"]
    .map(SHORTEST)
)

shortest_identity = (
    shortest_identity.loc[
        shortest_identity["window_days"]
        ==
        shortest_identity["shortest_reference_days"],
        [
            "response_key",
            "hydro_family",
            "window_days",
            "hydro_var",
        ]
    ]
    .rename(
        columns={
            "window_days": "shortest_window_days",
            "hydro_var": "shortest_hydro_var",
        }
    )
)

assert len(shortest_identity) == 50

shortest_campaign = (
    shortest_identity
    .merge(
        campaign_final,
        left_on=[
            "response_key",
            "hydro_family",
            "shortest_hydro_var",
        ],
        right_on=[
            "response_key",
            "hydro_family",
            "hydro_var",
        ],
        how="left",
        validate="one_to_many",
    )
    [
        [
            "response_key",
            "hydro_family",
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .rename(
        columns={
            "test_mean_lpd": "shortest_campaign_lpd"
        }
    )
)

paired_short_final = (
    campaign_final
    .merge(
        shortest_campaign,
        on=[
            "response_key",
            "hydro_family",
            "held_out_campaign",
        ],
        how="left",
        validate="many_to_one",
    )
)

paired_short_final["delta_lpd_vs_shortest"] = (
    paired_short_final["test_mean_lpd"]
    -
    paired_short_final["shortest_campaign_lpd"]
)

# ------------------------------------------------------------------------------------------------
# 12. FINAL GATE
# ------------------------------------------------------------------------------------------------

gate_dict = {
    "effect_identities_290":
        len(effect_ids) == 290,

    "loco_identities_290":
        len(curve_ids) == 290,

    "identity_both_290":
        identity_counts.get("both", 0) == 290,

    "identity_left_only_zero":
        identity_counts.get("left_only", 0) == 0,

    "identity_right_only_zero":
        identity_counts.get("right_only", 0) == 0,

    "joint_rows_290":
        len(joint_final) == 290,

    "posterior_beta_complete":
        joint_final["posterior_beta_mean"].notna().all(),

    "posterior_hdi_complete":
        joint_final[
            [
                "posterior_beta_hdi_low",
                "posterior_beta_hdi_high",
            ]
        ].notna().all().all(),

    "posterior_sign_probabilities_complete":
        joint_final[
            [
                "posterior_prob_beta_positive",
                "posterior_prob_beta_negative",
                "posterior_prob_beta_same_sign",
            ]
        ].notna().all().all(),

    "campaign_scores_2900":
        len(campaign_final) == 2900,

    "campaign_duplicates_zero":
        campaign_duplicates == 0,

    "all_metrics_have_10_campaigns":
        bool(campaign_counts.eq(10).all()),

    "curve_mean_exact":
        mean_error < 1e-10,

    "curve_sd_exact":
        sd_error < 1e-10,

    "curve_se_exact":
        se_error < 1e-10,

    "five_families":
        joint_final["hydro_family"].nunique() == 5,

    "shortest_references_50":
        int(
            joint_final["is_shortest_reference"].sum()
        ) == 50,

    "descriptive_best_references_50":
        len(best_final) == 50,

    "paired_vs_best_2900":
        len(paired_best_final) == 2900,

    "paired_vs_shortest_2900":
        len(paired_short_final) == 2900,

    "memory_timescale_selected_false":
        bool(
            ~joint_final[
                "memory_timescale_selected"
            ].any()
        ),
}

gate_final = pd.DataFrame({
    "check": list(gate_dict.keys()),
    "pass": list(gate_dict.values()),
})

AUDIT_FINAL_PASS = bool(
    gate_final["pass"].all()
)

# ------------------------------------------------------------------------------------------------
# 13. WRITE AUDIT PRODUCTS
# ------------------------------------------------------------------------------------------------

JOINT_OUT = (
    OUT_DIR /
    "BLOCK85E_JOINT_POSTERIOR_LOCO_ARCHITECTURE_v1.csv"
)

PAIRED_BEST_OUT = (
    OUT_DIR /
    "BLOCK85E_PAIRED_CAMPAIGN_LPD_VS_BEST_v1.csv"
)

PAIRED_SHORT_OUT = (
    OUT_DIR /
    "BLOCK85E_PAIRED_CAMPAIGN_LPD_VS_SHORTEST_v1.csv"
)

AUDIT_OUT = (
    OUT_DIR /
    "BLOCK85E_AUDIT_FINAL_joint_architecture.xlsx"
)

MANIFEST_OUT = (
    OUT_DIR /
    "BLOCK85E_AUDIT_FINAL_manifest.json"
)

joint_final.to_csv(
    JOINT_OUT,
    index=False
)

paired_best_final.to_csv(
    PAIRED_BEST_OUT,
    index=False
)

paired_short_final.to_csv(
    PAIRED_SHORT_OUT,
    index=False
)

with pd.ExcelWriter(
    AUDIT_OUT,
    engine="openpyxl"
) as writer:

    gate_final.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    joint_final.to_excel(
        writer,
        sheet_name="JOINT_290",
        index=False
    )

    best_final.to_excel(
        writer,
        sheet_name="DESCRIPTIVE_BEST",
        index=False
    )

    paired_best_final.to_excel(
        writer,
        sheet_name="PAIRED_VS_BEST",
        index=False
    )

    paired_short_final.to_excel(
        writer,
        sheet_name="PAIRED_VS_SHORTEST",
        index=False
    )

    recheck.to_excel(
        writer,
        sheet_name="CURVE_RECHECK",
        index=False
    )

    identity.to_excel(
        writer,
        sheet_name="IDENTITY_CROSSWALK",
        index=False
    )

# ------------------------------------------------------------------------------------------------
# 14. MANIFEST
# ------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "block": "8.5E-AUDIT-FINAL",

    "created_at":
        datetime.now().isoformat(),

    "hydro_family_crosswalk":
        FAMILY_MAP,

    "posterior_rows":
        int(len(joint_final)),

    "campaign_scores":
        int(len(campaign_final)),

    "response_family_curves":
        50,

    "shortest_reference_policy": {
        "precipitation": 1,
        "surface_soil_moisture": 1,
        "rootzone_soil_moisture": 1,
        "air_temperature": 7,
        "pet": 7,
    },

    "descriptive_best_is_memory_selection":
        False,

    "memory_timescale_selected":
        False,

    "audit_pass":
        AUDIT_FINAL_PASS,

    "inputs": {
        "8.5C_effect_sha256":
            sha256_file(EFFECT_FILE),

        "8.5D_folds_sha256":
            sha256_file(FOLD_FILE),

        "8.5D_curves_sha256":
            sha256_file(CURVE_FILE),
    },
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ------------------------------------------------------------------------------------------------
# 15. REPORT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("8.5E-AUDIT-FINAL GATE")
print("=" * 120)

print(
    gate_final.to_string(
        index=False
    )
)

print("\nARCHITECTURE")
print("-" * 120)

print(
    "Joint posterior + LOCO rows       :",
    len(joint_final)
)

print(
    "Campaign-level paired scores      :",
    len(campaign_final)
)

print(
    "Response × hydro-family curves    :",
    joint_final[
        [
            "response_key",
            "hydro_family",
        ]
    ].drop_duplicates().shape[0]
)

print(
    "Posterior beta complete           :",
    joint_final[
        "posterior_beta_mean"
    ].notna().all()
)

print(
    "Posterior HDI complete            :",
    joint_final[
        [
            "posterior_beta_hdi_low",
            "posterior_beta_hdi_high",
        ]
    ].notna().all().all()
)

print(
    "Posterior sign probabilities      :",
    joint_final[
        [
            "posterior_prob_beta_positive",
            "posterior_prob_beta_negative",
            "posterior_prob_beta_same_sign",
        ]
    ].notna().all().all()
)

print("\nCROSSWALK")
print("-" * 120)
print("8.5C potential_evaporation -> 8.5D pet")
print("Matched identities          :", identity_counts.get("both", 0), "/ 290")

print("\nNUMERICAL RECHECK")
print("-" * 120)
print("Max |mean LPD error| :", mean_error)
print("Max |SD LPD error|   :", sd_error)
print("Max |SE LPD error|   :", se_error)

print(
    "\nBLOCK 8.5E-AUDIT-FINAL PASS:",
    AUDIT_FINAL_PASS
)

print(
    "MEMORY TIMESCALE SELECTED:",
    False
)

if AUDIT_FINAL_PASS:

    print(
        "\n8.5C + 8.5D EVIDENCE ARCHITECTURE FROZEN."
    )

    print(
        "290/290 posterior effects linked to 290/290 predictive windows."
    )

    print(
        "2,900/2,900 campaign-level predictive comparisons available."
    )

    print(
        "READY TO FREEZE THE SUPPORTED-MEMORY / PLATEAU INFERENCE RULE."
    )

else:

    print(
        "\nSTOP — DO NOT APPLY MEMORY SUPPORT RULE."
    )

---
### Imported master block 94

`BLOCK 8.5E-RULE-FREEZE`


In [ ]:
# ================================================================================================================
# BLOCK 8.5E-RULE-FREEZE
# PRE-SPECIFIED SUPPORTED HYDROCLIMATIC MEMORY INFERENCE RULE
#
# PURPOSE:
#   Freeze the inference rules BEFORE applying them to the 50 response × hydro-family curves.
#
# IMPORTANT:
#   - DOES NOT load posterior-effect results
#   - DOES NOT load LOCO results
#   - DOES NOT inspect Fe, Al, NH4, etc.
#   - DOES NOT classify any response
#   - DOES NOT select any memory timescale
#   - DOES NOT fit any model
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import json
import hashlib

print("=" * 120)
print("BLOCK 8.5E-RULE-FREEZE — PRE-SPECIFIED SUPPORTED-MEMORY INFERENCE RULE")
print("=" * 120)

# ================================================================================================================
# 1. OUTPUT DIRECTORY
# ================================================================================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/MEMORY_SPECTRUM"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RULE_CSV = (
    OUT_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.csv"
)

FAMILY_CSV = (
    OUT_DIR /
    "BLOCK85E_HYDRO_FAMILY_REFERENCE_POLICY_v1.csv"
)

CLASS_CSV = (
    OUT_DIR /
    "BLOCK85E_MEMORY_CLASSIFICATION_POLICY_v1.csv"
)

RULE_JSON = (
    OUT_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.json"
)

FREEZE_MANIFEST = (
    OUT_DIR /
    "BLOCK85E_RULE_FREEZE_manifest.json"
)

# ================================================================================================================
# 2. FIXED NUMERICAL THRESHOLDS
# ================================================================================================================

POSTERIOR_SIGN_PROBABILITY_THRESHOLD = 0.95
POSTERIOR_HDI_LEVEL = 0.95

PLATEAU_SE_MULTIPLIER = 1.0

ANTECEDENT_CI_LEVEL = 0.95
ANTECEDENT_CI_ALPHA = 0.05

N_LOCO_CAMPAIGNS_EXPECTED = 10

# ================================================================================================================
# 3. CORE RULES
# ================================================================================================================

rules = pd.DataFrame(
    [
        {
            "rule_id": "R01",
            "rule_name": "posterior_directional_support",
            "criterion":
                "Posterior dominant-sign probability >= 0.95 AND 95% posterior HDI excludes zero.",
            "mathematical_rule":
                "max[P(beta>0), P(beta<0)] >= 0.95 AND (HDI_low > 0 OR HDI_high < 0)",
            "purpose":
                "Require a directionally coherent full-data posterior hydroclimatic effect.",
            "threshold":
                "0.95 sign probability; 95% HDI",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R02",
            "rule_name": "descriptive_predictive_reference",
            "criterion":
                "Within each response x hydro-family, define w* as the window with the highest campaign-equal-weighted mean LOCO LPD.",
            "mathematical_rule":
                "w* = argmax_w mean_c[LPD_c(w)]",
            "purpose":
                "Provide a predictive reference only; w* is NOT interpreted as an exact memory timescale.",
            "threshold":
                "None",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R03",
            "rule_name": "paired_predictive_plateau",
            "criterion":
                "A window belongs to the predictive plateau when its campaign-paired mean LPD loss relative to w* is no greater than one paired standard error.",
            "mathematical_rule":
                "mean_c[Delta_best(w)] >= -1.0 * SE_c[Delta_best(w)], where Delta_best(w)=LPD_c(w)-LPD_c(w*)",
            "purpose":
                "Identify predictively indistinguishable/supportable neighboring timescales without false exact-day precision.",
            "threshold":
                "1 paired SE",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R04",
            "rule_name": "joint_supported_window",
            "criterion":
                "A window is jointly supported only when it satisfies posterior directional support AND belongs to the paired predictive plateau.",
            "mathematical_rule":
                "JOINT_SUPPORT(w) = R01(w) AND R03(w)",
            "purpose":
                "Require both posterior effect evidence and out-of-sample predictive support.",
            "threshold":
                "R01 AND R03",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R05",
            "rule_name": "antecedent_memory_advantage",
            "criterion":
                "For precipitation, surface soil moisture, and root-zone soil moisture only, a longer window can support antecedent memory if it is jointly supported and its paired LOCO improvement over the 1-day reference has a two-sided 95% t confidence interval entirely above zero.",
            "mathematical_rule":
                "w>1 AND JOINT_SUPPORT(w) AND CI95{mean_c[LPD_c(w)-LPD_c(1d)]}.lower > 0",
            "purpose":
                "Distinguish evidence for antecedent information from mere temporal smoothing or a broad predictive plateau.",
            "threshold":
                "paired two-sided 95% t-CI lower bound > 0",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R06",
            "rule_name": "supported_range_reporting",
            "criterion":
                "If multiple windows are jointly supported, report the supported temporal set/range rather than the single descriptive optimum.",
            "mathematical_rule":
                "SUPPORTED_SET = {w : JOINT_SUPPORT(w)}",
            "purpose":
                "Avoid interpreting nested correlated windows as an exact hydrological memory duration.",
            "threshold":
                "None",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R07",
            "rule_name": "antecedent_memory_range_reporting",
            "criterion":
                "When multiple longer windows satisfy the antecedent-memory criterion, report their supported range/set; do not report argmax LPD or argmax |beta| as memory duration.",
            "mathematical_rule":
                "MEMORY_SET = {w>1 : R05(w)}",
            "purpose":
                "Represent uncertainty and nested-timescale equivalence explicitly.",
            "threshold":
                "None",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R08",
            "rule_name": "air_temperature_pet_interpretation",
            "criterion":
                "Air temperature and PET have a shortest available window of 7 days and therefore cannot test antecedent versus instantaneous conditions.",
            "mathematical_rule":
                "reference_window = 7d; no literal instantaneous-memory contrast",
            "purpose":
                "Restrict interpretation to supported temporal-response timescale/range.",
            "threshold":
                "7-day reference",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R09",
            "rule_name": "water_temperature_interpretation",
            "criterion":
                "Water-temperature responses to air temperature/PET are interpreted as thermal-response references rather than evidence of hydrochemical storage memory.",
            "mathematical_rule":
                "thermal_reference != hydrochemical_storage_memory",
            "purpose":
                "Prevent category error between thermal persistence and hydrochemical memory.",
            "threshold":
                "Not applicable",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R10",
            "rule_name": "campaign_pairing",
            "criterion":
                "All predictive window contrasts must be calculated within the same held-out campaign before averaging across campaigns.",
            "mathematical_rule":
                "Delta_c(w1,w2)=LPD_c(w1)-LPD_c(w2)",
            "purpose":
                "Preserve campaign as the independent predictive-validation unit.",
            "threshold":
                "10 paired campaigns expected",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R11",
            "rule_name": "paired_uncertainty",
            "criterion":
                "Uncertainty for predictive differences is computed from the 10 campaign-level paired differences, not from independent window SEs.",
            "mathematical_rule":
                "SE_Delta = SD(Delta_1,...,Delta_10)/sqrt(10)",
            "purpose":
                "Respect pairing and avoid treating nested-window predictive scores as independent.",
            "threshold":
                "n=10 campaigns",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R12",
            "rule_name": "no_cross_response_lpd_ranking",
            "criterion":
                "Absolute standardized-scale LPD values are not compared across different responses.",
            "mathematical_rule":
                "LPD comparisons restricted to windows within the same response and held-out campaign/family architecture",
            "purpose":
                "Avoid invalid comparisons caused by response-specific scaling and likelihood structures.",
            "threshold":
                "Not applicable",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R13",
            "rule_name": "no_argmax_memory",
            "criterion":
                "Neither maximum posterior effect magnitude nor maximum mean LOCO LPD alone defines memory.",
            "mathematical_rule":
                "memory != argmax|beta(w)|; memory != argmax meanLPD(w)",
            "purpose":
                "Prevent circular or overprecise memory-timescale claims.",
            "threshold":
                "Prohibited",
            "memory_selection":
                False,
        },

        {
            "rule_id": "R14",
            "rule_name": "no_connectivity_inference",
            "criterion":
                "Hydrological connectivity is not inferred from the memory spectrum until explicit spatial/network covariates are introduced.",
            "mathematical_rule":
                "memory evidence != connectivity evidence",
            "purpose":
                "Keep H3 separate from temporal-memory inference.",
            "threshold":
                "Not applicable",
            "memory_selection":
                False,
        },
    ]
)

# ================================================================================================================
# 4. HYDRO-FAMILY REFERENCE POLICY
# ================================================================================================================

family_policy = pd.DataFrame(
    [
        {
            "hydro_family": "precipitation",
            "available_windows_days": "1,3,7,14,30,60,90",
            "shortest_reference_days": 1,
            "antecedent_memory_test_allowed": True,
            "interpretation":
                "Antecedent-memory inference allowed relative to 1-day precipitation state.",
        },

        {
            "hydro_family": "surface_soil_moisture",
            "available_windows_days": "1,7,14,30,60,90",
            "shortest_reference_days": 1,
            "antecedent_memory_test_allowed": True,
            "interpretation":
                "Antecedent-memory inference allowed relative to 1-day surface wetness state.",
        },

        {
            "hydro_family": "rootzone_soil_moisture",
            "available_windows_days": "1,7,14,30,60,90",
            "shortest_reference_days": 1,
            "antecedent_memory_test_allowed": True,
            "interpretation":
                "Antecedent-memory inference allowed relative to 1-day root-zone wetness state.",
        },

        {
            "hydro_family": "air_temperature",
            "available_windows_days": "7,14,30,60,90",
            "shortest_reference_days": 7,
            "antecedent_memory_test_allowed": False,
            "interpretation":
                "Supported temporal-response range only; no instantaneous-versus-antecedent test.",
        },

        {
            "hydro_family": "pet",
            "available_windows_days": "7,14,30,60,90",
            "shortest_reference_days": 7,
            "antecedent_memory_test_allowed": False,
            "interpretation":
                "Supported temporal-response range only; no instantaneous-versus-antecedent test.",
        },
    ]
)

# ================================================================================================================
# 5. PRE-SPECIFIED CLASSIFICATION POLICY
# ================================================================================================================

classification_policy = pd.DataFrame(
    [
        {
            "classification": "SUPPORTED_ANTECEDENT_MEMORY",
            "definition":
                "At least one window longer than 1 day satisfies joint posterior-predictive support and has a paired 95% LOCO difference CI entirely above the 1-day reference.",
            "applicable_families":
                "precipitation;surface_soil_moisture;rootzone_soil_moisture",
        },

        {
            "classification": "SUPPORTED_TEMPORAL_RESPONSE_RANGE",
            "definition":
                "One or more windows satisfy joint posterior-predictive support, but the family either lacks an instantaneous/1-day comparison or does not satisfy the stronger antecedent-memory advantage criterion.",
            "applicable_families":
                "all",
        },

        {
            "classification": "SHORT_STATE_RESPONSE",
            "definition":
                "The shortest reference is jointly supported and no longer window demonstrates a positive paired 95% LOCO advantage over the shortest reference.",
            "applicable_families":
                "precipitation;surface_soil_moisture;rootzone_soil_moisture",
        },

        {
            "classification": "NO_ROBUST_SUPPORT",
            "definition":
                "No candidate window simultaneously satisfies posterior directional support and predictive-plateau support.",
            "applicable_families":
                "all",
        },
    ]
)

# ================================================================================================================
# 6. INTERNAL QA — RULE FREEZE ONLY
# ================================================================================================================

gate_dict = {
    "rules_14":
        len(rules) == 14,

    "unique_rule_ids":
        rules["rule_id"].nunique() == 14,

    "posterior_sign_threshold_0_95":
        POSTERIOR_SIGN_PROBABILITY_THRESHOLD == 0.95,

    "posterior_hdi_95":
        POSTERIOR_HDI_LEVEL == 0.95,

    "plateau_one_paired_se":
        PLATEAU_SE_MULTIPLIER == 1.0,

    "antecedent_ci_95":
        ANTECEDENT_CI_LEVEL == 0.95,

    "expected_loco_campaigns_10":
        N_LOCO_CAMPAIGNS_EXPECTED == 10,

    "five_hydro_families":
        len(family_policy) == 5,

    "three_families_allow_antecedent_test":
        int(
            family_policy[
                "antecedent_memory_test_allowed"
            ].sum()
        ) == 3,

    "air_temperature_reference_7d":
        int(
            family_policy.loc[
                family_policy["hydro_family"].eq("air_temperature"),
                "shortest_reference_days"
            ].iloc[0]
        ) == 7,

    "pet_reference_7d":
        int(
            family_policy.loc[
                family_policy["hydro_family"].eq("pet"),
                "shortest_reference_days"
            ].iloc[0]
        ) == 7,

    "three_wetness_precip_references_1d":
        bool(
            family_policy.loc[
                family_policy[
                    "antecedent_memory_test_allowed"
                ],
                "shortest_reference_days"
            ].eq(1).all()
        ),

    "four_classifications":
        len(classification_policy) == 4,

    "no_rule_applied_to_results":
        True,

    "memory_timescale_selected_false":
        True,
}

gate = pd.DataFrame(
    {
        "check": list(gate_dict.keys()),
        "pass": list(gate_dict.values()),
    }
)

RULE_FREEZE_PASS = bool(
    gate["pass"].all()
)

# ================================================================================================================
# 7. WRITE FROZEN RULE FILES
# ================================================================================================================

rules.to_csv(
    RULE_CSV,
    index=False
)

family_policy.to_csv(
    FAMILY_CSV,
    index=False
)

classification_policy.to_csv(
    CLASS_CSV,
    index=False
)

rule_json = {
    "rule_version":
        "SUPPORTED_MEMORY_RULE_v1",

    "frozen_before_application":
        True,

    "posterior_directional_support": {
        "dominant_sign_probability_threshold":
            POSTERIOR_SIGN_PROBABILITY_THRESHOLD,

        "require_95pct_hdi_excludes_zero":
            True,
    },

    "predictive_plateau": {
        "reference":
            "maximum campaign-equal-weighted mean LOCO LPD within response x hydro-family",

        "criterion":
            "mean paired delta LPD versus reference >= -1 paired SE",

        "paired_se_multiplier":
            PLATEAU_SE_MULTIPLIER,

        "exact_best_is_memory":
            False,
    },

    "antecedent_memory": {
        "eligible_families": [
            "precipitation",
            "surface_soil_moisture",
            "rootzone_soil_moisture",
        ],

        "shortest_reference_days":
            1,

        "candidate_window_must_be_longer_than_reference":
            True,

        "require_joint_support":
            True,

        "paired_difference":
            "LPD_campaign(candidate) - LPD_campaign(1d)",

        "confidence_interval":
            "two-sided 95% Student-t CI across 10 paired campaign differences",

        "criterion":
            "lower 95% CI > 0",
    },

    "air_temperature_pet": {
        "shortest_reference_days":
            7,

        "antecedent_memory_test":
            False,

        "interpretation":
            "supported temporal-response range",
    },

    "reporting": {
        "report_supported_set_or_range":
            True,

        "report_argmax_as_exact_memory":
            False,

        "cross_response_absolute_lpd_comparison":
            False,

        "connectivity_inference":
            False,
    },

    "n_expected_loco_campaigns":
        N_LOCO_CAMPAIGNS_EXPECTED,

    "memory_timescale_selected":
        False,
}

with open(
    RULE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rule_json,
        f,
        indent=2
    )

# ================================================================================================================
# 8. HASH FROZEN RULE PRODUCTS
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


freeze_manifest = {
    "block":
        "8.5E-RULE-FREEZE",

    "created_at":
        datetime.now().isoformat(),

    "rule_version":
        "SUPPORTED_MEMORY_RULE_v1",

    "frozen_before_application":
        True,

    "result_files_loaded":
        False,

    "models_fitted":
        False,

    "memory_timescale_selected":
        False,

    "rule_freeze_pass":
        RULE_FREEZE_PASS,

    "files": {
        RULE_CSV.name:
            sha256_file(RULE_CSV),

        FAMILY_CSV.name:
            sha256_file(FAMILY_CSV),

        CLASS_CSV.name:
            sha256_file(CLASS_CSV),

        RULE_JSON.name:
            sha256_file(RULE_JSON),
    },
}

with open(
    FREEZE_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze_manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 9. REPORT
# ================================================================================================================

print("\nFROZEN NUMERICAL RULE")
print("-" * 120)

print(
    "Posterior dominant-sign probability : >=",
    POSTERIOR_SIGN_PROBABILITY_THRESHOLD
)

print(
    "Posterior HDI                       : 95%, must exclude zero"
)

print(
    "Predictive plateau                  : mean paired ΔLPD(best) >= -1 paired SE"
)

print(
    "Antecedent-memory advantage         : paired 95% t-CI ΔLPD(vs 1d) lower bound > 0"
)

print(
    "LOCO campaigns                      :",
    N_LOCO_CAMPAIGNS_EXPECTED
)

print("\nHYDRO-FAMILY POLICY")
print("-" * 120)

print(
    family_policy[
        [
            "hydro_family",
            "shortest_reference_days",
            "antecedent_memory_test_allowed",
        ]
    ].to_string(index=False)
)

print("\nCLASSIFICATION POLICY")
print("-" * 120)

print(
    classification_policy[
        [
            "classification",
            "applicable_families",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 120)
print("8.5E-RULE-FREEZE GATE")
print("=" * 120)

print(
    gate.to_string(index=False)
)

print(
    "\nBLOCK 8.5E-RULE-FREEZE PASS:",
    RULE_FREEZE_PASS
)

print(
    "RESULT FILES LOADED:",
    False
)

print(
    "MEMORY TIMESCALE SELECTED:",
    False
)

print("\nFROZEN FILES")
print("-" * 120)

for p in [
    RULE_CSV,
    FAMILY_CSV,
    CLASS_CSV,
    RULE_JSON,
    FREEZE_MANIFEST,
]:
    print(
        p,
        "| exists:",
        p.exists(),
        "| bytes:",
        p.stat().st_size if p.exists() else None
    )

if RULE_FREEZE_PASS:

    print(
        "\nSUPPORTED-MEMORY RULE FROZEN BEFORE APPLICATION."
    )

    print(
        "NEXT BLOCK MAY APPLY THIS RULE WITHOUT ALTERING THRESHOLDS."
    )

else:

    print(
        "\nSTOP — RULE FREEZE FAILED."
    )

---
### Imported master block 95

`BLOCK 8.5E-INFERENCE`


In [ ]:
# ================================================================================================================
# BLOCK 8.5E-INFERENCE
# APPLICATION OF THE PRE-FROZEN SUPPORTED HYDROCLIMATIC MEMORY RULE
#
# INPUT:
#   - 8.5E joint posterior + LOCO architecture (290 windows)
#   - 8.5D full LOCO folds (2900 campaign-held-out fits)
#   - PRE-FROZEN 8.5E rule files
#
# OUTPUT:
#   - window-level inference: 290 rows
#   - response × hydro-family inference: 50 rows
#   - campaign-paired contrasts
#
# CRITICAL:
#   - NO threshold changes
#   - NO model fitting
#   - NO argmax-as-memory
#   - NO post-hoc variable/window selection
#   - NO connectivity inference
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from scipy.stats import t
import hashlib
import json

print("=" * 120)
print("BLOCK 8.5E-INFERENCE — SUPPORTED HYDROCLIMATIC MEMORY SPECTRUM")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

MEM_DIR = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/MEMORY_SPECTRUM"
)

FULL_RESULTS = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/FULL_LOCO/RESULTS"
)

JOINT_FILE = (
    MEM_DIR /
    "BLOCK85E_JOINT_POSTERIOR_LOCO_ARCHITECTURE_v1.csv"
)

FOLD_FILE = (
    FULL_RESULTS /
    "LOCO_FULL_FOLD_RESULTS_v1.csv"
)

RULE_FILE = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.json"
)

RULE_MANIFEST = (
    MEM_DIR /
    "BLOCK85E_RULE_FREEZE_manifest.json"
)

FAMILY_POLICY_FILE = (
    MEM_DIR /
    "BLOCK85E_HYDRO_FAMILY_REFERENCE_POLICY_v1.csv"
)

for p in [
    JOINT_FILE,
    FOLD_FILE,
    RULE_FILE,
    RULE_MANIFEST,
    FAMILY_POLICY_FILE,
]:
    assert p.exists(), f"Missing required frozen input: {p}"

# ================================================================================================================
# 2. LOAD FROZEN RULE — DO NOT REDEFINE THRESHOLDS
# ================================================================================================================

with open(
    RULE_FILE,
    "r",
    encoding="utf-8"
) as f:
    frozen_rule = json.load(f)

with open(
    RULE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    freeze_manifest = json.load(f)

assert frozen_rule["frozen_before_application"] is True
assert frozen_rule["memory_timescale_selected"] is False
assert freeze_manifest["rule_freeze_pass"] is True
assert freeze_manifest["result_files_loaded"] is False

SIGN_THRESHOLD = float(
    frozen_rule[
        "posterior_directional_support"
    ][
        "dominant_sign_probability_threshold"
    ]
)

PLATEAU_SE_MULTIPLIER = float(
    frozen_rule[
        "predictive_plateau"
    ][
        "paired_se_multiplier"
    ]
)

EXPECTED_CAMPAIGNS = int(
    frozen_rule[
        "n_expected_loco_campaigns"
    ]
)

assert SIGN_THRESHOLD == 0.95
assert PLATEAU_SE_MULTIPLIER == 1.0
assert EXPECTED_CAMPAIGNS == 10

# ================================================================================================================
# 3. LOAD RESULTS
# ================================================================================================================

joint = pd.read_csv(
    JOINT_FILE,
    low_memory=False
)

folds = pd.read_csv(
    FOLD_FILE,
    low_memory=False
)

family_policy = pd.read_csv(
    FAMILY_POLICY_FILE,
    low_memory=False
)

print("\nINPUT ARCHITECTURE")
print("-" * 120)
print("Joint posterior + LOCO windows :", joint.shape)
print("LOCO campaign folds            :", folds.shape)
print("Frozen hydro-family policies   :", family_policy.shape)

assert len(joint) == 290
assert len(folds) == 2900

# ================================================================================================================
# 4. IDENTITY
# ================================================================================================================

keys = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
]

assert joint.duplicated(keys).sum() == 0

fold_identity = (
    folds[
        keys + [
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .copy()
)

assert (
    fold_identity
    .duplicated(
        keys + ["held_out_campaign"]
    )
    .sum()
    == 0
)

assert (
    fold_identity
    .groupby(keys)["held_out_campaign"]
    .nunique()
    .eq(10)
    .all()
)

# ================================================================================================================
# 5. R01 — POSTERIOR DIRECTIONAL SUPPORT
# ================================================================================================================

joint["posterior_dominant_sign_probability"] = (
    joint[
        [
            "posterior_prob_beta_positive",
            "posterior_prob_beta_negative",
        ]
    ]
    .max(axis=1)
)

joint["posterior_hdi_excludes_zero"] = (
    (joint["posterior_beta_hdi_low"] > 0)
    |
    (joint["posterior_beta_hdi_high"] < 0)
)

joint["posterior_directional_support"] = (
    (
        joint[
            "posterior_dominant_sign_probability"
        ]
        >= SIGN_THRESHOLD
    )
    &
    joint[
        "posterior_hdi_excludes_zero"
    ]
)

joint["posterior_effect_direction"] = np.where(
    joint["posterior_beta_mean"] > 0,
    "POSITIVE",
    np.where(
        joint["posterior_beta_mean"] < 0,
        "NEGATIVE",
        "ZERO"
    )
)

# ================================================================================================================
# 6. DESCRIPTIVE BEST WINDOW PER RESPONSE × FAMILY
#    Reference only.
# ================================================================================================================

best = (
    joint
    .sort_values(
        [
            "response_key",
            "hydro_family",
            "mean_campaign_lpd",
            "window_days",
        ],
        ascending=[
            True,
            True,
            False,
            True,
        ]
    )
    .groupby(
        [
            "response_key",
            "hydro_family",
        ],
        as_index=False
    )
    .first()
    [
        [
            "response_key",
            "hydro_family",
            "window_days",
            "hydro_var",
            "mean_campaign_lpd",
        ]
    ]
    .rename(
        columns={
            "window_days":
                "descriptive_best_window_days",

            "hydro_var":
                "descriptive_best_hydro_var",

            "mean_campaign_lpd":
                "descriptive_best_mean_lpd",
        }
    )
)

assert len(best) == 50

# ================================================================================================================
# 7. BUILD CAMPAIGN-PAIRED DELTA VS BEST
# ================================================================================================================

best_campaign = (
    best
    .merge(
        fold_identity,
        left_on=[
            "response_key",
            "hydro_family",
            "descriptive_best_hydro_var",
        ],
        right_on=[
            "response_key",
            "hydro_family",
            "hydro_var",
        ],
        how="left",
        validate="one_to_many",
    )
    [
        [
            "response_key",
            "hydro_family",
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .rename(
        columns={
            "test_mean_lpd":
                "best_reference_lpd"
        }
    )
)

paired_best = (
    fold_identity
    .merge(
        best_campaign,
        on=[
            "response_key",
            "hydro_family",
            "held_out_campaign",
        ],
        how="left",
        validate="many_to_one",
    )
)

paired_best["delta_lpd_vs_best"] = (
    paired_best["test_mean_lpd"]
    -
    paired_best["best_reference_lpd"]
)

# ================================================================================================================
# 8. SUMMARIZE PAIRED DELTA VS BEST
# ================================================================================================================

def paired_summary(
    frame,
    delta_col,
    prefix
):

    rows = []

    for identity, g in frame.groupby(
        keys,
        sort=False
    ):

        x = (
            g[delta_col]
            .astype(float)
            .to_numpy()
        )

        n = len(x)

        assert n == EXPECTED_CAMPAIGNS

        mean = float(
            np.mean(x)
        )

        sd = float(
            np.std(
                x,
                ddof=1
            )
        )

        se = float(
            sd / np.sqrt(n)
        )

        tcrit = float(
            t.ppf(
                0.975,
                df=n - 1
            )
        )

        ci_low = float(
            mean - tcrit * se
        )

        ci_high = float(
            mean + tcrit * se
        )

        row = dict(
            zip(
                keys,
                identity
            )
        )

        row.update(
            {
                f"{prefix}_n":
                    n,

                f"{prefix}_mean":
                    mean,

                f"{prefix}_sd":
                    sd,

                f"{prefix}_se":
                    se,

                f"{prefix}_ci95_low":
                    ci_low,

                f"{prefix}_ci95_high":
                    ci_high,
            }
        )

        rows.append(row)

    return pd.DataFrame(rows)


best_summary = paired_summary(
    paired_best,
    "delta_lpd_vs_best",
    "delta_best"
)

assert len(best_summary) == 290

# ================================================================================================================
# 9. R03 — PREDICTIVE PLATEAU
# ================================================================================================================

joint = (
    joint
    .merge(
        best,
        on=[
            "response_key",
            "hydro_family",
        ],
        how="left",
        validate="many_to_one",
    )
    .merge(
        best_summary,
        on=keys,
        how="left",
        validate="one_to_one",
    )
)

joint["predictive_plateau_support"] = (
    joint["delta_best_mean"]
    >=
    (
        -PLATEAU_SE_MULTIPLIER
        *
        joint["delta_best_se"]
    )
)

# Exact descriptive optimum should satisfy plateau by construction.
joint["is_descriptive_best"] = (
    joint["hydro_var"]
    ==
    joint["descriptive_best_hydro_var"]
)

assert bool(
    joint.loc[
        joint["is_descriptive_best"],
        "predictive_plateau_support"
    ].all()
)

# ================================================================================================================
# 10. R04 — JOINT POSTERIOR + PREDICTIVE SUPPORT
# ================================================================================================================

joint["joint_supported_window"] = (
    joint["posterior_directional_support"]
    &
    joint["predictive_plateau_support"]
)

# ================================================================================================================
# 11. SHORTEST REFERENCES FROM FROZEN POLICY
# ================================================================================================================

shortest_map = dict(
    zip(
        family_policy[
            "hydro_family"
        ],
        family_policy[
            "shortest_reference_days"
        ].astype(int)
    )
)

antecedent_allowed_map = dict(
    zip(
        family_policy[
            "hydro_family"
        ],
        family_policy[
            "antecedent_memory_test_allowed"
        ].astype(bool)
    )
)

joint["shortest_reference_days"] = (
    joint["hydro_family"]
    .map(shortest_map)
)

joint["antecedent_memory_test_allowed"] = (
    joint["hydro_family"]
    .map(antecedent_allowed_map)
)

assert joint["shortest_reference_days"].notna().all()
assert joint["antecedent_memory_test_allowed"].notna().all()

joint["is_shortest_reference"] = (
    joint["window_days"]
    ==
    joint["shortest_reference_days"]
)

assert int(
    joint[
        "is_shortest_reference"
    ].sum()
) == 50

# ================================================================================================================
# 12. BUILD PAIRED DELTA VS SHORTEST
# ================================================================================================================

shortest_identity = (
    joint.loc[
        joint["is_shortest_reference"],
        [
            "response_key",
            "hydro_family",
            "window_days",
            "hydro_var",
        ]
    ]
    .rename(
        columns={
            "window_days":
                "shortest_window_days",

            "hydro_var":
                "shortest_hydro_var",
        }
    )
)

assert len(shortest_identity) == 50

shortest_campaign = (
    shortest_identity
    .merge(
        fold_identity,
        left_on=[
            "response_key",
            "hydro_family",
            "shortest_hydro_var",
        ],
        right_on=[
            "response_key",
            "hydro_family",
            "hydro_var",
        ],
        how="left",
        validate="one_to_many",
    )
    [
        [
            "response_key",
            "hydro_family",
            "held_out_campaign",
            "test_mean_lpd",
        ]
    ]
    .rename(
        columns={
            "test_mean_lpd":
                "shortest_reference_lpd"
        }
    )
)

paired_short = (
    fold_identity
    .merge(
        shortest_campaign,
        on=[
            "response_key",
            "hydro_family",
            "held_out_campaign",
        ],
        how="left",
        validate="many_to_one",
    )
)

paired_short["delta_lpd_vs_shortest"] = (
    paired_short["test_mean_lpd"]
    -
    paired_short["shortest_reference_lpd"]
)

short_summary = paired_summary(
    paired_short,
    "delta_lpd_vs_shortest",
    "delta_short"
)

assert len(short_summary) == 290

joint = (
    joint
    .merge(
        short_summary,
        on=keys,
        how="left",
        validate="one_to_one",
    )
)

# ================================================================================================================
# 13. R05 — STRICT ANTECEDENT-MEMORY ADVANTAGE
# ================================================================================================================

joint["longer_than_shortest"] = (
    joint["window_days"]
    >
    joint["shortest_reference_days"]
)

joint["predictive_advantage_over_shortest"] = (
    joint["delta_short_ci95_low"] > 0
)

joint["supported_antecedent_memory_window"] = (
    joint[
        "antecedent_memory_test_allowed"
    ]
    &
    joint[
        "longer_than_shortest"
    ]
    &
    joint[
        "joint_supported_window"
    ]
    &
    joint[
        "predictive_advantage_over_shortest"
    ]
)

# ================================================================================================================
# 14. FAMILY-LEVEL CLASSIFICATION
# ================================================================================================================

def format_window_set(values):

    values = sorted(
        {
            int(v)
            for v in values
        }
    )

    if not values:
        return ""

    return ",".join(
        map(
            str,
            values
        )
    )


def format_supported_range(values):

    values = sorted(
        {
            int(v)
            for v in values
        }
    )

    if not values:
        return ""

    if len(values) == 1:
        return f"{values[0]} d"

    return f"{values[0]}–{values[-1]} d"


family_rows = []

for (
    response_key,
    hydro_family
), g in joint.groupby(
    [
        "response_key",
        "hydro_family",
    ],
    sort=True
):

    g = g.sort_values(
        "window_days"
    )

    joint_windows = (
        g.loc[
            g[
                "joint_supported_window"
            ],
            "window_days",
        ]
        .astype(int)
        .tolist()
    )

    memory_windows = (
        g.loc[
            g[
                "supported_antecedent_memory_window"
            ],
            "window_days",
        ]
        .astype(int)
        .tolist()
    )

    posterior_windows = (
        g.loc[
            g[
                "posterior_directional_support"
            ],
            "window_days",
        ]
        .astype(int)
        .tolist()
    )

    plateau_windows = (
        g.loc[
            g[
                "predictive_plateau_support"
            ],
            "window_days",
        ]
        .astype(int)
        .tolist()
    )

    shortest_days = int(
        g[
            "shortest_reference_days"
        ].iloc[0]
    )

    shortest_joint = bool(
        g.loc[
            g[
                "window_days"
            ].eq(
                shortest_days
            ),
            "joint_supported_window",
        ].iloc[0]
    )

    antecedent_allowed = bool(
        g[
            "antecedent_memory_test_allowed"
        ].iloc[0]
    )

    # --------------------------------------------------------------------------
    # PRE-FROZEN classification hierarchy
    # --------------------------------------------------------------------------

    if (
        antecedent_allowed
        and
        len(memory_windows) > 0
    ):

        classification = (
            "SUPPORTED_ANTECEDENT_MEMORY"
        )

    elif len(joint_windows) == 0:

        classification = (
            "NO_ROBUST_SUPPORT"
        )

    elif (
        antecedent_allowed
        and
        shortest_joint
        and
        len(memory_windows) == 0
    ):

        classification = (
            "SHORT_STATE_RESPONSE"
        )

    else:

        classification = (
            "SUPPORTED_TEMPORAL_RESPONSE_RANGE"
        )

    best_row = (
        g.loc[
            g[
                "is_descriptive_best"
            ]
        ]
        .iloc[0]
    )

    family_rows.append(
        {
            "response_key":
                response_key,

            "hydro_family":
                hydro_family,

            "classification":
                classification,

            "shortest_reference_days":
                shortest_days,

            "antecedent_memory_test_allowed":
                antecedent_allowed,

            "descriptive_best_window_days":
                int(
                    best_row[
                        "window_days"
                    ]
                ),

            "descriptive_best_mean_lpd":
                float(
                    best_row[
                        "mean_campaign_lpd"
                    ]
                ),

            "posterior_supported_windows":
                format_window_set(
                    posterior_windows
                ),

            "predictive_plateau_windows":
                format_window_set(
                    plateau_windows
                ),

            "joint_supported_windows":
                format_window_set(
                    joint_windows
                ),

            "joint_supported_range":
                format_supported_range(
                    joint_windows
                ),

            "antecedent_memory_windows":
                format_window_set(
                    memory_windows
                ),

            "antecedent_memory_range":
                format_supported_range(
                    memory_windows
                ),

            "shortest_window_jointly_supported":
                shortest_joint,

            "n_posterior_supported_windows":
                len(
                    posterior_windows
                ),

            "n_predictive_plateau_windows":
                len(
                    plateau_windows
                ),

            "n_joint_supported_windows":
                len(
                    joint_windows
                ),

            "n_antecedent_memory_windows":
                len(
                    memory_windows
                ),

            "memory_timescale_selected":
                False,
        }
    )

family_result = pd.DataFrame(
    family_rows
)

assert len(family_result) == 50

# ================================================================================================================
# 15. IMPORTANT: RANGE CONTIGUITY AUDIT
#
# Do not imply all intermediate candidate windows are supported when support is disjoint.
# ================================================================================================================

available_windows = (
    joint
    .groupby(
        "hydro_family"
    )[
        "window_days"
    ]
    .apply(
        lambda x: sorted(
            set(
                x.astype(int)
            )
        )
    )
    .to_dict()
)


def is_contiguous_candidate_set(
    supported,
    available
):

    supported = sorted(
        set(
            supported
        )
    )

    if len(
        supported
    ) <= 1:
        return True

    lo = supported[0]
    hi = supported[-1]

    expected_inside = [
        w
        for w in available
        if lo <= w <= hi
    ]

    return (
        supported
        ==
        expected_inside
    )


contiguity_rows = []

for _, row in family_result.iterrows():

    fam = row[
        "hydro_family"
    ]

    joint_set = (
        []
        if row[
            "joint_supported_windows"
        ] == ""
        else [
            int(x)
            for x in str(
                row[
                    "joint_supported_windows"
                ]
            ).split(",")
        ]
    )

    memory_set = (
        []
        if row[
            "antecedent_memory_windows"
        ] == ""
        else [
            int(x)
            for x in str(
                row[
                    "antecedent_memory_windows"
                ]
            ).split(",")
        ]
    )

    contiguity_rows.append(
        {
            "response_key":
                row[
                    "response_key"
                ],

            "hydro_family":
                fam,

            "joint_support_contiguous":
                is_contiguous_candidate_set(
                    joint_set,
                    available_windows[
                        fam
                    ]
                ),

            "memory_support_contiguous":
                is_contiguous_candidate_set(
                    memory_set,
                    available_windows[
                        fam
                    ]
                ),
        }
    )

contiguity = pd.DataFrame(
    contiguity_rows
)

family_result = (
    family_result
    .merge(
        contiguity,
        on=[
            "response_key",
            "hydro_family",
        ],
        how="left",
        validate="one_to_one",
    )
)

# If support is disjoint, blank the simple range label and preserve explicit set.
family_result.loc[
    ~family_result[
        "joint_support_contiguous"
    ],
    "joint_supported_range"
] = ""

family_result.loc[
    ~family_result[
        "memory_support_contiguous"
    ],
    "antecedent_memory_range"
] = ""

# ================================================================================================================
# 16. CLASSIFICATION COUNTS
# ================================================================================================================

class_counts = (
    family_result[
        "classification"
    ]
    .value_counts()
    .rename_axis(
        "classification"
    )
    .reset_index(
        name="n_response_family_curves"
    )
)

# ================================================================================================================
# 17. FINAL QA
# ================================================================================================================

gate_dict = {

    "joint_windows_290":
        len(joint) == 290,

    "family_curves_50":
        len(family_result) == 50,

    "campaign_folds_2900":
        len(fold_identity) == 2900,

    "all_windows_have_10_campaigns":
        bool(
            fold_identity
            .groupby(keys)[
                "held_out_campaign"
            ]
            .nunique()
            .eq(10)
            .all()
        ),

    "posterior_sign_threshold_frozen_0_95":
        SIGN_THRESHOLD == 0.95,

    "plateau_threshold_frozen_1se":
        PLATEAU_SE_MULTIPLIER == 1.0,

    "posterior_support_boolean_complete":
        joint[
            "posterior_directional_support"
        ].notna().all(),

    "plateau_support_boolean_complete":
        joint[
            "predictive_plateau_support"
        ].notna().all(),

    "joint_support_boolean_complete":
        joint[
            "joint_supported_window"
        ].notna().all(),

    "antecedent_support_boolean_complete":
        joint[
            "supported_antecedent_memory_window"
        ].notna().all(),

    "paired_best_n_10":
        bool(
            joint[
                "delta_best_n"
            ].eq(10).all()
        ),

    "paired_short_n_10":
        bool(
            joint[
                "delta_short_n"
            ].eq(10).all()
        ),

    "descriptive_best_50":
        int(
            joint[
                "is_descriptive_best"
            ].sum()
        ) == 50,

    "shortest_reference_50":
        int(
            joint[
                "is_shortest_reference"
            ].sum()
        ) == 50,

    "antecedent_memory_only_allowed_families":
        bool(
            joint.loc[
                joint[
                    "supported_antecedent_memory_window"
                ],
                "antecedent_memory_test_allowed",
            ].all()
        ),

    "antecedent_memory_only_longer_windows":
        bool(
            joint.loc[
                joint[
                    "supported_antecedent_memory_window"
                ],
                "longer_than_shortest",
            ].all()
        ),

    "antecedent_memory_requires_joint_support":
        bool(
            joint.loc[
                joint[
                    "supported_antecedent_memory_window"
                ],
                "joint_supported_window",
            ].all()
        ),

    "antecedent_memory_requires_ci_above_zero":
        bool(
            (
                joint.loc[
                    joint[
                        "supported_antecedent_memory_window"
                    ],
                    "delta_short_ci95_low",
                ]
                > 0
            ).all()
        ),

    "no_argmax_declared_memory":
        True,

    "no_connectivity_inference":
        True,

    "memory_timescale_selected_false":
        bool(
            ~family_result[
                "memory_timescale_selected"
            ].any()
        ),
}

gate = pd.DataFrame(
    {
        "check":
            list(
                gate_dict.keys()
            ),

        "pass":
            list(
                gate_dict.values()
            ),
    }
)

INFERENCE_PASS = bool(
    gate[
        "pass"
    ].all()
)

# ================================================================================================================
# 18. OUTPUTS
# ================================================================================================================

WINDOW_OUT = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_WINDOW_INFERENCE_v1.csv"
)

FAMILY_OUT = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_FAMILY_INFERENCE_v1.csv"
)

PAIRED_BEST_OUT = (
    MEM_DIR /
    "BLOCK85E_PAIRED_LPD_VS_BEST_INFERENCE_v1.csv"
)

PAIRED_SHORT_OUT = (
    MEM_DIR /
    "BLOCK85E_PAIRED_LPD_VS_SHORTEST_INFERENCE_v1.csv"
)

AUDIT_OUT = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_INFERENCE_audit.xlsx"
)

MANIFEST_OUT = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_INFERENCE_manifest.json"
)

joint.to_csv(
    WINDOW_OUT,
    index=False
)

family_result.to_csv(
    FAMILY_OUT,
    index=False
)

paired_best.to_csv(
    PAIRED_BEST_OUT,
    index=False
)

paired_short.to_csv(
    PAIRED_SHORT_OUT,
    index=False
)

with pd.ExcelWriter(
    AUDIT_OUT,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    family_result.to_excel(
        writer,
        sheet_name="FAMILY_50",
        index=False
    )

    joint.to_excel(
        writer,
        sheet_name="WINDOW_290",
        index=False
    )

    class_counts.to_excel(
        writer,
        sheet_name="CLASS_COUNTS",
        index=False
    )

    paired_best.to_excel(
        writer,
        sheet_name="PAIRED_VS_BEST",
        index=False
    )

    paired_short.to_excel(
        writer,
        sheet_name="PAIRED_VS_SHORTEST",
        index=False
    )

# ================================================================================================================
# 19. MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


manifest = {

    "block":
        "8.5E-INFERENCE",

    "created_at":
        datetime.now().isoformat(),

    "rule_file":
        str(
            RULE_FILE
        ),

    "rule_file_sha256":
        sha256_file(
            RULE_FILE
        ),

    "rule_freeze_manifest_sha256":
        sha256_file(
            RULE_MANIFEST
        ),

    "posterior_sign_threshold":
        SIGN_THRESHOLD,

    "plateau_se_multiplier":
        PLATEAU_SE_MULTIPLIER,

    "paired_ci":
        "two-sided Student-t 95% CI; df=9",

    "n_windows":
        int(
            len(joint)
        ),

    "n_response_family_curves":
        int(
            len(family_result)
        ),

    "classification_counts":
        {
            str(
                row[
                    "classification"
                ]
            ):
            int(
                row[
                    "n_response_family_curves"
                ]
            )

            for _, row
            in class_counts.iterrows()
        },

    "memory_timescale_selected":
        False,

    "exact_argmax_interpreted_as_memory":
        False,

    "connectivity_inferred":
        False,

    "inference_pass":
        INFERENCE_PASS,
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 20. REPORT
# ================================================================================================================

print("\n" + "=" * 120)
print("8.5E-INFERENCE FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print("\nCLASSIFICATION COUNTS")
print("-" * 120)

print(
    class_counts.to_string(
        index=False
    )
)

print("\nSUPPORTED MEMORY / TEMPORAL RESPONSE SPECTRUM")
print("-" * 120)

display_cols = [
    "response_key",
    "hydro_family",
    "classification",
    "descriptive_best_window_days",
    "posterior_supported_windows",
    "predictive_plateau_windows",
    "joint_supported_windows",
    "joint_supported_range",
    "antecedent_memory_windows",
    "antecedent_memory_range",
    "joint_support_contiguous",
    "memory_support_contiguous",
]

print(
    family_result[
        display_cols
    ]
    .sort_values(
        [
            "response_key",
            "hydro_family",
        ]
    )
    .to_string(
        index=False
    )
)

print("\nWINDOW-LEVEL COUNTS")
print("-" * 120)

print(
    "Posterior-supported windows      :",
    int(
        joint[
            "posterior_directional_support"
        ].sum()
    )
)

print(
    "Predictive plateau windows       :",
    int(
        joint[
            "predictive_plateau_support"
        ].sum()
    )
)

print(
    "Joint-supported windows          :",
    int(
        joint[
            "joint_supported_window"
        ].sum()
    )
)

print(
    "Strict antecedent-memory windows :",
    int(
        joint[
            "supported_antecedent_memory_window"
        ].sum()
    )
)

print("\n" + "=" * 120)

print(
    "BLOCK 8.5E-INFERENCE PASS:",
    INFERENCE_PASS
)

print(
    "RULE THRESHOLDS ALTERED:",
    False
)

print(
    "ARGMAX DECLARED AS MEMORY:",
    False
)

print(
    "CONNECTIVITY INFERRED:",
    False
)

if INFERENCE_PASS:

    print(
        "\nSUPPORTED-MEMORY INFERENCE COMPLETE."
    )

    print(
        "Interpret the supported SET/RANGE, not the descriptive optimum alone."
    )

else:

    print(
        "\nSTOP — INFERENCE QA FAILED."
    )

---
### Imported master block 96

`BLOCK S4 / 8.5E-FINALIZE`


In [ ]:
# ================================================================================================================
# BLOCK S4 / 8.5E-FINALIZE
# FREEZE + VERIFY + SNAPSHOT OF SUPPORTED HYDROCLIMATIC MEMORY INFERENCE
#
# DOES NOT:
#   - fit models
#   - recompute posterior inference
#   - change thresholds
#   - change classifications
#   - select an exact memory timescale
#   - infer connectivity
#
# DOES:
#   - verify frozen rule + 8.5E products
#   - verify 290-window / 50-family architecture
#   - verify the strict-memory flags already produced by 8.5E
#   - create a quantitative table of strict antecedent-memory windows
#   - copy all critical products into an immutable-style snapshot
#   - verify source/snapshot SHA256
#   - update a latest-pointer JSON
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import json
import shutil

print("=" * 120)
print("BLOCK S4 / 8.5E-FINALIZE — FREEZE + VERIFY + SNAPSHOT")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

MEM_DIR = (
    PROJECT_ROOT /
    "MEMORY_SPECTRUM"
)

FULL_RESULTS = (
    PROJECT_ROOT /
    "FULL_LOCO" /
    "RESULTS"
)

# 8.5E rule products
RULE_CSV = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.csv"
)

FAMILY_POLICY_CSV = (
    MEM_DIR /
    "BLOCK85E_HYDRO_FAMILY_REFERENCE_POLICY_v1.csv"
)

CLASS_POLICY_CSV = (
    MEM_DIR /
    "BLOCK85E_MEMORY_CLASSIFICATION_POLICY_v1.csv"
)

RULE_JSON = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.json"
)

RULE_FREEZE_MANIFEST = (
    MEM_DIR /
    "BLOCK85E_RULE_FREEZE_manifest.json"
)

# 8.5E architecture products
JOINT_ARCH = (
    MEM_DIR /
    "BLOCK85E_JOINT_POSTERIOR_LOCO_ARCHITECTURE_v1.csv"
)

AUDIT_FINAL_XLSX = (
    MEM_DIR /
    "BLOCK85E_AUDIT_FINAL_joint_architecture.xlsx"
)

AUDIT_FINAL_MANIFEST = (
    MEM_DIR /
    "BLOCK85E_AUDIT_FINAL_manifest.json"
)

# 8.5E inference products
WINDOW_INFERENCE = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_WINDOW_INFERENCE_v1.csv"
)

FAMILY_INFERENCE = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_FAMILY_INFERENCE_v1.csv"
)

PAIRED_BEST = (
    MEM_DIR /
    "BLOCK85E_PAIRED_LPD_VS_BEST_INFERENCE_v1.csv"
)

PAIRED_SHORT = (
    MEM_DIR /
    "BLOCK85E_PAIRED_LPD_VS_SHORTEST_INFERENCE_v1.csv"
)

INFERENCE_AUDIT = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_INFERENCE_audit.xlsx"
)

INFERENCE_MANIFEST = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_INFERENCE_manifest.json"
)

# 8.5D final products retained as provenance
LOCO_FOLDS = (
    FULL_RESULTS /
    "LOCO_FULL_FOLD_RESULTS_v1.csv"
)

LOCO_CURVES = (
    FULL_RESULTS /
    "LOCO_FULL_PREDICTIVE_CURVES_v1.csv"
)

# New S4 derived products
STRICT_TABLE = (
    MEM_DIR /
    "BLOCK85E_STRICT_ANTECEDENT_MEMORY_WINDOWS_v1.csv"
)

S4_MANIFEST = (
    MEM_DIR /
    "BLOCK_S4_85E_FINALIZE_manifest.json"
)

# ================================================================================================================
# 2. REQUIRED INPUT EXISTENCE
# ================================================================================================================

required_files = [
    RULE_CSV,
    FAMILY_POLICY_CSV,
    CLASS_POLICY_CSV,
    RULE_JSON,
    RULE_FREEZE_MANIFEST,
    JOINT_ARCH,
    AUDIT_FINAL_XLSX,
    AUDIT_FINAL_MANIFEST,
    WINDOW_INFERENCE,
    FAMILY_INFERENCE,
    PAIRED_BEST,
    PAIRED_SHORT,
    INFERENCE_AUDIT,
    INFERENCE_MANIFEST,
    LOCO_FOLDS,
    LOCO_CURVES,
]

missing_files = [
    str(p)
    for p in required_files
    if not p.exists()
]

print("\nREQUIRED PRODUCTS")
print("-" * 120)
print("Required files :", len(required_files))
print("Missing files  :", len(missing_files))

if missing_files:
    for x in missing_files:
        print("MISSING:", x)

assert len(missing_files) == 0

# ================================================================================================================
# 3. HASH FUNCTION
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# ================================================================================================================
# 4. LOAD FROZEN PRODUCTS
# ================================================================================================================

window = pd.read_csv(
    WINDOW_INFERENCE,
    low_memory=False
)

family = pd.read_csv(
    FAMILY_INFERENCE,
    low_memory=False
)

loco_folds = pd.read_csv(
    LOCO_FOLDS,
    low_memory=False
)

loco_curves = pd.read_csv(
    LOCO_CURVES,
    low_memory=False
)

with open(
    RULE_JSON,
    "r",
    encoding="utf-8"
) as f:
    rule = json.load(f)

with open(
    RULE_FREEZE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    rule_manifest = json.load(f)

with open(
    INFERENCE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    inference_manifest = json.load(f)

print("\nLOADED ARCHITECTURE")
print("-" * 120)
print("Window inference rows :", len(window))
print("Family inference rows :", len(family))
print("LOCO fold rows        :", len(loco_folds))
print("LOCO curve rows       :", len(loco_curves))

# ================================================================================================================
# 5. VERIFY RULE REALLY PRECEDES APPLICATION
# ================================================================================================================

rule_hash_now = sha256_file(
    RULE_JSON
)

rule_manifest_hash_now = sha256_file(
    RULE_FREEZE_MANIFEST
)

rule_hash_recorded_in_inference = (
    inference_manifest.get(
        "rule_file_sha256"
    )
)

rule_manifest_hash_recorded = (
    inference_manifest.get(
        "rule_freeze_manifest_sha256"
    )
)

rule_hash_match = (
    rule_hash_recorded_in_inference
    ==
    rule_hash_now
)

rule_manifest_hash_match = (
    rule_manifest_hash_recorded
    ==
    rule_manifest_hash_now
)

# ================================================================================================================
# 6. VERIFY FROZEN THRESHOLDS
# ================================================================================================================

sign_threshold = float(
    rule[
        "posterior_directional_support"
    ][
        "dominant_sign_probability_threshold"
    ]
)

plateau_multiplier = float(
    rule[
        "predictive_plateau"
    ][
        "paired_se_multiplier"
    ]
)

expected_campaigns = int(
    rule[
        "n_expected_loco_campaigns"
    ]
)

# ================================================================================================================
# 7. VERIFY ARCHITECTURE
# ================================================================================================================

keys = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
]

window_duplicates = int(
    window
    .duplicated(
        keys
    )
    .sum()
)

family_duplicates = int(
    family
    .duplicated(
        [
            "response_key",
            "hydro_family",
        ]
    )
    .sum()
)

fold_duplicates = int(
    loco_folds
    .duplicated(
        keys + [
            "held_out_campaign"
        ]
    )
    .sum()
)

fold_campaign_counts = (
    loco_folds
    .groupby(
        keys
    )[
        "held_out_campaign"
    ]
    .nunique()
)

# ================================================================================================================
# 8. VERIFY CLASSIFICATION COUNTS AGAINST FROZEN INFERENCE
# ================================================================================================================

class_counts = (
    family[
        "classification"
    ]
    .value_counts()
    .to_dict()
)

expected_class_counts = {
    "SUPPORTED_TEMPORAL_RESPONSE_RANGE": 25,
    "NO_ROBUST_SUPPORT": 12,
    "SHORT_STATE_RESPONSE": 9,
    "SUPPORTED_ANTECEDENT_MEMORY": 4,
}

class_counts_match = (
    class_counts
    ==
    expected_class_counts
)

# ================================================================================================================
# 9. STRICT ANTECEDENT-MEMORY WINDOWS
#
# IMPORTANT:
# This does NOT create a new criterion.
# It merely extracts rows already flagged True by BLOCK 8.5E-INFERENCE.
# ================================================================================================================

strict = (
    window.loc[
        window[
            "supported_antecedent_memory_window"
        ].astype(bool)
    ]
    .copy()
)

strict = strict.sort_values(
    [
        "response_key",
        "hydro_family",
        "window_days",
    ]
).reset_index(drop=True)

# Columns that must exist for later Results writing
strict_required_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "posterior_beta_mean",
    "posterior_beta_hdi_low",
    "posterior_beta_hdi_high",
    "posterior_dominant_sign_probability",
    "mean_campaign_lpd",
    "delta_short_mean",
    "delta_short_se",
    "delta_short_ci95_low",
    "delta_short_ci95_high",
    "posterior_directional_support",
    "predictive_plateau_support",
    "joint_supported_window",
    "supported_antecedent_memory_window",
]

strict_missing_cols = [
    c
    for c in strict_required_cols
    if c not in strict.columns
]

assert len(strict_missing_cols) == 0, (
    f"Missing strict-memory columns: {strict_missing_cols}"
)

strict_table = (
    strict[
        strict_required_cols
    ]
    .copy()
)

strict_table["effect_direction"] = np.where(
    strict_table[
        "posterior_beta_mean"
    ] > 0,
    "POSITIVE",
    "NEGATIVE"
)

# Preserve interpretation role without changing statistical classification.
strict_table["response_interpretation_role"] = np.where(
    strict_table[
        "response_key"
    ].eq(
        "FIELD::Water temperature::degC"
    ),
    "THERMAL_RESPONSE_REFERENCE",
    "HYDROCHEMICAL_RESPONSE"
)

strict_table["exact_timescale_selected"] = False

strict_table.to_csv(
    STRICT_TABLE,
    index=False
)

# ================================================================================================================
# 10. VERIFY STRICT-MEMORY ARCHITECTURE
# ================================================================================================================

strict_count = len(
    strict_table
)

strict_family_count = (
    family[
        "classification"
    ]
    .eq(
        "SUPPORTED_ANTECEDENT_MEMORY"
    )
    .sum()
)

strict_hydrochemical_count = int(
    strict_table[
        "response_interpretation_role"
    ]
    .eq(
        "HYDROCHEMICAL_RESPONSE"
    )
    .sum()
)

strict_thermal_count = int(
    strict_table[
        "response_interpretation_role"
    ]
    .eq(
        "THERMAL_RESPONSE_REFERENCE"
    )
    .sum()
)

# These identities are already implied by the accepted 8.5E output.
# They are used here as freeze-time corruption guards.
expected_strict_identity = {
    (
        "AC::C.E._SOL::dS/m",
        "precipitation",
        14,
    ),
    (
        "AC::C.E._SOL::dS/m",
        "precipitation",
        60,
    ),
    (
        "AC::Na::mg/L",
        "precipitation",
        60,
    ),
    (
        "AC::SO4_ppm::mg/L",
        "precipitation",
        60,
    ),
    (
        "FIELD::Water temperature::degC",
        "rootzone_soil_moisture",
        90,
    ),
}

actual_strict_identity = set(
    zip(
        strict_table[
            "response_key"
        ],
        strict_table[
            "hydro_family"
        ],
        strict_table[
            "window_days"
        ].astype(int),
    )
)

strict_identity_match = (
    actual_strict_identity
    ==
    expected_strict_identity
)

# ================================================================================================================
# 11. VERIFY STRICT WINDOWS SATISFY THE FROZEN RULE
# ================================================================================================================

strict_all_posterior = bool(
    strict_table[
        "posterior_directional_support"
    ].all()
)

strict_all_plateau = bool(
    strict_table[
        "predictive_plateau_support"
    ].all()
)

strict_all_joint = bool(
    strict_table[
        "joint_supported_window"
    ].all()
)

strict_all_ci_above_zero = bool(
    (
        strict_table[
            "delta_short_ci95_low"
        ]
        > 0
    ).all()
)

strict_all_longer_than_1d = bool(
    (
        strict_table[
            "window_days"
        ]
        > 1
    ).all()
)

# ================================================================================================================
# 12. FINAL GATE BEFORE SNAPSHOT
# ================================================================================================================

gate_dict = {

    "required_files_16_present":
        len(missing_files) == 0,

    "rule_freeze_pass_recorded":
        bool(
            rule_manifest[
                "rule_freeze_pass"
            ]
        ),

    "rule_frozen_before_application":
        bool(
            rule[
                "frozen_before_application"
            ]
        ),

    "rule_sha256_matches_inference_manifest":
        rule_hash_match,

    "rule_manifest_sha256_matches_inference_manifest":
        rule_manifest_hash_match,

    "sign_threshold_0_95":
        sign_threshold == 0.95,

    "plateau_multiplier_1se":
        plateau_multiplier == 1.0,

    "expected_campaigns_10":
        expected_campaigns == 10,

    "window_rows_290":
        len(window) == 290,

    "window_duplicates_zero":
        window_duplicates == 0,

    "family_rows_50":
        len(family) == 50,

    "family_duplicates_zero":
        family_duplicates == 0,

    "loco_fold_rows_2900":
        len(loco_folds) == 2900,

    "loco_fold_duplicates_zero":
        fold_duplicates == 0,

    "all_windows_have_10_campaigns":
        bool(
            fold_campaign_counts
            .eq(10)
            .all()
        ),

    "loco_curves_290":
        len(loco_curves) == 290,

    "classification_counts_match":
        class_counts_match,

    "strict_memory_windows_5":
        strict_count == 5,

    "strict_memory_family_curves_4":
        int(
            strict_family_count
        ) == 4,

    "strict_hydrochemical_windows_4":
        strict_hydrochemical_count == 4,

    "strict_thermal_windows_1":
        strict_thermal_count == 1,

    "strict_identity_exact_match":
        strict_identity_match,

    "strict_all_posterior_supported":
        strict_all_posterior,

    "strict_all_predictive_plateau":
        strict_all_plateau,

    "strict_all_joint_supported":
        strict_all_joint,

    "strict_all_ci95_above_zero":
        strict_all_ci_above_zero,

    "strict_all_longer_than_1d":
        strict_all_longer_than_1d,

    "memory_timescale_selected_false":
        bool(
            ~family[
                "memory_timescale_selected"
            ].astype(bool).any()
        ),

    "exact_timescale_selected_false":
        bool(
            ~strict_table[
                "exact_timescale_selected"
            ].any()
        ),

    "connectivity_inference_false":
        inference_manifest.get(
            "connectivity_inferred"
        ) is False,
}

gate = pd.DataFrame(
    {
        "check":
            list(
                gate_dict.keys()
            ),

        "pass":
            list(
                gate_dict.values()
            ),
    }
)

FINALIZE_PASS = bool(
    gate[
        "pass"
    ].all()
)

print("\n" + "=" * 120)
print("8.5E FINALIZE PRE-SNAPSHOT GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

assert FINALIZE_PASS, (
    "STOP — 8.5E finalize QA failed. "
    "No snapshot should be accepted."
)

# ================================================================================================================
# 13. BUILD S4 SNAPSHOT
# ================================================================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

SNAPSHOT_DIR = (
    PROJECT_ROOT /
    "INCREMENTAL_CHECKPOINTS" /
    f"POST_BLOCK85E_{timestamp}"
)

SNAPSHOT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

snapshot_sources = [
    RULE_CSV,
    FAMILY_POLICY_CSV,
    CLASS_POLICY_CSV,
    RULE_JSON,
    RULE_FREEZE_MANIFEST,
    JOINT_ARCH,
    AUDIT_FINAL_XLSX,
    AUDIT_FINAL_MANIFEST,
    WINDOW_INFERENCE,
    FAMILY_INFERENCE,
    PAIRED_BEST,
    PAIRED_SHORT,
    INFERENCE_AUDIT,
    INFERENCE_MANIFEST,
    STRICT_TABLE,
    LOCO_CURVES,
]

copy_records = []

for source in snapshot_sources:

    destination = (
        SNAPSHOT_DIR /
        source.name
    )

    shutil.copy2(
        source,
        destination
    )

    source_hash = sha256_file(
        source
    )

    snapshot_hash = sha256_file(
        destination
    )

    copy_records.append(
        {
            "file":
                source.name,

            "source_path":
                str(source),

            "snapshot_path":
                str(destination),

            "source_size":
                int(
                    source.stat().st_size
                ),

            "snapshot_size":
                int(
                    destination.stat().st_size
                ),

            "source_sha256":
                source_hash,

            "snapshot_sha256":
                snapshot_hash,

            "size_match":
                source.stat().st_size
                ==
                destination.stat().st_size,

            "sha256_match":
                source_hash
                ==
                snapshot_hash,
        }
    )

snapshot_verification = pd.DataFrame(
    copy_records
)

SNAPSHOT_VERIFY_CSV = (
    SNAPSHOT_DIR /
    "S4_SNAPSHOT_VERIFICATION.csv"
)

snapshot_verification.to_csv(
    SNAPSHOT_VERIFY_CSV,
    index=False
)

snapshot_pass = bool(
    snapshot_verification[
        "size_match"
    ].all()
    and
    snapshot_verification[
        "sha256_match"
    ].all()
)

# ================================================================================================================
# 14. S4 MANIFEST
# ================================================================================================================

s4_manifest = {

    "block":
        "S4 / 8.5E-FINALIZE",

    "created_at":
        datetime.now().isoformat(),

    "snapshot":
        str(
            SNAPSHOT_DIR
        ),

    "rule_version":
        "SUPPORTED_MEMORY_RULE_v1",

    "rule_sha256":
        rule_hash_now,

    "rule_freeze_manifest_sha256":
        rule_manifest_hash_now,

    "window_inference_rows":
        int(
            len(window)
        ),

    "family_inference_rows":
        int(
            len(family)
        ),

    "strict_antecedent_memory_windows":
        int(
            strict_count
        ),

    "strict_hydrochemical_windows":
        strict_hydrochemical_count,

    "strict_thermal_reference_windows":
        strict_thermal_count,

    "classification_counts":
        {
            str(k):
                int(v)
            for k, v
            in class_counts.items()
        },

    "thresholds_altered":
        False,

    "models_fitted":
        False,

    "inference_recomputed":
        False,

    "exact_memory_timescale_selected":
        False,

    "connectivity_inferred":
        False,

    "finalize_pass":
        FINALIZE_PASS,

    "snapshot_pass":
        snapshot_pass,

    "snapshot_files":
        int(
            len(
                snapshot_sources
            )
        ),
}

with open(
    S4_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        s4_manifest,
        f,
        indent=2
    )

# Also copy the S4 manifest into snapshot.
S4_MANIFEST_SNAPSHOT = (
    SNAPSHOT_DIR /
    S4_MANIFEST.name
)

shutil.copy2(
    S4_MANIFEST,
    S4_MANIFEST_SNAPSHOT
)

# ================================================================================================================
# 15. LATEST POINTER
# ================================================================================================================

LATEST_POINTER = (
    PROJECT_ROOT /
    "LATEST_POST_BLOCK85E.json"
)

latest_pointer = {
    "created_at":
        datetime.now().isoformat(),

    "block":
        "S4 / 8.5E-FINALIZE",

    "snapshot":
        str(
            SNAPSHOT_DIR
        ),

    "snapshot_pass":
        snapshot_pass,

    "finalize_pass":
        FINALIZE_PASS,

    "strict_memory_table":
        str(
            STRICT_TABLE
        ),

    "s4_manifest":
        str(
            S4_MANIFEST
        ),
}

with open(
    LATEST_POINTER,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        latest_pointer,
        f,
        indent=2
    )

# ================================================================================================================
# 16. FINAL REPORT
# ================================================================================================================

print("\nSTRICT ANTECEDENT-MEMORY WINDOWS")
print("-" * 120)

strict_display_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "posterior_beta_mean",
    "posterior_beta_hdi_low",
    "posterior_beta_hdi_high",
    "posterior_dominant_sign_probability",
    "delta_short_mean",
    "delta_short_ci95_low",
    "delta_short_ci95_high",
    "response_interpretation_role",
]

print(
    strict_table[
        strict_display_cols
    ]
    .to_string(
        index=False
    )
)

print("\nCLASSIFICATION COUNTS")
print("-" * 120)

print(
    pd.DataFrame(
        [
            {
                "classification": k,
                "n": v,
            }
            for k, v
            in class_counts.items()
        ]
    ).to_string(
        index=False
    )
)

print("\nSNAPSHOT VERIFICATION")
print("-" * 120)

print(
    snapshot_verification[
        [
            "file",
            "source_size",
            "snapshot_size",
            "size_match",
            "sha256_match",
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n" + "=" * 120)
print("BLOCK S4 / 8.5E-FINALIZE FINAL GATE")
print("=" * 120)

print(
    "8.5E FINALIZE PASS             :",
    FINALIZE_PASS
)

print(
    "S4 SNAPSHOT PASS              :",
    snapshot_pass
)

print(
    "Window inference rows         :",
    len(window)
)

print(
    "Family inference curves       :",
    len(family)
)

print(
    "Strict antecedent windows     :",
    strict_count
)

print(
    "  Hydrochemical               :",
    strict_hydrochemical_count
)

print(
    "  Thermal reference           :",
    strict_thermal_count
)

print(
    "Thresholds altered            :",
    False
)

print(
    "Models fitted                 :",
    False
)

print(
    "Inference recomputed          :",
    False
)

print(
    "Exact memory timescale chosen :",
    False
)

print(
    "Connectivity inferred         :",
    False
)

print(
    "\nSnapshot:",
    SNAPSHOT_DIR
)

print(
    "Latest pointer:",
    LATEST_POINTER
)

if FINALIZE_PASS and snapshot_pass:

    print(
        "\nBLOCK 8.5E PASS / FROZEN."
    )

    print(
        "POST-8.5E SAVE GAME VERIFIED."
    )

    print(
        "READY FOR QUANTITATIVE RESULTS EXTRACTION + MEMORY-SPECTRUM FIGURE."
    )

else:

    print(
        "\nSTOP — DO NOT PROCEED TO RESULTS WRITING."
    )

---
### Imported master block 97

`BLOCK 8.5F-A`


In [ ]:
# ================================================================================================================
# BLOCK 8.5F-A
# PAPER-READY MEMORY-SPECTRUM RESULTS EXTRACTION + FIGURE-DATA FREEZE
#
# INPUT:
#   Frozen BLOCK 8.5E outputs only.
#
# PURPOSE:
#   1. Convert frozen inference into paper-ready tables.
#   2. Build the exact 10 response x 5 hydro-family figure matrix.
#   3. Preserve supported SETS (including disjoint support).
#   4. Separate hydrochemical evidence from thermal reference.
#
# DOES NOT:
#   - refit models
#   - recompute 8.5E inference
#   - alter thresholds
#   - select a new window
#   - infer connectivity
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import json

print("=" * 120)
print("BLOCK 8.5F-A — PAPER-READY MEMORY-SPECTRUM RESULTS EXTRACTION")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

MEM = ROOT / "MEMORY_SPECTRUM"

WINDOW_FILE = (
    MEM /
    "BLOCK85E_SUPPORTED_MEMORY_WINDOW_INFERENCE_v1.csv"
)

FAMILY_FILE = (
    MEM /
    "BLOCK85E_SUPPORTED_MEMORY_FAMILY_INFERENCE_v1.csv"
)

STRICT_FILE = (
    MEM /
    "BLOCK85E_STRICT_ANTECEDENT_MEMORY_WINDOWS_v1.csv"
)

S4_MANIFEST_FILE = (
    MEM /
    "BLOCK_S4_85E_FINALIZE_manifest.json"
)

RULE_FILE = (
    MEM /
    "BLOCK85E_SUPPORTED_MEMORY_RULE_v1.json"
)

for p in [
    WINDOW_FILE,
    FAMILY_FILE,
    STRICT_FILE,
    S4_MANIFEST_FILE,
    RULE_FILE,
]:
    assert p.exists(), f"Missing frozen input: {p}"

# ================================================================================================================
# 2. LOAD FROZEN DATA
# ================================================================================================================

window = pd.read_csv(
    WINDOW_FILE,
    low_memory=False
)

family = pd.read_csv(
    FAMILY_FILE,
    low_memory=False
)

strict = pd.read_csv(
    STRICT_FILE,
    low_memory=False
)

with open(
    S4_MANIFEST_FILE,
    "r",
    encoding="utf-8"
) as f:
    s4 = json.load(f)

with open(
    RULE_FILE,
    "r",
    encoding="utf-8"
) as f:
    rule = json.load(f)

assert s4["finalize_pass"] is True
assert s4["snapshot_pass"] is True
assert s4["thresholds_altered"] is False
assert s4["exact_memory_timescale_selected"] is False

print("\nFROZEN INPUT")
print("-" * 120)
print("Window rows       :", len(window))
print("Family curves     :", len(family))
print("Strict windows    :", len(strict))

# ================================================================================================================
# 3. FIXED DISPLAY LABELS
#
# Purely editorial labels.
# No scientific classifications are changed.
# ================================================================================================================

response_labels = {
    "AC::pH_SOL::NO_UNIT":
        "pH",

    "AC::Na::mg/L":
        "Na",

    "AC::C.E._SOL::dS/m":
        "Electrical conductivity",

    "COLIFORMS::Total coliforms::NMP/100 mL":
        "Total coliforms",

    "AC::NH4::mg/L":
        "NH₄",

    "METALS::Aluminio::mg/L":
        "Al",

    "METALS::Hierro::mg/L":
        "Fe",

    "METALS::Silicio::mg/L":
        "Si",

    "AC::SO4_ppm::mg/L":
        "SO₄",

    "FIELD::Water temperature::degC":
        "Water temperature",
}

response_order = [
    "Electrical conductivity",
    "Na",
    "SO₄",
    "Al",
    "Fe",
    "Si",
    "NH₄",
    "Total coliforms",
    "pH",
    "Water temperature",
]

family_labels = {
    "precipitation":
        "Precipitation",

    "surface_soil_moisture":
        "Surface soil moisture",

    "rootzone_soil_moisture":
        "Root-zone soil moisture",

    "air_temperature":
        "Air temperature",

    "pet":
        "Potential evaporation",
}

family_order = [
    "Precipitation",
    "Surface soil moisture",
    "Root-zone soil moisture",
    "Air temperature",
    "Potential evaporation",
]

# ================================================================================================================
# 4. VERIFY EXACT RESPONSE/FAMILY ARCHITECTURE
# ================================================================================================================

assert set(
    family["response_key"]
) == set(
    response_labels.keys()
)

assert set(
    family["hydro_family"]
) == set(
    family_labels.keys()
)

assert family[
    [
        "response_key",
        "hydro_family",
    ]
].duplicated().sum() == 0

assert len(family) == 50

# ================================================================================================================
# 5. ADD PAPER LABELS
# ================================================================================================================

family["response_label"] = (
    family["response_key"]
    .map(response_labels)
)

family["hydro_family_label"] = (
    family["hydro_family"]
    .map(family_labels)
)

window["response_label"] = (
    window["response_key"]
    .map(response_labels)
)

window["hydro_family_label"] = (
    window["hydro_family"]
    .map(family_labels)
)

strict["response_label"] = (
    strict["response_key"]
    .map(response_labels)
)

strict["hydro_family_label"] = (
    strict["hydro_family"]
    .map(family_labels)
)

assert family["response_label"].notna().all()
assert family["hydro_family_label"].notna().all()

# ================================================================================================================
# 6. CLASSIFICATION DISPLAY LABELS
#
# Internal frozen class remains untouched.
# These are manuscript-facing descriptions only.
# ================================================================================================================

class_display = {
    "SUPPORTED_ANTECEDENT_MEMORY":
        "Supported antecedent memory",

    "SUPPORTED_TEMPORAL_RESPONSE_RANGE":
        "Supported temporal-response range",

    "SHORT_STATE_RESPONSE":
        "Short-state / no incremental antecedent advantage",

    "NO_ROBUST_SUPPORT":
        "No robust joint support",
}

family["classification_display"] = (
    family["classification"]
    .map(class_display)
)

assert family[
    "classification_display"
].notna().all()

# ================================================================================================================
# 7. RESPONSE INTERPRETATION ROLE
# ================================================================================================================

family["response_interpretation_role"] = np.where(
    family["response_key"].eq(
        "FIELD::Water temperature::degC"
    ),
    "THERMAL_RESPONSE_REFERENCE",
    "HYDROCHEMICAL_RESPONSE"
)

window["response_interpretation_role"] = np.where(
    window["response_key"].eq(
        "FIELD::Water temperature::degC"
    ),
    "THERMAL_RESPONSE_REFERENCE",
    "HYDROCHEMICAL_RESPONSE"
)

# ================================================================================================================
# 8. DIRECTION OF JOINTLY SUPPORTED EFFECTS
#
# We DO NOT summarize direction from unsupported windows.
# ================================================================================================================

joint_direction = (
    window.loc[
        window[
            "joint_supported_window"
        ].astype(bool)
    ]
    .groupby(
        [
            "response_key",
            "hydro_family",
        ]
    )[
        "posterior_beta_mean"
    ]
    .agg(
        [
            "min",
            "max",
        ]
    )
    .reset_index()
)

joint_direction["joint_effect_direction"] = np.select(
    [
        joint_direction["min"] > 0,
        joint_direction["max"] < 0,
    ],
    [
        "POSITIVE",
        "NEGATIVE",
    ],
    default="MIXED"
)

family = (
    family
    .merge(
        joint_direction[
            [
                "response_key",
                "hydro_family",
                "joint_effect_direction",
            ]
        ],
        on=[
            "response_key",
            "hydro_family",
        ],
        how="left",
        validate="one_to_one",
    )
)

family["joint_effect_direction"] = (
    family[
        "joint_effect_direction"
    ]
    .fillna(
        "NO_JOINT_SUPPORT"
    )
)

# ================================================================================================================
# 9. PAPER-FACING SUPPORT TEXT
#
# Preserve explicit sets when support is disjoint.
# ================================================================================================================

def clean_set_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x in [
        "",
        "nan",
        "None",
    ]:
        return ""

    return x


def support_text(row):

    support_set = clean_set_text(
        row[
            "joint_supported_windows"
        ]
    )

    support_range = clean_set_text(
        row[
            "joint_supported_range"
        ]
    )

    if support_set == "":
        return "None"

    if support_range != "":
        return support_range

    # Explicitly preserve disjoint support.
    return "{" + support_set.replace(",", ", ") + "} d"


def memory_text(row):

    memory_set = clean_set_text(
        row[
            "antecedent_memory_windows"
        ]
    )

    memory_range = clean_set_text(
        row[
            "antecedent_memory_range"
        ]
    )

    if memory_set == "":
        return "None"

    if memory_range != "":
        return memory_range

    return "{" + memory_set.replace(",", ", ") + "} d"


family["joint_support_text"] = (
    family.apply(
        support_text,
        axis=1
    )
)

family["antecedent_memory_text"] = (
    family.apply(
        memory_text,
        axis=1
    )
)

# ================================================================================================================
# 10. BUILD 50-CURVE PAPER TABLE
# ================================================================================================================

paper_family = (
    family[
        [
            "response_key",
            "response_label",
            "response_interpretation_role",
            "hydro_family",
            "hydro_family_label",
            "classification",
            "classification_display",
            "joint_effect_direction",
            "descriptive_best_window_days",
            "posterior_supported_windows",
            "predictive_plateau_windows",
            "joint_supported_windows",
            "joint_support_text",
            "antecedent_memory_windows",
            "antecedent_memory_text",
            "joint_support_contiguous",
            "memory_support_contiguous",
            "memory_timescale_selected",
        ]
    ]
    .copy()
)

# ================================================================================================================
# 11. BUILD 290-WINDOW PAPER TABLE
# ================================================================================================================

paper_window_cols = [
    "response_key",
    "response_label",
    "response_interpretation_role",
    "hydro_family",
    "hydro_family_label",
    "window_days",
    "hydro_var",

    "posterior_beta_mean",
    "posterior_beta_hdi_low",
    "posterior_beta_hdi_high",
    "posterior_dominant_sign_probability",

    "mean_campaign_lpd",

    "delta_best_mean",
    "delta_best_se",
    "delta_best_ci95_low",
    "delta_best_ci95_high",

    "delta_short_mean",
    "delta_short_se",
    "delta_short_ci95_low",
    "delta_short_ci95_high",

    "posterior_directional_support",
    "predictive_plateau_support",
    "joint_supported_window",
    "supported_antecedent_memory_window",

    "is_descriptive_best",
    "is_shortest_reference",
]

missing_window_cols = [
    c
    for c in paper_window_cols
    if c not in window.columns
]

assert len(
    missing_window_cols
) == 0, missing_window_cols

paper_window = (
    window[
        paper_window_cols
    ]
    .copy()
)

# ================================================================================================================
# 12. STRICT MEMORY TABLE — ALL 5
# ================================================================================================================

strict_all = (
    strict[
        [
            "response_key",
            "response_label",
            "response_interpretation_role",
            "hydro_family",
            "hydro_family_label",
            "window_days",
            "posterior_beta_mean",
            "posterior_beta_hdi_low",
            "posterior_beta_hdi_high",
            "posterior_dominant_sign_probability",
            "delta_short_mean",
            "delta_short_se",
            "delta_short_ci95_low",
            "delta_short_ci95_high",
            "effect_direction",
        ]
    ]
    .copy()
)

# ================================================================================================================
# 13. STRICT HYDROCHEMICAL TABLE — MAIN PAPER EVIDENCE
# ================================================================================================================

strict_hydrochemical = (
    strict_all.loc[
        strict_all[
            "response_interpretation_role"
        ].eq(
            "HYDROCHEMICAL_RESPONSE"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

assert len(
    strict_hydrochemical
) == 4

# ================================================================================================================
# 14. BUILD 10 x 5 FIGURE MATRIX
#
# Cell contains supported timescale set/range.
# Classification and direction are stored separately.
# ================================================================================================================

figure_matrix = (
    paper_family
    .pivot(
        index="response_label",
        columns="hydro_family_label",
        values="joint_support_text",
    )
    .reindex(
        index=response_order,
        columns=family_order,
    )
)

assert figure_matrix.shape == (
    10,
    5,
)

figure_class_matrix = (
    paper_family
    .pivot(
        index="response_label",
        columns="hydro_family_label",
        values="classification",
    )
    .reindex(
        index=response_order,
        columns=family_order,
    )
)

figure_direction_matrix = (
    paper_family
    .pivot(
        index="response_label",
        columns="hydro_family_label",
        values="joint_effect_direction",
    )
    .reindex(
        index=response_order,
        columns=family_order,
    )
)

# ================================================================================================================
# 15. FIGURE CELL LONG TABLE
#
# This is the safest input for plotting in the next block.
# ================================================================================================================

figure_cells = (
    paper_family[
        [
            "response_key",
            "response_label",
            "response_interpretation_role",
            "hydro_family",
            "hydro_family_label",
            "classification",
            "classification_display",
            "joint_effect_direction",
            "joint_supported_windows",
            "joint_support_text",
            "antecedent_memory_windows",
            "antecedent_memory_text",
            "joint_support_contiguous",
            "memory_support_contiguous",
        ]
    ]
    .copy()
)

figure_cells["response_order"] = (
    figure_cells[
        "response_label"
    ]
    .map(
        {
            x: i
            for i, x
            in enumerate(
                response_order
            )
        }
    )
)

figure_cells["family_order"] = (
    figure_cells[
        "hydro_family_label"
    ]
    .map(
        {
            x: i
            for i, x
            in enumerate(
                family_order
            )
        }
    )
)

figure_cells = (
    figure_cells
    .sort_values(
        [
            "response_order",
            "family_order",
        ]
    )
    .reset_index(
        drop=True
    )
)

# ================================================================================================================
# 16. KEY PAPER COUNTS
# ================================================================================================================

n_posterior_supported = int(
    paper_window[
        "posterior_directional_support"
    ].sum()
)

n_plateau = int(
    paper_window[
        "predictive_plateau_support"
    ].sum()
)

n_joint = int(
    paper_window[
        "joint_supported_window"
    ].sum()
)

n_strict = int(
    paper_window[
        "supported_antecedent_memory_window"
    ].sum()
)

n_strict_hydro = int(
    strict_hydrochemical.shape[0]
)

classification_counts = (
    paper_family[
        "classification"
    ]
    .value_counts()
    .rename_axis(
        "classification"
    )
    .reset_index(
        name="n_curves"
    )
)

# ================================================================================================================
# 17. QA
# ================================================================================================================

expected_strict_hydro = {
    (
        "AC::C.E._SOL::dS/m",
        "precipitation",
        14,
    ),
    (
        "AC::C.E._SOL::dS/m",
        "precipitation",
        60,
    ),
    (
        "AC::Na::mg/L",
        "precipitation",
        60,
    ),
    (
        "AC::SO4_ppm::mg/L",
        "precipitation",
        60,
    ),
}

actual_strict_hydro = set(
    zip(
        strict_hydrochemical[
            "response_key"
        ],
        strict_hydrochemical[
            "hydro_family"
        ],
        strict_hydrochemical[
            "window_days"
        ].astype(int),
    )
)

gate_dict = {

    "s4_finalize_pass":
        s4["finalize_pass"] is True,

    "s4_snapshot_pass":
        s4["snapshot_pass"] is True,

    "window_rows_290":
        len(paper_window) == 290,

    "family_curves_50":
        len(paper_family) == 50,

    "figure_cells_50":
        len(figure_cells) == 50,

    "figure_matrix_10x5":
        figure_matrix.shape == (10, 5),

    "figure_matrix_complete":
        bool(
            figure_matrix
            .notna()
            .all()
            .all()
        ),

    "classification_matrix_complete":
        bool(
            figure_class_matrix
            .notna()
            .all()
            .all()
        ),

    "direction_matrix_complete":
        bool(
            figure_direction_matrix
            .notna()
            .all()
            .all()
        ),

    "posterior_supported_174":
        n_posterior_supported == 174,

    "predictive_plateau_160":
        n_plateau == 160,

    "joint_supported_99":
        n_joint == 99,

    "strict_windows_5":
        n_strict == 5,

    "strict_hydrochemical_4":
        n_strict_hydro == 4,

    "strict_hydrochemical_identity_match":
        actual_strict_hydro
        ==
        expected_strict_hydro,

    "no_exact_memory_selection":
        bool(
            ~paper_family[
                "memory_timescale_selected"
            ].astype(bool).any()
        ),

    "no_connectivity_inference":
        True,
}

gate = pd.DataFrame(
    {
        "check":
            list(
                gate_dict.keys()
            ),

        "pass":
            list(
                gate_dict.values()
            ),
    }
)

PASS = bool(
    gate[
        "pass"
    ].all()
)

# ================================================================================================================
# 18. OUTPUT PATHS
# ================================================================================================================

PAPER_FAMILY_OUT = (
    MEM /
    "BLOCK85F_A_PAPER_MEMORY_FAMILY_TABLE_v1.csv"
)

PAPER_WINDOW_OUT = (
    MEM /
    "BLOCK85F_A_PAPER_MEMORY_WINDOW_TABLE_v1.csv"
)

STRICT_ALL_OUT = (
    MEM /
    "BLOCK85F_A_STRICT_MEMORY_ALL_v1.csv"
)

STRICT_HYDRO_OUT = (
    MEM /
    "BLOCK85F_A_STRICT_HYDROCHEMICAL_MEMORY_v1.csv"
)

FIGURE_CELLS_OUT = (
    MEM /
    "BLOCK85F_A_MEMORY_SPECTRUM_FIGURE_CELLS_v1.csv"
)

FIGURE_MATRIX_OUT = (
    MEM /
    "BLOCK85F_A_MEMORY_SPECTRUM_MATRIX_v1.csv"
)

AUDIT_OUT = (
    MEM /
    "BLOCK85F_A_results_extraction_audit.xlsx"
)

MANIFEST_OUT = (
    MEM /
    "BLOCK85F_A_results_extraction_manifest.json"
)

# ================================================================================================================
# 19. SAVE
# ================================================================================================================

paper_family.to_csv(
    PAPER_FAMILY_OUT,
    index=False
)

paper_window.to_csv(
    PAPER_WINDOW_OUT,
    index=False
)

strict_all.to_csv(
    STRICT_ALL_OUT,
    index=False
)

strict_hydrochemical.to_csv(
    STRICT_HYDRO_OUT,
    index=False
)

figure_cells.to_csv(
    FIGURE_CELLS_OUT,
    index=False
)

figure_matrix.to_csv(
    FIGURE_MATRIX_OUT
)

with pd.ExcelWriter(
    AUDIT_OUT,
    engine="openpyxl"
) as writer:

    gate.to_excel(
        writer,
        sheet_name="FINAL_GATE",
        index=False
    )

    paper_family.to_excel(
        writer,
        sheet_name="FAMILY_50",
        index=False
    )

    paper_window.to_excel(
        writer,
        sheet_name="WINDOW_290",
        index=False
    )

    strict_all.to_excel(
        writer,
        sheet_name="STRICT_ALL_5",
        index=False
    )

    strict_hydrochemical.to_excel(
        writer,
        sheet_name="STRICT_HYDRO_4",
        index=False
    )

    figure_cells.to_excel(
        writer,
        sheet_name="FIGURE_CELLS_50",
        index=False
    )

    figure_matrix.to_excel(
        writer,
        sheet_name="FIGURE_MATRIX_10x5"
    )

    figure_class_matrix.to_excel(
        writer,
        sheet_name="CLASS_MATRIX_10x5"
    )

    figure_direction_matrix.to_excel(
        writer,
        sheet_name="DIRECTION_MATRIX_10x5"
    )

    classification_counts.to_excel(
        writer,
        sheet_name="CLASS_COUNTS",
        index=False
    )

# ================================================================================================================
# 20. HASH + MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


outputs = [
    PAPER_FAMILY_OUT,
    PAPER_WINDOW_OUT,
    STRICT_ALL_OUT,
    STRICT_HYDRO_OUT,
    FIGURE_CELLS_OUT,
    FIGURE_MATRIX_OUT,
    AUDIT_OUT,
]

manifest = {
    "block":
        "8.5F-A",

    "created_at":
        datetime.now().isoformat(),

    "source_s4_snapshot":
        s4.get(
            "snapshot"
        ),

    "s4_finalize_pass":
        s4[
            "finalize_pass"
        ],

    "thresholds_changed":
        False,

    "models_fitted":
        False,

    "inference_recomputed":
        False,

    "memory_timescale_selected":
        False,

    "connectivity_inferred":
        False,

    "n_window_rows":
        int(
            len(
                paper_window
            )
        ),

    "n_family_curves":
        int(
            len(
                paper_family
            )
        ),

    "n_posterior_supported_windows":
        n_posterior_supported,

    "n_predictive_plateau_windows":
        n_plateau,

    "n_joint_supported_windows":
        n_joint,

    "n_strict_antecedent_windows":
        n_strict,

    "n_strict_hydrochemical_windows":
        n_strict_hydro,

    "classification_counts":
        {
            str(
                row[
                    "classification"
                ]
            ):
            int(
                row[
                    "n_curves"
                ]
            )
            for _, row
            in classification_counts.iterrows()
        },

    "pass":
        PASS,

    "files": {
        p.name:
            sha256_file(
                p
            )
        for p in outputs
    },
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 21. REPORT
# ================================================================================================================

print("\n" + "=" * 120)
print("8.5F-A FINAL GATE")
print("=" * 120)

print(
    gate.to_string(
        index=False
    )
)

print("\nPAPER COUNTS")
print("-" * 120)

print(
    "Posterior-supported windows      :",
    n_posterior_supported
)

print(
    "Predictive-plateau windows       :",
    n_plateau
)

print(
    "Joint-supported windows          :",
    n_joint
)

print(
    "Strict antecedent windows        :",
    n_strict
)

print(
    "Strict hydrochemical windows     :",
    n_strict_hydro
)

print("\nSTRICT HYDROCHEMICAL MEMORY — MAIN PAPER EVIDENCE")
print("-" * 120)

print(
    strict_hydrochemical[
        [
            "response_label",
            "hydro_family_label",
            "window_days",
            "posterior_beta_mean",
            "posterior_beta_hdi_low",
            "posterior_beta_hdi_high",
            "posterior_dominant_sign_probability",
            "delta_short_mean",
            "delta_short_ci95_low",
            "delta_short_ci95_high",
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n10 x 5 FIGURE MATRIX — JOINT SUPPORTED TIMESCALES")
print("-" * 120)

print(
    figure_matrix.to_string()
)

print("\nCLASSIFICATION COUNTS")
print("-" * 120)

print(
    classification_counts.to_string(
        index=False
    )
)

print("\nBLOCK 8.5F-A PASS:", PASS)
print("MODELS FITTED:", False)
print("INFERENCE RECOMPUTED:", False)
print("MEMORY TIMESCALE SELECTED:", False)
print("CONNECTIVITY INFERRED:", False)

if PASS:

    print(
        "\nPAPER-READY MEMORY-SPECTRUM DATA FROZEN."
    )

    print(
        "READY FOR BLOCK 8.5F-B — MAIN MEMORY-SPECTRUM FIGURE."
    )

else:

    print(
        "\nSTOP — DO NOT DRAW THE FINAL FIGURE."
    )

---
### Imported master block 98

`BLOCK 8.5F-B — MAIN HYDROCLIMATIC MEMORY-SPECTRUM FIGURE`


In [ ]:
# ================================================================================================================
# BLOCK 8.5F-B — MAIN HYDROCLIMATIC MEMORY-SPECTRUM FIGURE
# ================================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import hashlib
import json
from datetime import datetime

print("=" * 120)
print("BLOCK 8.5F-B — MAIN HYDROCLIMATIC MEMORY-SPECTRUM FIGURE")
print("=" * 120)

ROOT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
MEM = ROOT / "MEMORY_SPECTRUM"

WINDOW_FILE = MEM / "BLOCK85F_A_PAPER_MEMORY_WINDOW_TABLE_v1.csv"
CELLS_FILE  = MEM / "BLOCK85F_A_MEMORY_SPECTRUM_FIGURE_CELLS_v1.csv"
MANIFEST_A  = MEM / "BLOCK85F_A_results_extraction_manifest.json"

for p in [WINDOW_FILE, CELLS_FILE, MANIFEST_A]:
    assert p.exists(), f"Missing frozen input: {p}"

window = pd.read_csv(WINDOW_FILE, low_memory=False)
cells  = pd.read_csv(CELLS_FILE, low_memory=False)

with open(MANIFEST_A, "r", encoding="utf-8") as f:
    manifest_a = json.load(f)

assert manifest_a["pass"] is True
assert len(window) == 290
assert len(cells) == 50

# ------------------------------------------------------------------------------------------------
# DISPLAY ORDER
# ------------------------------------------------------------------------------------------------

response_order = [
    "Electrical conductivity",
    "Na",
    "SO₄",
    "Al",
    "Fe",
    "Si",
    "NH₄",
    "Total coliforms",
    "pH",
    "Water temperature",
]

family_order = [
    "Precipitation",
    "Surface soil moisture",
    "Root-zone soil moisture",
    "Air temperature",
    "Potential evaporation",
]

# Candidate windows represented on one common categorical temporal axis.
all_windows = [1, 3, 7, 14, 30, 60, 90]
xpos = {w: i for i, w in enumerate(all_windows)}

# ------------------------------------------------------------------------------------------------
# BUILD LOOKUP
# ------------------------------------------------------------------------------------------------

window["window_days"] = window["window_days"].astype(int)

lookup = {}

for (resp, fam), g in window.groupby(
    ["response_label", "hydro_family_label"]
):
    lookup[(resp, fam)] = g.copy()

assert len(lookup) == 50

# ------------------------------------------------------------------------------------------------
# QA COUNTS
# ------------------------------------------------------------------------------------------------

n_joint = int(window["joint_supported_window"].astype(bool).sum())
n_strict = int(window["supported_antecedent_memory_window"].astype(bool).sum())

assert n_joint == 99
assert n_strict == 5

# ------------------------------------------------------------------------------------------------
# FIGURE
# ------------------------------------------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=10,
    ncols=5,
    figsize=(15.5, 18),
    sharex=False,
    sharey=False,
)

for r, resp in enumerate(response_order):

    for c, fam in enumerate(family_order):

        ax = axes[r, c]

        g = lookup[(resp, fam)].sort_values("window_days")

        # Candidate windows physically available for this family.
        available = set(g["window_days"].astype(int))

        # Horizontal guide.
        available_x = [xpos[w] for w in all_windows if w in available]

        if available_x:
            ax.plot(
                [min(available_x), max(available_x)],
                [0, 0],
                linewidth=0.7,
                alpha=0.35,
            )

        # ------------------------------------------------------------
        # Candidate windows: tiny neutral points.
        # ------------------------------------------------------------
        for _, row in g.iterrows():

            w = int(row["window_days"])
            x = xpos[w]

            ax.scatter(
                x,
                0,
                s=13,
                marker="o",
                facecolors="none",
                edgecolors="0.72",
                linewidths=0.6,
                zorder=2,
            )

        # ------------------------------------------------------------
        # Jointly supported windows.
        # Shape encodes posterior effect direction.
        #   upward triangle = positive beta
        #   downward triangle = negative beta
        # ------------------------------------------------------------
        supported = g.loc[
            g["joint_supported_window"].astype(bool)
        ]

        for _, row in supported.iterrows():

            w = int(row["window_days"])
            beta = float(row["posterior_beta_mean"])

            marker = "^" if beta > 0 else "v"

            ax.scatter(
                xpos[w],
                0,
                s=78,
                marker=marker,
                zorder=4,
            )

        # ------------------------------------------------------------
        # Strict antecedent-memory windows.
        # Large star overlays joint-support symbol.
        # ------------------------------------------------------------
        strict = g.loc[
            g["supported_antecedent_memory_window"].astype(bool)
        ]

        for _, row in strict.iterrows():

            w = int(row["window_days"])

            ax.scatter(
                xpos[w],
                0,
                s=210,
                marker="*",
                facecolors="none",
                linewidths=1.6,
                zorder=6,
            )

        # ------------------------------------------------------------
        # No robust joint support: small × in cell.
        # ------------------------------------------------------------
        if len(supported) == 0:
            ax.text(
                0.5,
                0.66,
                "×",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=12,
                alpha=0.55,
            )

        # ------------------------------------------------------------
        # Thermal response row visual distinction.
        # ------------------------------------------------------------
        if resp == "Water temperature":
            ax.text(
                0.98,
                0.92,
                "T",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8,
                fontweight="bold",
            )

        # ------------------------------------------------------------
        # Axis formatting
        # ------------------------------------------------------------
        ax.set_ylim(-0.7, 0.7)
        ax.set_xlim(-0.45, len(all_windows) - 0.55)

        ax.set_yticks([])

        # Show temporal ticks on every cell for unambiguous interpretation.
        ax.set_xticks(
            [xpos[w] for w in all_windows]
        )

        labels = [
            str(w) if w in available else ""
            for w in all_windows
        ]

        ax.set_xticklabels(
            labels,
            fontsize=7,
        )

        ax.tick_params(
            axis="x",
            length=2,
            pad=1,
        )

        # Column headers.
        if r == 0:
            ax.set_title(
                fam,
                fontsize=10,
                fontweight="bold",
                pad=10,
            )

        # Row labels.
        if c == 0:
            ax.set_ylabel(
                resp,
                rotation=0,
                ha="right",
                va="center",
                fontsize=10,
                labelpad=12,
            )

# ------------------------------------------------------------------------------------------------
# GLOBAL LABELS
# ------------------------------------------------------------------------------------------------

fig.supxlabel(
    "Antecedent / averaging window (days)",
    fontsize=11,
    y=0.035,
)

fig.suptitle(
    "Hydroclimatic memory spectrum across water-quality responses",
    fontsize=15,
    fontweight="bold",
    y=0.985,
)

# ------------------------------------------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------------------------------------------

legend_handles = [
    Line2D(
        [0], [0],
        marker="^",
        linestyle="None",
        markersize=8,
        label="Joint support, positive β",
    ),
    Line2D(
        [0], [0],
        marker="v",
        linestyle="None",
        markersize=8,
        label="Joint support, negative β",
    ),
    Line2D(
        [0], [0],
        marker="*",
        markerfacecolor="none",
        linestyle="None",
        markersize=13,
        label="Strict antecedent-memory support",
    ),
    Line2D(
        [0], [0],
        marker="o",
        markerfacecolor="none",
        markeredgecolor="0.65",
        linestyle="None",
        markersize=5,
        label="Candidate window",
    ),
    Line2D(
        [0], [0],
        marker="$×$",
        linestyle="None",
        markersize=8,
        label="No robust joint support",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=5,
    frameon=False,
    fontsize=9,
)

plt.tight_layout(
    rect=[
        0.08,
        0.06,
        1,
        0.965,
    ]
)

# ------------------------------------------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------------------------------------------

SVG_OUT = MEM / "FIGURE_MEMORY_SPECTRUM_v1.svg"
PDF_OUT = MEM / "FIGURE_MEMORY_SPECTRUM_v1.pdf"
PNG_OUT = MEM / "FIGURE_MEMORY_SPECTRUM_v1.png"

fig.savefig(
    SVG_OUT,
    bbox_inches="tight",
)

fig.savefig(
    PDF_OUT,
    bbox_inches="tight",
)

fig.savefig(
    PNG_OUT,
    dpi=600,
    bbox_inches="tight",
)

plt.show()

# ------------------------------------------------------------------------------------------------
# OUTPUT QA
# ------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


qa = pd.DataFrame(
    {
        "check": [
            "input_window_rows_290",
            "input_cells_50",
            "response_rows_10",
            "hydro_families_5",
            "joint_supported_windows_99",
            "strict_windows_5",
            "svg_exists",
            "pdf_exists",
            "png_exists",
            "models_fitted_false",
            "inference_recomputed_false",
            "memory_timescale_selected_false",
            "connectivity_inferred_false",
        ],
        "pass": [
            len(window) == 290,
            len(cells) == 50,
            len(response_order) == 10,
            len(family_order) == 5,
            n_joint == 99,
            n_strict == 5,
            SVG_OUT.exists(),
            PDF_OUT.exists(),
            PNG_OUT.exists(),
            True,
            True,
            True,
            True,
        ],
    }
)

PASS = bool(qa["pass"].all())

MANIFEST_OUT = MEM / "BLOCK85F_B_memory_spectrum_figure_manifest.json"

manifest = {
    "block": "8.5F-B",
    "created_at": datetime.now().isoformat(),
    "source_block": "8.5F-A",
    "figure_design":
        "10x5 temporal-spectrum matrix; candidate windows + joint posterior/LOCO support + strict antecedent-memory overlay",
    "n_joint_supported_windows": n_joint,
    "n_strict_antecedent_windows": n_strict,
    "models_fitted": False,
    "inference_recomputed": False,
    "memory_timescale_selected": False,
    "connectivity_inferred": False,
    "pass": PASS,
    "files": {
        SVG_OUT.name: sha256_file(SVG_OUT),
        PDF_OUT.name: sha256_file(PDF_OUT),
        PNG_OUT.name: sha256_file(PNG_OUT),
    },
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print("\n" + "=" * 120)
print("8.5F-B FIGURE QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

print("\nOUTPUT FILES")
print("-" * 120)

for p in [
    SVG_OUT,
    PDF_OUT,
    PNG_OUT,
    MANIFEST_OUT,
]:
    print(
        p.name,
        "| exists:",
        p.exists(),
        "| bytes:",
        p.stat().st_size if p.exists() else None
    )

print("\nBLOCK 8.5F-B PASS:", PASS)
print("MODELS FITTED:", False)
print("INFERENCE RECOMPUTED:", False)
print("MEMORY TIMESCALE SELECTED:", False)
print("CONNECTIVITY INFERRED:", False)

if PASS:
    print(
        "\nMAIN MEMORY-SPECTRUM FIGURE GENERATED."
    )
else:
    print(
        "\nSTOP — FIGURE QA FAILED."
    )

---
### Imported master block 99

`BLOCK 8.5F-C — PUBLICATION FIGURE + MAIN/SUPPLEMENTARY TABLES`


In [ ]:
# ================================================================================================================
# BLOCK 8.5F-C — PUBLICATION FIGURE + MAIN/SUPPLEMENTARY TABLES
# FINAL EDITORIAL FREEZE OF HYDROCLIMATIC MEMORY SPECTRUM
#
# INPUTS:
#   Frozen BLOCK 8.5F-A products only.
#
# OUTPUTS:
#   Figure: SVG + PDF + PNG 600 dpi
#   Table 1: strict hydrochemical antecedent-memory evidence (main manuscript)
#   Table S1: all 50 response × hydro-family curves
#   Table S2: all 290 response × hydroclimate-window results
#   Excel workbook + QA + manifest
#
# DOES NOT:
#   - fit models
#   - recompute inference
#   - change thresholds
#   - change classifications
#   - choose an exact memory timescale
#   - infer connectivity
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import hashlib
import json

print("=" * 120)
print("BLOCK 8.5F-C — PUBLICATION FIGURE + MANUSCRIPT/SUPPLEMENTARY TABLES")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

MEM = ROOT / "MEMORY_SPECTRUM"

FAMILY_FILE = (
    MEM /
    "BLOCK85F_A_PAPER_MEMORY_FAMILY_TABLE_v1.csv"
)

WINDOW_FILE = (
    MEM /
    "BLOCK85F_A_PAPER_MEMORY_WINDOW_TABLE_v1.csv"
)

STRICT_FILE = (
    MEM /
    "BLOCK85F_A_STRICT_HYDROCHEMICAL_MEMORY_v1.csv"
)

FIGURE_CELLS_FILE = (
    MEM /
    "BLOCK85F_A_MEMORY_SPECTRUM_FIGURE_CELLS_v1.csv"
)

MANIFEST_A_FILE = (
    MEM /
    "BLOCK85F_A_results_extraction_manifest.json"
)

for p in [
    FAMILY_FILE,
    WINDOW_FILE,
    STRICT_FILE,
    FIGURE_CELLS_FILE,
    MANIFEST_A_FILE,
]:
    assert p.exists(), f"Missing frozen input: {p}"

# ================================================================================================================
# 2. LOAD FROZEN PRODUCTS
# ================================================================================================================

family = pd.read_csv(
    FAMILY_FILE,
    low_memory=False
)

window = pd.read_csv(
    WINDOW_FILE,
    low_memory=False
)

strict = pd.read_csv(
    STRICT_FILE,
    low_memory=False
)

figure_cells = pd.read_csv(
    FIGURE_CELLS_FILE,
    low_memory=False
)

with open(
    MANIFEST_A_FILE,
    "r",
    encoding="utf-8"
) as f:
    manifest_a = json.load(f)

assert manifest_a["pass"] is True
assert manifest_a["thresholds_changed"] is False
assert manifest_a["models_fitted"] is False
assert manifest_a["inference_recomputed"] is False
assert manifest_a["memory_timescale_selected"] is False

assert len(family) == 50
assert len(window) == 290
assert len(strict) == 4
assert len(figure_cells) == 50

print("\nFROZEN INPUT")
print("-" * 120)
print("Family curves              :", len(family))
print("Window-level results       :", len(window))
print("Strict hydrochemical rows  :", len(strict))
print("Figure cells               :", len(figure_cells))

# ================================================================================================================
# 3. FIXED DISPLAY ORDER
# ================================================================================================================

response_order = [
    "Electrical conductivity",
    "Na",
    "SO₄",
    "Al",
    "Fe",
    "Si",
    "NH₄",
    "Total coliforms",
    "pH",
    "Water temperature",
]

family_order = [
    "Precipitation",
    "Surface soil moisture",
    "Root-zone soil moisture",
    "Air temperature",
    "Potential evaporation",
]

all_windows = [
    1,
    3,
    7,
    14,
    30,
    60,
    90,
]

xpos = {
    w: i
    for i, w in enumerate(all_windows)
}

# ================================================================================================================
# 4. EXACT ARCHITECTURE QA
# ================================================================================================================

assert set(
    family["response_label"]
) == set(response_order)

assert set(
    family["hydro_family_label"]
) == set(family_order)

assert family[
    [
        "response_label",
        "hydro_family_label",
    ]
].duplicated().sum() == 0

assert window[
    [
        "response_key",
        "hydro_family",
        "window_days",
        "hydro_var",
    ]
].duplicated().sum() == 0

window["window_days"] = (
    window["window_days"]
    .astype(int)
)

n_joint = int(
    window[
        "joint_supported_window"
    ]
    .astype(bool)
    .sum()
)

n_strict_all = int(
    window[
        "supported_antecedent_memory_window"
    ]
    .astype(bool)
    .sum()
)

assert n_joint == 99
assert n_strict_all == 5

# ================================================================================================================
# 5. BUILD LOOKUP FOR FIGURE
# ================================================================================================================

lookup = {}

for (
    response_label,
    hydro_family_label
), g in window.groupby(
    [
        "response_label",
        "hydro_family_label",
    ]
):

    lookup[
        (
            response_label,
            hydro_family_label,
        )
    ] = (
        g
        .sort_values(
            "window_days"
        )
        .copy()
    )

assert len(lookup) == 50

# ================================================================================================================
# 6. PUBLICATION FIGURE
#
# ENCODING:
#   small open circle = candidate window
#   ▲ = jointly supported positive posterior effect
#   ▼ = jointly supported negative posterior effect
#   ★ = strict antecedent-memory support
#   NS = no robust joint support for response × driver curve
#   T = water-temperature thermal reference
#
# IMPORTANT:
#   No color scale represents effect magnitude.
#   Shape carries scientific meaning.
# ================================================================================================================

fig, axes = plt.subplots(
    nrows=10,
    ncols=5,
    figsize=(13.8, 14.2),
    sharex=False,
    sharey=False,
)

for r, response_label in enumerate(
    response_order
):

    for c, hydro_family_label in enumerate(
        family_order
    ):

        ax = axes[r, c]

        g = lookup[
            (
                response_label,
                hydro_family_label,
            )
        ].copy()

        available = set(
            g[
                "window_days"
            ].astype(int)
        )

        available_x = [
            xpos[w]
            for w in all_windows
            if w in available
        ]

        # ------------------------------------------------------------------------------------------
        # Baseline connecting candidate windows
        # ------------------------------------------------------------------------------------------

        if len(available_x) > 1:

            ax.plot(
                [
                    min(available_x),
                    max(available_x),
                ],
                [0, 0],
                linewidth=0.65,
                alpha=0.30,
                zorder=1,
            )

        # ------------------------------------------------------------------------------------------
        # Candidate windows
        # ------------------------------------------------------------------------------------------

        for _, row in g.iterrows():

            w = int(
                row[
                    "window_days"
                ]
            )

            ax.scatter(
                xpos[w],
                0,
                s=12,
                marker="o",
                facecolors="white",
                edgecolors="0.68",
                linewidths=0.55,
                zorder=2,
            )

        # ------------------------------------------------------------------------------------------
        # Joint support
        # ------------------------------------------------------------------------------------------

        supported = g.loc[
            g[
                "joint_supported_window"
            ].astype(bool)
        ].copy()

        for _, row in supported.iterrows():

            w = int(
                row[
                    "window_days"
                ]
            )

            beta = float(
                row[
                    "posterior_beta_mean"
                ]
            )

            marker = (
                "^"
                if beta > 0
                else "v"
            )

            # Single neutral dark tone.
            # Shape, not color, carries sign.
            ax.scatter(
                xpos[w],
                0,
                s=62,
                marker=marker,
                facecolors="0.18",
                edgecolors="0.18",
                linewidths=0.7,
                zorder=4,
            )

        # ------------------------------------------------------------------------------------------
        # Strict antecedent-memory support
        #
        # Star overlay around the same x-position.
        # ------------------------------------------------------------------------------------------

        strict_g = g.loc[
            g[
                "supported_antecedent_memory_window"
            ].astype(bool)
        ].copy()

        for _, row in strict_g.iterrows():

            w = int(
                row[
                    "window_days"
                ]
            )

            ax.scatter(
                xpos[w],
                0,
                s=205,
                marker="*",
                facecolors="white",
                edgecolors="0.05",
                linewidths=1.25,
                zorder=5,
            )

            # Re-overlay sign symbol in center so direction remains visible.
            beta = float(
                row[
                    "posterior_beta_mean"
                ]
            )

            marker = (
                "^"
                if beta > 0
                else "v"
            )

            ax.scatter(
                xpos[w],
                0,
                s=42,
                marker=marker,
                facecolors="0.05",
                edgecolors="0.05",
                linewidths=0.5,
                zorder=6,
            )

        # ------------------------------------------------------------------------------------------
        # No robust joint support
        # ------------------------------------------------------------------------------------------

        if len(supported) == 0:

            ax.text(
                0.94,
                0.82,
                "NS",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=7.3,
                fontweight="bold",
                alpha=0.65,
            )

        # ------------------------------------------------------------------------------------------
        # Thermal-reference row
        # ------------------------------------------------------------------------------------------

        if response_label == "Water temperature":

            ax.text(
                0.04,
                0.82,
                "T",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=7.5,
                fontweight="bold",
            )

        # ------------------------------------------------------------------------------------------
        # Panel formatting
        # ------------------------------------------------------------------------------------------

        ax.set_xlim(
            -0.45,
            len(all_windows) - 0.55
        )

        ax.set_ylim(
            -0.52,
            0.52
        )

        ax.set_yticks([])

        ax.set_xticks(
            [
                xpos[w]
                for w in all_windows
            ]
        )

        # Only bottom row gets day labels.
        if r == len(response_order) - 1:

            labels = [
                str(w)
                if w in available
                else ""
                for w in all_windows
            ]

            ax.set_xticklabels(
                labels,
                fontsize=7.3,
            )

            ax.tick_params(
                axis="x",
                length=2.2,
                pad=2,
            )

        else:

            ax.set_xticklabels([])

            ax.tick_params(
                axis="x",
                length=0,
            )

        # Column titles
        if r == 0:

            ax.set_title(
                hydro_family_label,
                fontsize=9.5,
                fontweight="bold",
                pad=8,
            )

        # Row labels
        if c == 0:

            ax.set_ylabel(
                response_label,
                rotation=0,
                ha="right",
                va="center",
                fontsize=9.2,
                labelpad=11,
            )

        # Thin publication-style panel frame.
        for spine in ax.spines.values():
            spine.set_linewidth(0.55)
            spine.set_alpha(0.65)

# ================================================================================================================
# 7. FIGURE GLOBAL LABELS
# ================================================================================================================

fig.suptitle(
    "Hydroclimatic memory spectrum across water-quality responses",
    fontsize=14.5,
    fontweight="bold",
    y=0.985,
)

fig.supxlabel(
    "Antecedent / averaging window (days)",
    fontsize=10.2,
    y=0.050,
)

legend_handles = [

    Line2D(
        [0],
        [0],
        marker="^",
        linestyle="None",
        markerfacecolor="0.18",
        markeredgecolor="0.18",
        markersize=7,
        label="Joint support, positive β",
    ),

    Line2D(
        [0],
        [0],
        marker="v",
        linestyle="None",
        markerfacecolor="0.18",
        markeredgecolor="0.18",
        markersize=7,
        label="Joint support, negative β",
    ),

    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="0.05",
        markeredgewidth=1.1,
        markersize=11,
        label="Strict antecedent-memory support",
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="0.68",
        markersize=4.5,
        label="Candidate window",
    ),

    Line2D(
        [0],
        [0],
        linestyle="None",
        marker="",
        label="NS = no robust joint support",
    ),

    Line2D(
        [0],
        [0],
        linestyle="None",
        marker="",
        label="T = thermal-response reference",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(
        0.5,
        0.006
    ),
    ncol=3,
    frameon=False,
    fontsize=8.1,
    columnspacing=1.6,
    handletextpad=0.5,
)

plt.tight_layout(
    rect=[
        0.075,
        0.075,
        1.0,
        0.965,
    ],
    h_pad=0.65,
    w_pad=0.55,
)

# ================================================================================================================
# 8. FIGURE OUTPUTS
# ================================================================================================================

FIG_SVG = (
    MEM /
    "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.svg"
)

FIG_PDF = (
    MEM /
    "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.pdf"
)

FIG_PNG = (
    MEM /
    "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.png"
)

fig.savefig(
    FIG_SVG,
    bbox_inches="tight"
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight"
)

fig.savefig(
    FIG_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# ================================================================================================================
# 9. MAIN MANUSCRIPT TABLE
#
# Strict hydrochemical antecedent-memory evidence only.
# This is deliberately NOT a table of "best windows".
# ================================================================================================================

main_table = strict[
    [
        "response_label",
        "hydro_family_label",
        "window_days",
        "posterior_beta_mean",
        "posterior_beta_hdi_low",
        "posterior_beta_hdi_high",
        "posterior_dominant_sign_probability",
        "delta_short_mean",
        "delta_short_ci95_low",
        "delta_short_ci95_high",
    ]
].copy()

main_table = main_table.rename(
    columns={
        "response_label":
            "Response",

        "hydro_family_label":
            "Hydroclimatic driver",

        "window_days":
            "Window (days)",

        "posterior_beta_mean":
            "Posterior beta",

        "posterior_beta_hdi_low":
            "Beta HDI 95% low",

        "posterior_beta_hdi_high":
            "Beta HDI 95% high",

        "posterior_dominant_sign_probability":
            "P(dominant sign)",

        "delta_short_mean":
            "Delta LPD vs shortest",

        "delta_short_ci95_low":
            "Delta LPD 95% CI low",

        "delta_short_ci95_high":
            "Delta LPD 95% CI high",
    }
)

# Round only manuscript-facing display table.
main_table_display = (
    main_table.copy()
)

numeric_cols_main = [
    "Posterior beta",
    "Beta HDI 95% low",
    "Beta HDI 95% high",
    "P(dominant sign)",
    "Delta LPD vs shortest",
    "Delta LPD 95% CI low",
    "Delta LPD 95% CI high",
]

for col in numeric_cols_main:
    main_table_display[col] = (
        main_table_display[col]
        .round(3)
    )

# ================================================================================================================
# 10. SUPPLEMENTARY TABLE S1 — ALL 50 CURVES
# ================================================================================================================

table_s1 = family[
    [
        "response_label",
        "response_interpretation_role",
        "hydro_family_label",
        "classification",
        "classification_display",
        "joint_effect_direction",
        "descriptive_best_window_days",
        "posterior_supported_windows",
        "predictive_plateau_windows",
        "joint_supported_windows",
        "joint_support_text",
        "antecedent_memory_windows",
        "antecedent_memory_text",
        "joint_support_contiguous",
        "memory_support_contiguous",
    ]
].copy()

table_s1 = table_s1.rename(
    columns={
        "response_label":
            "Response",

        "response_interpretation_role":
            "Interpretation role",

        "hydro_family_label":
            "Hydroclimatic driver",

        "classification":
            "Frozen classification",

        "classification_display":
            "Manuscript classification",

        "joint_effect_direction":
            "Direction of jointly supported effect",

        "descriptive_best_window_days":
            "Descriptive best window (days; not memory estimate)",

        "posterior_supported_windows":
            "Posterior-supported windows",

        "predictive_plateau_windows":
            "Predictive-plateau windows",

        "joint_supported_windows":
            "Joint-supported windows",

        "joint_support_text":
            "Joint-supported set/range",

        "antecedent_memory_windows":
            "Strict antecedent-memory windows",

        "antecedent_memory_text":
            "Strict antecedent-memory set/range",

        "joint_support_contiguous":
            "Joint support contiguous",

        "memory_support_contiguous":
            "Memory support contiguous",
    }
)

# ================================================================================================================
# 11. SUPPLEMENTARY TABLE S2 — ALL 290 WINDOWS
#
# Full transparent inference trace.
# ================================================================================================================

table_s2 = window.copy()

# Keep all frozen columns; add concise manuscript identifiers first.
front_cols = [
    "response_label",
    "hydro_family_label",
    "window_days",
    "hydro_var",
]

remaining_cols = [
    c
    for c in table_s2.columns
    if c not in front_cols
]

table_s2 = table_s2[
    front_cols
    +
    remaining_cols
]

# ================================================================================================================
# 12. CAPTION + TABLE NOTES
# ================================================================================================================

figure_caption = (
    "Hydroclimatic memory spectrum across water-quality responses. "
    "Open circles denote pre-specified candidate antecedent or averaging windows. "
    "Upward and downward triangles denote windows with joint posterior and campaign-held-out "
    "predictive support for positive and negative effects, respectively. "
    "Stars identify windows meeting the pre-specified strict antecedent-memory criterion, "
    "which required posterior directional support, predictive-plateau support, and a positive "
    "paired campaign-held-out predictive advantage relative to the shortest reference window. "
    "NS denotes response-driver combinations without robust joint support. "
    "Water temperature (T) is retained as a thermal-response reference and is not interpreted "
    "as primary hydrochemical-memory evidence. Supported sets rather than descriptive optima "
    "should be interpreted as the temporal evidence."
)

main_table_note = (
    "Only hydrochemical windows satisfying the pre-specified strict antecedent-memory criterion "
    "are shown. Posterior beta values are standardized effects. Delta LPD is the campaign-paired "
    "difference in held-out log predictive density relative to the 1-day reference for precipitation. "
    "A positive 95% CI entirely above zero was required for strict antecedent-memory support. "
    "The listed candidate window is not interpreted as an exact physical memory duration."
)

s1_note = (
    "Table S1 reports all 50 response-by-driver temporal curves. "
    "The descriptive best window is retained only as a predictive reference and must not be interpreted "
    "as an estimated memory timescale. Disjoint supported sets are reported explicitly rather than "
    "converted to continuous ranges."
)

s2_note = (
    "Table S2 provides the complete 290-window inference trace used to construct the hydroclimatic "
    "memory spectrum, including posterior effects, campaign-held-out predictive quantities, "
    "joint-support flags, and strict antecedent-memory flags."
)

# ================================================================================================================
# 13. SAVE TABLES
# ================================================================================================================

MAIN_TABLE_CSV = (
    MEM /
    "TABLE_MAIN_STRICT_HYDROCHEMICAL_MEMORY_v1.csv"
)

S1_CSV = (
    MEM /
    "TABLE_S1_MEMORY_RESPONSE_DRIVER_CURVES_v1.csv"
)

S2_CSV = (
    MEM /
    "TABLE_S2_MEMORY_WINDOW_FULL_RESULTS_v1.csv"
)

CAPTION_TXT = (
    MEM /
    "FIGURE_MEMORY_SPECTRUM_caption_v1.txt"
)

NOTES_TXT = (
    MEM /
    "MEMORY_TABLE_NOTES_v1.txt"
)

WORKBOOK = (
    MEM /
    "MEMORY_SPECTRUM_MANUSCRIPT_TABLES_v1.xlsx"
)

main_table_display.to_csv(
    MAIN_TABLE_CSV,
    index=False
)

table_s1.to_csv(
    S1_CSV,
    index=False
)

table_s2.to_csv(
    S2_CSV,
    index=False
)

with open(
    CAPTION_TXT,
    "w",
    encoding="utf-8"
) as f:
    f.write(
        figure_caption
    )

with open(
    NOTES_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MAIN TABLE NOTE\n"
        +
        main_table_note
        +
        "\n\nTABLE S1 NOTE\n"
        +
        s1_note
        +
        "\n\nTABLE S2 NOTE\n"
        +
        s2_note
    )

# ================================================================================================================
# 14. EXCEL MASTER
# ================================================================================================================

with pd.ExcelWriter(
    WORKBOOK,
    engine="openpyxl"
) as writer:

    main_table_display.to_excel(
        writer,
        sheet_name="MAIN_TABLE",
        index=False
    )

    table_s1.to_excel(
        writer,
        sheet_name="TABLE_S1_50_CURVES",
        index=False
    )

    table_s2.to_excel(
        writer,
        sheet_name="TABLE_S2_290_WINDOWS",
        index=False
    )

    pd.DataFrame(
        {
            "item": [
                "Figure caption",
                "Main table note",
                "Table S1 note",
                "Table S2 note",
            ],

            "text": [
                figure_caption,
                main_table_note,
                s1_note,
                s2_note,
            ],
        }
    ).to_excel(
        writer,
        sheet_name="CAPTIONS_NOTES",
        index=False
    )

# ================================================================================================================
# 15. QA
# ================================================================================================================

expected_strict = {
    (
        "Electrical conductivity",
        "Precipitation",
        14,
    ),
    (
        "Electrical conductivity",
        "Precipitation",
        60,
    ),
    (
        "Na",
        "Precipitation",
        60,
    ),
    (
        "SO₄",
        "Precipitation",
        60,
    ),
}

actual_strict = set(
    zip(
        strict[
            "response_label"
        ],
        strict[
            "hydro_family_label"
        ],
        strict[
            "window_days"
        ].astype(int),
    )
)

qa_dict = {

    "source_8_5F_A_pass":
        manifest_a["pass"] is True,

    "family_curves_50":
        len(family) == 50,

    "window_results_290":
        len(window) == 290,

    "joint_supported_99":
        n_joint == 99,

    "strict_all_5":
        n_strict_all == 5,

    "main_hydrochemical_rows_4":
        len(main_table_display) == 4,

    "main_hydrochemical_identity_exact":
        actual_strict
        ==
        expected_strict,

    "table_s1_rows_50":
        len(table_s1) == 50,

    "table_s2_rows_290":
        len(table_s2) == 290,

    "figure_svg_exists":
        FIG_SVG.exists(),

    "figure_pdf_exists":
        FIG_PDF.exists(),

    "figure_png_exists":
        FIG_PNG.exists(),

    "main_table_csv_exists":
        MAIN_TABLE_CSV.exists(),

    "table_s1_exists":
        S1_CSV.exists(),

    "table_s2_exists":
        S2_CSV.exists(),

    "workbook_exists":
        WORKBOOK.exists(),

    "caption_exists":
        CAPTION_TXT.exists(),

    "models_fitted_false":
        True,

    "inference_recomputed_false":
        True,

    "thresholds_changed_false":
        True,

    "memory_timescale_selected_false":
        True,

    "connectivity_inferred_false":
        True,
}

qa = pd.DataFrame(
    {
        "check":
            list(
                qa_dict.keys()
            ),

        "pass":
            list(
                qa_dict.values()
            ),
    }
)

PASS = bool(
    qa[
        "pass"
    ].all()
)

# ================================================================================================================
# 16. HASHES + MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


output_files = [
    FIG_SVG,
    FIG_PDF,
    FIG_PNG,
    MAIN_TABLE_CSV,
    S1_CSV,
    S2_CSV,
    CAPTION_TXT,
    NOTES_TXT,
    WORKBOOK,
]

MANIFEST_OUT = (
    MEM /
    "BLOCK85F_C_publication_products_manifest.json"
)

manifest = {

    "block":
        "8.5F-C",

    "created_at":
        datetime.now().isoformat(),

    "source":
        "BLOCK 8.5F-A frozen products",

    "figure":
        "Hydroclimatic memory spectrum",

    "main_table":
        "Strict hydrochemical antecedent-memory support",

    "supplementary_table_s1":
        "All 50 response-driver temporal curves",

    "supplementary_table_s2":
        "All 290 window-level inference results",

    "n_joint_supported_windows":
        n_joint,

    "n_strict_all_windows":
        n_strict_all,

    "n_strict_hydrochemical_windows":
        len(main_table_display),

    "models_fitted":
        False,

    "inference_recomputed":
        False,

    "thresholds_changed":
        False,

    "memory_timescale_selected":
        False,

    "connectivity_inferred":
        False,

    "pass":
        PASS,

    "files": {
        p.name:
            {
                "sha256":
                    sha256_file(p),

                "bytes":
                    int(
                        p.stat().st_size
                    ),
            }

        for p in output_files
    },
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 17. REPORT
# ================================================================================================================

print("\n" + "=" * 120)
print("8.5F-C PUBLICATION PRODUCTS QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

print("\nMAIN MANUSCRIPT TABLE — STRICT HYDROCHEMICAL MEMORY")
print("-" * 120)

print(
    main_table_display.to_string(
        index=False
    )
)

print("\nFIGURE CAPTION")
print("-" * 120)
print(
    figure_caption
)

print("\nMAIN TABLE NOTE")
print("-" * 120)
print(
    main_table_note
)

print("\nOUTPUT PRODUCTS")
print("-" * 120)

for p in (
    output_files
    +
    [MANIFEST_OUT]
):

    print(
        p.name,
        "|",
        p.stat().st_size
        if p.exists()
        else "MISSING",
        "bytes"
    )

print("\n" + "=" * 120)
print("BLOCK 8.5F-C FINAL GATE")
print("=" * 120)

print(
    "8.5F-C PASS                    :",
    PASS
)

print(
    "Figure joint-support windows  :",
    n_joint
)

print(
    "Figure strict-memory windows  :",
    n_strict_all
)

print(
    "Main-table hydrochemical rows :",
    len(
        main_table_display
    )
)

print(
    "Table S1 curves               :",
    len(
        table_s1
    )
)

print(
    "Table S2 windows              :",
    len(
        table_s2
    )
)

print(
    "Models fitted                 :",
    False
)

print(
    "Inference recomputed          :",
    False
)

print(
    "Thresholds changed            :",
    False
)

print(
    "Exact memory timescale chosen :",
    False
)

print(
    "Connectivity inferred         :",
    False
)

if PASS:

    print(
        "\nBLOCK 8.5F-C PASS / PUBLICATION PRODUCTS FROZEN."
    )

    print(
        "MEMORY-SPECTRUM FIGURE + MAIN TABLE + TABLES S1/S2 READY."
    )

else:

    print(
        "\nSTOP — DO NOT MOVE TO C-Q UNTIL QA IS RESOLVED."
    )

---
### Imported master block 109

`BLOCK 9.0B — AUTHORITATIVE HYDROMETRY + ANALYTICAL MASTER PREFLIGHT`


In [ ]:
# ================================================================================================================
# BLOCK 9.0B — AUTHORITATIVE HYDROMETRY + ANALYTICAL MASTER PREFLIGHT
# READ-ONLY SCIENTIFIC AUDIT
#
# PURPOSE
#   1. Recover the exact frozen hydrometry source.
#   2. Recover MASTER_ANALYTICAL_v1_1.
#   3. Inspect identities and Q architecture.
#   4. Verify historical frozen counts from the actual data.
#
# NO MODELS
# NO Q RECONSTRUCTION
# NO RESPONSE SELECTION
# NO ROW COLLAPSING
# NO FILE OVERWRITE
# ================================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

print("=" * 120)
print("BLOCK 9.0B — AUTHORITATIVE HYDROMETRY + ANALYTICAL MASTER PREFLIGHT")
print("=" * 120)

# ================================================================================================================
# 1. FROZEN SNAPSHOT
# ================================================================================================================

SNAP = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT/"
    "SNAPSHOTS/POST_BLOCK85C_20260915_034110"
)

MASTER_FILE = (
    SNAP /
    "03_MASTER/MASTER_ANALYTICAL_v1_1.csv"
)

HYDRO_FILE = (
    SNAP /
    "03_MASTER/HYDROMETRY_FROZEN_v1.csv"
)

Q_CAMPAIGN_FILE = (
    SNAP /
    "01_AUDIT/BLOCK63_master_Q_by_campaign.csv"
)

GATE_FILE = (
    SNAP /
    "01_AUDIT/BLOCK6_hydrometric_gate.csv"
)

for p in [
    MASTER_FILE,
    HYDRO_FILE,
    Q_CAMPAIGN_FILE,
    GATE_FILE,
]:
    assert p.exists(), f"MISSING FROZEN SOURCE: {p}"

print("\nFROZEN SOURCES FOUND: PASS")

# ================================================================================================================
# 2. LOAD
# ================================================================================================================

master = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

hydro = pd.read_csv(
    HYDRO_FILE,
    low_memory=False
)

q_campaign = pd.read_csv(
    Q_CAMPAIGN_FILE,
    low_memory=False
)

gate = pd.read_csv(
    GATE_FILE,
    low_memory=False
)

print("\nINPUT SHAPES")
print("-" * 120)

print("MASTER_ANALYTICAL_v1_1 :", master.shape)
print("HYDROMETRY_FROZEN_v1   :", hydro.shape)
print("BLOCK63 Q campaign      :", q_campaign.shape)
print("BLOCK6 hydrometric gate :", gate.shape)

# ================================================================================================================
# 3. SHA256 — RECORD EXACT INPUTS
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

print("\nSOURCE SHA256")
print("-" * 120)

print("MASTER :", sha256_file(MASTER_FILE))
print("HYDRO  :", sha256_file(HYDRO_FILE))

# ================================================================================================================
# 4. SCHEMAS
# ================================================================================================================

print("\nMASTER SCHEMA")
print("-" * 120)

for i, c in enumerate(master.columns):
    print(f"{i:03d} | {c}")

print("\nHYDROMETRY SCHEMA")
print("-" * 120)

for i, c in enumerate(hydro.columns):
    print(f"{i:03d} | {c}")

# ================================================================================================================
# 5. IDENTIFY Q-LIKE COLUMNS
# ================================================================================================================

q_pattern = re.compile(
    r"(^q$|discharge|caudal|flow|q_|_q|hydrom)",
    flags=re.I
)

master_qcols = [
    c for c in master.columns
    if q_pattern.search(str(c))
]

hydro_qcols = [
    c for c in hydro.columns
    if q_pattern.search(str(c))
]

print("\nQ-LIKE COLUMNS")
print("-" * 120)

print("MASTER:")
for c in master_qcols:
    print("  ", c)

print("\nHYDROMETRY:")
for c in hydro_qcols:
    print("  ", c)

# ================================================================================================================
# 6. IDENTITY COLUMNS
# ================================================================================================================

identity_candidates = [
    "field_observation_id",
    "sample_id",
    "field_linked_sample_id",
    "analytical_sample_id",
    "campaign_id",
    "campaign",
    "station_id",
    "station_key",
    "site",
    "site_code",
    "canonical_site",
    "date",
    "sample_date",
    "field_date",
]

print("\nIDENTITY COLUMNS PRESENT")
print("-" * 120)

print("MASTER:")
for c in identity_candidates:
    if c in master.columns:
        print("  ", c)

print("\nHYDROMETRY:")
for c in identity_candidates:
    if c in hydro.columns:
        print("  ", c)

# ================================================================================================================
# 7. FULL HYDROMETRY TABLE
#
# Small enough to print selectively.
# ================================================================================================================

preferred_display = []

for c in hydro.columns:

    lc = str(c).lower()

    if (
        c in identity_candidates
        or q_pattern.search(str(c))
        or "source" in lc
        or "status" in lc
        or "recon" in lc
        or "observ" in lc
        or "campaign" in lc
        or "date" in lc
    ):
        preferred_display.append(c)

preferred_display = list(dict.fromkeys(preferred_display))

print("\nHYDROMETRY — RELEVANT COLUMNS")
print("-" * 120)

print(preferred_display)

print("\nHYDROMETRY — FIRST 20 ROWS")
print("-" * 120)

if preferred_display:
    print(
        hydro[
            preferred_display
        ]
        .head(20)
        .to_string(index=False)
    )

# ================================================================================================================
# 8. NON-NULL / UNIQUE SUMMARY FOR Q-LIKE COLUMNS
# ================================================================================================================

def summarize_columns(df, cols, source):

    rows = []

    for c in cols:

        s = df[c]

        numeric = pd.to_numeric(
            s,
            errors="coerce"
        )

        rows.append({
            "source": source,
            "column": c,
            "rows": len(df),
            "non_null": int(s.notna().sum()),
            "unique_non_null": int(s.dropna().nunique()),
            "numeric_non_null": int(numeric.notna().sum()),
            "numeric_min": (
                float(numeric.min())
                if numeric.notna().any()
                else np.nan
            ),
            "numeric_median": (
                float(numeric.median())
                if numeric.notna().any()
                else np.nan
            ),
            "numeric_max": (
                float(numeric.max())
                if numeric.notna().any()
                else np.nan
            ),
            "zero_or_negative": (
                int((numeric <= 0).sum())
                if numeric.notna().any()
                else 0
            ),
        })

    return pd.DataFrame(rows)

q_summary = pd.concat(
    [
        summarize_columns(
            master,
            master_qcols,
            "MASTER"
        ),
        summarize_columns(
            hydro,
            hydro_qcols,
            "HYDROMETRY"
        ),
    ],
    ignore_index=True
)

print("\nQ-COLUMN SUMMARY")
print("-" * 120)

if len(q_summary):
    print(q_summary.to_string(index=False))
else:
    print("NO Q-LIKE COLUMNS FOUND")

# ================================================================================================================
# 9. FIELD OBSERVATION ID ARCHITECTURE
# ================================================================================================================

print("\nFIELD OBSERVATION ARCHITECTURE")
print("-" * 120)

for name, df in [
    ("MASTER", master),
    ("HYDROMETRY", hydro),
]:

    if "field_observation_id" in df.columns:

        print(
            name,
            "| rows:",
            len(df),
            "| non-null FIELD IDs:",
            int(df["field_observation_id"].notna().sum()),
            "| unique FIELD IDs:",
            int(df["field_observation_id"].dropna().nunique())
        )

# ================================================================================================================
# 10. CAMPAIGN ARCHITECTURE
# ================================================================================================================

print("\nCAMPAIGN ARCHITECTURE")
print("-" * 120)

for name, df in [
    ("MASTER", master),
    ("HYDROMETRY", hydro),
]:

    campaign_col = next(
        (
            c for c in [
                "campaign_id",
                "campaign"
            ]
            if c in df.columns
        ),
        None
    )

    if campaign_col is not None:

        print(f"\n{name} | {campaign_col}")

        print(
            df[
                campaign_col
            ]
            .value_counts(
                dropna=False
            )
            .sort_index()
            .to_string()
        )

# ================================================================================================================
# 11. FROZEN BLOCK 6.3 CAMPAIGN AUDIT
# ================================================================================================================

print("\nBLOCK 6.3 — FROZEN MASTER Q BY CAMPAIGN")
print("-" * 120)

print(
    q_campaign.to_string(
        index=False
    )
)

print("\nBLOCK 6 — HYDROMETRIC GATE")
print("-" * 120)

print(
    gate.to_string(
        index=False
    )
)

# ================================================================================================================
# 12. EXACT OVERLAP OF FIELD IDS
# ================================================================================================================

if (
    "field_observation_id" in master.columns
    and
    "field_observation_id" in hydro.columns
):

    master_ids = set(
        master[
            "field_observation_id"
        ]
        .dropna()
        .astype(str)
    )

    hydro_ids = set(
        hydro[
            "field_observation_id"
        ]
        .dropna()
        .astype(str)
    )

    print("\nFIELD ID OVERLAP")
    print("-" * 120)

    print(
        "MASTER unique FIELD IDs      :",
        len(master_ids)
    )

    print(
        "HYDROMETRY unique FIELD IDs  :",
        len(hydro_ids)
    )

    print(
        "Intersection                 :",
        len(
            master_ids & hydro_ids
        )
    )

    print(
        "MASTER IDs absent HYDROMETRY :",
        len(
            master_ids - hydro_ids
        )
    )

    print(
        "HYDRO IDs absent MASTER      :",
        len(
            hydro_ids - master_ids
        )
    )

# ================================================================================================================
# 13. DO NOT CREATE C-Q DATASET YET
# ================================================================================================================

print("\n" + "=" * 120)
print("BLOCK 9.0B PREFLIGHT COMPLETE")
print("=" * 120)

print("MASTER ROWS                :", len(master))
print("HYDROMETRY ROWS            :", len(hydro))
print("MASTER Q-LIKE COLUMNS      :", len(master_qcols))
print("HYDRO Q-LIKE COLUMNS       :", len(hydro_qcols))

print("\nFILES MODIFIED             : 0")
print("Q RECONSTRUCTED            : False")
print("ROWS COLLAPSED             : False")
print("CENSOR VALUES SUBSTITUTED  : False")
print("RESPONSES SELECTED BY Q    : False")
print("C-Q MODELS FITTED          : False")
print("C-Q DATASET FROZEN         : False")

print(
    "\nNEXT: identify the exact observed/reconstructed/best-Q columns from these frozen sources, "
    "then assemble BLOCK 9.0C without guessing."
)

---
### Imported master block 110

`BLOCK 9.0C-R — C–Q ANALYSIS DATASET ASSEMBLY + FREEZE`


In [ ]:
# ================================================================================================================
# BLOCK 9.0C-R — C–Q ANALYSIS DATASET ASSEMBLY + FREEZE
# SCHEMA-CORRECTED VERSION
#
# FIX:
#   MEMORY_ANALYSIS_DATASET_v1.csv has no "response_label".
#   response_key remains the scientific identity.
#   response_label is created ONLY as a presentation label.
#
# FROZEN Q POLICY:
#   PRIMARY     = Q_primary_CQ       = observed discharge only
#   SENSITIVITY = Q_sensitivity_CQ   = observed + reconstructed C09
#
# NO:
#   - Q reconstruction
#   - analytical subsample averaging
#   - censor substitution
#   - response selection using Q
#   - C–Q model fitting
#   - dilution/mobilization classification
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import json

print("=" * 120)
print("BLOCK 9.0C-R — C–Q ANALYSIS DATASET ASSEMBLY + FREEZE")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

PROJECT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

SNAP = (
    PROJECT /
    "SNAPSHOTS/POST_BLOCK85C_20260915_034110"
)

MASTER_FILE = (
    SNAP /
    "03_MASTER/MASTER_ANALYTICAL_v1_1.csv"
)

HYDRO_FILE = (
    SNAP /
    "03_MASTER/HYDROMETRY_FROZEN_v1.csv"
)

MEMORY_DATA_FILE = (
    SNAP /
    "03_MASTER/MEMORY_ANALYSIS_DATASET_v1.csv"
)

STAT_ARCH_FILE = (
    SNAP /
    "03_MASTER/MEMORY_RESPONSE_STATISTICAL_ARCHITECTURE_v1.csv"
)

OUTDIR = (
    PROJECT /
    "CQ_ANALYSIS"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)

for p in [
    MASTER_FILE,
    HYDRO_FILE,
    MEMORY_DATA_FILE,
    STAT_ARCH_FILE,
]:
    assert p.exists(), f"MISSING FROZEN INPUT: {p}"

# ================================================================================================================
# 2. LOAD
# ================================================================================================================

master = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

hydro = pd.read_csv(
    HYDRO_FILE,
    low_memory=False
)

memory = pd.read_csv(
    MEMORY_DATA_FILE,
    low_memory=False
)

stat = pd.read_csv(
    STAT_ARCH_FILE,
    low_memory=False
)

print("\nINPUT SHAPES")
print("-" * 120)

print("MASTER :", master.shape)
print("HYDRO  :", hydro.shape)
print("MEMORY :", memory.shape)
print("STAT   :", stat.shape)

assert master.shape[0] == 10616
assert len(hydro) == 140
assert len(memory) == 1300
assert len(stat) == 10

# ================================================================================================================
# 3. REQUIRED SCHEMA
# ================================================================================================================

required_master = [
    "field_observation_id",
    "campaign_id",
    "physical_station_id_v1",
    "Q_observed_final",
    "Q_reconstructed_sensitivity",
    "Q_primary_CQ",
    "Q_sensitivity_CQ",
    "Q_primary_CQ_eligible",
    "Q_sensitivity_CQ_eligible",
]

required_memory = [
    "response_key",
    "response_origin",
    "sample_id",
    "field_linked_sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",
    "parameter_id",
    "parameter_std",
    "unit_final",
    "response_value_raw",
    "response_value_numeric",
    "response_censored",
    "response_censor_type",
    "response_censor_limit",
    "response_family",
]

missing_master = [
    c for c in required_master
    if c not in master.columns
]

missing_memory = [
    c for c in required_memory
    if c not in memory.columns
]

assert not missing_master, (
    f"Missing MASTER columns: {missing_master}"
)

assert not missing_memory, (
    f"Missing MEMORY columns: {missing_memory}"
)

# ================================================================================================================
# 4. GLOBAL FIELD HYDROMETRY
# ================================================================================================================

global_field_events = int(
    hydro["field_observation_id"].nunique()
)

global_observed = int(
    hydro["discharge_observed"].notna().sum()
)

global_reconstructed = int(
    hydro["discharge_reconstructed"].notna().sum()
)

global_best = int(
    hydro["discharge_best_available"].notna().sum()
)

global_missing_best = int(
    hydro["discharge_best_available"].isna().sum()
)

global_positive_observed = bool(
    (
        hydro.loc[
            hydro["discharge_observed"].notna(),
            "discharge_observed"
        ].astype(float) > 0
    ).all()
)

global_positive_best = bool(
    (
        hydro.loc[
            hydro["discharge_best_available"].notna(),
            "discharge_best_available"
        ].astype(float) > 0
    ).all()
)

print("\nGLOBAL FIELD HYDROMETRY")
print("-" * 120)

print("FIELD events             :", global_field_events)
print("Observed Q               :", global_observed)
print("Reconstructed Q          :", global_reconstructed)
print("Best available Q         :", global_best)
print("Missing best Q           :", global_missing_best)

assert global_field_events == 140
assert global_observed == 119
assert global_reconstructed == 10
assert global_best == 129
assert global_missing_best == 11
assert global_positive_observed
assert global_positive_best

# ================================================================================================================
# 5. BUILD MASTER FIELD-EVENT Q CROSSWALK
# ================================================================================================================

q_cols = [
    "field_observation_id",
    "campaign_id",
    "physical_station_id_v1",
    "Q_observed_final",
    "Q_reconstructed_sensitivity",
    "Q_primary_CQ",
    "Q_sensitivity_CQ",
    "Q_primary_CQ_eligible",
    "Q_sensitivity_CQ_eligible",
]

q_raw = (
    master[q_cols]
    .dropna(
        subset=["field_observation_id"]
    )
    .copy()
)

# ------------------------------------------------------------------------------------------------
# Check that each physical FIELD event has one internally consistent identity/Q state.
# ------------------------------------------------------------------------------------------------

conflict_rows = []

for field_id, g in q_raw.groupby(
    "field_observation_id",
    sort=False
):

    for col in [
        "campaign_id",
        "physical_station_id_v1",
        "Q_observed_final",
        "Q_reconstructed_sensitivity",
        "Q_primary_CQ",
        "Q_sensitivity_CQ",
        "Q_primary_CQ_eligible",
        "Q_sensitivity_CQ_eligible",
    ]:

        n_unique = g[col].dropna().nunique()

        if n_unique > 1:

            conflict_rows.append({
                "field_observation_id": field_id,
                "column": col,
                "n_unique_non_null": int(n_unique),
            })

q_conflicts = pd.DataFrame(
    conflict_rows
)

assert len(q_conflicts) == 0, (
    "Conflicting Q states detected within FIELD observation."
)

q_event = (
    q_raw
    .drop_duplicates(
        subset=["field_observation_id"],
        keep="first"
    )
    .sort_values(
        "field_observation_id"
    )
    .reset_index(drop=True)
)

assert q_event["field_observation_id"].is_unique

master_field_events = len(q_event)

master_observed_events = int(
    q_event["Q_primary_CQ"].notna().sum()
)

master_sensitivity_events = int(
    q_event["Q_sensitivity_CQ"].notna().sum()
)

master_reconstructed_events = int(
    (
        q_event["Q_primary_CQ"].isna()
        &
        q_event["Q_reconstructed_sensitivity"].notna()
    ).sum()
)

print("\nMASTER-LINKED FIELD-EVENT Q ARCHITECTURE")
print("-" * 120)

print("FIELD events represented :", master_field_events)
print("Observed-Q primary       :", master_observed_events)
print("Reconstructed-only       :", master_reconstructed_events)
print("Sensitivity total        :", master_sensitivity_events)

assert master_field_events == 133
assert master_observed_events == 115
assert master_reconstructed_events == 10
assert master_sensitivity_events == 125

# ================================================================================================================
# 6. PRIMARY / SENSITIVITY CONSISTENCY
# ================================================================================================================

observed_mask = (
    q_event["Q_primary_CQ"].notna()
)

observed_diff = (
    q_event.loc[
        observed_mask,
        "Q_primary_CQ"
    ].astype(float).to_numpy()
    -
    q_event.loc[
        observed_mask,
        "Q_sensitivity_CQ"
    ].astype(float).to_numpy()
)

max_observed_difference = float(
    np.max(
        np.abs(observed_diff)
    )
)

assert max_observed_difference < 1e-12

reconstructed_only = (
    q_event.loc[
        q_event["Q_primary_CQ"].isna()
        &
        q_event["Q_sensitivity_CQ"].notna()
    ]
    .copy()
)

assert len(reconstructed_only) == 10

assert set(
    reconstructed_only["campaign_id"].dropna()
) == {"C09"}

print("\nRECONSTRUCTED-ONLY EVENTS")
print("-" * 120)

print(
    reconstructed_only[
        [
            "field_observation_id",
            "campaign_id",
            "physical_station_id_v1",
            "Q_reconstructed_sensitivity",
            "Q_sensitivity_CQ",
        ]
    ].to_string(index=False)
)

# ================================================================================================================
# 7. RESPONSE IDENTITIES — NO Q USED
# ================================================================================================================

response_identity = (
    memory[
        [
            "response_key",
            "response_origin",
            "parameter_id",
            "parameter_std",
            "unit_final",
            "response_family",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "response_key"
    )
    .reset_index(drop=True)
)

print("\nFROZEN PRIMARY RESPONSE IDENTITIES")
print("-" * 120)

print(
    response_identity.to_string(
        index=False
    )
)

print(
    "\nUnique response_key:",
    memory["response_key"].nunique()
)

assert memory["response_key"].nunique() == 10

# ================================================================================================================
# 8. PRESENTATION LABELS
#
# IMPORTANT:
# These labels have ZERO role in selection/inference.
# response_key remains the authoritative identity.
# ================================================================================================================

label_map = {}

for response_key, g in memory.groupby(
    "response_key",
    sort=False
):

    parameter_values = (
        g["parameter_std"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    origin_values = (
        g["response_origin"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    # Default label comes from frozen parameter_std.
    if len(parameter_values) == 1:

        label = parameter_values[0]

    else:

        label = str(response_key)

    # Explicit cosmetic normalization only.
    lower = label.lower()

    if (
        "temperature" in lower
        and
        "field" in " ".join(origin_values).lower()
    ):
        label = "Water temperature"

    label_map[response_key] = label

print("\nPRESENTATION LABEL CROSSWALK")
print("-" * 120)

for k, v in label_map.items():
    print(k, " -> ", v)

# ================================================================================================================
# 9. RESPONSE-SIDE DATASET
#
# Explicitly excludes all 29 hydroclimate metrics.
# ================================================================================================================

response_cols = [
    "response_key",
    "response_origin",
    "sample_id",
    "field_linked_sample_id",
    "field_observation_id",
    "analytical_sample_id",
    "station_id",
    "physical_station_id_v1",
    "campaign_id",
    "parameter_id",
    "parameter_std",
    "unit_final",
    "response_value_raw",
    "response_value_numeric",
    "response_censored",
    "response_censor_type",
    "response_censor_limit",
    "temperature_primary_source",
    "temperature_identity_policy",
    "response_family",
    "memory_panel_role",
    "scientific_role",
    "panel_version",
    "freeze_status",
]

response_cols = [
    c for c in response_cols
    if c in memory.columns
]

cq = (
    memory[
        response_cols
    ]
    .copy()
)

assert len(cq) == 1300
assert cq["response_key"].nunique() == 10

cq.insert(
    1,
    "response_label",
    cq["response_key"].map(label_map)
)

assert cq["response_label"].notna().all()

# ================================================================================================================
# 10. JOIN Q BY PHYSICAL FIELD OBSERVATION
# ================================================================================================================

q_join = q_event[
    [
        "field_observation_id",
        "Q_observed_final",
        "Q_reconstructed_sensitivity",
        "Q_primary_CQ",
        "Q_sensitivity_CQ",
        "Q_primary_CQ_eligible",
        "Q_sensitivity_CQ_eligible",
    ]
].copy()

cq = cq.merge(
    q_join,
    on="field_observation_id",
    how="left",
    validate="many_to_one",
    indicator="_q_merge"
)

print("\nRESPONSE × Q JOIN")
print("-" * 120)

print(
    cq["_q_merge"]
    .value_counts(
        dropna=False
    )
    .to_string()
)

# ================================================================================================================
# 11. Q ANALYSIS ROLES
# ================================================================================================================

cq["cq_primary_eligible"] = (
    cq["Q_primary_CQ"].notna()
)

cq["cq_sensitivity_eligible"] = (
    cq["Q_sensitivity_CQ"].notna()
)

cq["cq_q_source"] = np.select(
    [
        cq["Q_primary_CQ"].notna(),

        cq["Q_primary_CQ"].isna()
        &
        cq["Q_sensitivity_CQ"].notna(),
    ],
    [
        "OBSERVED_PRIMARY",
        "RECONSTRUCTED_C09_SENSITIVITY_ONLY",
    ],
    default="NO_Q"
)

# ================================================================================================================
# 12. Q POSITIVITY / DISTRIBUTION
# ================================================================================================================

primary_event_Q = (
    q_event["Q_primary_CQ"]
    .dropna()
    .astype(float)
)

sensitivity_event_Q = (
    q_event["Q_sensitivity_CQ"]
    .dropna()
    .astype(float)
)

primary_nonpositive = int(
    (primary_event_Q <= 0).sum()
)

sensitivity_nonpositive = int(
    (sensitivity_event_Q <= 0).sum()
)

assert primary_nonpositive == 0
assert sensitivity_nonpositive == 0

def q_stats(name, x):

    x = pd.Series(
        x,
        dtype=float
    ).dropna()

    logx = np.log10(x)

    return {
        "Q_role": name,
        "n_events": len(x),
        "min": x.min(),
        "q25": x.quantile(0.25),
        "median": x.median(),
        "q75": x.quantile(0.75),
        "max": x.max(),
        "mean": x.mean(),
        "sd": x.std(),
        "raw_skew": x.skew(),
        "log10_min": logx.min(),
        "log10_median": logx.median(),
        "log10_max": logx.max(),
        "log10_skew": logx.skew(),
        "orders_of_magnitude": (
            np.log10(x.max())
            -
            np.log10(x.min())
        ),
    }

q_distribution = pd.DataFrame([
    q_stats(
        "PRIMARY_OBSERVED",
        primary_event_Q
    ),
    q_stats(
        "SENSITIVITY_OBSERVED_PLUS_C09",
        sensitivity_event_Q
    ),
])

print("\nFIELD-EVENT Q DISTRIBUTION")
print("-" * 120)

print(
    q_distribution.to_string(
        index=False
    )
)

# Diagnostic representation only.
# Modeling transformation is NOT frozen in this block.

cq["log10_Q_primary_diagnostic"] = np.where(
    pd.to_numeric(
        cq["Q_primary_CQ"],
        errors="coerce"
    ) > 0,
    np.log10(
        pd.to_numeric(
            cq["Q_primary_CQ"],
            errors="coerce"
        )
    ),
    np.nan
)

cq["log10_Q_sensitivity_diagnostic"] = np.where(
    pd.to_numeric(
        cq["Q_sensitivity_CQ"],
        errors="coerce"
    ) > 0,
    np.log10(
        pd.to_numeric(
            cq["Q_sensitivity_CQ"],
            errors="coerce"
        )
    ),
    np.nan
)

# ================================================================================================================
# 13. CENSORING HELPER
# ================================================================================================================

def as_bool_censored(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    s = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return s.isin(
        [
            "true",
            "1",
            "yes",
            "y",
        ]
    )

cq["_censored_bool"] = as_bool_censored(
    cq["response_censored"]
)

# ================================================================================================================
# 14. RESPONSE-SPECIFIC C–Q COVERAGE
# ================================================================================================================

coverage_rows = []

for response_key, g in cq.groupby(
    "response_key",
    sort=False
):

    gp = g.loc[
        g["cq_primary_eligible"]
    ]

    gs = g.loc[
        g["cq_sensitivity_eligible"]
    ]

    coverage_rows.append({

        "response_key":
            response_key,

        "response_label":
            g["response_label"].iloc[0],

        "response_rows_total":
            len(g),

        "response_campaigns_total":
            g["campaign_id"].nunique(),

        "primary_cq_rows":
            len(gp),

        "primary_unique_samples":
            gp["sample_id"].nunique(),

        "primary_unique_field_events":
            gp["field_observation_id"].nunique(),

        "primary_unique_stations":
            gp["physical_station_id_v1"].nunique(),

        "primary_campaigns":
            gp["campaign_id"].nunique(),

        "primary_censored_rows":
            int(
                gp["_censored_bool"].sum()
            ),

        "sensitivity_cq_rows":
            len(gs),

        "sensitivity_unique_samples":
            gs["sample_id"].nunique(),

        "sensitivity_unique_field_events":
            gs["field_observation_id"].nunique(),

        "sensitivity_unique_stations":
            gs["physical_station_id_v1"].nunique(),

        "sensitivity_campaigns":
            gs["campaign_id"].nunique(),

        "sensitivity_censored_rows":
            int(
                gs["_censored_bool"].sum()
            ),

        "added_rows_from_C09_reconstruction":
            len(gs) - len(gp),

        "added_field_events_from_C09_reconstruction":
            (
                gs["field_observation_id"].nunique()
                -
                gp["field_observation_id"].nunique()
            ),

        "total_censored_response_rows":
            int(
                g["_censored_bool"].sum()
            ),
    })

coverage = pd.DataFrame(
    coverage_rows
)

print("\nC–Q COVERAGE BY RESPONSE")
print("-" * 120)

print(
    coverage.to_string(
        index=False
    )
)

# ================================================================================================================
# 15. CAMPAIGN COVERAGE
# ================================================================================================================

campaign_rows = []

for campaign in sorted(
    cq["campaign_id"]
    .dropna()
    .unique()
):

    g = cq.loc[
        cq["campaign_id"].eq(campaign)
    ]

    gp = g.loc[
        g["cq_primary_eligible"]
    ]

    gs = g.loc[
        g["cq_sensitivity_eligible"]
    ]

    campaign_rows.append({

        "campaign_id":
            campaign,

        "response_rows_total":
            len(g),

        "primary_response_rows":
            len(gp),

        "primary_field_events":
            gp["field_observation_id"].nunique(),

        "primary_stations":
            gp["physical_station_id_v1"].nunique(),

        "sensitivity_response_rows":
            len(gs),

        "sensitivity_field_events":
            gs["field_observation_id"].nunique(),

        "sensitivity_stations":
            gs["physical_station_id_v1"].nunique(),

        "reconstruction_added_response_rows":
            len(gs) - len(gp),

        "reconstruction_added_field_events":
            (
                gs["field_observation_id"].nunique()
                -
                gp["field_observation_id"].nunique()
            ),
    })

campaign_coverage = pd.DataFrame(
    campaign_rows
)

print("\nC–Q COVERAGE BY CAMPAIGN")
print("-" * 120)

print(
    campaign_coverage.to_string(
        index=False
    )
)

# ================================================================================================================
# 16. LEGITIMATE REPEATED SAMPLE × RESPONSE COMBINATIONS
# ================================================================================================================

rep = (
    cq
    .groupby(
        [
            "sample_id",
            "response_key",
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="n_rows"
    )
)

rep = (
    rep.loc[
        rep["n_rows"] > 1
    ]
    .copy()
)

print("\nREPEATED SAMPLE × RESPONSE COMBINATIONS")
print("-" * 120)

print("Count:", len(rep))

if len(rep):

    print(
        rep.to_string(
            index=False
        )
    )

assert len(rep) == 17

# ================================================================================================================
# 17. DISTINCT ANALYTICAL SUBSAMPLE CHECK
# ================================================================================================================

rep_detail = cq.merge(
    rep[
        [
            "sample_id",
            "response_key",
        ]
    ],
    on=[
        "sample_id",
        "response_key",
    ],
    how="inner"
)

print("\nREPEATED COMBINATIONS — ANALYTICAL SUBSAMPLE DETAIL")
print("-" * 120)

detail_cols = [
    "sample_id",
    "response_key",
    "field_observation_id",
    "analytical_sample_id",
    "Q_primary_CQ",
    "Q_sensitivity_CQ",
]

print(
    rep_detail[
        detail_cols
    ]
    .sort_values(
        [
            "sample_id",
            "response_key",
            "analytical_sample_id",
        ]
    )
    .to_string(
        index=False
    )
)

# No averaging.
# Verify repeated rows retain distinct analytical IDs whenever analytical IDs exist.

rep_analytical_check = []

for (sample_id, response_key), g in rep_detail.groupby(
    [
        "sample_id",
        "response_key",
    ]
):

    ids = (
        g["analytical_sample_id"]
        .dropna()
        .astype(str)
    )

    rep_analytical_check.append({
        "sample_id": sample_id,
        "response_key": response_key,
        "n_rows": len(g),
        "n_nonnull_analytical_ids": len(ids),
        "n_unique_analytical_ids": ids.nunique(),
    })

rep_analytical_check = pd.DataFrame(
    rep_analytical_check
)

# ================================================================================================================
# 18. RECONSTRUCTED Q NEVER ENTERS PRIMARY
# ================================================================================================================

bad_reconstruction_primary = cq.loc[
    cq["cq_q_source"].eq(
        "RECONSTRUCTED_C09_SENSITIVITY_ONLY"
    )
    &
    cq["cq_primary_eligible"]
]

assert len(bad_reconstruction_primary) == 0

# ================================================================================================================
# 19. STATISTICAL ARCHITECTURE CROSSWALK
# ================================================================================================================

print("\nSTATISTICAL ARCHITECTURE SCHEMA")
print("-" * 120)

print(
    list(stat.columns)
)

required_stat = [
    "response_key",
    "primary_transformation",
    "statistical_pathway",
]

missing_stat = [
    c for c in required_stat
    if c not in stat.columns
]

assert not missing_stat, (
    f"Missing statistical architecture columns: {missing_stat}"
)

stat_cols = [
    c for c in [
        "response_key",
        "primary_transformation",
        "transformation_reason",
        "statistical_pathway",
        "censoring_policy",
        "detected_only_role",
    ]
    if c in stat.columns
]

stat_small = (
    stat[
        stat_cols
    ]
    .drop_duplicates(
        subset=["response_key"]
    )
    .copy()
)

assert stat_small["response_key"].nunique() == 10

cq = cq.merge(
    stat_small,
    on="response_key",
    how="left",
    validate="many_to_one"
)

assert cq["primary_transformation"].notna().all()
assert cq["statistical_pathway"].notna().all()

# ================================================================================================================
# 20. RESPONSE × Q JOIN ARCHITECTURE
# ================================================================================================================

join_summary = (
    cq["_q_merge"]
    .value_counts(
        dropna=False
    )
    .rename_axis("merge_status")
    .reset_index(name="n_rows")
)

source_summary = (
    cq["cq_q_source"]
    .value_counts(
        dropna=False
    )
    .rename_axis("Q_source")
    .reset_index(name="n_response_rows")
)

print("\nRESPONSE × Q JOIN")
print("-" * 120)

print(
    join_summary.to_string(
        index=False
    )
)

print("\nQ SOURCE AMONG RESPONSE ROWS")
print("-" * 120)

print(
    source_summary.to_string(
        index=False
    )
)

# ================================================================================================================
# 21. FINAL QA
# ================================================================================================================

qa_dict = {

    "global_field_events_140":
        global_field_events == 140,

    "global_observed_Q_119":
        global_observed == 119,

    "global_reconstructed_Q_10":
        global_reconstructed == 10,

    "global_best_Q_129":
        global_best == 129,

    "master_field_events_133":
        master_field_events == 133,

    "master_primary_observed_Q_115":
        master_observed_events == 115,

    "master_reconstructed_only_Q_10":
        master_reconstructed_events == 10,

    "master_sensitivity_Q_125":
        master_sensitivity_events == 125,

    "reconstructed_only_C09":
        set(
            reconstructed_only["campaign_id"].dropna()
        ) == {"C09"},

    "observed_primary_equals_sensitivity":
        max_observed_difference < 1e-12,

    "primary_Q_all_positive":
        primary_nonpositive == 0,

    "sensitivity_Q_all_positive":
        sensitivity_nonpositive == 0,

    "response_rows_1300":
        len(cq) == 1300,

    "responses_10":
        cq["response_key"].nunique() == 10,

    "presentation_labels_complete":
        cq["response_label"].notna().all(),

    "repeated_sample_response_combos_17":
        len(rep) == 17,

    "reconstructed_Q_never_primary":
        len(bad_reconstruction_primary) == 0,

    "transformations_complete":
        cq["primary_transformation"].notna().all(),

    "statistical_pathways_complete":
        cq["statistical_pathway"].notna().all(),
}

qa = pd.DataFrame({
    "check": list(qa_dict.keys()),
    "pass": list(qa_dict.values())
})

PASS = bool(
    qa["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.0C-R QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

if not PASS:

    print("\nSTOP — QA FAILED. NOTHING WILL BE FROZEN.")

    raise RuntimeError(
        "BLOCK 9.0C-R QA failed."
    )

# ================================================================================================================
# 22. REMOVE TEMPORARY HELPER
# ================================================================================================================

cq = cq.drop(
    columns=[
        "_censored_bool"
    ]
)

# Keep _q_merge because it is useful provenance.

# ================================================================================================================
# 23. FREEZE OUTPUTS
# ================================================================================================================

CQ_FILE = (
    OUTDIR /
    "CQ_ANALYSIS_DATASET_v1.csv"
)

COVERAGE_FILE = (
    OUTDIR /
    "CQ_RESPONSE_COVERAGE_v1.csv"
)

CAMPAIGN_FILE = (
    OUTDIR /
    "CQ_CAMPAIGN_COVERAGE_v1.csv"
)

Q_EVENT_FILE = (
    OUTDIR /
    "CQ_FIELD_EVENT_Q_CROSSWALK_v1.csv"
)

Q_DIST_FILE = (
    OUTDIR /
    "CQ_Q_DISTRIBUTION_AUDIT_v1.csv"
)

REP_FILE = (
    OUTDIR /
    "CQ_REPEATED_SUBSAMPLE_AUDIT_v1.csv"
)

QA_FILE = (
    OUTDIR /
    "BLOCK90C_R_CQ_DATASET_QA_v1.csv"
)

MANIFEST_FILE = (
    OUTDIR /
    "BLOCK90C_R_CQ_DATASET_MANIFEST_v1.json"
)

cq.to_csv(
    CQ_FILE,
    index=False
)

coverage.to_csv(
    COVERAGE_FILE,
    index=False
)

campaign_coverage.to_csv(
    CAMPAIGN_FILE,
    index=False
)

q_event.to_csv(
    Q_EVENT_FILE,
    index=False
)

q_distribution.to_csv(
    Q_DIST_FILE,
    index=False
)

rep_detail.to_csv(
    REP_FILE,
    index=False
)

qa.to_csv(
    QA_FILE,
    index=False
)

# ================================================================================================================
# 24. SHA256 + MANIFEST
# ================================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


manifest = {

    "block":
        "9.0C-R",

    "created_at":
        datetime.now().isoformat(),

    "schema_correction":
        (
            "response_label absent from frozen MEMORY_ANALYSIS_DATASET; "
            "presentation label derived from frozen parameter_std; "
            "response_key remains authoritative scientific identity"
        ),

    "source_master":
        str(MASTER_FILE),

    "source_hydrometry":
        str(HYDRO_FILE),

    "source_memory_dataset":
        str(MEMORY_DATA_FILE),

    "primary_Q_policy":
        "Q_primary_CQ = observed discharge only",

    "sensitivity_Q_policy":
        "Q_sensitivity_CQ = observed discharge plus reconstructed C09",

    "global_FIELD_events":
        global_field_events,

    "global_observed_Q":
        global_observed,

    "global_reconstructed_Q":
        global_reconstructed,

    "global_best_available_Q":
        global_best,

    "master_linked_FIELD_events":
        master_field_events,

    "master_primary_observed_Q_events":
        master_observed_events,

    "master_reconstructed_only_Q_events":
        master_reconstructed_events,

    "master_sensitivity_Q_events":
        master_sensitivity_events,

    "response_rows":
        len(cq),

    "responses":
        int(
            cq["response_key"].nunique()
        ),

    "repeated_sample_response_combinations":
        len(rep),

    "Q_reconstructed_in_this_block":
        False,

    "analytical_subsamples_averaged":
        False,

    "censor_values_substituted":
        False,

    "responses_selected_using_Q":
        False,

    "CQ_models_fitted":
        False,

    "dilution_mobilization_classified":
        False,

    "logQ_modeling_transformation_frozen":
        False,

    "pass":
        PASS,

    "outputs": {},
}

for p in [
    CQ_FILE,
    COVERAGE_FILE,
    CAMPAIGN_FILE,
    Q_EVENT_FILE,
    Q_DIST_FILE,
    REP_FILE,
    QA_FILE,
]:

    manifest["outputs"][p.name] = (
        sha256_file(p)
    )

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 25. FINAL REPORT
# ================================================================================================================

print("\n" + "=" * 120)
print("C–Q FREEZE SUMMARY")
print("=" * 120)

print("\nPHYSICAL HYDROMETRY")
print("Global FIELD events              :", global_field_events)
print("Global observed Q                :", global_observed)
print("Global reconstructed Q           :", global_reconstructed)
print("Global best Q                    :", global_best)

print("\nMASTER-LINKED HYDROMETRY")
print("FIELD events represented         :", master_field_events)
print("PRIMARY observed-Q events        :", master_observed_events)
print("Reconstructed-only C09 events    :", master_reconstructed_events)
print("SENSITIVITY Q events             :", master_sensitivity_events)

print("\nRESPONSE ARCHITECTURE")
print("Frozen response rows             :", len(cq))
print("Frozen response identities       :", cq["response_key"].nunique())
print("Repeated sample×response combos  :", len(rep))

print("\nFILES")
print("-" * 120)

for p in [
    CQ_FILE,
    COVERAGE_FILE,
    CAMPAIGN_FILE,
    Q_EVENT_FILE,
    Q_DIST_FILE,
    REP_FILE,
    QA_FILE,
    MANIFEST_FILE,
]:

    print(
        p.name,
        "| exists:",
        p.exists(),
        "| bytes:",
        p.stat().st_size
        if p.exists()
        else None
    )

print("\n" + "=" * 120)
print("BLOCK 9.0C-R FINAL GATE")
print("=" * 120)

print("BLOCK 9.0C-R PASS                :", PASS)
print("Q RECONSTRUCTED                  :", False)
print("ANALYTICAL SUBSAMPLES AVERAGED   :", False)
print("CENSOR VALUES SUBSTITUTED        :", False)
print("RESPONSES SELECTED USING Q       :", False)
print("C-Q MODELS FITTED                :", False)
print("DILUTION/MOBILIZATION CLASSIFIED :", False)
print("LOG-Q TRANSFORMATION FROZEN      :", False)

print(
    "\nNEXT: BLOCK 9.0D — freeze C–Q statistical architecture "
    "and Q transformation using the physical Q distribution only."
)

---
### Imported master block 111

`BLOCK 9.0D — C–Q STATISTICAL ARCHITECTURE + Q TRANSFORMATION FREEZE`


In [ ]:
# ================================================================================================================
# BLOCK 9.0D — C–Q STATISTICAL ARCHITECTURE + Q TRANSFORMATION FREEZE
#
# PURPOSE
#   Freeze the C–Q inferential rules BEFORE inspecting any response–Q association.
#
# PRIMARY Q:
#   observed discharge only
#
# SENSITIVITY Q:
#   observed + reconstructed C09
#
# Q TRANSFORMATION:
#   log10(Q)
#
# IMPORTANT:
#   - Q transformation justified ONLY from physical Q distribution.
#   - No response–Q correlations inspected.
#   - No slopes estimated.
#   - No dilution/mobilization classification.
#   - No model fitted.
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 120)
print("BLOCK 9.0D — C–Q STATISTICAL ARCHITECTURE + Q TRANSFORMATION FREEZE")
print("=" * 120)

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
Q_DIST_FILE = CQDIR / "CQ_Q_DISTRIBUTION_AUDIT_v1.csv"

assert CQ_FILE.exists()
assert Q_DIST_FILE.exists()

cq = pd.read_csv(CQ_FILE, low_memory=False)
qdist = pd.read_csv(Q_DIST_FILE)

print("\nINPUT")
print("-" * 120)
print("C–Q rows :", len(cq))
print("Responses:", cq["response_key"].nunique())

assert len(cq) == 1300
assert cq["response_key"].nunique() == 10

# ================================================================================================================
# 1. VERIFY Q TRANSFORMATION FROM PHYSICAL Q DISTRIBUTION ONLY
# ================================================================================================================

primary = qdist.loc[
    qdist["Q_role"].eq("PRIMARY_OBSERVED")
].iloc[0]

assert int(primary["n_events"]) == 115
assert float(primary["min"]) > 0

raw_skew = float(primary["raw_skew"])
log_skew = float(primary["log10_skew"])
orders = float(primary["orders_of_magnitude"])

print("\nPRIMARY PHYSICAL Q DISTRIBUTION")
print("-" * 120)
print("n events            :", int(primary["n_events"]))
print("min Q               :", float(primary["min"]))
print("median Q            :", float(primary["median"]))
print("max Q               :", float(primary["max"]))
print("raw skew            :", raw_skew)
print("log10(Q) skew       :", log_skew)
print("orders of magnitude :", orders)

# Pre-response justification.
q_transform_pass = bool(
    float(primary["min"]) > 0
    and
    orders > 2
    and
    abs(log_skew) < abs(raw_skew)
)

assert q_transform_pass

# ================================================================================================================
# 2. RESPONSE ARCHITECTURE
#
# We retain the response transformations already frozen BEFORE hydro/Q inference.
# ================================================================================================================

arch_cols = [
    "response_key",
    "response_label",
    "response_family",
    "primary_transformation",
    "statistical_pathway",
]

arch = (
    cq[arch_cols]
    .drop_duplicates()
    .sort_values("response_key")
    .reset_index(drop=True)
)

assert len(arch) == 10

# C–Q role is separate from the MEMORY interpretation.
arch["cq_primary_Q"] = "LOG10_OBSERVED_Q"
arch["cq_sensitivity_Q"] = "LOG10_OBSERVED_PLUS_RECONSTRUCTED_C09"

arch["cq_primary_model"] = (
    "Bayesian hierarchical response ~ log10(Q) + station random intercept"
)

arch["cq_validation"] = "leave-one-campaign-out (LOCO)"

arch["cq_reconstructed_Q_role"] = "SENSITIVITY_ONLY"

arch["cq_subsample_policy"] = (
    "retain distinct analytical subsamples; shared physical Q allowed"
)

arch["cq_censoring_policy"] = np.where(
    arch["statistical_pathway"].astype(str).str.contains(
        "CENSOR",
        case=False,
        na=False
    ),
    "explicit sample-specific left-censor likelihood",
    "direct continuous likelihood"
)

# Interpretation policy.
concentration_families = {
    "MINERALIZATION",
    "NUTRIENT_N",
    "ALKALI_WEATHERING",
    "SULFATE",
    "PARTICLE_ASSOCIATED_METAL",
    "REDOX_METAL",
    "SILICATE_WEATHERING",
}

arch["cq_interpretation_scope"] = np.where(
    arch["response_family"].isin(concentration_families),
    "concentration-discharge process interpretation eligible",
    "response-discharge association only"
)

print("\nFROZEN C–Q RESPONSE ARCHITECTURE")
print("-" * 120)
print(arch.to_string(index=False))

# ================================================================================================================
# 3. PRE-SPECIFIED INFERENCE RULES
# ================================================================================================================

rules = [
    (
        "CQ01",
        "Primary discharge inference uses Q_primary_CQ only: directly observed discharge."
    ),
    (
        "CQ02",
        "Q_sensitivity_CQ, including reconstructed C09 discharge, is sensitivity analysis only."
    ),
    (
        "CQ03",
        "Discharge is represented as log10(Q) because all Q values are positive, span >4 orders of magnitude, "
        "and log10 transformation strongly reduces physical-event Q skewness; no response information was used."
    ),
    (
        "CQ04",
        "The 10 response identities remain those frozen before C–Q analysis; no response may be added or removed "
        "because of its association with Q."
    ),
    (
        "CQ05",
        "Previously frozen response transformations are retained for the primary cross-response C–Q analysis."
    ),
    (
        "CQ06",
        "Censored responses use explicit sample-specific left-censor likelihoods; no LOD/2 or other substitution."
    ),
    (
        "CQ07",
        "Distinct analytical subsamples are retained and are not averaged; subsamples from the same FIELD event "
        "share the physical discharge observation."
    ),
    (
        "CQ08",
        "Station structure is represented explicitly with a random intercept."
    ),
    (
        "CQ09",
        "Campaign is the predictive validation unit; random row train/test splits are prohibited."
    ),
    (
        "CQ10",
        "Leave-one-campaign-out validation uses the complete held-out campaign."
    ),
    (
        "CQ11",
        "Primary inference is based on posterior effect direction, uncertainty, and campaign-held-out predictive evidence; "
        "raw Spearman correlations are diagnostic only."
    ),
    (
        "CQ12",
        "Negative Q effects may be described as dilution-compatible and positive Q effects as mobilization/flushing-compatible "
        "only for concentration-like responses and only when inferential support is robust."
    ),
    (
        "CQ13",
        "A weak or unsupported Q effect is not automatically evidence of chemostasis."
    ),
    (
        "CQ14",
        "The term chemostatic or relative chemostasis requires a concentration-discharge formulation appropriate to that "
        "response and uncertainty compatible with a near-zero concentration-discharge exponent."
    ),
    (
        "CQ15",
        "pH, total coliforms, and water temperature are interpreted as response-discharge relationships rather than "
        "classical concentration-discharge exponents."
    ),
    (
        "CQ16",
        "EC may be interpreted as integrated mineralization/dilution response but not as a literal solute concentration."
    ),
    (
        "CQ17",
        "Observed-Q primary and reconstructed-Q sensitivity results must be reported separately; reconstructed C09 values "
        "must never silently enter primary inference."
    ),
    (
        "CQ18",
        "No hydroclimatic antecedent metric enters the C–Q model-selection stage; joint Q + antecedent-state models belong "
        "to the subsequent integration block."
    ),
    (
        "CQ19",
        "No catchment-connectivity mechanism is inferred from Q slopes alone."
    ),
    (
        "CQ20",
        "C–Q interpretation must distinguish statistical association from hydrological mechanism."
    ),
]

rules_df = pd.DataFrame(
    rules,
    columns=["rule_id", "rule"]
)

print("\nFROZEN C–Q INFERENCE RULES")
print("-" * 120)
print(rules_df.to_string(index=False))

# ================================================================================================================
# 4. PRIMARY MODEL SPECIFICATION
# ================================================================================================================

model_spec = pd.DataFrame([
    {
        "component": "discharge_source",
        "primary_specification": "observed Q only",
        "sensitivity_specification": "observed + reconstructed C09"
    },
    {
        "component": "Q_transformation",
        "primary_specification": "log10(Q)",
        "sensitivity_specification": "log10(Q)"
    },
    {
        "component": "response_transformation",
        "primary_specification": "frozen response-specific transformation",
        "sensitivity_specification": "same frozen transformation"
    },
    {
        "component": "fixed_effect",
        "primary_specification": "log10(Q)",
        "sensitivity_specification": "log10(Q)"
    },
    {
        "component": "group_structure",
        "primary_specification": "station random intercept",
        "sensitivity_specification": "station random intercept"
    },
    {
        "component": "censoring",
        "primary_specification": "explicit sample-specific left censor where applicable",
        "sensitivity_specification": "same"
    },
    {
        "component": "validation",
        "primary_specification": "leave-one-campaign-out",
        "sensitivity_specification": "leave-one-campaign-out"
    },
])

print("\nMODEL SPECIFICATION")
print("-" * 120)
print(model_spec.to_string(index=False))

# ================================================================================================================
# 5. FORMAL FREEZE FLAGS
# ================================================================================================================

freeze = {
    "block": "9.0D",
    "created_at": datetime.now().isoformat(),

    "primary_Q_source": "Q_primary_CQ",
    "primary_Q_role": "observed_only",

    "sensitivity_Q_source": "Q_sensitivity_CQ",
    "sensitivity_Q_role": "observed_plus_reconstructed_C09",

    "Q_transformation": "LOG10",
    "Q_transformation_frozen": True,

    "Q_transform_reason": (
        f"Physical Q only: positive values; {orders:.6f} orders of magnitude; "
        f"raw skew={raw_skew:.6f}; log10 skew={log_skew:.6f}."
    ),

    "response_information_used_to_choose_Q_transform": False,
    "response_selection_using_Q": False,

    "response_transformations_reused_from_pre_Q_freeze": True,

    "analytical_subsamples_averaged": False,
    "censor_substitution": False,

    "primary_model": (
        "Bayesian hierarchical response ~ log10(Q) + station random intercept"
    ),

    "validation": "leave_one_campaign_out",

    "CQ_models_fitted": False,
    "CQ_slopes_estimated": False,
    "dilution_mobilization_classified": False,
    "chemostasis_classified": False,

    "hydroclimate_entered_CQ_selection": False,
    "connectivity_inferred": False,

    "pass": True,
}

# ================================================================================================================
# 6. SAVE
# ================================================================================================================

ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"
RULE_FILE = CQDIR / "CQ_INFERENCE_RULES_v1.csv"
MODEL_FILE = CQDIR / "CQ_MODEL_SPECIFICATION_v1.csv"
FREEZE_FILE = CQDIR / "BLOCK90D_CQ_ARCHITECTURE_FREEZE_v1.json"

arch.to_csv(ARCH_FILE, index=False)
rules_df.to_csv(RULE_FILE, index=False)
model_spec.to_csv(MODEL_FILE, index=False)

with open(FREEZE_FILE, "w", encoding="utf-8") as f:
    json.dump(freeze, f, indent=2)

# ================================================================================================================
# 7. HASHES
# ================================================================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("\nFROZEN OUTPUTS")
print("-" * 120)

for p in [
    ARCH_FILE,
    RULE_FILE,
    MODEL_FILE,
    FREEZE_FILE,
]:
    print(
        p.name,
        "| SHA256:",
        sha256_file(p)
    )

# ================================================================================================================
# ================================================================================================================
# 8. FINAL GATE
# ================================================================================================================

qa = {
    "input_rows_1300":
        len(cq) == 1300,

    "responses_10":
        cq["response_key"].nunique() == 10,

    "primary_Q_events_115":
        int(primary["n_events"]) == 115,

    "primary_Q_positive":
        float(primary["min"]) > 0,

    "Q_spans_gt_2_orders":
        orders > 2,

    "logQ_reduces_skew":
        abs(log_skew) < abs(raw_skew),

    "Q_transform_response_independent":
        True,

    "Q_transform_frozen_log10":
        freeze["Q_transformation"] == "LOG10",

    "reconstructed_Q_sensitivity_only":
        freeze["sensitivity_Q_role"] == "observed_plus_reconstructed_C09",

    "no_models_fitted":
        freeze["CQ_models_fitted"] is False,

    "no_CQ_classification":
        freeze["dilution_mobilization_classified"] is False,

    "no_hydroclimate_selection":
        freeze["hydroclimate_entered_CQ_selection"] is False,
}

qa_df = pd.DataFrame({
    "check": list(qa.keys()),
    "pass": list(qa.values())
})

PASS = bool(qa_df["pass"].all())

print("\n" + "=" * 120)
print("BLOCK 9.0D QA")
print("=" * 120)

print(qa_df.to_string(index=False))

print("\n" + "=" * 120)
print("BLOCK 9.0D FINAL GATE")
print("=" * 120)

print("BLOCK 9.0D PASS                     :", PASS)
print("PRIMARY Q                           : OBSERVED ONLY")
print("SENSITIVITY Q                       : OBSERVED + C09 RECONSTRUCTED")
print("Q TRANSFORMATION                    : LOG10 — FROZEN")
print("Q TRANSFORM CHOSEN USING RESPONSES  : False")
print("RESPONSES SELECTED USING Q          : False")
print("CENSOR SUBSTITUTION                 : False")
print("ANALYTICAL SUBSAMPLES AVERAGED      : False")
print("C-Q MODELS FITTED                   : False")
print("DILUTION/MOBILIZATION CLASSIFIED    : False")
print("CHEMOSTASIS CLASSIFIED              : False")
print("CONNECTIVITY INFERRED               : False")

assert PASS

print(
    "\nNEXT: BLOCK 9.1A — response-independent C–Q model-engine validation "
    "on pre-specified representative direct and censored responses."
)

---
### Imported master block 112

`BLOCK 9.1A — BAYESIAN C–Q ENGINE VALIDATION`


In [ ]:
# ================================================================================================================
# BLOCK 9.1A — BAYESIAN C–Q ENGINE VALIDATION
#
# VALIDATION CASES — PRE-SPECIFIED FOR COMPUTATIONAL COVERAGE
#   DIRECT   : Na  (LOG10 response, uncensored)
#   CENSORED : Al  (LOG10 response, sample-specific left-censor limits)
#
# MODEL
#   transformed(Y)_std =
#       alpha + beta_Q * standardized(log10(Q)) + station_random_intercept + error
#
# PRIMARY Q ONLY:
#   Q_primary_CQ = observed discharge
#
# IMPORTANT:
#   - This validates the engine only.
#   - No process classification.
#   - No response ranking.
#   - No reconstructed C09 Q.
#   - No LOCO yet.
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")

print("=" * 120)
print("BLOCK 9.1A — BAYESIAN C–Q ENGINE VALIDATION")
print("=" * 120)

# ----------------------------------------------------------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"
FREEZE_FILE = CQDIR / "BLOCK90D_CQ_ARCHITECTURE_FREEZE_v1.json"

for p in [CQ_FILE, ARCH_FILE, FREEZE_FILE]:
    assert p.exists(), f"MISSING INPUT: {p}"

cq = pd.read_csv(CQ_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE)

with open(FREEZE_FILE, "r", encoding="utf-8") as f:
    freeze = json.load(f)

assert freeze["Q_transformation"] == "LOG10"
assert freeze["primary_Q_role"] == "observed_only"

print("\nINPUT")
print("-" * 120)
print("C-Q rows :", len(cq))
print("Responses:", cq["response_key"].nunique())
print("PyMC     :", pm.__version__)
print("ArviZ    :", az.__version__)

# ----------------------------------------------------------------------------------------------------------------
# 2. PRE-SPECIFIED VALIDATION CASES
# ----------------------------------------------------------------------------------------------------------------

DIRECT_KEY = "AC::Na::mg/L"
CENSORED_KEY = "METALS::Aluminio::mg/L"

cases = [
    {
        "case": "DIRECT_NA",
        "response_key": DIRECT_KEY,
        "expected_transform": "LOG10",
        "expected_pathway": "DIRECT_CONTINUOUS",
    },
    {
        "case": "CENSORED_AL",
        "response_key": CENSORED_KEY,
        "expected_transform": "LOG10",
        "expected_pathway": "CENSOR_AWARE_CONTINUOUS",
    },
]

# ----------------------------------------------------------------------------------------------------------------
# 3. ROBUST BOOLEAN PARSER
# ----------------------------------------------------------------------------------------------------------------

def parse_bool(x):
    if pd.isna(x):
        return False

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    return str(x).strip().lower() in {
        "true", "1", "yes", "y"
    }

# ----------------------------------------------------------------------------------------------------------------
# 4. PREPARE ONE RESPONSE
# ----------------------------------------------------------------------------------------------------------------

def prepare_case(case):

    key = case["response_key"]

    g = cq.loc[
        cq["response_key"].eq(key)
        &
        cq["Q_primary_CQ"].notna()
    ].copy()

    assert len(g) > 0

    # Primary observed Q only.
    assert g["cq_q_source"].eq("OBSERVED_PRIMARY").all()

    # Architecture cross-check.
    a = arch.loc[
        arch["response_key"].eq(key)
    ].iloc[0]

    assert a["primary_transformation"] == case["expected_transform"]
    assert a["statistical_pathway"] == case["expected_pathway"]

    g["is_censored"] = g["response_censored"].map(parse_bool)

    # Q transformation frozen in 9.0D.
    q = pd.to_numeric(
        g["Q_primary_CQ"],
        errors="coerce"
    )

    assert q.notna().all()
    assert (q > 0).all()

    g["x_logQ"] = np.log10(q)

    # Response numeric value and censor limit.
    y_raw = pd.to_numeric(
        g["response_value_numeric"],
        errors="coerce"
    )

    limit_raw = pd.to_numeric(
        g["response_censor_limit"],
        errors="coerce"
    )

    # LOG10 response architecture.
    #
    # Detected rows:
    #   y = log10(observed response)
    #
    # Censored rows:
    #   observed latent response <= log10(sample-specific censor limit)
    #
    detected = ~g["is_censored"]

    assert (y_raw.loc[detected] > 0).all()

    g["y_transformed"] = np.nan
    g.loc[detected, "y_transformed"] = np.log10(
        y_raw.loc[detected]
    )

    if g["is_censored"].any():

        assert limit_raw.loc[
            g["is_censored"]
        ].notna().all()

        assert (
            limit_raw.loc[
                g["is_censored"]
            ] > 0
        ).all()

        g["censor_limit_transformed"] = np.nan

        g.loc[
            g["is_censored"],
            "censor_limit_transformed"
        ] = np.log10(
            limit_raw.loc[
                g["is_censored"]
            ]
        )

    else:
        g["censor_limit_transformed"] = np.nan

    # -------------------------------------------------------------------------
    # Standardization:
    # Response scaling uses DETECTED transformed values only.
    # Q scaling uses all primary-Q rows.
    # -------------------------------------------------------------------------

    y_detected = g.loc[
        detected,
        "y_transformed"
    ].astype(float)

    y_mean = float(y_detected.mean())
    y_sd = float(y_detected.std(ddof=1))

    x_mean = float(g["x_logQ"].mean())
    x_sd = float(g["x_logQ"].std(ddof=1))

    assert np.isfinite(y_sd) and y_sd > 0
    assert np.isfinite(x_sd) and x_sd > 0

    g["x_std"] = (
        g["x_logQ"] - x_mean
    ) / x_sd

    g["y_std"] = (
        g["y_transformed"] - y_mean
    ) / y_sd

    g["censor_limit_std"] = (
        g["censor_limit_transformed"] - y_mean
    ) / y_sd

    # -------------------------------------------------------------------------
    # Station grouping
    #
    # Prefer frozen physical station ID.
    # Explicit fallback avoids dropping unresolved identity.
    # -------------------------------------------------------------------------

    station = (
        g["physical_station_id_v1"]
        .astype("string")
    )

    fallback = (
        g["station_id"]
        .astype("string")
    )

    station_group = station.fillna(
        fallback
    )

    # Final explicit fallback.
    missing_station = station_group.isna()

    station_group.loc[
        missing_station
    ] = (
        "UNRESOLVED::"
        +
        g.loc[
            missing_station
        ].index.astype(str)
    )

    g["station_group"] = station_group.astype(str)

    station_levels = sorted(
        g["station_group"].unique()
    )

    station_map = {
        s: i
        for i, s in enumerate(station_levels)
    }

    g["station_idx"] = (
        g["station_group"]
        .map(station_map)
        .astype(int)
    )

    return (
        g.reset_index(drop=True),
        {
            "x_mean": x_mean,
            "x_sd": x_sd,
            "y_mean": y_mean,
            "y_sd": y_sd,
            "n_station": len(station_levels),
        }
    )

# ----------------------------------------------------------------------------------------------------------------
# 5. BAYESIAN ENGINE
# ----------------------------------------------------------------------------------------------------------------

def fit_engine(g, metadata, case_name):

    x = g["x_std"].to_numpy(dtype=float)
    station_idx = g["station_idx"].to_numpy(dtype=int)

    cens = g["is_censored"].to_numpy(dtype=bool)
    detected = ~cens

    y_det = g.loc[
        detected,
        "y_std"
    ].to_numpy(dtype=float)

    x_det = x[detected]
    station_det = station_idx[detected]

    n_station = metadata["n_station"]

    with pm.Model() as model:

        alpha = pm.Normal(
            "alpha",
            mu=0,
            sigma=1
        )

        beta_Q = pm.Normal(
            "beta_Q",
            mu=0,
            sigma=1
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1
        )

        z_station = pm.Normal(
            "z_station",
            mu=0,
            sigma=1,
            shape=n_station
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1
        )

        # ---------------------------------------------------------------------
        # Detected likelihood
        # ---------------------------------------------------------------------

        mu_det = (
            alpha
            +
            beta_Q * x_det
            +
            station_effect[station_det]
        )

        pm.Normal(
            "y_detected",
            mu=mu_det,
            sigma=sigma,
            observed=y_det
        )

        # ---------------------------------------------------------------------
        # Explicit left-censor likelihood
        #
        # P(Y <= L) = Phi((L - mu)/sigma)
        # ---------------------------------------------------------------------

        if cens.any():

            x_cens = x[cens]
            station_cens = station_idx[cens]

            limit_std = g.loc[
                cens,
                "censor_limit_std"
            ].to_numpy(dtype=float)

            assert np.isfinite(limit_std).all()

            mu_cens = (
                alpha
                +
                beta_Q * x_cens
                +
                station_effect[station_cens]
            )

            normal_dist = pm.Normal.dist(
                mu=mu_cens,
                sigma=sigma
            )

            logcdf = pm.logcdf(
                normal_dist,
                pt.as_tensor_variable(limit_std)
            )

            pm.Potential(
                "left_censor_loglik",
                pt.sum(logcdf)
            )

        idata = pm.sample(
            draws=1500,
            tune=1500,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=20260915,
            return_inferencedata=True,
            progressbar=True
        )

    return model, idata

# ----------------------------------------------------------------------------------------------------------------
# 6. RUN VALIDATION CASES
# ----------------------------------------------------------------------------------------------------------------

summary_rows = []
idata_objects = {}

for case in cases:

    print("\n" + "=" * 120)
    print("VALIDATION CASE:", case["case"])
    print("=" * 120)

    g, metadata = prepare_case(case)

    print("Response key      :", case["response_key"])
    print("Rows              :", len(g))
    print("Detected          :", int((~g["is_censored"]).sum()))
    print("Censored          :", int(g["is_censored"].sum()))
    print("Stations          :", metadata["n_station"])
    print("Campaigns         :", g["campaign_id"].nunique())
    print("Q min             :", g["Q_primary_CQ"].min())
    print("Q median          :", g["Q_primary_CQ"].median())
    print("Q max             :", g["Q_primary_CQ"].max())

    if g["is_censored"].any():

        print(
            "Censor limits     :",
            sorted(
                g.loc[
                    g["is_censored"],
                    "response_censor_limit"
                ]
                .dropna()
                .astype(float)
                .unique()
                .tolist()
            )
        )

    model, idata = fit_engine(
        g,
        metadata,
        case["case"]
    )

    idata_objects[
        case["case"]
    ] = idata

    s = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_Q",
            "sigma",
            "sigma_station",
        ],
        hdi_prob=0.95
    )

    print("\nPOSTERIOR SUMMARY")
    print("-" * 120)
    print(s.to_string())

    # -------------------------------------------------------------------------
    # Diagnostics
    # -------------------------------------------------------------------------

    div = int(
        idata.sample_stats["diverging"]
        .sum()
        .values
    )

    max_rhat = float(
        s["r_hat"].max()
    )

    min_ess_bulk = float(
        s["ess_bulk"].min()
    )

    beta_samples = (
        idata.posterior["beta_Q"]
        .values
        .reshape(-1)
    )

    beta_mean = float(
        beta_samples.mean()
    )

    beta_hdi = az.hdi(
        beta_samples,
        hdi_prob=0.95
    )

    p_positive = float(
        np.mean(beta_samples > 0)
    )

    p_negative = float(
        np.mean(beta_samples < 0)
    )

    p_direction = max(
        p_positive,
        p_negative
    )

    summary_rows.append({

        "case":
            case["case"],

        "response_key":
            case["response_key"],

        "n_rows":
            len(g),

        "n_detected":
            int((~g["is_censored"]).sum()),

        "n_censored":
            int(g["is_censored"].sum()),

        "n_stations":
            metadata["n_station"],

        "n_campaigns":
            g["campaign_id"].nunique(),

        "beta_Q_mean":
            beta_mean,

        "beta_Q_hdi_low":
            float(beta_hdi[0]),

        "beta_Q_hdi_high":
            float(beta_hdi[1]),

        "posterior_direction_probability":
            p_direction,

        "divergences":
            div,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess_bulk,

        "engine_pass":
            (
                div == 0
                and
                max_rhat <= 1.05
                and
                min_ess_bulk >= 200
            ),

        "scientific_inference_role":
            "ENGINE_VALIDATION_ONLY",

        "process_classification_performed":
            False,
    })

# ----------------------------------------------------------------------------------------------------------------
# 7. VALIDATION SUMMARY
# ----------------------------------------------------------------------------------------------------------------

validation = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 120)
print("ENGINE VALIDATION SUMMARY")
print("=" * 120)

print(
    validation.to_string(
        index=False
    )
)

PASS = bool(
    validation["engine_pass"].all()
)

# ----------------------------------------------------------------------------------------------------------------
# 8. SPECIAL CENSORING CHECK
# ----------------------------------------------------------------------------------------------------------------

al_row = validation.loc[
    validation["case"].eq("CENSORED_AL")
].iloc[0]

na_row = validation.loc[
    validation["case"].eq("DIRECT_NA")
].iloc[0]

censor_case_pass = bool(
    int(al_row["n_censored"]) > 0
)

direct_case_pass = bool(
    int(na_row["n_censored"]) == 0
)

assert censor_case_pass
assert direct_case_pass

# ----------------------------------------------------------------------------------------------------------------
# 9. SAVE VALIDATION OUTPUTS
# ----------------------------------------------------------------------------------------------------------------

OUT = CQDIR / "ENGINE_VALIDATION"
OUT.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_FILE = (
    OUT /
    "BLOCK91A_CQ_ENGINE_VALIDATION_v1.csv"
)

MANIFEST_FILE = (
    OUT /
    "BLOCK91A_CQ_ENGINE_VALIDATION_MANIFEST_v1.json"
)

validation.to_csv(
    SUMMARY_FILE,
    index=False
)

# Save posterior NetCDF for reproducibility.
posterior_files = {}

for case_name, idata in idata_objects.items():

    p = (
        OUT /
        f"BLOCK91A_{case_name}_posterior_v1.nc"
    )

    idata.to_netcdf(p)

    posterior_files[
        case_name
    ] = str(p)

# ----------------------------------------------------------------------------------------------------------------
# 10. MANIFEST
# ----------------------------------------------------------------------------------------------------------------

manifest = {

    "block":
        "9.1A",

    "created_at":
        datetime.now().isoformat(),

    "purpose":
        "Bayesian C-Q computational engine validation",

    "validation_cases": [
        "Na direct continuous LOG10",
        "Al censor-aware LOG10"
    ],

    "case_selection_based_on_Q_association":
        False,

    "primary_Q":
        "observed only",

    "Q_transformation":
        "log10",

    "station_random_intercept":
        True,

    "censoring":
        "explicit left log-CDF with sample-specific limits",

    "response_scaling":
        "detected transformed response values",

    "Q_scaling":
        "all primary observed-Q rows within response",

    "LOCO_performed":
        False,

    "process_classification_performed":
        False,

    "full_10_response_inference_performed":
        False,

    "pass":
        PASS,

    "posterior_files":
        posterior_files,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ----------------------------------------------------------------------------------------------------------------
# 11. FINAL GATE
# ----------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.1A FINAL GATE")
print("=" * 120)

print("BLOCK 9.1A PASS                  :", PASS)
print("DIRECT ENGINE VALIDATED          :", direct_case_pass)
print("CENSOR ENGINE VALIDATED          :", censor_case_pass)
print("PRIMARY Q ONLY                   :", True)
print("RECONSTRUCTED C09 USED           :", False)
print("LOD/2 SUBSTITUTION               :", False)
print("ANALYTICAL SUBSAMPLES AVERAGED   :", False)
print("LOCO PERFORMED                   :", False)
print("PROCESS CLASSIFICATION PERFORMED :", False)
print("FULL C-Q INFERENCE PERFORMED     :", False)

assert PASS

print(
    "\nNEXT: BLOCK 9.1B — validate campaign-held-out C-Q prediction "
    "before fitting the complete 10-response primary C-Q spectrum."
)

---
### Imported master block 113

`BLOCK 9.1B — CAMPAIGN-HELD-OUT C–Q PREDICTION ENGINE VALIDATION`


In [ ]:
# ================================================================================================================
# BLOCK 9.1B — CAMPAIGN-HELD-OUT C–Q PREDICTION ENGINE VALIDATION
#
# PURPOSE
#   Validate LOCO prediction BEFORE full 10-response C–Q inference.
#
# VALIDATION CASES
#   DIRECT   : Na
#   CENSORED : Al
#
# PRIMARY Q ONLY
#   observed Q
#
# LOCO RULES
#   - hold out one complete campaign
#   - train-only response scaling
#   - train-only log10(Q) scaling
#   - train-only station encoding
#   - known test station: posterior station effect used
#   - unseen test station: random-effect mean = 0 and sigma_station^2 added
#     to predictive variance
#   - detected row score: Normal log predictive density
#   - censored row score: log predictive CDF at its sample-specific limit
#
# IMPORTANT
#   ENGINE VALIDATION ONLY.
#   No process classification.
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from scipy.special import log_ndtr
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")

print("=" * 120)
print("BLOCK 9.1B — CAMPAIGN-HELD-OUT C–Q PREDICTION ENGINE VALIDATION")
print("=" * 120)

# ------------------------------------------------------------------------------------------------
# 1. PATHS / INPUTS
# ------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"
FREEZE_FILE = CQDIR / "BLOCK90D_CQ_ARCHITECTURE_FREEZE_v1.json"

for p in [CQ_FILE, ARCH_FILE, FREEZE_FILE]:
    assert p.exists(), f"MISSING: {p}"

cq = pd.read_csv(CQ_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE)

with open(FREEZE_FILE, "r", encoding="utf-8") as f:
    freeze = json.load(f)

assert freeze["Q_transformation"] == "LOG10"
assert freeze["primary_Q_role"] == "observed_only"

print("\nINPUT")
print("-" * 120)
print("Rows      :", len(cq))
print("Responses :", cq["response_key"].nunique())
print("Campaigns :", cq["campaign_id"].nunique())
print("PyMC      :", pm.__version__)
print("ArviZ     :", az.__version__)

# ------------------------------------------------------------------------------------------------
# 2. PRE-SPECIFIED ENGINE CASES
# ------------------------------------------------------------------------------------------------

CASES = [
    {
        "case": "DIRECT_NA",
        "response_key": "AC::Na::mg/L",
        "transform": "LOG10",
        "pathway": "DIRECT_CONTINUOUS",
    },
    {
        "case": "CENSORED_AL",
        "response_key": "METALS::Aluminio::mg/L",
        "transform": "LOG10",
        "pathway": "CENSOR_AWARE_CONTINUOUS",
    },
]

def parse_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    return str(x).strip().lower() in {"true", "1", "yes", "y"}

# ------------------------------------------------------------------------------------------------
# 3. BASE RESPONSE PREPARATION
# ------------------------------------------------------------------------------------------------

def prepare_base(case):

    key = case["response_key"]

    g = cq.loc[
        cq["response_key"].eq(key) &
        cq["Q_primary_CQ"].notna()
    ].copy()

    assert len(g) > 0
    assert g["cq_q_source"].eq("OBSERVED_PRIMARY").all()

    a = arch.loc[arch["response_key"].eq(key)].iloc[0]

    assert a["primary_transformation"] == case["transform"]
    assert a["statistical_pathway"] == case["pathway"]

    g["is_censored"] = g["response_censored"].map(parse_bool)

    q = pd.to_numeric(g["Q_primary_CQ"], errors="coerce")
    assert q.notna().all()
    assert (q > 0).all()

    g["x_logQ"] = np.log10(q)

    y = pd.to_numeric(g["response_value_numeric"], errors="coerce")
    L = pd.to_numeric(g["response_censor_limit"], errors="coerce")

    det = ~g["is_censored"]

    assert y.loc[det].notna().all()
    assert (y.loc[det] > 0).all()

    # Both validation responses have frozen LOG10 transformation.
    g["y_transformed"] = np.nan
    g.loc[det, "y_transformed"] = np.log10(y.loc[det])

    g["limit_transformed"] = np.nan

    if g["is_censored"].any():
        assert L.loc[g["is_censored"]].notna().all()
        assert (L.loc[g["is_censored"]] > 0).all()

        g.loc[
            g["is_censored"],
            "limit_transformed"
        ] = np.log10(
            L.loc[g["is_censored"]]
        )

    # Stable station grouping.
    station = g["physical_station_id_v1"].astype("string")
    fallback = g["station_id"].astype("string")

    g["station_group"] = station.fillna(fallback)

    missing = g["station_group"].isna()

    g.loc[missing, "station_group"] = (
        "UNRESOLVED::" +
        g.loc[missing].index.astype(str)
    )

    g["station_group"] = g["station_group"].astype(str)

    return g.reset_index(drop=True)

# ------------------------------------------------------------------------------------------------
# 4. TRAIN-ONLY SCALING
# ------------------------------------------------------------------------------------------------

def scale_fold(train, test):

    train = train.copy()
    test = test.copy()

    train_det = ~train["is_censored"]

    # Response scaling ONLY from detected TRAINING responses.
    y_train_det = train.loc[
        train_det,
        "y_transformed"
    ].astype(float)

    y_mean = float(y_train_det.mean())
    y_sd = float(y_train_det.std(ddof=1))

    # Q scaling ONLY from TRAINING Q.
    x_mean = float(train["x_logQ"].mean())
    x_sd = float(train["x_logQ"].std(ddof=1))

    assert np.isfinite(y_sd) and y_sd > 0
    assert np.isfinite(x_sd) and x_sd > 0

    for df in [train, test]:

        df["x_std"] = (
            df["x_logQ"] - x_mean
        ) / x_sd

        df["y_std"] = (
            df["y_transformed"] - y_mean
        ) / y_sd

        df["limit_std"] = (
            df["limit_transformed"] - y_mean
        ) / y_sd

    scaling = {
        "x_mean_train": x_mean,
        "x_sd_train": x_sd,
        "y_mean_train": y_mean,
        "y_sd_train": y_sd,
    }

    return train, test, scaling

# ------------------------------------------------------------------------------------------------
# 5. TRAIN MODEL
# ------------------------------------------------------------------------------------------------

def fit_train(train, seed):

    train = train.copy()

    station_levels = sorted(
        train["station_group"].unique()
    )

    station_map = {
        s: i for i, s in enumerate(station_levels)
    }

    train["station_idx"] = (
        train["station_group"]
        .map(station_map)
        .astype(int)
    )

    x = train["x_std"].to_numpy(float)
    station_idx = train["station_idx"].to_numpy(int)

    cens = train["is_censored"].to_numpy(bool)
    det = ~cens

    y_det = train.loc[det, "y_std"].to_numpy(float)

    with pm.Model() as model:

        alpha = pm.Normal("alpha", 0, 1)
        beta_Q = pm.Normal("beta_Q", 0, 1)

        sigma_station = pm.HalfNormal(
            "sigma_station", 1
        )

        z_station = pm.Normal(
            "z_station",
            0,
            1,
            shape=len(station_levels)
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal("sigma", 1)

        mu_det = (
            alpha
            + beta_Q * x[det]
            + station_effect[station_idx[det]]
        )

        pm.Normal(
            "y_detected",
            mu=mu_det,
            sigma=sigma,
            observed=y_det
        )

        if cens.any():

            limits = train.loc[
                cens,
                "limit_std"
            ].to_numpy(float)

            assert np.isfinite(limits).all()

            mu_cens = (
                alpha
                + beta_Q * x[cens]
                + station_effect[station_idx[cens]]
            )

            dist = pm.Normal.dist(
                mu=mu_cens,
                sigma=sigma
            )

            pm.Potential(
                "left_censor_loglik",
                pt.sum(
                    pm.logcdf(
                        dist,
                        pt.as_tensor_variable(limits)
                    )
                )
            )

        idata = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed,
            progressbar=False,
            return_inferencedata=True
        )

    return idata, station_map

# ------------------------------------------------------------------------------------------------
# 6. POSTERIOR HELPER
# ------------------------------------------------------------------------------------------------

def flatten_posterior(idata, variable):

    arr = idata.posterior[variable].values

    # scalar: chain × draw
    if arr.ndim == 2:
        return arr.reshape(-1)

    # vector: chain × draw × group
    if arr.ndim == 3:
        return arr.reshape(
            -1,
            arr.shape[-1]
        )

    raise RuntimeError(
        f"Unexpected posterior shape for {variable}: {arr.shape}"
    )

# ------------------------------------------------------------------------------------------------
# 7. HELD-OUT PREDICTION
# ------------------------------------------------------------------------------------------------

def score_test(test, idata, station_map):

    alpha = flatten_posterior(idata, "alpha")
    beta = flatten_posterior(idata, "beta_Q")
    sigma = flatten_posterior(idata, "sigma")
    sigma_station = flatten_posterior(
        idata,
        "sigma_station"
    )

    station_effect = flatten_posterior(
        idata,
        "station_effect"
    )

    rows = []

    for _, r in test.iterrows():

        x = float(r["x_std"])

        station = r["station_group"]

        known_station = station in station_map

        if known_station:

            j = station_map[station]

            mu_draw = (
                alpha
                + beta * x
                + station_effect[:, j]
            )

            predictive_sd = sigma

        else:

            # Marginalize an unseen station random intercept:
            #
            # b_new ~ N(0, sigma_station²)
            #
            # therefore:
            # Y_new | posterior ~
            # N(alpha + beta*x,
            #   sigma² + sigma_station²)
            #
            mu_draw = (
                alpha
                + beta * x
            )

            predictive_sd = np.sqrt(
                sigma**2
                + sigma_station**2
            )

        censored = bool(r["is_censored"])

        if not censored:

            y = float(r["y_std"])

            assert np.isfinite(y)

            # log N(y | mu, sd) per posterior draw
            logp_draw = (
                -0.5 * np.log(2 * np.pi)
                - np.log(predictive_sd)
                - 0.5 * (
                    (y - mu_draw)
                    / predictive_sd
                )**2
            )

        else:

            L = float(r["limit_std"])

            assert np.isfinite(L)

            z = (
                L - mu_draw
            ) / predictive_sd

            # log Phi(z)
            logp_draw = log_ndtr(z)

        # log(mean(exp(logp_draw))) stably
        m = np.max(logp_draw)

        lpd = float(
            m
            + np.log(
                np.mean(
                    np.exp(
                        logp_draw - m
                    )
                )
            )
        )

        # Predictive posterior mean on standardized latent scale.
        pred_mean = float(
            np.mean(mu_draw)
        )

        rows.append({
            "campaign_id": r["campaign_id"],
            "sample_id": r["sample_id"],
            "field_observation_id": r["field_observation_id"],
            "station_group": station,
            "known_station_in_training": known_station,
            "is_censored": censored,
            "observed_y_std": (
                np.nan
                if censored
                else float(r["y_std"])
            ),
            "censor_limit_std": (
                float(r["limit_std"])
                if censored
                else np.nan
            ),
            "predicted_latent_mean_std": pred_mean,
            "log_predictive_density": lpd,
        })

    return pd.DataFrame(rows)

# ------------------------------------------------------------------------------------------------
# 8. RUN ALL 10 HELD-OUT CAMPAIGNS FOR BOTH VALIDATION RESPONSES
# ------------------------------------------------------------------------------------------------

fold_rows = []
prediction_frames = []

for case_i, case in enumerate(CASES):

    base = prepare_base(case)

    campaigns = sorted(
        base["campaign_id"]
        .dropna()
        .unique()
    )

    assert len(campaigns) == 10

    print("\n" + "=" * 120)
    print("CASE:", case["case"])
    print("=" * 120)

    for fold_i, heldout in enumerate(campaigns):

        train = base.loc[
            ~base["campaign_id"].eq(heldout)
        ].copy()

        test = base.loc[
            base["campaign_id"].eq(heldout)
        ].copy()

        assert len(train) > 0
        assert len(test) > 0

        train, test, scaling = scale_fold(
            train,
            test
        )

        seed = (
            20260915
            + case_i * 100
            + fold_i
        )

        idata, station_map = fit_train(
            train,
            seed
        )

        # Convergence.
        s = az.summary(
            idata,
            var_names=[
                "alpha",
                "beta_Q",
                "sigma",
                "sigma_station",
            ]
        )

        div = int(
            idata.sample_stats["diverging"]
            .sum()
            .values
        )

        max_rhat = float(
            s["r_hat"].max()
        )

        min_ess = float(
            s["ess_bulk"].min()
        )

        pred = score_test(
            test,
            idata,
            station_map
        )

        pred.insert(
            0,
            "case",
            case["case"]
        )

        pred.insert(
            1,
            "response_key",
            case["response_key"]
        )

        prediction_frames.append(pred)

        n_unseen = int(
            (~pred["known_station_in_training"]).sum()
        )

        n_cens_test = int(
            pred["is_censored"].sum()
        )

        mean_lpd = float(
            pred["log_predictive_density"].mean()
        )

        # RMSE / MAE only for detected held-out rows.
        det_pred = pred.loc[
            ~pred["is_censored"]
        ].copy()

        if len(det_pred):

            residual = (
                det_pred["observed_y_std"]
                - det_pred["predicted_latent_mean_std"]
            )

            rmse = float(
                np.sqrt(
                    np.mean(
                        residual**2
                    )
                )
            )

            mae = float(
                np.mean(
                    np.abs(residual)
                )
            )

        else:
            rmse = np.nan
            mae = np.nan

        fold_pass = bool(
            div == 0
            and max_rhat <= 1.05
            and min_ess >= 200
            and np.isfinite(
                pred["log_predictive_density"]
            ).all()
        )

        fold_rows.append({
            "case": case["case"],
            "response_key": case["response_key"],
            "heldout_campaign": heldout,

            "n_train": len(train),
            "n_test": len(test),

            "n_test_detected":
                int((~test["is_censored"]).sum()),

            "n_test_censored":
                n_cens_test,

            "n_train_stations":
                len(station_map),

            "n_unseen_station_rows":
                n_unseen,

            "x_mean_train":
                scaling["x_mean_train"],

            "x_sd_train":
                scaling["x_sd_train"],

            "y_mean_train":
                scaling["y_mean_train"],

            "y_sd_train":
                scaling["y_sd_train"],

            "mean_test_lpd":
                mean_lpd,

            "rmse_detected_std":
                rmse,

            "mae_detected_std":
                mae,

            "divergences":
                div,

            "max_rhat":
                max_rhat,

            "min_ess_bulk":
                min_ess,

            "fold_pass":
                fold_pass,
        })

        print(
            f"{case['case']} | "
            f"holdout={heldout} | "
            f"train={len(train):3d} | "
            f"test={len(test):2d} | "
            f"cens={n_cens_test:2d} | "
            f"unseen={n_unseen:2d} | "
            f"LPD={mean_lpd: .4f} | "
            f"div={div} | "
            f"Rhat={max_rhat:.3f} | "
            f"ESS={min_ess:.0f} | "
            f"PASS={fold_pass}"
        )

# ------------------------------------------------------------------------------------------------
# 9. COMBINE RESULTS
# ------------------------------------------------------------------------------------------------

folds = pd.DataFrame(fold_rows)

predictions = pd.concat(
    prediction_frames,
    ignore_index=True
)

print("\n" + "=" * 120)
print("LOCO FOLD SUMMARY")
print("=" * 120)

print(
    folds[
        [
            "case",
            "heldout_campaign",
            "n_train",
            "n_test",
            "n_test_censored",
            "n_unseen_station_rows",
            "mean_test_lpd",
            "rmse_detected_std",
            "divergences",
            "max_rhat",
            "min_ess_bulk",
            "fold_pass",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------------------------
# 10. CASE-LEVEL SUMMARY
# ------------------------------------------------------------------------------------------------

case_summary = (
    folds.groupby(
        ["case", "response_key"],
        as_index=False
    )
    .agg(
        n_folds=(
            "heldout_campaign",
            "nunique"
        ),
        total_test_rows=(
            "n_test",
            "sum"
        ),
        total_test_censored=(
            "n_test_censored",
            "sum"
        ),
        total_unseen_station_rows=(
            "n_unseen_station_rows",
            "sum"
        ),
        campaign_equal_mean_LPD=(
            "mean_test_lpd",
            "mean"
        ),
        worst_rhat=(
            "max_rhat",
            "max"
        ),
        minimum_ess=(
            "min_ess_bulk",
            "min"
        ),
        total_divergences=(
            "divergences",
            "sum"
        ),
        passed_folds=(
            "fold_pass",
            "sum"
        ),
    )
)

print("\n" + "=" * 120)
print("CASE-LEVEL VALIDATION")
print("=" * 120)

print(
    case_summary.to_string(
        index=False
    )
)

# ------------------------------------------------------------------------------------------------
# 11. STRUCTURAL QA
# ------------------------------------------------------------------------------------------------

expected_test_rows = {
    "DIRECT_NA": 115,
    "CENSORED_AL": 114,
}

qa_rows = []

for case in CASES:

    name = case["case"]

    f = folds.loc[
        folds["case"].eq(name)
    ]

    p = predictions.loc[
        predictions["case"].eq(name)
    ]

    qa_rows.extend([
        {
            "check": f"{name}_10_folds",
            "pass": len(f) == 10
        },
        {
            "check": f"{name}_each_campaign_once",
            "pass": f["heldout_campaign"].nunique() == 10
        },
        {
            "check": f"{name}_prediction_rows_exact",
            "pass": len(p) == expected_test_rows[name]
        },
        {
            "check": f"{name}_finite_LPD",
            "pass": np.isfinite(
                p["log_predictive_density"]
            ).all()
        },
        {
            "check": f"{name}_all_folds_pass",
            "pass": f["fold_pass"].all()
        },
    ])

# Specific censoring validation:
al_pred = predictions.loc[
    predictions["case"].eq("CENSORED_AL")
]

qa_rows.append({
    "check": "AL_censored_rows_scored",
    "pass": int(
        al_pred["is_censored"].sum()
    ) == 13
})

qa_rows.append({
    "check": "AL_censored_LPD_finite",
    "pass": np.isfinite(
        al_pred.loc[
            al_pred["is_censored"],
            "log_predictive_density"
        ]
    ).all()
})

qa = pd.DataFrame(qa_rows)

PASS = bool(
    qa["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.1B QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

# ------------------------------------------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------------------------------------------

OUT = CQDIR / "LOCO_ENGINE_VALIDATION"
OUT.mkdir(
    parents=True,
    exist_ok=True
)

FOLD_FILE = (
    OUT /
    "BLOCK91B_CQ_LOCO_FOLDS_v1.csv"
)

PRED_FILE = (
    OUT /
    "BLOCK91B_CQ_LOCO_PREDICTIONS_v1.csv"
)

CASE_FILE = (
    OUT /
    "BLOCK91B_CQ_LOCO_CASE_SUMMARY_v1.csv"
)

QA_FILE = (
    OUT /
    "BLOCK91B_CQ_LOCO_QA_v1.csv"
)

MANIFEST_FILE = (
    OUT /
    "BLOCK91B_CQ_LOCO_MANIFEST_v1.json"
)

folds.to_csv(
    FOLD_FILE,
    index=False
)

predictions.to_csv(
    PRED_FILE,
    index=False
)

case_summary.to_csv(
    CASE_FILE,
    index=False
)

qa.to_csv(
    QA_FILE,
    index=False
)

manifest = {
    "block": "9.1B",
    "created_at": datetime.now().isoformat(),

    "purpose":
        "campaign-held-out C-Q predictive engine validation",

    "validation_cases": [
        "Na direct",
        "Al censor-aware"
    ],

    "primary_Q":
        "observed only",

    "Q_transformation":
        "log10",

    "fold_unit":
        "campaign",

    "n_campaigns":
        10,

    "train_only_scaling":
        True,

    "train_only_station_encoding":
        True,

    "unseen_station_prediction":
        "random-effect mean zero with sigma_station variance marginalized",

    "detected_scoring":
        "posterior-integrated Normal log predictive density",

    "censored_scoring":
        "posterior-integrated log predictive CDF at sample-specific censor limit",

    "rmse_mae_role":
        "detected held-out observations only",

    "reconstructed_C09_used":
        False,

    "process_classification_performed":
        False,

    "full_CQ_inference_performed":
        False,

    "pass":
        PASS,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

# ------------------------------------------------------------------------------------------------
# 13. FINAL GATE
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.1B FINAL GATE")
print("=" * 120)

print("BLOCK 9.1B PASS                  :", PASS)
print("LOCO FOLDS                       :", len(folds))
print("HELD-OUT PREDICTIONS             :", len(predictions))
print("PRIMARY OBSERVED Q ONLY          :", True)
print("TRAIN-ONLY SCALING               :", True)
print("TRAIN-ONLY STATION ENCODING      :", True)
print("UNSEEN STATIONS MARGINALIZED     :", True)
print("CENSORED TEST ROWS SCORED BY CDF :", True)
print("RECONSTRUCTED C09 USED           :", False)
print("PROCESS CLASSIFICATION PERFORMED :", False)
print("FULL 10-RESPONSE C-Q FIT         :", False)

assert PASS

print(
    "\nNEXT: BLOCK 9.2A — fit the complete primary Bayesian C-Q effect spectrum "
    "for all 10 frozen responses using observed Q only."
)

---
### Imported master block 114

`BLOCK 9.2A — FULL PRIMARY BAYESIAN C–Q EFFECT SPECTRUM`


In [ ]:
# ================================================================================================================
# BLOCK 9.2A — FULL PRIMARY BAYESIAN C–Q EFFECT SPECTRUM
#
# PURPOSE
#   Fit the complete frozen 10-response primary C–Q spectrum.
#
# PRIMARY Q:
#   observed Q only
#
# FROZEN Q:
#   log10(Q)
#
# MODEL:
#   Y*_std = alpha + beta_Q * log10(Q)_std + station random intercept + error
#
# CENSORING:
#   explicit sample-specific left-censor likelihood
#
# IMPORTANT:
#   - All 10 responses fitted.
#   - No reconstructed C09 Q.
#   - No LOD substitution.
#   - No response selection.
#   - No process classification yet.
#   - LOCO comes in 9.2B.
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")

print("=" * 120)
print("BLOCK 9.2A — FULL PRIMARY BAYESIAN C–Q EFFECT SPECTRUM")
print("=" * 120)

# ================================================================================================================
# 1. PATHS
# ================================================================================================================

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"
FREEZE_FILE = CQDIR / "BLOCK90D_CQ_ARCHITECTURE_FREEZE_v1.json"

for p in [CQ_FILE, ARCH_FILE, FREEZE_FILE]:
    assert p.exists(), f"MISSING INPUT: {p}"

cq = pd.read_csv(CQ_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE)

with open(FREEZE_FILE, "r", encoding="utf-8") as f:
    freeze = json.load(f)

assert freeze["Q_transformation"] == "LOG10"
assert freeze["primary_Q_role"] == "observed_only"

print("\nINPUT")
print("-" * 120)
print("C-Q rows :", len(cq))
print("Responses:", cq["response_key"].nunique())
print("PyMC     :", pm.__version__)
print("ArviZ    :", az.__version__)

assert len(cq) == 1300
assert cq["response_key"].nunique() == 10
assert len(arch) == 10

# ================================================================================================================
# 2. HELPERS
# ================================================================================================================

def parse_bool(x):

    if pd.isna(x):
        return False

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    return str(x).strip().lower() in {
        "true", "1", "yes", "y"
    }


def flatten_scalar(idata, var):

    return (
        idata.posterior[var]
        .values
        .reshape(-1)
    )


def prepare_response(response_key):

    a = arch.loc[
        arch["response_key"].eq(response_key)
    ]

    assert len(a) == 1

    a = a.iloc[0]

    transform = str(
        a["primary_transformation"]
    ).upper()

    pathway = str(
        a["statistical_pathway"]
    ).upper()

    g = cq.loc[
        cq["response_key"].eq(response_key)
        &
        cq["Q_primary_CQ"].notna()
    ].copy()

    assert len(g) > 0

    # -------------------------------------------------------------------------
    # PRIMARY observed Q only
    # -------------------------------------------------------------------------

    assert g["cq_q_source"].eq(
        "OBSERVED_PRIMARY"
    ).all()

    q = pd.to_numeric(
        g["Q_primary_CQ"],
        errors="coerce"
    )

    assert q.notna().all()
    assert (q > 0).all()

    g["x_logQ"] = np.log10(q)

    # -------------------------------------------------------------------------
    # Censoring
    # -------------------------------------------------------------------------

    g["is_censored"] = (
        g["response_censored"]
        .map(parse_bool)
    )

    y_raw = pd.to_numeric(
        g["response_value_numeric"],
        errors="coerce"
    )

    L_raw = pd.to_numeric(
        g["response_censor_limit"],
        errors="coerce"
    )

    det = ~g["is_censored"]

    assert y_raw.loc[det].notna().all()

    # -------------------------------------------------------------------------
    # Frozen response transformation
    # -------------------------------------------------------------------------

    g["y_transformed"] = np.nan
    g["limit_transformed"] = np.nan

    if transform == "LOG10":

        assert (
            y_raw.loc[det] > 0
        ).all()

        g.loc[
            det,
            "y_transformed"
        ] = np.log10(
            y_raw.loc[det]
        )

        if g["is_censored"].any():

            assert L_raw.loc[
                g["is_censored"]
            ].notna().all()

            assert (
                L_raw.loc[
                    g["is_censored"]
                ] > 0
            ).all()

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = np.log10(
                L_raw.loc[
                    g["is_censored"]
                ]
            )

    elif transform == "ORIGINAL":

        g.loc[
            det,
            "y_transformed"
        ] = y_raw.loc[det]

        if g["is_censored"].any():

            assert L_raw.loc[
                g["is_censored"]
            ].notna().all()

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = L_raw.loc[
                g["is_censored"]
            ]

    else:

        raise RuntimeError(
            f"Unsupported frozen transform: {transform}"
        )

    # -------------------------------------------------------------------------
    # Full-primary-dataset scaling
    # -------------------------------------------------------------------------

    y_det = g.loc[
        det,
        "y_transformed"
    ].astype(float)

    y_mean = float(
        y_det.mean()
    )

    y_sd = float(
        y_det.std(ddof=1)
    )

    x_mean = float(
        g["x_logQ"].mean()
    )

    x_sd = float(
        g["x_logQ"].std(ddof=1)
    )

    assert np.isfinite(y_sd) and y_sd > 0
    assert np.isfinite(x_sd) and x_sd > 0

    g["x_std"] = (
        g["x_logQ"] - x_mean
    ) / x_sd

    g["y_std"] = (
        g["y_transformed"] - y_mean
    ) / y_sd

    g["limit_std"] = (
        g["limit_transformed"] - y_mean
    ) / y_sd

    # -------------------------------------------------------------------------
    # Station grouping
    # -------------------------------------------------------------------------

    station = (
        g["physical_station_id_v1"]
        .astype("string")
    )

    fallback = (
        g["station_id"]
        .astype("string")
    )

    g["station_group"] = (
        station.fillna(fallback)
    )

    missing = (
        g["station_group"].isna()
    )

    g.loc[
        missing,
        "station_group"
    ] = (
        "UNRESOLVED::"
        +
        g.loc[
            missing
        ].index.astype(str)
    )

    g["station_group"] = (
        g["station_group"]
        .astype(str)
    )

    station_levels = sorted(
        g["station_group"].unique()
    )

    station_map = {
        s: i
        for i, s in enumerate(station_levels)
    }

    g["station_idx"] = (
        g["station_group"]
        .map(station_map)
        .astype(int)
    )

    meta = {
        "response_key":
            response_key,

        "response_label":
            a["response_label"],

        "response_family":
            a["response_family"],

        "primary_transformation":
            transform,

        "statistical_pathway":
            pathway,

        "n_rows":
            len(g),

        "n_detected":
            int(det.sum()),

        "n_censored":
            int(g["is_censored"].sum()),

        "n_stations":
            len(station_levels),

        "n_campaigns":
            int(g["campaign_id"].nunique()),

        "x_mean":
            x_mean,

        "x_sd":
            x_sd,

        "y_mean":
            y_mean,

        "y_sd":
            y_sd,
    }

    return (
        g.reset_index(drop=True),
        meta
    )

# ================================================================================================================
# 3. FIT ONE RESPONSE
# ================================================================================================================

def fit_response(g, meta, seed):

    x = g["x_std"].to_numpy(
        dtype=float
    )

    station_idx = g[
        "station_idx"
    ].to_numpy(
        dtype=int
    )

    cens = g[
        "is_censored"
    ].to_numpy(
        dtype=bool
    )

    det = ~cens

    y_det = g.loc[
        det,
        "y_std"
    ].to_numpy(
        dtype=float
    )

    with pm.Model() as model:

        alpha = pm.Normal(
            "alpha",
            mu=0,
            sigma=1
        )

        beta_Q = pm.Normal(
            "beta_Q",
            mu=0,
            sigma=1
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1
        )

        z_station = pm.Normal(
            "z_station",
            mu=0,
            sigma=1,
            shape=meta["n_stations"]
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1
        )

        # ---------------------------------------------------------------------
        # Detected observations
        # ---------------------------------------------------------------------

        mu_det = (
            alpha
            +
            beta_Q * x[det]
            +
            station_effect[
                station_idx[det]
            ]
        )

        pm.Normal(
            "y_detected",
            mu=mu_det,
            sigma=sigma,
            observed=y_det
        )

        # ---------------------------------------------------------------------
        # Left-censored observations
        # ---------------------------------------------------------------------

        if cens.any():

            limits = g.loc[
                cens,
                "limit_std"
            ].to_numpy(
                dtype=float
            )

            assert np.isfinite(
                limits
            ).all()

            mu_cens = (
                alpha
                +
                beta_Q * x[cens]
                +
                station_effect[
                    station_idx[cens]
                ]
            )

            dist = pm.Normal.dist(
                mu=mu_cens,
                sigma=sigma
            )

            pm.Potential(
                "left_censor_loglik",
                pt.sum(
                    pm.logcdf(
                        dist,
                        pt.as_tensor_variable(
                            limits
                        )
                    )
                )
            )

        idata = pm.sample(
            draws=1500,
            tune=1500,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed,
            progressbar=True,
            return_inferencedata=True
        )

    return model, idata

# ================================================================================================================
# 4. OUTPUT DIRECTORY
# ================================================================================================================

OUT = (
    CQDIR /
    "PRIMARY_EFFECT_SPECTRUM"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

# ================================================================================================================
# 5. FIT ALL 10 FROZEN RESPONSES
# ================================================================================================================

response_order = (
    arch["response_key"]
    .tolist()
)

assert len(response_order) == 10
assert len(set(response_order)) == 10

summary_rows = []
idata_files = {}

for i, response_key in enumerate(
    response_order
):

    print("\n" + "=" * 120)
    print(
        f"PRIMARY C–Q MODEL {i+1}/10"
    )
    print("=" * 120)

    g, meta = prepare_response(
        response_key
    )

    print("Response       :", meta["response_label"])
    print("Key            :", response_key)
    print("Family         :", meta["response_family"])
    print("Transform      :", meta["primary_transformation"])
    print("Pathway        :", meta["statistical_pathway"])
    print("Rows           :", meta["n_rows"])
    print("Detected       :", meta["n_detected"])
    print("Censored       :", meta["n_censored"])
    print("Stations       :", meta["n_stations"])
    print("Campaigns      :", meta["n_campaigns"])

    seed = (
        2026091500 + i
    )

    model, idata = fit_response(
        g,
        meta,
        seed
    )

    # -------------------------------------------------------------------------
    # Posterior diagnostics
    # -------------------------------------------------------------------------

    s = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_Q",
            "sigma",
            "sigma_station",
        ],
        hdi_prob=0.95
    )

    print("\nPOSTERIOR SUMMARY")
    print("-" * 120)
    print(
        s.to_string()
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ].sum().values
    )

    max_rhat = float(
        s["r_hat"].max()
    )

    min_ess_bulk = float(
        s["ess_bulk"].min()
    )

    # -------------------------------------------------------------------------
    # beta_Q posterior
    # -------------------------------------------------------------------------

    beta = flatten_scalar(
        idata,
        "beta_Q"
    )

    beta_mean = float(
        beta.mean()
    )

    beta_median = float(
        np.median(beta)
    )

    beta_sd = float(
        beta.std(ddof=1)
    )

    beta_hdi = az.hdi(
        beta,
        hdi_prob=0.95
    )

    p_positive = float(
        np.mean(beta > 0)
    )

    p_negative = float(
        np.mean(beta < 0)
    )

    p_direction = float(
        max(
            p_positive,
            p_negative
        )
    )

    direction = (
        "POSITIVE"
        if p_positive >= p_negative
        else "NEGATIVE"
    )

    # -------------------------------------------------------------------------
    # Unstandardized slope on transformed-response scale
    #
    # beta_std = Δ(Y / y_sd) per Δ(logQ / x_sd)
    #
    # therefore:
    # beta_unstd = beta_std * y_sd / x_sd
    #
    # This is useful for physical interpretation later.
    # -------------------------------------------------------------------------

    beta_unstd = (
        beta
        *
        meta["y_sd"]
        /
        meta["x_sd"]
    )

    beta_unstd_mean = float(
        beta_unstd.mean()
    )

    beta_unstd_hdi = az.hdi(
        beta_unstd,
        hdi_prob=0.95
    )

    # -------------------------------------------------------------------------
    # For LOG10 concentration responses:
    #
    # log10(C) = a + b log10(Q)
    #
    # b is the classical C–Q exponent.
    #
    # For ORIGINAL responses this is NOT a classical C–Q exponent.
    # -------------------------------------------------------------------------

    if (
        meta["primary_transformation"]
        == "LOG10"
        and
        meta["response_family"]
        not in {
            "MICROBIAL",
            "ACID_BASE",
            "THERMAL"
        }
    ):

        classical_cq_exponent_eligible = True

        cq_exponent_mean = (
            beta_unstd_mean
        )

        cq_exponent_hdi_low = float(
            beta_unstd_hdi[0]
        )

        cq_exponent_hdi_high = float(
            beta_unstd_hdi[1]
        )

    else:

        classical_cq_exponent_eligible = False

        cq_exponent_mean = np.nan
        cq_exponent_hdi_low = np.nan
        cq_exponent_hdi_high = np.nan

    engine_pass = bool(
        divergences == 0
        and
        max_rhat <= 1.05
        and
        min_ess_bulk >= 200
    )

    summary_rows.append({

        "response_key":
            response_key,

        "response_label":
            meta["response_label"],

        "response_family":
            meta["response_family"],

        "primary_transformation":
            meta["primary_transformation"],

        "statistical_pathway":
            meta["statistical_pathway"],

        "n_rows":
            meta["n_rows"],

        "n_detected":
            meta["n_detected"],

        "n_censored":
            meta["n_censored"],

        "n_stations":
            meta["n_stations"],

        "n_campaigns":
            meta["n_campaigns"],

        "Q_source":
            "OBSERVED_ONLY",

        "Q_transformation":
            "LOG10",

        "beta_Q_standardized_mean":
            beta_mean,

        "beta_Q_standardized_median":
            beta_median,

        "beta_Q_standardized_sd":
            beta_sd,

        "beta_Q_standardized_hdi_low":
            float(beta_hdi[0]),

        "beta_Q_standardized_hdi_high":
            float(beta_hdi[1]),

        "P_beta_positive":
            p_positive,

        "P_beta_negative":
            p_negative,

        "posterior_direction":
            direction,

        "posterior_direction_probability":
            p_direction,

        "beta_Q_transformed_scale_mean":
            beta_unstd_mean,

        "beta_Q_transformed_scale_hdi_low":
            float(beta_unstd_hdi[0]),

        "beta_Q_transformed_scale_hdi_high":
            float(beta_unstd_hdi[1]),

        "classical_CQ_exponent_eligible":
            classical_cq_exponent_eligible,

        "CQ_exponent_mean":
            cq_exponent_mean,

        "CQ_exponent_hdi_low":
            cq_exponent_hdi_low,

        "CQ_exponent_hdi_high":
            cq_exponent_hdi_high,

        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess_bulk,

        "engine_pass":
            engine_pass,

        "process_classification":
            "NOT_PERFORMED",

        "LOCO_validated_for_this_response":
            False,
    })

    # -------------------------------------------------------------------------
    # Save posterior
    # -------------------------------------------------------------------------

    safe_name = (
        str(meta["response_label"])
        .replace("/", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

    posterior_file = (
        OUT /
        f"BLOCK92A_{i+1:02d}_{safe_name}_posterior_v1.nc"
    )

    idata.to_netcdf(
        posterior_file
    )

    idata_files[
        response_key
    ] = str(
        posterior_file
    )

# ================================================================================================================
# 6. FULL EFFECT TABLE
# ================================================================================================================

effects = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 120)
print("PRIMARY C–Q EFFECT SPECTRUM")
print("=" * 120)

display_cols = [
    "response_label",
    "response_family",
    "primary_transformation",
    "n_rows",
    "n_censored",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction",
    "posterior_direction_probability",
    "beta_Q_transformed_scale_mean",
    "classical_CQ_exponent_eligible",
    "CQ_exponent_mean",
    "divergences",
    "max_rhat",
    "min_ess_bulk",
    "engine_pass",
]

print(
    effects[
        display_cols
    ].to_string(
        index=False
    )
)

# ================================================================================================================
# 7. QA
# ================================================================================================================

qa = {
    "responses_exactly_10":
        len(effects) == 10,

    "all_frozen_responses_once":
        (
            set(effects["response_key"])
            ==
            set(arch["response_key"])
        ),

    "primary_observed_Q_only":
        effects["Q_source"]
        .eq("OBSERVED_ONLY")
        .all(),

    "Q_transform_log10":
        effects["Q_transformation"]
        .eq("LOG10")
        .all(),

    "all_campaigns_10":
        effects["n_campaigns"]
        .eq(10)
        .all(),

    "all_models_zero_divergences":
        effects["divergences"]
        .eq(0)
        .all(),

    "all_models_rhat_le_1_05":
        effects["max_rhat"]
        .le(1.05)
        .all(),

    "all_models_ess_ge_200":
        effects["min_ess_bulk"]
        .ge(200)
        .all(),

    "all_models_engine_pass":
        effects["engine_pass"]
        .all(),

    "no_process_classification":
        effects["process_classification"]
        .eq("NOT_PERFORMED")
        .all(),

    "LOCO_not_yet_used_for_classification":
        (
            ~effects[
                "LOCO_validated_for_this_response"
            ]
        ).all(),
}

qa_df = pd.DataFrame({
    "check":
        list(qa.keys()),

    "pass":
        list(qa.values())
})

PASS = bool(
    qa_df["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.2A QA")
print("=" * 120)

print(
    qa_df.to_string(
        index=False
    )
)

# ================================================================================================================
# 8. SAVE TABLES
# ================================================================================================================

EFFECT_FILE = (
    OUT /
    "BLOCK92A_PRIMARY_CQ_EFFECT_SPECTRUM_v1.csv"
)

QA_FILE = (
    OUT /
    "BLOCK92A_PRIMARY_CQ_QA_v1.csv"
)

MANIFEST_FILE = (
    OUT /
    "BLOCK92A_PRIMARY_CQ_MANIFEST_v1.json"
)

effects.to_csv(
    EFFECT_FILE,
    index=False
)

qa_df.to_csv(
    QA_FILE,
    index=False
)

manifest = {

    "block":
        "9.2A",

    "created_at":
        datetime.now().isoformat(),

    "purpose":
        "complete primary Bayesian C-Q effect spectrum",

    "n_responses":
        len(effects),

    "primary_Q":
        "observed only",

    "Q_transformation":
        "log10",

    "response_transformations":
        "frozen pre-Q architecture",

    "station_random_intercept":
        True,

    "censoring":
        "explicit sample-specific left-censor likelihood",

    "analytical_subsamples_averaged":
        False,

    "response_selection_using_Q":
        False,

    "reconstructed_C09_used":
        False,

    "process_classification_performed":
        False,

    "LOCO_full_spectrum_performed":
        False,

    "pass":
        PASS,

    "posterior_files":
        idata_files,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ================================================================================================================
# 9. FINAL GATE
# ================================================================================================================

print("\n" + "=" * 120)
print("BLOCK 9.2A FINAL GATE")
print("=" * 120)

print("BLOCK 9.2A PASS                  :", PASS)
print("PRIMARY MODELS FIT               :", len(effects))
print("PRIMARY OBSERVED Q ONLY          :", True)
print("RECONSTRUCTED C09 USED           :", False)
print("Q TRANSFORMATION                 : LOG10 — FROZEN")
print("CENSOR SUBSTITUTION              :", False)
print("ANALYTICAL SUBSAMPLES AVERAGED   :", False)
print("PROCESS CLASSIFICATION PERFORMED :", False)
print("FULL-SPECTRUM LOCO PERFORMED     :", False)

assert PASS

print(
    "\nNEXT: BLOCK 9.2B — full 10-response campaign-held-out C–Q validation, "
    "followed only then by pre-specified process-support classification."
)

---
### Imported master block 115

`BLOCK 9.2B — FULL 10-RESPONSE LOCO C–Q VALIDATION`


In [ ]:
# ================================================================================================================
# BLOCK 9.2B — FULL 10-RESPONSE LOCO C–Q VALIDATION
#
# PURPOSE
#   Test whether Q adds campaign-held-out predictive information for each of the
#   10 frozen responses.
#
# COMPETING MODELS
#
#   M_Q:
#       Y* = alpha + beta_Q*z(log10 Q) + station random intercept + error
#
#   M_0:
#       Y* = alpha + station random intercept + error
#
# PRIMARY DATA
#   observed Q only
#
# VALIDATION
#   leave-one-campaign-out (10 folds)
#
# SCORING
#   detected : posterior-integrated Normal LPD
#   censored : posterior-integrated log predictive CDF
#
# IMPORTANT
#   - train-only response/Q scaling
#   - train-only station encoding
#   - unseen stations marginalized
#   - no reconstructed C09
#   - no LOD substitution
#   - incremental checkpointing
#   - NO final hydrological classification in this block
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from scipy.special import log_ndtr
import json
import warnings
import gc

warnings.filterwarnings("ignore")

print("=" * 120)
print("BLOCK 9.2B — FULL 10-RESPONSE LOCO C–Q VALIDATION")
print("=" * 120)

# ----------------------------------------------------------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"
EFFECT_FILE = CQDIR / "PRIMARY_EFFECT_SPECTRUM/BLOCK92A_PRIMARY_CQ_EFFECT_SPECTRUM_v1.csv"

for p in [CQ_FILE, ARCH_FILE, EFFECT_FILE]:
    assert p.exists(), f"MISSING: {p}"

cq = pd.read_csv(CQ_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE)
effects92a = pd.read_csv(EFFECT_FILE)

OUT = CQDIR / "FULL_LOCO"
OUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = OUT / "BLOCK92B_LOCO_CHECKPOINT_v1.csv"

print("\nINPUT")
print("-" * 120)
print("Rows      :", len(cq))
print("Responses :", cq["response_key"].nunique())
print("Campaigns :", cq["campaign_id"].nunique())

assert len(cq) == 1300
assert cq["response_key"].nunique() == 10
assert len(effects92a) == 10

# ----------------------------------------------------------------------------------------------------------------
# 2. HELPERS
# ----------------------------------------------------------------------------------------------------------------

def parse_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def flatten(idata, var):
    a = idata.posterior[var].values
    if a.ndim == 2:
        return a.reshape(-1)
    if a.ndim == 3:
        return a.reshape(-1, a.shape[-1])
    raise RuntimeError(f"Unexpected posterior shape: {var} {a.shape}")


def logmeanexp(x):
    x = np.asarray(x, dtype=float)
    m = np.max(x)
    return float(m + np.log(np.mean(np.exp(x - m))))


# ----------------------------------------------------------------------------------------------------------------
# 3. RESPONSE PREPARATION
# ----------------------------------------------------------------------------------------------------------------

def prepare_base(response_key):

    a = arch.loc[arch["response_key"].eq(response_key)].iloc[0]

    transform = str(a["primary_transformation"]).upper()

    g = cq.loc[
        cq["response_key"].eq(response_key)
        & cq["Q_primary_CQ"].notna()
    ].copy()

    assert g["cq_q_source"].eq("OBSERVED_PRIMARY").all()

    g["is_censored"] = g["response_censored"].map(parse_bool)

    q = pd.to_numeric(g["Q_primary_CQ"], errors="coerce")
    assert q.notna().all()
    assert (q > 0).all()

    g["x_logQ"] = np.log10(q)

    y = pd.to_numeric(g["response_value_numeric"], errors="coerce")
    L = pd.to_numeric(g["response_censor_limit"], errors="coerce")

    det = ~g["is_censored"]

    g["y_transformed"] = np.nan
    g["limit_transformed"] = np.nan

    if transform == "LOG10":

        assert (y.loc[det] > 0).all()

        g.loc[det, "y_transformed"] = np.log10(y.loc[det])

        if g["is_censored"].any():
            assert L.loc[g["is_censored"]].notna().all()
            assert (L.loc[g["is_censored"]] > 0).all()

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = np.log10(L.loc[g["is_censored"]])

    elif transform == "ORIGINAL":

        g.loc[det, "y_transformed"] = y.loc[det]

        if g["is_censored"].any():
            assert L.loc[g["is_censored"]].notna().all()

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = L.loc[g["is_censored"]]

    else:
        raise RuntimeError(f"Unsupported transform: {transform}")

    station = g["physical_station_id_v1"].astype("string")
    fallback = g["station_id"].astype("string")

    g["station_group"] = station.fillna(fallback)

    missing = g["station_group"].isna()

    g.loc[missing, "station_group"] = (
        "UNRESOLVED::" + g.loc[missing].index.astype(str)
    )

    g["station_group"] = g["station_group"].astype(str)

    return g.reset_index(drop=True), a


# ----------------------------------------------------------------------------------------------------------------
# 4. TRAIN-ONLY SCALING
# ----------------------------------------------------------------------------------------------------------------

def scale_fold(train, test):

    train = train.copy()
    test = test.copy()

    det_train = ~train["is_censored"]

    y_train = train.loc[
        det_train,
        "y_transformed"
    ].astype(float)

    y_mean = float(y_train.mean())
    y_sd = float(y_train.std(ddof=1))

    x_mean = float(train["x_logQ"].mean())
    x_sd = float(train["x_logQ"].std(ddof=1))

    assert np.isfinite(y_sd) and y_sd > 0
    assert np.isfinite(x_sd) and x_sd > 0

    for df in [train, test]:

        df["x_std"] = (df["x_logQ"] - x_mean) / x_sd
        df["y_std"] = (df["y_transformed"] - y_mean) / y_sd
        df["limit_std"] = (df["limit_transformed"] - y_mean) / y_sd

    return train, test


# ----------------------------------------------------------------------------------------------------------------
# 5. FIT MODEL
# ----------------------------------------------------------------------------------------------------------------

def fit_model(train, include_Q, seed):

    train = train.copy()

    stations = sorted(train["station_group"].unique())
    station_map = {s: i for i, s in enumerate(stations)}

    train["station_idx"] = (
        train["station_group"].map(station_map).astype(int)
    )

    x = train["x_std"].to_numpy(float)
    station_idx = train["station_idx"].to_numpy(int)

    cens = train["is_censored"].to_numpy(bool)
    det = ~cens

    y_det = train.loc[det, "y_std"].to_numpy(float)

    with pm.Model() as model:

        alpha = pm.Normal("alpha", 0, 1)

        if include_Q:
            beta_Q = pm.Normal("beta_Q", 0, 1)

        sigma_station = pm.HalfNormal("sigma_station", 1)

        z_station = pm.Normal(
            "z_station",
            0,
            1,
            shape=len(stations)
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal("sigma", 1)

        mu_det = alpha + station_effect[station_idx[det]]

        if include_Q:
            mu_det = mu_det + beta_Q * x[det]

        pm.Normal(
            "y_detected",
            mu=mu_det,
            sigma=sigma,
            observed=y_det
        )

        if cens.any():

            limits = train.loc[cens, "limit_std"].to_numpy(float)
            assert np.isfinite(limits).all()

            mu_cens = alpha + station_effect[station_idx[cens]]

            if include_Q:
                mu_cens = mu_cens + beta_Q * x[cens]

            dist = pm.Normal.dist(
                mu=mu_cens,
                sigma=sigma
            )

            pm.Potential(
                "left_censor_loglik",
                pt.sum(
                    pm.logcdf(
                        dist,
                        pt.as_tensor_variable(limits)
                    )
                )
            )

        idata = pm.sample(
            draws=800,
            tune=800,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed,
            progressbar=False,
            return_inferencedata=True
        )

    return idata, station_map


# ----------------------------------------------------------------------------------------------------------------
# 6. SCORE HELD-OUT DATA
# ----------------------------------------------------------------------------------------------------------------

def score_test(test, idata, station_map, include_Q):

    alpha = flatten(idata, "alpha")
    sigma = flatten(idata, "sigma")
    sigma_station = flatten(idata, "sigma_station")
    station_effect = flatten(idata, "station_effect")

    beta = flatten(idata, "beta_Q") if include_Q else None

    out = []

    for _, r in test.iterrows():

        station = r["station_group"]
        known = station in station_map

        if known:

            b = station_effect[:, station_map[station]]

            mu = alpha + b

            if include_Q:
                mu = mu + beta * float(r["x_std"])

            pred_sd = sigma

        else:

            mu = alpha.copy()

            if include_Q:
                mu = mu + beta * float(r["x_std"])

            pred_sd = np.sqrt(
                sigma**2 + sigma_station**2
            )

        if not bool(r["is_censored"]):

            y = float(r["y_std"])

            logp = (
                -0.5*np.log(2*np.pi)
                - np.log(pred_sd)
                - 0.5*((y-mu)/pred_sd)**2
            )

        else:

            L = float(r["limit_std"])
            assert np.isfinite(L)

            z = (L-mu)/pred_sd
            logp = log_ndtr(z)

        out.append(
            logmeanexp(logp)
        )

    return np.asarray(out, dtype=float)


# ----------------------------------------------------------------------------------------------------------------
# 7. CHECKPOINT RECOVERY
# ----------------------------------------------------------------------------------------------------------------

if CHECKPOINT.exists():

    checkpoint = pd.read_csv(CHECKPOINT)

    completed = set(
        zip(
            checkpoint["response_key"],
            checkpoint["heldout_campaign"]
        )
    )

    rows = checkpoint.to_dict("records")

    print("\nCheckpoint recovered:", len(rows), "completed folds")

else:

    completed = set()
    rows = []


# ----------------------------------------------------------------------------------------------------------------
# 8. FULL 10 × 10 LOCO
# ----------------------------------------------------------------------------------------------------------------

response_order = arch["response_key"].tolist()

for response_i, response_key in enumerate(response_order):

    base, a = prepare_base(response_key)

    campaigns = sorted(
        base["campaign_id"].dropna().unique()
    )

    assert len(campaigns) == 10

    print("\n" + "=" * 120)
    print(
        f"RESPONSE {response_i+1}/10 — {a['response_label']}"
    )
    print("=" * 120)

    for fold_i, heldout in enumerate(campaigns):

        key = (response_key, heldout)

        if key in completed:
            print("SKIP completed:", heldout)
            continue

        train = base.loc[
            ~base["campaign_id"].eq(heldout)
        ].copy()

        test = base.loc[
            base["campaign_id"].eq(heldout)
        ].copy()

        train, test = scale_fold(train, test)

        # Q model
        seed_q = 2026092000 + response_i*100 + fold_i

        idata_q, map_q = fit_model(
            train,
            include_Q=True,
            seed=seed_q
        )

        # Null model
        seed_0 = 2026093000 + response_i*100 + fold_i

        idata_0, map_0 = fit_model(
            train,
            include_Q=False,
            seed=seed_0
        )

        # Both models trained on exactly same stations.
        assert set(map_q) == set(map_0)

        lpd_q = score_test(
            test,
            idata_q,
            map_q,
            include_Q=True
        )

        lpd_0 = score_test(
            test,
            idata_0,
            map_0,
            include_Q=False
        )

        assert len(lpd_q) == len(test)
        assert len(lpd_0) == len(test)
        assert np.isfinite(lpd_q).all()
        assert np.isfinite(lpd_0).all()

        delta = lpd_q - lpd_0

        # ------------------------------------------------------------
        # Diagnostics Q model
        # ------------------------------------------------------------

        sq = az.summary(
            idata_q,
            var_names=[
                "alpha",
                "beta_Q",
                "sigma",
                "sigma_station"
            ]
        )

        div_q = int(
            idata_q.sample_stats["diverging"].sum().values
        )

        rhat_q = float(sq["r_hat"].max())
        ess_q = float(sq["ess_bulk"].min())

        # ------------------------------------------------------------
        # Diagnostics null model
        # ------------------------------------------------------------

        s0 = az.summary(
            idata_0,
            var_names=[
                "alpha",
                "sigma",
                "sigma_station"
            ]
        )

        div_0 = int(
            idata_0.sample_stats["diverging"].sum().values
        )

        rhat_0 = float(s0["r_hat"].max())
        ess_0 = float(s0["ess_bulk"].min())

        unseen = int(
            (~test["station_group"].isin(map_q)).sum()
        )

        fold_pass = bool(
            div_q == 0
            and div_0 == 0
            and rhat_q <= 1.05
            and rhat_0 <= 1.05
            and ess_q >= 200
            and ess_0 >= 200
            and np.isfinite(delta).all()
        )

        row = {

            "response_key":
                response_key,

            "response_label":
                a["response_label"],

            "response_family":
                a["response_family"],

            "heldout_campaign":
                heldout,

            "n_train":
                len(train),

            "n_test":
                len(test),

            "n_test_censored":
                int(test["is_censored"].sum()),

            "n_unseen_station_rows":
                unseen,

            "mean_LPD_Q":
                float(np.mean(lpd_q)),

            "mean_LPD_null":
                float(np.mean(lpd_0)),

            "mean_delta_LPD_Q_minus_null":
                float(np.mean(delta)),

            "sum_delta_LPD_Q_minus_null":
                float(np.sum(delta)),

            "fraction_rows_Q_better":
                float(np.mean(delta > 0)),

            "Q_divergences":
                div_q,

            "Q_max_rhat":
                rhat_q,

            "Q_min_ess":
                ess_q,

            "null_divergences":
                div_0,

            "null_max_rhat":
                rhat_0,

            "null_min_ess":
                ess_0,

            "fold_pass":
                fold_pass,
        }

        rows.append(row)

        # Save after EVERY fold.
        pd.DataFrame(rows).to_csv(
            CHECKPOINT,
            index=False
        )

        print(
            f"{heldout} | "
            f"n={len(test):2d} | "
            f"cens={int(test['is_censored'].sum()):2d} | "
            f"unseen={unseen:2d} | "
            f"ΔLPD={np.mean(delta): .4f} | "
            f"Q[Rhat={rhat_q:.3f},ESS={ess_q:.0f}] | "
            f"N[Rhat={rhat_0:.3f},ESS={ess_0:.0f}] | "
            f"PASS={fold_pass}"
        )

        del idata_q, idata_0
        gc.collect()


# ----------------------------------------------------------------------------------------------------------------
# 9. COMPLETE FOLD TABLE
# ----------------------------------------------------------------------------------------------------------------

folds = pd.DataFrame(rows)

assert len(folds) == 100

assert (
    folds.groupby("response_key")[
        "heldout_campaign"
    ].nunique()
    .eq(10)
    .all()
)

print("\n" + "=" * 120)
print("FULL LOCO COMPLETE")
print("=" * 120)

print("Folds:", len(folds))
print("Passed:", int(folds["fold_pass"].sum()))


# ----------------------------------------------------------------------------------------------------------------
# 10. RESPONSE-LEVEL PREDICTIVE SUMMARY
#
# IMPORTANT:
# campaign_equal_mean_delta_LPD gives each campaign equal weight.
# pooled_sum_delta_LPD is retained only as a secondary diagnostic.
# ----------------------------------------------------------------------------------------------------------------

response_summary = (
    folds
    .groupby(
        [
            "response_key",
            "response_label",
            "response_family"
        ],
        as_index=False
    )
    .agg(

        n_folds=(
            "heldout_campaign",
            "nunique"
        ),

        total_test_rows=(
            "n_test",
            "sum"
        ),

        campaign_equal_mean_delta_LPD=(
            "mean_delta_LPD_Q_minus_null",
            "mean"
        ),

        campaign_median_delta_LPD=(
            "mean_delta_LPD_Q_minus_null",
            "median"
        ),

        campaigns_Q_better=(
            "mean_delta_LPD_Q_minus_null",
            lambda x: int((x > 0).sum())
        ),

        campaigns_Q_worse=(
            "mean_delta_LPD_Q_minus_null",
            lambda x: int((x < 0).sum())
        ),

        pooled_sum_delta_LPD=(
            "sum_delta_LPD_Q_minus_null",
            "sum"
        ),

        total_unseen_station_rows=(
            "n_unseen_station_rows",
            "sum"
        ),

        total_divergences_Q=(
            "Q_divergences",
            "sum"
        ),

        worst_rhat_Q=(
            "Q_max_rhat",
            "max"
        ),

        minimum_ess_Q=(
            "Q_min_ess",
            "min"
        ),

        total_divergences_null=(
            "null_divergences",
            "sum"
        ),

        worst_rhat_null=(
            "null_max_rhat",
            "max"
        ),

        minimum_ess_null=(
            "null_min_ess",
            "min"
        ),

        passed_folds=(
            "fold_pass",
            "sum"
        ),
    )
)


# ----------------------------------------------------------------------------------------------------------------
# 11. MERGE WITH 9.2A POSTERIOR EFFECTS
# ----------------------------------------------------------------------------------------------------------------

combined = effects92a.merge(
    response_summary,
    on=[
        "response_key",
        "response_label",
        "response_family"
    ],
    how="left",
    validate="one_to_one"
)

assert len(combined) == 10

# Descriptive predictive-support fields.
#
# IMPORTANT:
# These are NOT yet hydrological process labels.

combined["posterior_direction_robust_0_95"] = (
    combined[
        "posterior_direction_probability"
    ] >= 0.95
)

combined["LOCO_mean_supports_Q"] = (
    combined[
        "campaign_equal_mean_delta_LPD"
    ] > 0
)

combined["LOCO_majority_campaigns_support_Q"] = (
    combined[
        "campaigns_Q_better"
    ] >= 6
)

combined["joint_effect_predictive_support"] = (
    combined[
        "posterior_direction_robust_0_95"
    ]
    &
    combined[
        "LOCO_mean_supports_Q"
    ]
)

print("\n" + "=" * 120)
print("POSTERIOR + LOCO C–Q SUMMARY")
print("=" * 120)

show = [
    "response_label",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction_probability",
    "campaign_equal_mean_delta_LPD",
    "campaigns_Q_better",
    "campaigns_Q_worse",
    "joint_effect_predictive_support",
]

print(
    combined[show].to_string(
        index=False
    )
)


# ----------------------------------------------------------------------------------------------------------------
# 12. QA
# ----------------------------------------------------------------------------------------------------------------

qa = {

    "exactly_100_folds":
        len(folds) == 100,

    "10_responses":
        folds["response_key"].nunique() == 10,

    "10_campaigns_each_response":
        (
            folds.groupby("response_key")[
                "heldout_campaign"
            ].nunique()
            .eq(10)
            .all()
        ),

    "all_Q_models_zero_divergences":
        folds["Q_divergences"].eq(0).all(),

    "all_null_models_zero_divergences":
        folds["null_divergences"].eq(0).all(),

    "all_Q_rhat_le_1_05":
        folds["Q_max_rhat"].le(1.05).all(),

    "all_null_rhat_le_1_05":
        folds["null_max_rhat"].le(1.05).all(),

    "all_Q_ess_ge_200":
        folds["Q_min_ess"].ge(200).all(),

    "all_null_ess_ge_200":
        folds["null_min_ess"].ge(200).all(),

    "all_folds_pass":
        folds["fold_pass"].all(),

    "combined_responses_10":
        len(combined) == 10,

    "no_process_classification":
        True,

    "reconstructed_C09_not_used":
        True,
}

qa_df = pd.DataFrame({
    "check": list(qa.keys()),
    "pass": list(qa.values())
})

PASS = bool(qa_df["pass"].all())

print("\n" + "=" * 120)
print("BLOCK 9.2B QA")
print("=" * 120)

print(qa_df.to_string(index=False))


# ----------------------------------------------------------------------------------------------------------------
# 13. SAVE
# ----------------------------------------------------------------------------------------------------------------

FOLD_FILE = OUT / "BLOCK92B_FULL_LOCO_FOLDS_v1.csv"
SUMMARY_FILE = OUT / "BLOCK92B_RESPONSE_PREDICTIVE_SUMMARY_v1.csv"
COMBINED_FILE = OUT / "BLOCK92B_POSTERIOR_LOCO_COMBINED_v1.csv"
QA_FILE = OUT / "BLOCK92B_FULL_LOCO_QA_v1.csv"
MANIFEST_FILE = OUT / "BLOCK92B_FULL_LOCO_MANIFEST_v1.json"

folds.to_csv(FOLD_FILE, index=False)
response_summary.to_csv(SUMMARY_FILE, index=False)
combined.to_csv(COMBINED_FILE, index=False)
qa_df.to_csv(QA_FILE, index=False)

manifest = {

    "block": "9.2B",

    "created_at": datetime.now().isoformat(),

    "primary_Q": "observed only",

    "n_responses": 10,

    "n_campaigns": 10,

    "n_Q_models": 100,

    "n_null_models": 100,

    "validation_unit": "campaign",

    "Q_vs_null_comparison": True,

    "campaign_equal_weight_primary": True,

    "train_only_scaling": True,

    "unseen_station_marginalization": True,

    "censor_likelihood": "explicit sample-specific left CDF",

    "reconstructed_C09_used": False,

    "process_classification_performed": False,

    "pass": PASS,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)


# ----------------------------------------------------------------------------------------------------------------
# 14. FINAL GATE
# ----------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.2B FINAL GATE")
print("=" * 120)

print("BLOCK 9.2B PASS                  :", PASS)
print("TOTAL LOCO FOLDS                 :", len(folds))
print("Q MODELS                         :", 100)
print("NULL MODELS                      :", 100)
print("PRIMARY OBSERVED Q ONLY          :", True)
print("TRAIN-ONLY SCALING               :", True)
print("UNSEEN STATIONS MARGINALIZED     :", True)
print("CENSOR VALUES SUBSTITUTED        :", False)
print("RECONSTRUCTED C09 USED           :", False)
print("PROCESS CLASSIFICATION PERFORMED :", False)

assert PASS

print(
    "\nNEXT: BLOCK 9.2C — freeze C–Q support classes using posterior + LOCO evidence; "
    "then BLOCK 9.3 tests reconstructed-C09 sensitivity and creates the manuscript-ready C–Q figure/table."
)

---
### Imported master block 116

`BLOCK 9.2C — FREEZE PRIMARY C–Q SUPPORT CLASSES`


In [ ]:
# ================================================================================================================
# BLOCK 9.2C — FREEZE PRIMARY C–Q SUPPORT CLASSES
#
# PURPOSE
#   Convert the already-frozen posterior + LOCO evidence into conservative,
#   pre-specified C–Q support classes.
#
# NO NEW MODELS ARE FIT.
#
# PRIMARY SUPPORT RULE — already operationalized before seeing 9.2B results:
#
#   posterior directional probability >= 0.95
#   AND
#   campaign-equal mean ΔLPD(Q - null) > 0
#
# IMPORTANT
#   - campaigns_Q_better is descriptive, NOT a replacement gate.
#   - Al/Fe are NOT rescued because a majority of campaigns favor Q.
#   - pH / temperature are response–Q relationships, not dilution/flushing.
#   - EC can support integrated mineralization/dilution behavior, but is not a solute concentration.
#   - "chemostatic" is NOT assigned automatically.
#   - reconstructed C09 is still absent.
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 120)
print("BLOCK 9.2C — PRIMARY C–Q SUPPORT CLASS FREEZE")
print("=" * 120)

# ----------------------------------------------------------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

COMBINED_FILE = (
    CQDIR /
    "FULL_LOCO/BLOCK92B_POSTERIOR_LOCO_COMBINED_v1.csv"
)

FOLD_FILE = (
    CQDIR /
    "FULL_LOCO/BLOCK92B_FULL_LOCO_FOLDS_v1.csv"
)

assert COMBINED_FILE.exists()
assert FOLD_FILE.exists()

x = pd.read_csv(COMBINED_FILE)
folds = pd.read_csv(FOLD_FILE)

assert len(x) == 10
assert len(folds) == 100
assert x["response_key"].nunique() == 10

print("\nINPUT")
print("-" * 120)
print("Responses :", len(x))
print("LOCO folds:", len(folds))

# ----------------------------------------------------------------------------------------------------------------
# 2. IMMUTABLE SUPPORT GATES
# ----------------------------------------------------------------------------------------------------------------

POSTERIOR_DIRECTION_THRESHOLD = 0.95

x["posterior_gate"] = (
    x["posterior_direction_probability"]
    >= POSTERIOR_DIRECTION_THRESHOLD
)

x["predictive_gate"] = (
    x["campaign_equal_mean_delta_LPD"] > 0
)

x["primary_joint_support"] = (
    x["posterior_gate"]
    &
    x["predictive_gate"]
)

# Verify exact agreement with the field generated BEFORE results were inspected.
assert (
    x["primary_joint_support"].astype(bool).to_numpy()
    ==
    x["joint_effect_predictive_support"].astype(bool).to_numpy()
).all()

# Majority of campaigns is DESCRIPTIVE ONLY.
x["campaign_majority_Q_better"] = (
    x["campaigns_Q_better"] >= 6
)

# ----------------------------------------------------------------------------------------------------------------
# 3. RESPONSE INTERPRETATION TYPE
# ----------------------------------------------------------------------------------------------------------------

# These are genuine concentration-like hydrochemical responses for which
# Q direction may be described using dilution-/mobilization-compatible language.
#
# EC is handled separately because it is integrated mineralization, not a literal concentration.
#
# pH, microbial indicators and temperature are also handled separately.

CONCENTRATION_LIKE = {
    "AC::NH4::mg/L",
    "AC::Na::mg/L",
    "AC::SO4_ppm::mg/L",
    "METALS::Aluminio::mg/L",
    "METALS::Hierro::mg/L",
    "METALS::Silicio::mg/L",
}

EC_KEY = "AC::C.E._SOL::dS/m"

ASSOCIATION_ONLY = {
    "AC::pH_SOL::NO_UNIT",
    "COLIFORMS::Total coliforms::NMP/100 mL",
    "FIELD::Water temperature::degC",
}

assert (
    CONCENTRATION_LIKE
    | {EC_KEY}
    | ASSOCIATION_ONLY
) == set(x["response_key"])

# ----------------------------------------------------------------------------------------------------------------
# 4. CONSERVATIVE PRIMARY SUPPORT CLASSIFICATION
# ----------------------------------------------------------------------------------------------------------------

def classify(row):

    key = row["response_key"]

    posterior_ok = bool(row["posterior_gate"])
    predictive_ok = bool(row["predictive_gate"])

    direction = str(row["posterior_direction"]).upper()

    # --------------------------------------------------------------------------------
    # No robust posterior direction
    # --------------------------------------------------------------------------------

    if not posterior_ok:

        return pd.Series({
            "primary_CQ_support_class":
                "NO_ROBUST_Q_SUPPORT",

            "hydrological_process_language":
                "NO_PROCESS_CLASSIFICATION",

            "temporal_generalization":
                "NOT_SUPPORTED",

            "interpretation_strength":
                "NO_ROBUST_SUPPORT",
        })

    # --------------------------------------------------------------------------------
    # Strong posterior effect, but Q fails campaign-equal held-out predictive gate.
    #
    # This is deliberately NOT dilution/mobilization classification.
    # --------------------------------------------------------------------------------

    if posterior_ok and not predictive_ok:

        return pd.Series({
            "primary_CQ_support_class":
                f"{direction}_Q_ASSOCIATION_NOT_TEMPORALLY_GENERALIZABLE",

            "hydrological_process_language":
                "PROCESS_CLASSIFICATION_WITHHELD",

            "temporal_generalization":
                "NOT_SUPPORTED",

            "interpretation_strength":
                "POSTERIOR_ONLY_NOT_PREDICTIVELY_GENERALIZABLE",
        })

    # --------------------------------------------------------------------------------
    # Joint posterior + LOCO support
    # --------------------------------------------------------------------------------

    assert posterior_ok and predictive_ok

    # True concentration-like responses.
    if key in CONCENTRATION_LIKE:

        if direction == "NEGATIVE":

            return pd.Series({
                "primary_CQ_support_class":
                    "ROBUST_NEGATIVE_Q_RESPONSE",

                "hydrological_process_language":
                    "DILUTION_COMPATIBLE",

                "temporal_generalization":
                    "SUPPORTED",

                "interpretation_strength":
                    "JOINT_POSTERIOR_AND_LOCO_SUPPORT",
            })

        elif direction == "POSITIVE":

            return pd.Series({
                "primary_CQ_support_class":
                    "ROBUST_POSITIVE_Q_RESPONSE",

                "hydrological_process_language":
                    "MOBILIZATION_FLUSHING_COMPATIBLE",

                "temporal_generalization":
                    "SUPPORTED",

                "interpretation_strength":
                    "JOINT_POSTERIOR_AND_LOCO_SUPPORT",
            })

        else:
            raise RuntimeError(direction)

    # --------------------------------------------------------------------------------
    # EC — integrated mineralization response
    # --------------------------------------------------------------------------------

    if key == EC_KEY:

        if direction == "NEGATIVE":

            return pd.Series({
                "primary_CQ_support_class":
                    "ROBUST_NEGATIVE_Q_RESPONSE",

                "hydrological_process_language":
                    "INTEGRATED_MINERALIZATION_DILUTION_COMPATIBLE",

                "temporal_generalization":
                    "SUPPORTED",

                "interpretation_strength":
                    "JOINT_POSTERIOR_AND_LOCO_SUPPORT",
            })

        else:

            return pd.Series({
                "primary_CQ_support_class":
                    "ROBUST_POSITIVE_Q_RESPONSE",

                "hydrological_process_language":
                    "INTEGRATED_MINERALIZATION_INCREASE_WITH_Q",

                "temporal_generalization":
                    "SUPPORTED",

                "interpretation_strength":
                    "JOINT_POSTERIOR_AND_LOCO_SUPPORT",
            })

    # --------------------------------------------------------------------------------
    # pH / microbial / thermal:
    # association language only.
    # --------------------------------------------------------------------------------

    if key in ASSOCIATION_ONLY:

        return pd.Series({
            "primary_CQ_support_class":
                f"ROBUST_{direction}_Q_RESPONSE",

            "hydrological_process_language":
                "RESPONSE_DISCHARGE_ASSOCIATION_ONLY",

            "temporal_generalization":
                "SUPPORTED",

            "interpretation_strength":
                "JOINT_POSTERIOR_AND_LOCO_SUPPORT",
        })

    raise RuntimeError(
        f"Unclassified response: {key}"
    )


classification = x.apply(
    classify,
    axis=1
)

x = pd.concat(
    [
        x.reset_index(drop=True),
        classification.reset_index(drop=True)
    ],
    axis=1
)

# ----------------------------------------------------------------------------------------------------------------
# 5. CLASSICAL C–Q EXPONENT LANGUAGE
# ----------------------------------------------------------------------------------------------------------------

# Preserve exactly what 9.2A established.
#
# Na, NH4, Al, Fe:
#   log10(response) ~ log10(Q)
#
# But a mathematically defined exponent does NOT automatically imply
# robust hydrological process support.

x["CQ_exponent_reporting_status"] = np.where(
    x["classical_CQ_exponent_eligible"].astype(bool),
    "REPORTABLE_AS_LOGLOG_EXPONENT",
    "NOT_A_CLASSICAL_LOGLOG_EXPONENT"
)

# Explicitly prohibit chemostatic assignment here.
x["chemostasis_classification"] = "NOT_ASSIGNED"

# ----------------------------------------------------------------------------------------------------------------
# 6. C09 INFLUENCE DIAGNOSTIC
#
# DESCRIPTIVE ONLY.
#
# This does NOT alter the primary campaign-equal rule.
# It simply records whether C09 is influential for interpretation and motivates
# the already-planned reconstructed-C09 sensitivity.
# ----------------------------------------------------------------------------------------------------------------

c09 = (
    folds.loc[
        folds["heldout_campaign"].eq("C09"),
        [
            "response_key",
            "mean_delta_LPD_Q_minus_null"
        ]
    ]
    .rename(
        columns={
            "mean_delta_LPD_Q_minus_null":
                "C09_delta_LPD_Q_minus_null"
        }
    )
)

assert len(c09) == 10

other = (
    folds.loc[
        ~folds["heldout_campaign"].eq("C09")
    ]
    .groupby(
        "response_key",
        as_index=False
    )
    .agg(
        mean_delta_LPD_excluding_C09=(
            "mean_delta_LPD_Q_minus_null",
            "mean"
        ),
        campaigns_Q_better_excluding_C09=(
            "mean_delta_LPD_Q_minus_null",
            lambda z: int((z > 0).sum())
        )
    )
)

x = x.merge(
    c09,
    on="response_key",
    how="left",
    validate="one_to_one"
)

x = x.merge(
    other,
    on="response_key",
    how="left",
    validate="one_to_one"
)

x["C09_influence_flag"] = (
    np.sign(
        x["campaign_equal_mean_delta_LPD"]
    )
    !=
    np.sign(
        x["mean_delta_LPD_excluding_C09"]
    )
)

# IMPORTANT:
# Excluding C09 is NOT a primary analysis.
x["C09_exclusion_used_for_primary_inference"] = False

# ----------------------------------------------------------------------------------------------------------------
# 7. MANUSCRIPT-ORIENTED COMPACT TABLE
#
# This is an intermediate scientific table, NOT yet the final manuscript table,
# because reconstructed-C09 sensitivity has not been completed.
# ----------------------------------------------------------------------------------------------------------------

compact_cols = [
    "response_label",
    "response_family",
    "n_rows",
    "n_censored",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction_probability",
    "CQ_exponent_mean",
    "campaign_equal_mean_delta_LPD",
    "campaigns_Q_better",
    "campaigns_Q_worse",
    "C09_delta_LPD_Q_minus_null",
    "mean_delta_LPD_excluding_C09",
    "C09_influence_flag",
    "primary_CQ_support_class",
    "hydrological_process_language",
    "temporal_generalization",
]

compact = x[compact_cols].copy()

print("\n" + "=" * 120)
print("PRIMARY C–Q SUPPORT SPECTRUM — FROZEN")
print("=" * 120)

print(
    compact.to_string(
        index=False
    )
)

# ----------------------------------------------------------------------------------------------------------------
# 8. EXPECTED STRUCTURAL COUNTS
# ----------------------------------------------------------------------------------------------------------------

support_counts = (
    x["primary_CQ_support_class"]
    .value_counts()
)

print("\n" + "=" * 120)
print("SUPPORT CLASS COUNTS")
print("=" * 120)

print(
    support_counts.to_string()
)

print("\nJOINT POSTERIOR + LOCO SUPPORT")
print("-" * 120)

print(
    x.loc[
        x["primary_joint_support"],
        [
            "response_label",
            "posterior_direction",
            "campaign_equal_mean_delta_LPD",
            "campaigns_Q_better",
            "hydrological_process_language"
        ]
    ].to_string(index=False)
)

print("\nPOSTERIOR EFFECT BUT NO TEMPORAL GENERALIZATION")
print("-" * 120)

print(
    x.loc[
        x["posterior_gate"]
        &
        ~x["predictive_gate"],
        [
            "response_label",
            "posterior_direction",
            "posterior_direction_probability",
            "campaign_equal_mean_delta_LPD",
            "campaigns_Q_better",
            "C09_delta_LPD_Q_minus_null",
            "mean_delta_LPD_excluding_C09"
        ]
    ].to_string(index=False)
)

# ----------------------------------------------------------------------------------------------------------------
# 9. QA
# ----------------------------------------------------------------------------------------------------------------

qa = {

    "responses_10":
        len(x) == 10,

    "support_rule_matches_92B":
        (
            x["primary_joint_support"].astype(bool).to_numpy()
            ==
            x["joint_effect_predictive_support"].astype(bool).to_numpy()
        ).all(),

    "posterior_threshold_0_95":
        POSTERIOR_DIRECTION_THRESHOLD == 0.95,

    "campaign_equal_delta_primary":
        True,

    "campaign_majority_not_primary_gate":
        True,

    "C09_not_excluded":
        (
            ~x[
                "C09_exclusion_used_for_primary_inference"
            ]
        ).all(),

    "no_reconstructed_C09":
        True,

    "no_chemostasis_assigned":
        x[
            "chemostasis_classification"
        ].eq("NOT_ASSIGNED").all(),

    "association_only_responses_not_process_classified":
        x.loc[
            x["response_key"].isin(
                ASSOCIATION_ONLY
            ),
            "hydrological_process_language"
        ].isin(
            [
                "RESPONSE_DISCHARGE_ASSOCIATION_ONLY",
                "NO_PROCESS_CLASSIFICATION",
                "PROCESS_CLASSIFICATION_WITHHELD",
            ]
        ).all(),

    "posterior_only_effects_process_withheld":
        x.loc[
            x["posterior_gate"]
            &
            ~x["predictive_gate"],
            "hydrological_process_language"
        ].eq(
            "PROCESS_CLASSIFICATION_WITHHELD"
        ).all(),

    "all_classes_nonmissing":
        x[
            "primary_CQ_support_class"
        ].notna().all(),
}

qa_df = pd.DataFrame({
    "check": list(qa.keys()),
    "pass": list(qa.values())
})

PASS = bool(
    qa_df["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.2C QA")
print("=" * 120)

print(
    qa_df.to_string(
        index=False
    )
)

# ----------------------------------------------------------------------------------------------------------------
# 10. SAVE
# ----------------------------------------------------------------------------------------------------------------

OUT = CQDIR / "SUPPORT_CLASSIFICATION"
OUT.mkdir(
    parents=True,
    exist_ok=True
)

FULL_FILE = (
    OUT /
    "BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv"
)

COMPACT_FILE = (
    OUT /
    "BLOCK92C_PRIMARY_CQ_COMPACT_TABLE_v1.csv"
)

QA_FILE = (
    OUT /
    "BLOCK92C_PRIMARY_CQ_SUPPORT_QA_v1.csv"
)

MANIFEST_FILE = (
    OUT /
    "BLOCK92C_PRIMARY_CQ_SUPPORT_MANIFEST_v1.json"
)

x.to_csv(
    FULL_FILE,
    index=False
)

compact.to_csv(
    COMPACT_FILE,
    index=False
)

qa_df.to_csv(
    QA_FILE,
    index=False
)

manifest = {

    "block":
        "9.2C",

    "created_at":
        datetime.now().isoformat(),

    "purpose":
        "freeze conservative primary C-Q support classes",

    "posterior_direction_threshold":
        0.95,

    "predictive_gate":
        "campaign_equal_mean_delta_LPD_Q_minus_null > 0",

    "primary_joint_support":
        "posterior direction probability >= 0.95 AND campaign-equal mean delta LPD > 0",

    "campaign_majority_role":
        "descriptive only",

    "C09_exclusion_role":
        "diagnostic only; never primary inference",

    "process_language":
        {
            "negative_concentration":
                "dilution-compatible",

            "positive_concentration":
                "mobilization/flushing-compatible",

            "EC_negative":
                "integrated mineralization/dilution-compatible",

            "pH_microbiology_temperature":
                "response-discharge association only",

            "posterior_effect_without_predictive_gate":
                "process classification withheld"
        },

    "chemostasis_assigned":
        False,

    "reconstructed_C09_used":
        False,

    "new_models_fitted":
        False,

    "pass":
        PASS,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# ----------------------------------------------------------------------------------------------------------------
# 11. HASHES
# ----------------------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


print("\nFROZEN OUTPUTS")
print("-" * 120)

for p in [
    FULL_FILE,
    COMPACT_FILE,
    QA_FILE,
    MANIFEST_FILE
]:

    print(
        p.name,
        "| SHA256:",
        sha256_file(p)
    )

# ----------------------------------------------------------------------------------------------------------------
# 12. FINAL GATE
# ----------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.2C FINAL GATE")
print("=" * 120)

print("BLOCK 9.2C PASS                    :", PASS)
print("NEW MODELS FIT                     :", False)
print("PRIMARY SUPPORT RULE CHANGED       :", False)
print("C09 EXCLUDED FROM PRIMARY          :", False)
print("RECONSTRUCTED C09 USED             :", False)
print("CHEMOSTASIS ASSIGNED               :", False)
print("CONNECTIVITY MECHANISM INFERRED    :", False)

assert PASS

print(
    "\nNEXT: BLOCK 9.3 — observed+reconstructed C09 sensitivity. "
    "Only after that sensitivity is frozen will the manuscript-ready "
    "C-Q figure and table be generated."
)

---
### Imported master block 117

`BLOCK 9.3A — RECONSTRUCTED-C09 SENSITIVITY:`


In [ ]:
# ================================================================================================================
# BLOCK 9.3A — RECONSTRUCTED-C09 SENSITIVITY:
#              FULL BAYESIAN C–Q EFFECT SPECTRUM
#
# PURPOSE
#   Refit the same 10 frozen C–Q models using:
#
#       Q_sensitivity_CQ =
#       observed Q + reconstructed C09 Q
#
#   and compare posterior Q effects against the frozen primary 9.2A spectrum.
#
# IMPORTANT
#   - sensitivity analysis only
#   - primary inference remains BLOCK 9.2C
#   - same response identities
#   - same response transformations
#   - same censor likelihood
#   - same station random-intercept model
#   - same Q transformation: log10(Q)
#   - no LOD substitution
#   - no subsample averaging
#   - no process reclassification in this block
# ================================================================================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import pytensor.tensor as pt
import json
import hashlib
import gc
import warnings

warnings.filterwarnings("ignore")

print("=" * 120)
print("BLOCK 9.3A — RECONSTRUCTED-C09 SENSITIVITY: FULL BAYESIAN C–Q EFFECT SPECTRUM")
print("=" * 120)

# ----------------------------------------------------------------------------------------------------------------
# 1. PATHS
# ----------------------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")
CQDIR = PROJECT / "CQ_ANALYSIS"

CQ_FILE = CQDIR / "CQ_ANALYSIS_DATASET_v1.csv"
ARCH_FILE = CQDIR / "CQ_STATISTICAL_ARCHITECTURE_v1.csv"

PRIMARY_EFFECT_FILE = (
    CQDIR /
    "PRIMARY_EFFECT_SPECTRUM/BLOCK92A_PRIMARY_CQ_EFFECT_SPECTRUM_v1.csv"
)

for p in [
    CQ_FILE,
    ARCH_FILE,
    PRIMARY_EFFECT_FILE
]:
    assert p.exists(), f"MISSING: {p}"

cq = pd.read_csv(CQ_FILE, low_memory=False)
arch = pd.read_csv(ARCH_FILE)
primary = pd.read_csv(PRIMARY_EFFECT_FILE)

OUT = CQDIR / "C09_SENSITIVITY"
OUT.mkdir(parents=True, exist_ok=True)

print("\nINPUT")
print("-" * 120)
print("C-Q rows :", len(cq))
print("Responses:", cq["response_key"].nunique())
print("PyMC     :", pm.__version__)
print("ArviZ    :", az.__version__)

assert len(cq) == 1300
assert cq["response_key"].nunique() == 10
assert len(primary) == 10


# ----------------------------------------------------------------------------------------------------------------
# 2. HELPERS
# ----------------------------------------------------------------------------------------------------------------

def parse_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    return str(x).strip().lower() in {
        "true", "1", "yes", "y"
    }


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------------------------------------------
# 3. PREPARE ONE RESPONSE
# ----------------------------------------------------------------------------------------------------------------

def prepare_response(response_key):

    a = arch.loc[
        arch["response_key"].eq(response_key)
    ].iloc[0]

    transform = str(
        a["primary_transformation"]
    ).upper()

    # ------------------------------------------------------------
    # SENSITIVITY Q ONLY
    # ------------------------------------------------------------

    g = cq.loc[
        cq["response_key"].eq(response_key)
        &
        cq["Q_sensitivity_CQ"].notna()
    ].copy()

    q = pd.to_numeric(
        g["Q_sensitivity_CQ"],
        errors="coerce"
    )

    assert q.notna().all()
    assert (q > 0).all()

    g["x_logQ"] = np.log10(q)

    g["is_censored"] = (
        g["response_censored"]
        .map(parse_bool)
    )

    y = pd.to_numeric(
        g["response_value_numeric"],
        errors="coerce"
    )

    L = pd.to_numeric(
        g["response_censor_limit"],
        errors="coerce"
    )

    det = ~g["is_censored"]

    g["y_transformed"] = np.nan
    g["limit_transformed"] = np.nan

    if transform == "LOG10":

        assert (y.loc[det] > 0).all()

        g.loc[
            det,
            "y_transformed"
        ] = np.log10(
            y.loc[det]
        )

        if g["is_censored"].any():

            assert (
                L.loc[
                    g["is_censored"]
                ].notna().all()
            )

            assert (
                L.loc[
                    g["is_censored"]
                ] > 0
            ).all()

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = np.log10(
                L.loc[
                    g["is_censored"]
                ]
            )

    elif transform == "ORIGINAL":

        g.loc[
            det,
            "y_transformed"
        ] = y.loc[det]

        if g["is_censored"].any():

            assert (
                L.loc[
                    g["is_censored"]
                ].notna().all()
            )

            g.loc[
                g["is_censored"],
                "limit_transformed"
            ] = L.loc[
                g["is_censored"]
            ]

    else:
        raise RuntimeError(
            f"Unsupported transformation: {transform}"
        )

    station = (
        g["physical_station_id_v1"]
        .astype("string")
    )

    fallback = (
        g["station_id"]
        .astype("string")
    )

    g["station_group"] = (
        station.fillna(fallback)
    )

    missing = (
        g["station_group"].isna()
    )

    g.loc[
        missing,
        "station_group"
    ] = (
        "UNRESOLVED::"
        +
        g.loc[
            missing
        ].index.astype(str)
    )

    g["station_group"] = (
        g["station_group"]
        .astype(str)
    )

    return g.reset_index(drop=True), a


# ----------------------------------------------------------------------------------------------------------------
# 4. FULL-DATA STANDARDIZATION
#
# Same strategy as 9.2A:
# standardized effect for cross-response spectrum,
# while transformed-scale beta is recovered afterward.
# ----------------------------------------------------------------------------------------------------------------

def standardize_response(g):

    g = g.copy()

    det = ~g["is_censored"]

    y_det = (
        g.loc[
            det,
            "y_transformed"
        ]
        .astype(float)
    )

    y_mean = float(
        y_det.mean()
    )

    y_sd = float(
        y_det.std(ddof=1)
    )

    x_mean = float(
        g["x_logQ"].mean()
    )

    x_sd = float(
        g["x_logQ"].std(ddof=1)
    )

    assert np.isfinite(y_sd)
    assert y_sd > 0

    assert np.isfinite(x_sd)
    assert x_sd > 0

    g["x_std"] = (
        g["x_logQ"] - x_mean
    ) / x_sd

    g["y_std"] = (
        g["y_transformed"] - y_mean
    ) / y_sd

    g["limit_std"] = (
        g["limit_transformed"] - y_mean
    ) / y_sd

    scaling = {
        "x_mean": x_mean,
        "x_sd": x_sd,
        "y_mean": y_mean,
        "y_sd": y_sd,
    }

    return g, scaling


# ----------------------------------------------------------------------------------------------------------------
# 5. MODEL
# ----------------------------------------------------------------------------------------------------------------

def fit_full_model(g, seed):

    stations = sorted(
        g["station_group"].unique()
    )

    station_map = {
        s: i
        for i, s in enumerate(stations)
    }

    station_idx = (
        g["station_group"]
        .map(station_map)
        .astype(int)
        .to_numpy()
    )

    x = g[
        "x_std"
    ].to_numpy(float)

    cens = g[
        "is_censored"
    ].to_numpy(bool)

    det = ~cens

    y_det = (
        g.loc[
            det,
            "y_std"
        ]
        .to_numpy(float)
    )

    with pm.Model() as model:

        alpha = pm.Normal(
            "alpha",
            0,
            1
        )

        beta_Q = pm.Normal(
            "beta_Q",
            0,
            1
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            1
        )

        z_station = pm.Normal(
            "z_station",
            0,
            1,
            shape=len(stations)
        )

        station_effect = pm.Deterministic(
            "station_effect",
            z_station * sigma_station
        )

        sigma = pm.HalfNormal(
            "sigma",
            1
        )

        mu_det = (
            alpha
            +
            beta_Q * x[det]
            +
            station_effect[
                station_idx[det]
            ]
        )

        pm.Normal(
            "y_detected",
            mu=mu_det,
            sigma=sigma,
            observed=y_det
        )

        if cens.any():

            limits = (
                g.loc[
                    cens,
                    "limit_std"
                ]
                .to_numpy(float)
            )

            assert np.isfinite(
                limits
            ).all()

            mu_cens = (
                alpha
                +
                beta_Q * x[cens]
                +
                station_effect[
                    station_idx[cens]
                ]
            )

            dist = pm.Normal.dist(
                mu=mu_cens,
                sigma=sigma
            )

            pm.Potential(
                "left_censor_loglik",
                pt.sum(
                    pm.logcdf(
                        dist,
                        pt.as_tensor_variable(
                            limits
                        )
                    )
                )
            )

        idata = pm.sample(
            draws=2000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed,
            progressbar=True,
            return_inferencedata=True
        )

    return idata


# ----------------------------------------------------------------------------------------------------------------
# 6. FIT ALL 10 SENSITIVITY MODELS
# ----------------------------------------------------------------------------------------------------------------

results = []

response_order = (
    arch["response_key"]
    .tolist()
)

for i, response_key in enumerate(
    response_order
):

    g, a = prepare_response(
        response_key
    )

    g, scaling = standardize_response(
        g
    )

    print("\n" + "=" * 120)
    print(
        f"SENSITIVITY C–Q MODEL {i+1}/10"
    )
    print("=" * 120)

    print(
        "Response       :",
        a["response_label"]
    )

    print(
        "Key            :",
        response_key
    )

    print(
        "Family         :",
        a["response_family"]
    )

    print(
        "Transform      :",
        a["primary_transformation"]
    )

    print(
        "Pathway        :",
        a["statistical_pathway"]
    )

    print(
        "Rows           :",
        len(g)
    )

    print(
        "Detected       :",
        int(
            (~g["is_censored"]).sum()
        )
    )

    print(
        "Censored       :",
        int(
            g["is_censored"].sum()
        )
    )

    print(
        "Stations       :",
        g["station_group"].nunique()
    )

    print(
        "Campaigns      :",
        g["campaign_id"].nunique()
    )

    n_reconstructed = int(
        (
            g["cq_q_source"]
            ==
            "RECONSTRUCTED_C09_SENSITIVITY"
        ).sum()
    )

    # If the exact string differs in the frozen dataset,
    # derive reconstruction count from primary-vs-sensitivity Q instead.
    if n_reconstructed == 0:

        n_reconstructed = int(
            (
                g["Q_primary_CQ"].isna()
                &
                g["Q_sensitivity_CQ"].notna()
            ).sum()
        )

    print(
        "Reconstructed-Q rows:",
        n_reconstructed
    )

    seed = (
        2026094000 + i
    )

    idata = fit_full_model(
        g,
        seed
    )

    summary = az.summary(
        idata,
        var_names=[
            "alpha",
            "beta_Q",
            "sigma",
            "sigma_station"
        ],
        hdi_prob=0.95
    )

    print("\nPOSTERIOR SUMMARY")
    print("-" * 120)
    print(summary.to_string())

    beta_draws = (
        idata.posterior[
            "beta_Q"
        ]
        .values
        .reshape(-1)
    )

    beta_mean = float(
        beta_draws.mean()
    )

    hdi = az.hdi(
        beta_draws,
        hdi_prob=0.95
    )

    hdi_low = float(
        hdi[0]
    )

    hdi_high = float(
        hdi[1]
    )

    if beta_mean >= 0:

        direction = "POSITIVE"

        direction_prob = float(
            np.mean(
                beta_draws > 0
            )
        )

    else:

        direction = "NEGATIVE"

        direction_prob = float(
            np.mean(
                beta_draws < 0
            )
        )

    # ------------------------------------------------------------
    # Convert standardized slope back to transformed response scale:
    #
    # beta_transformed =
    # beta_std * y_sd / x_sd
    #
    # where x = log10(Q)
    # ------------------------------------------------------------

    beta_transformed_draws = (
        beta_draws
        *
        scaling["y_sd"]
        /
        scaling["x_sd"]
    )

    beta_transformed_mean = float(
        beta_transformed_draws.mean()
    )

    beta_transformed_hdi = az.hdi(
        beta_transformed_draws,
        hdi_prob=0.95
    )

    classical_eligible = bool(
        str(
            a[
                "primary_transformation"
            ]
        ).upper()
        ==
        "LOG10"
        and
        response_key
        in {
            "AC::NH4::mg/L",
            "AC::Na::mg/L",
            "METALS::Aluminio::mg/L",
            "METALS::Hierro::mg/L",
        }
    )

    cq_exponent = (
        beta_transformed_mean
        if classical_eligible
        else np.nan
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ]
        .sum()
        .values
    )

    max_rhat = float(
        summary[
            "r_hat"
        ].max()
    )

    min_ess = float(
        summary[
            "ess_bulk"
        ].min()
    )

    engine_pass = bool(
        divergences == 0
        and max_rhat <= 1.05
        and min_ess >= 200
    )

    results.append({

        "response_key":
            response_key,

        "response_label":
            a["response_label"],

        "response_family":
            a["response_family"],

        "primary_transformation":
            a["primary_transformation"],

        "statistical_pathway":
            a["statistical_pathway"],

        "n_rows":
            len(g),

        "n_censored":
            int(
                g["is_censored"].sum()
            ),

        "n_stations":
            g["station_group"].nunique(),

        "n_campaigns":
            g["campaign_id"].nunique(),

        "n_reconstructed_Q_rows":
            n_reconstructed,

        "beta_Q_standardized_mean":
            beta_mean,

        "beta_Q_standardized_hdi_low":
            hdi_low,

        "beta_Q_standardized_hdi_high":
            hdi_high,

        "posterior_direction":
            direction,

        "posterior_direction_probability":
            direction_prob,

        "beta_Q_transformed_scale_mean":
            beta_transformed_mean,

        "beta_Q_transformed_scale_hdi_low":
            float(
                beta_transformed_hdi[0]
            ),

        "beta_Q_transformed_scale_hdi_high":
            float(
                beta_transformed_hdi[1]
            ),

        "classical_CQ_exponent_eligible":
            classical_eligible,

        "CQ_exponent_mean":
            cq_exponent,

        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess,

        "engine_pass":
            engine_pass,
    })

    del idata
    gc.collect()


# ----------------------------------------------------------------------------------------------------------------
# 7. SENSITIVITY EFFECT SPECTRUM
# ----------------------------------------------------------------------------------------------------------------

sens = pd.DataFrame(
    results
)

assert len(sens) == 10
assert sens["response_key"].nunique() == 10

print("\n" + "=" * 120)
print("SENSITIVITY C–Q EFFECT SPECTRUM")
print("=" * 120)

show = [
    "response_label",
    "n_rows",
    "n_reconstructed_Q_rows",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction_probability",
    "CQ_exponent_mean",
    "divergences",
    "max_rhat",
    "min_ess_bulk",
    "engine_pass",
]

print(
    sens[
        show
    ].to_string(
        index=False
    )
)


# ----------------------------------------------------------------------------------------------------------------
# 8. PRIMARY vs SENSITIVITY COMPARISON
# ----------------------------------------------------------------------------------------------------------------

pcols = [
    "response_key",
    "response_label",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction",
    "posterior_direction_probability",
    "CQ_exponent_mean",
]

p = primary[
    pcols
].copy()

p = p.rename(
    columns={
        c: (
            "primary_" + c
            if c not in {
                "response_key",
                "response_label"
            }
            else c
        )
        for c in p.columns
    }
)

scols = [
    "response_key",
    "response_label",
    "n_rows",
    "n_reconstructed_Q_rows",
    "beta_Q_standardized_mean",
    "beta_Q_standardized_hdi_low",
    "beta_Q_standardized_hdi_high",
    "posterior_direction",
    "posterior_direction_probability",
    "CQ_exponent_mean",
]

s = sens[
    scols
].copy()

s = s.rename(
    columns={
        c: (
            "sensitivity_" + c
            if c not in {
                "response_key",
                "response_label"
            }
            else c
        )
        for c in s.columns
    }
)

comparison = p.merge(
    s,
    on=[
        "response_key",
        "response_label"
    ],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 10

comparison[
    "delta_beta_std_sensitivity_minus_primary"
] = (
    comparison[
        "sensitivity_beta_Q_standardized_mean"
    ]
    -
    comparison[
        "primary_beta_Q_standardized_mean"
    ]
)

comparison[
    "direction_preserved"
] = (
    comparison[
        "primary_posterior_direction"
    ]
    ==
    comparison[
        "sensitivity_posterior_direction"
    ]
)

comparison[
    "primary_posterior_gate"
] = (
    comparison[
        "primary_posterior_direction_probability"
    ] >= 0.95
)

comparison[
    "sensitivity_posterior_gate"
] = (
    comparison[
        "sensitivity_posterior_direction_probability"
    ] >= 0.95
)

comparison[
    "posterior_gate_preserved"
] = (
    comparison[
        "primary_posterior_gate"
    ]
    ==
    comparison[
        "sensitivity_posterior_gate"
    ]
)

print("\n" + "=" * 120)
print("PRIMARY vs RECONSTRUCTED-C09 SENSITIVITY")
print("=" * 120)

compare_show = [
    "response_label",
    "primary_beta_Q_standardized_mean",
    "sensitivity_beta_Q_standardized_mean",
    "delta_beta_std_sensitivity_minus_primary",
    "primary_posterior_direction_probability",
    "sensitivity_posterior_direction_probability",
    "direction_preserved",
    "posterior_gate_preserved",
]

print(
    comparison[
        compare_show
    ].to_string(
        index=False
    )
)


# ----------------------------------------------------------------------------------------------------------------
# 9. QA
# ----------------------------------------------------------------------------------------------------------------

qa = {

    "responses_exactly_10":
        len(sens) == 10,

    "all_frozen_responses_once":
        sens[
            "response_key"
        ].nunique() == 10,

    "sensitivity_Q_used":
        True,

    "reconstructed_Q_rows_present":
        sens[
            "n_reconstructed_Q_rows"
        ].sum() > 0,

    "all_campaigns_10":
        sens[
            "n_campaigns"
        ].eq(10).all(),

    "all_models_zero_divergences":
        sens[
            "divergences"
        ].eq(0).all(),

    "all_models_rhat_le_1_05":
        sens[
            "max_rhat"
        ].le(1.05).all(),

    "all_models_ess_ge_200":
        sens[
            "min_ess_bulk"
        ].ge(200).all(),

    "all_models_engine_pass":
        sens[
            "engine_pass"
        ].all(),

    "comparison_rows_10":
        len(comparison) == 10,

    "primary_results_not_modified":
        True,

    "process_reclassification_not_performed":
        True,
}

qa_df = pd.DataFrame({
    "check":
        list(qa.keys()),

    "pass":
        list(qa.values())
})

PASS = bool(
    qa_df["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.3A QA")
print("=" * 120)

print(
    qa_df.to_string(
        index=False
    )
)


# ----------------------------------------------------------------------------------------------------------------
# 10. SAVE
# ----------------------------------------------------------------------------------------------------------------

SENS_FILE = (
    OUT /
    "BLOCK93A_C09_SENSITIVITY_EFFECT_SPECTRUM_v1.csv"
)

COMPARE_FILE = (
    OUT /
    "BLOCK93A_PRIMARY_VS_C09_SENSITIVITY_v1.csv"
)

QA_FILE = (
    OUT /
    "BLOCK93A_C09_SENSITIVITY_QA_v1.csv"
)

MANIFEST_FILE = (
    OUT /
    "BLOCK93A_C09_SENSITIVITY_MANIFEST_v1.json"
)

sens.to_csv(
    SENS_FILE,
    index=False
)

comparison.to_csv(
    COMPARE_FILE,
    index=False
)

qa_df.to_csv(
    QA_FILE,
    index=False
)

manifest = {

    "block":
        "9.3A",

    "created_at":
        datetime.now().isoformat(),

    "role":
        "sensitivity only",

    "primary_inference_unchanged":
        True,

    "Q_source":
        "observed + reconstructed C09",

    "Q_transformation":
        "log10",

    "response_transformations":
        "frozen from pre-Q architecture",

    "station_random_intercept":
        True,

    "censoring":
        "explicit sample-specific left censor likelihood",

    "reconstructed_C09_allowed":
        True,

    "process_reclassification_performed":
        False,

    "LOCO_performed":
        False,

    "pass":
        PASS,
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ----------------------------------------------------------------------------------------------------------------
# 11. HASHES
# ----------------------------------------------------------------------------------------------------------------

print("\nFROZEN OUTPUTS")
print("-" * 120)

for pth in [
    SENS_FILE,
    COMPARE_FILE,
    QA_FILE,
    MANIFEST_FILE,
]:

    print(
        pth.name,
        "| SHA256:",
        sha256_file(pth)
    )


# ----------------------------------------------------------------------------------------------------------------
# 12. FINAL GATE
# ----------------------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.3A FINAL GATE")
print("=" * 120)

print(
    "BLOCK 9.3A PASS                    :",
    PASS
)

print(
    "SENSITIVITY MODELS FIT             :",
    10
)

print(
    "PRIMARY RESULTS MODIFIED            :",
    False
)

print(
    "OBSERVED Q INCLUDED                 :",
    True
)

print(
    "RECONSTRUCTED C09 INCLUDED          :",
    True
)

print(
    "Q TRANSFORMATION                    :",
    "LOG10 — FROZEN"
)

print(
    "CENSOR SUBSTITUTION                 :",
    False
)

print(
    "ANALYTICAL SUBSAMPLES AVERAGED      :",
    False
)

print(
    "LOCO SENSITIVITY PERFORMED          :",
    False
)

print(
    "PROCESS RECLASSIFICATION PERFORMED  :",
    False
)

assert PASS

print(
    "\nNEXT: BLOCK 9.3B — full campaign-held-out sensitivity using observed + reconstructed C09 Q. "
    "This is the decisive test of whether the Al/Fe temporal-generalization result is sensitive "
    "to the sparse observed-Q coverage of C09."
)

---
### Imported master block 118

`BLOCK 9.3B — FULL LOCO C–Q SENSITIVITY`


In [ ]:
# ==================================================================================================
# BLOCK 9.3B — FULL LOCO C–Q SENSITIVITY
# OBSERVED + RECONSTRUCTED C09 DISCHARGE
# ==================================================================================================
#
# PURPOSE
# -------
# Test whether the primary campaign-held-out C–Q conclusions are sensitive to the sparse observed-Q
# coverage of campaign C09.
#
# PRIMARY 9.2B:
#     Q_primary_CQ = observed discharge only
#
# SENSITIVITY 9.3B:
#     Q_sensitivity_CQ = observed discharge + reconstructed C09 discharge
#
# IMPORTANT
# ---------
# - Does NOT modify primary 9.2B results.
# - Does NOT reclassify hydrological processes.
# - Does NOT substitute censored observations.
# - Keeps distinct analytical subsamples.
# - Uses campaign-held-out LOCO.
# - Scaling is estimated from TRAINING DATA ONLY.
# - Known test stations use their learned station random intercept.
# - Unseen test stations are marginalized over the station random-effect distribution.
# - Censored held-out observations are scored with Normal log-CDF.
# - Detected held-out observations are scored with Normal log-PDF.
# - Q-model is compared with station-only NULL model.
# ==================================================================================================

import os
import json
import math
import hashlib
import warnings

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

from scipy.special import logsumexp
from scipy.stats import norm

warnings.filterwarnings("ignore", category=FutureWarning)

# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = "/content/drive/MyDrive/Huanta_JoH_PROJECT"
CQ_DIR = os.path.join(PROJECT, "CQ_ANALYSIS")
OUT_DIR = os.path.join(CQ_DIR, "C09_SENSITIVITY_LOCO")
os.makedirs(OUT_DIR, exist_ok=True)

CQ_PATH = os.path.join(CQ_DIR, "CQ_ANALYSIS_DATASET_v1.csv")

PRIMARY_EFFECT_PATH = os.path.join(
    CQ_DIR,
    "BLOCK92A_PRIMARY_CQ_EFFECT_SPECTRUM_v1.csv"
)

PRIMARY_SUPPORT_PATH = os.path.join(
    CQ_DIR,
    "BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv"
)

# Search safely for the 9.2B LOCO summary if exact filename differs.
candidate_92b = [
    os.path.join(CQ_DIR, "BLOCK92B_FULL_LOCO_CQ_SUMMARY_v1.csv"),
    os.path.join(CQ_DIR, "BLOCK92B_CQ_LOCO_SUMMARY_v1.csv"),
    os.path.join(CQ_DIR, "BLOCK92B_POSTERIOR_LOCO_CQ_SUMMARY_v1.csv"),
    os.path.join(CQ_DIR, "BLOCK92B_FULL_LOCO_SUMMARY_v1.csv"),
]

PRIMARY_LOCO_SUMMARY_PATH = None

for p in candidate_92b:
    if os.path.exists(p):
        PRIMARY_LOCO_SUMMARY_PATH = p
        break

if PRIMARY_LOCO_SUMMARY_PATH is None:
    # Recursive fallback.
    matches = []
    for root, dirs, files in os.walk(CQ_DIR):
        for f in files:
            fl = f.lower()
            if (
                f.endswith(".csv")
                and "92b" in fl
                and ("summary" in fl or "loco" in fl)
            ):
                matches.append(os.path.join(root, f))

    if matches:
        print("Possible 9.2B files found:")
        for m in matches:
            print("  ", m)

# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def parse_bool_series(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "t"])
    )


def first_existing(columns, candidates, required=True):
    for c in candidates:
        if c in columns:
            return c

    if required:
        raise KeyError(
            f"None of these candidate columns were found:\n{candidates}\n\n"
            f"Available columns:\n{list(columns)}"
        )

    return None


def safe_sd(x):
    x = np.asarray(x, dtype=float)
    s = np.nanstd(x, ddof=1)

    if (not np.isfinite(s)) or s <= 0:
        raise ValueError("Invalid standard deviation during training-only scaling.")

    return float(s)


def posterior_array(idata, var):
    arr = np.asarray(idata.posterior[var])
    return arr.reshape((-1,) + arr.shape[2:])


def diagnostics(idata):
    vars_to_check = [
        v for v in ["alpha", "beta_Q", "sigma", "sigma_station"]
        if v in idata.posterior
    ]

    summ = az.summary(
        idata,
        var_names=vars_to_check,
        round_to=None
    )

    max_rhat = float(summ["r_hat"].max())
    min_ess = float(summ["ess_bulk"].min())

    divergences = int(
        np.asarray(idata.sample_stats["diverging"]).sum()
    )

    return divergences, max_rhat, min_ess


# --------------------------------------------------------------------------------------------------
# 2. LOAD C-Q DATASET
# --------------------------------------------------------------------------------------------------

if not os.path.exists(CQ_PATH):
    raise FileNotFoundError(f"C-Q dataset not found:\n{CQ_PATH}")

cq = pd.read_csv(CQ_PATH, low_memory=False)

print("=" * 120)
print("BLOCK 9.3B — FULL 10-RESPONSE LOCO C–Q SENSITIVITY")
print("OBSERVED + RECONSTRUCTED C09 DISCHARGE")
print("=" * 120)

print("\nINPUT")
print("-" * 120)
print("C-Q rows :", len(cq))
print("Columns  :", cq.shape[1])
print("PyMC     :", pm.__version__)
print("ArviZ    :", az.__version__)


# --------------------------------------------------------------------------------------------------
# 3. SCHEMA RESOLUTION
# --------------------------------------------------------------------------------------------------

RESPONSE_KEY_COL = first_existing(
    cq.columns,
    ["response_key"]
)

RESPONSE_LABEL_COL = first_existing(
    cq.columns,
    ["response_label", "parameter_std", "parameter"]
)

CAMPAIGN_COL = first_existing(
    cq.columns,
    ["campaign_id"]
)

STATION_COL = first_existing(
    cq.columns,
    ["physical_station_id_v1", "station_id"]
)

VALUE_COL = first_existing(
    cq.columns,
    [
        "response_value_numeric",
        "value_numeric",
        "result_numeric",
        "numeric_value"
    ]
)

CENSOR_COL = first_existing(
    cq.columns,
    [
        "response_censored",
        "is_censored",
        "censored"
    ]
)

LIMIT_COL = first_existing(
    cq.columns,
    [
        "response_censor_limit",
        "censor_limit",
        "detection_limit_numeric",
        "reporting_limit_numeric"
    ]
)

TRANSFORM_COL = first_existing(
    cq.columns,
    ["primary_transformation"]
)

PATHWAY_COL = first_existing(
    cq.columns,
    ["statistical_pathway"]
)

Q_SENS_COL = first_existing(
    cq.columns,
    ["Q_sensitivity_CQ"]
)

Q_PRIMARY_COL = first_existing(
    cq.columns,
    ["Q_primary_CQ"]
)

Q_SOURCE_COL = first_existing(
    cq.columns,
    ["cq_q_source", "Q_source", "discharge_source"],
    required=False
)

print("\nRESOLVED SCHEMA")
print("-" * 120)
print("response key   :", RESPONSE_KEY_COL)
print("response label :", RESPONSE_LABEL_COL)
print("campaign       :", CAMPAIGN_COL)
print("station        :", STATION_COL)
print("value          :", VALUE_COL)
print("censor flag    :", CENSOR_COL)
print("censor limit   :", LIMIT_COL)
print("transformation :", TRANSFORM_COL)
print("pathway        :", PATHWAY_COL)
print("primary Q      :", Q_PRIMARY_COL)
print("sensitivity Q  :", Q_SENS_COL)
print("Q source       :", Q_SOURCE_COL)


# --------------------------------------------------------------------------------------------------
# 4. BASIC Q QA
# --------------------------------------------------------------------------------------------------

q_primary = pd.to_numeric(cq[Q_PRIMARY_COL], errors="coerce")
q_sens = pd.to_numeric(cq[Q_SENS_COL], errors="coerce")

assert (q_primary.dropna() > 0).all(), "Primary Q contains zero/negative values."
assert (q_sens.dropna() > 0).all(), "Sensitivity Q contains zero/negative values."

n_primary_rows = int(q_primary.notna().sum())
n_sens_rows = int(q_sens.notna().sum())
n_added_rows = int((q_sens.notna() & q_primary.isna()).sum())

print("\nQ AVAILABILITY")
print("-" * 120)
print("Rows with primary observed Q              :", n_primary_rows)
print("Rows with sensitivity observed+recon Q    :", n_sens_rows)
print("Rows added only by reconstructed C09 Q    :", n_added_rows)

if n_added_rows <= 0:
    raise AssertionError(
        "No reconstructed-Q rows were added in Q_sensitivity_CQ."
    )


# --------------------------------------------------------------------------------------------------
# 5. RESPONSE ARCHITECTURE
# --------------------------------------------------------------------------------------------------

arch_cols = [
    RESPONSE_KEY_COL,
    RESPONSE_LABEL_COL,
    TRANSFORM_COL,
    PATHWAY_COL
]

arch = (
    cq[arch_cols]
    .drop_duplicates(subset=[RESPONSE_KEY_COL])
    .copy()
)

if len(arch) != 10:
    print("\nResponse architecture found:")
    print(arch.to_string(index=False))
    raise AssertionError(
        f"Expected exactly 10 frozen responses, found {len(arch)}."
    )

print("\nRESPONSES")
print("-" * 120)
print(
    arch[
        [
            RESPONSE_LABEL_COL,
            RESPONSE_KEY_COL,
            TRANSFORM_COL,
            PATHWAY_COL
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 6. PREPARE RESPONSE-SPECIFIC DATA
# --------------------------------------------------------------------------------------------------

def prepare_response(response_key):

    g = cq.loc[cq[RESPONSE_KEY_COL] == response_key].copy()

    # Sensitivity Q only.
    g["_Q"] = pd.to_numeric(g[Q_SENS_COL], errors="coerce")

    g = g.loc[
        g["_Q"].notna() &
        (g["_Q"] > 0)
    ].copy()

    if len(g) == 0:
        raise ValueError(f"No sensitivity-Q observations for {response_key}")

    # Response values and censor limits.
    g["_value"] = pd.to_numeric(g[VALUE_COL], errors="coerce")
    g["_limit"] = pd.to_numeric(g[LIMIT_COL], errors="coerce")
    g["_censored"] = parse_bool_series(g[CENSOR_COL])

    # Keep:
    # detected -> numeric observed value required
    # censored -> numeric censor limit required
    valid = (
        (~g["_censored"] & g["_value"].notna()) |
        (g["_censored"] & g["_limit"].notna())
    )

    g = g.loc[valid].copy()

    transform_values = (
        g[TRANSFORM_COL]
        .dropna()
        .astype(str)
        .str.upper()
        .unique()
    )

    if len(transform_values) != 1:
        raise ValueError(
            f"Non-unique transformation for {response_key}: {transform_values}"
        )

    transform = transform_values[0]

    if transform not in ["ORIGINAL", "LOG10"]:
        raise ValueError(
            f"Unsupported transformation {transform} for {response_key}"
        )

    # Transform response.
    if transform == "LOG10":

        bad_detected = (
            (~g["_censored"]) &
            ((g["_value"] <= 0) | g["_value"].isna())
        )

        bad_censored = (
            g["_censored"] &
            ((g["_limit"] <= 0) | g["_limit"].isna())
        )

        if bad_detected.any():
            raise ValueError(
                f"Detected non-positive values under LOG10 for {response_key}"
            )

        if bad_censored.any():
            raise ValueError(
                f"Non-positive censor limits under LOG10 for {response_key}"
            )

        g["_y_trans"] = np.nan
        g.loc[~g["_censored"], "_y_trans"] = np.log10(
            g.loc[~g["_censored"], "_value"]
        )

        g["_limit_trans"] = np.nan
        g.loc[g["_censored"], "_limit_trans"] = np.log10(
            g.loc[g["_censored"], "_limit"]
        )

    else:

        g["_y_trans"] = np.nan
        g.loc[~g["_censored"], "_y_trans"] = (
            g.loc[~g["_censored"], "_value"]
        )

        g["_limit_trans"] = np.nan
        g.loc[g["_censored"], "_limit_trans"] = (
            g.loc[g["_censored"], "_limit"]
        )

    g["_x"] = np.log10(g["_Q"])

    # Explicit station strings.
    g["_station"] = (
        g[STATION_COL]
        .astype("string")
        .fillna("UNRESOLVED_STATION")
        .astype(str)
    )

    g["_campaign"] = g[CAMPAIGN_COL].astype(str)

    return g, transform


# --------------------------------------------------------------------------------------------------
# 7. MODEL FITTER
# --------------------------------------------------------------------------------------------------

def fit_training_model(
    train,
    include_Q=True,
    draws=1000,
    tune=1000,
    chains=4,
    cores=4,
    target_accept=0.95,
    random_seed=20260915
):

    # ----------------------------------------------------------------------------------------------
    # TRAIN-ONLY SCALING
    # ----------------------------------------------------------------------------------------------

    x_train = train["_x"].to_numpy(dtype=float)

    x_mean = float(np.mean(x_train))
    x_sd = safe_sd(x_train)

    detected_train = train.loc[~train["_censored"]].copy()

    if len(detected_train) < 3:
        raise ValueError(
            "Too few detected training observations for response scaling."
        )

    y_detected_train = detected_train["_y_trans"].to_numpy(dtype=float)

    y_mean = float(np.mean(y_detected_train))
    y_sd = safe_sd(y_detected_train)

    x_std = (x_train - x_mean) / x_sd

    # Station coding based ONLY on training stations.
    train_stations = pd.Index(train["_station"].unique())
    station_to_idx = {
        s: i for i, s in enumerate(train_stations)
    }

    station_idx = np.array(
        [station_to_idx[s] for s in train["_station"]],
        dtype=int
    )

    n_station = len(train_stations)

    det_mask = (~train["_censored"]).to_numpy(dtype=bool)
    cens_mask = train["_censored"].to_numpy(dtype=bool)

    y_std_all = np.full(len(train), np.nan, dtype=float)
    limit_std_all = np.full(len(train), np.nan, dtype=float)

    if det_mask.any():
        y_std_all[det_mask] = (
            train.loc[~train["_censored"], "_y_trans"].to_numpy(dtype=float)
            - y_mean
        ) / y_sd

    if cens_mask.any():
        limit_std_all[cens_mask] = (
            train.loc[train["_censored"], "_limit_trans"].to_numpy(dtype=float)
            - y_mean
        ) / y_sd

    coords = {
        "station": np.arange(n_station),
        "obs": np.arange(len(train))
    }

    with pm.Model(coords=coords) as model:

        alpha = pm.Normal(
            "alpha",
            mu=0.0,
            sigma=1.5
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1.0
        )

        z_station = pm.Normal(
            "z_station",
            mu=0.0,
            sigma=1.0,
            dims="station"
        )

        b_station = pm.Deterministic(
            "b_station",
            z_station * sigma_station,
            dims="station"
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1.0
        )

        if include_Q:

            beta_Q = pm.Normal(
                "beta_Q",
                mu=0.0,
                sigma=1.0
            )

            mu = (
                alpha
                + beta_Q * x_std
                + b_station[station_idx]
            )

        else:

            mu = (
                alpha
                + b_station[station_idx]
            )

        # Detected observations.
        if det_mask.any():

            pm.Normal(
                "y_detected",
                mu=mu[det_mask],
                sigma=sigma,
                observed=y_std_all[det_mask]
            )

        # Left-censored observations.
        if cens_mask.any():

            cens_dist = pm.Normal.dist(
                mu=mu[cens_mask],
                sigma=sigma
            )

            logcdf = pm.logcdf(
                cens_dist,
                limit_std_all[cens_mask]
            )

            pm.Potential(
                "left_censored_likelihood",
                logcdf.sum()
            )

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            cores=cores,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=False,
            compute_convergence_checks=True
        )

    scaling = {
        "x_mean": x_mean,
        "x_sd": x_sd,
        "y_mean": y_mean,
        "y_sd": y_sd,
        "train_stations": train_stations.tolist(),
        "station_to_idx": station_to_idx
    }

    return idata, scaling


# --------------------------------------------------------------------------------------------------
# 8. HELD-OUT PREDICTIVE LOG DENSITY
# --------------------------------------------------------------------------------------------------

def heldout_log_predictive_density(
    idata,
    scaling,
    test,
    include_Q=True
):

    # Flatten chains × draws.
    alpha = posterior_array(idata, "alpha").reshape(-1)
    sigma = posterior_array(idata, "sigma").reshape(-1)
    sigma_station = posterior_array(
        idata,
        "sigma_station"
    ).reshape(-1)

    b_station = posterior_array(
        idata,
        "b_station"
    )

    if b_station.ndim != 2:
        b_station = b_station.reshape(
            b_station.shape[0],
            -1
        )

    if include_Q:
        beta_Q = posterior_array(
            idata,
            "beta_Q"
        ).reshape(-1)
    else:
        beta_Q = np.zeros_like(alpha)

    n_draws = len(alpha)

    x_mean = scaling["x_mean"]
    x_sd = scaling["x_sd"]
    y_mean = scaling["y_mean"]
    y_sd = scaling["y_sd"]
    station_to_idx = scaling["station_to_idx"]

    pointwise = []

    for _, row in test.iterrows():

        x_std = (
            float(row["_x"]) - x_mean
        ) / x_sd

        station = row["_station"]

        known_station = station in station_to_idx

        # ------------------------------------------------------------------------------------------
        # KNOWN STATION
        # ------------------------------------------------------------------------------------------

        if known_station:

            j = station_to_idx[station]

            mu_draw = (
                alpha
                + beta_Q * x_std
                + b_station[:, j]
            )

            pred_sd = sigma

        # ------------------------------------------------------------------------------------------
        # UNSEEN STATION
        #
        # Integrate b_new ~ Normal(0, sigma_station)
        #
        # y | hyperparameters ~ Normal(
        #     alpha + beta*x,
        #     sqrt(sigma^2 + sigma_station^2)
        # )
        # ------------------------------------------------------------------------------------------

        else:

            mu_draw = (
                alpha
                + beta_Q * x_std
            )

            pred_sd = np.sqrt(
                sigma**2
                + sigma_station**2
            )

        # ------------------------------------------------------------------------------------------
        # CENSORED TEST OBSERVATION
        # ------------------------------------------------------------------------------------------

        if bool(row["_censored"]):

            limit_std = (
                float(row["_limit_trans"]) - y_mean
            ) / y_sd

            z = (
                limit_std - mu_draw
            ) / pred_sd

            logp_draw = norm.logcdf(z)

        # ------------------------------------------------------------------------------------------
        # DETECTED TEST OBSERVATION
        # ------------------------------------------------------------------------------------------

        else:

            y_std = (
                float(row["_y_trans"]) - y_mean
            ) / y_sd

            logp_draw = norm.logpdf(
                y_std,
                loc=mu_draw,
                scale=pred_sd
            )

        # Posterior predictive density:
        # log(mean(exp(log p(y | theta_s))))
        lpd = (
            logsumexp(logp_draw)
            - np.log(n_draws)
        )

        pointwise.append(
            {
                "campaign_id": row["_campaign"],
                "station": station,
                "station_seen_in_training": known_station,
                "censored": bool(row["_censored"]),
                "lpd": float(lpd)
            }
        )

    return pd.DataFrame(pointwise)


# --------------------------------------------------------------------------------------------------
# 9. FULL 10 × 10 LOCO
# --------------------------------------------------------------------------------------------------

fold_records = []

response_keys = arch[RESPONSE_KEY_COL].tolist()

for response_number, response_key in enumerate(
    response_keys,
    start=1
):

    g, transform = prepare_response(response_key)

    label = str(
        arch.loc[
            arch[RESPONSE_KEY_COL] == response_key,
            RESPONSE_LABEL_COL
        ].iloc[0]
    )

    campaigns = sorted(g["_campaign"].unique())

    if len(campaigns) != 10:
        raise AssertionError(
            f"{label}: expected 10 campaigns, found {len(campaigns)}"
        )

    print("\n" + "=" * 120)
    print(
        f"RESPONSE {response_number}/10 — {label}"
    )
    print("=" * 120)

    for fold_number, heldout_campaign in enumerate(
        campaigns,
        start=1
    ):

        train = g.loc[
            g["_campaign"] != heldout_campaign
        ].copy()

        test = g.loc[
            g["_campaign"] == heldout_campaign
        ].copy()

        if len(test) == 0:
            raise AssertionError(
                f"{label} {heldout_campaign}: empty test fold."
            )

        # ------------------------------------------------------------------------------------------
        # Q MODEL
        # ------------------------------------------------------------------------------------------

        seed_Q = (
            20260915
            + response_number * 1000
            + fold_number * 10
            + 1
        )

        idata_Q, scale_Q = fit_training_model(
            train=train,
            include_Q=True,
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed_Q
        )

        div_Q, rhat_Q, ess_Q = diagnostics(idata_Q)

        pred_Q = heldout_log_predictive_density(
            idata=idata_Q,
            scaling=scale_Q,
            test=test,
            include_Q=True
        )

        # ------------------------------------------------------------------------------------------
        # NULL MODEL
        # ------------------------------------------------------------------------------------------

        seed_N = (
            20260915
            + response_number * 1000
            + fold_number * 10
            + 2
        )

        idata_N, scale_N = fit_training_model(
            train=train,
            include_Q=False,
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            target_accept=0.95,
            random_seed=seed_N
        )

        div_N, rhat_N, ess_N = diagnostics(idata_N)

        pred_N = heldout_log_predictive_density(
            idata=idata_N,
            scaling=scale_N,
            test=test,
            include_Q=False
        )

        if len(pred_Q) != len(pred_N):
            raise AssertionError(
                f"{label} {heldout_campaign}: predictive row mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # IMPORTANT:
        #
        # We use MEAN pointwise LPD per campaign so each campaign has equal weight in the final
        # response-level summary regardless of number of observations.
        # ------------------------------------------------------------------------------------------

        mean_lpd_Q = float(pred_Q["lpd"].mean())
        mean_lpd_N = float(pred_N["lpd"].mean())

        delta_lpd = mean_lpd_Q - mean_lpd_N

        n_test = len(test)
        n_censored_test = int(test["_censored"].sum())

        train_station_set = set(train["_station"])
        unseen_station_mask = ~test["_station"].isin(
            train_station_set
        )

        n_unseen = int(unseen_station_mask.sum())

        n_reconstructed_test = int(
            (
                pd.to_numeric(
                    test[Q_PRIMARY_COL],
                    errors="coerce"
                ).isna()
                &
                pd.to_numeric(
                    test[Q_SENS_COL],
                    errors="coerce"
                ).notna()
            ).sum()
        )

        fold_pass = bool(
            div_Q == 0
            and div_N == 0
            and rhat_Q <= 1.05
            and rhat_N <= 1.05
            and ess_Q >= 200
            and ess_N >= 200
            and np.isfinite(delta_lpd)
        )

        fold_records.append(
            {
                "response_key": response_key,
                "response_label": label,
                "primary_transformation": transform,
                "heldout_campaign": heldout_campaign,
                "n_test": n_test,
                "n_censored_test": n_censored_test,
                "n_unseen_station_rows": n_unseen,
                "n_reconstructed_Q_test_rows": n_reconstructed_test,
                "mean_LPD_Q_model": mean_lpd_Q,
                "mean_LPD_null_model": mean_lpd_N,
                "delta_LPD_Q_minus_null": delta_lpd,
                "Q_model_divergences": div_Q,
                "Q_model_max_rhat": rhat_Q,
                "Q_model_min_ess_bulk": ess_Q,
                "null_model_divergences": div_N,
                "null_model_max_rhat": rhat_N,
                "null_model_min_ess_bulk": ess_N,
                "fold_pass": fold_pass
            }
        )

        print(
            f"{heldout_campaign} | "
            f"n={n_test:2d} | "
            f"cens={n_censored_test:2d} | "
            f"recon={n_reconstructed_test:2d} | "
            f"unseen={n_unseen:2d} | "
            f"ΔLPD={delta_lpd: .4f} | "
            f"Q[Rhat={rhat_Q:.3f},ESS={ess_Q:.0f}] | "
            f"N[Rhat={rhat_N:.3f},ESS={ess_N:.0f}] | "
            f"PASS={fold_pass}"
        )

        # Free large objects before next fit.
        del idata_Q
        del idata_N
        del pred_Q
        del pred_N


# --------------------------------------------------------------------------------------------------
# 10. FOLD TABLE
# --------------------------------------------------------------------------------------------------

folds = pd.DataFrame(fold_records)

print("\n" + "=" * 120)
print("FULL SENSITIVITY LOCO COMPLETE")
print("=" * 120)
print("Folds :", len(folds))
print("Passed:", int(folds["fold_pass"].sum()))


# --------------------------------------------------------------------------------------------------
# 11. RESPONSE-LEVEL SENSITIVITY LOCO SUMMARY
# --------------------------------------------------------------------------------------------------

summary_records = []

for response_key, rg in folds.groupby(
    "response_key",
    sort=False
):

    label = rg["response_label"].iloc[0]

    delta = rg["delta_LPD_Q_minus_null"].to_numpy(
        dtype=float
    )

    c09_row = rg.loc[
        rg["heldout_campaign"] == "C09"
    ]

    if len(c09_row) != 1:
        raise AssertionError(
            f"{label}: expected exactly one C09 fold."
        )

    c09_delta = float(
        c09_row["delta_LPD_Q_minus_null"].iloc[0]
    )

    excluding_c09 = rg.loc[
        rg["heldout_campaign"] != "C09",
        "delta_LPD_Q_minus_null"
    ].to_numpy(dtype=float)

    summary_records.append(
        {
            "response_key": response_key,
            "response_label": label,
            "campaign_equal_mean_delta_LPD_sensitivity": float(
                np.mean(delta)
            ),
            "campaign_equal_median_delta_LPD_sensitivity": float(
                np.median(delta)
            ),
            "campaigns_Q_better_sensitivity": int(
                np.sum(delta > 0)
            ),
            "campaigns_Q_worse_sensitivity": int(
                np.sum(delta < 0)
            ),
            "C09_delta_LPD_Q_minus_null_sensitivity": c09_delta,
            "mean_delta_LPD_excluding_C09_sensitivity": float(
                np.mean(excluding_c09)
            ),
            "total_reconstructed_Q_test_rows": int(
                rg["n_reconstructed_Q_test_rows"].sum()
            ),
            "all_folds_pass": bool(
                rg["fold_pass"].all()
            )
        }
    )

sens_summary = pd.DataFrame(summary_records)


print("\n" + "=" * 120)
print("SENSITIVITY LOCO RESPONSE SUMMARY")
print("=" * 120)

print(
    sens_summary[
        [
            "response_label",
            "campaign_equal_mean_delta_LPD_sensitivity",
            "campaigns_Q_better_sensitivity",
            "campaigns_Q_worse_sensitivity",
            "C09_delta_LPD_Q_minus_null_sensitivity",
            "mean_delta_LPD_excluding_C09_sensitivity",
            "total_reconstructed_Q_test_rows",
            "all_folds_pass"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 12. LOAD PRIMARY 9.2C SUPPORT SPECTRUM
#
# This contains the frozen primary LOCO summaries already used for support classification.
# We compare against it WITHOUT modifying it.
# --------------------------------------------------------------------------------------------------

if not os.path.exists(PRIMARY_SUPPORT_PATH):
    raise FileNotFoundError(
        "Frozen 9.2C primary support spectrum not found:\n"
        f"{PRIMARY_SUPPORT_PATH}"
    )

primary_support = pd.read_csv(
    PRIMARY_SUPPORT_PATH,
    low_memory=False
)

required_primary_cols = [
    "response_label",
    "campaign_equal_mean_delta_LPD",
    "campaigns_Q_better",
    "campaigns_Q_worse",
    "C09_delta_LPD_Q_minus_null",
    "mean_delta_LPD_excluding_C09",
    "primary_CQ_support_class",
    "hydrological_process_language",
    "temporal_generalization"
]

missing_primary_cols = [
    c for c in required_primary_cols
    if c not in primary_support.columns
]

if missing_primary_cols:
    raise KeyError(
        "Missing columns in frozen 9.2C support file:\n"
        f"{missing_primary_cols}\n\n"
        f"Available:\n{primary_support.columns.tolist()}"
    )


# --------------------------------------------------------------------------------------------------
# 13. PRIMARY vs SENSITIVITY LOCO COMPARISON
# --------------------------------------------------------------------------------------------------

comparison = primary_support[
    required_primary_cols
].merge(
    sens_summary,
    on="response_label",
    how="inner",
    validate="one_to_one"
)

if len(comparison) != 10:
    raise AssertionError(
        f"Expected 10 primary-vs-sensitivity comparisons, got {len(comparison)}."
    )

comparison[
    "delta_mean_LPD_sensitivity_minus_primary"
] = (
    comparison[
        "campaign_equal_mean_delta_LPD_sensitivity"
    ]
    -
    comparison[
        "campaign_equal_mean_delta_LPD"
    ]
)

comparison[
    "primary_mean_delta_positive"
] = (
    comparison[
        "campaign_equal_mean_delta_LPD"
    ] > 0
)

comparison[
    "sensitivity_mean_delta_positive"
] = (
    comparison[
        "campaign_equal_mean_delta_LPD_sensitivity"
    ] > 0
)

comparison[
    "predictive_sign_preserved"
] = (
    comparison[
        "primary_mean_delta_positive"
    ]
    ==
    comparison[
        "sensitivity_mean_delta_positive"
    ]
)

comparison[
    "C09_predictive_sign_primary"
] = np.sign(
    comparison[
        "C09_delta_LPD_Q_minus_null"
    ]
)

comparison[
    "C09_predictive_sign_sensitivity"
] = np.sign(
    comparison[
        "C09_delta_LPD_Q_minus_null_sensitivity"
    ]
)

comparison[
    "C09_predictive_sign_changed"
] = (
    comparison[
        "C09_predictive_sign_primary"
    ]
    !=
    comparison[
        "C09_predictive_sign_sensitivity"
    ]
)


print("\n" + "=" * 120)
print("PRIMARY 9.2B/9.2C vs RECONSTRUCTED-C09 SENSITIVITY 9.3B")
print("=" * 120)

display_cols = [
    "response_label",
    "campaign_equal_mean_delta_LPD",
    "campaign_equal_mean_delta_LPD_sensitivity",
    "delta_mean_LPD_sensitivity_minus_primary",
    "campaigns_Q_better",
    "campaigns_Q_better_sensitivity",
    "C09_delta_LPD_Q_minus_null",
    "C09_delta_LPD_Q_minus_null_sensitivity",
    "mean_delta_LPD_excluding_C09",
    "mean_delta_LPD_excluding_C09_sensitivity",
    "predictive_sign_preserved",
    "C09_predictive_sign_changed",
    "primary_CQ_support_class"
]

print(
    comparison[display_cols].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 14. SPECIAL DIAGNOSTIC: Al + Fe
#
# These are the two responses for which 9.2C found:
# posterior positive association BUT no temporal generalization.
#
# IMPORTANT:
# We are NOT changing their classification here.
# We are only testing whether reconstructed C09 materially changes the predictive evidence.
# --------------------------------------------------------------------------------------------------

metal_diag = comparison.loc[
    comparison["response_label"].isin(
        ["Aluminio", "Hierro"]
    ),
    display_cols
].copy()

print("\n" + "=" * 120)
print("DECISIVE Al / Fe C09 SENSITIVITY DIAGNOSTIC")
print("=" * 120)

print(
    metal_diag.to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 15. QA
# --------------------------------------------------------------------------------------------------

qa = pd.DataFrame(
    [
        {
            "check": "exactly_100_folds",
            "pass": len(folds) == 100
        },
        {
            "check": "10_responses",
            "pass": folds["response_key"].nunique() == 10
        },
        {
            "check": "10_campaigns_each_response",
            "pass": bool(
                (
                    folds.groupby("response_key")[
                        "heldout_campaign"
                    ].nunique() == 10
                ).all()
            )
        },
        {
            "check": "all_Q_models_zero_divergences",
            "pass": bool(
                (
                    folds["Q_model_divergences"] == 0
                ).all()
            )
        },
        {
            "check": "all_null_models_zero_divergences",
            "pass": bool(
                (
                    folds["null_model_divergences"] == 0
                ).all()
            )
        },
        {
            "check": "all_Q_rhat_le_1_05",
            "pass": bool(
                (
                    folds["Q_model_max_rhat"] <= 1.05
                ).all()
            )
        },
        {
            "check": "all_null_rhat_le_1_05",
            "pass": bool(
                (
                    folds["null_model_max_rhat"] <= 1.05
                ).all()
            )
        },
        {
            "check": "all_Q_ess_ge_200",
            "pass": bool(
                (
                    folds["Q_model_min_ess_bulk"] >= 200
                ).all()
            )
        },
        {
            "check": "all_null_ess_ge_200",
            "pass": bool(
                (
                    folds["null_model_min_ess_bulk"] >= 200
                ).all()
            )
        },
        {
            "check": "all_folds_pass",
            "pass": bool(
                folds["fold_pass"].all()
            )
        },
        {
            "check": "reconstructed_Q_used",
            "pass": bool(
                folds[
                    "n_reconstructed_Q_test_rows"
                ].sum() > 0
            )
        },
        {
            "check": "C09_contains_reconstructed_Q",
            "pass": bool(
                folds.loc[
                    folds["heldout_campaign"] == "C09",
                    "n_reconstructed_Q_test_rows"
                ].sum() > 0
            )
        },
        {
            "check": "comparison_responses_10",
            "pass": len(comparison) == 10
        },
        {
            "check": "primary_support_file_not_modified",
            "pass": True
        },
        {
            "check": "no_process_reclassification",
            "pass": True
        }
    ]
)

print("\n" + "=" * 120)
print("BLOCK 9.3B QA")
print("=" * 120)
print(qa.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 16. SAVE OUTPUTS
# --------------------------------------------------------------------------------------------------

FOLD_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_C09_SENSITIVITY_LOCO_FOLDS_v1.csv"
)

SUMMARY_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_C09_SENSITIVITY_LOCO_SUMMARY_v1.csv"
)

COMPARISON_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_PRIMARY_VS_C09_LOCO_COMPARISON_v1.csv"
)

METAL_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_AL_FE_C09_DIAGNOSTIC_v1.csv"
)

QA_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_C09_SENSITIVITY_LOCO_QA_v1.csv"
)

MANIFEST_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_C09_SENSITIVITY_LOCO_MANIFEST_v1.json"
)

folds.to_csv(
    FOLD_OUT,
    index=False
)

sens_summary.to_csv(
    SUMMARY_OUT,
    index=False
)

comparison.to_csv(
    COMPARISON_OUT,
    index=False
)

metal_diag.to_csv(
    METAL_OUT,
    index=False
)

qa.to_csv(
    QA_OUT,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL GATE
# --------------------------------------------------------------------------------------------------

block_pass = bool(qa["pass"].all())

manifest = {
    "block": "9.3B",
    "status": "PASS" if block_pass else "FAIL",
    "purpose": (
        "Campaign-held-out C-Q sensitivity using observed discharge plus "
        "reconstructed C09 discharge."
    ),
    "primary_results_modified": False,
    "process_reclassification_performed": False,
    "Q_transformation": "LOG10",
    "Q_source": "OBSERVED_PLUS_RECONSTRUCTED_C09",
    "validation": "LOCO_CAMPAIGN",
    "train_only_scaling": True,
    "unseen_station_handling": (
        "MARGINALIZED_OVER_STATION_RANDOM_EFFECT"
    ),
    "censor_handling": (
        "EXPLICIT_SAMPLE_SPECIFIC_LEFT_CENSOR_LIKELIHOOD"
    ),
    "n_responses": int(
        folds["response_key"].nunique()
    ),
    "n_folds": int(len(folds)),
    "n_reconstructed_test_rows": int(
        folds["n_reconstructed_Q_test_rows"].sum()
    ),
    "outputs": {}
}

for p in [
    FOLD_OUT,
    SUMMARY_OUT,
    COMPARISON_OUT,
    METAL_OUT,
    QA_OUT
]:
    manifest["outputs"][os.path.basename(p)] = {
        "sha256": sha256_file(p),
        "bytes": os.path.getsize(p)
    }

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

manifest_sha = sha256_file(
    MANIFEST_OUT
)

print("\nFROZEN OUTPUTS")
print("-" * 120)

for p in [
    FOLD_OUT,
    SUMMARY_OUT,
    COMPARISON_OUT,
    METAL_OUT,
    QA_OUT
]:
    print(
        os.path.basename(p),
        "| SHA256:",
        sha256_file(p)
    )

print(
    os.path.basename(MANIFEST_OUT),
    "| SHA256:",
    manifest_sha
)


print("\n" + "=" * 120)
print("BLOCK 9.3B FINAL GATE")
print("=" * 120)

print(
    "BLOCK 9.3B PASS                    :",
    block_pass
)

print(
    "TOTAL LOCO FOLDS                   :",
    len(folds)
)

print(
    "Q MODELS                           :",
    len(folds)
)

print(
    "NULL MODELS                        :",
    len(folds)
)

print(
    "OBSERVED Q INCLUDED                :",
    True
)

print(
    "RECONSTRUCTED C09 INCLUDED         :",
    bool(
        folds[
            "n_reconstructed_Q_test_rows"
        ].sum() > 0
    )
)

print(
    "TRAIN-ONLY SCALING                 :",
    True
)

print(
    "UNSEEN STATIONS MARGINALIZED       :",
    True
)

print(
    "CENSOR VALUES SUBSTITUTED          :",
    False
)

print(
    "PRIMARY RESULTS MODIFIED           :",
    False
)

print(
    "PROCESS RECLASSIFICATION PERFORMED :",
    False
)

if not block_pass:
    print(
        "\nSTOP: one or more QA gates failed. "
        "Do NOT proceed to manuscript figure/table."
    )
else:
    print(
        "\nNEXT: review the 9.3B sensitivity results. "
        "Only after review will the final C-Q sensitivity interpretation "
        "be frozen and the manuscript-ready figure/table generated."
    )

---
### Imported master block 119

`BLOCK 9.3B-R — SURGICAL REPAIR + FINAL FREEZE`


In [ ]:
# ==============================================================================================
# BLOCK 9.3B-R — SURGICAL REPAIR + FINAL FREEZE
#
# Repairs ONLY:
#   Fe / C08 sensitivity LOCO fold
#
# Then:
#   - recalculates sensitivity summary
#   - finds frozen BLOCK 9.2C automatically
#   - compares primary vs sensitivity
#   - runs final QA
#   - saves frozen 9.3B products
#
# DO NOT rerun the 100-fold block.
# ==============================================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

print("=" * 120)
print("BLOCK 9.3B-R — SURGICAL Fe-C08 REPAIR + FINAL FREEZE")
print("=" * 120)

# ----------------------------------------------------------------------------------------------
# 0. REQUIRE OBJECTS FROM THE PREVIOUS 9.3B RUN
# ----------------------------------------------------------------------------------------------

required_objects = [
    "cq",
    "arch",
    "folds",
    "prepare_response",
    "fit_training_model",
    "heldout_log_predictive_density",
    "diagnostics"
]

missing = [
    x for x in required_objects
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Required 9.3B objects are no longer in RAM:\n"
        + "\n".join(missing)
        + "\n\nDo NOT Run All. Tell ChatGPT which objects are missing."
    )

print("\nRecovered objects:")
print("folds:", folds.shape)
print("responses:", folds["response_key"].nunique())
print("campaigns:", folds["heldout_campaign"].nunique())

assert len(folds) == 100
assert folds["response_key"].nunique() == 10
assert folds["heldout_campaign"].nunique() == 10


# ----------------------------------------------------------------------------------------------
# 1. IDENTIFY THE SINGLE FAILED FOLD
# ----------------------------------------------------------------------------------------------

failed = folds.loc[~folds["fold_pass"]].copy()

print("\nFAILED FOLDS BEFORE REPAIR")
print("-" * 120)
print(failed.to_string(index=False))

assert len(failed) == 1, (
    f"Expected exactly one failed fold, found {len(failed)}."
)

failed_row = failed.iloc[0]

assert failed_row["response_label"] == "Hierro"
assert failed_row["heldout_campaign"] == "C08"

FE_KEY = failed_row["response_key"]

print("\nConfirmed surgical target:")
print("Response :", failed_row["response_label"])
print("Key      :", FE_KEY)
print("Campaign :", failed_row["heldout_campaign"])


# ----------------------------------------------------------------------------------------------
# 2. REBUILD EXACT Fe DATASET
# ----------------------------------------------------------------------------------------------

g_fe, transform_fe = prepare_response(FE_KEY)

train = g_fe.loc[
    g_fe["_campaign"] != "C08"
].copy()

test = g_fe.loc[
    g_fe["_campaign"] == "C08"
].copy()

print("\nFe-C08 fold")
print("-" * 120)
print("Train rows :", len(train))
print("Test rows  :", len(test))
print("Censored test :", int(test["_censored"].sum()))
print(
    "Reconstructed test Q :",
    int(
        (
            pd.to_numeric(test[Q_PRIMARY_COL], errors="coerce").isna()
            &
            pd.to_numeric(test[Q_SENS_COL], errors="coerce").notna()
        ).sum()
    )
)

assert len(test) == 11
assert int(test["_censored"].sum()) == 2


# ----------------------------------------------------------------------------------------------
# 3. REFIT ONLY Fe-C08 Q MODEL
#
# target_accept raised from 0.95 -> 0.99
# More draws/tuning for clean convergence.
# ----------------------------------------------------------------------------------------------

print("\nRefitting Fe-C08 Q model only...")

idata_Q_fix, scale_Q_fix = fit_training_model(
    train=train,
    include_Q=True,
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.99,
    random_seed=202693808
)

div_Q, rhat_Q, ess_Q = diagnostics(idata_Q_fix)

pred_Q_fix = heldout_log_predictive_density(
    idata=idata_Q_fix,
    scaling=scale_Q_fix,
    test=test,
    include_Q=True
)

print(
    f"Q model repaired | divergences={div_Q} | "
    f"Rhat={rhat_Q:.3f} | ESS={ess_Q:.0f}"
)


# ----------------------------------------------------------------------------------------------
# 4. REFIT Fe-C08 NULL MODEL TOO
#
# This keeps the Q-null comparison generated under the same repair run.
# ----------------------------------------------------------------------------------------------

print("\nRefitting Fe-C08 NULL model...")

idata_N_fix, scale_N_fix = fit_training_model(
    train=train,
    include_Q=False,
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.99,
    random_seed=202693809
)

div_N, rhat_N, ess_N = diagnostics(idata_N_fix)

pred_N_fix = heldout_log_predictive_density(
    idata=idata_N_fix,
    scaling=scale_N_fix,
    test=test,
    include_Q=False
)

print(
    f"NULL model repaired | divergences={div_N} | "
    f"Rhat={rhat_N:.3f} | ESS={ess_N:.0f}"
)


# ----------------------------------------------------------------------------------------------
# 5. RECOMPUTE Fe-C08 ΔLPD
# ----------------------------------------------------------------------------------------------

mean_lpd_Q = float(pred_Q_fix["lpd"].mean())
mean_lpd_N = float(pred_N_fix["lpd"].mean())
delta_lpd = mean_lpd_Q - mean_lpd_N

train_station_set = set(train["_station"])
n_unseen = int(
    (~test["_station"].isin(train_station_set)).sum()
)

n_reconstructed_test = int(
    (
        pd.to_numeric(test[Q_PRIMARY_COL], errors="coerce").isna()
        &
        pd.to_numeric(test[Q_SENS_COL], errors="coerce").notna()
    ).sum()
)

fold_pass_fix = bool(
    div_Q == 0
    and div_N == 0
    and rhat_Q <= 1.05
    and rhat_N <= 1.05
    and ess_Q >= 200
    and ess_N >= 200
    and np.isfinite(delta_lpd)
)

print("\nREPAIRED Fe-C08")
print("-" * 120)
print("Mean LPD Q    :", mean_lpd_Q)
print("Mean LPD null :", mean_lpd_N)
print("ΔLPD          :", delta_lpd)
print("PASS          :", fold_pass_fix)

if not fold_pass_fix:
    raise RuntimeError(
        "Fe-C08 still failed convergence after target_accept=0.99. "
        "STOP without freezing 9.3B."
    )


# ----------------------------------------------------------------------------------------------
# 6. REPLACE ONLY THE FAILED ROW
# ----------------------------------------------------------------------------------------------

mask_fix = (
    folds["response_key"].eq(FE_KEY)
    &
    folds["heldout_campaign"].eq("C08")
)

assert mask_fix.sum() == 1

folds_fixed = folds.copy()

replacement = {
    "response_key": FE_KEY,
    "response_label": "Hierro",
    "primary_transformation": transform_fe,
    "heldout_campaign": "C08",
    "n_test": len(test),
    "n_censored_test": int(test["_censored"].sum()),
    "n_unseen_station_rows": n_unseen,
    "n_reconstructed_Q_test_rows": n_reconstructed_test,
    "mean_LPD_Q_model": mean_lpd_Q,
    "mean_LPD_null_model": mean_lpd_N,
    "delta_LPD_Q_minus_null": delta_lpd,
    "Q_model_divergences": div_Q,
    "Q_model_max_rhat": rhat_Q,
    "Q_model_min_ess_bulk": ess_Q,
    "null_model_divergences": div_N,
    "null_model_max_rhat": rhat_N,
    "null_model_min_ess_bulk": ess_N,
    "fold_pass": fold_pass_fix
}

for col, value in replacement.items():
    folds_fixed.loc[mask_fix, col] = value

assert len(folds_fixed) == 100
assert folds_fixed["fold_pass"].all()

print("\nFold replacement complete.")
print("Passed folds:", int(folds_fixed["fold_pass"].sum()), "/ 100")


# ----------------------------------------------------------------------------------------------
# 7. RECOMPUTE RESPONSE-LEVEL SENSITIVITY SUMMARY
# ----------------------------------------------------------------------------------------------

summary_records = []

for response_key, rg in folds_fixed.groupby(
    "response_key",
    sort=False
):
    label = rg["response_label"].iloc[0]

    delta = rg[
        "delta_LPD_Q_minus_null"
    ].to_numpy(dtype=float)

    c09 = rg.loc[
        rg["heldout_campaign"].eq("C09")
    ]

    assert len(c09) == 1

    c09_delta = float(
        c09["delta_LPD_Q_minus_null"].iloc[0]
    )

    excluding_c09 = rg.loc[
        ~rg["heldout_campaign"].eq("C09"),
        "delta_LPD_Q_minus_null"
    ].to_numpy(dtype=float)

    summary_records.append({
        "response_key": response_key,
        "response_label": label,
        "campaign_equal_mean_delta_LPD_sensitivity":
            float(np.mean(delta)),
        "campaign_equal_median_delta_LPD_sensitivity":
            float(np.median(delta)),
        "campaigns_Q_better_sensitivity":
            int(np.sum(delta > 0)),
        "campaigns_Q_worse_sensitivity":
            int(np.sum(delta < 0)),
        "C09_delta_LPD_Q_minus_null_sensitivity":
            c09_delta,
        "mean_delta_LPD_excluding_C09_sensitivity":
            float(np.mean(excluding_c09)),
        "total_reconstructed_Q_test_rows":
            int(rg["n_reconstructed_Q_test_rows"].sum()),
        "all_folds_pass":
            bool(rg["fold_pass"].all())
    })

sens_summary_fixed = pd.DataFrame(summary_records)

assert len(sens_summary_fixed) == 10

print("\n" + "=" * 120)
print("FINAL REPAIRED SENSITIVITY LOCO SUMMARY")
print("=" * 120)

print(
    sens_summary_fixed.to_string(index=False)
)


# ----------------------------------------------------------------------------------------------
# 8. FIND THE ACTUAL FROZEN 9.2C FILE
# ----------------------------------------------------------------------------------------------

PROJECT = "/content/drive/MyDrive/Huanta_JoH_PROJECT"
CQ_DIR = os.path.join(PROJECT, "CQ_ANALYSIS")

matches_92c = []

for root, dirs, files in os.walk(CQ_DIR):
    for filename in files:
        if filename == "BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv":
            matches_92c.append(
                os.path.join(root, filename)
            )

print("\n9.2C FILE SEARCH")
print("-" * 120)

for m in matches_92c:
    print(m)

if len(matches_92c) == 0:
    raise FileNotFoundError(
        "Could not find BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv "
        "anywhere under CQ_ANALYSIS."
    )

if len(matches_92c) > 1:
    print(
        "\nWARNING: multiple exact 9.2C files found. "
        "No file will be silently selected."
    )
    raise RuntimeError(
        "Multiple exact frozen 9.2C support files found."
    )

PRIMARY_SUPPORT_PATH = matches_92c[0]

print("\nUsing frozen primary support:")
print(PRIMARY_SUPPORT_PATH)

primary_support = pd.read_csv(
    PRIMARY_SUPPORT_PATH,
    low_memory=False
)

print("Rows:", len(primary_support))
print("Columns:", primary_support.shape[1])

assert len(primary_support) == 10


# ----------------------------------------------------------------------------------------------
# 9. RESOLVE PRIMARY COLUMN NAMES ROBUSTLY
# ----------------------------------------------------------------------------------------------

def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            "Could not resolve any of:\n"
            + "\n".join(candidates)
            + "\n\nAvailable columns:\n"
            + "\n".join(df.columns)
        )
    return None


P_LABEL = find_col(
    primary_support,
    ["response_label"]
)

P_KEY = find_col(
    primary_support,
    ["response_key"]
)

P_MEAN = find_col(
    primary_support,
    [
        "campaign_equal_mean_delta_LPD",
        "mean_delta_LPD_Q_minus_null"
    ]
)

P_BETTER = find_col(
    primary_support,
    ["campaigns_Q_better"]
)

P_WORSE = find_col(
    primary_support,
    ["campaigns_Q_worse"]
)

P_C09 = find_col(
    primary_support,
    [
        "C09_delta_LPD_Q_minus_null",
        "C09_delta_LPD"
    ]
)

P_EXCL_C09 = find_col(
    primary_support,
    [
        "mean_delta_LPD_excluding_C09",
        "mean_delta_LPD_excluding_C09_diagnostic"
    ],
    required=False
)

P_CLASS = find_col(
    primary_support,
    ["primary_CQ_support_class"]
)

P_PROCESS = find_col(
    primary_support,
    ["hydrological_process_language"]
)

P_GENERAL = find_col(
    primary_support,
    [
        "temporal_generalization",
        "temporal_generalization_status"
    ]
)


# ----------------------------------------------------------------------------------------------
# 10. BUILD PRIMARY TABLE
# ----------------------------------------------------------------------------------------------

primary_compare = pd.DataFrame({
    "response_key": primary_support[P_KEY],
    "response_label": primary_support[P_LABEL],
    "primary_campaign_equal_mean_delta_LPD":
        pd.to_numeric(primary_support[P_MEAN], errors="coerce"),
    "primary_campaigns_Q_better":
        pd.to_numeric(primary_support[P_BETTER], errors="coerce"),
    "primary_campaigns_Q_worse":
        pd.to_numeric(primary_support[P_WORSE], errors="coerce"),
    "primary_C09_delta_LPD":
        pd.to_numeric(primary_support[P_C09], errors="coerce"),
    "primary_CQ_support_class":
        primary_support[P_CLASS],
    "primary_hydrological_process_language":
        primary_support[P_PROCESS],
    "primary_temporal_generalization":
        primary_support[P_GENERAL]
})

if P_EXCL_C09 is not None:
    primary_compare[
        "primary_mean_delta_LPD_excluding_C09"
    ] = pd.to_numeric(
        primary_support[P_EXCL_C09],
        errors="coerce"
    )
else:
    primary_compare[
        "primary_mean_delta_LPD_excluding_C09"
    ] = np.nan


# ----------------------------------------------------------------------------------------------
# 11. PRIMARY vs SENSITIVITY COMPARISON
# ----------------------------------------------------------------------------------------------

comparison = primary_compare.merge(
    sens_summary_fixed,
    on=["response_key", "response_label"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 10

comparison[
    "delta_mean_LPD_sensitivity_minus_primary"
] = (
    comparison[
        "campaign_equal_mean_delta_LPD_sensitivity"
    ]
    -
    comparison[
        "primary_campaign_equal_mean_delta_LPD"
    ]
)

comparison[
    "primary_predictive_gate"
] = (
    comparison[
        "primary_campaign_equal_mean_delta_LPD"
    ] > 0
)

comparison[
    "sensitivity_predictive_gate"
] = (
    comparison[
        "campaign_equal_mean_delta_LPD_sensitivity"
    ] > 0
)

comparison[
    "predictive_gate_preserved"
] = (
    comparison["primary_predictive_gate"]
    ==
    comparison["sensitivity_predictive_gate"]
)

comparison[
    "C09_delta_improvement"
] = (
    comparison[
        "C09_delta_LPD_Q_minus_null_sensitivity"
    ]
    -
    comparison[
        "primary_C09_delta_LPD"
    ]
)


# ----------------------------------------------------------------------------------------------
# 12. JOIN 9.3A POSTERIOR GATE
# ----------------------------------------------------------------------------------------------

SENS93A = os.path.join(
    CQ_DIR,
    "C09_SENSITIVITY",
    "BLOCK93A_C09_SENSITIVITY_EFFECT_SPECTRUM_v1.csv"
)

assert os.path.exists(SENS93A), SENS93A

effect93a = pd.read_csv(SENS93A)

effect93a = effect93a[
    [
        "response_key",
        "posterior_direction_probability",
        "beta_Q_standardized_mean",
        "beta_Q_standardized_hdi_low",
        "beta_Q_standardized_hdi_high",
        "CQ_exponent_mean"
    ]
].copy()

effect93a[
    "sensitivity_posterior_gate"
] = (
    effect93a[
        "posterior_direction_probability"
    ] >= 0.95
)

comparison = comparison.merge(
    effect93a,
    on="response_key",
    how="left",
    validate="one_to_one"
)

comparison[
    "sensitivity_joint_support"
] = (
    comparison[
        "sensitivity_posterior_gate"
    ]
    &
    comparison[
        "sensitivity_predictive_gate"
    ]
)


# ----------------------------------------------------------------------------------------------
# 13. PRINT DECISIVE RESULTS
# ----------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("PRIMARY vs C09 SENSITIVITY — FINAL COMPARISON")
print("=" * 120)

cols_show = [
    "response_label",
    "primary_campaign_equal_mean_delta_LPD",
    "campaign_equal_mean_delta_LPD_sensitivity",
    "primary_campaigns_Q_better",
    "campaigns_Q_better_sensitivity",
    "primary_C09_delta_LPD",
    "C09_delta_LPD_Q_minus_null_sensitivity",
    "C09_delta_improvement",
    "posterior_direction_probability",
    "primary_predictive_gate",
    "sensitivity_predictive_gate",
    "sensitivity_joint_support",
    "primary_CQ_support_class"
]

print(
    comparison[cols_show].to_string(index=False)
)


# ----------------------------------------------------------------------------------------------
# 14. Al / Fe DECISIVE DIAGNOSTIC
# ----------------------------------------------------------------------------------------------

metal_diag = comparison.loc[
    comparison["response_label"].isin(
        ["Aluminio", "Hierro"]
    )
].copy()

print("\n" + "=" * 120)
print("Al / Fe — FINAL C09 COVERAGE SENSITIVITY")
print("=" * 120)

print(
    metal_diag[cols_show].to_string(index=False)
)


# ----------------------------------------------------------------------------------------------
# 15. FINAL QA
# ----------------------------------------------------------------------------------------------

qa_rows = [
    ("exactly_100_folds", len(folds_fixed) == 100),
    ("all_100_folds_pass", folds_fixed["fold_pass"].all()),
    (
        "all_Q_zero_divergences",
        folds_fixed["Q_model_divergences"].eq(0).all()
    ),
    (
        "all_null_zero_divergences",
        folds_fixed["null_model_divergences"].eq(0).all()
    ),
    (
        "all_Q_rhat_le_1_05",
        folds_fixed["Q_model_max_rhat"].le(1.05).all()
    ),
    (
        "all_null_rhat_le_1_05",
        folds_fixed["null_model_max_rhat"].le(1.05).all()
    ),
    (
        "all_Q_ess_ge_200",
        folds_fixed["Q_model_min_ess_bulk"].ge(200).all()
    ),
    (
        "all_null_ess_ge_200",
        folds_fixed["null_model_min_ess_bulk"].ge(200).all()
    ),
    (
        "responses_exactly_10",
        comparison["response_key"].nunique() == 10
    ),
    (
        "reconstructed_Q_used",
        folds_fixed["n_reconstructed_Q_test_rows"].sum() == 94
    ),
    (
        "reconstructed_Q_only_in_C09_test",
        folds_fixed.loc[
            folds_fixed["n_reconstructed_Q_test_rows"] > 0,
            "heldout_campaign"
        ].eq("C09").all()
    ),
    ("frozen_primary_92C_loaded", len(primary_support) == 10),
    ("primary_results_not_modified", True),
    ("primary_support_rule_not_modified", True),
    ("process_reclassification_not_performed", True),
    ("chemostasis_not_assigned", True),
    ("connectivity_not_inferred", True)
]

qa_final = pd.DataFrame(
    qa_rows,
    columns=["check", "pass"]
)

PASS = bool(
    qa_final["pass"].all()
)

print("\n" + "=" * 120)
print("BLOCK 9.3B-R FINAL QA")
print("=" * 120)

print(
    qa_final.to_string(index=False)
)


# ----------------------------------------------------------------------------------------------
# 16. SAVE FINAL FROZEN PRODUCTS
# ----------------------------------------------------------------------------------------------

OUT_DIR = os.path.join(
    CQ_DIR,
    "C09_SENSITIVITY_LOCO"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

FOLD_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_C09_SENSITIVITY_LOCO_FOLDS_v1.csv"
)

SUMMARY_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_C09_SENSITIVITY_LOCO_SUMMARY_v1.csv"
)

COMPARE_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_PRIMARY_VS_C09_SENSITIVITY_v1.csv"
)

METAL_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_AL_FE_C09_DIAGNOSTIC_v1.csv"
)

QA_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_FINAL_QA_v1.csv"
)

MANIFEST_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93B_R_FINAL_MANIFEST_v1.json"
)

folds_fixed.to_csv(
    FOLD_OUT,
    index=False
)

sens_summary_fixed.to_csv(
    SUMMARY_OUT,
    index=False
)

comparison.to_csv(
    COMPARE_OUT,
    index=False
)

metal_diag.to_csv(
    METAL_OUT,
    index=False
)

qa_final.to_csv(
    QA_OUT,
    index=False
)


# ----------------------------------------------------------------------------------------------
# 17. HASH HELPER
# ----------------------------------------------------------------------------------------------

def sha256_file_local(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


manifest = {
    "block": "9.3B-R",
    "status": "PASS" if PASS else "FAIL",
    "scientific_role": "C09 reconstructed-discharge sensitivity only",
    "primary_inference": "BLOCK 9.2C remains authoritative",
    "repair": {
        "response": "Hierro",
        "campaign": "C08",
        "reason": "one divergence in initial 9.3B Q model",
        "target_accept": 0.99,
        "draws": 2000,
        "tune": 2000,
        "Q_and_null_refit": True
    },
    "Q_source": "observed + reconstructed C09",
    "Q_transformation": "log10",
    "validation": "leave-one-campaign-out",
    "campaign_equal_weighting": True,
    "train_only_scaling": True,
    "unseen_station_random_effect_marginalized": True,
    "sample_specific_left_censor_likelihood": True,
    "n_responses": 10,
    "n_folds": 100,
    "n_reconstructed_response_rows": int(
        folds_fixed["n_reconstructed_Q_test_rows"].sum()
    ),
    "primary_results_modified": False,
    "process_reclassification_performed": False,
    "pass": PASS
}

with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nFROZEN OUTPUTS")
print("-" * 120)

for p in [
    FOLD_OUT,
    SUMMARY_OUT,
    COMPARE_OUT,
    METAL_OUT,
    QA_OUT,
    MANIFEST_OUT
]:
    print(
        os.path.basename(p),
        "| SHA256:",
        sha256_file_local(p)
    )


# ----------------------------------------------------------------------------------------------
# 18. FINAL GATE
# ----------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 9.3B-R FINAL GATE")
print("=" * 120)

print("BLOCK 9.3B-R PASS                  :", PASS)
print("TOTAL FOLDS                        :", len(folds_fixed))
print("FAILED FOLDS AFTER REPAIR          :", int((~folds_fixed["fold_pass"]).sum()))
print("RECONSTRUCTED RESPONSE ROWS         :", int(folds_fixed["n_reconstructed_Q_test_rows"].sum()))
print("PRIMARY 9.2C MODIFIED               :", False)
print("PROCESS RECLASSIFICATION            :", False)
print("CHEMOSTASIS ASSIGNED                :", False)
print("CONNECTIVITY INFERRED               :", False)

assert PASS, "9.3B-R did not pass all QA gates."

print(
    "\nNEXT: BLOCK 9.3C — freeze sensitivity interpretation and generate "
    "the manuscript-ready C-Q figure, compact main-text table, and supplementary LOCO table."
)

---
### Imported master block 120

`BLOCK 9.3C — FINAL C–Q EDITORIAL FREEZE`


In [ ]:
# ==================================================================================================
# BLOCK 9.3C — FINAL C–Q EDITORIAL FREEZE
# Manuscript-ready figure + main table + supplementary LOCO tables
# ==================================================================================================
#
# NO NEW MODELS ARE FIT HERE.
#
# INPUTS (all previously frozen):
#   - 9.2A primary posterior C-Q spectrum
#   - 9.2C primary support classification
#   - 9.3A reconstructed-C09 posterior sensitivity
#   - 9.3B-R reconstructed-C09 LOCO sensitivity
#   - 9.2B primary LOCO folds
#
# OUTPUTS:
#   1. Main manuscript C-Q figure (PDF + 600 dpi PNG)
#   2. Figure-data CSV
#   3. Main manuscript C-Q compact table (CSV + XLSX)
#   4. Supplementary response-level table
#   5. Supplementary fold-level LOCO table
#   6. QA + manifest
#
# IMPORTANT:
#   - Primary inference remains 9.2C.
#   - C09 reconstruction remains sensitivity analysis.
#   - No process reclassification.
#   - No chemostasis assignment.
#   - No connectivity inference.
# ==================================================================================================

import os
import json
import hashlib
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = "/content/drive/MyDrive/Huanta_JoH_PROJECT"
CQ_DIR = os.path.join(PROJECT, "CQ_ANALYSIS")

OUT_DIR = os.path.join(
    CQ_DIR,
    "EDITORIAL_FREEZE"
)

os.makedirs(OUT_DIR, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 1. FILE FINDER
# --------------------------------------------------------------------------------------------------

def find_exact(root, filename):
    matches = []

    for r, d, files in os.walk(root):
        if filename in files:
            matches.append(
                os.path.join(r, filename)
            )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nCould not find:\n{filename}\nunder:\n{root}"
        )

    if len(matches) > 1:
        print("\nMultiple matches found for:")
        print(filename)

        for m in matches:
            print("  ", m)

        raise RuntimeError(
            "Multiple exact matches found. "
            "No file selected silently."
        )

    return matches[0]


def find_one_by_tokens(root, required_tokens):
    matches = []

    tokens = [
        x.lower()
        for x in required_tokens
    ]

    for r, d, files in os.walk(root):
        for f in files:
            fl = f.lower()

            if all(t in fl for t in tokens):
                matches.append(
                    os.path.join(r, f)
                )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No file found containing tokens: {required_tokens}"
        )

    if len(matches) > 1:
        print(
            "\nCandidate files for tokens",
            required_tokens
        )

        for m in matches:
            print("  ", m)

        raise RuntimeError(
            "Multiple candidate files found. "
            "No file selected silently."
        )

    return matches[0]


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# --------------------------------------------------------------------------------------------------
# 2. LOCATE FROZEN INPUTS
# --------------------------------------------------------------------------------------------------

PRIMARY_SUPPORT_PATH = find_exact(
    CQ_DIR,
    "BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv"
)

SENS_EFFECT_PATH = find_exact(
    CQ_DIR,
    "BLOCK93A_C09_SENSITIVITY_EFFECT_SPECTRUM_v1.csv"
)

SENS_COMPARE_PATH = find_exact(
    CQ_DIR,
    "BLOCK93B_R_PRIMARY_VS_C09_SENSITIVITY_v1.csv"
)

SENS_FOLDS_PATH = find_exact(
    CQ_DIR,
    "BLOCK93B_R_C09_SENSITIVITY_LOCO_FOLDS_v1.csv"
)

PRIMARY_FOLDS_PATH = find_exact(
    CQ_DIR,
    "BLOCK92B_FULL_LOCO_FOLDS_v1.csv"
)

# 9.2A exact filename may vary slightly.
try:
    PRIMARY_EFFECT_PATH = find_exact(
        CQ_DIR,
        "BLOCK92A_PRIMARY_CQ_EFFECT_SPECTRUM_v1.csv"
    )

except FileNotFoundError:
    PRIMARY_EFFECT_PATH = find_one_by_tokens(
        CQ_DIR,
        ["92a", "effect", "spectrum"]
    )


print("=" * 120)
print("BLOCK 9.3C — FINAL C–Q EDITORIAL FREEZE")
print("=" * 120)

print("\nFROZEN INPUTS")
print("-" * 120)

for p in [
    PRIMARY_EFFECT_PATH,
    PRIMARY_SUPPORT_PATH,
    SENS_EFFECT_PATH,
    SENS_COMPARE_PATH,
    PRIMARY_FOLDS_PATH,
    SENS_FOLDS_PATH
]:
    print(p)


# --------------------------------------------------------------------------------------------------
# 3. LOAD INPUTS
# --------------------------------------------------------------------------------------------------

primary_effect = pd.read_csv(
    PRIMARY_EFFECT_PATH,
    low_memory=False
)

primary_support = pd.read_csv(
    PRIMARY_SUPPORT_PATH,
    low_memory=False
)

sens_effect = pd.read_csv(
    SENS_EFFECT_PATH,
    low_memory=False
)

sens_compare = pd.read_csv(
    SENS_COMPARE_PATH,
    low_memory=False
)

primary_folds = pd.read_csv(
    PRIMARY_FOLDS_PATH,
    low_memory=False
)

sens_folds = pd.read_csv(
    SENS_FOLDS_PATH,
    low_memory=False
)


print("\nINPUT SHAPES")
print("-" * 120)
print("Primary effect :", primary_effect.shape)
print("Primary support:", primary_support.shape)
print("Sensitivity eff:", sens_effect.shape)
print("Sensitivity cmp:", sens_compare.shape)
print("Primary folds  :", primary_folds.shape)
print("Sensitivity fld:", sens_folds.shape)


# --------------------------------------------------------------------------------------------------
# 4. BASIC INPUT QA
# --------------------------------------------------------------------------------------------------

assert len(primary_effect) == 10
assert len(primary_support) == 10
assert len(sens_effect) == 10
assert len(sens_compare) == 10
assert len(primary_folds) == 100
assert len(sens_folds) == 100

assert primary_effect["response_key"].nunique() == 10
assert primary_support["response_key"].nunique() == 10
assert sens_effect["response_key"].nunique() == 10
assert sens_compare["response_key"].nunique() == 10


# --------------------------------------------------------------------------------------------------
# 5. COLUMN RESOLVER
# --------------------------------------------------------------------------------------------------

def col(df, candidates, required=True):

    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise KeyError(
            "\nCould not resolve any of:\n"
            + "\n".join(candidates)
            + "\n\nAvailable columns:\n"
            + "\n".join(df.columns)
        )

    return None


# Primary effect columns
PE_KEY = col(
    primary_effect,
    ["response_key"]
)

PE_LABEL = col(
    primary_effect,
    ["response_label"]
)

PE_N = col(
    primary_effect,
    ["n_rows", "n"]
)

PE_CENS = col(
    primary_effect,
    ["n_censored", "censored_n"]
)

PE_TRANS = col(
    primary_effect,
    ["primary_transformation"]
)

PE_BETA = col(
    primary_effect,
    ["beta_Q_standardized_mean"]
)

PE_LOW = col(
    primary_effect,
    ["beta_Q_standardized_hdi_low"]
)

PE_HIGH = col(
    primary_effect,
    ["beta_Q_standardized_hdi_high"]
)

PE_PDIR = col(
    primary_effect,
    ["posterior_direction_probability"]
)

PE_EXP = col(
    primary_effect,
    ["CQ_exponent_mean"],
    required=False
)

PE_RAW = col(
    primary_effect,
    [
        "beta_Q_transformed_scale",
        "beta_Q_transformed_scale_mean",
        "transformed_scale_beta_Q"
    ],
    required=False
)


# Primary support columns
PS_MEAN_LPD = col(
    primary_support,
    [
        "campaign_equal_mean_delta_LPD",
        "mean_delta_LPD_Q_minus_null"
    ]
)

PS_BETTER = col(
    primary_support,
    ["campaigns_Q_better"]
)

PS_WORSE = col(
    primary_support,
    ["campaigns_Q_worse"]
)

PS_C09 = col(
    primary_support,
    [
        "C09_delta_LPD_Q_minus_null",
        "C09_delta_LPD"
    ]
)

PS_CLASS = col(
    primary_support,
    ["primary_CQ_support_class"]
)

PS_PROCESS = col(
    primary_support,
    ["hydrological_process_language"]
)

PS_GENERAL = col(
    primary_support,
    [
        "temporal_generalization",
        "temporal_generalization_status"
    ]
)


# Sensitivity effect columns
SE_BETA = col(
    sens_effect,
    ["beta_Q_standardized_mean"]
)

SE_LOW = col(
    sens_effect,
    ["beta_Q_standardized_hdi_low"]
)

SE_HIGH = col(
    sens_effect,
    ["beta_Q_standardized_hdi_high"]
)

SE_PDIR = col(
    sens_effect,
    ["posterior_direction_probability"]
)

SE_EXP = col(
    sens_effect,
    ["CQ_exponent_mean"],
    required=False
)


# Sensitivity comparison columns
SC_MEAN = col(
    sens_compare,
    ["campaign_equal_mean_delta_LPD_sensitivity"]
)

SC_BETTER = col(
    sens_compare,
    ["campaigns_Q_better_sensitivity"]
)

SC_WORSE = col(
    sens_compare,
    ["campaigns_Q_worse_sensitivity"]
)

SC_C09 = col(
    sens_compare,
    ["C09_delta_LPD_Q_minus_null_sensitivity"]
)

SC_JOINT = col(
    sens_compare,
    ["sensitivity_joint_support"]
)


# --------------------------------------------------------------------------------------------------
# 6. BUILD SINGLE EDITORIAL DATASET
# --------------------------------------------------------------------------------------------------

base = pd.DataFrame({
    "response_key":
        primary_effect[PE_KEY],

    "response_label":
        primary_effect[PE_LABEL],

    "n_primary":
        pd.to_numeric(
            primary_effect[PE_N],
            errors="coerce"
        ),

    "n_censored_primary":
        pd.to_numeric(
            primary_effect[PE_CENS],
            errors="coerce"
        ),

    "transformation":
        primary_effect[PE_TRANS],

    "primary_beta_std":
        pd.to_numeric(
            primary_effect[PE_BETA],
            errors="coerce"
        ),

    "primary_hdi_low":
        pd.to_numeric(
            primary_effect[PE_LOW],
            errors="coerce"
        ),

    "primary_hdi_high":
        pd.to_numeric(
            primary_effect[PE_HIGH],
            errors="coerce"
        ),

    "primary_Pdirection":
        pd.to_numeric(
            primary_effect[PE_PDIR],
            errors="coerce"
        )
})


if PE_EXP is not None:
    base["primary_CQ_exponent"] = pd.to_numeric(
        primary_effect[PE_EXP],
        errors="coerce"
    )
else:
    base["primary_CQ_exponent"] = np.nan


if PE_RAW is not None:
    base["primary_transformed_scale_slope"] = pd.to_numeric(
        primary_effect[PE_RAW],
        errors="coerce"
    )
else:
    base["primary_transformed_scale_slope"] = np.nan


support_small = primary_support[
    [
        "response_key",
        PS_MEAN_LPD,
        PS_BETTER,
        PS_WORSE,
        PS_C09,
        PS_CLASS,
        PS_PROCESS,
        PS_GENERAL
    ]
].copy()

support_small.columns = [
    "response_key",
    "primary_mean_delta_LPD",
    "primary_campaigns_Q_better",
    "primary_campaigns_Q_worse",
    "primary_C09_delta_LPD",
    "primary_support_class",
    "primary_process_language",
    "primary_temporal_generalization"
]


sens_eff_small = pd.DataFrame({
    "response_key":
        sens_effect["response_key"],

    "sensitivity_beta_std":
        pd.to_numeric(
            sens_effect[SE_BETA],
            errors="coerce"
        ),

    "sensitivity_hdi_low":
        pd.to_numeric(
            sens_effect[SE_LOW],
            errors="coerce"
        ),

    "sensitivity_hdi_high":
        pd.to_numeric(
            sens_effect[SE_HIGH],
            errors="coerce"
        ),

    "sensitivity_Pdirection":
        pd.to_numeric(
            sens_effect[SE_PDIR],
            errors="coerce"
        )
})


if SE_EXP is not None:
    sens_eff_small[
        "sensitivity_CQ_exponent"
    ] = pd.to_numeric(
        sens_effect[SE_EXP],
        errors="coerce"
    )
else:
    sens_eff_small[
        "sensitivity_CQ_exponent"
    ] = np.nan


sens_cmp_small = sens_compare[
    [
        "response_key",
        SC_MEAN,
        SC_BETTER,
        SC_WORSE,
        SC_C09,
        SC_JOINT
    ]
].copy()

sens_cmp_small.columns = [
    "response_key",
    "sensitivity_mean_delta_LPD",
    "sensitivity_campaigns_Q_better",
    "sensitivity_campaigns_Q_worse",
    "sensitivity_C09_delta_LPD",
    "sensitivity_joint_support"
]


editorial = (
    base
    .merge(
        support_small,
        on="response_key",
        how="left",
        validate="one_to_one"
    )
    .merge(
        sens_eff_small,
        on="response_key",
        how="left",
        validate="one_to_one"
    )
    .merge(
        sens_cmp_small,
        on="response_key",
        how="left",
        validate="one_to_one"
    )
)


assert len(editorial) == 10
assert editorial["response_key"].nunique() == 10


# --------------------------------------------------------------------------------------------------
# 7. FIX MANUSCRIPT DISPLAY ORDER
#
# Conceptual order rather than sorting by effect size.
# Prevents visual ranking from dictating scientific architecture.
# --------------------------------------------------------------------------------------------------

display_order = [
    "C.E._SOL",
    "Na",
    "SO4_ppm",
    "pH_SOL",
    "NH4",
    "Total coliforms",
    "Aluminio",
    "Hierro",
    "Silicio",
    "Water temperature"
]

display_names = {
    "C.E._SOL": "Electrical conductivity",
    "Na": "Na",
    "SO4_ppm": "SO₄",
    "pH_SOL": "pH",
    "NH4": "NH₄",
    "Total coliforms": "Total coliforms",
    "Aluminio": "Al",
    "Hierro": "Fe",
    "Silicio": "Si",
    "Water temperature": "Water temperature"
}


editorial["display_order"] = editorial[
    "response_label"
].map(
    {
        k: i
        for i, k in enumerate(display_order)
    }
)

if editorial["display_order"].isna().any():
    bad = editorial.loc[
        editorial["display_order"].isna(),
        "response_label"
    ].tolist()

    raise ValueError(
        f"Unrecognized response labels: {bad}"
    )


editorial["display_name"] = editorial[
    "response_label"
].map(display_names)

editorial = (
    editorial
    .sort_values("display_order")
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 8. EXPLICIT CLASSICAL EXPONENT ELIGIBILITY
#
# Frozen architecture:
# only NH4, Na, Al, Fe are eligible for classical log10(C)-log10(Q) exponent interpretation.
# --------------------------------------------------------------------------------------------------

CLASSICAL_EXPONENT_RESPONSES = {
    "NH4",
    "Na",
    "Aluminio",
    "Hierro"
}

editorial[
    "classical_CQ_exponent_eligible"
] = editorial[
    "response_label"
].isin(
    CLASSICAL_EXPONENT_RESPONSES
)


# Never accidentally display an exponent for an ineligible response.
editorial.loc[
    ~editorial["classical_CQ_exponent_eligible"],
    [
        "primary_CQ_exponent",
        "sensitivity_CQ_exponent"
    ]
] = np.nan


# --------------------------------------------------------------------------------------------------
# 9. SENSITIVITY STATUS — DESCRIPTIVE ONLY
#
# IMPORTANT:
# Does NOT overwrite primary class.
# --------------------------------------------------------------------------------------------------

def sensitivity_status(row):

    primary_gate = (
        row["primary_Pdirection"] >= 0.95
        and row["primary_mean_delta_LPD"] > 0
    )

    sensitivity_gate = (
        row["sensitivity_Pdirection"] >= 0.95
        and row["sensitivity_mean_delta_LPD"] > 0
    )

    if primary_gate and sensitivity_gate:
        return "PRIMARY_AND_SENSITIVITY_SUPPORTED"

    if (not primary_gate) and sensitivity_gate:
        return "SENSITIVITY_SUPPORT_ONLY"

    if primary_gate and (not sensitivity_gate):
        return "PRIMARY_SUPPORT_NOT_PRESERVED"

    return "NO_JOINT_SUPPORT"


editorial[
    "sensitivity_status"
] = editorial.apply(
    sensitivity_status,
    axis=1
)


# --------------------------------------------------------------------------------------------------
# 10. CROSS-CHECK EXPECTED SCIENTIFIC PATTERN
# --------------------------------------------------------------------------------------------------

expected_primary_classes = {
    "C.E._SOL": "ROBUST_NEGATIVE_Q_RESPONSE",
    "Na": "ROBUST_NEGATIVE_Q_RESPONSE",
    "SO4_ppm": "ROBUST_NEGATIVE_Q_RESPONSE",
    "pH_SOL": "ROBUST_NEGATIVE_Q_RESPONSE",
    "NH4": "NO_ROBUST_Q_SUPPORT",
    "Total coliforms": "NO_ROBUST_Q_SUPPORT",
    "Aluminio": "POSITIVE_Q_ASSOCIATION_NOT_TEMPORALLY_GENERALIZABLE",
    "Hierro": "POSITIVE_Q_ASSOCIATION_NOT_TEMPORALLY_GENERALIZABLE",
    "Silicio": "ROBUST_NEGATIVE_Q_RESPONSE",
    "Water temperature": "ROBUST_NEGATIVE_Q_RESPONSE"
}


for label, expected in expected_primary_classes.items():

    actual = editorial.loc[
        editorial["response_label"] == label,
        "primary_support_class"
    ].iloc[0]

    assert actual == expected, (
        f"{label}: primary class changed unexpectedly.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )


# Al and Fe should be sensitivity-support only.
for label in ["Aluminio", "Hierro"]:

    status = editorial.loc[
        editorial["response_label"] == label,
        "sensitivity_status"
    ].iloc[0]

    assert status == "SENSITIVITY_SUPPORT_ONLY", (
        f"{label}: unexpected sensitivity status = {status}"
    )


# NH4 and coliforms should remain unsupported.
for label in ["NH4", "Total coliforms"]:

    status = editorial.loc[
        editorial["response_label"] == label,
        "sensitivity_status"
    ].iloc[0]

    assert status == "NO_JOINT_SUPPORT", (
        f"{label}: unexpected sensitivity status = {status}"
    )


# --------------------------------------------------------------------------------------------------
# 11. FIGURE DATA
# --------------------------------------------------------------------------------------------------

figure_data = editorial[
    [
        "response_key",
        "response_label",
        "display_name",
        "primary_beta_std",
        "primary_hdi_low",
        "primary_hdi_high",
        "sensitivity_beta_std",
        "sensitivity_hdi_low",
        "sensitivity_hdi_high",
        "primary_mean_delta_LPD",
        "sensitivity_mean_delta_LPD",
        "primary_support_class",
        "sensitivity_status"
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 12. MAIN MANUSCRIPT FIGURE
#
# Grayscale:
#   Primary     = filled circle
#   Sensitivity = open square
#
# No colors.
# No ranking by effect size.
# --------------------------------------------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.0, 6.2),
    sharey=True
)

ax1, ax2 = axes

y = np.arange(
    len(editorial)
)


# ----------------------------------
# Panel A — posterior Q effects
# ----------------------------------

for i, row in editorial.iterrows():

    # Primary 95% HDI
    ax1.plot(
        [
            row["primary_hdi_low"],
            row["primary_hdi_high"]
        ],
        [i, i],
        linewidth=1.6,
        zorder=2
    )

    ax1.scatter(
        row["primary_beta_std"],
        i,
        marker="o",
        s=48,
        zorder=3
    )

    # Sensitivity 95% HDI
    ax1.plot(
        [
            row["sensitivity_hdi_low"],
            row["sensitivity_hdi_high"]
        ],
        [i + 0.16, i + 0.16],
        linewidth=1.1,
        linestyle="--",
        zorder=2
    )

    ax1.scatter(
        row["sensitivity_beta_std"],
        i + 0.16,
        marker="s",
        s=42,
        facecolors="none",
        edgecolors="black",
        zorder=3
    )


ax1.axvline(
    0,
    linewidth=0.9,
    linestyle=":"
)

ax1.set_xlabel(
    "Standardized discharge effect, βQ"
)

ax1.set_ylabel("")

ax1.set_yticks(y)

ax1.set_yticklabels(
    editorial["display_name"]
)

ax1.invert_yaxis()

ax1.set_title(
    "A  Posterior discharge effects",
    loc="left",
    fontweight="bold"
)


# Legend via dummy handles
ax1.scatter(
    [],
    [],
    marker="o",
    s=48,
    label="Primary: observed Q"
)

ax1.scatter(
    [],
    [],
    marker="s",
    s=42,
    facecolors="none",
    edgecolors="black",
    label="Sensitivity: + reconstructed C09 Q"
)

ax1.legend(
    frameon=False,
    fontsize=8,
    loc="lower right"
)


# ----------------------------------
# Panel B — temporal generalization
# ----------------------------------

for i, row in editorial.iterrows():

    x1 = row["primary_mean_delta_LPD"]
    x2 = row["sensitivity_mean_delta_LPD"]

    # Connect primary and sensitivity.
    ax2.plot(
        [x1, x2],
        [i, i + 0.16],
        linewidth=0.8,
        linestyle=":"
    )

    ax2.scatter(
        x1,
        i,
        marker="o",
        s=48,
        zorder=3
    )

    ax2.scatter(
        x2,
        i + 0.16,
        marker="s",
        s=42,
        facecolors="none",
        edgecolors="black",
        zorder=3
    )


ax2.axvline(
    0,
    linewidth=0.9,
    linestyle=":"
)

ax2.set_xlabel(
    "Campaign-equal mean ΔLPD (Q model − null)"
)

ax2.set_title(
    "B  Held-out predictive contribution of discharge",
    loc="left",
    fontweight="bold"
)

ax2.tick_params(
    axis="y",
    labelleft=False
)


# ----------------------------------
# Figure-level formatting
# ----------------------------------

for ax in axes:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="x",
        linewidth=0.35,
        alpha=0.35
    )


fig.suptitle(
    "Discharge–water-quality response spectrum and sensitivity to C09 hydrometric coverage",
    fontsize=12,
    y=0.995
)

fig.tight_layout(
    rect=[0, 0, 1, 0.96]
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE FIGURE
# --------------------------------------------------------------------------------------------------

FIG_PDF = os.path.join(
    OUT_DIR,
    "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1.pdf"
)

FIG_PNG = os.path.join(
    OUT_DIR,
    "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1_600dpi.png"
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight"
)

fig.savefig(
    FIG_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# --------------------------------------------------------------------------------------------------
# 14. MAIN-TEXT TABLE
# --------------------------------------------------------------------------------------------------

def fmt_interval(mean, low, high, digits=3):

    if (
        pd.isna(mean)
        or pd.isna(low)
        or pd.isna(high)
    ):
        return ""

    return (
        f"{mean:.{digits}f} "
        f"[{low:.{digits}f}, {high:.{digits}f}]"
    )


def fmt_num(x, digits=3):

    if pd.isna(x):
        return ""

    return f"{x:.{digits}f}"


main_rows = []

for _, row in editorial.iterrows():

    exponent_primary = ""

    if (
        row["classical_CQ_exponent_eligible"]
        and pd.notna(row["primary_CQ_exponent"])
    ):
        exponent_primary = (
            f"{row['primary_CQ_exponent']:.3f}"
        )

    main_rows.append({
        "Response":
            row["display_name"],

        "n":
            int(row["n_primary"]),

        "Censored_n":
            int(row["n_censored_primary"]),

        "Response_transform":
            row["transformation"],

        "Primary_betaQ_std_95HDI":
            fmt_interval(
                row["primary_beta_std"],
                row["primary_hdi_low"],
                row["primary_hdi_high"]
            ),

        "Primary_Pdirection":
            fmt_num(
                row["primary_Pdirection"],
                3
            ),

        "Classical_CQ_exponent_if_eligible":
            exponent_primary,

        "Primary_mean_delta_LPD":
            fmt_num(
                row["primary_mean_delta_LPD"],
                3
            ),

        "Primary_campaigns_Q_better":
            f"{int(row['primary_campaigns_Q_better'])}/10",

        "Primary_inference":
            row["primary_support_class"],

        "C09_sensitivity":
            row["sensitivity_status"]
    })


main_table = pd.DataFrame(
    main_rows
)


# --------------------------------------------------------------------------------------------------
# 15. SUPPLEMENTARY RESPONSE-LEVEL TABLE
# --------------------------------------------------------------------------------------------------

supp_response = editorial[
    [
        "response_key",
        "display_name",
        "n_primary",
        "n_censored_primary",
        "transformation",
        "primary_beta_std",
        "primary_hdi_low",
        "primary_hdi_high",
        "primary_Pdirection",
        "primary_CQ_exponent",
        "primary_transformed_scale_slope",
        "primary_mean_delta_LPD",
        "primary_campaigns_Q_better",
        "primary_campaigns_Q_worse",
        "primary_C09_delta_LPD",
        "primary_support_class",
        "primary_process_language",
        "primary_temporal_generalization",
        "sensitivity_beta_std",
        "sensitivity_hdi_low",
        "sensitivity_hdi_high",
        "sensitivity_Pdirection",
        "sensitivity_CQ_exponent",
        "sensitivity_mean_delta_LPD",
        "sensitivity_campaigns_Q_better",
        "sensitivity_campaigns_Q_worse",
        "sensitivity_C09_delta_LPD",
        "sensitivity_joint_support",
        "sensitivity_status"
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 16. SUPPLEMENTARY FOLD TABLE
#
# Combine 100 primary + 100 sensitivity folds.
# --------------------------------------------------------------------------------------------------

primary_folds_s = primary_folds.copy()
primary_folds_s.insert(
    0,
    "analysis",
    "PRIMARY_OBSERVED_Q"
)

sens_folds_s = sens_folds.copy()
sens_folds_s.insert(
    0,
    "analysis",
    "SENSITIVITY_OBSERVED_PLUS_RECONSTRUCTED_C09_Q"
)

all_folds = pd.concat(
    [
        primary_folds_s,
        sens_folds_s
    ],
    ignore_index=True,
    sort=False
)

assert len(all_folds) == 200


# --------------------------------------------------------------------------------------------------
# 17. FIGURE CAPTION
# --------------------------------------------------------------------------------------------------

caption = """
Figure X. Discharge–water-quality response spectrum and sensitivity to hydrometric coverage in campaign C09.
(A) Posterior standardized discharge effects (βQ) from hierarchical response–discharge models with station-level
random intercepts. Filled circles show the primary analysis using observed discharge only; open squares show the
sensitivity analysis additionally including reconstructed C09 discharge. Horizontal intervals are 95% highest
density intervals. (B) Campaign-equal mean difference in held-out log predictive density (ΔLPD) between the
discharge model and a station-only null model under leave-one-campaign-out validation. Positive values indicate
improved held-out prediction after including discharge. The primary analysis remains the inferential reference;
reconstructed C09 discharge is used only as a sensitivity analysis. Response transformations and left-censoring
rules were specified before C–Q inference. Positive or negative discharge associations are not interpreted by
themselves as evidence of hydrological connectivity or a specific mobilization mechanism.
"""

caption = " ".join(
    caption.split()
)


# --------------------------------------------------------------------------------------------------
# 18. MANUSCRIPT-READY RESULT STATEMENT
#
# Stored as text for later manuscript assembly.
# --------------------------------------------------------------------------------------------------

result_text = """
Discharge effects differed markedly among water-quality domains. In the primary observed-discharge analysis,
electrical conductivity, Na, SO4, pH, Si and water temperature showed directionally supported discharge effects
that also improved campaign-held-out prediction relative to station-only models. NH4 and total coliforms showed
no robust discharge support. Al and Fe exhibited strong positive posterior discharge associations, but their
primary campaign-equal predictive gains were negative because of poor held-out performance in C09. When
reconstructed C09 discharge was included as a sensitivity analysis, the campaign-equal ΔLPD increased from
-0.038 to 0.129 for Al and from -0.070 to 0.046 for Fe. The C09-specific ΔLPD changed from -1.642 to 0.043 for
Al and from -1.311 to -0.184 for Fe. Thus, the apparent lack of temporal generalization of the positive Al and
Fe discharge associations in the primary analysis was sensitive to sparse hydrometric coverage in C09,
particularly for Al. These sensitivity results do not replace the observed-discharge primary classification and
do not by themselves establish a specific mobilization or connectivity mechanism.
"""

result_text = " ".join(
    result_text.split()
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

qa_rows = [
    (
        "primary_effect_rows_10",
        len(primary_effect) == 10
    ),
    (
        "primary_support_rows_10",
        len(primary_support) == 10
    ),
    (
        "sensitivity_effect_rows_10",
        len(sens_effect) == 10
    ),
    (
        "sensitivity_comparison_rows_10",
        len(sens_compare) == 10
    ),
    (
        "editorial_rows_10",
        len(editorial) == 10
    ),
    (
        "primary_folds_100",
        len(primary_folds) == 100
    ),
    (
        "sensitivity_folds_100",
        len(sens_folds) == 100
    ),
    (
        "combined_folds_200",
        len(all_folds) == 200
    ),
    (
        "all_sensitivity_folds_pass",
        sens_folds["fold_pass"].astype(bool).all()
    ),
    (
        "primary_classes_preserved",
        all(
            editorial.loc[
                editorial["response_label"] == k,
                "primary_support_class"
            ].iloc[0] == v
            for k, v in expected_primary_classes.items()
        )
    ),
    (
        "Al_sensitivity_support_only",
        editorial.loc[
            editorial["response_label"] == "Aluminio",
            "sensitivity_status"
        ].iloc[0] == "SENSITIVITY_SUPPORT_ONLY"
    ),
    (
        "Fe_sensitivity_support_only",
        editorial.loc[
            editorial["response_label"] == "Hierro",
            "sensitivity_status"
        ].iloc[0] == "SENSITIVITY_SUPPORT_ONLY"
    ),
    (
        "NH4_remains_unsupported",
        editorial.loc[
            editorial["response_label"] == "NH4",
            "sensitivity_status"
        ].iloc[0] == "NO_JOINT_SUPPORT"
    ),
    (
        "coliforms_remain_unsupported",
        editorial.loc[
            editorial["response_label"] == "Total coliforms",
            "sensitivity_status"
        ].iloc[0] == "NO_JOINT_SUPPORT"
    ),
    (
        "only_four_classical_exponent_responses",
        int(
            editorial[
                "classical_CQ_exponent_eligible"
            ].sum()
        ) == 4
    ),
    (
        "no_new_models_fit",
        True
    ),
    (
        "primary_inference_not_modified",
        True
    ),
    (
        "process_reclassification_not_performed",
        True
    ),
    (
        "chemostasis_not_assigned",
        True
    ),
    (
        "connectivity_not_inferred",
        True
    )
]

qa = pd.DataFrame(
    qa_rows,
    columns=[
        "check",
        "pass"
    ]
)

BLOCK_PASS = bool(
    qa["pass"].all()
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE DATA PRODUCTS
# --------------------------------------------------------------------------------------------------

FIG_DATA_OUT = os.path.join(
    OUT_DIR,
    "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_DATA_v1.csv"
)

MAIN_CSV = os.path.join(
    OUT_DIR,
    "TABLE_MAIN_CQ_RESPONSE_SPECTRUM_v1.csv"
)

MAIN_XLSX = os.path.join(
    OUT_DIR,
    "TABLE_MAIN_CQ_RESPONSE_SPECTRUM_v1.xlsx"
)

SUPP_RESPONSE_CSV = os.path.join(
    OUT_DIR,
    "TABLE_S_CQ_RESPONSE_LEVEL_RESULTS_v1.csv"
)

SUPP_LOCO_CSV = os.path.join(
    OUT_DIR,
    "TABLE_S_CQ_ALL_LOCO_FOLDS_v1.csv"
)

CAPTION_OUT = os.path.join(
    OUT_DIR,
    "FIG_CQ_CAPTION_v1.txt"
)

RESULT_OUT = os.path.join(
    OUT_DIR,
    "CQ_RESULTS_TEXT_v1.txt"
)

QA_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93C_EDITORIAL_FREEZE_QA_v1.csv"
)

MANIFEST_OUT = os.path.join(
    OUT_DIR,
    "BLOCK93C_EDITORIAL_FREEZE_MANIFEST_v1.json"
)


figure_data.to_csv(
    FIG_DATA_OUT,
    index=False
)

main_table.to_csv(
    MAIN_CSV,
    index=False
)

main_table.to_excel(
    MAIN_XLSX,
    index=False
)

supp_response.to_csv(
    SUPP_RESPONSE_CSV,
    index=False
)

all_folds.to_csv(
    SUPP_LOCO_CSV,
    index=False
)

with open(
    CAPTION_OUT,
    "w",
    encoding="utf-8"
) as f:
    f.write(caption)

with open(
    RESULT_OUT,
    "w",
    encoding="utf-8"
) as f:
    f.write(result_text)

qa.to_csv(
    QA_OUT,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 21. MANIFEST
# --------------------------------------------------------------------------------------------------

outputs = [
    FIG_PDF,
    FIG_PNG,
    FIG_DATA_OUT,
    MAIN_CSV,
    MAIN_XLSX,
    SUPP_RESPONSE_CSV,
    SUPP_LOCO_CSV,
    CAPTION_OUT,
    RESULT_OUT,
    QA_OUT
]

manifest = {
    "block": "9.3C",
    "status": (
        "PASS"
        if BLOCK_PASS
        else "FAIL"
    ),
    "purpose": (
        "Final editorial freeze of C-Q results and "
        "manuscript-ready figure/table generation."
    ),
    "new_models_fit": False,
    "primary_inference_source": (
        "BLOCK 9.2C observed-Q analysis"
    ),
    "sensitivity_role": (
        "Observed + reconstructed C09 discharge; "
        "does not replace primary inference."
    ),
    "primary_support_rule": (
        "posterior direction probability >= 0.95 "
        "AND campaign-equal mean delta LPD > 0"
    ),
    "classical_CQ_exponent_responses": [
        "NH4",
        "Na",
        "Al",
        "Fe"
    ],
    "process_reclassification_performed": False,
    "chemostasis_assigned": False,
    "connectivity_inferred": False,
    "outputs": {}
}

for p in outputs:

    manifest["outputs"][
        os.path.basename(p)
    ] = {
        "sha256": sha256_file(p),
        "bytes": os.path.getsize(p)
    }


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 22. PRINT FINAL PRODUCTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("MAIN MANUSCRIPT TABLE")
print("=" * 120)

print(
    main_table.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("SENSITIVITY STATUS")
print("=" * 120)

print(
    editorial[
        [
            "display_name",
            "primary_support_class",
            "sensitivity_status",
            "primary_mean_delta_LPD",
            "sensitivity_mean_delta_LPD"
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("BLOCK 9.3C QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)


print("\nFROZEN OUTPUTS")
print("-" * 120)

for p in outputs + [MANIFEST_OUT]:

    print(
        os.path.basename(p),
        "| SHA256:",
        sha256_file(p)
    )


print("\n" + "=" * 120)
print("BLOCK 9.3C FINAL GATE")
print("=" * 120)

print(
    "BLOCK 9.3C PASS                   :",
    BLOCK_PASS
)

print(
    "NEW MODELS FIT                    :",
    False
)

print(
    "PRIMARY INFERENCE MODIFIED         :",
    False
)

print(
    "MAIN FIGURE PDF CREATED            :",
    os.path.exists(FIG_PDF)
)

print(
    "MAIN FIGURE 600-DPI PNG CREATED    :",
    os.path.exists(FIG_PNG)
)

print(
    "MAIN TABLE CREATED                 :",
    os.path.exists(MAIN_XLSX)
)

print(
    "SUPPLEMENTARY 200-FOLD TABLE       :",
    len(all_folds) == 200
)

print(
    "PROCESS RECLASSIFICATION           :",
    False
)

print(
    "CHEMOSTASIS ASSIGNED               :",
    False
)

print(
    "CONNECTIVITY INFERRED              :",
    False
)


assert BLOCK_PASS, (
    "BLOCK 9.3C QA FAILED — do not proceed to Block 10."
)

print(
    "\nC-Q ANALYSIS CLOSED/FROZEN."
)

print(
    "NEXT: BLOCK 10 — joint antecedent hydroclimate + discharge models."
)

---
### Imported master block 123

`BLOCK 10.0B — JOINT HYDROCLIMATE + DISCHARGE ARCHITECTURE FREEZE`


In [ ]:
# ==================================================================================================
# BLOCK 10.0B — JOINT HYDROCLIMATE + DISCHARGE ARCHITECTURE FREEZE
# Pre-specified H+Q analysis before observing any joint-model result
# ==================================================================================================
#
# NO MODEL FITTING.
# NO H+Q PERFORMANCE INSPECTION.
# NO WINDOW OPTIMIZATION.
#
# PRIMARY QUESTION:
# Does antecedent hydroclimate provide predictive information on water quality
# beyond instantaneous discharge?
#
# PRIMARY INCREMENTAL CONTRAST:
#     ΔLPD(H+Q − Q)
#
# Contextual contrasts:
#     ΔLPD(H − NULL)
#     ΔLPD(Q − NULL)
#     ΔLPD(H+Q − H)
#
# Primary Q:
#     observed discharge only
#
# Validation:
#     leave-one-campaign-out (LOCO)
#
# Station:
#     random intercept
#
# Censoring:
#     explicit sample-specific left-censor likelihood
#
# ==================================================================================================

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")

MEM_DIR = PROJECT / "MEMORY_SPECTRUM"
CQ_DIR = PROJECT / "CQ_ANALYSIS"

OUT_DIR = PROJECT / "JOINT_HQ_ANALYSIS" / "ARCHITECTURE_FREEZE"
OUT_DIR.mkdir(parents=True, exist_ok=True)


STRICT_PATH = (
    MEM_DIR /
    "BLOCK85E_STRICT_ANTECEDENT_MEMORY_WINDOWS_v1.csv"
)

FAMILY_PATH = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_FAMILY_INFERENCE_v1.csv"
)

WINDOW_PATH = (
    MEM_DIR /
    "BLOCK85E_SUPPORTED_MEMORY_WINDOW_INFERENCE_v1.csv"
)

CQ_SUPPORT_PATH = (
    CQ_DIR /
    "SUPPORT_CLASSIFICATION" /
    "BLOCK92C_PRIMARY_CQ_SUPPORT_SPECTRUM_v1.csv"
)

RESTART_PATH = (
    CQ_DIR /
    "FINAL_BLOCK9_PACKAGE" /
    "BLOCK10_START_HERE.json"
)


# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def assert_unique_rows(df, cols, label):

    n_dup = int(
        df.duplicated(cols).sum()
    )

    assert n_dup == 0, (
        f"{label}: {n_dup} duplicated rows for {cols}"
    )


# --------------------------------------------------------------------------------------------------
# 2. VERIFY INPUTS
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("BLOCK 10.0B — JOINT HYDROCLIMATE + DISCHARGE ARCHITECTURE FREEZE")
print("=" * 120)


required = [
    STRICT_PATH,
    FAMILY_PATH,
    WINDOW_PATH,
    CQ_SUPPORT_PATH,
    RESTART_PATH
]


for p in required:

    print(
        f"{p.name:55s}",
        "FOUND" if p.exists() else "MISSING"
    )

    assert p.exists(), f"Missing frozen input: {p}"


# --------------------------------------------------------------------------------------------------
# 3. LOAD FROZEN PRODUCTS
# --------------------------------------------------------------------------------------------------

strict = pd.read_csv(
    STRICT_PATH,
    low_memory=False
)

family = pd.read_csv(
    FAMILY_PATH,
    low_memory=False
)

window = pd.read_csv(
    WINDOW_PATH,
    low_memory=False
)

cq = pd.read_csv(
    CQ_SUPPORT_PATH,
    low_memory=False
)

with open(
    RESTART_PATH,
    "r",
    encoding="utf-8"
) as f:

    restart = json.load(f)


print("\nFrozen input shapes:")
print("strict :", strict.shape)
print("family :", family.shape)
print("window :", window.shape)
print("C-Q    :", cq.shape)


assert strict.shape[0] == 5
assert family.shape[0] == 50
assert window.shape[0] == 290
assert cq.shape[0] == 10


# --------------------------------------------------------------------------------------------------
# 4. VERIFY STRICT MEMORY ROWS
# --------------------------------------------------------------------------------------------------

required_strict_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "posterior_directional_support",
    "predictive_plateau_support",
    "joint_supported_window",
    "supported_antecedent_memory_window",
    "effect_direction",
    "response_interpretation_role"
]


missing = [
    c for c in required_strict_cols
    if c not in strict.columns
]

assert not missing, (
    f"Missing strict-memory columns: {missing}"
)


assert strict[
    "posterior_directional_support"
].astype(bool).all()

assert strict[
    "predictive_plateau_support"
].astype(bool).all()

assert strict[
    "joint_supported_window"
].astype(bool).all()

assert strict[
    "supported_antecedent_memory_window"
].astype(bool).all()


assert_unique_rows(
    strict,
    [
        "response_key",
        "hydro_var"
    ],
    "strict memory table"
)


print("\nFrozen strict antecedent-memory rows:")

print(
    strict[
        [
            "response_key",
            "hydro_family",
            "window_days",
            "hydro_var",
            "effect_direction",
            "response_interpretation_role"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 5. PRE-SPECIFY PRIMARY AND REFERENCE JOINT TESTS
# --------------------------------------------------------------------------------------------------

# Scientific rule:
#
# PRIMARY:
#   strict antecedent-memory windows whose response role was already frozen
#   as HYDROCHEMICAL_RESPONSE.
#
# REFERENCE:
#   strict antecedent-memory window whose role was already frozen as
#   THERMAL_RESPONSE_REFERENCE.
#
# This decision is based ONLY on Block 8.
# No H+Q result exists yet.

joint_plan = strict.copy()


joint_plan["joint_analysis_role"] = np.where(
    joint_plan[
        "response_interpretation_role"
    ].eq(
        "HYDROCHEMICAL_RESPONSE"
    ),
    "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST",
    np.where(
        joint_plan[
            "response_interpretation_role"
        ].eq(
            "THERMAL_RESPONSE_REFERENCE"
        ),
        "SECONDARY_THERMAL_REFERENCE",
        "NOT_INCLUDED"
    )
)


joint_plan["primary_HQ_test"] = (
    joint_plan[
        "joint_analysis_role"
    ]
    ==
    "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST"
)


joint_plan["secondary_reference"] = (
    joint_plan[
        "joint_analysis_role"
    ]
    ==
    "SECONDARY_THERMAL_REFERENCE"
)


assert int(
    joint_plan[
        "primary_HQ_test"
    ].sum()
) == 4


assert int(
    joint_plan[
        "secondary_reference"
    ].sum()
) == 1


# --------------------------------------------------------------------------------------------------
# 6. FREEZE EXACT MODEL SET
# --------------------------------------------------------------------------------------------------

# Same-row paired comparison is mandatory.
#
# Within every response × H pair and every held-out campaign:
#
# M0: station-only null
# MQ: station + log10(Q)
# MH: station + H
# MHQ: station + log10(Q) + H
#
# Q-only MUST be refit on the exact same rows as MHQ.
# We do NOT recycle Block-9 fold LPD blindly.


joint_plan["model_null"] = (
    "Y* ~ 1 + (1|station)"
)

joint_plan["model_Q"] = (
    "Y* ~ log10(Q_observed) + (1|station)"
)

joint_plan["model_H"] = (
    "Y* ~ H + (1|station)"
)

joint_plan["model_HQ"] = (
    "Y* ~ log10(Q_observed) + H + (1|station)"
)


joint_plan["primary_predictive_contrast"] = (
    "delta_LPD_HQ_minus_Q"
)

joint_plan["contextual_contrast_H_vs_null"] = (
    "delta_LPD_H_minus_null"
)

joint_plan["contextual_contrast_Q_vs_null"] = (
    "delta_LPD_Q_minus_null"
)

joint_plan["contextual_contrast_HQ_vs_H"] = (
    "delta_LPD_HQ_minus_H"
)


# --------------------------------------------------------------------------------------------------
# 7. RESPONSE TRANSFORMATION / CENSORING CROSSWALK
# --------------------------------------------------------------------------------------------------

# Frozen from Block 8.4B.
# These MUST NOT be changed based on Block-10 performance.

response_architecture = pd.DataFrame(
    [
        [
            "AC::C.E._SOL::dS/m",
            "ORIGINAL",
            False
        ],

        [
            "AC::Na::mg/L",
            "LOG10",
            False
        ],

        [
            "AC::SO4_ppm::mg/L",
            "ORIGINAL",
            True
        ],

        [
            "FIELD::Water temperature::degC",
            "ORIGINAL",
            False
        ]
    ],
    columns=[
        "response_key",
        "response_transform",
        "censor_aware_required"
    ]
)


# Note:
# EC appears twice because P14 and P60 are two independently frozen
# strict memory windows.

joint_plan = joint_plan.merge(
    response_architecture,
    on="response_key",
    how="left",
    validate="many_to_one"
)


assert joint_plan[
    "response_transform"
].notna().all()


# --------------------------------------------------------------------------------------------------
# 8. ADD BLOCK-9 C-Q CONTEXT — WITHOUT ALTERING ELIGIBILITY
# --------------------------------------------------------------------------------------------------

cq_cols = [
    "response_key",
    "primary_CQ_support_class"
]


for c in cq_cols:

    assert c in cq.columns


joint_plan = joint_plan.merge(
    cq[
        cq_cols
    ],
    on="response_key",
    how="left",
    validate="many_to_one"
)


assert joint_plan[
    "primary_CQ_support_class"
].notna().all()


# IMPORTANT:
# C-Q classification is context only.
# It did NOT determine which H windows were selected.


# --------------------------------------------------------------------------------------------------
# 9. SPECIAL EC MULTI-WINDOW RULE
# --------------------------------------------------------------------------------------------------

# EC has two strict antecedent-memory windows:
# precipitation 14 d and precipitation 60 d.
#
# They remain TWO PRE-SPECIFIED TESTS.
#
# We will NOT:
#   - choose whichever gives the larger H+Q ΔLPD,
#   - call the better one "the EC memory timescale",
#   - combine P14 and P60 in the same model.
#
# Interpretation must acknowledge correlated nested precipitation windows.


joint_plan["multi_window_response_rule"] = np.where(
    joint_plan[
        "response_key"
    ].eq(
        "AC::C.E._SOL::dS/m"
    ),
    (
        "TWO_PRE_SPECIFIED_CORRELATED_WINDOWS;"
        "NO_POST_HOC_WINNER;"
        "NO_SIMULTANEOUS_P14_P60_MODEL"
    ),
    "SINGLE_PRE_SPECIFIED_STRICT_WINDOW"
)


# --------------------------------------------------------------------------------------------------
# 10. FREEZE PRIMARY SUPPORT RULE
# --------------------------------------------------------------------------------------------------

# We need both:
#
# A) posterior evidence that H contributes conditionally on Q
# B) out-of-campaign predictive improvement of H+Q over Q
#
# Posterior criterion:
#     P(direction of beta_H | Q) >= 0.95
#
# Predictive criterion:
#     campaign-equal mean ΔLPD(H+Q - Q) > 0
#
# BOTH required for primary incremental antecedent-information support.
#
# Campaign majority is descriptive only.
#
# This mirrors the philosophy frozen before Block-9 inference.


support_rule = pd.DataFrame(
    [
        [
            "HQ01",
            "Primary Q source",
            "Observed discharge only"
        ],

        [
            "HQ02",
            "Primary H source",
            "Only strict Block-8 antecedent-memory windows frozen before H+Q modeling"
        ],

        [
            "HQ03",
            "Primary incremental contrast",
            "Campaign-equal mean delta_LPD(H+Q minus Q)"
        ],

        [
            "HQ04",
            "Posterior conditional-H gate",
            "P(direction of beta_H conditional on Q) >= 0.95"
        ],

        [
            "HQ05",
            "Predictive incremental-H gate",
            "Campaign-equal mean delta_LPD(H+Q minus Q) > 0"
        ],

        [
            "HQ06",
            "Primary joint support",
            "HQ04 AND HQ05"
        ],

        [
            "HQ07",
            "Validation",
            "Leave one complete campaign out"
        ],

        [
            "HQ08",
            "Scaling in LOCO",
            "Q and H scaling estimated from training campaigns only"
        ],

        [
            "HQ09",
            "Q transformation",
            "log10(Q)"
        ],

        [
            "HQ10",
            "Station structure",
            "Random intercept; unseen held-out stations marginalized predictively"
        ],

        [
            "HQ11",
            "Censoring",
            "Explicit sample-specific left-censor likelihood; no substitution"
        ],

        [
            "HQ12",
            "Repeated laboratory subsamples",
            "Retained as distinct analytical subsamples sharing physical Q and hydroclimate"
        ],

        [
            "HQ13",
            "Comparator fairness",
            "NULL, Q, H and H+Q fitted on identical response-specific rows within each fold"
        ],

        [
            "HQ14",
            "EC precipitation windows",
            "P14 and P60 are separate pre-specified correlated tests; no post-hoc winner"
        ],

        [
            "HQ15",
            "Hydroclimate window search",
            "No screening of the remaining 29 windows in primary Block 10"
        ],

        [
            "HQ16",
            "Interaction",
            "No H x Q interaction in primary additive analysis"
        ],

        [
            "HQ17",
            "C09 reconstructed Q",
            "Excluded from primary; may enter a separately labeled sensitivity analysis"
        ],

        [
            "HQ18",
            "Campaign majority",
            "Descriptive only; not part of primary support gate"
        ],

        [
            "HQ19",
            "Causal language",
            "Incremental predictive information is not proof of causal memory mechanism"
        ],

        [
            "HQ20",
            "Connectivity",
            "No connectivity inference in Block 10; H3 remains deferred to Block 11"
        ],

        [
            "HQ21",
            "Al and Fe",
            (
                "Not part of primary H1 strict-window test unless independently "
                "pre-specified later as diagnostic analysis; no retrospective "
                "window selection from Block-8 spectrum"
            )
        ],

        [
            "HQ22",
            "Response transforms",
            "Frozen Block-8.4B transforms retained unchanged"
        ]
    ],
    columns=[
        "rule_id",
        "rule",
        "frozen_decision"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. FREEZE INTERPRETATION CLASSES
# --------------------------------------------------------------------------------------------------

interpretation_classes = pd.DataFrame(
    [
        [
            "INCREMENTAL_ANTECEDENT_INFORMATION_SUPPORTED",
            (
                "Conditional H posterior gate passes AND "
                "campaign-equal mean delta_LPD(H+Q-Q) > 0"
            )
        ],

        [
            "CONDITIONAL_ASSOCIATION_WITHOUT_PREDICTIVE_GENERALIZATION",
            (
                "Conditional H posterior gate passes but "
                "campaign-equal mean delta_LPD(H+Q-Q) <= 0"
            )
        ],

        [
            "PREDICTIVE_GAIN_WITHOUT_DIRECTIONAL_POSTERIOR_SUPPORT",
            (
                "delta_LPD(H+Q-Q) > 0 but conditional H posterior "
                "direction probability < 0.95"
            )
        ],

        [
            "NO_ROBUST_INCREMENTAL_HYDROCLIMATE_SUPPORT",
            (
                "Neither required primary gate is jointly satisfied"
            )
        ]
    ],
    columns=[
        "future_classification",
        "definition"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. QA — ENSURE NO SCIENTIFIC DRIFT
# --------------------------------------------------------------------------------------------------

expected_pairs = {
    (
        "AC::C.E._SOL::dS/m",
        "ppt_sum_14d"
    ),

    (
        "AC::C.E._SOL::dS/m",
        "ppt_sum_60d"
    ),

    (
        "AC::Na::mg/L",
        "ppt_sum_60d"
    ),

    (
        "AC::SO4_ppm::mg/L",
        "ppt_sum_60d"
    ),

    (
        "FIELD::Water temperature::degC",
        "sm_rootzone_mean_90d"
    )
}


observed_pairs = set(
    zip(
        joint_plan[
            "response_key"
        ],
        joint_plan[
            "hydro_var"
        ]
    )
)


assert observed_pairs == expected_pairs


assert (
    joint_plan[
        "hydro_family"
    ].eq(
        "precipitation"
    ).sum()
    == 4
)


assert (
    joint_plan[
        "hydro_family"
    ].eq(
        "rootzone_soil_moisture"
    ).sum()
    == 1
)


assert (
    joint_plan[
        "response_key"
    ].eq(
        "AC::C.E._SOL::dS/m"
    ).sum()
    == 2
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE ARCHITECTURE FREEZE
# --------------------------------------------------------------------------------------------------

PLAN_OUT = (
    OUT_DIR /
    "BLOCK100B_JOINT_HQ_ANALYSIS_PLAN_v1.csv"
)

RULE_OUT = (
    OUT_DIR /
    "BLOCK100B_JOINT_HQ_SUPPORT_RULES_v1.csv"
)

CLASS_OUT = (
    OUT_DIR /
    "BLOCK100B_JOINT_HQ_CLASSIFICATION_POLICY_v1.csv"
)


joint_plan.to_csv(
    PLAN_OUT,
    index=False
)

support_rule.to_csv(
    RULE_OUT,
    index=False
)

interpretation_classes.to_csv(
    CLASS_OUT,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 14. MANIFEST
# --------------------------------------------------------------------------------------------------

manifest = {

    "block":
        "10.0B",

    "status":
        "FROZEN_BEFORE_HQ_MODELING",

    "primary_scientific_question":
        (
            "Does antecedent hydroclimate provide predictive "
            "information on water quality beyond instantaneous discharge?"
        ),

    "primary_incremental_contrast":
        "campaign_equal_mean_delta_LPD_HQ_minus_Q",

    "n_primary_tests":
        int(
            joint_plan[
                "primary_HQ_test"
            ].sum()
        ),

    "n_secondary_reference_tests":
        int(
            joint_plan[
                "secondary_reference"
            ].sum()
        ),

    "primary_Q":
        "observed_only",

    "C09_reconstructed_Q":
        "sensitivity_only",

    "validation":
        "leave_one_campaign_out",

    "station_structure":
        "random_intercept",

    "interaction":
        "not_in_primary_model",

    "connectivity":
        "deferred_to_Block_11",

    "inputs": {
        str(p): sha256_file(p)
        for p in required
    },

    "outputs": {
        str(PLAN_OUT): sha256_file(PLAN_OUT),
        str(RULE_OUT): sha256_file(RULE_OUT),
        str(CLASS_OUT): sha256_file(CLASS_OUT)
    }
}


MANIFEST_OUT = (
    OUT_DIR /
    "BLOCK100B_JOINT_HQ_ARCHITECTURE_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 15. FINAL QA
# --------------------------------------------------------------------------------------------------

qa = pd.DataFrame(
    [
        [
            "strict_memory_rows_5",
            len(strict) == 5
        ],

        [
            "primary_hydrochemical_tests_4",
            int(
                joint_plan[
                    "primary_HQ_test"
                ].sum()
            ) == 4
        ],

        [
            "secondary_thermal_reference_1",
            int(
                joint_plan[
                    "secondary_reference"
                ].sum()
            ) == 1
        ],

        [
            "EC_has_two_prespecified_windows",
            int(
                joint_plan[
                    "response_key"
                ].eq(
                    "AC::C.E._SOL::dS/m"
                ).sum()
            ) == 2
        ],

        [
            "all_primary_Q_observed_only",
            True
        ],

        [
            "all_H_windows_preselected_from_Block8",
            True
        ],

        [
            "no_HQ_results_used_for_selection",
            True
        ],

        [
            "campaign_LOCO_frozen",
            True
        ],

        [
            "same_row_comparator_frozen",
            True
        ],

        [
            "censor_likelihood_frozen",
            True
        ],

        [
            "no_HxQ_interaction_primary",
            True
        ],

        [
            "C09_reconstruction_sensitivity_only",
            True
        ],

        [
            "Al_Fe_not_added_post_hoc",
            True
        ],

        [
            "connectivity_deferred",
            True
        ],

        [
            "no_models_fit",
            True
        ]
    ],
    columns=[
        "check",
        "pass"
    ]
)


QA_OUT = (
    OUT_DIR /
    "BLOCK100B_JOINT_HQ_ARCHITECTURE_QA_v1.csv"
)


qa.to_csv(
    QA_OUT,
    index=False
)


BLOCK_PASS = bool(
    qa[
        "pass"
    ].all()
)


# --------------------------------------------------------------------------------------------------
# 16. PRINT FROZEN PLAN
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("FROZEN BLOCK-10 JOINT TEST PLAN")
print("=" * 120)


display_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "effect_direction",
    "joint_analysis_role",
    "response_transform",
    "censor_aware_required",
    "primary_CQ_support_class"
]


print(
    joint_plan[
        display_cols
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("FROZEN SUPPORT RULES")
print("=" * 120)


print(
    support_rule.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("QA")
print("=" * 120)


print(
    qa.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("BLOCK 10.0B FINAL GATE")
print("=" * 120)


print(
    "BLOCK 10.0B PASS                         :",
    BLOCK_PASS
)

print(
    "PRIMARY HYDROCHEMICAL TESTS              :",
    int(
        joint_plan[
            "primary_HQ_test"
        ].sum()
    )
)

print(
    "SECONDARY THERMAL REFERENCES             :",
    int(
        joint_plan[
            "secondary_reference"
        ].sum()
    )
)

print(
    "ALL H WINDOWS SELECTED BEFORE H+Q RESULTS:",
    True
)

print(
    "NEW MODELS FIT                           :",
    False
)

print(
    "READY FOR BLOCK 10.1 ENGINE VALIDATION   :",
    BLOCK_PASS
)


print("\nFiles written:")

for p in [
    PLAN_OUT,
    RULE_OUT,
    CLASS_OUT,
    QA_OUT,
    MANIFEST_OUT
]:
    print(" ", p)


assert BLOCK_PASS


print("\n" + "=" * 120)
print("BLOCK 10.0B FROZEN — DO NOT MODIFY AFTER H+Q RESULTS ARE OBSERVED")
print("=" * 120)

---
### Imported master block 124

`BLOCK 10.1A-R — JOINT H+Q ENGINE VALIDATION — SCHEMA-RESOLVED VERSION`


In [ ]:
# ==================================================================================================
# BLOCK 10.1A-R — JOINT H+Q ENGINE VALIDATION — SCHEMA-RESOLVED VERSION
# Pilot validation: Na–P60 (direct) + SO4–P60 (left-censored)
# ==================================================================================================
#
# PURPOSE
# -------
# Validate the joint Bayesian H+Q engine BEFORE full Block-10 inference.
#
# PRE-SPECIFIED PILOTS
# --------------------
# 1. Na  ~ Q + precipitation_60d
#       response transform: LOG10
#       censoring: none
#
# 2. SO4 ~ Q + precipitation_60d
#       response transform: ORIGINAL
#       censoring: explicit left-censor likelihood
#
# MODELS
# ------
# M0  : station random intercept
# MQ  : log10(Q) + station random intercept
# MH  : H + station random intercept
# MHQ : log10(Q) + H + station random intercept
#
# VALIDATION
# ----------
# - Full-data pilot posterior validation
# - One pre-specified pilot LOCO fold: C10
# - Same rows for all model comparisons
# - Training-only scaling in LOCO
# - Unseen station effects marginalized predictively
#
# IMPORTANT
# ---------
# NO final Block-10 scientific classification.
# NO hydroclimate-window search.
# NO reconstructed C09 Q.
# NO H×Q interaction.
# NO connectivity inference.
#
# ==================================================================================================

import os
import json
import math
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ==================================================================================================
# 0. BAYESIAN STACK
# ==================================================================================================

try:
    import pymc as pm
    import arviz as az
    import pytensor.tensor as pt

except Exception:

    !pip -q install pymc arviz

    import pymc as pm
    import arviz as az
    import pytensor.tensor as pt

from scipy.special import ndtr


# ==================================================================================================
# 1. PROJECT PATHS
# ==================================================================================================

PROJECT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

ARCH_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "ARCHITECTURE_FREEZE"
)

OUT_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "ENGINE_VALIDATION"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PLAN_PATH = (
    ARCH_DIR /
    "BLOCK100B_JOINT_HQ_ANALYSIS_PLAN_v1.csv"
)

MASTER_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MASTER_ANALYTICAL_v1_1.csv"
)

MEMORY_DATA_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)


# ==================================================================================================
# 2. VERIFY FROZEN INPUTS
# ==================================================================================================

print("=" * 120)
print("BLOCK 10.1A-R — JOINT H+Q ENGINE VALIDATION")
print("=" * 120)

for p in [
    PLAN_PATH,
    MASTER_PATH,
    MEMORY_DATA_PATH
]:

    print(
        f"{p.name:55s}",
        "FOUND" if p.exists() else "MISSING"
    )

    assert p.exists(), (
        f"Required frozen input missing:\n{p}"
    )


plan = pd.read_csv(
    PLAN_PATH,
    low_memory=False
)

master = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

mem = pd.read_csv(
    MEMORY_DATA_PATH,
    low_memory=False
)


print("\nArchitecture freeze:")
print(PLAN_PATH)
print("Plan shape:", plan.shape)

print("\nAuthoritative master:")
print(MASTER_PATH)
print("Shape:", master.shape)

print("\nFrozen memory-analysis dataset:")
print(MEMORY_DATA_PATH)
print("Shape:", mem.shape)


assert plan.shape[0] == 5
assert master.shape == (10616, 76)
assert mem.shape == (1300, 60)


# ==================================================================================================
# 3. EXACT SCHEMA — RESOLVED BY BLOCK 10.1A-R0
# ==================================================================================================

M_RESPONSE = "response_key"
M_SAMPLE = "sample_id"
M_FIELD = "field_observation_id"
M_CAMPAIGN = "campaign_id"
M_STATION = "station_id"

M_VALUE = "response_value_numeric"
M_CENSORED = "response_censored"
M_CENSOR_TYPE = "response_censor_type"
M_LIMIT = "response_censor_limit"

A_FIELD = "field_observation_id"
A_Q = "Q_primary_CQ"


required_mem_cols = [
    M_RESPONSE,
    M_SAMPLE,
    M_FIELD,
    M_CAMPAIGN,
    M_STATION,
    M_VALUE,
    M_CENSORED,
    M_CENSOR_TYPE,
    M_LIMIT,
    "ppt_sum_14d",
    "ppt_sum_60d",
    "sm_rootzone_mean_90d"
]


required_master_cols = [
    A_FIELD,
    A_Q
]


missing_mem = [
    c for c in required_mem_cols
    if c not in mem.columns
]

missing_master = [
    c for c in required_master_cols
    if c not in master.columns
]


assert not missing_mem, (
    f"Missing MEMORY columns: {missing_mem}"
)

assert not missing_master, (
    f"Missing MASTER columns: {missing_master}"
)


print("\nExact schema resolved:")
print("Response key        :", M_RESPONSE)
print("Response numeric    :", M_VALUE)
print("Censored flag       :", M_CENSORED)
print("Censor type         :", M_CENSOR_TYPE)
print("Censor limit        :", M_LIMIT)
print("FIELD ID            :", M_FIELD)
print("Station             :", M_STATION)
print("Campaign            :", M_CAMPAIGN)
print("Primary Q           :", A_Q)


# ==================================================================================================
# 4. GLOBAL CENSORING QA
# ==================================================================================================

n_censored_global = int(
    mem[
        M_CENSORED
    ].sum()
)

n_limits_global = int(
    mem[
        M_LIMIT
    ].notna()
    .sum()
)

n_censor_types_global = int(
    mem[
        M_CENSOR_TYPE
    ].notna()
    .sum()
)


print("\nGlobal censoring representation:")
print("Censored rows       :", n_censored_global)
print("Censor limits       :", n_limits_global)
print("Censor-type rows    :", n_censor_types_global)


assert n_censored_global == 84
assert n_limits_global == 84
assert n_censor_types_global == 84


assert (
    mem.loc[
        mem[M_CENSORED],
        M_CENSOR_TYPE
    ]
    .eq("LEFT")
    .all()
)


assert (
    mem.loc[
        mem[M_CENSORED],
        M_LIMIT
    ]
    .notna()
    .all()
)


# ==================================================================================================
# 5. PRE-SPECIFIED PILOTS
# ==================================================================================================

PILOTS = pd.DataFrame(
    [
        {
            "response_key":
                "AC::Na::mg/L",

            "hydro_var":
                "ppt_sum_60d",

            "pilot_role":
                "DIRECT_UNCENSORED_VALIDATION"
        },

        {
            "response_key":
                "AC::SO4_ppm::mg/L",

            "hydro_var":
                "ppt_sum_60d",

            "pilot_role":
                "LEFT_CENSORED_VALIDATION"
        }
    ]
)


pilot_plan = PILOTS.merge(
    plan,
    on=[
        "response_key",
        "hydro_var"
    ],
    how="left",
    validate="one_to_one"
)


assert pilot_plan[
    "joint_analysis_role"
].notna().all()


assert (
    pilot_plan[
        "joint_analysis_role"
    ]
    ==
    "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST"
).all()


print("\nFrozen pilot pairs:")

print(
    pilot_plan[
        [
            "response_key",
            "hydro_var",
            "response_transform",
            "censor_aware_required",
            "pilot_role"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 6. PRIMARY OBSERVED-Q LOOKUP
# ==================================================================================================

q_lookup = master[
    [
        A_FIELD,
        A_Q
    ]
].copy()


q_lookup = q_lookup[
    q_lookup[
        A_FIELD
    ].notna()
].copy()


q_lookup[
    A_Q
] = pd.to_numeric(
    q_lookup[
        A_Q
    ],
    errors="coerce"
)


# Verify that a physical FIELD event never carries conflicting primary Q.

q_conflicts = (
    q_lookup
    .dropna(
        subset=[
            A_Q
        ]
    )
    .groupby(
        A_FIELD
    )[
        A_Q
    ]
    .nunique()
)


assert (
    q_conflicts <= 1
).all(), (
    "Conflicting Q_primary_CQ values within a FIELD observation."
)


q_lookup = (
    q_lookup[
        [
            A_FIELD,
            A_Q
        ]
    ]
    .drop_duplicates()
    .groupby(
        A_FIELD,
        as_index=False
    )[
        A_Q
    ]
    .first()
)


q_lookup = q_lookup.rename(
    columns={
        A_FIELD:
            "field_observation_id_join",

        A_Q:
            "Q_primary"
    }
)


print("\nPrimary-Q lookup:")
print("FIELD IDs:", q_lookup["field_observation_id_join"].nunique())
print("Observed Q:", int(q_lookup["Q_primary"].notna().sum()))


assert (
    q_lookup.loc[
        q_lookup[
            "Q_primary"
        ].notna(),
        "Q_primary"
    ] > 0
).all()


# ==================================================================================================
# 7. BUILD PILOT DATASETS
# ==================================================================================================

pilot_datasets = {}


for _, spec in pilot_plan.iterrows():

    response = spec[
        "response_key"
    ]

    hydro = spec[
        "hydro_var"
    ]

    d = mem[
        mem[
            M_RESPONSE
        ].astype(str)
        .eq(
            response
        )
    ].copy()


    # Original response coverage before Q filtering.
    assert len(d) == 138


    d = d.rename(
        columns={
            M_FIELD:
                "field_observation_id_join"
        }
    )


    d = d.merge(
        q_lookup,
        on="field_observation_id_join",
        how="left",
        validate="many_to_one"
    )


    d[
        "response_numeric"
    ] = pd.to_numeric(
        d[
            M_VALUE
        ],
        errors="coerce"
    )


    d[
        "is_censored_model"
    ] = (
        d[
            M_CENSORED
        ]
        .fillna(False)
        .astype(bool)
    )


    d[
        "censor_limit_numeric"
    ] = pd.to_numeric(
        d[
            M_LIMIT
        ],
        errors="coerce"
    )


    d[
        "H_raw"
    ] = pd.to_numeric(
        d[
            hydro
        ],
        errors="coerce"
    )


    d[
        "Q_primary"
    ] = pd.to_numeric(
        d[
            "Q_primary"
        ],
        errors="coerce"
    )


    # ----------------------------------------------------------------------------------------------
    # PRIMARY ANALYSIS = OBSERVED Q ONLY
    # ----------------------------------------------------------------------------------------------

    d = d[
        d[
            "Q_primary"
        ].notna()
        &
        (
            d[
                "Q_primary"
            ] > 0
        )
        &
        d[
            "H_raw"
        ].notna()
    ].copy()


    # ----------------------------------------------------------------------------------------------
    # CENSORING QA
    # ----------------------------------------------------------------------------------------------

    cens = d[
        "is_censored_model"
    ]

    unc = ~cens


    # Every uncensored observation must have a numeric response.
    assert d.loc[
        unc,
        "response_numeric"
    ].notna().all()


    # Every censored observation must have a positive censor threshold.
    assert d.loc[
        cens,
        "censor_limit_numeric"
    ].notna().all()


    if cens.any():

        assert (
            d.loc[
                cens,
                "censor_limit_numeric"
            ] > 0
        ).all()

        assert (
            d.loc[
                cens,
                M_CENSOR_TYPE
            ].eq(
                "LEFT"
            ).all()
        )


    # Na is frozen as uncensored.
    if response == "AC::Na::mg/L":

        assert int(
            cens.sum()
        ) == 0


    # SO4 frozen primary-Q coverage has 23 censored rows.
    if response == "AC::SO4_ppm::mg/L":

        assert int(
            cens.sum()
        ) == 23


    # ----------------------------------------------------------------------------------------------
    # Q TRANSFORM
    # ----------------------------------------------------------------------------------------------

    d[
        "log10Q"
    ] = np.log10(
        d[
            "Q_primary"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # RESPONSE TRANSFORM — FROZEN FROM BLOCK 8.4B
    # ----------------------------------------------------------------------------------------------

    response_transform = spec[
        "response_transform"
    ]


    if response_transform == "LOG10":

        # Only uncensored observed response values are directly transformed.
        assert (
            d.loc[
                unc,
                "response_numeric"
            ] > 0
        ).all()


        d[
            "Y_model_raw"
        ] = np.nan


        d.loc[
            unc,
            "Y_model_raw"
        ] = np.log10(
            d.loc[
                unc,
                "response_numeric"
            ]
        )


        # Censor threshold must be on the SAME transformed scale.
        d[
            "censor_limit_model_raw"
        ] = np.nan


        d.loc[
            cens,
            "censor_limit_model_raw"
        ] = np.log10(
            d.loc[
                cens,
                "censor_limit_numeric"
            ]
        )


    elif response_transform == "ORIGINAL":

        d[
            "Y_model_raw"
        ] = d[
            "response_numeric"
        ]


        d[
            "censor_limit_model_raw"
        ] = d[
            "censor_limit_numeric"
        ]


    else:

        raise RuntimeError(
            f"Unknown frozen response transform: {response_transform}"
        )


    # ----------------------------------------------------------------------------------------------
    # FINAL MODEL IDENTIFIERS
    # ----------------------------------------------------------------------------------------------

    d[
        "response_key_model"
    ] = response


    d[
        "hydro_var_model"
    ] = hydro


    pilot_datasets[
        response
    ] = d


# ==================================================================================================
# 8. PILOT COVERAGE AUDIT
# ==================================================================================================

coverage_rows = []


for _, spec in pilot_plan.iterrows():

    response = spec[
        "response_key"
    ]

    d = pilot_datasets[
        response
    ]


    coverage_rows.append(
        {
            "response_key":
                response,

            "hydro_var":
                spec[
                    "hydro_var"
                ],

            "n_rows_primary_Q":
                len(d),

            "n_field_events":
                d[
                    "field_observation_id_join"
                ].nunique(),

            "n_stations":
                d[
                    M_STATION
                ].nunique(),

            "n_campaigns":
                d[
                    M_CAMPAIGN
                ].nunique(),

            "n_censored":
                int(
                    d[
                        "is_censored_model"
                    ].sum()
                ),

            "Q_min":
                d[
                    "Q_primary"
                ].min(),

            "Q_median":
                d[
                    "Q_primary"
                ].median(),

            "Q_max":
                d[
                    "Q_primary"
                ].max(),

            "H_min":
                d[
                    "H_raw"
                ].min(),

            "H_median":
                d[
                    "H_raw"
                ].median(),

            "H_max":
                d[
                    "H_raw"
                ].max()
        }
    )


coverage = pd.DataFrame(
    coverage_rows
)


print("\n" + "=" * 120)
print("PILOT COVERAGE")
print("=" * 120)

print(
    coverage.to_string(
        index=False
    )
)


# Frozen Block-9 primary-Q coverage.

expected_n = {
    "AC::Na::mg/L": 115,
    "AC::SO4_ppm::mg/L": 115
}


for response, expected in expected_n.items():

    actual = len(
        pilot_datasets[
            response
        ]
    )

    assert actual == expected, (
        f"{response}: expected {expected} primary-Q rows; found {actual}"
    )


assert int(
    pilot_datasets[
        "AC::SO4_ppm::mg/L"
    ][
        "is_censored_model"
    ].sum()
) == 23


assert (
    coverage[
        "n_campaigns"
    ] == 10
).all()


# ==================================================================================================
# 9. TRAINING-SCALE FUNCTIONS
# ==================================================================================================

def prepare_training_scale(
    df,
    train_mask=None
):

    """
    Scaling architecture
    --------------------
    Y:
        mean/SD estimated from uncensored training observations.

    Q:
        mean/SD of log10(Q) from training observations.

    H:
        mean/SD from training observations.

    In full-data fits, all available rows are training rows.

    In LOCO, held-out campaign NEVER contributes to scaling.
    """

    if train_mask is None:

        train_mask = np.ones(
            len(df),
            dtype=bool
        )


    train_mask = np.asarray(
        train_mask,
        dtype=bool
    )


    uncensored = (
        ~df[
            "is_censored_model"
        ].to_numpy(
            dtype=bool
        )
    )


    y_scale_mask = (
        train_mask
        &
        uncensored
        &
        np.isfinite(
            df[
                "Y_model_raw"
            ].to_numpy(
                dtype=float
            )
        )
    )


    assert int(
        y_scale_mask.sum()
    ) >= 5


    y_mean = float(
        df.loc[
            y_scale_mask,
            "Y_model_raw"
        ].mean()
    )


    y_sd = float(
        df.loc[
            y_scale_mask,
            "Y_model_raw"
        ].std(
            ddof=1
        )
    )


    q_mean = float(
        df.loc[
            train_mask,
            "log10Q"
        ].mean()
    )


    q_sd = float(
        df.loc[
            train_mask,
            "log10Q"
        ].std(
            ddof=1
        )
    )


    h_mean = float(
        df.loc[
            train_mask,
            "H_raw"
        ].mean()
    )


    h_sd = float(
        df.loc[
            train_mask,
            "H_raw"
        ].std(
            ddof=1
        )
    )


    assert np.isfinite(
        y_sd
    ) and y_sd > 0


    assert np.isfinite(
        q_sd
    ) and q_sd > 0


    assert np.isfinite(
        h_sd
    ) and h_sd > 0


    return {
        "y_mean": y_mean,
        "y_sd": y_sd,
        "q_mean": q_mean,
        "q_sd": q_sd,
        "h_mean": h_mean,
        "h_sd": h_sd
    }


def transform_with_scale(
    df,
    scale
):

    out = df.copy()


    out[
        "Q_z"
    ] = (
        out[
            "log10Q"
        ]
        -
        scale[
            "q_mean"
        ]
    ) / scale[
        "q_sd"
    ]


    out[
        "H_z"
    ] = (
        out[
            "H_raw"
        ]
        -
        scale[
            "h_mean"
        ]
    ) / scale[
        "h_sd"
    ]


    out[
        "Y_z"
    ] = (
        out[
            "Y_model_raw"
        ]
        -
        scale[
            "y_mean"
        ]
    ) / scale[
        "y_sd"
    ]


    out[
        "censor_z"
    ] = (
        out[
            "censor_limit_model_raw"
        ]
        -
        scale[
            "y_mean"
        ]
    ) / scale[
        "y_sd"
    ]


    return out


# ==================================================================================================
# 10. BAYESIAN ENGINE
# ==================================================================================================

def fit_joint_model(
    df_train,
    model_type,
    draws=1200,
    tune=1200,
    chains=4,
    target_accept=0.95,
    seed=20260916
):

    """
    model_type
    ----------
    NULL : station only
    Q    : station + log10(Q)
    H    : station + antecedent hydroclimate
    HQ   : station + log10(Q) + antecedent hydroclimate
    """

    assert model_type in {
        "NULL",
        "Q",
        "H",
        "HQ"
    }


    stations = pd.Index(
        sorted(
            df_train[
                M_STATION
            ]
            .astype(str)
            .unique()
        )
    )


    station_map = {
        s: i
        for i, s in enumerate(
            stations
        )
    }


    station_idx = (
        df_train[
            M_STATION
        ]
        .astype(str)
        .map(
            station_map
        )
        .to_numpy(
            dtype=int
        )
    )


    cens = (
        df_train[
            "is_censored_model"
        ]
        .to_numpy(
            dtype=bool
        )
    )


    unc = ~cens


    y_obs = (
        df_train.loc[
            unc,
            "Y_z"
        ]
        .to_numpy(
            dtype=float
        )
    )


    censor_limit = (
        df_train.loc[
            cens,
            "censor_z"
        ]
        .to_numpy(
            dtype=float
        )
    )


    with pm.Model() as model:

        alpha = pm.Normal(
            "alpha",
            mu=0,
            sigma=1
        )


        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1
        )


        z_station = pm.Normal(
            "z_station",
            mu=0,
            sigma=1,
            shape=len(stations)
        )


        station_effect = pm.Deterministic(
            "station_effect",
            z_station
            *
            sigma_station
        )


        sigma = pm.HalfNormal(
            "sigma",
            sigma=1
        )


        mu = (
            alpha
            +
            station_effect[
                station_idx
            ]
        )


        if model_type in {
            "Q",
            "HQ"
        }:

            beta_Q = pm.Normal(
                "beta_Q",
                mu=0,
                sigma=1
            )


            mu = (
                mu
                +
                beta_Q
                *
                df_train[
                    "Q_z"
                ].to_numpy(
                    dtype=float
                )
            )


        if model_type in {
            "H",
            "HQ"
        }:

            beta_H = pm.Normal(
                "beta_H",
                mu=0,
                sigma=1
            )


            mu = (
                mu
                +
                beta_H
                *
                df_train[
                    "H_z"
                ].to_numpy(
                    dtype=float
                )
            )


        # ------------------------------------------------------------------------------------------
        # UNCENSORED LIKELIHOOD
        # ------------------------------------------------------------------------------------------

        if int(
            unc.sum()
        ) > 0:

            pm.Normal(
                "y_obs",
                mu=mu[
                    unc
                ],
                sigma=sigma,
                observed=y_obs
            )


        # ------------------------------------------------------------------------------------------
        # LEFT-CENSOR LIKELIHOOD
        #
        # P(Y <= L | mu, sigma)
        # ------------------------------------------------------------------------------------------

        if int(
            cens.sum()
        ) > 0:

            z = (
                censor_limit
                -
                mu[
                    cens
                ]
            ) / sigma


            logcdf = pm.logcdf(
                pm.Normal.dist(
                    mu=0,
                    sigma=1
                ),
                z
            )


            pm.Potential(
                "left_censor_likelihood",
                pt.sum(
                    logcdf
                )
            )


        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            cores=min(
                chains,
                4
            ),
            target_accept=target_accept,
            random_seed=seed,
            return_inferencedata=True,
            progressbar=True
        )


    return {
        "model": model,
        "idata": idata,
        "stations": stations,
        "station_map": station_map
    }


# ==================================================================================================
# 11. POSTERIOR DIAGNOSTICS
# ==================================================================================================

def posterior_diagnostics(
    fit,
    model_type
):

    vars_keep = [
        "alpha",
        "sigma",
        "sigma_station"
    ]


    if model_type in {
        "Q",
        "HQ"
    }:

        vars_keep.append(
            "beta_Q"
        )


    if model_type in {
        "H",
        "HQ"
    }:

        vars_keep.append(
            "beta_H"
        )


    summ = az.summary(
        fit[
            "idata"
        ],
        var_names=vars_keep,
        hdi_prob=0.95
    )


    divergences = int(
        fit[
            "idata"
        ]
        .sample_stats[
            "diverging"
        ]
        .sum()
        .values
    )


    max_rhat = float(
        summ[
            "r_hat"
        ].max()
    )


    min_ess = float(
        summ[
            "ess_bulk"
        ].min()
    )


    return {
        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess,

        "summary":
            summ
    }


def posterior_direction_probability(
    fit,
    variable
):

    x = (
        fit[
            "idata"
        ]
        .posterior[
            variable
        ]
        .values
        .reshape(-1)
    )


    p_pos = float(
        np.mean(
            x > 0
        )
    )


    p_neg = float(
        np.mean(
            x < 0
        )
    )


    if p_pos >= p_neg:

        return (
            p_pos,
            "POSITIVE"
        )

    else:

        return (
            p_neg,
            "NEGATIVE"
        )


# ==================================================================================================
# 12. FULL-DATA PILOT FITS
# ==================================================================================================

full_fit_rows = []

full_fits = {}


for pilot_number, (_, spec) in enumerate(
    pilot_plan.iterrows(),
    start=1
):

    response = spec[
        "response_key"
    ]


    d0 = pilot_datasets[
        response
    ].copy()


    scale = prepare_training_scale(
        d0
    )


    d = transform_with_scale(
        d0,
        scale
    )


    full_fits[
        response
    ] = {}


    print("\n" + "=" * 120)

    print(
        f"FULL-DATA PILOT {pilot_number}: "
        f"{response} + {spec['hydro_var']}"
    )

    print("=" * 120)


    for j, model_type in enumerate(
        [
            "NULL",
            "Q",
            "H",
            "HQ"
        ]
    ):

        print(
            f"\nFitting {model_type} ..."
        )


        fit = fit_joint_model(
            d,
            model_type=model_type,
            draws=1200,
            tune=1200,
            chains=4,
            target_accept=0.95,
            seed=(
                20260916
                +
                pilot_number * 100
                +
                j
            )
        )


        full_fits[
            response
        ][
            model_type
        ] = fit


        diag = posterior_diagnostics(
            fit,
            model_type
        )


        row = {
            "response_key":
                response,

            "hydro_var":
                spec[
                    "hydro_var"
                ],

            "model_type":
                model_type,

            "n":
                len(d),

            "n_censored":
                int(
                    d[
                        "is_censored_model"
                    ].sum()
                ),

            "n_stations":
                d[
                    M_STATION
                ].nunique(),

            "n_campaigns":
                d[
                    M_CAMPAIGN
                ].nunique(),

            "divergences":
                diag[
                    "divergences"
                ],

            "max_rhat":
                diag[
                    "max_rhat"
                ],

            "min_ess_bulk":
                diag[
                    "min_ess_bulk"
                ]
        }


        if model_type in {
            "Q",
            "HQ"
        }:

            bq = (
                fit[
                    "idata"
                ]
                .posterior[
                    "beta_Q"
                ]
                .values
                .reshape(-1)
            )


            row[
                "beta_Q_median_std"
            ] = float(
                np.median(
                    bq
                )
            )


        if model_type in {
            "H",
            "HQ"
        }:

            bh = (
                fit[
                    "idata"
                ]
                .posterior[
                    "beta_H"
                ]
                .values
                .reshape(-1)
            )


            hdi = np.asarray(
                az.hdi(
                    bh,
                    hdi_prob=0.95
                )
            )


            pdir, direction = (
                posterior_direction_probability(
                    fit,
                    "beta_H"
                )
            )


            row[
                "beta_H_median_std"
            ] = float(
                np.median(
                    bh
                )
            )


            row[
                "beta_H_hdi_low_std"
            ] = float(
                hdi[0]
            )


            row[
                "beta_H_hdi_high_std"
            ] = float(
                hdi[1]
            )


            row[
                "beta_H_Pdirection"
            ] = float(
                pdir
            )


            row[
                "beta_H_direction"
            ] = direction


            # Back-transform conditional H slope:
            # transformed-response units per raw H unit.

            row[
                "beta_H_raw_scale"
            ] = float(
                np.median(
                    bh
                )
                *
                scale[
                    "y_sd"
                ]
                /
                scale[
                    "h_sd"
                ]
            )


        full_fit_rows.append(
            row
        )


        print(
            diag[
                "summary"
            ]
        )


        print(
            "Divergences:",
            diag[
                "divergences"
            ]
        )


        print(
            "Max Rhat:",
            diag[
                "max_rhat"
            ]
        )


        print(
            "Min ESS bulk:",
            diag[
                "min_ess_bulk"
            ]
        )


full_fit_summary = pd.DataFrame(
    full_fit_rows
)


# ==================================================================================================
# 13. PREDICTIVE SCORING HELPERS
# ==================================================================================================

def normal_logpdf(
    y,
    mu,
    sigma
):

    return (
        -0.5
        *
        np.log(
            2.0
            *
            np.pi
        )
        -
        np.log(
            sigma
        )
        -
        0.5
        *
        (
            (
                y
                -
                mu
            )
            /
            sigma
        ) ** 2
    )


def logmeanexp(
    x,
    axis=0
):

    x = np.asarray(
        x
    )


    m = np.max(
        x,
        axis=axis,
        keepdims=True
    )


    result = (
        m
        +
        np.log(
            np.mean(
                np.exp(
                    x
                    -
                    m
                ),
                axis=axis,
                keepdims=True
            )
        )
    )


    return np.squeeze(
        result,
        axis=axis
    )


def heldout_lpd(
    fit,
    test_df,
    model_type
):

    """
    Posterior predictive log density.

    Known station:
        posterior station effect used.

    Unseen held-out station:
        random intercept marginalized by
        sqrt(sigma^2 + sigma_station^2).

    Censored held-out observation:
        Normal CDF probability scored.
    """

    post = fit[
        "idata"
    ].posterior


    alpha = (
        post[
            "alpha"
        ]
        .values
        .reshape(-1)
    )


    sigma = (
        post[
            "sigma"
        ]
        .values
        .reshape(-1)
    )


    sigma_station = (
        post[
            "sigma_station"
        ]
        .values
        .reshape(-1)
    )


    station_effect = (
        post[
            "station_effect"
        ]
        .values
        .reshape(
            -1,
            post[
                "station_effect"
            ].shape[-1]
        )
    )


    if model_type in {
        "Q",
        "HQ"
    }:

        beta_Q = (
            post[
                "beta_Q"
            ]
            .values
            .reshape(-1)
        )


    if model_type in {
        "H",
        "HQ"
    }:

        beta_H = (
            post[
                "beta_H"
            ]
            .values
            .reshape(-1)
        )


    train_station_map = {
        str(s): i
        for i, s in enumerate(
            fit[
                "stations"
            ]
        )
    }


    point_lpd = []


    for _, row in test_df.iterrows():

        station = str(
            row[
                M_STATION
            ]
        )


        mu = alpha.copy()


        if station in train_station_map:

            idx = train_station_map[
                station
            ]


            mu = (
                mu
                +
                station_effect[
                    :,
                    idx
                ]
            )


            sigma_pred = sigma


        else:

            # Marginalize unknown station-level random effect.

            sigma_pred = np.sqrt(
                sigma ** 2
                +
                sigma_station ** 2
            )


        if model_type in {
            "Q",
            "HQ"
        }:

            mu = (
                mu
                +
                beta_Q
                *
                float(
                    row[
                        "Q_z"
                    ]
                )
            )


        if model_type in {
            "H",
            "HQ"
        }:

            mu = (
                mu
                +
                beta_H
                *
                float(
                    row[
                        "H_z"
                    ]
                )
            )


        if bool(
            row[
                "is_censored_model"
            ]
        ):

            z = (
                float(
                    row[
                        "censor_z"
                    ]
                )
                -
                mu
            ) / sigma_pred


            probs = np.clip(
                ndtr(
                    z
                ),
                1e-300,
                1.0
            )


            draw_loglik = np.log(
                probs
            )


        else:

            y = float(
                row[
                    "Y_z"
                ]
            )


            draw_loglik = normal_logpdf(
                y,
                mu,
                sigma_pred
            )


        point_lpd.append(
            float(
                logmeanexp(
                    draw_loglik
                )
            )
        )


    return np.asarray(
        point_lpd,
        dtype=float
    )


# ==================================================================================================
# 14. PRE-SPECIFIED PILOT LOCO — C10
# ==================================================================================================

PILOT_HELDOUT_CAMPAIGN = "C10"

loco_rows = []


for pilot_number, (_, spec) in enumerate(
    pilot_plan.iterrows(),
    start=1
):

    response = spec[
        "response_key"
    ]


    d0 = pilot_datasets[
        response
    ].copy()


    campaigns = set(
        d0[
            M_CAMPAIGN
        ].astype(str)
    )


    assert (
        PILOT_HELDOUT_CAMPAIGN
        in campaigns
    )


    train_mask = (
        d0[
            M_CAMPAIGN
        ]
        .astype(str)
        .ne(
            PILOT_HELDOUT_CAMPAIGN
        )
        .to_numpy()
    )


    test_mask = ~train_mask


    # TRAIN-ONLY SCALING

    scale = prepare_training_scale(
        d0,
        train_mask=train_mask
    )


    d = transform_with_scale(
        d0,
        scale
    )


    train = d.loc[
        train_mask
    ].copy()


    test = d.loc[
        test_mask
    ].copy()


    assert len(
        train
    ) > 0


    assert len(
        test
    ) > 0


    print("\n" + "=" * 120)

    print(
        f"PILOT LOCO — {response} — "
        f"HELD OUT {PILOT_HELDOUT_CAMPAIGN}"
    )

    print("=" * 120)


    print(
        "Train:",
        len(train),
        "| Test:",
        len(test),
        "| Test censored:",
        int(
            test[
                "is_censored_model"
            ].sum()
        )
    )


    for j, model_type in enumerate(
        [
            "NULL",
            "Q",
            "H",
            "HQ"
        ]
    ):

        print(
            f"\nFitting LOCO {model_type} ..."
        )


        fit = fit_joint_model(
            train,
            model_type=model_type,
            draws=1000,
            tune=1000,
            chains=4,
            target_accept=0.95,
            seed=(
                20261916
                +
                pilot_number * 100
                +
                j
            )
        )


        diag = posterior_diagnostics(
            fit,
            model_type
        )


        lpd = heldout_lpd(
            fit,
            test,
            model_type
        )


        loco_rows.append(
            {
                "response_key":
                    response,

                "hydro_var":
                    spec[
                        "hydro_var"
                    ],

                "heldout_campaign":
                    PILOT_HELDOUT_CAMPAIGN,

                "model_type":
                    model_type,

                "n_train":
                    len(train),

                "n_test":
                    len(test),

                "n_test_censored":
                    int(
                        test[
                            "is_censored_model"
                        ].sum()
                    ),

                "mean_test_LPD":
                    float(
                        np.mean(
                            lpd
                        )
                    ),

                "sum_test_LPD":
                    float(
                        np.sum(
                            lpd
                        )
                    ),

                "divergences":
                    diag[
                        "divergences"
                    ],

                "max_rhat":
                    diag[
                        "max_rhat"
                    ],

                "min_ess_bulk":
                    diag[
                        "min_ess_bulk"
                    ]
            }
        )


pilot_loco = pd.DataFrame(
    loco_rows
)


# ==================================================================================================
# 15. PAIRED PILOT CONTRASTS
# ==================================================================================================

contrast_rows = []


for response in PILOTS[
    "response_key"
]:

    sub = (
        pilot_loco[
            pilot_loco[
                "response_key"
            ].eq(
                response
            )
        ]
        .set_index(
            "model_type"
        )
    )


    assert set(
        sub.index
    ) == {
        "NULL",
        "Q",
        "H",
        "HQ"
    }


    contrast_rows.append(
        {
            "response_key":
                response,

            "heldout_campaign":
                PILOT_HELDOUT_CAMPAIGN,

            "delta_LPD_Q_minus_NULL":
                float(
                    sub.loc[
                        "Q",
                        "mean_test_LPD"
                    ]
                    -
                    sub.loc[
                        "NULL",
                        "mean_test_LPD"
                    ]
                ),

            "delta_LPD_H_minus_NULL":
                float(
                    sub.loc[
                        "H",
                        "mean_test_LPD"
                    ]
                    -
                    sub.loc[
                        "NULL",
                        "mean_test_LPD"
                    ]
                ),

            "delta_LPD_HQ_minus_Q":
                float(
                    sub.loc[
                        "HQ",
                        "mean_test_LPD"
                    ]
                    -
                    sub.loc[
                        "Q",
                        "mean_test_LPD"
                    ]
                ),

            "delta_LPD_HQ_minus_H":
                float(
                    sub.loc[
                        "HQ",
                        "mean_test_LPD"
                    ]
                    -
                    sub.loc[
                        "H",
                        "mean_test_LPD"
                    ]
                )
        }
    )


pilot_contrasts = pd.DataFrame(
    contrast_rows
)


# ==================================================================================================
# 16. COMPUTATIONAL QA
# ==================================================================================================

# IMPORTANT:
# Positive H+Q predictive improvement is NOT required here.
# A null/negative scientific result must not fail engine validation.

full_diag_pass = bool(
    (
        (
            full_fit_summary[
                "divergences"
            ] == 0
        )
        &
        (
            full_fit_summary[
                "max_rhat"
            ] <= 1.01
        )
        &
        (
            full_fit_summary[
                "min_ess_bulk"
            ] >= 400
        )
    ).all()
)


loco_diag_pass = bool(
    (
        (
            pilot_loco[
                "divergences"
            ] == 0
        )
        &
        (
            pilot_loco[
                "max_rhat"
            ] <= 1.01
        )
        &
        (
            pilot_loco[
                "min_ess_bulk"
            ] >= 400
        )
        &
        np.isfinite(
            pilot_loco[
                "mean_test_LPD"
            ]
        )
    ).all()
)


qa = pd.DataFrame(
    [
        [
            "exact_memory_schema_resolved",
            True
        ],

        [
            "global_censored_rows_84",
            n_censored_global == 84
        ],

        [
            "global_censor_limits_84",
            n_limits_global == 84
        ],

        [
            "all_censoring_left",
            bool(
                mem.loc[
                    mem[
                        M_CENSORED
                    ],
                    M_CENSOR_TYPE
                ]
                .eq(
                    "LEFT"
                )
                .all()
            )
        ],

        [
            "Na_primary_Q_rows_115",
            len(
                pilot_datasets[
                    "AC::Na::mg/L"
                ]
            ) == 115
        ],

        [
            "Na_censored_rows_0",
            int(
                pilot_datasets[
                    "AC::Na::mg/L"
                ][
                    "is_censored_model"
                ].sum()
            ) == 0
        ],

        [
            "SO4_primary_Q_rows_115",
            len(
                pilot_datasets[
                    "AC::SO4_ppm::mg/L"
                ]
            ) == 115
        ],

        [
            "SO4_censored_rows_23",
            int(
                pilot_datasets[
                    "AC::SO4_ppm::mg/L"
                ][
                    "is_censored_model"
                ].sum()
            ) == 23
        ],

        [
            "only_frozen_P60_used",
            set(
                PILOTS[
                    "hydro_var"
                ]
            )
            ==
            {
                "ppt_sum_60d"
            }
        ],

        [
            "four_models_per_pilot_full_data",
            len(
                full_fit_summary
            ) == 8
        ],

        [
            "four_models_per_pilot_LOCO",
            len(
                pilot_loco
            ) == 8
        ],

        [
            "full_model_diagnostics_pass",
            full_diag_pass
        ],

        [
            "pilot_LOCO_diagnostics_pass",
            loco_diag_pass
        ],

        [
            "heldout_LPD_finite",
            bool(
                np.isfinite(
                    pilot_loco[
                        "mean_test_LPD"
                    ]
                ).all()
            )
        ],

        [
            "same_rows_all_comparators",
            True
        ],

        [
            "train_only_LOCO_scaling",
            True
        ],

        [
            "explicit_sample_specific_censor_likelihood",
            True
        ],

        [
            "unseen_station_marginalization",
            True
        ],

        [
            "observed_Q_only",
            True
        ],

        [
            "no_window_search",
            True
        ],

        [
            "no_HxQ_interaction",
            True
        ],

        [
            "no_C09_reconstructed_Q",
            True
        ],

        [
            "no_final_support_classification",
            True
        ]
    ],
    columns=[
        "check",
        "pass"
    ]
)


BLOCK_PASS = bool(
    qa[
        "pass"
    ].all()
)


# ==================================================================================================
# 17. SAVE OUTPUTS
# ==================================================================================================

COVERAGE_OUT = (
    OUT_DIR /
    "BLOCK101A_R_PILOT_DATA_COVERAGE_v1.csv"
)

FULL_OUT = (
    OUT_DIR /
    "BLOCK101A_R_FULLDATA_PILOT_MODEL_DIAGNOSTICS_v1.csv"
)

LOCO_OUT = (
    OUT_DIR /
    "BLOCK101A_R_PILOT_LOCO_MODEL_DIAGNOSTICS_v1.csv"
)

CONTRAST_OUT = (
    OUT_DIR /
    "BLOCK101A_R_PILOT_LOCO_CONTRASTS_v1.csv"
)

QA_OUT = (
    OUT_DIR /
    "BLOCK101A_R_ENGINE_VALIDATION_QA_v1.csv"
)


coverage.to_csv(
    COVERAGE_OUT,
    index=False
)

full_fit_summary.to_csv(
    FULL_OUT,
    index=False
)

pilot_loco.to_csv(
    LOCO_OUT,
    index=False
)

pilot_contrasts.to_csv(
    CONTRAST_OUT,
    index=False
)

qa.to_csv(
    QA_OUT,
    index=False
)


# ==================================================================================================
# 18. MANIFEST
# ==================================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


manifest = {
    "block":
        "10.1A-R",

    "status":
        (
            "PASS"
            if BLOCK_PASS
            else "FAIL"
        ),

    "purpose":
        "schema-resolved joint H+Q engine validation",

    "schema_source":
        "BLOCK 10.1A-R0 reconnaissance",

    "response_numeric_column":
        M_VALUE,

    "response_censored_column":
        M_CENSORED,

    "response_censor_type_column":
        M_CENSOR_TYPE,

    "response_censor_limit_column":
        M_LIMIT,

    "pilots": [
        {
            "response":
                "AC::Na::mg/L",

            "hydro":
                "ppt_sum_60d",

            "role":
                "direct uncensored validation"
        },

        {
            "response":
                "AC::SO4_ppm::mg/L",

            "hydro":
                "ppt_sum_60d",

            "role":
                "left-censored validation"
        }
    ],

    "pilot_LOCO_campaign":
        PILOT_HELDOUT_CAMPAIGN,

    "primary_Q":
        "observed_only",

    "final_support_inference_performed":
        False,

    "window_search_performed":
        False,

    "outputs": {
        str(COVERAGE_OUT):
            sha256_file(
                COVERAGE_OUT
            ),

        str(FULL_OUT):
            sha256_file(
                FULL_OUT
            ),

        str(LOCO_OUT):
            sha256_file(
                LOCO_OUT
            ),

        str(CONTRAST_OUT):
            sha256_file(
                CONTRAST_OUT
            ),

        str(QA_OUT):
            sha256_file(
                QA_OUT
            )
    }
}


MANIFEST_OUT = (
    OUT_DIR /
    "BLOCK101A_R_ENGINE_VALIDATION_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 19. PRINT FINAL RESULTS
# ==================================================================================================

print("\n" + "=" * 120)
print("PILOT COVERAGE")
print("=" * 120)

print(
    coverage.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("FULL-DATA PILOT MODEL DIAGNOSTICS")
print("=" * 120)

print(
    full_fit_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("PILOT C10 LOCO MODEL DIAGNOSTICS")
print("=" * 120)

print(
    pilot_loco.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("PILOT C10 PAIRED PREDICTIVE CONTRASTS")
print("=" * 120)

print(
    pilot_contrasts.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("ENGINE QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("BLOCK 10.1A-R FINAL GATE")
print("=" * 120)


print(
    "BLOCK 10.1A-R PASS                 :",
    BLOCK_PASS
)

print(
    "FULL-DATA PILOT FITS               :",
    len(
        full_fit_summary
    )
)

print(
    "PILOT LOCO FITS                    :",
    len(
        pilot_loco
    )
)

print(
    "SCHEMA RESOLVED                    :",
    True
)

print(
    "PRIMARY Q                          :",
    "OBSERVED ONLY"
)

print(
    "FINAL SCIENTIFIC CLASSIFICATION    :",
    False
)

print(
    "WINDOW SEARCH PERFORMED            :",
    False
)

print(
    "C09 RECONSTRUCTED Q USED           :",
    False
)

print(
    "READY FOR FULL BLOCK 10.2          :",
    BLOCK_PASS
)


print("\nFiles written:")

for p in [
    COVERAGE_OUT,
    FULL_OUT,
    LOCO_OUT,
    CONTRAST_OUT,
    QA_OUT,
    MANIFEST_OUT
]:

    print(
        " ",
        p
    )


assert BLOCK_PASS, (
    "\nBLOCK 10.1A-R computational validation FAILED.\n"
    "DO NOT proceed to Block 10.2.\n"
    "Review diagnostics before any additional model fitting."
)


print("\n" + "=" * 120)
print("BLOCK 10.1A-R PASS — ENGINE VALIDATED")
print("=" * 120)

print(
    """
If this gate is PASS:

DO NOT rerun Block 8.
DO NOT rerun Block 9.
DO NOT modify Block 10.0B.
DO NOT select a different hydroclimate window.

Return the following outputs for review:

1. PILOT COVERAGE
2. FULL-DATA PILOT MODEL DIAGNOSTICS
3. PILOT C10 LOCO MODEL DIAGNOSTICS
4. PILOT C10 PAIRED PREDICTIVE CONTRASTS
5. ENGINE QA
6. BLOCK 10.1A-R FINAL GATE

Only after reviewing these results should the full Block 10.2 analysis be launched.
"""
)

---
### Imported master block 131

`BLOCK 10.2A — FULL-DATA CONDITIONAL H|Q POSTERIOR SPECTRUM`


In [ ]:
# ==================================================================================================
# BLOCK 10.2A — FULL-DATA CONDITIONAL H|Q POSTERIOR SPECTRUM
#
# PURPOSE
#   Fit the five PRE-SPECIFIED additive H+Q models frozen in Block 10.0B.
#
#   Y* = alpha + beta_Q * z(log10 Q) + beta_H * z(H) + station random intercept + error
#
# IMPORTANT
#   - Primary observed Q only
#   - Only the 5 frozen H windows
#   - No window search
#   - No HxQ interaction
#   - Sample-specific left-censor likelihood
#   - Frozen response transformations
#   - Distinct analytical subsamples retained
#   - NO final joint-support classification here
#   - Predictive gate remains deferred to Block 10.2B LOCO
#
# OUTPUT
#   Full-data posterior spectrum of beta_H conditional on Q.
# ==================================================================================================

import json
import hashlib
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import arviz as az

warnings.filterwarnings("ignore")


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

PROJECT = Path("/content/drive/MyDrive/Huanta_JoH_PROJECT")

ARCH_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "ARCHITECTURE_FREEZE"
)

ENGINE_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "ENGINE_VALIDATION"
)

OUT_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULLDATA_POSTERIOR"
)

TRACE_DIR = OUT_DIR / "TRACES"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)


PLAN_PATH = (
    ARCH_DIR /
    "BLOCK100B_JOINT_HQ_ANALYSIS_PLAN_v1.csv"
)

ENGINE_QA_PATH = (
    ENGINE_DIR /
    "BLOCK101A_R4_FINAL_ENGINE_QA_v1.csv"
)

MEMORY_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

MASTER_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MASTER_ANALYTICAL_v1_1.csv"
)


for p in [
    PLAN_PATH,
    ENGINE_QA_PATH,
    MEMORY_PATH,
    MASTER_PATH
]:
    assert p.exists(), p


print("=" * 120)
print("BLOCK 10.2A — FULL-DATA CONDITIONAL H|Q POSTERIOR SPECTRUM")
print("=" * 120)

print("\nFrozen inputs:")
print("Plan   :", PLAN_PATH)
print("Engine :", ENGINE_QA_PATH)
print("Memory :", MEMORY_PATH)
print("Master :", MASTER_PATH)


# ==================================================================================================
# 1. VERIFY ENGINE VALIDATION
# ==================================================================================================

engine_qa = pd.read_csv(
    ENGINE_QA_PATH,
    keep_default_na=False
)

assert len(engine_qa) > 0

assert engine_qa["pass"].astype(str).str.lower().eq("true").all(), (
    "Block 10.1 engine validation is not fully PASS."
)

print("\nBlock 10.1 engine validation: PASS")


# ==================================================================================================
# 2. LOAD FROZEN PLAN
# ==================================================================================================

plan = pd.read_csv(
    PLAN_PATH,
    keep_default_na=False
)

print("\nFrozen plan shape:", plan.shape)

required_plan_cols = [
    "response_key",
    "hydro_family",
    "window_days",
    "hydro_var",
    "effect_direction",
    "joint_analysis_role",
    "response_transform",
    "censor_aware_required"
]

for c in required_plan_cols:
    assert c in plan.columns, c


assert len(plan) == 5
assert plan["response_key"].nunique() == 4

assert set(plan["hydro_var"]) == {
    "ppt_sum_14d",
    "ppt_sum_60d",
    "sm_rootzone_mean_90d"
}


print("\nFrozen five H+Q pairs:")

print(
    plan[
        required_plan_cols
    ].to_string(index=False)
)


# ==================================================================================================
# 3. EXACT FROZEN EXPECTATION
# ==================================================================================================

expected_pairs = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"),
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"),
    ("AC::Na::mg/L", "ppt_sum_60d"),
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"),
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d")
}

actual_pairs = set(
    zip(
        plan["response_key"],
        plan["hydro_var"]
    )
)

assert actual_pairs == expected_pairs


# ==================================================================================================
# 4. LOAD AUTHORITATIVE DATA
# ==================================================================================================

mem = pd.read_csv(
    MEMORY_PATH,
    low_memory=False
)

master = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

print("\nMemory shape:", mem.shape)
print("Master shape:", master.shape)


# ==================================================================================================
# 5. EXACT SCHEMA
# ==================================================================================================

M_RESPONSE = "response_key"
M_VALUE = "response_value_numeric"
M_CENS = "response_censored"
M_CTYPE = "response_censor_type"
M_CLIMIT = "response_censor_limit"
M_FIELD = "field_observation_id"
M_STATION = "station_id"
M_CAMPAIGN = "campaign_id"

Q_FIELD = "field_observation_id"
Q_PRIMARY = "Q_primary_CQ"


required_mem_cols = [
    M_RESPONSE,
    M_VALUE,
    M_CENS,
    M_CTYPE,
    M_CLIMIT,
    M_FIELD,
    M_STATION,
    M_CAMPAIGN,
    "ppt_sum_14d",
    "ppt_sum_60d",
    "sm_rootzone_mean_90d"
]

for c in required_mem_cols:
    assert c in mem.columns, c

for c in [Q_FIELD, Q_PRIMARY]:
    assert c in master.columns, c


# ==================================================================================================
# 6. PRIMARY OBSERVED-Q LOOKUP
# ==================================================================================================

q_lookup = (
    master[
        [Q_FIELD, Q_PRIMARY]
    ]
    .dropna(subset=[Q_FIELD])
    .copy()
)

# There can be many analytical rows per FIELD observation in master.
# Q must be physically unique within FIELD event.

q_nunique = (
    q_lookup
    .groupby(Q_FIELD)[Q_PRIMARY]
    .nunique(dropna=True)
)

assert (q_nunique <= 1).all()

q_lookup = (
    q_lookup
    .drop_duplicates(
        subset=[Q_FIELD, Q_PRIMARY]
    )
    .drop_duplicates(
        subset=[Q_FIELD],
        keep="first"
    )
)

assert q_lookup[Q_FIELD].is_unique


print("\nPrimary Q lookup:")
print("FIELD IDs :", q_lookup[Q_FIELD].nunique())
print("Observed Q:", q_lookup[Q_PRIMARY].notna().sum())


# ==================================================================================================
# 7. RESPONSE TRANSFORM FUNCTION
# ==================================================================================================

def apply_response_transform(df, transform):

    out = df.copy()

    y = pd.to_numeric(
        out[M_VALUE],
        errors="coerce"
    )

    censor = (
        out[M_CENS]
        .fillna(False)
        .astype(bool)
    )

    limit = pd.to_numeric(
        out[M_CLIMIT],
        errors="coerce"
    )

    transform = str(transform).upper()

    if transform == "ORIGINAL":

        out["y_model"] = y

        out["censor_limit_model"] = limit

    elif transform == "LOG10":

        # Detected observations require positive Y.
        bad_detected = (
            (~censor)
            &
            (
                y.isna()
                |
                (y <= 0)
            )
        )

        assert not bad_detected.any(), (
            "Non-positive detected value under LOG10 transform."
        )

        # Censored observations require a positive censor limit.
        bad_censored = (
            censor
            &
            (
                limit.isna()
                |
                (limit <= 0)
            )
        )

        assert not bad_censored.any(), (
            "Invalid censor limit under LOG10 transform."
        )

        out["y_model"] = np.where(
            censor,
            np.nan,
            np.log10(y)
        )

        out["censor_limit_model"] = np.where(
            censor,
            np.log10(limit),
            np.nan
        )

    else:

        raise RuntimeError(
            f"Unsupported frozen response transform: {transform}"
        )

    out["is_censored_model"] = censor

    return out


# ==================================================================================================
# 8. BUILD EXACT FIVE FULL-DATA DATASETS
# ==================================================================================================

datasets = {}
coverage_rows = []


for _, r in plan.iterrows():

    response = r["response_key"]
    hydro_var = r["hydro_var"]
    transform = r["response_transform"]

    key = (response, hydro_var)

    d = mem[
        mem[M_RESPONSE].eq(response)
    ].copy()

    assert len(d) > 0

    d = d.merge(
        q_lookup,
        on=M_FIELD,
        how="left",
        validate="many_to_one"
    )

    # Primary analysis = observed Q only.
    d = d[
        d[Q_PRIMARY].notna()
    ].copy()

    d[Q_PRIMARY] = pd.to_numeric(
        d[Q_PRIMARY],
        errors="coerce"
    )

    d[hydro_var] = pd.to_numeric(
        d[hydro_var],
        errors="coerce"
    )

    assert (d[Q_PRIMARY] > 0).all()

    # H must be available.
    d = d[
        d[hydro_var].notna()
    ].copy()

    d = apply_response_transform(
        d,
        transform
    )

    # Keep rows with either:
    #  - detected Y
    #  - valid censor limit
    usable = (
        (
            ~d["is_censored_model"]
            &
            d["y_model"].notna()
        )
        |
        (
            d["is_censored_model"]
            &
            d["censor_limit_model"].notna()
        )
    )

    d = d[
        usable
    ].copy()

    # Model predictors.
    d["log10_Q"] = np.log10(
        d[Q_PRIMARY].astype(float)
    )

    d["H_raw"] = d[hydro_var].astype(float)

    # Full-data descriptive standardization.
    # LOCO later MUST use training-only scaling.
    q_mean = d["log10_Q"].mean()
    q_sd = d["log10_Q"].std(ddof=1)

    h_mean = d["H_raw"].mean()
    h_sd = d["H_raw"].std(ddof=1)

    assert np.isfinite(q_sd) and q_sd > 0
    assert np.isfinite(h_sd) and h_sd > 0

    d["Q_z"] = (
        d["log10_Q"] - q_mean
    ) / q_sd

    d["H_z"] = (
        d["H_raw"] - h_mean
    ) / h_sd

    # Y scaling must use the same conceptual approach as validated engine.
    # For censored rows, scale from detected observations.
    detected_y = d.loc[
        ~d["is_censored_model"],
        "y_model"
    ].astype(float)

    assert len(detected_y) > 2

    y_mean = detected_y.mean()
    y_sd = detected_y.std(ddof=1)

    assert np.isfinite(y_sd) and y_sd > 0

    d["y_z"] = (
        d["y_model"] - y_mean
    ) / y_sd

    d["censor_limit_z"] = (
        d["censor_limit_model"] - y_mean
    ) / y_sd

    # Freeze scaling metadata.
    d.attrs["q_mean"] = float(q_mean)
    d.attrs["q_sd"] = float(q_sd)
    d.attrs["h_mean"] = float(h_mean)
    d.attrs["h_sd"] = float(h_sd)
    d.attrs["y_mean"] = float(y_mean)
    d.attrs["y_sd"] = float(y_sd)

    datasets[key] = d

    coverage_rows.append(
        {
            "response_key":
                response,

            "hydro_var":
                hydro_var,

            "hydro_family":
                r["hydro_family"],

            "window_days":
                r["window_days"],

            "joint_analysis_role":
                r["joint_analysis_role"],

            "response_transform":
                transform,

            "n_rows":
                len(d),

            "n_field_events":
                d[M_FIELD].nunique(),

            "n_stations":
                d[M_STATION].nunique(),

            "n_campaigns":
                d[M_CAMPAIGN].nunique(),

            "n_censored":
                int(
                    d["is_censored_model"].sum()
                ),

            "Q_min":
                d[Q_PRIMARY].min(),

            "Q_median":
                d[Q_PRIMARY].median(),

            "Q_max":
                d[Q_PRIMARY].max(),

            "H_min":
                d["H_raw"].min(),

            "H_median":
                d["H_raw"].median(),

            "H_max":
                d["H_raw"].max(),

            "Q_H_raw_spearman":
                d[
                    ["log10_Q", "H_raw"]
                ]
                .corr(method="spearman")
                .iloc[0, 1]
        }
    )


coverage = pd.DataFrame(
    coverage_rows
)


print("\n" + "=" * 120)
print("FULL-DATA H+Q COVERAGE")
print("=" * 120)

print(
    coverage.to_string(
        index=False
    )
)


# ==================================================================================================
# 9. COVERAGE EXPECTATIONS
# ==================================================================================================

# EC / Na / SO4 should reproduce Block-9 primary observed-Q coverage.
# Temperature should reproduce its own physical-response coverage.

expected_n = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"): 115,
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"): 115,
    ("AC::Na::mg/L", "ppt_sum_60d"): 115,
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"): 115,
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d"): 111
}

expected_censored = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"): 0,
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"): 0,
    ("AC::Na::mg/L", "ppt_sum_60d"): 0,
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"): 23,
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d"): 0
}


for key, d in datasets.items():

    assert len(d) == expected_n[key], (
        f"{key}: expected {expected_n[key]} rows, got {len(d)}"
    )

    assert int(
        d["is_censored_model"].sum()
    ) == expected_censored[key]

    assert d[M_CAMPAIGN].nunique() == 10


# ==================================================================================================
# 10. ENGINE FUNCTIONS MUST EXIST FROM VALIDATED 10.1 SESSION
# ==================================================================================================

assert "fit_joint_model" in globals(), (
    "fit_joint_model not found in runtime. "
    "Do NOT improvise a new engine. Reload/re-execute the validated "
    "Block 10.1 engine-definition cells only."
)

assert "posterior_diagnostics" in globals(), (
    "posterior_diagnostics not found."
)


# ==================================================================================================
# 11. FIT FIVE PRE-SPECIFIED FULL-DATA H+Q MODELS
# ==================================================================================================

result_rows = []
trace_paths = []


for i, r in plan.reset_index(drop=True).iterrows():

    response = r["response_key"]
    hydro_var = r["hydro_var"]
    key = (response, hydro_var)

    d = datasets[key]

    print("\n" + "=" * 120)
    print(
        f"FULL H+Q MODEL {i+1}/5 — {response} + {hydro_var}"
    )
    print("=" * 120)

    print(
        "Rows:",
        len(d),
        "| censored:",
        int(d["is_censored_model"].sum()),
        "| stations:",
        d[M_STATION].nunique(),
        "| campaigns:",
        d[M_CAMPAIGN].nunique()
    )

    # ----------------------------------------------------------------------------------------------
    # Fit only H+Q.
    # ----------------------------------------------------------------------------------------------

    fit = fit_joint_model(
        d,
        model_type="HQ",
        draws=2000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        seed=20261020 + i
    )

    diag = posterior_diagnostics(
        fit,
        "HQ"
    )

    # ----------------------------------------------------------------------------------------------
    # Extract posterior beta_Q and beta_H.
    # ----------------------------------------------------------------------------------------------

    beta_q = (
        fit.posterior["beta_Q"]
        .values
        .reshape(-1)
    )

    beta_h = (
        fit.posterior["beta_H"]
        .values
        .reshape(-1)
    )

    assert len(beta_q) > 0
    assert len(beta_h) > 0

    bq_med = float(
        np.median(beta_q)
    )

    bh_med = float(
        np.median(beta_h)
    )

    bq_low, bq_high = np.quantile(
        beta_q,
        [0.025, 0.975]
    )

    bh_low, bh_high = np.quantile(
        beta_h,
        [0.025, 0.975]
    )

    # Frozen expected memory direction from Block 8.
    expected_direction = str(
        r["effect_direction"]
    ).upper()

    if expected_direction == "NEGATIVE":

        p_direction = float(
            np.mean(beta_h < 0)
        )

    elif expected_direction == "POSITIVE":

        p_direction = float(
            np.mean(beta_h > 0)
        )

    else:

        raise RuntimeError(
            f"Unknown frozen direction: {expected_direction}"
        )

    # ----------------------------------------------------------------------------------------------
    # Raw-scale slopes.
    #
    # beta_Q raw:
    #   response-model units per 1 log10(Q), i.e. per 10-fold Q increase.
    #
    # beta_H raw:
    #   response-model units per one raw hydroclimate unit.
    #
    # For LOG10 response, raw response-model unit = log10(response).
    # ----------------------------------------------------------------------------------------------

    y_sd = float(d.attrs["y_sd"])
    q_sd = float(d.attrs["q_sd"])
    h_sd = float(d.attrs["h_sd"])

    beta_q_raw = (
        bq_med * y_sd / q_sd
    )

    beta_h_raw = (
        bh_med * y_sd / h_sd
    )

    beta_q_raw_low = (
        float(bq_low) * y_sd / q_sd
    )

    beta_q_raw_high = (
        float(bq_high) * y_sd / q_sd
    )

    beta_h_raw_low = (
        float(bh_low) * y_sd / h_sd
    )

    beta_h_raw_high = (
        float(bh_high) * y_sd / h_sd
    )

    posterior_gate = (
        p_direction >= 0.95
    )

    # ----------------------------------------------------------------------------------------------
    # Persist trace immediately.
    # ----------------------------------------------------------------------------------------------

    safe_response = (
        response
        .replace("::", "__")
        .replace("/", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

    safe_hydro = (
        hydro_var
        .replace("/", "_")
        .replace(" ", "_")
    )

    trace_path = (
        TRACE_DIR /
        f"BLOCK102A_{i+1:02d}_{safe_response}__{safe_hydro}_HQ.nc"
    )

    az.to_netcdf(
        fit,
        trace_path
    )

    trace_paths.append(
        trace_path
    )

    result_rows.append(
        {
            "test_id":
                f"HQ_TEST_{i+1:02d}",

            "response_key":
                response,

            "hydro_family":
                r["hydro_family"],

            "window_days":
                r["window_days"],

            "hydro_var":
                hydro_var,

            "joint_analysis_role":
                r["joint_analysis_role"],

            "response_transform":
                r["response_transform"],

            "censor_aware_required":
                r["censor_aware_required"],

            "frozen_expected_H_direction":
                expected_direction,

            "n":
                len(d),

            "n_censored":
                int(
                    d["is_censored_model"].sum()
                ),

            "n_field_events":
                d[M_FIELD].nunique(),

            "n_stations":
                d[M_STATION].nunique(),

            "n_campaigns":
                d[M_CAMPAIGN].nunique(),

            "beta_Q_median_std":
                bq_med,

            "beta_Q_hdi_low_std":
                float(bq_low),

            "beta_Q_hdi_high_std":
                float(bq_high),

            "beta_H_conditional_median_std":
                bh_med,

            "beta_H_conditional_hdi_low_std":
                float(bh_low),

            "beta_H_conditional_hdi_high_std":
                float(bh_high),

            "beta_H_conditional_Pdirection":
                p_direction,

            "beta_H_conditional_posterior_gate":
                posterior_gate,

            "beta_Q_raw_scale":
                beta_q_raw,

            "beta_Q_raw_hdi_low":
                beta_q_raw_low,

            "beta_Q_raw_hdi_high":
                beta_q_raw_high,

            "beta_H_raw_scale":
                beta_h_raw,

            "beta_H_raw_hdi_low":
                beta_h_raw_low,

            "beta_H_raw_hdi_high":
                beta_h_raw_high,

            "Q_log10_mean":
                float(d.attrs["q_mean"]),

            "Q_log10_sd":
                q_sd,

            "H_raw_mean":
                float(d.attrs["h_mean"]),

            "H_raw_sd":
                h_sd,

            "Y_model_mean_detected":
                float(d.attrs["y_mean"]),

            "Y_model_sd_detected":
                y_sd,

            "divergences":
                diag["divergences"],

            "max_rhat":
                diag["max_rhat"],

            "min_ess_bulk":
                diag["min_ess_bulk"],

            "trace_path":
                str(trace_path),

            # IMPORTANT:
            # predictive support is deliberately unavailable until 10.2B.
            "predictive_gate":
                np.nan,

            "final_joint_support":
                np.nan,

            "classification_status":
                "DEFERRED_TO_BLOCK_10.2B_LOCO"
        }
    )

    print("\nPosterior conditional effects:")
    print(
        f"beta_Q std = {bq_med:+.4f} "
        f"[{bq_low:+.4f}, {bq_high:+.4f}]"
    )

    print(
        f"beta_H|Q std = {bh_med:+.4f} "
        f"[{bh_low:+.4f}, {bh_high:+.4f}]"
    )

    print(
        "P(frozen H direction | Q) =",
        round(p_direction, 6)
    )

    print(
        "Posterior directional gate =",
        posterior_gate
    )

    print("\nDiagnostics:")
    print("Divergences :", diag["divergences"])
    print("Max Rhat    :", diag["max_rhat"])
    print("Min ESS bulk:", diag["min_ess_bulk"])
    print("Trace saved :", trace_path)


# ==================================================================================================
# 12. POSTERIOR SPECTRUM
# ==================================================================================================

results = pd.DataFrame(
    result_rows
)


print("\n" + "=" * 120)
print("BLOCK 10.2A — CONDITIONAL H|Q POSTERIOR SPECTRUM")
print("=" * 120)

display_cols = [
    "test_id",
    "response_key",
    "hydro_var",
    "joint_analysis_role",
    "n",
    "n_censored",
    "beta_Q_median_std",
    "beta_H_conditional_median_std",
    "beta_H_conditional_hdi_low_std",
    "beta_H_conditional_hdi_high_std",
    "beta_H_conditional_Pdirection",
    "beta_H_conditional_posterior_gate",
    "divergences",
    "max_rhat",
    "min_ess_bulk"
]

print(
    results[
        display_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 13. COMPUTATIONAL QA
# ==================================================================================================

qa = pd.DataFrame(
    [
        [
            "frozen_plan_rows_5",
            len(plan) == 5
        ],

        [
            "exact_five_prespecified_pairs",
            actual_pairs == expected_pairs
        ],

        [
            "full_models_fit_5",
            len(results) == 5
        ],

        [
            "all_models_HQ_only",
            len(trace_paths) == 5
        ],

        [
            "all_10_campaigns_present",
            bool(
                (
                    results["n_campaigns"]
                    == 10
                ).all()
            )
        ],

        [
            "expected_primary_Q_coverage",
            bool(
                all(
                    len(datasets[k])
                    ==
                    expected_n[k]
                    for k in expected_pairs
                )
            )
        ],

        [
            "SO4_censored_23",
            int(
                results.loc[
                    results["response_key"].eq(
                        "AC::SO4_ppm::mg/L"
                    ),
                    "n_censored"
                ].iloc[0]
            )
            == 23
        ],

        [
            "all_zero_divergences",
            bool(
                (
                    results["divergences"]
                    == 0
                ).all()
            )
        ],

        [
            "all_Rhat_le_1_01",
            bool(
                (
                    results["max_rhat"]
                    <= 1.01
                ).all()
            )
        ],

        [
            "all_ESS_ge_400",
            bool(
                (
                    results["min_ess_bulk"]
                    >= 400
                ).all()
            )
        ],

        [
            "all_beta_H_finite",
            bool(
                np.isfinite(
                    results[
                        "beta_H_conditional_median_std"
                    ]
                ).all()
            )
        ],

        [
            "all_Pdirection_valid",
            bool(
                results[
                    "beta_H_conditional_Pdirection"
                ]
                .between(0, 1)
                .all()
            )
        ],

        [
            "all_traces_exist",
            all(
                p.exists()
                for p in trace_paths
            )
        ],

        [
            "observed_Q_only",
            True
        ],

        [
            "frozen_response_transforms",
            True
        ],

        [
            "sample_specific_censoring",
            True
        ],

        [
            "station_random_intercept",
            True
        ],

        [
            "window_search_performed",
            False
        ],

        [
            "HxQ_interaction_used",
            False
        ],

        [
            "C09_reconstructed_Q_used",
            False
        ],

        [
            "predictive_gate_evaluated",
            False
        ],

        [
            "final_joint_classification_performed",
            False
        ]
    ],
    columns=[
        "check",
        "state"
    ]
)


# Positive requirements.
positive_names = {
    "frozen_plan_rows_5",
    "exact_five_prespecified_pairs",
    "full_models_fit_5",
    "all_models_HQ_only",
    "all_10_campaigns_present",
    "expected_primary_Q_coverage",
    "SO4_censored_23",
    "all_zero_divergences",
    "all_Rhat_le_1_01",
    "all_ESS_ge_400",
    "all_beta_H_finite",
    "all_Pdirection_valid",
    "all_traces_exist",
    "observed_Q_only",
    "frozen_response_transforms",
    "sample_specific_censoring",
    "station_random_intercept"
}

# These are prohibited/premature actions and must be FALSE.
negative_names = {
    "window_search_performed",
    "HxQ_interaction_used",
    "C09_reconstructed_Q_used",
    "predictive_gate_evaluated",
    "final_joint_classification_performed"
}


qa["pass"] = qa.apply(
    lambda r:
        bool(r["state"])
        if r["check"] in positive_names
        else (
            not bool(r["state"])
            if r["check"] in negative_names
            else False
        ),
    axis=1
)


BLOCK_PASS = bool(
    qa["pass"].all()
)


print("\n" + "=" * 120)
print("BLOCK 10.2A QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)


# ==================================================================================================
# 14. SAVE RESULTS
# ==================================================================================================

RESULTS_OUT = (
    OUT_DIR /
    "BLOCK102A_FULLDATA_CONDITIONAL_HQ_POSTERIOR_SPECTRUM_v1.csv"
)

COVERAGE_OUT = (
    OUT_DIR /
    "BLOCK102A_FULLDATA_HQ_COVERAGE_v1.csv"
)

QA_OUT = (
    OUT_DIR /
    "BLOCK102A_FULLDATA_HQ_QA_v1.csv"
)


results.to_csv(
    RESULTS_OUT,
    index=False
)

coverage.to_csv(
    COVERAGE_OUT,
    index=False
)

qa.to_csv(
    QA_OUT,
    index=False
)


# ==================================================================================================
# 15. HASHES / MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "block":
        "10.2A",

    "status":
        "PASS" if BLOCK_PASS else "FAIL",

    "analysis":
        "five pre-specified full-data additive hydroclimate + discharge models",

    "primary_question":
        "posterior hydroclimate effect conditional on instantaneous discharge",

    "n_prespecified_tests":
        5,

    "primary_hydrochemical_tests":
        4,

    "secondary_thermal_reference":
        1,

    "primary_Q":
        "observed discharge only",

    "predictive_support":
        "deferred to Block 10.2B campaign LOCO",

    "window_search":
        False,

    "interaction":
        False,

    "C09_reconstructed_Q":
        False,

    "final_joint_classification":
        False,

    "outputs": {
        str(RESULTS_OUT):
            sha256_file(RESULTS_OUT),

        str(COVERAGE_OUT):
            sha256_file(COVERAGE_OUT),

        str(QA_OUT):
            sha256_file(QA_OUT)
    },

    "traces": {
        str(p):
            sha256_file(p)
        for p in trace_paths
    }
}


MANIFEST_OUT = (
    OUT_DIR /
    "BLOCK102A_FULLDATA_HQ_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 16. FINAL GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("BLOCK 10.2A FINAL GATE")
print("=" * 120)

print(
    "BLOCK 10.2A PASS                       :",
    BLOCK_PASS
)

print(
    "FULL-DATA H+Q MODELS                   :",
    len(results)
)

print(
    "PRIMARY HYDROCHEMICAL TESTS            :",
    int(
        (
            results["joint_analysis_role"]
            ==
            "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST"
        ).sum()
    )
)

print(
    "SECONDARY THERMAL REFERENCES           :",
    int(
        (
            results["joint_analysis_role"]
            ==
            "SECONDARY_THERMAL_REFERENCE"
        ).sum()
    )
)

print(
    "POSTERIOR H|Q DIRECTIONAL GATES PASSED :",
    int(
        results[
            "beta_H_conditional_posterior_gate"
        ].sum()
    ),
    "/ 5"
)

print(
    "PREDICTIVE GATE EVALUATED              :",
    False
)

print(
    "FINAL JOINT SUPPORT CLASSIFIED         :",
    False
)

print(
    "WINDOW SEARCH PERFORMED                :",
    False
)

print(
    "READY FOR BLOCK 10.2B LOCO             :",
    BLOCK_PASS
)


print("\nFiles written:")

for p in [
    RESULTS_OUT,
    COVERAGE_OUT,
    QA_OUT,
    MANIFEST_OUT
]:
    print(" ", p)

print("\nPosterior traces:")
for p in trace_paths:
    print(" ", p)


assert BLOCK_PASS, (
    "\nBLOCK 10.2A computational QA failed.\n"
    "Do not proceed to Block 10.2B."
)


print("\n" + "=" * 120)
print("BLOCK 10.2A PASS — CONDITIONAL H|Q POSTERIOR SPECTRUM FROZEN")
print("=" * 120)

---
### Imported master block 134

`BLOCK 10.2A-R2 — FIVE PRE-SPECIFIED FULL-DATA H+Q MODELS`


In [ ]:
# ==================================================================================================
# BLOCK 10.2A-R2 — FIVE PRE-SPECIFIED FULL-DATA H+Q MODELS
#
# Requires:
#   - Block 10.2A-R1 PASS in current runtime
#   - datasets: 5 repaired datasets
#   - plan: frozen Block-10.0B analysis plan
#   - fit_joint_model(): validated Block-10.1 engine
#
# Fits ONLY:
#   EC  + P14
#   EC  + P60
#   Na  + P60
#   SO4 + P60
#   Water temperature + root-zone SM90
#
# NO LOCO
# NO window search
# NO HxQ interaction
# NO reconstructed C09 Q
# NO final scientific classification
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import arviz as az

print("=" * 120)
print("BLOCK 10.2A-R2 — FULL-DATA CONDITIONAL H|Q POSTERIOR SPECTRUM")
print("=" * 120)


# ==================================================================================================
# 1. PRE-FLIGHT
# ==================================================================================================

assert "datasets" in globals()
assert len(datasets) == 5

assert "plan" in globals()
assert len(plan) == 5

assert "fit_joint_model" in globals()

assert "M_STATION" in globals()
assert "M_CAMPAIGN" in globals()
assert "M_FIELD" in globals()

required_engine_cols = [
    M_STATION,
    "is_censored_model",
    "Y_z",
    "censor_z",
    "Q_z",
    "H_z"
]

for key, d in datasets.items():

    missing = [
        c for c in required_engine_cols
        if c not in d.columns
    ]

    assert not missing, (
        f"{key}: missing engine columns {missing}"
    )


expected_pairs = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"),
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"),
    ("AC::Na::mg/L", "ppt_sum_60d"),
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"),
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d")
}

actual_pairs = set(datasets.keys())

assert actual_pairs == expected_pairs


print("\nPre-flight:")
print("Datasets                    :", len(datasets))
print("Frozen pairs exact          :", actual_pairs == expected_pairs)
print("Validated engine available  :", True)
print("Engine contract repaired    :", True)


# ==================================================================================================
# 2. OUTPUT DIRECTORY
# ==================================================================================================

PROJECT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

OUT_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULLDATA_POSTERIOR"
)

TRACE_DIR = OUT_DIR / "TRACES"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRACE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 3. HELPER FUNCTIONS
# ==================================================================================================

def safe_name(x):

    return (
        str(x)
        .replace("::", "__")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def summarize_diagnostics(idata, variables):

    summary = az.summary(
        idata,
        var_names=variables,
        kind="diagnostics"
    )

    max_rhat = float(
        summary["r_hat"].max()
    )

    min_ess_bulk = float(
        summary["ess_bulk"].min()
    )

    div = 0

    if "diverging" in idata.sample_stats:

        div = int(
            idata.sample_stats[
                "diverging"
            ]
            .values
            .sum()
        )

    return {
        "divergences": div,
        "max_rhat": max_rhat,
        "min_ess_bulk": min_ess_bulk
    }


def posterior_vector(idata, variable):

    assert variable in idata.posterior.data_vars, (
        f"{variable} absent from posterior."
    )

    return (
        idata.posterior[variable]
        .values
        .reshape(-1)
        .astype(float)
    )


# ==================================================================================================
# 4. ORDER EXACTLY AS FROZEN IN PLAN
# ==================================================================================================

fit_plan = plan[
    [
        "response_key",
        "hydro_family",
        "window_days",
        "hydro_var",
        "effect_direction",
        "joint_analysis_role",
        "response_transform",
        "censor_aware_required"
    ]
].copy()


assert len(fit_plan) == 5


print("\n" + "=" * 120)
print("FROZEN MODEL ORDER")
print("=" * 120)

print(
    fit_plan.to_string(
        index=False
    )
)


# ==================================================================================================
# 5. FIT FIVE H+Q MODELS
# ==================================================================================================

result_rows = []
trace_paths = []


for i, r in fit_plan.reset_index(drop=True).iterrows():

    response = r["response_key"]
    hydro_var = r["hydro_var"]

    key = (
        response,
        hydro_var
    )

    assert key in datasets

    d = datasets[key].copy()

    # ----------------------------------------------------------------------------------------------
    # Final dataset contract check immediately before sampling.
    # ----------------------------------------------------------------------------------------------

    cens = (
        d["is_censored_model"]
        .astype(bool)
    )

    unc = ~cens

    assert d.loc[
        unc,
        "Y_z"
    ].notna().all()

    assert d.loc[
        cens,
        "censor_z"
    ].notna().all()

    assert d["Q_z"].notna().all()
    assert d["H_z"].notna().all()
    assert d[M_STATION].notna().all()

    print("\n" + "=" * 120)
    print(
        f"MODEL {i+1}/5 — {response} + {hydro_var}"
    )
    print("=" * 120)

    print(
        "Rows      :",
        len(d)
    )

    print(
        "Censored  :",
        int(cens.sum())
    )

    print(
        "Stations  :",
        d[M_STATION].nunique()
    )

    print(
        "Campaigns :",
        d[M_CAMPAIGN].nunique()
    )

    # ----------------------------------------------------------------------------------------------
    # FIT VALIDATED ENGINE
    # ----------------------------------------------------------------------------------------------

    fit = fit_joint_model(
        df_train=d,
        model_type="HQ",
        draws=2000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        seed=20262000 + i
    )

    # Exact validated engine return contract.
    assert isinstance(fit, dict)
    assert "idata" in fit

    idata = fit["idata"]

    assert hasattr(
        idata,
        "posterior"
    )

    assert "beta_Q" in idata.posterior
    assert "beta_H" in idata.posterior

    # ----------------------------------------------------------------------------------------------
    # POSTERIOR
    # ----------------------------------------------------------------------------------------------

    beta_Q = posterior_vector(
        idata,
        "beta_Q"
    )

    beta_H = posterior_vector(
        idata,
        "beta_H"
    )

    bq_med = float(
        np.median(beta_Q)
    )

    bh_med = float(
        np.median(beta_H)
    )

    bq_low, bq_high = np.quantile(
        beta_Q,
        [0.025, 0.975]
    )

    bh_low, bh_high = np.quantile(
        beta_H,
        [0.025, 0.975]
    )

    expected_direction = (
        str(
            r["effect_direction"]
        )
        .strip()
        .upper()
    )

    if expected_direction == "NEGATIVE":

        Pdirection = float(
            np.mean(
                beta_H < 0
            )
        )

    elif expected_direction == "POSITIVE":

        Pdirection = float(
            np.mean(
                beta_H > 0
            )
        )

    else:

        raise RuntimeError(
            "Unexpected frozen H direction: "
            + expected_direction
        )

    posterior_gate = bool(
        Pdirection >= 0.95
    )

    # ----------------------------------------------------------------------------------------------
    # DIAGNOSTICS
    # ----------------------------------------------------------------------------------------------

    diag = summarize_diagnostics(
        idata,
        [
            "alpha",
            "sigma",
            "sigma_station",
            "beta_Q",
            "beta_H"
        ]
    )

    diagnostic_pass = bool(
        diag["divergences"] == 0
        and diag["max_rhat"] <= 1.01
        and diag["min_ess_bulk"] >= 400
    )

    # ----------------------------------------------------------------------------------------------
    # RAW-SCALE CONDITIONAL SLOPES
    #
    # These attrs were frozen when datasets were built in 10.2A.
    # ----------------------------------------------------------------------------------------------

    for attr in [
        "q_sd",
        "h_sd",
        "y_sd"
    ]:
        assert attr in d.attrs, (
            f"{key}: missing {attr} scaling metadata."
        )

    q_sd = float(
        d.attrs["q_sd"]
    )

    h_sd = float(
        d.attrs["h_sd"]
    )

    y_sd = float(
        d.attrs["y_sd"]
    )

    beta_Q_raw = (
        bq_med
        *
        y_sd
        /
        q_sd
    )

    beta_H_raw = (
        bh_med
        *
        y_sd
        /
        h_sd
    )

    beta_Q_raw_low = (
        float(bq_low)
        *
        y_sd
        /
        q_sd
    )

    beta_Q_raw_high = (
        float(bq_high)
        *
        y_sd
        /
        q_sd
    )

    beta_H_raw_low = (
        float(bh_low)
        *
        y_sd
        /
        h_sd
    )

    beta_H_raw_high = (
        float(bh_high)
        *
        y_sd
        /
        h_sd
    )

    # ----------------------------------------------------------------------------------------------
    # SAVE TRACE IMMEDIATELY
    # ----------------------------------------------------------------------------------------------

    trace_path = (
        TRACE_DIR /
        (
            f"BLOCK102A_R2_{i+1:02d}_"
            f"{safe_name(response)}__"
            f"{safe_name(hydro_var)}__HQ.nc"
        )
    )

    az.to_netcdf(
        idata,
        trace_path
    )

    assert trace_path.exists()

    trace_paths.append(
        trace_path
    )

    # ----------------------------------------------------------------------------------------------
    # SAVE ONE ROW IMMEDIATELY AS CHECKPOINT
    # ----------------------------------------------------------------------------------------------

    row = {
        "test_id":
            f"HQ_TEST_{i+1:02d}",

        "response_key":
            response,

        "hydro_family":
            r["hydro_family"],

        "window_days":
            int(r["window_days"]),

        "hydro_var":
            hydro_var,

        "joint_analysis_role":
            r["joint_analysis_role"],

        "response_transform":
            r["response_transform"],

        "censor_aware_required":
            r["censor_aware_required"],

        "frozen_expected_H_direction":
            expected_direction,

        "n":
            len(d),

        "n_censored":
            int(cens.sum()),

        "n_field_events":
            d[M_FIELD].nunique(),

        "n_stations":
            d[M_STATION].nunique(),

        "n_campaigns":
            d[M_CAMPAIGN].nunique(),

        "beta_Q_median_std":
            bq_med,

        "beta_Q_hdi_low_std":
            float(bq_low),

        "beta_Q_hdi_high_std":
            float(bq_high),

        "beta_H_conditional_median_std":
            bh_med,

        "beta_H_conditional_hdi_low_std":
            float(bh_low),

        "beta_H_conditional_hdi_high_std":
            float(bh_high),

        "beta_H_conditional_Pdirection":
            Pdirection,

        "posterior_directional_gate":
            posterior_gate,

        "beta_Q_raw_scale":
            beta_Q_raw,

        "beta_Q_raw_hdi_low":
            beta_Q_raw_low,

        "beta_Q_raw_hdi_high":
            beta_Q_raw_high,

        "beta_H_raw_scale":
            beta_H_raw,

        "beta_H_raw_hdi_low":
            beta_H_raw_low,

        "beta_H_raw_hdi_high":
            beta_H_raw_high,

        "divergences":
            diag["divergences"],

        "max_rhat":
            diag["max_rhat"],

        "min_ess_bulk":
            diag["min_ess_bulk"],

        "diagnostic_pass":
            diagnostic_pass,

        "predictive_gate":
            np.nan,

        "final_joint_support":
            np.nan,

        "classification_status":
            "DEFERRED_TO_BLOCK_10.2B_LOCO",

        "trace_path":
            str(trace_path)
    }

    result_rows.append(
        row
    )

    checkpoint = pd.DataFrame(
        result_rows
    )

    checkpoint_path = (
        OUT_DIR /
        "BLOCK102A_R2_RUNNING_CHECKPOINT_v1.csv"
    )

    checkpoint.to_csv(
        checkpoint_path,
        index=False
    )

    # ----------------------------------------------------------------------------------------------
    # PRINT MODEL RESULT
    # ----------------------------------------------------------------------------------------------

    print("\nConditional posterior:")

    print(
        f"beta_Q = "
        f"{bq_med:+.6f} "
        f"[{bq_low:+.6f}, {bq_high:+.6f}]"
    )

    print(
        f"beta_H|Q = "
        f"{bh_med:+.6f} "
        f"[{bh_low:+.6f}, {bh_high:+.6f}]"
    )

    print(
        "P(frozen H direction | Q) =",
        round(
            Pdirection,
            6
        )
    )

    print(
        "Posterior directional gate =",
        posterior_gate
    )

    print("\nDiagnostics:")

    print(
        "Divergences :",
        diag["divergences"]
    )

    print(
        "Max Rhat    :",
        diag["max_rhat"]
    )

    print(
        "Min ESS bulk:",
        diag["min_ess_bulk"]
    )

    print(
        "Diagnostic PASS:",
        diagnostic_pass
    )

    print(
        "Trace saved:",
        trace_path
    )


# ==================================================================================================
# 6. COMBINE RESULTS
# ==================================================================================================

results = pd.DataFrame(
    result_rows
)

assert len(results) == 5


print("\n" + "=" * 120)
print("BLOCK 10.2A-R2 — CONDITIONAL H|Q POSTERIOR SPECTRUM")
print("=" * 120)

display_cols = [
    "test_id",
    "response_key",
    "hydro_var",
    "joint_analysis_role",
    "n",
    "n_censored",
    "beta_Q_median_std",
    "beta_H_conditional_median_std",
    "beta_H_conditional_hdi_low_std",
    "beta_H_conditional_hdi_high_std",
    "beta_H_conditional_Pdirection",
    "posterior_directional_gate",
    "divergences",
    "max_rhat",
    "min_ess_bulk",
    "diagnostic_pass"
]

print(
    results[
        display_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 7. QA
# ==================================================================================================

qa_rows = [
    (
        "five_prespecified_models",
        len(results) == 5
    ),

    (
        "exact_frozen_pairs",
        set(
            zip(
                results["response_key"],
                results["hydro_var"]
            )
        )
        == expected_pairs
    ),

    (
        "all_models_10_campaigns",
        bool(
            (
                results["n_campaigns"]
                == 10
            ).all()
        )
    ),

    (
        "EC_P14_n115",
        int(
            results.loc[
                (
                    results["response_key"]
                    ==
                    "AC::C.E._SOL::dS/m"
                )
                &
                (
                    results["hydro_var"]
                    ==
                    "ppt_sum_14d"
                ),
                "n"
            ].iloc[0]
        )
        == 115
    ),

    (
        "EC_P60_n115",
        int(
            results.loc[
                (
                    results["response_key"]
                    ==
                    "AC::C.E._SOL::dS/m"
                )
                &
                (
                    results["hydro_var"]
                    ==
                    "ppt_sum_60d"
                ),
                "n"
            ].iloc[0]
        )
        == 115
    ),

    (
        "Na_P60_n115",
        int(
            results.loc[
                results["response_key"]
                ==
                "AC::Na::mg/L",
                "n"
            ].iloc[0]
        )
        == 115
    ),

    (
        "SO4_P60_n115",
        int(
            results.loc[
                results["response_key"]
                ==
                "AC::SO4_ppm::mg/L",
                "n"
            ].iloc[0]
        )
        == 115
    ),

    (
        "SO4_censored_23",
        int(
            results.loc[
                results["response_key"]
                ==
                "AC::SO4_ppm::mg/L",
                "n_censored"
            ].iloc[0]
        )
        == 23
    ),

    (
        "temperature_n111",
        int(
            results.loc[
                results["response_key"]
                ==
                "FIELD::Water temperature::degC",
                "n"
            ].iloc[0]
        )
        == 111
    ),

    (
        "all_zero_divergences",
        bool(
            (
                results["divergences"]
                == 0
            ).all()
        )
    ),

    (
        "all_Rhat_le_1_01",
        bool(
            (
                results["max_rhat"]
                <= 1.01
            ).all()
        )
    ),

    (
        "all_ESS_ge_400",
        bool(
            (
                results["min_ess_bulk"]
                >= 400
            ).all()
        )
    ),

    (
        "all_diagnostics_pass",
        bool(
            results[
                "diagnostic_pass"
            ].all()
        )
    ),

    (
        "all_traces_saved",
        all(
            p.exists()
            for p in trace_paths
        )
    ),

    (
        "observed_Q_only",
        True
    ),

    (
        "window_search_performed",
        False
    ),

    (
        "HxQ_interaction_used",
        False
    ),

    (
        "C09_reconstructed_Q_used",
        False
    ),

    (
        "predictive_gate_evaluated",
        False
    ),

    (
        "final_classification_performed",
        False
    )
]


qa = pd.DataFrame(
    qa_rows,
    columns=[
        "check",
        "state"
    ]
)


negative_checks = {
    "window_search_performed",
    "HxQ_interaction_used",
    "C09_reconstructed_Q_used",
    "predictive_gate_evaluated",
    "final_classification_performed"
}


qa["pass"] = qa.apply(
    lambda r:
        (
            not bool(r["state"])
            if r["check"] in negative_checks
            else bool(r["state"])
        ),
    axis=1
)


BLOCK_PASS = bool(
    qa["pass"].all()
)


print("\n" + "=" * 120)
print("BLOCK 10.2A-R2 QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)


# ==================================================================================================
# 8. SAVE FROZEN OUTPUTS
# ==================================================================================================

RESULTS_OUT = (
    OUT_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_POSTERIOR_SPECTRUM_v1.csv"
)

QA_OUT = (
    OUT_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_QA_v1.csv"
)


results.to_csv(
    RESULTS_OUT,
    index=False
)

qa.to_csv(
    QA_OUT,
    index=False
)


manifest = {
    "block":
        "10.2A-R2",

    "status":
        "PASS"
        if BLOCK_PASS
        else "FAIL",

    "models_fit":
        5,

    "model_type":
        "HQ",

    "purpose":
        "Full-data conditional antecedent-hydroclimate posterior given observed discharge",

    "primary_Q":
        "OBSERVED_ONLY",

    "window_search":
        False,

    "HxQ_interaction":
        False,

    "C09_reconstructed_Q":
        False,

    "predictive_gate_evaluated":
        False,

    "final_joint_classification":
        False,

    "results_file":
        str(RESULTS_OUT),

    "results_sha256":
        sha256_file(
            RESULTS_OUT
        ),

    "QA_file":
        str(QA_OUT),

    "QA_sha256":
        sha256_file(
            QA_OUT
        ),

    "traces": {
        str(p):
            sha256_file(p)
        for p in trace_paths
    }
}


MANIFEST_OUT = (
    OUT_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 9. FINAL GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("BLOCK 10.2A-R2 FINAL GATE")
print("=" * 120)

print(
    "BLOCK 10.2A-R2 PASS                   :",
    BLOCK_PASS
)

print(
    "FULL-DATA H+Q MODELS                 :",
    len(results)
)

print(
    "POSTERIOR DIRECTIONAL GATES PASSED   :",
    int(
        results[
            "posterior_directional_gate"
        ].sum()
    ),
    "/ 5"
)

print(
    "ALL MODEL DIAGNOSTICS PASS           :",
    bool(
        results[
            "diagnostic_pass"
        ].all()
    )
)

print(
    "PREDICTIVE GATE EVALUATED            :",
    False
)

print(
    "FINAL JOINT SUPPORT CLASSIFIED       :",
    False
)

print(
    "WINDOW SEARCH PERFORMED              :",
    False
)

print(
    "C09 RECONSTRUCTED Q USED             :",
    False
)

print(
    "READY FOR BLOCK 10.2B CAMPAIGN LOCO :",
    BLOCK_PASS
)


print("\nFiles written:")
print(" ", RESULTS_OUT)
print(" ", QA_OUT)
print(" ", MANIFEST_OUT)

print("\nPosterior traces:")
for p in trace_paths:
    print(" ", p)


assert BLOCK_PASS, (
    "\nBLOCK 10.2A-R2 failed computational QA.\n"
    "Do NOT proceed to Block 10.2B."
)


print("\n" + "=" * 120)
print("BLOCK 10.2A-R2 PASS — FULL-DATA CONDITIONAL H|Q POSTERIORS FROZEN")
print("=" * 120)

---
### Imported master block 135

`BLOCK 10.2B-A — FULL CAMPAIGN-LOCO PREFLIGHT / DATA & SCORING CONTRACT FREEZE`


In [ ]:
# ==================================================================================================
# BLOCK 10.2B-A — FULL CAMPAIGN-LOCO PREFLIGHT / DATA & SCORING CONTRACT FREEZE
#
# PURPOSE
#   Prepare and audit the complete campaign-LOCO architecture for the five
#   pre-specified H+Q tests BEFORE fitting the 200 Bayesian models.
#
# DESIGN
#   5 frozen response-H pairs
#   x 10 held-out campaigns
#   x 4 comparators: NULL / Q / H / HQ
#   = 50 train/test folds and, later, 200 Bayesian fits.
#
# CRITICAL
#   - NO PyMC models fitted here.
#   - Q = observed discharge only.
#   - C09 reconstructed Q prohibited.
#   - Scaling is recomputed FROM TRAINING DATA ONLY in every fold.
#   - The exact same train/test rows are used by NULL/Q/H/HQ within a fold.
#   - Sample-specific censoring preserved.
#   - Distinct analytical subsamples preserved.
#   - Unseen held-out stations will be marginalized in predictive scoring.
#   - No window search.
#   - No HxQ interaction.
#   - No scientific classification.
#
# OUTPUT
#   Frozen 50-fold LOCO manifest + QA + checkpoint architecture.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 120)
print("BLOCK 10.2B-A — CAMPAIGN-LOCO PREFLIGHT / CONTRACT FREEZE")
print("=" * 120)


# ==================================================================================================
# 0. REQUIRED RUNTIME OBJECTS
# ==================================================================================================

assert "datasets" in globals(), (
    "datasets missing. Recover Block 10.2A-R1/R2 runtime objects first."
)

assert "plan" in globals(), (
    "Frozen Block-10 plan missing."
)

assert len(datasets) == 5
assert len(plan) == 5

assert "M_STATION" in globals()
assert "M_CAMPAIGN" in globals()
assert "M_FIELD" in globals()

print("\nRuntime prerequisites: PASS")


# ==================================================================================================
# 1. VERIFY BLOCK 10.2A-R2 FROZEN OUTPUT
# ==================================================================================================

PROJECT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

FULLDATA_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULLDATA_POSTERIOR"
)

FULLDATA_RESULTS = (
    FULLDATA_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_POSTERIOR_SPECTRUM_v1.csv"
)

FULLDATA_QA = (
    FULLDATA_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_QA_v1.csv"
)

FULLDATA_MANIFEST = (
    FULLDATA_DIR /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_manifest.json"
)

for p in [
    FULLDATA_RESULTS,
    FULLDATA_QA,
    FULLDATA_MANIFEST
]:
    assert p.exists(), p


qa102a = pd.read_csv(
    FULLDATA_QA,
    keep_default_na=False
)

assert qa102a["pass"].astype(str).str.lower().eq("true").all()

with open(
    FULLDATA_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    manifest102a = json.load(f)

assert manifest102a["status"] == "PASS"

print("Block 10.2A-R2 frozen state: PASS")


# ==================================================================================================
# 2. EXACT FROZEN TESTS
# ==================================================================================================

expected_pairs = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"),
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"),
    ("AC::Na::mg/L", "ppt_sum_60d"),
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"),
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d")
}

assert set(datasets.keys()) == expected_pairs


campaigns_expected = [
    f"C{i:02d}"
    for i in range(1, 11)
]

models_expected = [
    "NULL",
    "Q",
    "H",
    "HQ"
]


# ==================================================================================================
# 3. EXPECTED FULL-DATA COVERAGE
# ==================================================================================================

expected_n = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"): 115,
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"): 115,
    ("AC::Na::mg/L", "ppt_sum_60d"): 115,
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"): 115,
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d"): 111
}

expected_censored = {
    ("AC::C.E._SOL::dS/m", "ppt_sum_14d"): 0,
    ("AC::C.E._SOL::dS/m", "ppt_sum_60d"): 0,
    ("AC::Na::mg/L", "ppt_sum_60d"): 0,
    ("AC::SO4_ppm::mg/L", "ppt_sum_60d"): 23,
    ("FIELD::Water temperature::degC", "sm_rootzone_mean_90d"): 0
}


# ==================================================================================================
# 4. TRAIN-ONLY SCALING FUNCTION
#
# IMPORTANT:
#   We deliberately DO NOT reuse Q_z/H_z/Y_z from full-data 10.2A.
# ==================================================================================================

def prepare_loco_split(full_df, heldout_campaign):

    full = full_df.copy()

    assert heldout_campaign in set(
        full[M_CAMPAIGN].astype(str)
    )

    train = full[
        full[M_CAMPAIGN].astype(str)
        != heldout_campaign
    ].copy()

    test = full[
        full[M_CAMPAIGN].astype(str)
        == heldout_campaign
    ].copy()

    assert len(train) > 0
    assert len(test) > 0

    # ----------------------------------------------------------------------------------------------
    # 4.1 Verify no campaign leakage
    # ----------------------------------------------------------------------------------------------

    assert heldout_campaign not in set(
        train[M_CAMPAIGN].astype(str)
    )

    assert set(
        test[M_CAMPAIGN].astype(str)
    ) == {
        heldout_campaign
    }

    # ----------------------------------------------------------------------------------------------
    # 4.2 Predictors in RAW modeling space
    #
    # These were already constructed from:
    #   log10_Q = log10(observed Q)
    #   H_raw   = frozen antecedent metric
    #
    # We use raw columns, not full-data standardized columns.
    # ----------------------------------------------------------------------------------------------

    for col in [
        "log10_Q",
        "H_raw",
        "y_model",
        "censor_limit_model",
        "is_censored_model"
    ]:
        assert col in full.columns, col

    # ----------------------------------------------------------------------------------------------
    # 4.3 TRAIN-ONLY Q scaling
    # ----------------------------------------------------------------------------------------------

    q_mean = float(
        train["log10_Q"].mean()
    )

    q_sd = float(
        train["log10_Q"].std(ddof=1)
    )

    assert np.isfinite(q_mean)
    assert np.isfinite(q_sd)
    assert q_sd > 0

    train["Q_z"] = (
        train["log10_Q"] - q_mean
    ) / q_sd

    test["Q_z"] = (
        test["log10_Q"] - q_mean
    ) / q_sd

    # ----------------------------------------------------------------------------------------------
    # 4.4 TRAIN-ONLY H scaling
    # ----------------------------------------------------------------------------------------------

    h_mean = float(
        train["H_raw"].mean()
    )

    h_sd = float(
        train["H_raw"].std(ddof=1)
    )

    assert np.isfinite(h_mean)
    assert np.isfinite(h_sd)
    assert h_sd > 0

    train["H_z"] = (
        train["H_raw"] - h_mean
    ) / h_sd

    test["H_z"] = (
        test["H_raw"] - h_mean
    ) / h_sd

    # ----------------------------------------------------------------------------------------------
    # 4.5 TRAIN-ONLY Y scaling
    #
    # Frozen engine architecture:
    # scale from detected TRAINING observations.
    # Censored observations contribute through their transformed censor limit.
    # ----------------------------------------------------------------------------------------------

    train_cens = (
        train["is_censored_model"]
        .astype(bool)
    )

    test_cens = (
        test["is_censored_model"]
        .astype(bool)
    )

    detected_train_y = pd.to_numeric(
        train.loc[
            ~train_cens,
            "y_model"
        ],
        errors="coerce"
    )

    assert detected_train_y.notna().all()
    assert len(detected_train_y) > 2

    y_mean = float(
        detected_train_y.mean()
    )

    y_sd = float(
        detected_train_y.std(ddof=1)
    )

    assert np.isfinite(y_mean)
    assert np.isfinite(y_sd)
    assert y_sd > 0

    # Engine contract.
    train["Y_z"] = (
        pd.to_numeric(
            train["y_model"],
            errors="coerce"
        )
        - y_mean
    ) / y_sd

    test["Y_z"] = (
        pd.to_numeric(
            test["y_model"],
            errors="coerce"
        )
        - y_mean
    ) / y_sd

    train["censor_z"] = (
        pd.to_numeric(
            train["censor_limit_model"],
            errors="coerce"
        )
        - y_mean
    ) / y_sd

    test["censor_z"] = (
        pd.to_numeric(
            test["censor_limit_model"],
            errors="coerce"
        )
        - y_mean
    ) / y_sd

    # ----------------------------------------------------------------------------------------------
    # 4.6 Contract QA
    # ----------------------------------------------------------------------------------------------

    assert train.loc[
        ~train_cens,
        "Y_z"
    ].notna().all()

    assert test.loc[
        ~test_cens,
        "Y_z"
    ].notna().all()

    assert train.loc[
        train_cens,
        "censor_z"
    ].notna().all()

    assert test.loc[
        test_cens,
        "censor_z"
    ].notna().all()

    assert train["Q_z"].notna().all()
    assert test["Q_z"].notna().all()

    assert train["H_z"].notna().all()
    assert test["H_z"].notna().all()

    # ----------------------------------------------------------------------------------------------
    # 4.7 Row identity integrity
    # ----------------------------------------------------------------------------------------------

    # No physical FIELD observation can cross campaigns.
    train_fields = set(
        train[M_FIELD]
        .dropna()
        .astype(str)
    )

    test_fields = set(
        test[M_FIELD]
        .dropna()
        .astype(str)
    )

    assert train_fields.isdisjoint(
        test_fields
    )

    scaling = {
        "Q_train_mean": q_mean,
        "Q_train_sd": q_sd,
        "H_train_mean": h_mean,
        "H_train_sd": h_sd,
        "Y_train_detected_mean": y_mean,
        "Y_train_detected_sd": y_sd
    }

    return train, test, scaling


# ==================================================================================================
# 5. BUILD ALL 50 FOLDS — NO MODELS
# ==================================================================================================

folds = {}
fold_rows = []


plan_lookup = (
    plan.set_index(
        [
            "response_key",
            "hydro_var"
        ]
    )
)


for pair_index, key in enumerate(
    sorted(
        expected_pairs,
        key=lambda x: (
            list(
                zip(
                    plan["response_key"],
                    plan["hydro_var"]
                )
            ).index(x)
        )
    ),
    start=1
):

    response, hydro_var = key

    full = datasets[key].copy()

    assert len(full) == expected_n[key]

    assert int(
        full["is_censored_model"]
        .astype(bool)
        .sum()
    ) == expected_censored[key]

    actual_campaigns = sorted(
        full[M_CAMPAIGN]
        .astype(str)
        .unique()
        .tolist()
    )

    assert actual_campaigns == campaigns_expected

    plan_row = plan_lookup.loc[
        key
    ]

    for heldout in campaigns_expected:

        train, test, scaling = prepare_loco_split(
            full,
            heldout
        )

        fold_id = (
            f"HQ{pair_index:02d}_"
            f"{heldout}"
        )

        folds[
            (
                response,
                hydro_var,
                heldout
            )
        ] = {
            "fold_id": fold_id,
            "train": train,
            "test": test,
            "scaling": scaling
        }

        # Stations unseen in training.
        train_stations = set(
            train[M_STATION]
            .astype(str)
        )

        test_stations = set(
            test[M_STATION]
            .astype(str)
        )

        unseen_test_stations = (
            test_stations
            -
            train_stations
        )

        fold_rows.append(
            {
                "fold_id":
                    fold_id,

                "pair_index":
                    pair_index,

                "response_key":
                    response,

                "hydro_var":
                    hydro_var,

                "hydro_family":
                    plan_row["hydro_family"],

                "window_days":
                    int(
                        plan_row["window_days"]
                    ),

                "joint_analysis_role":
                    plan_row[
                        "joint_analysis_role"
                    ],

                "response_transform":
                    plan_row[
                        "response_transform"
                    ],

                "heldout_campaign":
                    heldout,

                "n_full":
                    len(full),

                "n_train":
                    len(train),

                "n_test":
                    len(test),

                "n_train_censored":
                    int(
                        train[
                            "is_censored_model"
                        ]
                        .astype(bool)
                        .sum()
                    ),

                "n_test_censored":
                    int(
                        test[
                            "is_censored_model"
                        ]
                        .astype(bool)
                        .sum()
                    ),

                "n_train_stations":
                    train[
                        M_STATION
                    ].nunique(),

                "n_test_stations":
                    test[
                        M_STATION
                    ].nunique(),

                "n_unseen_test_stations":
                    len(
                        unseen_test_stations
                    ),

                "unseen_test_stations":
                    "|".join(
                        sorted(
                            unseen_test_stations
                        )
                    ),

                "Q_train_mean":
                    scaling[
                        "Q_train_mean"
                    ],

                "Q_train_sd":
                    scaling[
                        "Q_train_sd"
                    ],

                "H_train_mean":
                    scaling[
                        "H_train_mean"
                    ],

                "H_train_sd":
                    scaling[
                        "H_train_sd"
                    ],

                "Y_train_detected_mean":
                    scaling[
                        "Y_train_detected_mean"
                    ],

                "Y_train_detected_sd":
                    scaling[
                        "Y_train_detected_sd"
                    ],

                "same_rows_for_NULL_Q_H_HQ":
                    True,

                "training_only_scaling":
                    True,

                "observed_Q_only":
                    True,

                "sample_specific_censoring":
                    True
            }
        )


fold_manifest = pd.DataFrame(
    fold_rows
)


# ==================================================================================================
# 6. GLOBAL FOLD STRUCTURE
# ==================================================================================================

print("\n" + "=" * 120)
print("50-FOLD LOCO MANIFEST")
print("=" * 120)

print(
    fold_manifest[
        [
            "fold_id",
            "response_key",
            "hydro_var",
            "heldout_campaign",
            "n_train",
            "n_test",
            "n_train_censored",
            "n_test_censored",
            "n_train_stations",
            "n_test_stations",
            "n_unseen_test_stations"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 7. CAMPAIGN COVERAGE MATRIX
# ==================================================================================================

coverage_matrix = (
    fold_manifest
    .pivot_table(
        index=[
            "response_key",
            "hydro_var"
        ],
        columns="heldout_campaign",
        values="n_test",
        aggfunc="first"
    )
    .reset_index()
)


print("\n" + "=" * 120)
print("HELD-OUT CAMPAIGN SAMPLE COUNTS")
print("=" * 120)

print(
    coverage_matrix.to_string(
        index=False
    )
)


# ==================================================================================================
# 8. VERIFY EACH PAIR PARTITIONS FULL DATA EXACTLY
# ==================================================================================================

partition_rows = []


for key in expected_pairs:

    response, hydro_var = key

    sub = fold_manifest[
        (
            fold_manifest["response_key"]
            == response
        )
        &
        (
            fold_manifest["hydro_var"]
            == hydro_var
        )
    ].copy()

    assert len(sub) == 10

    n_test_sum = int(
        sub["n_test"].sum()
    )

    full_n = expected_n[key]

    partition_rows.append(
        {
            "response_key":
                response,

            "hydro_var":
                hydro_var,

            "full_n":
                full_n,

            "sum_heldout_n":
                n_test_sum,

            "exact_partition":
                n_test_sum == full_n,

            "campaigns":
                sub[
                    "heldout_campaign"
                ].nunique()
        }
    )


partition_audit = pd.DataFrame(
    partition_rows
)


print("\n" + "=" * 120)
print("PAIRWISE PARTITION AUDIT")
print("=" * 120)

print(
    partition_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 9. VERIFY NO FULL-DATA STANDARDIZATION LEAKAGE
#
# At least one fold should differ from full-data means/SDs.
# We do NOT demand all folds differ numerically.
# ==================================================================================================

leakage_rows = []


for key in expected_pairs:

    response, hydro_var = key

    full = datasets[key]

    full_q_mean = float(
        full["log10_Q"].mean()
    )

    full_h_mean = float(
        full["H_raw"].mean()
    )

    sub = fold_manifest[
        (
            fold_manifest["response_key"]
            == response
        )
        &
        (
            fold_manifest["hydro_var"]
            == hydro_var
        )
    ]

    q_diff = np.abs(
        sub["Q_train_mean"]
        -
        full_q_mean
    )

    h_diff = np.abs(
        sub["H_train_mean"]
        -
        full_h_mean
    )

    leakage_rows.append(
        {
            "response_key":
                response,

            "hydro_var":
                hydro_var,

            "at_least_one_Q_training_mean_differs_from_full":
                bool(
                    (q_diff > 1e-12)
                    .any()
                ),

            "at_least_one_H_training_mean_differs_from_full":
                bool(
                    (h_diff > 1e-12)
                    .any()
                )
        }
    )


leakage_audit = pd.DataFrame(
    leakage_rows
)


print("\n" + "=" * 120)
print("TRAIN-ONLY SCALING LEAKAGE AUDIT")
print("=" * 120)

print(
    leakage_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 10. FREEZE PREDICTIVE SCORING CONTRACT
# ==================================================================================================

scoring_rules = pd.DataFrame(
    [
        [
            "L01",
            "Campaign is the held-out temporal unit.",
            True
        ],

        [
            "L02",
            "NULL/Q/H/HQ use exactly the same train and test rows within each fold.",
            True
        ],

        [
            "L03",
            "Q, H and response scaling are estimated from training data only.",
            True
        ],

        [
            "L04",
            "Held-out observations are transformed using training-set scaling parameters.",
            True
        ],

        [
            "L05",
            "Observed discharge only is used in primary LOCO.",
            True
        ],

        [
            "L06",
            "Reconstructed C09 discharge is prohibited in primary LOCO.",
            True
        ],

        [
            "L07",
            "Detected held-out responses are scored using Normal posterior predictive density.",
            True
        ],

        [
            "L08",
            "Left-censored held-out responses are scored using posterior predictive Normal CDF at the sample-specific censor limit.",
            True
        ],

        [
            "L09",
            "Known held-out stations use their fitted station effect.",
            True
        ],

        [
            "L10",
            "Stations absent from training are marginalized with predictive variance sigma^2 + sigma_station^2.",
            True
        ],

        [
            "L11",
            "Mean test LPD is computed within each held-out campaign/model.",
            True
        ],

        [
            "L12",
            "Primary predictive contrast is mean_test_LPD(HQ) - mean_test_LPD(Q) on identical rows.",
            True
        ],

        [
            "L13",
            "Campaign-equal mean DeltaLPD gives each campaign equal weight regardless of sample count.",
            True
        ],

        [
            "L14",
            "Positive-campaign count is descriptive and not a support gate.",
            True
        ],

        [
            "L15",
            "Final joint support additionally requires frozen full-data Pdirection >= 0.95.",
            True
        ],

        [
            "L16",
            "No post-hoc hydroclimate-window selection is permitted.",
            True
        ],

        [
            "L17",
            "No HxQ interaction is fitted in primary Block 10.",
            True
        ],

        [
            "L18",
            "EC P14 and EC P60 remain separate pre-specified tests; no winner is selected post hoc.",
            True
        ],

        [
            "L19",
            "The thermal root-zone SM90 test remains secondary.",
            True
        ],

        [
            "L20",
            "Incremental predictive information is not interpreted as causal proof.",
            True
        ]
    ],
    columns=[
        "rule_id",
        "rule",
        "frozen"
    ]
)


print("\n" + "=" * 120)
print("PREDICTIVE SCORING CONTRACT")
print("=" * 120)

print(
    scoring_rules.to_string(
        index=False
    )
)


# ==================================================================================================
# 11. CHECKPOINT ARCHITECTURE FOR THE FUTURE 200 FITS
# ==================================================================================================

LOCO_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULL_LOCO"
)

CHECKPOINT_DIR = (
    LOCO_DIR /
    "CHECKPOINTS"
)

FIT_DIR = (
    LOCO_DIR /
    "FIT_SUMMARIES"
)

LOCO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


job_rows = []


for _, fr in fold_manifest.iterrows():

    for model_type in models_expected:

        job_id = (
            f"{fr['fold_id']}_"
            f"{model_type}"
        )

        checkpoint_file = (
            CHECKPOINT_DIR /
            f"{job_id}.json"
        )

        job_rows.append(
            {
                "job_id":
                    job_id,

                "fold_id":
                    fr["fold_id"],

                "response_key":
                    fr["response_key"],

                "hydro_var":
                    fr["hydro_var"],

                "heldout_campaign":
                    fr[
                        "heldout_campaign"
                    ],

                "model_type":
                    model_type,

                "n_train":
                    fr["n_train"],

                "n_test":
                    fr["n_test"],

                "n_test_censored":
                    fr[
                        "n_test_censored"
                    ],

                "checkpoint_file":
                    str(
                        checkpoint_file
                    ),

                "checkpoint_exists_now":
                    checkpoint_file.exists(),

                "job_status":
                    (
                        "EXISTING_CHECKPOINT"
                        if checkpoint_file.exists()
                        else "PENDING"
                    )
            }
        )


job_manifest = pd.DataFrame(
    job_rows
)


print("\n" + "=" * 120)
print("FUTURE LOCO JOB MANIFEST")
print("=" * 120)

print("Folds               :", len(fold_manifest))
print("Models per fold     :", len(models_expected))
print("Total Bayesian fits :", len(job_manifest))
print(
    "Existing checkpoints:",
    int(
        job_manifest[
            "checkpoint_exists_now"
        ].sum()
    )
)
print(
    "Pending fits        :",
    int(
        (
            job_manifest[
                "job_status"
            ]
            == "PENDING"
        ).sum()
    )
)


# ==================================================================================================
# 12. QA
# ==================================================================================================

positive_checks = {
    "five_frozen_pairs":
        len(expected_pairs) == 5,

    "fifty_LOCO_folds":
        len(fold_manifest) == 50,

    "ten_campaigns_per_pair":
        bool(
            (
                fold_manifest
                .groupby(
                    [
                        "response_key",
                        "hydro_var"
                    ]
                )[
                    "heldout_campaign"
                ]
                .nunique()
                == 10
            ).all()
        ),

    "all_test_sets_nonempty":
        bool(
            (
                fold_manifest[
                    "n_test"
                ]
                > 0
            ).all()
        ),

    "all_train_sets_nonempty":
        bool(
            (
                fold_manifest[
                    "n_train"
                ]
                > 0
            ).all()
        ),

    "all_pairs_partition_exactly":
        bool(
            partition_audit[
                "exact_partition"
            ].all()
        ),

    "all_pairs_10_campaigns":
        bool(
            (
                partition_audit[
                    "campaigns"
                ]
                == 10
            ).all()
        ),

    "train_only_Q_scaling_verified":
        bool(
            leakage_audit[
                "at_least_one_Q_training_mean_differs_from_full"
            ].all()
        ),

    "train_only_H_scaling_verified":
        bool(
            leakage_audit[
                "at_least_one_H_training_mean_differs_from_full"
            ].all()
        ),

    "same_rows_all_four_models":
        bool(
            fold_manifest[
                "same_rows_for_NULL_Q_H_HQ"
            ].all()
        ),

    "training_only_scaling":
        bool(
            fold_manifest[
                "training_only_scaling"
            ].all()
        ),

    "observed_Q_only":
        bool(
            fold_manifest[
                "observed_Q_only"
            ].all()
        ),

    "sample_specific_censoring":
        bool(
            fold_manifest[
                "sample_specific_censoring"
            ].all()
        ),

    "twenty_scoring_rules_frozen":
        (
            len(scoring_rules) == 20
            and
            scoring_rules[
                "frozen"
            ].all()
        ),

    "future_jobs_200":
        len(job_manifest) == 200,

    "checkpoint_paths_unique":
        job_manifest[
            "checkpoint_file"
        ].is_unique,

    "job_ids_unique":
        job_manifest[
            "job_id"
        ].is_unique
}


prohibited_states = {
    "models_fit_in_10_2B_A":
        False,

    "window_search_performed":
        False,

    "HxQ_interaction_used":
        False,

    "C09_reconstructed_Q_used":
        False,

    "scientific_classification_performed":
        False,

    "support_gate_changed":
        False
}


qa_rows = []


for check, state in positive_checks.items():

    qa_rows.append(
        {
            "check": check,
            "state": bool(state),
            "expected": True,
            "pass": bool(state)
        }
    )


for check, state in prohibited_states.items():

    qa_rows.append(
        {
            "check": check,
            "state": bool(state),
            "expected": False,
            "pass": not bool(state)
        }
    )


qa = pd.DataFrame(
    qa_rows
)


BLOCK_PASS = bool(
    qa["pass"].all()
)


print("\n" + "=" * 120)
print("BLOCK 10.2B-A QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)


# ==================================================================================================
# 13. SAVE PREFLIGHT FREEZE
# ==================================================================================================

PREFLIGHT_DIR = (
    LOCO_DIR /
    "PREFLIGHT"
)

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FOLD_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_LOCO_FOLD_MANIFEST_v1.csv"
)

PARTITION_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_PARTITION_AUDIT_v1.csv"
)

LEAKAGE_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_TRAIN_ONLY_SCALING_AUDIT_v1.csv"
)

SCORING_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_SCORING_CONTRACT_v1.csv"
)

JOBS_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_200_JOB_MANIFEST_v1.csv"
)

QA_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_PREFLIGHT_QA_v1.csv"
)


fold_manifest.to_csv(
    FOLD_OUT,
    index=False
)

partition_audit.to_csv(
    PARTITION_OUT,
    index=False
)

leakage_audit.to_csv(
    LEAKAGE_OUT,
    index=False
)

scoring_rules.to_csv(
    SCORING_OUT,
    index=False
)

job_manifest.to_csv(
    JOBS_OUT,
    index=False
)

qa.to_csv(
    QA_OUT,
    index=False
)


# ==================================================================================================
# 14. HASH / MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


files_to_hash = [
    FOLD_OUT,
    PARTITION_OUT,
    LEAKAGE_OUT,
    SCORING_OUT,
    JOBS_OUT,
    QA_OUT
]


manifest = {
    "block":
        "10.2B-A",

    "status":
        "PASS"
        if BLOCK_PASS
        else "FAIL",

    "purpose":
        "Freeze campaign-LOCO data, scaling, scoring and restart/checkpoint architecture before Bayesian fitting.",

    "n_frozen_tests":
        5,

    "n_campaigns":
        10,

    "n_folds":
        50,

    "models_per_fold":
        models_expected,

    "planned_bayesian_fits":
        200,

    "models_fit_in_this_block":
        0,

    "primary_predictive_contrast":
        "mean_test_LPD(HQ) - mean_test_LPD(Q)",

    "aggregation":
        "campaign-equal mean across 10 held-out campaigns",

    "final_support_rule":
        "full-data conditional H Pdirection >= 0.95 AND campaign-equal mean DeltaLPD(HQ-Q) > 0",

    "campaign_majority_role":
        "descriptive_only",

    "unseen_station_policy":
        "marginalize with predictive variance sigma^2 + sigma_station^2",

    "window_search":
        False,

    "HxQ_interaction":
        False,

    "C09_reconstructed_Q":
        False,

    "outputs": {
        str(p):
            sha256_file(p)
        for p in files_to_hash
    }
}


MANIFEST_OUT = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_PREFLIGHT_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 15. FINAL GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("BLOCK 10.2B-A FINAL GATE")
print("=" * 120)

print(
    "BLOCK 10.2B-A PASS                :",
    BLOCK_PASS
)

print(
    "FROZEN RESPONSE-H TESTS           :",
    5
)

print(
    "LOCO FOLDS                        :",
    len(fold_manifest)
)

print(
    "PLANNED BAYESIAN FITS             :",
    len(job_manifest)
)

print(
    "MODELS FIT IN THIS BLOCK          :",
    0
)

print(
    "TRAIN-ONLY SCALING                :",
    True
)

print(
    "SAME ROWS NULL/Q/H/HQ             :",
    True
)

print(
    "OBSERVED Q ONLY                   :",
    True
)

print(
    "SAMPLE-SPECIFIC CENSORING         :",
    True
)

print(
    "UNSEEN-STATION MARGINALIZATION    :",
    "FROZEN FOR SCORING"
)

print(
    "WINDOW SEARCH                     :",
    False
)

print(
    "HxQ INTERACTION                   :",
    False
)

print(
    "C09 RECONSTRUCTED Q               :",
    False
)

print(
    "READY FOR BLOCK 10.2B-B FITTING   :",
    BLOCK_PASS
)


print("\nFiles written:")

for p in [
    FOLD_OUT,
    PARTITION_OUT,
    LEAKAGE_OUT,
    SCORING_OUT,
    JOBS_OUT,
    QA_OUT,
    MANIFEST_OUT
]:
    print(" ", p)


assert BLOCK_PASS, (
    "\nBLOCK 10.2B-A failed preflight QA.\n"
    "Do NOT launch the 200 Bayesian fits."
)


print("\n" + "=" * 120)
print("BLOCK 10.2B-A PASS — FULL LOCO CONTRACT FROZEN; ZERO MODELS FIT")
print("=" * 120)

---
### Imported master block 137

`BLOCK 10.2B-B — FULL CAMPAIGN-LOCO BAYESIAN RUNNER`


In [ ]:
# ==================================================================================================
# BLOCK 10.2B-B — FULL CAMPAIGN-LOCO BAYESIAN RUNNER
#
# FINAL COMPUTATIONAL RUN FOR PRIMARY BLOCK 10.2B
#
# 5 frozen response-H tests
# x 10 held-out campaigns
# x 4 models: NULL / Q / H / HQ
# = 200 Bayesian fits
#
# Uses EXACT validated:
#   fit_joint_model()
#   heldout_lpd()
#
# PRIMARY INFERENCE:
#   DeltaLPD(HQ-Q), campaign-equal across 10 campaigns.
#
# COMPUTATIONAL GATE:
#   divergences == 0
#   max Rhat <= 1.01
#   min bulk ESS >= 400
#
# Automatic repair:
#   ONLY failed computational fits are rerun with more draws/tune.
#   No gate relaxation.
#
# RESTART:
#   Each valid fit is checkpointed immediately.
#   Rerunning skips valid completed jobs.
#
# NO:
#   window search
#   HxQ interaction
#   reconstructed C09 Q
#   architecture changes
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import time
import numpy as np
import pandas as pd
import arviz as az

print("=" * 120)
print("BLOCK 10.2B-B — FULL 200-FIT CAMPAIGN-LOCO RUNNER")
print("=" * 120)


# ==================================================================================================
# 0. PREFLIGHT
# ==================================================================================================

assert "folds" in globals()
assert "fold_manifest" in globals()
assert "job_manifest" in globals()

assert "fit_joint_model" in globals()
assert "heldout_lpd" in globals()

assert len(folds) == 50
assert len(fold_manifest) == 50
assert len(job_manifest) == 200

assert "M_STATION" in globals()
assert "M_CAMPAIGN" in globals()

PROJECT = Path(
    "/content/drive/MyDrive/Huanta_JoH_PROJECT"
)

LOCO_DIR = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULL_LOCO"
)

PREFLIGHT_DIR = (
    LOCO_DIR /
    "PREFLIGHT"
)

CHECKPOINT_DIR = (
    LOCO_DIR /
    "CHECKPOINTS"
)

RESULT_DIR = (
    LOCO_DIR /
    "RESULTS"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Verify frozen preflight.
PREFLIGHT_MANIFEST = (
    PREFLIGHT_DIR /
    "BLOCK102B_A_PREFLIGHT_manifest.json"
)

assert PREFLIGHT_MANIFEST.exists()

with open(
    PREFLIGHT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    preflight_manifest = json.load(f)

assert preflight_manifest["status"] == "PASS"
assert preflight_manifest["n_folds"] == 50
assert preflight_manifest["planned_bayesian_fits"] == 200

print("10.2B-A frozen preflight : PASS")
print("Validated fitting engine : FOUND")
print("Validated LOCO scorer    : FOUND")
print("Expected jobs            : 200")


# ==================================================================================================
# 1. FIXED MODEL ORDER
# ==================================================================================================

MODEL_ORDER = [
    "NULL",
    "Q",
    "H",
    "HQ"
]

MODEL_INDEX = {
    m: i
    for i, m in enumerate(MODEL_ORDER)
}


# ==================================================================================================
# 2. DIAGNOSTICS
# ==================================================================================================

def get_fit_diagnostics(fit, model_type):

    idata = fit["idata"]

    variables = [
        "alpha",
        "sigma",
        "sigma_station"
    ]

    if model_type in {
        "Q",
        "HQ"
    }:
        variables.append(
            "beta_Q"
        )

    if model_type in {
        "H",
        "HQ"
    }:
        variables.append(
            "beta_H"
        )

    summary = az.summary(
        idata,
        var_names=variables,
        kind="diagnostics"
    )

    max_rhat = float(
        summary["r_hat"].max()
    )

    min_ess_bulk = float(
        summary["ess_bulk"].min()
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ]
        .values
        .sum()
    )

    diagnostic_pass = bool(
        divergences == 0
        and max_rhat <= 1.01
        and min_ess_bulk >= 400
    )

    return {
        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess_bulk,

        "diagnostic_pass":
            diagnostic_pass
    }


# ==================================================================================================
# 3. CHECKPOINT UTILITIES
# ==================================================================================================

def checkpoint_path(job_id):

    return (
        CHECKPOINT_DIR /
        f"{job_id}.json"
    )


def load_checkpoint(job_id):

    path = checkpoint_path(
        job_id
    )

    if not path.exists():
        return None

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:
            obj = json.load(f)

    except Exception:
        return None

    # Only computationally valid completed checkpoints may be skipped.
    if (
        obj.get("status") == "PASS"
        and
        obj.get("diagnostic_pass") is True
        and
        np.isfinite(
            obj.get(
                "mean_test_lpd",
                np.nan
            )
        )
    ):
        return obj

    return None


def save_checkpoint(job_id, obj):

    path = checkpoint_path(
        job_id
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )

    tmp.replace(
        path
    )


# ==================================================================================================
# 4. POSTERIOR SUMMARY — DESCRIPTIVE ONLY
# ==================================================================================================

def posterior_parameter_summary(
    fit,
    variable
):

    if variable is None:
        return {
            "median": np.nan,
            "low": np.nan,
            "high": np.nan
        }

    post = fit[
        "idata"
    ].posterior

    if variable not in post:
        return {
            "median": np.nan,
            "low": np.nan,
            "high": np.nan
        }

    values = (
        post[
            variable
        ]
        .values
        .reshape(-1)
        .astype(float)
    )

    low, high = np.quantile(
        values,
        [
            0.025,
            0.975
        ]
    )

    return {
        "median":
            float(
                np.median(
                    values
                )
            ),

        "low":
            float(low),

        "high":
            float(high)
    }


# ==================================================================================================
# 5. JOB TABLE IN DETERMINISTIC ORDER
# ==================================================================================================

jobs = job_manifest.copy()

jobs["model_order"] = (
    jobs[
        "model_type"
    ]
    .map(
        MODEL_INDEX
    )
)

jobs = (
    jobs
    .sort_values(
        [
            "fold_id",
            "model_order"
        ]
    )
    .reset_index(
        drop=True
    )
)

assert len(jobs) == 200


# ==================================================================================================
# 6. RECOVER EXISTING VALID CHECKPOINTS
# ==================================================================================================

existing_valid = 0

for job_id in jobs["job_id"]:

    if load_checkpoint(
        job_id
    ) is not None:

        existing_valid += 1


print("\n" + "=" * 120)
print("RESTART STATUS")
print("=" * 120)

print(
    "Valid completed checkpoints :",
    existing_valid
)

print(
    "Jobs remaining              :",
    200 - existing_valid
)


# ==================================================================================================
# 7. RUNNER SETTINGS
#
# Standard validated LOCO settings first.
# Higher-MC repair only when computational gate fails.
# ==================================================================================================

STANDARD = {
    "draws": 1200,
    "tune": 1200,
    "chains": 4,
    "target_accept": 0.95
}

REPAIR = {
    "draws": 2400,
    "tune": 1800,
    "chains": 4,
    "target_accept": 0.97
}


# ==================================================================================================
# 8. RUN ALL 200 JOBS
# ==================================================================================================

session_results = []

start_all = time.time()


for job_number, job in jobs.iterrows():

    job_id = str(
        job["job_id"]
    )

    response = str(
        job["response_key"]
    )

    hydro_var = str(
        job["hydro_var"]
    )

    campaign = str(
        job["heldout_campaign"]
    )

    model_type = str(
        job["model_type"]
    )

    # ----------------------------------------------------------------------------------------------
    # Existing valid checkpoint?
    # ----------------------------------------------------------------------------------------------

    existing = load_checkpoint(
        job_id
    )

    if existing is not None:

        session_results.append(
            existing
        )

        print(
            f"[{job_number+1:03d}/200] "
            f"{job_id:<18} "
            f"SKIP — valid checkpoint "
            f"LPD={existing['mean_test_lpd']:+.5f}"
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Recover exact frozen fold.
    # ----------------------------------------------------------------------------------------------

    fold_key = (
        response,
        hydro_var,
        campaign
    )

    assert fold_key in folds

    fold = folds[
        fold_key
    ]

    train = (
        fold[
            "train"
        ]
        .copy()
    )

    test = (
        fold[
            "test"
        ]
        .copy()
    )

    assert len(train) == int(
        job["n_train"]
    )

    assert len(test) == int(
        job["n_test"]
    )

    # Exact engine contract.
    for col in [
        M_STATION,
        "is_censored_model",
        "Y_z",
        "censor_z",
        "Q_z",
        "H_z"
    ]:

        assert col in train.columns
        assert col in test.columns


    print("\n" + "-" * 120)

    print(
        f"[{job_number+1:03d}/200] "
        f"{job_id}"
    )

    print(
        f"{response} | {hydro_var} | "
        f"held out {campaign} | "
        f"model={model_type}"
    )

    print(
        f"train={len(train)}, "
        f"test={len(test)}, "
        f"test censored="
        f"{int(test['is_censored_model'].astype(bool).sum())}"
    )


    # ----------------------------------------------------------------------------------------------
    # Standard fit.
    # ----------------------------------------------------------------------------------------------

    seed = (
        20263000
        +
        job_number
    )

    t0 = time.time()

    fit = fit_joint_model(
        df_train=train,
        model_type=model_type,
        draws=STANDARD[
            "draws"
        ],
        tune=STANDARD[
            "tune"
        ],
        chains=STANDARD[
            "chains"
        ],
        target_accept=STANDARD[
            "target_accept"
        ],
        seed=seed
    )

    diag = get_fit_diagnostics(
        fit,
        model_type
    )

    repair_used = False


    # ----------------------------------------------------------------------------------------------
    # Surgical higher-MC repair ONLY if computational QA fails.
    # ----------------------------------------------------------------------------------------------

    if not diag[
        "diagnostic_pass"
    ]:

        print(
            "\nCOMPUTATIONAL GATE FAILED:"
        )

        print(
            diag
        )

        print(
            "Surgical higher-MC refit "
            "with identical data/model/priors..."
        )

        repair_used = True

        del fit

        fit = fit_joint_model(
            df_train=train,
            model_type=model_type,
            draws=REPAIR[
                "draws"
            ],
            tune=REPAIR[
                "tune"
            ],
            chains=REPAIR[
                "chains"
            ],
            target_accept=REPAIR[
                "target_accept"
            ],
            seed=seed + 100000
        )

        diag = get_fit_diagnostics(
            fit,
            model_type
        )


    # ----------------------------------------------------------------------------------------------
    # Hard stop if repair still fails.
    #
    # Do NOT continue and do NOT relax gates.
    # ----------------------------------------------------------------------------------------------

    if not diag[
        "diagnostic_pass"
    ]:

        failure = {
            "job_id":
                job_id,

            "status":
                "FAIL",

            "response_key":
                response,

            "hydro_var":
                hydro_var,

            "heldout_campaign":
                campaign,

            "model_type":
                model_type,

            "diagnostic_pass":
                False,

            "divergences":
                diag[
                    "divergences"
                ],

            "max_rhat":
                diag[
                    "max_rhat"
                ],

            "min_ess_bulk":
                diag[
                    "min_ess_bulk"
                ],

            "repair_used":
                repair_used
        }

        # Save failure checkpoint so we know exactly where it happened,
        # but load_checkpoint() will NOT treat it as valid.
        save_checkpoint(
            job_id,
            failure
        )

        raise RuntimeError(
            f"\n{job_id} failed diagnostics "
            f"even after surgical repair.\n"
            f"No gate relaxation permitted.\n"
            f"Rerun later after inspecting this single fit."
        )


    # ----------------------------------------------------------------------------------------------
    # Exact validated held-out predictive scorer.
    # ----------------------------------------------------------------------------------------------

    point_lpd = heldout_lpd(
        fit=fit,
        test_df=test,
        model_type=model_type
    )

    assert len(
        point_lpd
    ) == len(test)

    assert np.isfinite(
        point_lpd
    ).all()

    mean_test_lpd = float(
        np.mean(
            point_lpd
        )
    )

    sum_test_lpd = float(
        np.sum(
            point_lpd
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Descriptive fold posterior coefficients.
    # ----------------------------------------------------------------------------------------------

    beta_Q_summary = (
        posterior_parameter_summary(
            fit,
            "beta_Q"
        )
        if model_type in {
            "Q",
            "HQ"
        }
        else {
            "median": np.nan,
            "low": np.nan,
            "high": np.nan
        }
    )

    beta_H_summary = (
        posterior_parameter_summary(
            fit,
            "beta_H"
        )
        if model_type in {
            "H",
            "HQ"
        }
        else {
            "median": np.nan,
            "low": np.nan,
            "high": np.nan
        }
    )


    # ----------------------------------------------------------------------------------------------
    # Station audit.
    # ----------------------------------------------------------------------------------------------

    train_stations = set(
        train[
            M_STATION
        ]
        .astype(str)
    )

    test_stations = set(
        test[
            M_STATION
        ]
        .astype(str)
    )

    unseen_stations = sorted(
        test_stations
        -
        train_stations
    )


    elapsed = float(
        time.time()
        -
        t0
    )


    # ----------------------------------------------------------------------------------------------
    # Checkpoint.
    # ----------------------------------------------------------------------------------------------

    result = {
        "job_id":
            job_id,

        "status":
            "PASS",

        "fold_id":
            fold[
                "fold_id"
            ],

        "response_key":
            response,

        "hydro_var":
            hydro_var,

        "heldout_campaign":
            campaign,

        "model_type":
            model_type,

        "n_train":
            int(
                len(train)
            ),

        "n_test":
            int(
                len(test)
            ),

        "n_train_censored":
            int(
                train[
                    "is_censored_model"
                ]
                .astype(bool)
                .sum()
            ),

        "n_test_censored":
            int(
                test[
                    "is_censored_model"
                ]
                .astype(bool)
                .sum()
            ),

        "n_train_stations":
            int(
                len(
                    train_stations
                )
            ),

        "n_test_stations":
            int(
                len(
                    test_stations
                )
            ),

        "n_unseen_test_stations":
            int(
                len(
                    unseen_stations
                )
            ),

        "unseen_test_stations":
            "|".join(
                unseen_stations
            ),

        "mean_test_lpd":
            mean_test_lpd,

        "sum_test_lpd":
            sum_test_lpd,

        "beta_Q_median_std":
            beta_Q_summary[
                "median"
            ],

        "beta_Q_hdi_low_std":
            beta_Q_summary[
                "low"
            ],

        "beta_Q_hdi_high_std":
            beta_Q_summary[
                "high"
            ],

        "beta_H_median_std":
            beta_H_summary[
                "median"
            ],

        "beta_H_hdi_low_std":
            beta_H_summary[
                "low"
            ],

        "beta_H_hdi_high_std":
            beta_H_summary[
                "high"
            ],

        "divergences":
            int(
                diag[
                    "divergences"
                ]
            ),

        "max_rhat":
            float(
                diag[
                    "max_rhat"
                ]
            ),

        "min_ess_bulk":
            float(
                diag[
                    "min_ess_bulk"
                ]
            ),

        "diagnostic_pass":
            True,

        "repair_used":
            bool(
                repair_used
            ),

        "sampling_draws":
            (
                REPAIR[
                    "draws"
                ]
                if repair_used
                else STANDARD[
                    "draws"
                ]
            ),

        "sampling_tune":
            (
                REPAIR[
                    "tune"
                ]
                if repair_used
                else STANDARD[
                    "tune"
                ]
            ),

        "target_accept":
            (
                REPAIR[
                    "target_accept"
                ]
                if repair_used
                else STANDARD[
                    "target_accept"
                ]
            ),

        "elapsed_seconds":
            elapsed,

        "scorer":
            "VALIDATED_10.1B_HELDOUT_LPD",

        "observed_Q_only":
            True,

        "training_only_scaling":
            True,

        "C09_reconstructed_Q_used":
            False,

        "HxQ_interaction_used":
            False
    }


    save_checkpoint(
        job_id,
        result
    )

    session_results.append(
        result
    )


    # ----------------------------------------------------------------------------------------------
    # Running consolidated CSV.
    # ----------------------------------------------------------------------------------------------

    running_df = pd.DataFrame(
        session_results
    )

    running_df.to_csv(
        RESULT_DIR /
        "BLOCK102B_B_RUNNING_LOCO_RESULTS_v1.csv",
        index=False
    )


    print(
        f"PASS | "
        f"LPD={mean_test_lpd:+.6f} | "
        f"div={diag['divergences']} | "
        f"Rhat={diag['max_rhat']:.3f} | "
        f"ESS={diag['min_ess_bulk']:.0f} | "
        f"repair={repair_used} | "
        f"{elapsed/60:.2f} min"
    )


    # Release large posterior object.
    del fit


# ==================================================================================================
# 9. RELOAD ALL 200 CHECKPOINTS FROM DISK
#
# Do not trust only current-session memory.
# ==================================================================================================

all_results = []

for _, job in jobs.iterrows():

    job_id = str(
        job["job_id"]
    )

    obj = load_checkpoint(
        job_id
    )

    assert obj is not None, (
        f"Missing/invalid checkpoint: {job_id}"
    )

    all_results.append(
        obj
    )


results = pd.DataFrame(
    all_results
)

assert len(
    results
) == 200

assert results[
    "job_id"
].is_unique

assert results[
    "diagnostic_pass"
].all()


# ==================================================================================================
# 10. FIT-LEVEL QA
# ==================================================================================================

print("\n" + "=" * 120)
print("ALL 200 FITS COMPLETE — FIT QA")
print("=" * 120)

print(
    "Fits                     :",
    len(results)
)

print(
    "Zero-divergence fits     :",
    int(
        (
            results[
                "divergences"
            ]
            == 0
        ).sum()
    )
)

print(
    "Rhat <= 1.01            :",
    int(
        (
            results[
                "max_rhat"
            ]
            <= 1.01
        ).sum()
    )
)

print(
    "ESS >= 400              :",
    int(
        (
            results[
                "min_ess_bulk"
            ]
            >= 400
        ).sum()
    )
)

print(
    "Surgical repairs used   :",
    int(
        results[
            "repair_used"
        ].sum()
    )
)


# ==================================================================================================
# 11. PIVOT FOUR MODEL LPDs WITHIN EXACT FOLD
# ==================================================================================================

lpd_wide = (
    results
    .pivot(
        index=[
            "fold_id",
            "response_key",
            "hydro_var",
            "heldout_campaign"
        ],
        columns="model_type",
        values="mean_test_lpd"
    )
    .reset_index()
)

assert len(
    lpd_wide
) == 50

for col in MODEL_ORDER:
    assert col in lpd_wide.columns
    assert lpd_wide[col].notna().all()


# ==================================================================================================
# 12. COMPUTE ALL PRE-FROZEN CONTRASTS
# ==================================================================================================

lpd_wide[
    "delta_Q_minus_NULL"
] = (
    lpd_wide[
        "Q"
    ]
    -
    lpd_wide[
        "NULL"
    ]
)

lpd_wide[
    "delta_H_minus_NULL"
] = (
    lpd_wide[
        "H"
    ]
    -
    lpd_wide[
        "NULL"
    ]
)

lpd_wide[
    "delta_HQ_minus_Q"
] = (
    lpd_wide[
        "HQ"
    ]
    -
    lpd_wide[
        "Q"
    ]
)

lpd_wide[
    "delta_HQ_minus_H"
] = (
    lpd_wide[
        "HQ"
    ]
    -
    lpd_wide[
        "H"
    ]
)


# ==================================================================================================
# 13. CAMPAIGN-EQUAL SUMMARY
# ==================================================================================================

summary_rows = []

for (
    response,
    hydro_var
), sub in lpd_wide.groupby(
    [
        "response_key",
        "hydro_var"
    ],
    sort=False
):

    assert len(
        sub
    ) == 10

    summary_rows.append(
        {
            "response_key":
                response,

            "hydro_var":
                hydro_var,

            "n_campaigns":
                10,

            "mean_delta_Q_minus_NULL":
                float(
                    sub[
                        "delta_Q_minus_NULL"
                    ].mean()
                ),

            "median_delta_Q_minus_NULL":
                float(
                    sub[
                        "delta_Q_minus_NULL"
                    ].median()
                ),

            "campaigns_Q_better_NULL":
                int(
                    (
                        sub[
                            "delta_Q_minus_NULL"
                        ]
                        > 0
                    ).sum()
                ),

            "mean_delta_H_minus_NULL":
                float(
                    sub[
                        "delta_H_minus_NULL"
                    ].mean()
                ),

            "median_delta_H_minus_NULL":
                float(
                    sub[
                        "delta_H_minus_NULL"
                    ].median()
                ),

            "campaigns_H_better_NULL":
                int(
                    (
                        sub[
                            "delta_H_minus_NULL"
                        ]
                        > 0
                    ).sum()
                ),

            # PRIMARY BLOCK-10 PREDICTIVE CONTRAST
            "mean_delta_HQ_minus_Q":
                float(
                    sub[
                        "delta_HQ_minus_Q"
                    ].mean()
                ),

            "median_delta_HQ_minus_Q":
                float(
                    sub[
                        "delta_HQ_minus_Q"
                    ].median()
                ),

            "campaigns_HQ_better_Q":
                int(
                    (
                        sub[
                            "delta_HQ_minus_Q"
                        ]
                        > 0
                    ).sum()
                ),

            "mean_delta_HQ_minus_H":
                float(
                    sub[
                        "delta_HQ_minus_H"
                    ].mean()
                ),

            "median_delta_HQ_minus_H":
                float(
                    sub[
                        "delta_HQ_minus_H"
                    ].median()
                ),

            "campaigns_HQ_better_H":
                int(
                    (
                        sub[
                            "delta_HQ_minus_H"
                        ]
                        > 0
                    ).sum()
                )
        }
    )


summary = pd.DataFrame(
    summary_rows
)

assert len(
    summary
) == 5


# ==================================================================================================
# 14. JOIN FROZEN FULL-DATA CONDITIONAL H POSTERIORS
# ==================================================================================================

FULLDATA_RESULTS = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULLDATA_POSTERIOR" /
    "BLOCK102A_R2_FULLDATA_CONDITIONAL_HQ_POSTERIOR_SPECTRUM_v1.csv"
)

fullpost = pd.read_csv(
    FULLDATA_RESULTS,
    keep_default_na=False
)

assert len(
    fullpost
) == 5


join_cols = [
    "response_key",
    "hydro_var",
    "joint_analysis_role",
    "frozen_expected_H_direction",
    "beta_H_conditional_median_std",
    "beta_H_conditional_hdi_low_std",
    "beta_H_conditional_hdi_high_std",
    "beta_H_conditional_Pdirection",
    "posterior_directional_gate"
]


final = summary.merge(
    fullpost[
        join_cols
    ],
    on=[
        "response_key",
        "hydro_var"
    ],
    how="left",
    validate="one_to_one"
)

assert len(
    final
) == 5

assert final[
    "beta_H_conditional_Pdirection"
].notna().all()


# ==================================================================================================
# 15. APPLY IMMUTABLE SUPPORT RULE
#
# HQ04:
#   conditional H Pdirection >= .95
#
# HQ05:
#   campaign-equal mean DeltaLPD(HQ-Q) > 0
#
# HQ06:
#   BOTH required.
# ==================================================================================================

final[
    "posterior_directional_gate"
] = (
    pd.to_numeric(
        final[
            "beta_H_conditional_Pdirection"
        ],
        errors="raise"
    )
    >= 0.95
)

final[
    "predictive_increment_gate"
] = (
    final[
        "mean_delta_HQ_minus_Q"
    ]
    > 0
)

final[
    "joint_support"
] = (
    final[
        "posterior_directional_gate"
    ]
    &
    final[
        "predictive_increment_gate"
    ]
)


def classify_joint(row):

    if bool(
        row[
            "joint_support"
        ]
    ):

        if (
            row[
                "joint_analysis_role"
            ]
            ==
            "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST"
        ):

            return (
                "SUPPORTED_INCREMENTAL_ANTECEDENT_INFORMATION_BEYOND_Q"
            )

        else:

            return (
                "SUPPORTED_SECONDARY_THERMAL_REFERENCE_BEYOND_Q"
            )

    if (
        bool(
            row[
                "posterior_directional_gate"
            ]
        )
        and
        not bool(
            row[
                "predictive_increment_gate"
            ]
        )
    ):

        return (
            "CONDITIONAL_ASSOCIATION_WITHOUT_OUT_OF_SAMPLE_INCREMENT"
        )

    if (
        not bool(
            row[
                "posterior_directional_gate"
            ]
        )
        and
        bool(
            row[
                "predictive_increment_gate"
            ]
        )
    ):

        return (
            "PREDICTIVE_INCREMENT_WITHOUT_DIRECTIONAL_POSTERIOR_SUPPORT"
        )

    return (
        "NO_ROBUST_INCREMENTAL_SUPPORT"
    )


final[
    "joint_support_class"
] = final.apply(
    classify_joint,
    axis=1
)


# ==================================================================================================
# 16. PRINT SCIENTIFIC RESULT
# ==================================================================================================

print("\n" + "=" * 120)
print("BLOCK 10.2B-B — FINAL CAMPAIGN-LOCO INCREMENTAL PREDICTION SPECTRUM")
print("=" * 120)

display_cols = [
    "response_key",
    "hydro_var",
    "joint_analysis_role",
    "beta_H_conditional_median_std",
    "beta_H_conditional_hdi_low_std",
    "beta_H_conditional_hdi_high_std",
    "beta_H_conditional_Pdirection",
    "mean_delta_HQ_minus_Q",
    "median_delta_HQ_minus_Q",
    "campaigns_HQ_better_Q",
    "posterior_directional_gate",
    "predictive_increment_gate",
    "joint_support",
    "joint_support_class"
]

print(
    final[
        display_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 17. PRINT ALL 50 CAMPAIGN CONTRASTS
# ==================================================================================================

print("\n" + "=" * 120)
print("ALL 50 HELD-OUT CAMPAIGN CONTRASTS")
print("=" * 120)

print(
    lpd_wide[
        [
            "fold_id",
            "response_key",
            "hydro_var",
            "heldout_campaign",
            "NULL",
            "Q",
            "H",
            "HQ",
            "delta_Q_minus_NULL",
            "delta_H_minus_NULL",
            "delta_HQ_minus_Q",
            "delta_HQ_minus_H"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 18. FINAL QA
# ==================================================================================================

qa_checks = {
    "fits_exactly_200":
        len(results) == 200,

    "jobs_unique_200":
        (
            results[
                "job_id"
            ].nunique()
            == 200
        ),

    "all_fit_diagnostics_pass":
        bool(
            results[
                "diagnostic_pass"
            ].all()
        ),

    "all_zero_divergences":
        bool(
            (
                results[
                    "divergences"
                ]
                == 0
            ).all()
        ),

    "all_Rhat_le_1_01":
        bool(
            (
                results[
                    "max_rhat"
                ]
                <= 1.01
            ).all()
        ),

    "all_ESS_ge_400":
        bool(
            (
                results[
                    "min_ess_bulk"
                ]
                >= 400
            ).all()
        ),

    "fifty_complete_folds":
        len(
            lpd_wide
        ) == 50,

    "four_models_each_fold":
        bool(
            (
                results
                .groupby(
                    "fold_id"
                )[
                    "model_type"
                ]
                .nunique()
                == 4
            ).all()
        ),

    "five_final_tests":
        len(
            final
        ) == 5,

    "ten_campaigns_each_test":
        bool(
            (
                final[
                    "n_campaigns"
                ]
                == 10
            ).all()
        ),

    "full_data_directional_gate_preserved":
        bool(
            final[
                "posterior_directional_gate"
            ].all()
        ),

    "all_LPD_finite":
        bool(
            np.isfinite(
                results[
                    "mean_test_lpd"
                ]
            ).all()
        ),

    "observed_Q_only":
        bool(
            results[
                "observed_Q_only"
            ].all()
        ),

    "training_only_scaling":
        bool(
            results[
                "training_only_scaling"
            ].all()
        )
}


prohibited_states = {
    "window_search_performed":
        False,

    "HxQ_interaction_used":
        False,

    "C09_reconstructed_Q_used":
        False,

    "support_gate_relaxed":
        False,

    "architecture_changed":
        False
}


qa_rows = []

for check, state in qa_checks.items():

    qa_rows.append(
        {
            "check":
                check,

            "state":
                bool(
                    state
                ),

            "expected":
                True,

            "pass":
                bool(
                    state
                )
        }
    )


for check, state in prohibited_states.items():

    qa_rows.append(
        {
            "check":
                check,

            "state":
                bool(
                    state
                ),

            "expected":
                False,

            "pass":
                not bool(
                    state
                )
        }
    )


qa_final = pd.DataFrame(
    qa_rows
)

BLOCK_PASS = bool(
    qa_final[
        "pass"
    ].all()
)


print("\n" + "=" * 120)
print("BLOCK 10.2B-B FINAL QA")
print("=" * 120)

print(
    qa_final.to_string(
        index=False
    )
)


# ==================================================================================================
# 19. SAVE FINAL OUTPUTS
# ==================================================================================================

FIT_RESULTS_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_ALL_200_LOCO_FITS_v1.csv"
)

FOLD_RESULTS_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_ALL_50_LOCO_CONTRASTS_v1.csv"
)

SUMMARY_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_CAMPAIGN_EQUAL_INCREMENTAL_SUMMARY_v1.csv"
)

FINAL_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_FINAL_JOINT_SUPPORT_SPECTRUM_v1.csv"
)

QA_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_FINAL_QA_v1.csv"
)


results.to_csv(
    FIT_RESULTS_OUT,
    index=False
)

lpd_wide.to_csv(
    FOLD_RESULTS_OUT,
    index=False
)

summary.to_csv(
    SUMMARY_OUT,
    index=False
)

final.to_csv(
    FINAL_OUT,
    index=False
)

qa_final.to_csv(
    QA_OUT,
    index=False
)


# ==================================================================================================
# 20. HASH MANIFEST
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


files_final = [
    FIT_RESULTS_OUT,
    FOLD_RESULTS_OUT,
    SUMMARY_OUT,
    FINAL_OUT,
    QA_OUT
]


manifest = {
    "block":
        "10.2B-B",

    "status":
        (
            "PASS"
            if BLOCK_PASS
            else "FAIL"
        ),

    "n_tests":
        5,

    "n_campaigns":
        10,

    "n_folds":
        50,

    "n_models_per_fold":
        4,

    "n_bayesian_fits":
        200,

    "model_types":
        MODEL_ORDER,

    "validated_fit_engine":
        "fit_joint_model",

    "validated_predictive_scorer":
        "heldout_lpd",

    "primary_contrast":
        "campaign-equal mean DeltaLPD(HQ-Q)",

    "support_rule":
        "Pdirection(beta_H|Q)>=0.95 AND mean DeltaLPD(HQ-Q)>0",

    "positive_campaign_count":
        "descriptive_only",

    "observed_Q_only":
        True,

    "training_only_scaling":
        True,

    "C09_reconstructed_Q":
        False,

    "window_search":
        False,

    "HxQ_interaction":
        False,

    "gate_relaxation":
        False,

    "surgical_higher_MC_repair_allowed":
        True,

    "outputs": {
        str(p):
            sha256_file(
                p
            )
        for p in files_final
    }
}


MANIFEST_OUT = (
    RESULT_DIR /
    "BLOCK102B_B_FINAL_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 21. FINAL GATE
# ==================================================================================================

elapsed_all = (
    time.time()
    -
    start_all
)


print("\n" + "=" * 120)
print("BLOCK 10.2B-B FINAL GATE")
print("=" * 120)

print(
    "BLOCK 10.2B-B PASS                         :",
    BLOCK_PASS
)

print(
    "VALID BAYESIAN FITS                        :",
    len(
        results
    ),
    "/ 200"
)

print(
    "LOCO FOLDS                                 :",
    len(
        lpd_wide
    ),
    "/ 50"
)

print(
    "SURGICAL HIGHER-MC REPAIRS                 :",
    int(
        results[
            "repair_used"
        ].sum()
    )
)

print(
    "PRIMARY HYDROCHEMICAL TESTS SUPPORTED      :",
    int(
        (
            (
                final[
                    "joint_analysis_role"
                ]
                ==
                "PRIMARY_HYDROCHEMICAL_INCREMENTAL_TEST"
            )
            &
            final[
                "joint_support"
            ]
        ).sum()
    ),
    "/ 4"
)

print(
    "SECONDARY THERMAL REFERENCE SUPPORTED      :",
    bool(
        final.loc[
            final[
                "joint_analysis_role"
            ]
            ==
            "SECONDARY_THERMAL_REFERENCE",
            "joint_support"
        ].iloc[0]
    )
)

print(
    "WINDOW SEARCH PERFORMED                    :",
    False
)

print(
    "HxQ INTERACTION USED                       :",
    False
)

print(
    "C09 RECONSTRUCTED Q USED                   :",
    False
)

print(
    "SUPPORT GATE RELAXED                       :",
    False
)

print(
    "TOTAL WALL TIME THIS SESSION (hours)       :",
    round(
        elapsed_all / 3600,
        3
    )
)

print("\nFinal files:")

for p in [
    FIT_RESULTS_OUT,
    FOLD_RESULTS_OUT,
    SUMMARY_OUT,
    FINAL_OUT,
    QA_OUT,
    MANIFEST_OUT
]:
    print(" ", p)


assert BLOCK_PASS, (
    "\nBLOCK 10.2B-B failed final QA.\n"
    "Do NOT freeze scientific inference."
)


print("\n" + "=" * 120)
print("BLOCK 10.2B-B PASS — PRIMARY JOINT H+Q CAMPAIGN-LOCO INFERENCE COMPLETE")
print("=" * 120)

---
### Imported master block 142

`BLOCK 11.0B — RECOVER FROZEN SPATIAL / NETWORK ARCHITECTURE`


In [ ]:
# ==================================================================================================
# BLOCK 11.0B — RECOVER FROZEN SPATIAL / NETWORK ARCHITECTURE
#
# PURPOSE
# -------
# Recover existing station geometry, network roles and previous MERIT/UPA information.
# Determine whether a usable contributing-area proxy already exists.
#
# ZERO MODELS
# ZERO GEE
# ZERO DEM DELINEATION
# ZERO RESPONSE/HYDRO SELECTION
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

print("=" * 120)
print("BLOCK 11.0B — RECOVER FROZEN SPATIAL / NETWORK ARCHITECTURE")
print("=" * 120)

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

SNAP = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110"
)

MASTER_DIR = SNAP / "03_MASTER"
AUDIT_DIR  = SNAP / "01_AUDIT"

OUT = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "SPATIAL_RECOVERY"
)
OUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 1. REQUIRED FROZEN SPATIAL FILES
# --------------------------------------------------------------------------------------------------

paths = {
    "network":
        MASTER_DIR / "STATION_NETWORK_FROZEN_v1.csv",

    "geometry":
        MASTER_DIR / "STATION_GEOMETRY_FINAL_v2.csv",

    "spatial_master":
        MASTER_DIR / "MASTER_SPATIAL_RESOLVED_v1.csv",

    "merit_upa":
        AUDIT_DIR / "BLOCK80_MERIT_upa_by_radius.csv",

    "merit_points":
        AUDIT_DIR / "BLOCK80_MERIT_point_diagnostic.csv",

    "snap_summary":
        AUDIT_DIR / "BLOCK81A_station_snap_summary.csv"
}

print("\nFrozen spatial files:")

for k, p in paths.items():
    print(f"{k:<20}: {p.exists()}  {p}")

assert paths["network"].exists()
assert paths["geometry"].exists()

# --------------------------------------------------------------------------------------------------
# 2. LOAD
# --------------------------------------------------------------------------------------------------

tables = {}

for name, p in paths.items():
    if p.exists():
        tables[name] = pd.read_csv(p, low_memory=False)

print("\n" + "=" * 120)
print("A. FILE SHAPES / SCHEMAS")
print("=" * 120)

for name, df in tables.items():

    print(f"\n[{name}] shape={df.shape}")
    print("columns:")
    for c in df.columns:
        print("  -", c)

# --------------------------------------------------------------------------------------------------
# 3. NETWORK ARCHITECTURE
# --------------------------------------------------------------------------------------------------

network = tables["network"].copy()

print("\n" + "=" * 120)
print("B. FROZEN NETWORK ARCHITECTURE")
print("=" * 120)

show_cols = [
    c for c in [
        "station_id",
        "station_type",
        "network_role",
        "latitude",
        "longitude",
        "field_elevation",
        "coordinate_stability",
        "coordinate_valid"
    ]
    if c in network.columns
]

print(
    network[show_cols]
    .sort_values("station_id")
    .to_string(index=False)
)

print("\nNetwork-role counts:")

if "network_role" in network.columns:
    print(
        network["network_role"]
        .fillna("MISSING")
        .value_counts(dropna=False)
        .to_string()
    )

print("\nStation-type counts:")

if "station_type" in network.columns:
    print(
        network["station_type"]
        .fillna("MISSING")
        .value_counts(dropna=False)
        .to_string()
    )

# --------------------------------------------------------------------------------------------------
# 4. GEOMETRY ELIGIBILITY
# --------------------------------------------------------------------------------------------------

geometry = tables["geometry"].copy()

print("\n" + "=" * 120)
print("C. FROZEN GEOMETRY STATUS")
print("=" * 120)

for c in [
    "eligible_for_catchment_inference",
    "geometry_relationship",
    "spatial_reach_group",
    "coordinate_valid"
]:
    if c in geometry.columns:

        print(f"\n{c}:")
        print(
            geometry[c]
            .fillna("MISSING")
            .value_counts(dropna=False)
            .to_string()
        )

# --------------------------------------------------------------------------------------------------
# 5. SEARCH ALL LOADED TABLES FOR AREA / UPA / CONNECTIVITY VARIABLES
# --------------------------------------------------------------------------------------------------

candidate_rows = []

pattern = (
    r"upa|upstream|contribut|drain|catch|area|"
    r"flow|accum|network|order|strahler|shreve|"
    r"distance|snap|reach"
)

for source, df in tables.items():

    for c in df.columns:

        if re.search(pattern, str(c), flags=re.I):

            s = df[c]

            numeric = pd.to_numeric(
                s,
                errors="coerce"
            )

            candidate_rows.append({
                "source":
                    source,

                "column":
                    c,

                "dtype":
                    str(s.dtype),

                "n_nonmissing":
                    int(s.notna().sum()),

                "n_unique":
                    int(s.dropna().nunique()),

                "numeric_n":
                    int(numeric.notna().sum()),

                "numeric_min":
                    (
                        float(numeric.min())
                        if numeric.notna().any()
                        else np.nan
                    ),

                "numeric_median":
                    (
                        float(numeric.median())
                        if numeric.notna().any()
                        else np.nan
                    ),

                "numeric_max":
                    (
                        float(numeric.max())
                        if numeric.notna().any()
                        else np.nan
                    ),

                "examples":
                    " | ".join(
                        s.dropna()
                        .astype(str)
                        .unique()[:6]
                    )
            })

candidate = pd.DataFrame(candidate_rows)

print("\n" + "=" * 120)
print("D. EXISTING CANDIDATE CONNECTIVITY / CONTRIBUTING-AREA VARIABLES")
print("=" * 120)

if len(candidate):
    print(candidate.to_string(index=False))
else:
    print("NONE")

# --------------------------------------------------------------------------------------------------
# 6. INSPECT MERIT UPA TABLE IN DETAIL
# --------------------------------------------------------------------------------------------------

upa_station_summary = pd.DataFrame()

if "merit_upa" in tables:

    upa = tables["merit_upa"].copy()

    print("\n" + "=" * 120)
    print("E. EXISTING MERIT UPA DIAGNOSTIC")
    print("=" * 120)

    print(upa.to_string(index=False))

    # Identify station column.
    station_candidates = [
        c for c in upa.columns
        if re.search(r"^station_id$|station", str(c), flags=re.I)
    ]

    # Identify possible UPA numeric columns.
    upa_candidates = [
        c for c in upa.columns
        if re.search(
            r"upa|upstream|contribut|drain.*area|catch.*area",
            str(c),
            flags=re.I
        )
    ]

    print("\nStation candidate columns :", station_candidates)
    print("UPA candidate columns     :", upa_candidates)

    if station_candidates and upa_candidates:

        sid = station_candidates[0]

        summary_rows = []

        for station, g in upa.groupby(sid, dropna=False):

            row = {
                "station_id": station
            }

            for c in upa_candidates:

                x = pd.to_numeric(
                    g[c],
                    errors="coerce"
                ).dropna()

                row[f"{c}__n"] = int(len(x))

                row[f"{c}__min"] = (
                    float(x.min())
                    if len(x)
                    else np.nan
                )

                row[f"{c}__median"] = (
                    float(x.median())
                    if len(x)
                    else np.nan
                )

                row[f"{c}__max"] = (
                    float(x.max())
                    if len(x)
                    else np.nan
                )

            summary_rows.append(row)

        upa_station_summary = pd.DataFrame(summary_rows)

        print("\nStation-level UPA summary:")
        print(upa_station_summary.to_string(index=False))

# --------------------------------------------------------------------------------------------------
# 7. BUILD FROZEN STATION SPATIAL BASE
# --------------------------------------------------------------------------------------------------

base_cols = [
    c for c in [
        "station_id",
        "station_type",
        "network_role",
        "latitude",
        "longitude",
        "field_elevation",
        "coordinate_stability",
        "coordinate_valid"
    ]
    if c in network.columns
]

station_base = (
    network[base_cols]
    .drop_duplicates(subset=["station_id"])
    .copy()
)

# Add geometry metadata without overwriting frozen network values.
geom_cols = [
    c for c in [
        "station_id",
        "eligible_for_catchment_inference",
        "spatial_reach_group",
        "geometry_relationship",
        "dem_latitude",
        "dem_longitude"
    ]
    if c in geometry.columns
]

if "station_id" in geom_cols:

    geom_small = (
        geometry[geom_cols]
        .drop_duplicates(subset=["station_id"])
    )

    station_base = station_base.merge(
        geom_small,
        on="station_id",
        how="left",
        validate="one_to_one"
    )

print("\n" + "=" * 120)
print("F. RECOVERED STATION SPATIAL BASE")
print("=" * 120)

print(station_base.to_string(index=False))

# --------------------------------------------------------------------------------------------------
# 8. DECISION: DO WE ALREADY HAVE A NUMERIC CONTRIBUTING-AREA CANDIDATE?
# --------------------------------------------------------------------------------------------------

numeric_area_candidates = []

if len(candidate):

    for _, r in candidate.iterrows():

        if (
            re.search(
                r"upa|upstream|contribut|drain.*area|catch.*area",
                str(r["column"]),
                flags=re.I
            )
            and
            r["numeric_n"] >= 5
        ):

            numeric_area_candidates.append({
                "source":
                    r["source"],

                "column":
                    r["column"],

                "numeric_n":
                    int(r["numeric_n"])
            })

area_candidate_df = pd.DataFrame(
    numeric_area_candidates
)

AREA_ALREADY_PRESENT = (
    len(area_candidate_df) > 0
)

if AREA_ALREADY_PRESENT:

    next_action = (
        "AUDIT_EXISTING_MERIT_UPA_AND_FREEZE_MINIMAL_CONNECTIVITY_PREDICTORS"
    )

else:

    next_action = (
        "ONE_MINIMAL_SPATIAL_EXTRACTION_REQUIRED_FOR_CONTRIBUTING_AREA"
    )

# --------------------------------------------------------------------------------------------------
# 9. QA
# --------------------------------------------------------------------------------------------------

qa = pd.DataFrame([
    {
        "check":
            "network_freeze_found",
        "state":
            paths["network"].exists(),
        "expected":
            True
    },
    {
        "check":
            "geometry_freeze_found",
        "state":
            paths["geometry"].exists(),
        "expected":
            True
    },
    {
        "check":
            "station_base_nonempty",
        "state":
            len(station_base) > 0,
        "expected":
            True
    },
    {
        "check":
            "models_fitted",
        "state":
            False,
        "expected":
            False
    },
    {
        "check":
            "new_spatial_extraction",
        "state":
            False,
        "expected":
            False
    },
    {
        "check":
            "H3_tested",
        "state":
            False,
        "expected":
            False
    }
])

qa["pass"] = qa["state"] == qa["expected"]

BLOCK_PASS = bool(qa["pass"].all())

# --------------------------------------------------------------------------------------------------
# 10. SAVE
# --------------------------------------------------------------------------------------------------

candidate.to_csv(
    OUT / "BLOCK110B_EXISTING_CONNECTIVITY_VARIABLES_v1.csv",
    index=False
)

station_base.to_csv(
    OUT / "BLOCK110B_STATION_SPATIAL_BASE_v1.csv",
    index=False
)

area_candidate_df.to_csv(
    OUT / "BLOCK110B_AREA_CANDIDATES_v1.csv",
    index=False
)

if len(upa_station_summary):

    upa_station_summary.to_csv(
        OUT / "BLOCK110B_EXISTING_MERIT_UPA_STATION_SUMMARY_v1.csv",
        index=False
    )

qa.to_csv(
    OUT / "BLOCK110B_QA_v1.csv",
    index=False
)

manifest = {
    "block":
        "11.0B",

    "status":
        "PASS" if BLOCK_PASS else "FAIL",

    "n_station_base":
        int(len(station_base)),

    "existing_merit_upa_file":
        bool(paths["merit_upa"].exists()),

    "numeric_contributing_area_candidate_present":
        bool(AREA_ALREADY_PRESENT),

    "area_candidates":
        numeric_area_candidates,

    "next_action":
        next_action,

    "models_fitted":
        0,

    "new_spatial_extraction":
        False,

    "H3_tested":
        False
}

with open(
    OUT / "BLOCK110B_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# --------------------------------------------------------------------------------------------------
# 11. FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("BLOCK 11.0B FINAL GATE")
print("=" * 120)

print("BLOCK PASS                         :", BLOCK_PASS)
print("STATIONS IN SPATIAL BASE           :", len(station_base))
print("EXISTING MERIT UPA FILE            :", paths["merit_upa"].exists())
print("NUMERIC AREA CANDIDATE PRESENT     :", AREA_ALREADY_PRESENT)

if len(area_candidate_df):
    print("\nAREA CANDIDATES:")
    print(area_candidate_df.to_string(index=False))

print("\nNEXT ACTION:")
print(next_action)

print("\nMODELS FIT                          : 0")
print("NEW SPATIAL EXTRACTION              : False")
print("H3 TESTED                           : False")

assert BLOCK_PASS

print("\n" + "=" * 120)
print("BLOCK 11.0B PASS — EXISTING SPATIAL ARCHITECTURE RECOVERED")
print("=" * 120)

---
### Imported master block 143

`BLOCK 11.0C — FREEZE MINIMAL H3 CONNECTIVITY ARCHITECTURE`


In [ ]:
# ==================================================================================================
# BLOCK 11.0C — FREEZE MINIMAL H3 CONNECTIVITY ARCHITECTURE
#
# PURPOSE
# -------
# 1. Recover the pre-existing MERIT point / radius / snap diagnostics.
# 2. Diagnose whether contributing-area values are spatially trustworthy.
# 3. Freeze the H3 stream-network population.
# 4. Freeze the minimal connectivity architecture BEFORE response modelling.
#
# IMPORTANT
# ---------
# ZERO RESPONSE VALUES USED
# ZERO H3 MODELS
# ZERO OUTCOME-BASED SELECTION
# ZERO NEW GEE / DEM / DELINEATION
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

print("=" * 125)
print("BLOCK 11.0C — FREEZE MINIMAL H3 CONNECTIVITY ARCHITECTURE")
print("=" * 125)

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

SNAP = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110"
)

MASTER_DIR = SNAP / "03_MASTER"
AUDIT_DIR  = SNAP / "01_AUDIT"

RECOVERY_DIR = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "SPATIAL_RECOVERY"
)

OUT = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_FREEZE"
)

OUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 1. INPUTS
# --------------------------------------------------------------------------------------------------

P_NETWORK = MASTER_DIR / "STATION_NETWORK_FROZEN_v1.csv"
P_GEOM    = MASTER_DIR / "STATION_GEOMETRY_FINAL_v2.csv"
P_UPA     = AUDIT_DIR  / "BLOCK80_MERIT_upa_by_radius.csv"
P_POINTS  = AUDIT_DIR  / "BLOCK80_MERIT_point_diagnostic.csv"
P_SNAP    = AUDIT_DIR  / "BLOCK81A_station_snap_summary.csv"

for p in [P_NETWORK, P_GEOM, P_UPA, P_POINTS]:
    assert p.exists(), f"Missing required frozen input: {p}"

network = pd.read_csv(P_NETWORK, low_memory=False)
geom    = pd.read_csv(P_GEOM, low_memory=False)
upa     = pd.read_csv(P_UPA, low_memory=False)
points  = pd.read_csv(P_POINTS, low_memory=False)

snap = (
    pd.read_csv(P_SNAP, low_memory=False)
    if P_SNAP.exists()
    else pd.DataFrame()
)

print("\nInputs recovered:")
print("network :", network.shape)
print("geometry:", geom.shape)
print("UPA     :", upa.shape)
print("points  :", points.shape)
print("snap    :", snap.shape if len(snap) else "not available")


# --------------------------------------------------------------------------------------------------
# 2. ONE ROW PER FROZEN STATION
# --------------------------------------------------------------------------------------------------

base_cols = [
    "station_id",
    "station_type",
    "network_role",
    "latitude",
    "longitude",
    "field_elevation",
    "coordinate_stability",
    "coordinate_valid"
]

base = (
    network[
        [c for c in base_cols if c in network.columns]
    ]
    .drop_duplicates("station_id")
    .copy()
)

geom_cols = [
    "station_id",
    "eligible_for_catchment_inference",
    "geometry_status",
    "geometry_issue",
    "geometry_confidence",
    "geometry_relationship",
    "spatial_reach_group",
    "dem_latitude",
    "dem_longitude"
]

geom_small = (
    geom[
        [c for c in geom_cols if c in geom.columns]
    ]
    .drop_duplicates("station_id")
)

base = base.merge(
    geom_small,
    on="station_id",
    how="left",
    validate="one_to_one"
)


# --------------------------------------------------------------------------------------------------
# 3. MERGE EXISTING MERIT INFORMATION
# --------------------------------------------------------------------------------------------------

upa_cols = [
    "station_id",
    "upa_max_within_100m_km2",
    "upa_max_within_200m_km2",
    "upa_max_within_300m_km2",
    "upa_max_within_500m_km2",
    "upa_max_within_750m_km2",
    "upa_max_within_1000m_km2"
]

base = base.merge(
    upa[
        [c for c in upa_cols if c in upa.columns]
    ].drop_duplicates("station_id"),
    on="station_id",
    how="left",
    validate="one_to_one"
)

point_cols = [
    "station_id",
    "merit_upa_at_point_km2",
    "merit_elv_at_point",
    "merit_upg_at_point",
    "merit_hnd_at_point",
    "merit_wth_at_point",
    "merit_wat_at_point",
    "abs_field_merit_elv_diff_m"
]

base = base.merge(
    points[
        [c for c in point_cols if c in points.columns]
    ].drop_duplicates("station_id"),
    on="station_id",
    how="left",
    validate="one_to_one"
)

if len(snap):

    snap_cols = [
        "station_id",
        "recorded_upa_km2",
        "min_candidate_distance_m",
        "max_upa_300m_km2",
        "min_candidate_elv_diff_m",
        "any_large_upa_jump",
        "n_plausible_candidates"
    ]

    snap_small = (
        snap[
            [c for c in snap_cols if c in snap.columns]
        ]
        .drop_duplicates("station_id")
    )

    base = base.merge(
        snap_small,
        on="station_id",
        how="left",
        validate="one_to_one"
    )


# --------------------------------------------------------------------------------------------------
# 4. FREEZE POPULATION FIRST
#
# H3 concerns organization of the stream network.
# Springs / spring outflows / irrigation canal are NOT primary H3 connectivity units.
# Geometry-ineligible stations are excluded.
# --------------------------------------------------------------------------------------------------

base["is_stream_network"] = (
    base["station_type"]
    .astype(str)
    .eq("STREAM_NETWORK")
)

base["geometry_eligible"] = (
    base["eligible_for_catchment_inference"]
    .fillna(False)
    .astype(bool)
)

base["H3_population_eligible"] = (
    base["is_stream_network"] &
    base["geometry_eligible"]
)

base["H3_exclusion_reason"] = np.select(
    [
        ~base["is_stream_network"],
        (
            base["is_stream_network"] &
            ~base["geometry_eligible"]
        )
    ],
    [
        "SPECIAL_SOURCE_OR_NONSTREAM",
        "GEOMETRY_NOT_ELIGIBLE"
    ],
    default="PRIMARY_STREAM_NETWORK"
)


# --------------------------------------------------------------------------------------------------
# 5. DIAGNOSE UPA STABILITY ACROSS SEARCH RADII
#
# The maximum UPA in a large radius is NOT automatically the station's drainage area.
# Large jumps imply interception of another major channel.
# --------------------------------------------------------------------------------------------------

radii = [100, 200, 300, 500, 750, 1000]

upa_radius_cols = [
    f"upa_max_within_{r}m_km2"
    for r in radii
]

for c in upa_radius_cols:
    if c in base.columns:
        base[c] = pd.to_numeric(
            base[c],
            errors="coerce"
        )

base["merit_upa_at_point_km2"] = pd.to_numeric(
    base["merit_upa_at_point_km2"],
    errors="coerce"
)

if "recorded_upa_km2" in base.columns:
    base["recorded_upa_km2"] = pd.to_numeric(
        base["recorded_upa_km2"],
        errors="coerce"
    )


def safe_ratio(a, b):

    if pd.isna(a) or pd.isna(b):
        return np.nan

    if a <= 0 or b <= 0:
        return np.nan

    return a / b


# Diagnostic jumps.
base["jump_100_to_200"] = [
    safe_ratio(b, a)
    for a, b in zip(
        base["upa_max_within_100m_km2"],
        base["upa_max_within_200m_km2"]
    )
]

base["jump_200_to_300"] = [
    safe_ratio(b, a)
    for a, b in zip(
        base["upa_max_within_200m_km2"],
        base["upa_max_within_300m_km2"]
    )
]

base["jump_300_to_500"] = [
    safe_ratio(b, a)
    for a, b in zip(
        base["upa_max_within_300m_km2"],
        base["upa_max_within_500m_km2"]
    )
]

base["jump_500_to_750"] = [
    safe_ratio(b, a)
    for a, b in zip(
        base["upa_max_within_500m_km2"],
        base["upa_max_within_750m_km2"]
    )
]

base["jump_750_to_1000"] = [
    safe_ratio(b, a)
    for a, b in zip(
        base["upa_max_within_750m_km2"],
        base["upa_max_within_1000m_km2"]
    )
]

jump_cols = [
    "jump_100_to_200",
    "jump_200_to_300",
    "jump_300_to_500",
    "jump_500_to_750",
    "jump_750_to_1000"
]

base["max_adjacent_UPA_jump"] = (
    base[jump_cols]
    .replace([np.inf, -np.inf], np.nan)
    .max(axis=1)
)

# >10x between adjacent search radii is a strong warning,
# not an automatic replacement rule.
base["radius_jump_flag_10x"] = (
    base["max_adjacent_UPA_jump"] > 10
)


# --------------------------------------------------------------------------------------------------
# 6. PRIMARY CONTRIBUTING-AREA SOURCE
#
# IMPORTANT:
# We do NOT choose the radius giving the largest area.
#
# Existing snap diagnostics already contain "recorded_upa_km2" for stream-network stations.
# Therefore:
#
# PRIMARY = recorded_upa_km2 where available.
#
# This is the UPA at the recorded/frozen station geometry and avoids selecting a neighbouring
# high-UPA channel simply because it lies inside an arbitrary buffer.
#
# merit_upa_at_point_km2 is retained as an independent cross-check.
# --------------------------------------------------------------------------------------------------

if "recorded_upa_km2" in base.columns:

    base["contributing_area_primary_km2"] = (
        base["recorded_upa_km2"]
    )

    base["contributing_area_source"] = np.where(
        base["recorded_upa_km2"].notna(),
        "PREEXISTING_RECORDED_UPA",
        "MISSING"
    )

else:

    base["contributing_area_primary_km2"] = (
        base["merit_upa_at_point_km2"]
    )

    base["contributing_area_source"] = np.where(
        base["merit_upa_at_point_km2"].notna(),
        "MERIT_UPA_AT_FROZEN_POINT",
        "MISSING"
    )


# --------------------------------------------------------------------------------------------------
# 7. CROSS-CHECK PRIMARY AREA AGAINST MERIT POINT VALUE
# --------------------------------------------------------------------------------------------------

base["point_vs_primary_ratio"] = np.nan

mask = (
    base["contributing_area_primary_km2"].gt(0) &
    base["merit_upa_at_point_km2"].gt(0)
)

base.loc[
    mask,
    "point_vs_primary_ratio"
] = (
    base.loc[
        mask,
        "contributing_area_primary_km2"
    ] /
    base.loc[
        mask,
        "merit_upa_at_point_km2"
    ]
)

base["point_primary_agree_factor2"] = (
    base["point_vs_primary_ratio"]
    .between(
        0.5,
        2.0,
        inclusive="both"
    )
)

# Missing comparison stays missing.
base.loc[
    base["point_vs_primary_ratio"].isna(),
    "point_primary_agree_factor2"
] = np.nan


# --------------------------------------------------------------------------------------------------
# 8. AREA QUALITY CLASS
#
# This class is diagnostic and frozen BEFORE any response model.
# It does not inspect EC, Na, SO4, temperature, H, Q, or model outcomes.
# --------------------------------------------------------------------------------------------------

def classify_area(row):

    if not row["H3_population_eligible"]:
        return "NOT_IN_PRIMARY_H3_POPULATION"

    a = row["contributing_area_primary_km2"]

    if pd.isna(a) or a <= 0:
        return "MISSING_AREA"

    agree = row.get(
        "point_primary_agree_factor2",
        np.nan
    )

    jump = row.get(
        "radius_jump_flag_10x",
        False
    )

    # Recorded and point UPA agree.
    if agree is True or agree == True:
        if jump:
            return "PRIMARY_VALID_LARGE_RADIUS_JUMP_NEARBY"
        return "PRIMARY_VALID_STABLE"

    # Recorded UPA exists but point diagnostic differs substantially.
    if agree is False or agree == False:
        return "PRIMARY_USABLE_POINT_DISAGREEMENT_FLAG"

    return "PRIMARY_USABLE_NO_POINT_CROSSCHECK"


base["contributing_area_quality"] = base.apply(
    classify_area,
    axis=1
)


# --------------------------------------------------------------------------------------------------
# 9. LOG10 AREA — FROZEN CONNECTIVITY GRADIENT
# --------------------------------------------------------------------------------------------------

base["log10_contributing_area_km2"] = np.where(
    base["contributing_area_primary_km2"] > 0,
    np.log10(
        base["contributing_area_primary_km2"]
    ),
    np.nan
)


# --------------------------------------------------------------------------------------------------
# 10. FREEZE H3 MODEL-ELIGIBLE POPULATION
#
# We do NOT automatically remove stations merely because a large UPA channel exists nearby.
# We use the pre-existing recorded UPA and retain quality flags.
# --------------------------------------------------------------------------------------------------

base["H3_area_model_eligible"] = (
    base["H3_population_eligible"] &
    base["log10_contributing_area_km2"].notna()
)

primary = (
    base[
        base["H3_area_model_eligible"]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 11. DESCRIPTIVE CHECKS ONLY — NO RESPONSE DATA
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 125)
print("A. H3 POPULATION FREEZE")
print("=" * 125)

print(
    base[
        [
            "station_id",
            "station_type",
            "network_role",
            "H3_population_eligible",
            "H3_exclusion_reason"
        ]
    ]
    .sort_values("station_id")
    .to_string(index=False)
)

print("\nCounts:")
print(
    base["H3_exclusion_reason"]
    .value_counts(dropna=False)
    .to_string()
)


print("\n" + "=" * 125)
print("B. CONTRIBUTING-AREA FREEZE")
print("=" * 125)

display_cols = [
    "station_id",
    "network_role",
    "contributing_area_primary_km2",
    "merit_upa_at_point_km2",
    "point_vs_primary_ratio",
    "max_adjacent_UPA_jump",
    "radius_jump_flag_10x",
    "contributing_area_quality",
    "log10_contributing_area_km2",
    "H3_area_model_eligible"
]

print(
    base[
        display_cols
    ]
    .sort_values(
        [
            "H3_area_model_eligible",
            "contributing_area_primary_km2"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(index=False)
)


print("\n" + "=" * 125)
print("C. PRIMARY H3 CONNECTIVITY GRADIENT")
print("=" * 125)

print("Eligible stations :", len(primary))

if len(primary):

    print(
        primary[
            "contributing_area_primary_km2"
        ]
        .describe()
        .to_string()
    )

    print("\nlog10(area) distribution:")

    print(
        primary[
            "log10_contributing_area_km2"
        ]
        .describe()
        .to_string()
    )

    print("\nArea-quality counts:")

    print(
        primary[
            "contributing_area_quality"
        ]
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 12. FREEZE THE SCIENTIFIC H3 ARCHITECTURE
#
# NOTE:
# Elevation is retained as a secondary spatial descriptor / sensitivity covariate.
#
# The PRIMARY H3 target is NOT an additive area effect:
# station random intercepts already absorb persistent station differences.
#
# The target is hydroclimate × contributing-area interaction.
# --------------------------------------------------------------------------------------------------

H3_FREEZE = {
    "hypothesis":
        (
            "The expression of prespecified antecedent hydroclimatic memory "
            "varies systematically with catchment contributing area across "
            "the eligible stream network."
        ),

    "primary_population":
        (
            "STREAM_NETWORK stations with frozen geometry eligible for "
            "catchment inference and non-missing pre-existing contributing area."
        ),

    "excluded_primary_population":
        [
            "SPRING",
            "SPRING_OR_OUTFLOW",
            "IRRIGATION_CANAL",
            "geometry-ineligible stations"
        ],

    "primary_connectivity_variable":
        "log10_contributing_area_km2",

    "primary_area_source":
        "pre-existing recorded MERIT UPA at frozen station geometry",

    "secondary_spatial_descriptor":
        "field_elevation",

    "network_role_usage":
        (
            "descriptive/stratification variable; not treated as an ordinal "
            "continuous connectivity score"
        ),

    "primary_model_concept":
        "Y ~ Q + H + H:C + station random intercept",

    "primary_parameter":
        "beta_HxC",

    "interpretation":
        (
            "beta_HxC tests whether the magnitude/direction of the frozen "
            "antecedent hydroclimate association changes along the "
            "contributing-area gradient."
        ),

    "response_hydro_pairs":
        [
            {
                "response": "AC::C.E._SOL::dS/m",
                "hydro": "ppt_sum_14d"
            },
            {
                "response": "AC::C.E._SOL::dS/m",
                "hydro": "ppt_sum_60d"
            },
            {
                "response": "AC::Na::mg/L",
                "hydro": "ppt_sum_60d"
            },
            {
                "response": "AC::SO4_ppm::mg/L",
                "hydro": "ppt_sum_60d"
            }
        ],

    "secondary_reference":
        {
            "response":
                "FIELD::Water temperature::degC",

            "hydro":
                "sm_rootzone_mean_90d",

            "role":
                "secondary thermal reference only"
        },

    "selection_policy":
        (
            "No spatial predictor, response, hydroclimate window, interaction "
            "or model is selected based on H3 outcome."
        ),

    "discharge_policy":
        "observed Q only for primary inference",

    "validation":
        "campaign LOCO if full-data interaction estimation is computationally and statistically viable",

    "causal_claim":
        False
}


# --------------------------------------------------------------------------------------------------
# 13. QA
# --------------------------------------------------------------------------------------------------

qa_rows = [
    {
        "check": "network_input_frozen",
        "pass": P_NETWORK.exists()
    },
    {
        "check": "geometry_input_frozen",
        "pass": P_GEOM.exists()
    },
    {
        "check": "MERIT_UPA_preexisting",
        "pass": P_UPA.exists()
    },
    {
        "check": "primary_population_nonempty",
        "pass": len(primary) > 0
    },
    {
        "check": "primary_area_positive",
        "pass": bool(
            (
                primary["contributing_area_primary_km2"] > 0
            ).all()
        ) if len(primary) else False
    },
    {
        "check": "log_area_finite",
        "pass": bool(
            np.isfinite(
                primary["log10_contributing_area_km2"]
            ).all()
        ) if len(primary) else False
    },
    {
        "check": "no_response_values_used",
        "pass": True
    },
    {
        "check": "no_H3_models_fit",
        "pass": True
    },
    {
        "check": "no_new_spatial_extraction",
        "pass": True
    },
    {
        "check": "no_window_search",
        "pass": True
    }
]

qa = pd.DataFrame(qa_rows)

BLOCK_PASS = bool(
    qa["pass"].all()
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE FROZEN ARTIFACTS
# --------------------------------------------------------------------------------------------------

base.to_csv(
    OUT /
    "BLOCK110C_STATION_CONNECTIVITY_FREEZE_v1.csv",
    index=False
)

primary.to_csv(
    OUT /
    "BLOCK110C_PRIMARY_H3_STATIONS_v1.csv",
    index=False
)

qa.to_csv(
    OUT /
    "BLOCK110C_QA_v1.csv",
    index=False
)

with open(
    OUT /
    "BLOCK110C_H3_ARCHITECTURE_FREEZE_v1.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        H3_FREEZE,
        f,
        indent=2,
        ensure_ascii=False
    )

manifest = {
    "block":
        "11.0C",

    "status":
        "PASS" if BLOCK_PASS else "FAIL",

    "n_total_frozen_stations":
        int(len(base)),

    "n_stream_network":
        int(base["is_stream_network"].sum()),

    "n_geometry_eligible_stream":
        int(base["H3_population_eligible"].sum()),

    "n_primary_area_model_stations":
        int(base["H3_area_model_eligible"].sum()),

    "primary_connectivity_variable":
        "log10_contributing_area_km2",

    "response_values_used":
        False,

    "H3_models_fit":
        0,

    "new_spatial_extraction":
        False,

    "H3_architecture_frozen":
        True
}

with open(
    OUT /
    "BLOCK110C_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 15. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 125)
print("BLOCK 11.0C FINAL GATE")
print("=" * 125)

print("BLOCK PASS                     :", BLOCK_PASS)
print("TOTAL FROZEN STATIONS          :", len(base))
print("STREAM-NETWORK STATIONS        :", int(base["is_stream_network"].sum()))
print("GEOMETRY-ELIGIBLE STREAM       :", int(base["H3_population_eligible"].sum()))
print("PRIMARY AREA-MODEL STATIONS    :", int(base["H3_area_model_eligible"].sum()))

print(
    "AREA RANGE km2                 :",
    (
        float(primary["contributing_area_primary_km2"].min()),
        float(primary["contributing_area_primary_km2"].max())
    )
    if len(primary)
    else None
)

print(
    "LOG10 AREA RANGE               :",
    (
        float(primary["log10_contributing_area_km2"].min()),
        float(primary["log10_contributing_area_km2"].max())
    )
    if len(primary)
    else None
)

print("\nPRIMARY H3 VARIABLE:")
print("log10_contributing_area_km2")

print("\nPRIMARY H3 TARGET:")
print("beta_HxC in Y ~ Q + H + H:C + station random intercept")

print("\nFROZEN PRIMARY TESTS:")
for x in H3_FREEZE["response_hydro_pairs"]:
    print(" ", x["response"], " × ", x["hydro"])

print("\nSECONDARY:")
print(
    H3_FREEZE["secondary_reference"]["response"],
    " × ",
    H3_FREEZE["secondary_reference"]["hydro"]
)

print("\nRESPONSE VALUES USED            : False")
print("H3 MODELS FIT                  : 0")
print("NEW GEE / DEM EXTRACTION       : False")
print("WINDOW SEARCH                  : False")
print("H3 ARCHITECTURE FROZEN         : True")

print("\nOutputs:")
for p in sorted(OUT.glob("BLOCK110C*")):
    print(" ", p)

assert BLOCK_PASS

print("\n" + "=" * 125)
print("BLOCK 11.0C PASS — H3 CONNECTIVITY ARCHITECTURE FROZEN BEFORE OUTCOME MODELLING")
print("=" * 125)

---
### Imported master block 146

`BLOCK 11.1A — FINAL H3 MODEL PANEL ASSEMBLY + IDENTIFIABILITY QA`


In [ ]:
# ==================================================================================================
# BLOCK 11.1A — FINAL H3 MODEL PANEL ASSEMBLY + IDENTIFIABILITY QA
#
# H3:
# Does fixed catchment organization modulate the expression of a shared antecedent
# hydroclimatic forcing?
#
# PRIMARY TARGET:
# beta_HxC in
#
#   Y_it = alpha + station_RE_i + beta_Q*Q_it + beta_H*H_t
#          + beta_HxC*(H_t * C_i) + error
#
# where:
#   H_t = frozen common event-date antecedent hydroclimate
#   C_i = frozen log10 contributing area
#   Q_it = OBSERVED discharge only
#
# IMPORTANT:
# - NO beta_C main effect: C is time-invariant within station and is absorbed by station RE.
# - NO window search.
# - NO reconstructed C09 Q.
# - NO HxQ.
# - NO DEM/GEE/delineation.
# - NO outcome-based spatial selection.
# - Blocks 8–10 remain frozen.
#
# THIS BLOCK FITS ZERO MODELS.
# It creates the exact model-ready H3 panel and final pre-fit gate.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print("=" * 130)
print("BLOCK 11.1A — FINAL H3 MODEL PANEL ASSEMBLY + IDENTIFIABILITY QA")
print("=" * 130)


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

MASTER_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MASTER_ANALYTICAL_v1_1.csv"
)

MEMORY_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "MEMORY_ANALYSIS_DATASET_v1.csv"
)

HYDROMETRY_PATH = (
    PROJECT /
    "SNAPSHOTS" /
    "POST_BLOCK85C_20260915_034110" /
    "03_MASTER" /
    "HYDROMETRY_FROZEN_v1.csv"
)

CONNECTIVITY_PATH = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_FREEZE" /
    "BLOCK110C_STATION_CONNECTIVITY_FREEZE_v1.csv"
)

H3_STATIONS_PATH = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_FREEZE" /
    "BLOCK110C_PRIMARY_H3_STATIONS_v1.csv"
)

BLOCK10_SUPPORT_PATH = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULL_LOCO" /
    "RESULTS" /
    "BLOCK102B_B_FINAL_JOINT_SUPPORT_SPECTRUM_v1.csv"
)

BLOCK10_MANIFEST_PATH = (
    PROJECT /
    "JOINT_HQ_ANALYSIS" /
    "FULL_LOCO" /
    "RESULTS" /
    "BLOCK102B_B_FINAL_manifest.json"
)

BLOCK110E_MANIFEST_PATH = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_FREEZE" /
    "BLOCK110E_manifest.json"
)

OUT = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_MODELS"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 1. REQUIRED FILE GATE
# --------------------------------------------------------------------------------------------------

required = {
    "MASTER": MASTER_PATH,
    "MEMORY": MEMORY_PATH,
    "HYDROMETRY": HYDROMETRY_PATH,
    "CONNECTIVITY": CONNECTIVITY_PATH,
    "H3_STATIONS": H3_STATIONS_PATH,
    "BLOCK10_SUPPORT": BLOCK10_SUPPORT_PATH,
    "BLOCK10_MANIFEST": BLOCK10_MANIFEST_PATH,
    "BLOCK110E_MANIFEST": BLOCK110E_MANIFEST_PATH
}

print("\nA. REQUIRED INPUTS")
print("-" * 130)

missing = []

for name, path in required.items():

    exists = path.exists()

    print(
        f"{name:20s}",
        "FOUND" if exists else "MISSING",
        path
    )

    if not exists:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing required frozen inputs: {missing}"
    )


# --------------------------------------------------------------------------------------------------
# 2. VERIFY UPSTREAM GATES
# --------------------------------------------------------------------------------------------------

with open(
    BLOCK10_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:
    block10_manifest = json.load(f)

with open(
    BLOCK110E_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:
    block110e_manifest = json.load(f)

print("\n" + "=" * 130)
print("B. UPSTREAM FREEZE GATES")
print("=" * 130)

print(
    "Block 10 status:",
    block10_manifest.get(
        "status",
        "UNKNOWN"
    )
)

print(
    "Block 11.0E classification:",
    block110e_manifest.get(
        "classification",
        "UNKNOWN"
    )
)

if block110e_manifest.get("classification") != \
        "HYDROCLIMATE_IS_COMMON_TEMPORAL_FORCING":

    raise RuntimeError(
        "H3 STOP: Block 11.0E did not classify H as common temporal forcing."
    )

print("COMMON TEMPORAL H GATE: PASS")


# --------------------------------------------------------------------------------------------------
# 3. LOAD FROZEN DATA
# --------------------------------------------------------------------------------------------------

memory = pd.read_csv(
    MEMORY_PATH,
    low_memory=False
)

hydro = pd.read_csv(
    HYDROMETRY_PATH,
    low_memory=False
)

conn = pd.read_csv(
    CONNECTIVITY_PATH,
    low_memory=False
)

h3stations = pd.read_csv(
    H3_STATIONS_PATH,
    low_memory=False
)

b10 = pd.read_csv(
    BLOCK10_SUPPORT_PATH,
    low_memory=False
)

print("\n" + "=" * 130)
print("C. INPUT SHAPES")
print("=" * 130)

print("MEMORY       :", memory.shape)
print("HYDROMETRY   :", hydro.shape)
print("CONNECTIVITY :", conn.shape)
print("H3 STATIONS  :", h3stations.shape)
print("BLOCK10      :", b10.shape)


# --------------------------------------------------------------------------------------------------
# 4. FREEZE EXACT H3 TESTS
#
# These are inherited from Block 10.
# ZERO SEARCH.
# --------------------------------------------------------------------------------------------------

tests = pd.DataFrame([
    {
        "test_id": "H3_EC_P14",
        "response_key": "AC::C.E._SOL::dS/m",
        "H_variable": "ppt_sum_14d",
        "role": "PRIMARY"
    },
    {
        "test_id": "H3_EC_P60",
        "response_key": "AC::C.E._SOL::dS/m",
        "H_variable": "ppt_sum_60d",
        "role": "PRIMARY"
    },
    {
        "test_id": "H3_NA_P60",
        "response_key": "AC::Na::mg/L",
        "H_variable": "ppt_sum_60d",
        "role": "PRIMARY"
    },
    {
        "test_id": "H3_SO4_P60",
        "response_key": "AC::SO4_ppm::mg/L",
        "H_variable": "ppt_sum_60d",
        "role": "PRIMARY"
    },
    {
        "test_id": "H3_TEMP_SMROOT90",
        "response_key": "FIELD::Water temperature::degC",
        "H_variable": "sm_rootzone_mean_90d",
        "role": "SECONDARY_THERMAL"
    }
])

print("\n" + "=" * 130)
print("D. FROZEN H3 TEST BANK")
print("=" * 130)

print(
    tests.to_string(
        index=False
    )
)

print("\nPrimary tests:", (tests["role"] == "PRIMARY").sum())
print("Secondary tests:", (tests["role"] != "PRIMARY").sum())
print("WINDOW SEARCH: FALSE")


# --------------------------------------------------------------------------------------------------
# 5. DETERMINE FROZEN AREA COLUMN
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 130)
print("E. CONNECTIVITY VARIABLE")
print("=" * 130)

print("Connectivity columns:")
print(list(conn.columns))

area_log_candidates = [
    c for c in conn.columns
    if c.lower() in {
        "log10_contributing_area_km2",
        "log10_contributing_area",
        "log10_upa_km2"
    }
]

if not area_log_candidates:

    # Allow reconstruction ONLY from the frozen primary area variable,
    # never from another spatial candidate.
    area_candidates = [
        c for c in conn.columns
        if c.lower() in {
            "contributing_area_primary_km2",
            "recorded_upa_km2",
            "contributing_area_km2"
        }
    ]

    if not area_candidates:
        raise RuntimeError(
            "Frozen primary contributing-area variable not found."
        )

    area_raw_col = area_candidates[0]

    if (pd.to_numeric(conn[area_raw_col], errors="coerce") <= 0).any():
        raise RuntimeError(
            "Non-positive frozen contributing area detected."
        )

    conn["log10_contributing_area_km2"] = np.log10(
        pd.to_numeric(
            conn[area_raw_col],
            errors="coerce"
        )
    )

    area_log_col = "log10_contributing_area_km2"

else:

    area_log_col = area_log_candidates[0]

print("Frozen H3 C variable:", area_log_col)


# --------------------------------------------------------------------------------------------------
# 6. IDENTIFY PRIMARY H3 STATION POPULATION
# --------------------------------------------------------------------------------------------------

station_col_candidates = [
    c for c in h3stations.columns
    if c.lower() in {
        "station_id",
        "physical_station_id_v1"
    }
]

if not station_col_candidates:
    raise RuntimeError(
        "Station identifier missing from frozen H3 station file."
    )

h3_station_col = station_col_candidates[0]

primary_station_ids = set(
    h3stations[
        h3_station_col
    ]
    .dropna()
    .astype(str)
)

print("\nFrozen primary H3 stations:", len(primary_station_ids))

if len(primary_station_ids) != 32:
    raise RuntimeError(
        f"Expected 32 frozen H3 stations, found {len(primary_station_ids)}."
    )


# --------------------------------------------------------------------------------------------------
# 7. BUILD ONE-ROW-PER-STATION CONNECTIVITY LOOKUP
# --------------------------------------------------------------------------------------------------

if "station_id" not in conn.columns:

    if "physical_station_id_v1" in conn.columns:
        conn["station_id"] = conn["physical_station_id_v1"]
    else:
        raise RuntimeError(
            "No station_id in connectivity freeze."
        )

conn["station_id"] = conn["station_id"].astype(str)

conn_lookup = (
    conn[
        conn["station_id"].isin(
            primary_station_ids
        )
    ][[
        "station_id",
        area_log_col
    ]]
    .copy()
)

conn_lookup[area_log_col] = pd.to_numeric(
    conn_lookup[area_log_col],
    errors="coerce"
)

# Every station must have one unique C.
c_unique = (
    conn_lookup
    .groupby("station_id")[area_log_col]
    .nunique(dropna=True)
)

bad_c = c_unique[
    c_unique != 1
]

if len(bad_c):
    raise RuntimeError(
        "Connectivity is not unique within station:\n"
        + bad_c.to_string()
    )

conn_lookup = (
    conn_lookup
    .drop_duplicates(
        subset=["station_id"]
    )
)

print(
    "Stations with frozen C:",
    conn_lookup["station_id"].nunique()
)

print(
    "C range:",
    float(conn_lookup[area_log_col].min()),
    "to",
    float(conn_lookup[area_log_col].max())
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD OBSERVED-Q LOOKUP
#
# Primary H3 must use observed Q only.
# Reconstructed C09 Q is prohibited.
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 130)
print("F. OBSERVED-Q ARCHITECTURE")
print("=" * 130)

print("Hydrometry columns:")
print(list(hydro.columns))

if "field_observation_id" not in hydro.columns:
    raise RuntimeError(
        "field_observation_id missing from frozen hydrometry."
    )

q_candidates = [
    "Q_observed_final",
    "discharge_observed"
]

q_col = None

for c in q_candidates:
    if c in hydro.columns:
        q_col = c
        break

if q_col is None:
    raise RuntimeError(
        "Observed discharge column not found."
    )

print("Observed Q source:", q_col)

q_lookup = (
    hydro[[
        "field_observation_id",
        q_col
    ]]
    .copy()
)

q_lookup[q_col] = pd.to_numeric(
    q_lookup[q_col],
    errors="coerce"
)

# Multiple rows for same field event are allowed only if Q agrees.
q_conflict = (
    q_lookup
    .groupby("field_observation_id")[q_col]
    .nunique(dropna=True)
)

q_conflict = q_conflict[
    q_conflict > 1
]

if len(q_conflict):
    raise RuntimeError(
        "Conflicting observed Q values within field observation."
    )

q_lookup = (
    q_lookup
    .drop_duplicates(
        subset=["field_observation_id"]
    )
)

print(
    "Observed Q non-null events:",
    int(q_lookup[q_col].notna().sum())
)


# --------------------------------------------------------------------------------------------------
# 9. SUBSET MEMORY TO EXACT H3 RESPONSES
# --------------------------------------------------------------------------------------------------

wanted_responses = set(
    tests["response_key"]
)

panel = memory[
    memory["response_key"].isin(
        wanted_responses
    )
].copy()

print("\n" + "=" * 130)
print("G. RESPONSE PANEL BEFORE H3 FILTERING")
print("=" * 130)

print(
    panel.groupby(
        "response_key"
    )
    .size()
    .rename("rows")
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 10. MERGE OBSERVED Q
# --------------------------------------------------------------------------------------------------

panel = panel.merge(
    q_lookup,
    on="field_observation_id",
    how="left",
    validate="many_to_one"
)

panel = panel.rename(
    columns={
        q_col: "Q_observed_H3"
    }
)

# Q must be positive for log10(Q).
panel["Q_observed_H3"] = pd.to_numeric(
    panel["Q_observed_H3"],
    errors="coerce"
)

panel["observed_Q_eligible_H3"] = (
    panel["Q_observed_H3"].notna()
    &
    (panel["Q_observed_H3"] > 0)
)

print(
    "\nRows with positive observed Q:",
    int(panel["observed_Q_eligible_H3"].sum()),
    "/",
    len(panel)
)


# --------------------------------------------------------------------------------------------------
# 11. MERGE FROZEN CONNECTIVITY
# --------------------------------------------------------------------------------------------------

panel["station_id"] = panel["station_id"].astype(str)

panel = panel.merge(
    conn_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

panel["H3_primary_station"] = (
    panel["station_id"].isin(
        primary_station_ids
    )
)

panel["H3_connectivity_available"] = (
    panel[area_log_col].notna()
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL ROW ELIGIBILITY
# --------------------------------------------------------------------------------------------------

panel["response_value_numeric"] = pd.to_numeric(
    panel["response_value_numeric"],
    errors="coerce"
)

panel["H3_base_eligible"] = (
    panel["H3_primary_station"]
    &
    panel["H3_connectivity_available"]
    &
    panel["observed_Q_eligible_H3"]
    &
    panel["response_value_numeric"].notna()
    &
    panel["hydroclimate_integration_eligible"].fillna(False).astype(bool)
)

print("\n" + "=" * 130)
print("H. BASE H3 ELIGIBILITY")
print("=" * 130)

print(
    panel.groupby(
        "response_key"
    )[
        [
            "H3_primary_station",
            "H3_connectivity_available",
            "observed_Q_eligible_H3",
            "H3_base_eligible"
        ]
    ]
    .sum()
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 13. EXPAND INTO EXACT TEST-SPECIFIC MODEL PANELS
# --------------------------------------------------------------------------------------------------

test_panels = []

qa_rows = []

for _, spec in tests.iterrows():

    test_id = spec["test_id"]
    response_key = spec["response_key"]
    H = spec["H_variable"]
    role = spec["role"]

    if H not in panel.columns:
        raise RuntimeError(
            f"{test_id}: frozen H variable missing: {H}"
        )

    d = panel[
        panel["response_key"] == response_key
    ].copy()

    d[H] = pd.to_numeric(
        d[H],
        errors="coerce"
    )

    d = d[
        d["H3_base_eligible"]
        &
        d[H].notna()
    ].copy()

    d["test_id"] = test_id
    d["H_variable"] = H
    d["H3_role"] = role

    d["H_raw"] = d[H]

    d["C_raw"] = pd.to_numeric(
        d[area_log_col],
        errors="coerce"
    )

    d["log10_Q_raw"] = np.log10(
        d["Q_observed_H3"]
    )

    # ----------------------------------------------------------------------------------------------
    # IMPORTANT:
    # Do NOT standardize here permanently.
    #
    # Full-data model:
    # scaling will be calculated from the exact model rows.
    #
    # LOCO:
    # scaling MUST later be calculated from training rows only.
    #
    # Saving raw values here prevents leakage.
    # ----------------------------------------------------------------------------------------------

    # C must be invariant within station.
    c_check = (
        d.groupby(
            "station_id"
        )["C_raw"]
        .nunique(
            dropna=True
        )
    )

    if (c_check > 1).any():
        raise RuntimeError(
            f"{test_id}: C varies within station."
        )

    # H should be common within hydroclimate event date.
    h_check = (
        d.groupby(
            "hydroclimate_event_date"
        )["H_raw"]
        .nunique(
            dropna=True
        )
    )

    if (h_check > 1).any():
        raise RuntimeError(
            f"{test_id}: H varies within event date."
        )

    d["H3_model_row"] = True

    test_panels.append(
        d
    )

    qa_rows.append({
        "test_id":
            test_id,

        "role":
            role,

        "response_key":
            response_key,

        "H_variable":
            H,

        "n_rows":
            len(d),

        "n_stations":
            d["station_id"].nunique(),

        "n_campaigns":
            d["campaign_id"].nunique(),

        "n_event_dates":
            d["hydroclimate_event_date"].nunique(),

        "n_censored":
            int(
                d["response_censored"]
                .fillna(False)
                .astype(bool)
                .sum()
            ),

        "Q_min":
            d["Q_observed_H3"].min(),

        "Q_max":
            d["Q_observed_H3"].max(),

        "H_min":
            d["H_raw"].min(),

        "H_max":
            d["H_raw"].max(),

        "C_min":
            d["C_raw"].min(),

        "C_max":
            d["C_raw"].max(),

        "C_invariant_within_station":
            bool(
                (c_check <= 1).all()
            ),

        "H_common_within_event_date":
            bool(
                (h_check <= 1).all()
            )
    })


model_panel = pd.concat(
    test_panels,
    ignore_index=True
)

qa = pd.DataFrame(
    qa_rows
)


# --------------------------------------------------------------------------------------------------
# 14. IDENTIFIABILITY QA
#
# Because C is constant within station:
# station random intercept + beta_C*C is not separately identifiable.
#
# Therefore beta_C is intentionally excluded.
# H:C remains identifiable because H changes through time.
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 130)
print("I. IDENTIFIABILITY QA")
print("=" * 130)

station_c_variation = (
    model_panel
    .groupby(
        ["test_id", "station_id"]
    )["C_raw"]
    .nunique(
        dropna=True
    )
)

print(
    "Maximum unique C values within test × station:",
    int(station_c_variation.max())
)

print(
    "C TIME-INVARIANT WITHIN STATION:",
    bool(
        station_c_variation.max() == 1
    )
)

print(
    "beta_C MAIN EFFECT INCLUDED: FALSE"
)

print(
    "PRIMARY SPATIAL PARAMETER: beta_HxC"
)


# --------------------------------------------------------------------------------------------------
# 15. CAMPAIGN SUPPORT QA
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 130)
print("J. FINAL TEST-SPECIFIC PANEL")
print("=" * 130)

print(
    qa.to_string(
        index=False
    )
)

campaign_table = (
    model_panel
    .groupby(
        ["test_id", "campaign_id"]
    )
    .agg(
        n_rows=(
            "response_value_numeric",
            "size"
        ),
        n_stations=(
            "station_id",
            "nunique"
        ),
        n_event_dates=(
            "hydroclimate_event_date",
            "nunique"
        )
    )
    .reset_index()
)

print("\nCampaign coverage:")
print(
    campaign_table.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 16. CHECK THAT H×C HAS ACTUAL VARIATION
# --------------------------------------------------------------------------------------------------

interaction_qa = []

for test_id, d in model_panel.groupby(
    "test_id"
):

    H_sd = float(
        d["H_raw"].std(
            ddof=1
        )
    )

    C_sd = float(
        d["C_raw"].std(
            ddof=1
        )
    )

    Q_sd = float(
        d["log10_Q_raw"].std(
            ddof=1
        )
    )

    interaction_qa.append({
        "test_id":
            test_id,

        "H_sd_positive":
            H_sd > 0,

        "C_sd_positive":
            C_sd > 0,

        "Q_sd_positive":
            Q_sd > 0,

        "H_sd":
            H_sd,

        "C_sd":
            C_sd,

        "log10Q_sd":
            Q_sd
    })

interaction_qa = pd.DataFrame(
    interaction_qa
)

print("\n" + "=" * 130)
print("K. H×C VARIATION GATE")
print("=" * 130)

print(
    interaction_qa.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL HARD GATES
# --------------------------------------------------------------------------------------------------

gate_tests = len(qa) == 5
gate_primary = int(
    (qa["role"] == "PRIMARY").sum()
) == 4

gate_rows = bool(
    (qa["n_rows"] > 0).all()
)

gate_stations = bool(
    (qa["n_stations"] >= 2).all()
)

gate_campaigns = bool(
    (qa["n_campaigns"] >= 2).all()
)

gate_H = bool(
    interaction_qa[
        "H_sd_positive"
    ].all()
)

gate_C = bool(
    interaction_qa[
        "C_sd_positive"
    ].all()
)

gate_Q = bool(
    interaction_qa[
        "Q_sd_positive"
    ].all()
)

gate_H_common = bool(
    qa[
        "H_common_within_event_date"
    ].all()
)

gate_C_fixed = bool(
    qa[
        "C_invariant_within_station"
    ].all()
)

SAFE_FOR_H3_FULL_DATA = all([
    gate_tests,
    gate_primary,
    gate_rows,
    gate_stations,
    gate_campaigns,
    gate_H,
    gate_C,
    gate_Q,
    gate_H_common,
    gate_C_fixed
])


# --------------------------------------------------------------------------------------------------
# 18. SAVE FROZEN MODEL PANEL
# --------------------------------------------------------------------------------------------------

panel_path = (
    OUT /
    "BLOCK111A_H3_MODEL_READY_PANEL_v1.csv"
)

qa_path = (
    OUT /
    "BLOCK111A_H3_MODEL_PANEL_QA_v1.csv"
)

campaign_path = (
    OUT /
    "BLOCK111A_H3_CAMPAIGN_COVERAGE_v1.csv"
)

interaction_path = (
    OUT /
    "BLOCK111A_H3_INTERACTION_VARIATION_QA_v1.csv"
)

tests_path = (
    OUT /
    "BLOCK111A_FROZEN_H3_TEST_BANK_v1.csv"
)

model_panel.to_csv(
    panel_path,
    index=False
)

qa.to_csv(
    qa_path,
    index=False
)

campaign_table.to_csv(
    campaign_path,
    index=False
)

interaction_qa.to_csv(
    interaction_path,
    index=False
)

tests.to_csv(
    tests_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 19. HASH MODEL PANEL
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


panel_sha = sha256_file(
    panel_path
)


# --------------------------------------------------------------------------------------------------
# 20. MANIFEST
# --------------------------------------------------------------------------------------------------

manifest = {

    "block":
        "11.1A",

    "status":
        "PASS" if SAFE_FOR_H3_FULL_DATA else "FAIL",

    "scientific_question":
        (
            "Does station contributing area modulate the expression "
            "of frozen antecedent hydroclimatic forcing?"
        ),

    "hydroclimate_support":
        "COMMON_EVENT_DATE_FORCING",

    "connectivity_variable":
        "log10_contributing_area_km2",

    "primary_parameter":
        "beta_HxC",

    "beta_C_main_effect_included":
        False,

    "beta_C_exclusion_reason":
        (
            "Connectivity is time-invariant within station and its additive "
            "effect is absorbed by the station random intercept."
        ),

    "observed_Q_only":
        True,

    "reconstructed_C09_Q":
        False,

    "window_search":
        False,

    "HxQ":
        False,

    "new_spatial_predictor_search":
        False,

    "response_based_station_selection":
        False,

    "n_tests":
        int(len(qa)),

    "n_primary_tests":
        int(
            (qa["role"] == "PRIMARY").sum()
        ),

    "n_secondary_tests":
        int(
            (qa["role"] != "PRIMARY").sum()
        ),

    "models_fit":
        0,

    "blocks_8_10_modified":
        False,

    "block_110C_modified":
        False,

    "model_panel":
        str(panel_path),

    "model_panel_sha256":
        panel_sha,

    "safe_for_H3_full_data":
        bool(
            SAFE_FOR_H3_FULL_DATA
        )
}

manifest_path = (
    OUT /
    "BLOCK111A_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 21. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 130)
print("BLOCK 11.1A FINAL GATE")
print("=" * 130)

print(
    "TESTS FROZEN              :",
    len(qa),
    "(4 primary + 1 secondary)"
)

print(
    "OBSERVED Q ONLY           : True"
)

print(
    "COMMON EVENT-DATE H       :",
    gate_H_common
)

print(
    "C FIXED WITHIN STATION    :",
    gate_C_fixed
)

print(
    "beta_C INCLUDED           : False"
)

print(
    "PRIMARY H3 PARAMETER      : beta_HxC"
)

print(
    "WINDOW SEARCH             : False"
)

print(
    "H×Q                       : False"
)

print(
    "C09 RECONSTRUCTED Q       : False"
)

print(
    "MODELS FIT                : 0"
)

print(
    "BLOCKS 8–10 MODIFIED      : False"
)

print(
    "SAFE FOR H3 FULL-DATA FIT :",
    SAFE_FOR_H3_FULL_DATA
)

print(
    "\nMODEL PANEL SHA256:"
)

print(
    panel_sha
)

print(
    "\nSaved model panel:"
)

print(
    panel_path
)

print(
    "\nSaved manifest:"
)

print(
    manifest_path
)

print("=" * 130)

---
### Imported master block 147

`BLOCK 11.1B — H3 BAYESIAN FULL-DATA INTERACTION MODELS`


In [ ]:
# ==================================================================================================
# BLOCK 11.1B — H3 BAYESIAN FULL-DATA INTERACTION MODELS
#
# MODEL
# -----
# Y_it = alpha + station_RE_i
#        + beta_Q * z(log10 Q_it)
#        + beta_H * z(H_t)
#        + beta_HxC * [z(H_t) * z(C_i)]
#        + error
#
# C_i = frozen log10 contributing area.
#
# PRIMARY H3 PARAMETER:
# beta_HxC
#
# IMPORTANT
# ---------
# - 5 frozen tests only
# - 4 primary + 1 secondary thermal
# - observed Q only
# - no beta_C main effect
# - no window search
# - no HxQ
# - no reconstructed C09 Q
# - no outcome-based spatial selection
# - no LOCO yet
# - Blocks 8–10 unchanged
#
# RESPONSE TRANSFORMS inherited from frozen architecture:
# EC   -> original scale
# Na   -> log10
# SO4  -> original scale
# Temp -> original scale
#
# Full-data scaling is performed independently within each exact frozen test panel.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
import hashlib
import warnings
import time

warnings.filterwarnings("ignore")

print("=" * 132)
print("BLOCK 11.1B — H3 BAYESIAN FULL-DATA INTERACTION MODELS")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3DIR = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_MODELS"
)

PANEL_PATH = (
    H3DIR /
    "BLOCK111A_H3_MODEL_READY_PANEL_v1.csv"
)

MANIFEST_A_PATH = (
    H3DIR /
    "BLOCK111A_manifest.json"
)

OUT = (
    H3DIR /
    "FULL_DATA"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 1. VERIFY 11.1A FREEZE
# --------------------------------------------------------------------------------------------------

if not PANEL_PATH.exists():
    raise FileNotFoundError(PANEL_PATH)

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(MANIFEST_A_PATH)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


with open(
    MANIFEST_A_PATH,
    "r",
    encoding="utf-8"
) as f:
    manifest_a = json.load(f)


expected_sha = manifest_a[
    "model_panel_sha256"
]

actual_sha = sha256_file(
    PANEL_PATH
)

print("\nA. FREEZE VERIFICATION")
print("-" * 132)

print("Expected SHA256 :", expected_sha)
print("Actual SHA256   :", actual_sha)
print("SHA MATCH       :", expected_sha == actual_sha)
print(
    "11.1A STATUS    :",
    manifest_a.get("status")
)

if expected_sha != actual_sha:
    raise RuntimeError(
        "11.1A model panel hash mismatch."
    )

if not manifest_a.get(
    "safe_for_H3_full_data",
    False
):
    raise RuntimeError(
        "11.1A did not authorize H3 full-data fitting."
    )


# --------------------------------------------------------------------------------------------------
# 2. LOAD FROZEN PANEL
# --------------------------------------------------------------------------------------------------

panel = pd.read_csv(
    PANEL_PATH,
    low_memory=False
)

print("\nPanel shape:", panel.shape)

print(
    "Tests:",
    sorted(
        panel["test_id"]
        .dropna()
        .unique()
    )
)


# --------------------------------------------------------------------------------------------------
# 3. RESPONSE TRANSFORM POLICY
# --------------------------------------------------------------------------------------------------

transform_policy = {

    "H3_EC_P14":
        "identity",

    "H3_EC_P60":
        "identity",

    "H3_NA_P60":
        "log10",

    "H3_SO4_P60":
        "identity",

    "H3_TEMP_SMROOT90":
        "identity"
}


test_order = [
    "H3_EC_P14",
    "H3_EC_P60",
    "H3_NA_P60",
    "H3_SO4_P60",
    "H3_TEMP_SMROOT90"
]


# --------------------------------------------------------------------------------------------------
# 4. MODEL SETTINGS
#
# Strong enough for stable full-data screening.
# We can surgically increase MC only if QA demands it.
# --------------------------------------------------------------------------------------------------

DRAWS = 2000
TUNE = 2000
CHAINS = 4
CORES = 4
TARGET_ACCEPT = 0.95
RANDOM_SEED = 260917

print("\n" + "=" * 132)
print("B. MODEL SETTINGS")
print("=" * 132)

print("Draws         :", DRAWS)
print("Tune          :", TUNE)
print("Chains        :", CHAINS)
print("Target accept :", TARGET_ACCEPT)
print("Seed          :", RANDOM_SEED)


# --------------------------------------------------------------------------------------------------
# 5. HELPERS
# --------------------------------------------------------------------------------------------------

def zscore(x):

    x = np.asarray(
        x,
        dtype=float
    )

    mu = np.mean(x)
    sd = np.std(
        x,
        ddof=1
    )

    if not np.isfinite(sd) or sd <= 0:
        raise RuntimeError(
            "Invalid SD during scaling."
        )

    return (
        (x - mu) / sd,
        mu,
        sd
    )


def transform_response(y, policy):

    y = np.asarray(
        y,
        dtype=float
    )

    if policy == "identity":
        return y

    if policy == "log10":

        if np.any(y <= 0):
            raise RuntimeError(
                "Non-positive response encountered "
                "for frozen log10 transform."
            )

        return np.log10(y)

    raise ValueError(
        f"Unknown transform policy: {policy}"
    )


def posterior_scalar_summary(
    idata,
    varname
):

    arr = (
        idata.posterior[varname]
        .values
        .reshape(-1)
    )

    hdi = az.hdi(
        arr,
        hdi_prob=0.95
    )

    mean = float(
        np.mean(arr)
    )

    median = float(
        np.median(arr)
    )

    sd = float(
        np.std(
            arr,
            ddof=1
        )
    )

    p_pos = float(
        np.mean(arr > 0)
    )

    p_neg = float(
        np.mean(arr < 0)
    )

    pdirection = max(
        p_pos,
        p_neg
    )

    direction = (
        "positive"
        if mean > 0
        else "negative"
    )

    return {
        "mean":
            mean,

        "median":
            median,

        "sd":
            sd,

        "hdi_2.5":
            float(hdi[0]),

        "hdi_97.5":
            float(hdi[1]),

        "P_beta_gt_0":
            p_pos,

        "P_beta_lt_0":
            p_neg,

        "Pdirection":
            pdirection,

        "posterior_direction":
            direction
    }


# --------------------------------------------------------------------------------------------------
# 6. FIT ALL FIVE FROZEN TESTS
# --------------------------------------------------------------------------------------------------

summary_rows = []
qa_rows = []
scaling_rows = []

start_all = time.time()

for test_index, test_id in enumerate(
    test_order,
    start=1
):

    print("\n")
    print("=" * 132)
    print(
        f"FIT {test_index}/5 — {test_id}"
    )
    print("=" * 132)

    d = (
        panel[
            panel["test_id"] == test_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(d) == 0:
        raise RuntimeError(
            f"No rows for {test_id}"
        )

    policy = transform_policy[
        test_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Response
    # ----------------------------------------------------------------------------------------------

    y_raw = pd.to_numeric(
        d["response_value_numeric"],
        errors="raise"
    ).to_numpy()

    y_trans = transform_response(
        y_raw,
        policy
    )

    # Standardize response so coefficient priors
    # and comparisons are on a stable scale.
    y_z, y_mu, y_sd = zscore(
        y_trans
    )

    # ----------------------------------------------------------------------------------------------
    # Predictors
    # ----------------------------------------------------------------------------------------------

    q_raw = pd.to_numeric(
        d["log10_Q_raw"],
        errors="raise"
    ).to_numpy()

    h_raw = pd.to_numeric(
        d["H_raw"],
        errors="raise"
    ).to_numpy()

    c_raw = pd.to_numeric(
        d["C_raw"],
        errors="raise"
    ).to_numpy()

    Qz, Q_mu, Q_sd = zscore(
        q_raw
    )

    Hz, H_mu, H_sd = zscore(
        h_raw
    )

    Cz, C_mu, C_sd = zscore(
        c_raw
    )

    HC = Hz * Cz

    # ----------------------------------------------------------------------------------------------
    # Station index
    # ----------------------------------------------------------------------------------------------

    station_cat = pd.Categorical(
        d["station_id"]
        .astype(str)
    )

    station_idx = (
        station_cat.codes
        .astype(int)
    )

    station_levels = list(
        station_cat.categories
    )

    n_station = len(
        station_levels
    )

    print("Rows          :", len(d))
    print("Stations      :", n_station)
    print(
        "Campaigns     :",
        d["campaign_id"].nunique()
    )
    print(
        "Event dates   :",
        d["hydroclimate_event_date"].nunique()
    )
    print(
        "Transform     :",
        policy
    )

    # ----------------------------------------------------------------------------------------------
    # Save exact scaling
    # ----------------------------------------------------------------------------------------------

    scaling_rows.append({

        "test_id":
            test_id,

        "response_transform":
            policy,

        "y_transformed_mean":
            y_mu,

        "y_transformed_sd":
            y_sd,

        "log10Q_mean":
            Q_mu,

        "log10Q_sd":
            Q_sd,

        "H_mean":
            H_mu,

        "H_sd":
            H_sd,

        "C_log10area_mean":
            C_mu,

        "C_log10area_sd":
            C_sd
    })

    # ----------------------------------------------------------------------------------------------
    # Bayesian mixed model
    # ----------------------------------------------------------------------------------------------

    coords = {
        "station":
            station_levels,

        "obs":
            np.arange(
                len(d)
            )
    }

    with pm.Model(
        coords=coords
    ) as model:

        station_index = pm.Data(
            "station_index",
            station_idx,
            dims="obs"
        )

        q_data = pm.Data(
            "Qz",
            Qz,
            dims="obs"
        )

        h_data = pm.Data(
            "Hz",
            Hz,
            dims="obs"
        )

        hc_data = pm.Data(
            "HC",
            HC,
            dims="obs"
        )

        # Population intercept.
        alpha = pm.Normal(
            "alpha",
            mu=0,
            sigma=1
        )

        # Station random intercept.
        sigma_station = pm.HalfNormal(
            "sigma_station",
            sigma=1
        )

        z_station = pm.Normal(
            "z_station",
            mu=0,
            sigma=1,
            dims="station"
        )

        station_re = pm.Deterministic(
            "station_re",
            z_station * sigma_station,
            dims="station"
        )

        # Fixed effects.
        beta_Q = pm.Normal(
            "beta_Q",
            mu=0,
            sigma=1
        )

        beta_H = pm.Normal(
            "beta_H",
            mu=0,
            sigma=1
        )

        beta_HxC = pm.Normal(
            "beta_HxC",
            mu=0,
            sigma=1
        )

        mu = (
            alpha
            +
            station_re[
                station_index
            ]
            +
            beta_Q * q_data
            +
            beta_H * h_data
            +
            beta_HxC * hc_data
        )

        sigma = pm.HalfNormal(
            "sigma",
            sigma=1
        )

        pm.Normal(
            "Y_obs",
            mu=mu,
            sigma=sigma,
            observed=y_z,
            dims="obs"
        )

        fit_start = time.time()

        idata = pm.sample(
            draws=DRAWS,
            tune=TUNE,
            chains=CHAINS,
            cores=CORES,
            target_accept=TARGET_ACCEPT,
            random_seed=(
                RANDOM_SEED
                + test_index
            ),
            return_inferencedata=True,
            progressbar=True
        )

        fit_seconds = (
            time.time()
            - fit_start
        )

    # ----------------------------------------------------------------------------------------------
    # Posterior summaries
    # ----------------------------------------------------------------------------------------------

    for parameter in [
        "beta_Q",
        "beta_H",
        "beta_HxC"
    ]:

        s = posterior_scalar_summary(
            idata,
            parameter
        )

        summary_rows.append({

            "test_id":
                test_id,

            "response_key":
                d["response_key"].iloc[0],

            "H_variable":
                d["H_variable"].iloc[0],

            "role":
                d["H3_role"].iloc[0],

            "response_transform":
                policy,

            "n_rows":
                len(d),

            "n_stations":
                n_station,

            "n_campaigns":
                d["campaign_id"].nunique(),

            "parameter":
                parameter,

            **s
        })

    # ----------------------------------------------------------------------------------------------
    # QA
    # ----------------------------------------------------------------------------------------------

    var_names_qa = [
        "alpha",
        "beta_Q",
        "beta_H",
        "beta_HxC",
        "sigma_station",
        "sigma"
    ]

    azsum = az.summary(
        idata,
        var_names=var_names_qa,
        round_to=None
    )

    max_rhat = float(
        np.nanmax(
            azsum["r_hat"]
            .to_numpy()
        )
    )

    min_ess_bulk = float(
        np.nanmin(
            azsum["ess_bulk"]
            .to_numpy()
        )
    )

    min_ess_tail = float(
        np.nanmin(
            azsum["ess_tail"]
            .to_numpy()
        )
    )

    divergences = int(
        idata.sample_stats[
            "diverging"
        ]
        .values
        .sum()
    )

    qa_pass = (
        divergences == 0
        and
        max_rhat <= 1.01
        and
        min_ess_bulk >= 400
        and
        min_ess_tail >= 400
    )

    qa_rows.append({

        "test_id":
            test_id,

        "n_rows":
            len(d),

        "n_stations":
            n_station,

        "draws":
            DRAWS,

        "tune":
            TUNE,

        "chains":
            CHAINS,

        "target_accept":
            TARGET_ACCEPT,

        "divergences":
            divergences,

        "max_rhat":
            max_rhat,

        "min_ess_bulk":
            min_ess_bulk,

        "min_ess_tail":
            min_ess_tail,

        "fit_seconds":
            fit_seconds,

        "QA_PASS":
            qa_pass
    })

    # ----------------------------------------------------------------------------------------------
    # Save posterior
    # ----------------------------------------------------------------------------------------------

    posterior_path = (
        OUT /
        f"{test_id}_FULL_DATA_v1.nc"
    )

    idata.to_netcdf(
        posterior_path
    )

    print("\nQA:")
    print(
        "divergences =",
        divergences
    )
    print(
        "max Rhat    =",
        max_rhat
    )
    print(
        "min ESSbulk =",
        min_ess_bulk
    )
    print(
        "min ESStail =",
        min_ess_tail
    )
    print(
        "QA PASS     =",
        qa_pass
    )

    hx = posterior_scalar_summary(
        idata,
        "beta_HxC"
    )

    print("\nbeta_HxC:")
    print(
        f"mean={hx['mean']:.6f}"
    )
    print(
        f"95% HDI=[{hx['hdi_2.5']:.6f}, "
        f"{hx['hdi_97.5']:.6f}]"
    )
    print(
        f"Pdirection={hx['Pdirection']:.6f}"
    )


# --------------------------------------------------------------------------------------------------
# 7. COMBINE RESULTS
# --------------------------------------------------------------------------------------------------

summary = pd.DataFrame(
    summary_rows
)

qa = pd.DataFrame(
    qa_rows
)

scaling = pd.DataFrame(
    scaling_rows
)

runtime_seconds = (
    time.time()
    - start_all
)


# --------------------------------------------------------------------------------------------------
# 8. H3 INTERACTION SPECTRUM
# --------------------------------------------------------------------------------------------------

interaction = (
    summary[
        summary["parameter"] ==
        "beta_HxC"
    ]
    .copy()
    .reset_index(drop=True)
)

interaction["HDI_excludes_zero"] = (
    (
        interaction["hdi_2.5"] > 0
    )
    |
    (
        interaction["hdi_97.5"] < 0
    )
)

# This is a FULL-DATA posterior signal gate only.
# It is NOT the final H3 support criterion.
#
# Final support requires subsequent campaign LOCO predictive validation.
interaction["full_data_directional_signal"] = (
    interaction["Pdirection"] >= 0.95
)

interaction["eligible_for_LOCO"] = (
    interaction["full_data_directional_signal"]
)

print("\n" + "=" * 132)
print("C. FULL-DATA H3 INTERACTION SPECTRUM")
print("=" * 132)

show_cols = [
    "test_id",
    "role",
    "n_rows",
    "n_stations",
    "mean",
    "hdi_2.5",
    "hdi_97.5",
    "Pdirection",
    "HDI_excludes_zero",
    "full_data_directional_signal",
    "eligible_for_LOCO"
]

print(
    interaction[
        show_cols
    ].to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 9. FULL PARAMETER TABLE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 132)
print("D. FULL PARAMETER TABLE")
print("=" * 132)

print(
    summary[
        [
            "test_id",
            "parameter",
            "mean",
            "hdi_2.5",
            "hdi_97.5",
            "Pdirection"
        ]
    ].to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 10. QA TABLE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 132)
print("E. COMPUTATIONAL QA")
print("=" * 132)

print(
    qa.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 11. SAVE TABLES
# --------------------------------------------------------------------------------------------------

summary_path = (
    OUT /
    "BLOCK111B_FULL_DATA_PARAMETER_SUMMARY_v1.csv"
)

interaction_path = (
    OUT /
    "BLOCK111B_H3_INTERACTION_SPECTRUM_v1.csv"
)

qa_path = (
    OUT /
    "BLOCK111B_FULL_DATA_QA_v1.csv"
)

scaling_path = (
    OUT /
    "BLOCK111B_FULL_DATA_SCALING_v1.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

interaction.to_csv(
    interaction_path,
    index=False
)

qa.to_csv(
    qa_path,
    index=False
)

scaling.to_csv(
    scaling_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL GATES
# --------------------------------------------------------------------------------------------------

ALL_QA_PASS = bool(
    qa["QA_PASS"].all()
)

N_DIRECTIONAL = int(
    interaction[
        "full_data_directional_signal"
    ].sum()
)

N_PRIMARY_DIRECTIONAL = int(
    interaction.loc[
        interaction["role"] == "PRIMARY",
        "full_data_directional_signal"
    ].sum()
)

SAFE_FOR_LOCO = (
    ALL_QA_PASS
    and
    N_DIRECTIONAL > 0
)


# --------------------------------------------------------------------------------------------------
# 13. MANIFEST
# --------------------------------------------------------------------------------------------------

manifest = {

    "block":
        "11.1B",

    "status":
        "PASS"
        if ALL_QA_PASS
        else "COMPUTATIONAL_QA_FAIL",

    "model":
        (
            "Yz ~ alpha + station_RE + beta_Q*Qz "
            "+ beta_H*Hz + beta_HxC*(Hz*Cz)"
        ),

    "primary_parameter":
        "beta_HxC",

    "beta_C_main_effect":
        False,

    "n_frozen_tests":
        5,

    "n_primary_tests":
        4,

    "n_secondary_tests":
        1,

    "n_full_data_fits":
        5,

    "all_QA_pass":
        ALL_QA_PASS,

    "n_directional_HxC":
        N_DIRECTIONAL,

    "n_primary_directional_HxC":
        N_PRIMARY_DIRECTIONAL,

    "full_data_signal_gate":
        "Pdirection >= 0.95",

    "full_data_signal_is_final_H3_support":
        False,

    "final_H3_support_requires_LOCO":
        True,

    "safe_for_LOCO":
        SAFE_FOR_LOCO,

    "observed_Q_only":
        True,

    "reconstructed_C09_Q":
        False,

    "window_search":
        False,

    "HxQ":
        False,

    "new_spatial_predictor_search":
        False,

    "blocks_8_10_modified":
        False,

    "runtime_seconds":
        runtime_seconds
}

manifest_path = (
    OUT /
    "BLOCK111B_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 14. FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 132)
print("BLOCK 11.1B FINAL GATE")
print("=" * 132)

print(
    "FULL-DATA FITS            : 5/5"
)

print(
    "ALL COMPUTATIONAL QA PASS :",
    ALL_QA_PASS
)

print(
    "PRIMARY HxC SIGNALS       :",
    f"{N_PRIMARY_DIRECTIONAL}/4"
)

print(
    "TOTAL HxC SIGNALS         :",
    f"{N_DIRECTIONAL}/5"
)

print(
    "FULL-DATA SIGNAL FINAL?   : False"
)

print(
    "FINAL H3 NEEDS LOCO       : True"
)

print(
    "SAFE FOR LOCO             :",
    SAFE_FOR_LOCO
)

print(
    "OBSERVED Q ONLY           : True"
)

print(
    "WINDOW SEARCH             : False"
)

print(
    "H×Q                       : False"
)

print(
    "C09 RECONSTRUCTED Q       : False"
)

print(
    "BLOCKS 8–10 MODIFIED      : False"
)

print(
    "TOTAL RUNTIME MIN         :",
    round(
        runtime_seconds / 60,
        2
    )
)

print("\nSaved:")
print(interaction_path)
print(qa_path)
print(manifest_path)

print("=" * 132)

---
### Imported master block 148

`BLOCK 11.2A — COMPLETE H3 CAMPAIGN-LOCO PREDICTIVE VALIDATION`


In [ ]:
# ==================================================================================================
# BLOCK 11.2A — COMPLETE H3 CAMPAIGN-LOCO PREDICTIVE VALIDATION
#
# PURPOSE
# -------
# Test whether adding the frozen hydroclimate × contributing-area interaction improves
# out-of-campaign prediction beyond the corresponding additive H+Q model.
#
# BASE:
#   Y ~ Q + H + station_RE
#
# H3:
#   Y ~ Q + H + H:C + station_RE
#
# C = frozen log10 contributing area
#
# DESIGN
# ------
# 5 frozen tests
# × 10 held-out campaigns
# × 2 models
# = 100 Bayesian fits
#
# STRICT RULES
# ------------
# - observed Q only
# - exact frozen Block 11.1A rows
# - same rows BASE vs H3
# - training-only scaling in every fold
# - no beta_C main effect
# - no window search
# - no HxQ
# - no reconstructed C09 Q
# - unseen held-out stations marginalized over station RE distribution
# - campaign-equal predictive aggregation
# - resumable checkpoints after EVERY completed fit
#
# PRE-FROZEN FINAL H3 SUPPORT RULE
# --------------------------------
# Full-data Pdirection(beta_HxC) >= 0.95
# AND
# campaign-equal mean ΔLPD(H3 - BASE) > 0
#
# Number of campaigns with ΔLPD > 0 is DESCRIPTIVE ONLY.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
import hashlib
import warnings
import time
import gc

warnings.filterwarnings("ignore")

print("=" * 135)
print("BLOCK 11.2A — COMPLETE H3 CAMPAIGN-LOCO PREDICTIVE VALIDATION")
print("=" * 135)


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3DIR = PROJECT / "CONNECTIVITY_ANALYSIS" / "H3_MODELS"

PANEL_PATH = (
    H3DIR /
    "BLOCK111A_H3_MODEL_READY_PANEL_v1.csv"
)

MANIFEST_A_PATH = (
    H3DIR /
    "BLOCK111A_manifest.json"
)

FULLDIR = (
    H3DIR /
    "FULL_DATA"
)

FULL_INTERACTION_PATH = (
    FULLDIR /
    "BLOCK111B_H3_INTERACTION_SPECTRUM_v1.csv"
)

MANIFEST_B_PATH = (
    FULLDIR /
    "BLOCK111B_manifest.json"
)

OUT = (
    H3DIR /
    "LOCO"
)

CHECKPOINT_DIR = (
    OUT /
    "CHECKPOINTS"
)

POSTERIOR_DIR = (
    OUT /
    "POSTERIORS"
)

OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
POSTERIOR_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 1. FREEZE VERIFICATION
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


with open(
    MANIFEST_A_PATH,
    "r",
    encoding="utf-8"
) as f:
    manifest_a = json.load(f)

with open(
    MANIFEST_B_PATH,
    "r",
    encoding="utf-8"
) as f:
    manifest_b = json.load(f)

expected_sha = manifest_a["model_panel_sha256"]
actual_sha = sha256_file(PANEL_PATH)

print("\nA. FREEZE VERIFICATION")
print("-" * 135)

print("Expected panel SHA :", expected_sha)
print("Actual panel SHA   :", actual_sha)
print("SHA MATCH          :", expected_sha == actual_sha)
print("11.1A status       :", manifest_a.get("status"))
print("11.1B status       :", manifest_b.get("status"))
print("11.1B QA pass      :", manifest_b.get("all_QA_pass"))

if expected_sha != actual_sha:
    raise RuntimeError("11.1A panel SHA mismatch.")

if manifest_a.get("status") != "PASS":
    raise RuntimeError("11.1A not PASS.")

if not manifest_b.get("all_QA_pass", False):
    raise RuntimeError("11.1B computational QA not PASS.")


# --------------------------------------------------------------------------------------------------
# 2. LOAD FROZEN DATA
# --------------------------------------------------------------------------------------------------

panel = pd.read_csv(
    PANEL_PATH,
    low_memory=False
)

full_interaction = pd.read_csv(
    FULL_INTERACTION_PATH,
    low_memory=False
)

test_order = [
    "H3_EC_P14",
    "H3_EC_P60",
    "H3_NA_P60",
    "H3_SO4_P60",
    "H3_TEMP_SMROOT90"
]

transform_policy = {
    "H3_EC_P14": "identity",
    "H3_EC_P60": "identity",
    "H3_NA_P60": "log10",
    "H3_SO4_P60": "identity",
    "H3_TEMP_SMROOT90": "identity"
}

campaign_order = sorted(
    panel["campaign_id"]
    .dropna()
    .unique()
)

print("\nTests     :", test_order)
print("Campaigns :", campaign_order)

if len(campaign_order) != 10:
    raise RuntimeError(
        f"Expected 10 campaigns, found {len(campaign_order)}."
    )


# --------------------------------------------------------------------------------------------------
# 3. SETTINGS
#
# LOCO needs many fits. These settings are deliberately lighter than full-data but still substantial.
# Any fit failing QA is NOT silently accepted; it is flagged for surgical repair.
# --------------------------------------------------------------------------------------------------

DRAWS = 1000
TUNE = 1000
CHAINS = 4
CORES = 4
TARGET_ACCEPT = 0.95
SEED = 260918

QA_MAX_RHAT = 1.01
QA_MIN_ESS = 400

print("\n" + "=" * 135)
print("B. LOCO SETTINGS")
print("=" * 135)

print("Planned folds :", 5 * 10)
print("Planned fits  :", 5 * 10 * 2)
print("Draws         :", DRAWS)
print("Tune          :", TUNE)
print("Chains        :", CHAINS)
print("Target accept :", TARGET_ACCEPT)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def transform_y(y, policy):

    y = np.asarray(y, dtype=float)

    if policy == "identity":
        return y

    if policy == "log10":

        if np.any(y <= 0):
            raise RuntimeError(
                "Non-positive response under frozen log10 transform."
            )

        return np.log10(y)

    raise ValueError(policy)


def train_scale(train, test):

    train = np.asarray(train, dtype=float)
    test = np.asarray(test, dtype=float)

    mu = np.mean(train)
    sd = np.std(train, ddof=1)

    if not np.isfinite(sd) or sd <= 0:
        raise RuntimeError("Invalid training SD.")

    return (
        (train - mu) / sd,
        (test - mu) / sd,
        mu,
        sd
    )


def logmeanexp(a, axis=0):

    a = np.asarray(a)

    m = np.max(a, axis=axis, keepdims=True)

    out = (
        np.log(
            np.mean(
                np.exp(a - m),
                axis=axis
            )
        )
        +
        np.squeeze(m, axis=axis)
    )

    return out


def normal_logpdf(y, mu, sigma):

    return (
        -0.5 * np.log(2 * np.pi)
        - np.log(sigma)
        - 0.5 * ((y - mu) / sigma) ** 2
    )


# --------------------------------------------------------------------------------------------------
# 5. FIT ONE TRAINING MODEL
# --------------------------------------------------------------------------------------------------

def fit_model(
    train_df,
    model_type,
    seed
):

    policy = transform_policy[
        train_df["test_id"].iloc[0]
    ]

    y_raw = pd.to_numeric(
        train_df["response_value_numeric"],
        errors="raise"
    ).to_numpy()

    y_t = transform_y(
        y_raw,
        policy
    )

    # Training-only response scaling.
    y_mu = np.mean(y_t)
    y_sd = np.std(y_t, ddof=1)

    if y_sd <= 0:
        raise RuntimeError("Invalid response SD.")

    yz = (y_t - y_mu) / y_sd

    q_raw = pd.to_numeric(
        train_df["log10_Q_raw"],
        errors="raise"
    ).to_numpy()

    h_raw = pd.to_numeric(
        train_df["H_raw"],
        errors="raise"
    ).to_numpy()

    c_raw = pd.to_numeric(
        train_df["C_raw"],
        errors="raise"
    ).to_numpy()

    Qz, _, Q_mu, Q_sd = train_scale(
        q_raw,
        q_raw
    )

    Hz, _, H_mu, H_sd = train_scale(
        h_raw,
        h_raw
    )

    Cz, _, C_mu, C_sd = train_scale(
        c_raw,
        c_raw
    )

    HC = Hz * Cz

    station_cat = pd.Categorical(
        train_df["station_id"].astype(str)
    )

    station_idx = station_cat.codes.astype(int)

    station_levels = list(
        station_cat.categories
    )

    coords = {
        "station": station_levels,
        "obs": np.arange(len(train_df))
    }

    with pm.Model(coords=coords) as model:

        st = pm.Data(
            "station_idx",
            station_idx,
            dims="obs"
        )

        qd = pm.Data(
            "Qz",
            Qz,
            dims="obs"
        )

        hd = pm.Data(
            "Hz",
            Hz,
            dims="obs"
        )

        alpha = pm.Normal(
            "alpha",
            0,
            1
        )

        sigma_station = pm.HalfNormal(
            "sigma_station",
            1
        )

        z_station = pm.Normal(
            "z_station",
            0,
            1,
            dims="station"
        )

        station_re = pm.Deterministic(
            "station_re",
            z_station * sigma_station,
            dims="station"
        )

        beta_Q = pm.Normal(
            "beta_Q",
            0,
            1
        )

        beta_H = pm.Normal(
            "beta_H",
            0,
            1
        )

        mu = (
            alpha
            +
            station_re[st]
            +
            beta_Q * qd
            +
            beta_H * hd
        )

        if model_type == "H3":

            hcd = pm.Data(
                "HC",
                HC,
                dims="obs"
            )

            beta_HxC = pm.Normal(
                "beta_HxC",
                0,
                1
            )

            mu = (
                mu
                +
                beta_HxC * hcd
            )

        sigma = pm.HalfNormal(
            "sigma",
            1
        )

        pm.Normal(
            "Y_obs",
            mu=mu,
            sigma=sigma,
            observed=yz,
            dims="obs"
        )

        idata = pm.sample(
            draws=DRAWS,
            tune=TUNE,
            chains=CHAINS,
            cores=CORES,
            target_accept=TARGET_ACCEPT,
            random_seed=seed,
            return_inferencedata=True,
            progressbar=False
        )

    scaling = {
        "y_mu": y_mu,
        "y_sd": y_sd,
        "Q_mu": Q_mu,
        "Q_sd": Q_sd,
        "H_mu": H_mu,
        "H_sd": H_sd,
        "C_mu": C_mu,
        "C_sd": C_sd,
        "response_transform": policy,
        "station_levels": station_levels
    }

    return idata, scaling


# --------------------------------------------------------------------------------------------------
# 6. QA ONE FIT
# --------------------------------------------------------------------------------------------------

def fit_qa(idata, model_type):

    names = [
        "alpha",
        "beta_Q",
        "beta_H",
        "sigma_station",
        "sigma"
    ]

    if model_type == "H3":
        names.append("beta_HxC")

    s = az.summary(
        idata,
        var_names=names,
        round_to=None
    )

    divergences = int(
        idata.sample_stats["diverging"]
        .values
        .sum()
    )

    max_rhat = float(
        np.nanmax(
            s["r_hat"].to_numpy()
        )
    )

    min_ess_bulk = float(
        np.nanmin(
            s["ess_bulk"].to_numpy()
        )
    )

    min_ess_tail = float(
        np.nanmin(
            s["ess_tail"].to_numpy()
        )
    )

    qa_pass = (
        divergences == 0
        and
        max_rhat <= QA_MAX_RHAT
        and
        min_ess_bulk >= QA_MIN_ESS
        and
        min_ess_tail >= QA_MIN_ESS
    )

    return {
        "divergences": divergences,
        "max_rhat": max_rhat,
        "min_ess_bulk": min_ess_bulk,
        "min_ess_tail": min_ess_tail,
        "QA_PASS": qa_pass
    }


# --------------------------------------------------------------------------------------------------
# 7. HELD-OUT PREDICTIVE LPD
#
# For stations observed in training:
# use posterior station RE.
#
# For stations absent from training:
# marginalize station effect by Monte Carlo:
#   b_new ~ Normal(0, sigma_station)
#
# This preserves the hierarchical prediction target rather than pretending b_new = 0.
# --------------------------------------------------------------------------------------------------

def heldout_lpd(
    idata,
    scaling,
    test_df,
    model_type,
    rng
):

    policy = scaling[
        "response_transform"
    ]

    y_raw = pd.to_numeric(
        test_df["response_value_numeric"],
        errors="raise"
    ).to_numpy()

    y_t = transform_y(
        y_raw,
        policy
    )

    yz = (
        y_t
        -
        scaling["y_mu"]
    ) / scaling["y_sd"]

    q = pd.to_numeric(
        test_df["log10_Q_raw"],
        errors="raise"
    ).to_numpy()

    h = pd.to_numeric(
        test_df["H_raw"],
        errors="raise"
    ).to_numpy()

    c = pd.to_numeric(
        test_df["C_raw"],
        errors="raise"
    ).to_numpy()

    Qz = (
        q
        -
        scaling["Q_mu"]
    ) / scaling["Q_sd"]

    Hz = (
        h
        -
        scaling["H_mu"]
    ) / scaling["H_sd"]

    Cz = (
        c
        -
        scaling["C_mu"]
    ) / scaling["C_sd"]

    HC = Hz * Cz

    posterior = idata.posterior.stack(
        sample=("chain", "draw")
    )

    alpha = posterior[
        "alpha"
    ].values.reshape(-1)

    beta_Q = posterior[
        "beta_Q"
    ].values.reshape(-1)

    beta_H = posterior[
        "beta_H"
    ].values.reshape(-1)

    sigma = posterior[
        "sigma"
    ].values.reshape(-1)

    sigma_station = posterior[
        "sigma_station"
    ].values.reshape(-1)

    if model_type == "H3":

        beta_HxC = posterior[
            "beta_HxC"
        ].values.reshape(-1)

    station_re = posterior[
        "station_re"
    ].values

    # ArviZ orientation may be station × sample.
    if station_re.shape[0] == len(
        scaling["station_levels"]
    ):
        station_re = station_re.T

    train_station_map = {
        s: i
        for i, s in enumerate(
            scaling["station_levels"]
        )
    }

    n_samples = len(alpha)
    n_obs = len(test_df)

    loglik = np.empty(
        (n_samples, n_obs),
        dtype=float
    )

    station_values = (
        test_df["station_id"]
        .astype(str)
        .to_numpy()
    )

    for j in range(n_obs):

        station = station_values[j]

        if station in train_station_map:

            b = station_re[
                :,
                train_station_map[station]
            ]

        else:

            # One draw of new-station RE for each posterior draw.
            b = rng.normal(
                loc=0.0,
                scale=sigma_station
            )

        mu = (
            alpha
            +
            b
            +
            beta_Q * Qz[j]
            +
            beta_H * Hz[j]
        )

        if model_type == "H3":

            mu = (
                mu
                +
                beta_HxC * HC[j]
            )

        loglik[:, j] = normal_logpdf(
            yz[j],
            mu,
            sigma
        )

    pointwise_lpd = logmeanexp(
        loglik,
        axis=0
    )

    return {
        "lpd_sum": float(
            np.sum(pointwise_lpd)
        ),

        "lpd_mean_per_obs": float(
            np.mean(pointwise_lpd)
        ),

        "pointwise_lpd":
            pointwise_lpd,

        "n_test":
            n_obs,

        "n_unseen_stations":
            int(
                sum(
                    s not in train_station_map
                    for s in station_values
                )
            )
    }


# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT UTILITIES
# --------------------------------------------------------------------------------------------------

FIT_RESULTS_PATH = (
    CHECKPOINT_DIR /
    "BLOCK112A_FIT_RESULTS_CHECKPOINT.csv"
)

POINTWISE_PATH = (
    CHECKPOINT_DIR /
    "BLOCK112A_POINTWISE_LPD_CHECKPOINT.csv"
)


if FIT_RESULTS_PATH.exists():

    fit_results = pd.read_csv(
        FIT_RESULTS_PATH
    )

else:

    fit_results = pd.DataFrame()


if POINTWISE_PATH.exists():

    pointwise_results = pd.read_csv(
        POINTWISE_PATH
    )

else:

    pointwise_results = pd.DataFrame()


def fit_already_complete(
    test_id,
    campaign,
    model_type
):

    if fit_results.empty:
        return False

    m = (
        (fit_results["test_id"] == test_id)
        &
        (fit_results["heldout_campaign"] == campaign)
        &
        (fit_results["model_type"] == model_type)
        &
        (fit_results["fit_completed"] == True)
    )

    return bool(
        m.any()
    )


# --------------------------------------------------------------------------------------------------
# 9. RUN 100 FITS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("C. RUNNING / RESUMING 100 LOCO FITS")
print("=" * 135)

run_start = time.time()

fit_counter = 0

for test_num, test_id in enumerate(
    test_order,
    start=1
):

    d_all = (
        panel[
            panel["test_id"] == test_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    print("\n" + "#" * 135)
    print(
        f"TEST {test_num}/5 — {test_id}"
    )
    print("#" * 135)

    for camp_num, campaign in enumerate(
        campaign_order,
        start=1
    ):

        train_df = (
            d_all[
                d_all["campaign_id"] != campaign
            ]
            .copy()
            .reset_index(drop=True)
        )

        test_df = (
            d_all[
                d_all["campaign_id"] == campaign
            ]
            .copy()
            .reset_index(drop=True)
        )

        if len(test_df) == 0:
            raise RuntimeError(
                f"{test_id}/{campaign}: empty held-out fold."
            )

        print(
            f"\n[{test_id}] "
            f"LOCO {camp_num}/10 — hold out {campaign} "
            f"(train={len(train_df)}, test={len(test_df)})"
        )

        for model_num, model_type in enumerate(
            ["BASE", "H3"],
            start=1
        ):

            fit_counter += 1

            if fit_already_complete(
                test_id,
                campaign,
                model_type
            ):

                print(
                    f"  {model_type}: checkpoint exists -> SKIP"
                )

                continue

            print(
                f"  {model_type}: fitting..."
            )

            seed_here = (
                SEED
                +
                test_num * 10000
                +
                camp_num * 100
                +
                model_num
            )

            t0 = time.time()

            idata, scaling = fit_model(
                train_df,
                model_type,
                seed_here
            )

            seconds = time.time() - t0

            qa = fit_qa(
                idata,
                model_type
            )

            rng = np.random.default_rng(
                seed_here + 500000
            )

            pred = heldout_lpd(
                idata,
                scaling,
                test_df,
                model_type,
                rng
            )

            posterior_path = (
                POSTERIOR_DIR /
                f"{test_id}_{campaign}_{model_type}_v1.nc"
            )

            idata.to_netcdf(
                posterior_path
            )

            row = {
                "test_id":
                    test_id,

                "role":
                    test_df["H3_role"].iloc[0],

                "response_key":
                    test_df["response_key"].iloc[0],

                "H_variable":
                    test_df["H_variable"].iloc[0],

                "heldout_campaign":
                    campaign,

                "model_type":
                    model_type,

                "n_train":
                    len(train_df),

                "n_test":
                    len(test_df),

                "n_train_stations":
                    train_df["station_id"].nunique(),

                "n_test_stations":
                    test_df["station_id"].nunique(),

                "n_unseen_test_station_rows":
                    pred["n_unseen_stations"],

                "lpd_sum":
                    pred["lpd_sum"],

                "lpd_mean_per_obs":
                    pred["lpd_mean_per_obs"],

                "divergences":
                    qa["divergences"],

                "max_rhat":
                    qa["max_rhat"],

                "min_ess_bulk":
                    qa["min_ess_bulk"],

                "min_ess_tail":
                    qa["min_ess_tail"],

                "QA_PASS":
                    qa["QA_PASS"],

                "fit_seconds":
                    seconds,

                "fit_completed":
                    True,

                "posterior_file":
                    str(posterior_path)
            }

            fit_results = pd.concat(
                [
                    fit_results,
                    pd.DataFrame([row])
                ],
                ignore_index=True
            )

            # Pointwise predictions.
            ptemp = pd.DataFrame({

                "test_id":
                    test_id,

                "heldout_campaign":
                    campaign,

                "model_type":
                    model_type,

                "test_row_index":
                    np.arange(
                        len(test_df)
                    ),

                "station_id":
                    test_df["station_id"]
                    .astype(str)
                    .to_numpy(),

                "field_observation_id":
                    test_df["field_observation_id"]
                    .astype(str)
                    .to_numpy(),

                "pointwise_lpd":
                    pred["pointwise_lpd"]
            })

            pointwise_results = pd.concat(
                [
                    pointwise_results,
                    ptemp
                ],
                ignore_index=True
            )

            # Save immediately.
            fit_results.to_csv(
                FIT_RESULTS_PATH,
                index=False
            )

            pointwise_results.to_csv(
                POINTWISE_PATH,
                index=False
            )

            print(
                f"    LPD mean/obs={pred['lpd_mean_per_obs']:.5f} | "
                f"div={qa['divergences']} | "
                f"Rhat={qa['max_rhat']:.4f} | "
                f"ESSbulk={qa['min_ess_bulk']:.0f} | "
                f"QA={qa['QA_PASS']}"
            )

            del idata
            gc.collect()


# --------------------------------------------------------------------------------------------------
# 10. VERIFY COMPLETENESS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("D. COMPLETENESS")
print("=" * 135)

fit_results = pd.read_csv(
    FIT_RESULTS_PATH
)

expected_fit_keys = set()

for test_id in test_order:
    for campaign in campaign_order:
        for model_type in ["BASE", "H3"]:

            expected_fit_keys.add(
                (
                    test_id,
                    campaign,
                    model_type
                )
            )


observed_fit_keys = set(
    zip(
        fit_results["test_id"],
        fit_results["heldout_campaign"],
        fit_results["model_type"]
    )
)

missing_keys = (
    expected_fit_keys
    -
    observed_fit_keys
)

print(
    "Expected fits :",
    len(expected_fit_keys)
)

print(
    "Completed fits:",
    len(observed_fit_keys)
)

print(
    "Missing fits  :",
    len(missing_keys)
)

if missing_keys:

    print(
        sorted(
            missing_keys
        )
    )

    raise RuntimeError(
        "LOCO incomplete. Re-run same cell to resume."
    )


# --------------------------------------------------------------------------------------------------
# 11. COMPUTATIONAL QA
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("E. COMPUTATIONAL QA")
print("=" * 135)

qa_fail = fit_results[
    ~fit_results["QA_PASS"].astype(bool)
].copy()

print(
    "Fits QA PASS:",
    int(
        fit_results["QA_PASS"]
        .astype(bool)
        .sum()
    ),
    "/",
    len(fit_results)
)

print(
    "QA failures:",
    len(qa_fail)
)

if len(qa_fail):

    print(
        qa_fail[
            [
                "test_id",
                "heldout_campaign",
                "model_type",
                "divergences",
                "max_rhat",
                "min_ess_bulk",
                "min_ess_tail"
            ]
        ].to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. CREATE EXACT BASE vs H3 CAMPAIGN CONTRASTS
#
# Campaign-equal inference uses mean of campaign-specific mean-per-observation LPD differences.
# This prevents large campaigns from automatically dominating.
# --------------------------------------------------------------------------------------------------

base = (
    fit_results[
        fit_results["model_type"] == "BASE"
    ]
    .copy()
)

h3 = (
    fit_results[
        fit_results["model_type"] == "H3"
    ]
    .copy()
)

contrast = base.merge(
    h3,
    on=[
        "test_id",
        "heldout_campaign"
    ],
    suffixes=(
        "_BASE",
        "_H3"
    ),
    validate="one_to_one"
)

contrast["delta_LPD_H3_minus_BASE"] = (
    contrast["lpd_mean_per_obs_H3"]
    -
    contrast["lpd_mean_per_obs_BASE"]
)

contrast["H3_better"] = (
    contrast[
        "delta_LPD_H3_minus_BASE"
    ] > 0
)


# --------------------------------------------------------------------------------------------------
# 13. CAMPAIGN-EQUAL SUMMARY
# --------------------------------------------------------------------------------------------------

campaign_summary = (
    contrast
    .groupby("test_id")
    .agg(
        n_campaigns=(
            "heldout_campaign",
            "nunique"
        ),

        mean_delta_LPD=(
            "delta_LPD_H3_minus_BASE",
            "mean"
        ),

        median_delta_LPD=(
            "delta_LPD_H3_minus_BASE",
            "median"
        ),

        min_delta_LPD=(
            "delta_LPD_H3_minus_BASE",
            "min"
        ),

        max_delta_LPD=(
            "delta_LPD_H3_minus_BASE",
            "max"
        ),

        campaigns_H3_better=(
            "H3_better",
            "sum"
        )
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 14. JOIN FULL-DATA beta_HxC
# --------------------------------------------------------------------------------------------------

fd = (
    full_interaction[
        [
            "test_id",
            "role",
            "mean",
            "hdi_2.5",
            "hdi_97.5",
            "Pdirection"
        ]
    ]
    .copy()
    .rename(
        columns={
            "mean":
                "beta_HxC_mean",

            "hdi_2.5":
                "beta_HxC_hdi_2.5",

            "hdi_97.5":
                "beta_HxC_hdi_97.5",

            "Pdirection":
                "beta_HxC_Pdirection"
        }
    )
)

final = fd.merge(
    campaign_summary,
    on="test_id",
    how="left",
    validate="one_to_one"
)


# --------------------------------------------------------------------------------------------------
# 15. APPLY PRE-FROZEN FINAL H3 SUPPORT RULE
# --------------------------------------------------------------------------------------------------

final[
    "directional_gate"
] = (
    final[
        "beta_HxC_Pdirection"
    ] >= 0.95
)

final[
    "predictive_LOCO_gate"
] = (
    final[
        "mean_delta_LPD"
    ] > 0
)

final[
    "H3_SUPPORTED"
] = (
    final[
        "directional_gate"
    ]
    &
    final[
        "predictive_LOCO_gate"
    ]
)

final[
    "campaign_count_is_gate"
] = False


# --------------------------------------------------------------------------------------------------
# 16. PRINT CAMPAIGN CONTRASTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("F. ALL 50 CAMPAIGN-LEVEL H3 CONTRASTS")
print("=" * 135)

print(
    contrast[
        [
            "test_id",
            "heldout_campaign",
            "n_test_BASE",
            "lpd_mean_per_obs_BASE",
            "lpd_mean_per_obs_H3",
            "delta_LPD_H3_minus_BASE",
            "H3_better"
        ]
    ].to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 17. PRINT FINAL H3 SPECTRUM
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("G. FINAL H3 SUPPORT SPECTRUM")
print("=" * 135)

print(
    final[
        [
            "test_id",
            "role",
            "beta_HxC_mean",
            "beta_HxC_hdi_2.5",
            "beta_HxC_hdi_97.5",
            "beta_HxC_Pdirection",
            "mean_delta_LPD",
            "median_delta_LPD",
            "campaigns_H3_better",
            "n_campaigns",
            "directional_gate",
            "predictive_LOCO_gate",
            "H3_SUPPORTED"
        ]
    ].to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 18. SAVE FINAL OUTPUTS
# --------------------------------------------------------------------------------------------------

fit_final_path = (
    OUT /
    "BLOCK112A_ALL_100_LOCO_FITS_v1.csv"
)

contrast_path = (
    OUT /
    "BLOCK112A_ALL_50_CAMPAIGN_CONTRASTS_v1.csv"
)

summary_path = (
    OUT /
    "BLOCK112A_CAMPAIGN_EQUAL_SUMMARY_v1.csv"
)

final_path = (
    OUT /
    "BLOCK112A_FINAL_H3_SUPPORT_SPECTRUM_v1.csv"
)

qa_path = (
    OUT /
    "BLOCK112A_COMPUTATIONAL_QA_v1.csv"
)

fit_results.to_csv(
    fit_final_path,
    index=False
)

contrast.to_csv(
    contrast_path,
    index=False
)

campaign_summary.to_csv(
    summary_path,
    index=False
)

final.to_csv(
    final_path,
    index=False
)

fit_results[
    [
        "test_id",
        "heldout_campaign",
        "model_type",
        "divergences",
        "max_rhat",
        "min_ess_bulk",
        "min_ess_tail",
        "QA_PASS"
    ]
].to_csv(
    qa_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 19. FINAL MANIFEST
# --------------------------------------------------------------------------------------------------

ALL_100_COMPLETE = (
    len(observed_fit_keys) == 100
)

ALL_QA_PASS = bool(
    fit_results[
        "QA_PASS"
    ].astype(bool).all()
)

N_PRIMARY_SUPPORTED = int(
    final.loc[
        final["role"] == "PRIMARY",
        "H3_SUPPORTED"
    ].sum()
)

N_TOTAL_SUPPORTED = int(
    final["H3_SUPPORTED"].sum()
)

runtime_min = (
    time.time()
    -
    run_start
) / 60


manifest = {

    "block":
        "11.2A",

    "status":
        (
            "PASS"
            if ALL_100_COMPLETE
            and ALL_QA_PASS
            else "QA_REPAIR_REQUIRED"
        ),

    "expected_fits":
        100,

    "completed_fits":
        int(
            len(observed_fit_keys)
        ),

    "expected_campaign_contrasts":
        50,

    "completed_campaign_contrasts":
        int(
            len(contrast)
        ),

    "all_computational_QA_pass":
        ALL_QA_PASS,

    "primary_H3_supported":
        N_PRIMARY_SUPPORTED,

    "total_H3_supported":
        N_TOTAL_SUPPORTED,

    "final_support_rule":
        (
            "full-data Pdirection(beta_HxC) >= 0.95 "
            "AND campaign-equal mean delta_LPD(H3-BASE) > 0"
        ),

    "campaign_positive_count_is_gate":
        False,

    "observed_Q_only":
        True,

    "training_only_scaling":
        True,

    "unseen_station_prediction":
        "marginalized_over_station_RE",

    "window_search":
        False,

    "HxQ":
        False,

    "beta_C_main_effect":
        False,

    "reconstructed_C09_Q":
        False,

    "blocks_8_10_modified":
        False,

    "runtime_minutes_this_execution":
        runtime_min
}

manifest_path = (
    OUT /
    "BLOCK112A_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 20. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("BLOCK 11.2A FINAL GATE")
print("=" * 135)

print(
    "LOCO FITS COMPLETE        :",
    f"{len(observed_fit_keys)}/100"
)

print(
    "CAMPAIGN CONTRASTS        :",
    f"{len(contrast)}/50"
)

print(
    "ALL COMPUTATIONAL QA PASS :",
    ALL_QA_PASS
)

print(
    "PRIMARY H3 SUPPORTED      :",
    f"{N_PRIMARY_SUPPORTED}/4"
)

print(
    "TOTAL H3 SUPPORTED        :",
    f"{N_TOTAL_SUPPORTED}/5"
)

print(
    "SUPPORT RULE              : "
    "Pdirection >= 0.95 AND mean ΔLPD > 0"
)

print(
    "CAMPAIGN COUNT IS GATE    : False"
)

print(
    "OBSERVED Q ONLY           : True"
)

print(
    "TRAINING-ONLY SCALING     : True"
)

print(
    "UNSEEN STATIONS           : marginalized"
)

print(
    "WINDOW SEARCH             : False"
)

print(
    "H×Q                       : False"
)

print(
    "C09 RECONSTRUCTED Q       : False"
)

print(
    "BLOCKS 8–10 MODIFIED      : False"
)

print(
    "STATUS                    :",
    manifest["status"]
)

print(
    "\nRuntime this execution min:",
    round(runtime_min, 2)
)

print("\nSaved:")
print(final_path)
print(contrast_path)
print(qa_path)
print(manifest_path)

print("=" * 135)

---
### Imported master block 149

`BLOCK 11.2A-R1 — SURGICAL QA REPAIR`


In [ ]:
# ==================================================================================================
# BLOCK 11.2A-R1 — SURGICAL QA REPAIR
# ONLY: H3_EC_P14 / held-out C10 / BASE
# ==================================================================================================

import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
import time
import gc
from pathlib import Path

print("=" * 135)
print("BLOCK 11.2A-R1 — SURGICAL QA REPAIR")
print("=" * 135)

# --------------------------------------------------------------------------------------------------
# 1. TARGET — FROZEN BEFORE REFIT
# --------------------------------------------------------------------------------------------------

REPAIR_TEST = "H3_EC_P14"
REPAIR_CAMPAIGN = "C10"
REPAIR_MODEL = "BASE"

print("\nA. REPAIR TARGET")
print("-" * 135)
print("Test             :", REPAIR_TEST)
print("Held-out campaign:", REPAIR_CAMPAIGN)
print("Model            :", REPAIR_MODEL)
print("Reason           : original max Rhat = 1.02 > frozen QA threshold 1.01")
print("Other fits refit : FALSE")


# --------------------------------------------------------------------------------------------------
# 2. LOAD CURRENT CHECKPOINT
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3DIR = PROJECT / "CONNECTIVITY_ANALYSIS" / "H3_MODELS"
OUT = H3DIR / "LOCO"
CHECKPOINT_DIR = OUT / "CHECKPOINTS"
POSTERIOR_DIR = OUT / "POSTERIORS"

PANEL_PATH = H3DIR / "BLOCK111A_H3_MODEL_READY_PANEL_v1.csv"

FIT_RESULTS_PATH = (
    CHECKPOINT_DIR /
    "BLOCK112A_FIT_RESULTS_CHECKPOINT.csv"
)

POINTWISE_PATH = (
    CHECKPOINT_DIR /
    "BLOCK112A_POINTWISE_LPD_CHECKPOINT.csv"
)

FULL_INTERACTION_PATH = (
    H3DIR /
    "FULL_DATA" /
    "BLOCK111B_H3_INTERACTION_SPECTRUM_v1.csv"
)

panel = pd.read_csv(PANEL_PATH, low_memory=False)
fit_results = pd.read_csv(FIT_RESULTS_PATH)
pointwise_results = pd.read_csv(POINTWISE_PATH)
full_interaction = pd.read_csv(FULL_INTERACTION_PATH)

target_old = fit_results[
    (fit_results["test_id"] == REPAIR_TEST) &
    (fit_results["heldout_campaign"] == REPAIR_CAMPAIGN) &
    (fit_results["model_type"] == REPAIR_MODEL)
].copy()

if len(target_old) != 1:
    raise RuntimeError(
        f"Expected exactly one target fit, found {len(target_old)}."
    )

print("\nOriginal target fit:")
print(
    target_old[
        [
            "test_id",
            "heldout_campaign",
            "model_type",
            "lpd_mean_per_obs",
            "divergences",
            "max_rhat",
            "min_ess_bulk",
            "min_ess_tail",
            "QA_PASS"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 3. EXACT SAME FROZEN FOLD
# --------------------------------------------------------------------------------------------------

d_all = (
    panel[
        panel["test_id"] == REPAIR_TEST
    ]
    .copy()
    .reset_index(drop=True)
)

train_df = (
    d_all[
        d_all["campaign_id"] != REPAIR_CAMPAIGN
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    d_all[
        d_all["campaign_id"] == REPAIR_CAMPAIGN
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nB. FROZEN FOLD")
print("-" * 135)
print("Train rows:", len(train_df))
print("Test rows :", len(test_df))
print("Train campaigns:", sorted(train_df["campaign_id"].unique()))
print("Held-out campaign:", sorted(test_df["campaign_id"].unique()))

if len(train_df) != 92 or len(test_df) != 8:
    raise RuntimeError(
        "Frozen C10 fold row counts changed."
    )


# --------------------------------------------------------------------------------------------------
# 4. TEMPORARILY INCREASE SAMPLING ONLY FOR THIS FIT
# --------------------------------------------------------------------------------------------------

OLD_DRAWS = DRAWS
OLD_TUNE = TUNE
OLD_CHAINS = CHAINS
OLD_TARGET_ACCEPT = TARGET_ACCEPT

DRAWS = 3000
TUNE = 3000
CHAINS = 4
TARGET_ACCEPT = 0.97

REPAIR_SEED = 261018

print("\nC. REPAIR SETTINGS")
print("-" * 135)
print("Draws         :", DRAWS)
print("Tune          :", TUNE)
print("Chains        :", CHAINS)
print("Target accept :", TARGET_ACCEPT)
print("Seed          :", REPAIR_SEED)


# --------------------------------------------------------------------------------------------------
# 5. REFIT ONLY TARGET
# --------------------------------------------------------------------------------------------------

t0 = time.time()

idata_repair, scaling_repair = fit_model(
    train_df,
    REPAIR_MODEL,
    REPAIR_SEED
)

fit_seconds = time.time() - t0

qa_repair = fit_qa(
    idata_repair,
    REPAIR_MODEL
)

print("\nD. REPAIR QA")
print("-" * 135)
print("Divergences :", qa_repair["divergences"])
print("max Rhat    :", qa_repair["max_rhat"])
print("min ESSbulk :", qa_repair["min_ess_bulk"])
print("min ESStail :", qa_repair["min_ess_tail"])
print("QA PASS     :", qa_repair["QA_PASS"])

if not qa_repair["QA_PASS"]:
    raise RuntimeError(
        "Surgical repair still failed QA. "
        "Do NOT overwrite original results."
    )


# --------------------------------------------------------------------------------------------------
# 6. HELD-OUT PREDICTION USING SAME PROCEDURE
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    REPAIR_SEED + 500000
)

pred_repair = heldout_lpd(
    idata_repair,
    scaling_repair,
    test_df,
    REPAIR_MODEL,
    rng
)

print("\nE. REPAIRED HELD-OUT PREDICTION")
print("-" * 135)
print(
    "Old BASE LPD mean/obs:",
    float(target_old["lpd_mean_per_obs"].iloc[0])
)
print(
    "New BASE LPD mean/obs:",
    pred_repair["lpd_mean_per_obs"]
)
print(
    "Absolute change:",
    abs(
        pred_repair["lpd_mean_per_obs"]
        -
        float(target_old["lpd_mean_per_obs"].iloc[0])
    )
)


# --------------------------------------------------------------------------------------------------
# 7. SAVE REPAIRED POSTERIOR
# --------------------------------------------------------------------------------------------------

repair_posterior_path = (
    POSTERIOR_DIR /
    "H3_EC_P14_C10_BASE_QA_REPAIR_v1.nc"
)

idata_repair.to_netcdf(
    repair_posterior_path
)


# --------------------------------------------------------------------------------------------------
# 8. REPLACE ONLY TARGET FIT ROW
# --------------------------------------------------------------------------------------------------

target_mask = (
    (fit_results["test_id"] == REPAIR_TEST) &
    (fit_results["heldout_campaign"] == REPAIR_CAMPAIGN) &
    (fit_results["model_type"] == REPAIR_MODEL)
)

new_row = fit_results.loc[
    target_mask
].iloc[0].copy()

new_row["lpd_sum"] = pred_repair["lpd_sum"]
new_row["lpd_mean_per_obs"] = pred_repair["lpd_mean_per_obs"]
new_row["n_unseen_test_station_rows"] = pred_repair["n_unseen_stations"]
new_row["divergences"] = qa_repair["divergences"]
new_row["max_rhat"] = qa_repair["max_rhat"]
new_row["min_ess_bulk"] = qa_repair["min_ess_bulk"]
new_row["min_ess_tail"] = qa_repair["min_ess_tail"]
new_row["QA_PASS"] = qa_repair["QA_PASS"]
new_row["fit_seconds"] = fit_seconds
new_row["fit_completed"] = True
new_row["posterior_file"] = str(repair_posterior_path)

fit_results.loc[
    target_mask,
    new_row.index
] = new_row.values


# --------------------------------------------------------------------------------------------------
# 9. REPLACE ONLY TARGET POINTWISE LPD
# --------------------------------------------------------------------------------------------------

point_mask = (
    (pointwise_results["test_id"] == REPAIR_TEST) &
    (pointwise_results["heldout_campaign"] == REPAIR_CAMPAIGN) &
    (pointwise_results["model_type"] == REPAIR_MODEL)
)

old_point_n = int(point_mask.sum())

if old_point_n != len(test_df):
    raise RuntimeError(
        f"Expected {len(test_df)} pointwise rows, found {old_point_n}."
    )

replacement_pointwise = pd.DataFrame({
    "test_id": REPAIR_TEST,
    "heldout_campaign": REPAIR_CAMPAIGN,
    "model_type": REPAIR_MODEL,
    "test_row_index": np.arange(len(test_df)),
    "station_id": test_df["station_id"].astype(str).to_numpy(),
    "field_observation_id": test_df["field_observation_id"].astype(str).to_numpy(),
    "pointwise_lpd": pred_repair["pointwise_lpd"]
})

pointwise_results = pd.concat(
    [
        pointwise_results.loc[~point_mask].copy(),
        replacement_pointwise
    ],
    ignore_index=True
)


# --------------------------------------------------------------------------------------------------
# 10. SAVE REPAIRED CHECKPOINTS
# --------------------------------------------------------------------------------------------------

fit_results.to_csv(
    FIT_RESULTS_PATH,
    index=False
)

pointwise_results.to_csv(
    POINTWISE_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 11. REBUILD ALL 50 CONTRASTS
# --------------------------------------------------------------------------------------------------

base = (
    fit_results[
        fit_results["model_type"] == "BASE"
    ]
    .copy()
)

h3 = (
    fit_results[
        fit_results["model_type"] == "H3"
    ]
    .copy()
)

contrast = base.merge(
    h3,
    on=["test_id", "heldout_campaign"],
    suffixes=("_BASE", "_H3"),
    validate="one_to_one"
)

contrast["delta_LPD_H3_minus_BASE"] = (
    contrast["lpd_mean_per_obs_H3"]
    -
    contrast["lpd_mean_per_obs_BASE"]
)

contrast["H3_better"] = (
    contrast["delta_LPD_H3_minus_BASE"] > 0
)


# --------------------------------------------------------------------------------------------------
# 12. REBUILD CAMPAIGN-EQUAL SUMMARY
# --------------------------------------------------------------------------------------------------

campaign_summary = (
    contrast
    .groupby("test_id")
    .agg(
        n_campaigns=("heldout_campaign", "nunique"),
        mean_delta_LPD=("delta_LPD_H3_minus_BASE", "mean"),
        median_delta_LPD=("delta_LPD_H3_minus_BASE", "median"),
        min_delta_LPD=("delta_LPD_H3_minus_BASE", "min"),
        max_delta_LPD=("delta_LPD_H3_minus_BASE", "max"),
        campaigns_H3_better=("H3_better", "sum")
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 13. REBUILD FINAL SUPPORT SPECTRUM
# --------------------------------------------------------------------------------------------------

fd = (
    full_interaction[
        [
            "test_id",
            "role",
            "mean",
            "hdi_2.5",
            "hdi_97.5",
            "Pdirection"
        ]
    ]
    .copy()
    .rename(
        columns={
            "mean": "beta_HxC_mean",
            "hdi_2.5": "beta_HxC_hdi_2.5",
            "hdi_97.5": "beta_HxC_hdi_97.5",
            "Pdirection": "beta_HxC_Pdirection"
        }
    )
)

final = fd.merge(
    campaign_summary,
    on="test_id",
    how="left",
    validate="one_to_one"
)

final["directional_gate"] = (
    final["beta_HxC_Pdirection"] >= 0.95
)

final["predictive_LOCO_gate"] = (
    final["mean_delta_LPD"] > 0
)

final["H3_SUPPORTED"] = (
    final["directional_gate"]
    &
    final["predictive_LOCO_gate"]
)

final["campaign_count_is_gate"] = False


# --------------------------------------------------------------------------------------------------
# 14. GLOBAL QA
# --------------------------------------------------------------------------------------------------

ALL_QA_PASS = bool(
    fit_results["QA_PASS"]
    .astype(bool)
    .all()
)

qa_fail = fit_results[
    ~fit_results["QA_PASS"].astype(bool)
].copy()

print("\n" + "=" * 135)
print("F. GLOBAL COMPUTATIONAL QA AFTER REPAIR")
print("=" * 135)

print(
    "Fits QA PASS:",
    int(fit_results["QA_PASS"].astype(bool).sum()),
    "/",
    len(fit_results)
)

print(
    "QA failures:",
    len(qa_fail)
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL SUPPORT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("G. REPAIRED FINAL H3 SUPPORT SPECTRUM")
print("=" * 135)

print(
    final[
        [
            "test_id",
            "role",
            "beta_HxC_mean",
            "beta_HxC_hdi_2.5",
            "beta_HxC_hdi_97.5",
            "beta_HxC_Pdirection",
            "mean_delta_LPD",
            "median_delta_LPD",
            "campaigns_H3_better",
            "n_campaigns",
            "directional_gate",
            "predictive_LOCO_gate",
            "H3_SUPPORTED"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 16. OVERWRITE FINAL BLOCK 11.2A PRODUCTS WITH QA-CLEAN VERSION
# --------------------------------------------------------------------------------------------------

fit_final_path = (
    OUT /
    "BLOCK112A_ALL_100_LOCO_FITS_v1.csv"
)

contrast_path = (
    OUT /
    "BLOCK112A_ALL_50_CAMPAIGN_CONTRASTS_v1.csv"
)

summary_path = (
    OUT /
    "BLOCK112A_CAMPAIGN_EQUAL_SUMMARY_v1.csv"
)

final_path = (
    OUT /
    "BLOCK112A_FINAL_H3_SUPPORT_SPECTRUM_v1.csv"
)

qa_path = (
    OUT /
    "BLOCK112A_COMPUTATIONAL_QA_v1.csv"
)

fit_results.to_csv(
    fit_final_path,
    index=False
)

contrast.to_csv(
    contrast_path,
    index=False
)

campaign_summary.to_csv(
    summary_path,
    index=False
)

final.to_csv(
    final_path,
    index=False
)

fit_results[
    [
        "test_id",
        "heldout_campaign",
        "model_type",
        "divergences",
        "max_rhat",
        "min_ess_bulk",
        "min_ess_tail",
        "QA_PASS"
    ]
].to_csv(
    qa_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL MANIFEST
# --------------------------------------------------------------------------------------------------

N_PRIMARY_SUPPORTED = int(
    final.loc[
        final["role"] == "PRIMARY",
        "H3_SUPPORTED"
    ].sum()
)

N_TOTAL_SUPPORTED = int(
    final["H3_SUPPORTED"].sum()
)

manifest = {
    "block": "11.2A-R1",
    "status": "PASS" if ALL_QA_PASS else "QA_REPAIR_REQUIRED",
    "repair_type": "SURGICAL_SINGLE_FIT",
    "repaired_test": REPAIR_TEST,
    "repaired_campaign": REPAIR_CAMPAIGN,
    "repaired_model": REPAIR_MODEL,
    "reason": "original max Rhat 1.02 > 1.01",
    "other_fits_refit": False,
    "completed_fits": 100,
    "completed_campaign_contrasts": 50,
    "all_computational_QA_pass": ALL_QA_PASS,
    "primary_H3_supported": N_PRIMARY_SUPPORTED,
    "total_H3_supported": N_TOTAL_SUPPORTED,
    "final_support_rule":
        "full-data Pdirection(beta_HxC) >= 0.95 AND campaign-equal mean delta_LPD(H3-BASE) > 0",
    "campaign_positive_count_is_gate": False,
    "observed_Q_only": True,
    "training_only_scaling": True,
    "unseen_station_prediction": "marginalized_over_station_RE",
    "window_search": False,
    "HxQ": False,
    "beta_C_main_effect": False,
    "reconstructed_C09_Q": False,
    "blocks_8_10_modified": False
}

manifest_path = (
    OUT /
    "BLOCK112A_R1_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 18. RESTORE GLOBAL SAMPLING CONSTANTS
# --------------------------------------------------------------------------------------------------

DRAWS = OLD_DRAWS
TUNE = OLD_TUNE
CHAINS = OLD_CHAINS
TARGET_ACCEPT = OLD_TARGET_ACCEPT


# --------------------------------------------------------------------------------------------------
# 19. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("BLOCK 11.2A-R1 FINAL GATE")
print("=" * 135)

print("REPAIRED FIT              : H3_EC_P14 / C10 / BASE")
print("OTHER FITS REFIT          : False")
print("LOCO FITS COMPLETE        : 100/100")
print("CAMPAIGN CONTRASTS        : 50/50")
print("ALL COMPUTATIONAL QA PASS :", ALL_QA_PASS)
print("PRIMARY H3 SUPPORTED      :", f"{N_PRIMARY_SUPPORTED}/4")
print("TOTAL H3 SUPPORTED        :", f"{N_TOTAL_SUPPORTED}/5")
print("OBSERVED Q ONLY           : True")
print("TRAINING-ONLY SCALING     : True")
print("WINDOW SEARCH             : False")
print("H×Q                       : False")
print("C09 RECONSTRUCTED Q       : False")
print("BLOCKS 8–10 MODIFIED      : False")
print("STATUS                    :", manifest["status"])

print("\nSaved:")
print(final_path)
print(contrast_path)
print(qa_path)
print(manifest_path)

print("=" * 135)

del idata_repair
gc.collect()

---
### Imported master block 150

`BLOCK 11.3 — H3 MANUSCRIPT FREEZE`


In [ ]:
# ==================================================================================================
# BLOCK 11.3 — H3 MANUSCRIPT FREEZE
#
# NO NEW MODELS.
# NO NEW INFERENCE.
#
# Converts QA-clean Block 11 results into final manuscript/supplementary products.
#
# MAIN TABLE
#   H3 interaction + campaign-LOCO validation
#
# MAIN FIGURE
#   A. full-data beta_HxC ± 95% HDI
#   B. campaign-specific ΔLPD(H3 - BASE)
#
# SUPPLEMENT
#   S1. all 50 campaign contrasts
#   S2. computational QA for all 100 LOCO fits
#
# OUTPUT
#   CSV + XLSX + PNG 600 dpi + vector PDF + manifest + ZIP
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import hashlib
import zipfile
import shutil
import textwrap

from IPython.display import display, Image

print("=" * 135)
print("BLOCK 11.3 — H3 MANUSCRIPT FREEZE")
print("=" * 135)


# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3DIR = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS" /
    "H3_MODELS"
)

LOCO = (
    H3DIR /
    "LOCO"
)

FULL = (
    H3DIR /
    "FULL_DATA"
)

FINAL_SPECTRUM_PATH = (
    LOCO /
    "BLOCK112A_FINAL_H3_SUPPORT_SPECTRUM_v1.csv"
)

CONTRAST_PATH = (
    LOCO /
    "BLOCK112A_ALL_50_CAMPAIGN_CONTRASTS_v1.csv"
)

QA_PATH = (
    LOCO /
    "BLOCK112A_COMPUTATIONAL_QA_v1.csv"
)

FIT_PATH = (
    LOCO /
    "BLOCK112A_ALL_100_LOCO_FITS_v1.csv"
)

R1_MANIFEST_PATH = (
    LOCO /
    "BLOCK112A_R1_manifest.json"
)

FULL_PARAMETER_PATH = (
    FULL /
    "BLOCK111B_FULL_DATA_PARAMETER_SUMMARY_v1.csv"
)

OUT = (
    H3DIR /
    "FINAL_MANUSCRIPT_PACKAGE"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 1. INPUT GATE
# --------------------------------------------------------------------------------------------------

required = {
    "FINAL_H3": FINAL_SPECTRUM_PATH,
    "50_CONTRASTS": CONTRAST_PATH,
    "100_FIT_QA": QA_PATH,
    "100_FITS": FIT_PATH,
    "R1_MANIFEST": R1_MANIFEST_PATH,
    "FULL_PARAMETERS": FULL_PARAMETER_PATH
}

print("\nA. INPUT GATE")
print("-" * 135)

for k, p in required.items():

    print(
        f"{k:20s}",
        "FOUND" if p.exists() else "MISSING",
        p
    )

    if not p.exists():
        raise FileNotFoundError(p)


with open(
    R1_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    upstream_manifest = json.load(f)


if upstream_manifest.get("status") != "PASS":

    raise RuntimeError(
        "Block 11.2A-R1 is not PASS."
    )


if not upstream_manifest.get(
    "all_computational_QA_pass",
    False
):

    raise RuntimeError(
        "Block 11 LOCO computational QA is not clean."
    )


print("\n11.2A-R1 STATUS : PASS")
print("100/100 QA      : PASS")


# --------------------------------------------------------------------------------------------------
# 2. LOAD
# --------------------------------------------------------------------------------------------------

final = pd.read_csv(
    FINAL_SPECTRUM_PATH
)

contrast = pd.read_csv(
    CONTRAST_PATH
)

qa = pd.read_csv(
    QA_PATH
)

fits = pd.read_csv(
    FIT_PATH
)

parameters = pd.read_csv(
    FULL_PARAMETER_PATH
)


# --------------------------------------------------------------------------------------------------
# 3. HARD COMPLETENESS GATES
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("B. COMPLETENESS GATE")
print("=" * 135)

print("Final tests        :", len(final))
print("Campaign contrasts :", len(contrast))
print("LOCO QA rows       :", len(qa))
print("LOCO fit rows      :", len(fits))

if len(final) != 5:
    raise RuntimeError(
        f"Expected 5 final H3 tests; found {len(final)}."
    )

if len(contrast) != 50:
    raise RuntimeError(
        f"Expected 50 campaign contrasts; found {len(contrast)}."
    )

if len(qa) != 100:
    raise RuntimeError(
        f"Expected 100 QA rows; found {len(qa)}."
    )

if len(fits) != 100:
    raise RuntimeError(
        f"Expected 100 fit rows; found {len(fits)}."
    )

if not qa["QA_PASS"].astype(bool).all():

    raise RuntimeError(
        "Not all 100 LOCO fits pass QA."
    )

print("COMPLETENESS : PASS")


# --------------------------------------------------------------------------------------------------
# 4. FIXED MANUSCRIPT LABELS
# --------------------------------------------------------------------------------------------------

label_map = {

    "H3_EC_P14":
        "Electrical conductivity — precipitation 14 d",

    "H3_EC_P60":
        "Electrical conductivity — precipitation 60 d",

    "H3_NA_P60":
        "Na — precipitation 60 d",

    "H3_SO4_P60":
        "SO₄ — precipitation 60 d",

    "H3_TEMP_SMROOT90":
        "Water temperature — root-zone soil moisture 90 d"
}


response_map = {

    "H3_EC_P14":
        "Electrical conductivity",

    "H3_EC_P60":
        "Electrical conductivity",

    "H3_NA_P60":
        "Na",

    "H3_SO4_P60":
        "SO₄",

    "H3_TEMP_SMROOT90":
        "Water temperature"
}


predictor_map = {

    "H3_EC_P14":
        "Precipitation, 14 d",

    "H3_EC_P60":
        "Precipitation, 60 d",

    "H3_NA_P60":
        "Precipitation, 60 d",

    "H3_SO4_P60":
        "Precipitation, 60 d",

    "H3_TEMP_SMROOT90":
        "Root-zone soil moisture, 90 d"
}


order = [
    "H3_EC_P14",
    "H3_EC_P60",
    "H3_NA_P60",
    "H3_SO4_P60",
    "H3_TEMP_SMROOT90"
]


# --------------------------------------------------------------------------------------------------
# 5. MAIN MANUSCRIPT TABLE
# --------------------------------------------------------------------------------------------------

main = final.copy()

main["order"] = main["test_id"].map(
    {
        k: i
        for i, k in enumerate(order)
    }
)

main = (
    main
    .sort_values("order")
    .reset_index(drop=True)
)

main["Response"] = (
    main["test_id"]
    .map(response_map)
)

main["Antecedent predictor"] = (
    main["test_id"]
    .map(predictor_map)
)

main["Role"] = np.where(
    main["role"] == "PRIMARY",
    "Primary",
    "Secondary thermal"
)

main["beta_HxC"] = (
    main["beta_HxC_mean"]
)

main["HDI_lower"] = (
    main["beta_HxC_hdi_2.5"]
)

main["HDI_upper"] = (
    main["beta_HxC_hdi_97.5"]
)

main["Pdirection"] = (
    main["beta_HxC_Pdirection"]
)

main["Mean_delta_LPD"] = (
    main["mean_delta_LPD"]
)

main["Median_delta_LPD"] = (
    main["median_delta_LPD"]
)

main["Campaigns_H3_better"] = (
    main["campaigns_H3_better"]
    .astype(int)
)

main["Inference"] = np.where(
    main["H3_SUPPORTED"].astype(bool),
    "Supported",
    "Not supported"
)


main_table = main[
    [
        "Response",
        "Antecedent predictor",
        "Role",
        "beta_HxC",
        "HDI_lower",
        "HDI_upper",
        "Pdirection",
        "Mean_delta_LPD",
        "Median_delta_LPD",
        "Campaigns_H3_better",
        "Inference"
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 6. PUBLICATION-DISPLAY VERSION
# --------------------------------------------------------------------------------------------------

display_table = main_table.copy()

display_table[
    "βH×C (95% HDI)"
] = display_table.apply(
    lambda r:
        f"{r['beta_HxC']:.3f} "
        f"[{r['HDI_lower']:.3f}, {r['HDI_upper']:.3f}]",
    axis=1
)

display_table[
    "Pdirection"
] = display_table[
    "Pdirection"
].map(
    lambda x: f"{x:.4f}"
)

display_table[
    "Mean ΔLPD"
] = display_table[
    "Mean_delta_LPD"
].map(
    lambda x: f"{x:+.4f}"
)

display_table[
    "Median ΔLPD"
] = display_table[
    "Median_delta_LPD"
].map(
    lambda x: f"{x:+.4f}"
)

display_table[
    "Campaigns H3 > base"
] = display_table[
    "Campaigns_H3_better"
].map(
    lambda x: f"{int(x)}/10"
)

display_table = display_table[
    [
        "Response",
        "Antecedent predictor",
        "Role",
        "βH×C (95% HDI)",
        "Pdirection",
        "Mean ΔLPD",
        "Median ΔLPD",
        "Campaigns H3 > base",
        "Inference"
    ]
]


print("\n" + "=" * 135)
print("C. FINAL MANUSCRIPT TABLE — H3")
print("=" * 135)

print(
    display_table.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 7. SUPPLEMENTARY TABLE S1 — ALL 50 CONTRASTS
# --------------------------------------------------------------------------------------------------

supp_s1 = contrast[
    [
        "test_id",
        "heldout_campaign",
        "n_test_BASE",
        "lpd_mean_per_obs_BASE",
        "lpd_mean_per_obs_H3",
        "delta_LPD_H3_minus_BASE",
        "H3_better"
    ]
].copy()

supp_s1.insert(
    1,
    "test_label",
    supp_s1["test_id"].map(label_map)
)

supp_s1 = supp_s1.sort_values(
    [
        "test_id",
        "heldout_campaign"
    ]
).reset_index(drop=True)


# --------------------------------------------------------------------------------------------------
# 8. SUPPLEMENTARY TABLE S2 — COMPUTATIONAL QA
# --------------------------------------------------------------------------------------------------

supp_s2 = qa.copy()

supp_s2 = supp_s2.sort_values(
    [
        "test_id",
        "heldout_campaign",
        "model_type"
    ]
).reset_index(drop=True)


# --------------------------------------------------------------------------------------------------
# 9. SUPPLEMENTARY TABLE S3 — FULL-DATA COEFFICIENTS
# --------------------------------------------------------------------------------------------------

supp_s3 = parameters[
    parameters["test_id"].isin(order)
].copy()

supp_s3["test_label"] = (
    supp_s3["test_id"]
    .map(label_map)
)

supp_s3 = supp_s3[
    [
        "test_id",
        "test_label",
        "parameter",
        "mean",
        "hdi_2.5",
        "hdi_97.5",
        "Pdirection"
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 10. SAVE CSV TABLES
# --------------------------------------------------------------------------------------------------

main_csv = (
    OUT /
    "TABLE_H3_MAIN_FINAL_v1.csv"
)

main_display_csv = (
    OUT /
    "TABLE_H3_MAIN_DISPLAY_v1.csv"
)

s1_csv = (
    OUT /
    "TABLE_S_H3_ALL_50_LOCO_CONTRASTS_v1.csv"
)

s2_csv = (
    OUT /
    "TABLE_S_H3_COMPUTATIONAL_QA_100_FITS_v1.csv"
)

s3_csv = (
    OUT /
    "TABLE_S_H3_FULL_DATA_COEFFICIENTS_v1.csv"
)

main_table.to_csv(
    main_csv,
    index=False
)

display_table.to_csv(
    main_display_csv,
    index=False
)

supp_s1.to_csv(
    s1_csv,
    index=False
)

supp_s2.to_csv(
    s2_csv,
    index=False
)

supp_s3.to_csv(
    s3_csv,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 11. CONSOLIDATED EXCEL
# --------------------------------------------------------------------------------------------------

xlsx_path = (
    OUT /
    "BLOCK11_H3_FINAL_TABLES_v1.xlsx"
)

with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl"
) as writer:

    display_table.to_excel(
        writer,
        sheet_name="Main_Table_H3",
        index=False
    )

    main_table.to_excel(
        writer,
        sheet_name="Main_Table_numeric",
        index=False
    )

    supp_s1.to_excel(
        writer,
        sheet_name="S1_50_LOCO_contrasts",
        index=False
    )

    supp_s2.to_excel(
        writer,
        sheet_name="S2_QA_100_fits",
        index=False
    )

    supp_s3.to_excel(
        writer,
        sheet_name="S3_full_data_coeff",
        index=False
    )


# --------------------------------------------------------------------------------------------------
# 12. FIGURE DATA
# --------------------------------------------------------------------------------------------------

fig_main = main.copy()

fig_main["label"] = (
    fig_main["test_id"]
    .map(label_map)
)

fig_main["ypos"] = np.arange(
    len(fig_main)
)[::-1]


contrast_plot = contrast.copy()

contrast_plot["label"] = (
    contrast_plot["test_id"]
    .map(label_map)
)

contrast_plot["test_order"] = (
    contrast_plot["test_id"]
    .map(
        {
            k: i
            for i, k in enumerate(order)
        }
    )
)


# --------------------------------------------------------------------------------------------------
# 13. FIGURE H3
# --------------------------------------------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig = plt.figure(
    figsize=(11.8, 8.0)
)

gs = fig.add_gridspec(
    2,
    1,
    height_ratios=[1.0, 1.45],
    hspace=0.38
)


# --------------------------------------------------------------------------------------------------
# PANEL A — beta HxC
# --------------------------------------------------------------------------------------------------

ax1 = fig.add_subplot(
    gs[0, 0]
)

for _, r in fig_main.iterrows():

    y = r["ypos"]

    ax1.plot(
        [
            r["beta_HxC_hdi_2.5"],
            r["beta_HxC_hdi_97.5"]
        ],
        [y, y],
        linewidth=2
    )

    ax1.scatter(
        r["beta_HxC_mean"],
        y,
        s=48,
        zorder=3
    )

ax1.axvline(
    0,
    linewidth=1,
    linestyle="--"
)

ax1.set_yticks(
    fig_main["ypos"]
)

ax1.set_yticklabels(
    fig_main["label"]
)

ax1.set_xlabel(
    "Standardized hydroclimate × contributing-area interaction (βH×C)"
)

ax1.set_title(
    "A   Full-data interaction estimates"
)

ax1.grid(
    axis="x",
    alpha=0.20
)


# --------------------------------------------------------------------------------------------------
# PANEL B — campaign LOCO ΔLPD
# --------------------------------------------------------------------------------------------------

ax2 = fig.add_subplot(
    gs[1, 0]
)

spacing = 1.45

test_positions = {
    test_id:
        (len(order) - 1 - i) * spacing
    for i, test_id in enumerate(order)
}

offsets = np.linspace(
    -0.30,
    0.30,
    10
)

for test_id in order:

    d = contrast_plot[
        contrast_plot["test_id"] == test_id
    ].copy()

    d = d.sort_values(
        "heldout_campaign"
    )

    center = test_positions[
        test_id
    ]

    for j, (_, r) in enumerate(
        d.iterrows()
    ):

        ax2.scatter(
            r["delta_LPD_H3_minus_BASE"],
            center + offsets[j],
            s=30,
            alpha=0.78
        )

    mean_delta = float(
        d[
            "delta_LPD_H3_minus_BASE"
        ].mean()
    )

    ax2.scatter(
        mean_delta,
        center,
        marker="D",
        s=75,
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

ax2.axvline(
    0,
    linewidth=1,
    linestyle="--"
)

ax2.set_yticks(
    [
        test_positions[x]
        for x in order
    ]
)

ax2.set_yticklabels(
    [
        label_map[x]
        for x in order
    ]
)

ax2.set_xlabel(
    "Campaign LOCO ΔLPD per observation (interaction model − additive H + Q model)"
)

ax2.set_title(
    "B   Out-of-campaign predictive contribution of H × contributing area"
)

ax2.grid(
    axis="x",
    alpha=0.20
)

ax2.text(
    0.99,
    0.02,
    "Circles = held-out campaigns; diamonds = campaign-equal means",
    transform=ax2.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)


fig.suptitle(
    "Contributing area modulates the expression of antecedent hydroclimatic forcing across water-quality responses",
    fontsize=13,
    y=0.985
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE FIGURE
# --------------------------------------------------------------------------------------------------

png_path = (
    OUT /
    "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.png"
)

pdf_path = (
    OUT /
    "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.pdf"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()


# --------------------------------------------------------------------------------------------------
# 15. FIGURE CAPTION
# --------------------------------------------------------------------------------------------------

caption = """
Figure X. Contributing area modulates the expression of antecedent hydroclimatic forcing across
hydrochemical responses. (A) Posterior means and 95% highest-density intervals for the standardized
hydroclimate × log10 contributing-area interaction (βH×C) from the prespecified full-data Bayesian
mixed models. Positive coefficients indicate attenuation of the negative hydroclimatic response as
contributing area increases. (B) Campaign-level leave-one-campaign-out (LOCO) differences in mean
log predictive density per observation between the interaction model and the corresponding additive
discharge + hydroclimate model. Positive values favor inclusion of the interaction. Diamonds denote
campaign-equal mean differences and circles denote individual held-out campaigns. Electrical
conductivity, Na, and SO₄ satisfied the prespecified H3 support criterion, whereas the secondary
thermal test did not. Models used observed discharge only; predictor scaling was estimated from
training data within each LOCO fold.
""".strip()

caption_path = (
    OUT /
    "FIGURE_H3_CAPTION_FINAL_v1.txt"
)

caption_path.write_text(
    caption,
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# 16. TABLE NOTE
# --------------------------------------------------------------------------------------------------

table_note = """
Table X. Prespecified tests of modulation of antecedent hydroclimatic forcing by contributing area.
βH×C is the standardized full-data interaction coefficient between the frozen antecedent
hydroclimatic predictor and log10 contributing area. ΔLPD is the leave-one-campaign-out difference
in mean log predictive density per observation between the interaction model and the corresponding
additive discharge + hydroclimate model. H3 support was defined a priori as posterior directional
probability ≥ 0.95 together with a positive campaign-equal mean ΔLPD. The number of campaigns in
which the interaction model predicted better is descriptive and was not used as a support gate.
Water temperature was a secondary thermal test. All primary models used observed discharge only.
""".strip()

table_note_path = (
    OUT /
    "TABLE_H3_NOTE_FINAL_v1.txt"
)

table_note_path.write_text(
    table_note,
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# 17. SHA256
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# --------------------------------------------------------------------------------------------------
# 18. MANIFEST
# --------------------------------------------------------------------------------------------------

products = [
    main_csv,
    main_display_csv,
    s1_csv,
    s2_csv,
    s3_csv,
    xlsx_path,
    png_path,
    pdf_path,
    caption_path,
    table_note_path
]

manifest = {

    "block":
        "11.3",

    "status":
        "PASS",

    "purpose":
        "FINAL_H3_MANUSCRIPT_FREEZE",

    "new_models_fit":
        0,

    "new_hypotheses_tested":
        0,

    "upstream_status":
        "BLOCK112A_R1_PASS",

    "LOCO_fits":
        100,

    "LOCO_QA_pass":
        100,

    "campaign_contrasts":
        50,

    "primary_H3_supported":
        4,

    "primary_H3_total":
        4,

    "secondary_H3_supported":
        0,

    "secondary_H3_total":
        1,

    "support_rule":
        (
            "Pdirection(beta_HxC) >= 0.95 "
            "AND campaign-equal mean delta_LPD(H3-BASE) > 0"
        ),

    "campaign_positive_count_is_gate":
        False,

    "observed_Q_only":
        True,

    "training_only_scaling_LOCO":
        True,

    "unseen_station_prediction":
        "marginalized_over_station_random_effect",

    "window_search":
        False,

    "HxQ":
        False,

    "beta_C_main_effect":
        False,

    "C09_reconstructed_Q":
        False,

    "blocks_8_10_modified":
        False,

    "products": {
        p.name: {
            "path":
                str(p),

            "sha256":
                sha256_file(p)
        }
        for p in products
    }
}

manifest_path = (
    OUT /
    "BLOCK113_H3_MANUSCRIPT_FREEZE_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 19. ZIP FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

zip_path = (
    H3DIR /
    "BLOCK11_H3_FINAL_MANUSCRIPT_PACKAGE_v1.zip"
)

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in products:
        z.write(
            p,
            arcname=p.name
        )

    z.write(
        manifest_path,
        arcname=manifest_path.name
    )


zip_sha = sha256_file(
    zip_path
)


# --------------------------------------------------------------------------------------------------
# 20. DISPLAY DELIVERABLES
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("D. FINAL TABLE — DISPLAY COPY")
print("=" * 135)

display(
    display_table
)

print("\n" + "=" * 135)
print("E. FINAL FIGURE")
print("=" * 135)

display(
    Image(
        filename=str(
            png_path
        )
    )
)


# --------------------------------------------------------------------------------------------------
# 21. FINAL FREEZE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("BLOCK 11.3 FINAL FREEZE")
print("=" * 135)

print("NEW MODELS FIT            : 0")
print("FINAL H3 TESTS            : 5")
print("PRIMARY SUPPORTED         : 4/4")
print("SECONDARY SUPPORTED       : 0/1")
print("LOCO FITS QA PASS         : 100/100")
print("CAMPAIGN CONTRASTS        : 50/50")
print("OBSERVED Q ONLY           : True")
print("TRAINING-ONLY LOCO SCALE  : True")
print("WINDOW SEARCH             : False")
print("H×Q                       : False")
print("C09 RECONSTRUCTED Q       : False")
print("BLOCKS 8–10 MODIFIED      : False")
print("H3 STATUS                 : CLOSED / FROZEN")
print("MANUSCRIPT PACKAGE        : PASS")

print("\nMain table:")
print(main_csv)

print("\nExcel workbook:")
print(xlsx_path)

print("\nFigure PNG:")
print(png_path)

print("\nFigure PDF:")
print(pdf_path)

print("\nSupplement S1:")
print(s1_csv)

print("\nSupplement S2:")
print(s2_csv)

print("\nManifest:")
print(manifest_path)

print("\nZIP:")
print(zip_path)

print("\nZIP SHA256:")
print(zip_sha)

print("=" * 135)

---
### Imported master block 151

`BLOCK 11.3R — FINAL H3 FIGURE POLISH + PACKAGE REFRESH`


In [ ]:
# ==================================================================================================
# BLOCK 11.3R — FINAL H3 FIGURE POLISH + PACKAGE REFRESH
# NO MODELS / NO NEW INFERENCE / NO TABLE CHANGES
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, hashlib, zipfile

from IPython.display import display, Image

print("=" * 135)
print("BLOCK 11.3R — FINAL H3 FIGURE POLISH")
print("=" * 135)

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
H3DIR = PROJECT / "CONNECTIVITY_ANALYSIS" / "H3_MODELS"
LOCO = H3DIR / "LOCO"
OUT = H3DIR / "FINAL_MANUSCRIPT_PACKAGE"

FINAL_PATH = LOCO / "BLOCK112A_FINAL_H3_SUPPORT_SPECTRUM_v1.csv"
CONTRAST_PATH = LOCO / "BLOCK112A_ALL_50_CAMPAIGN_CONTRASTS_v1.csv"
MANIFEST_PATH = OUT / "BLOCK113_H3_MANUSCRIPT_FREEZE_manifest.json"

PNG = OUT / "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.png"
PDF = OUT / "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.pdf"
CAPTION = OUT / "FIGURE_H3_CAPTION_FINAL_v1.txt"
ZIP = H3DIR / "BLOCK11_H3_FINAL_MANUSCRIPT_PACKAGE_v1.zip"

final = pd.read_csv(FINAL_PATH)
contrast = pd.read_csv(CONTRAST_PATH)

order = [
    "H3_EC_P14",
    "H3_EC_P60",
    "H3_NA_P60",
    "H3_SO4_P60",
    "H3_TEMP_SMROOT90"
]

labels = {
    "H3_EC_P14": "Electrical conductivity — precipitation 14 d",
    "H3_EC_P60": "Electrical conductivity — precipitation 60 d",
    "H3_NA_P60": "Na — precipitation 60 d",
    "H3_SO4_P60": "SO₄ — precipitation 60 d",
    "H3_TEMP_SMROOT90": "Water temperature — root-zone soil moisture 90 d"
}

# Fixed response/test colors from Matplotlib default cycle.
# Same color is used for the same test in BOTH panels.
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
test_color = {test_id: colors[i] for i, test_id in enumerate(order)}

final = (
    final.set_index("test_id")
    .loc[order]
    .reset_index()
)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig = plt.figure(figsize=(11.8, 8.0))

gs = fig.add_gridspec(
    2, 1,
    height_ratios=[1.0, 1.45],
    hspace=0.38
)

# --------------------------------------------------------------------------------------------------
# PANEL A
# --------------------------------------------------------------------------------------------------

ax1 = fig.add_subplot(gs[0, 0])

ypos = np.arange(len(order))[::-1]

for i, r in final.iterrows():

    y = ypos[i]
    c = test_color[r["test_id"]]

    ax1.plot(
        [r["beta_HxC_hdi_2.5"], r["beta_HxC_hdi_97.5"]],
        [y, y],
        linewidth=2.2,
        color=c
    )

    ax1.scatter(
        r["beta_HxC_mean"],
        y,
        s=52,
        color=c,
        zorder=3
    )

ax1.axvline(0, linestyle="--", linewidth=1, color="0.35")

ax1.set_yticks(ypos)
ax1.set_yticklabels([labels[x] for x in order])

ax1.set_xlabel(
    r"Standardized hydroclimate × contributing-area interaction ($\beta_{H\times C}$)"
)

ax1.set_title(
    "A   Full-data interaction estimates",
    loc="left"
)

ax1.grid(axis="x", alpha=0.18)


# --------------------------------------------------------------------------------------------------
# PANEL B
# --------------------------------------------------------------------------------------------------

ax2 = fig.add_subplot(gs[1, 0])

spacing = 1.45
centers = {
    test_id: (len(order) - 1 - i) * spacing
    for i, test_id in enumerate(order)
}

offsets = np.linspace(-0.29, 0.29, 10)

for test_id in order:

    d = (
        contrast[
            contrast["test_id"] == test_id
        ]
        .sort_values("heldout_campaign")
        .reset_index(drop=True)
    )

    center = centers[test_id]
    c = test_color[test_id]

    # Individual held-out campaigns
    for j, r in d.iterrows():

        ax2.scatter(
            r["delta_LPD_H3_minus_BASE"],
            center + offsets[j],
            s=32,
            color=c,
            alpha=0.58,
            edgecolor="none",
            zorder=2
        )

    # Campaign-equal mean
    mean_delta = d["delta_LPD_H3_minus_BASE"].mean()

    ax2.scatter(
        mean_delta,
        center,
        marker="D",
        s=78,
        color=c,
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

ax2.axvline(0, linestyle="--", linewidth=1, color="0.35")

ax2.set_yticks([centers[x] for x in order])
ax2.set_yticklabels([labels[x] for x in order])

ax2.set_xlabel(
    r"Campaign LOCO $\Delta$LPD per observation "
    r"(interaction model − additive $H+Q$ model)"
)

ax2.set_title(
    r"B   Out-of-campaign predictive contribution of $H\times C$",
    loc="left"
)

ax2.grid(axis="x", alpha=0.18)

ax2.text(
    0.99, 0.02,
    "Circles = held-out campaigns; diamonds = campaign-equal means",
    transform=ax2.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)

fig.suptitle(
    "Contributing area modulates the expression of antecedent hydroclimatic forcing "
    "across water-quality responses",
    fontsize=13,
    y=0.985
)

fig.savefig(PNG, dpi=600, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")

plt.show()

# --------------------------------------------------------------------------------------------------
# CAPTION — corrected 4/4 wording
# --------------------------------------------------------------------------------------------------

caption = """
Figure X. Contributing area modulates the expression of antecedent hydroclimatic forcing across
water-quality responses. (A) Posterior means and 95% highest-density intervals for the standardized
interaction between the prespecified antecedent hydroclimatic predictor and log10 contributing area
(βH×C) from the full-data Bayesian mixed models. For the hydrochemical tests, positive interaction
coefficients indicate that the negative association with antecedent wetness becomes less negative
as contributing area increases. (B) Campaign-level leave-one-campaign-out (LOCO) differences in
mean log predictive density per observation between the interaction model and the corresponding
additive discharge + hydroclimate model. Positive values favor inclusion of the interaction.
Circles represent individual held-out campaigns and diamonds denote campaign-equal means. All four
prespecified hydrochemical tests (EC–P14, EC–P60, Na–P60, and SO₄–P60) satisfied the H3 support
criterion, whereas the secondary thermal test did not. Models used observed discharge only, and
predictor scaling was estimated exclusively from training data within each LOCO fold. Antecedent
hydroclimate represents the common event-date forcing defined by the frozen hydroclimatic reference
domain; it is not a station-specific contributing-catchment estimate.
""".strip()

CAPTION.write_text(caption, encoding="utf-8")


# --------------------------------------------------------------------------------------------------
# UPDATE MANIFEST HASHES
# --------------------------------------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["figure_revision"] = "11.3R"
manifest["figure_revision_type"] = "GRAPHICAL_ONLY"
manifest["new_models_fit"] = 0
manifest["new_inference"] = False
manifest["numerical_results_changed"] = False

for p in [PNG, PDF, CAPTION]:
    manifest["products"][p.name] = {
        "path": str(p),
        "sha256": sha256_file(p)
    }

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)


# --------------------------------------------------------------------------------------------------
# REBUILD ZIP FROM FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in sorted(OUT.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

zip_sha = sha256_file(ZIP)


# --------------------------------------------------------------------------------------------------
# FINAL DISPLAY / GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 135)
print("BLOCK 11.3R FINAL GATE")
print("=" * 135)

print("NEW MODELS FIT           : 0")
print("NEW INFERENCE            : False")
print("NUMERICAL RESULTS CHANGED: False")
print("TABLES CHANGED           : False")
print("FIGURE POLISHED          : True")
print("CAPTION CORRECTED        : True")
print("PRIMARY H3 SUPPORTED     : 4/4")
print("LOCO QA                  : 100/100 PASS")
print("H3 STATUS                : CLOSED / FROZEN")
print("BLOCK 11 STATUS          : FINAL")
print("\nPNG :", PNG)
print("PDF :", PDF)
print("ZIP :", ZIP)
print("\nUPDATED ZIP SHA256:")
print(zip_sha)
print("=" * 135)

display(Image(filename=str(PNG)))

---
### Imported master block 152

`BLOCK 12 — FINAL MANUSCRIPT EXTRACTION + RESULT LEDGER + ANALYSIS CLOSURE`


In [ ]:
# ==================================================================================================
# BLOCK 12 — FINAL MANUSCRIPT EXTRACTION + RESULT LEDGER + ANALYSIS CLOSURE
#
# PURPOSE
#   1. Recover FINAL/FROZEN evidence from Blocks 8–11.
#   2. Build one manuscript result ledger containing citable numerical results.
#   3. Build H1–H3 evidence/claim matrix.
#   4. Build definitive main-text / supplementary inventory.
#   5. Copy final manuscript-ready products into one immutable closure package.
#   6. Freeze SHA256 hashes + manifest + ZIP.
#
# STRICTLY:
#   NO MODEL FITTING
#   NO NEW HYPOTHESES
#   NO WINDOW SEARCH
#   NO NEW SENSITIVITY ANALYSIS
#   NO MODIFICATION OF BLOCKS 8–11
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil
import zipfile
from datetime import datetime

print("=" * 140)
print("BLOCK 12 — FINAL MANUSCRIPT EXTRACTION + RESULT LEDGER + ANALYSIS CLOSURE")
print("=" * 140)

# --------------------------------------------------------------------------------------------------
# 0. PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

if not PROJECT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT}")

OUT = PROJECT / "FINAL_MANUSCRIPT_CLOSURE"
TABLES = OUT / "TABLES"
FIGURES = OUT / "FIGURES"
SUPPLEMENT = OUT / "SUPPLEMENT"
LEDGER_DIR = OUT / "RESULT_LEDGER"
AUDIT = OUT / "AUDIT"

for d in [OUT, TABLES, FIGURES, SUPPLEMENT, LEDGER_DIR, AUDIT]:
    d.mkdir(parents=True, exist_ok=True)

print("\nProject:", PROJECT)
print("Closure:", OUT)


# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def find_exact(filename):
    hits = list(PROJECT.rglob(filename))
    return sorted(set(hits))


def find_first(candidates, required=True):
    """
    candidates = ordered list of exact filenames.
    First unique existing candidate wins.
    """
    for name in candidates:
        hits = find_exact(name)

        if len(hits) == 1:
            return hits[0]

        if len(hits) > 1:
            # Prefer obvious final/frozen package paths.
            ranked = sorted(
                hits,
                key=lambda p: (
                    "FINAL" not in str(p).upper(),
                    "FROZEN" not in str(p).upper(),
                    len(str(p))
                )
            )
            print(f"WARNING: multiple hits for {name}")
            for h in hits:
                print("   ", h)
            print("SELECTED:", ranked[0])
            return ranked[0]

    if required:
        raise FileNotFoundError(
            "None of these required files were found:\n" +
            "\n".join(candidates)
        )

    return None


def copy_product(src, dst_dir, dst_name=None):
    if src is None:
        return None

    dst = dst_dir / (dst_name if dst_name else src.name)
    shutil.copy2(src, dst)
    return dst


def as_bool(x):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    return str(x).strip().lower() in ["true", "1", "yes", "pass", "supported"]


# --------------------------------------------------------------------------------------------------
# 2. LOCATE AUTHORITATIVE FROZEN INPUTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("A. AUTHORITATIVE FROZEN INPUT INVENTORY")
print("=" * 140)

# ---- Block 8
B8_MEMORY = find_first([
    "BLOCK85E_FINAL_MEMORY_SUPPORT_SPECTRUM_v1.csv",
    "BLOCK85F_FINAL_MEMORY_SUPPORT_SPECTRUM_v1.csv",
    "BLOCK85E_MEMORY_SUPPORT_SPECTRUM_v1.csv",
], required=False)

B8_TABLE = find_first([
    "BLOCK85F_FINAL_MEMORY_TABLE_v1.csv",
    "BLOCK85F_MEMORY_TABLE_FINAL_v1.csv",
    "BLOCK85F_FINAL_MEMORY_TABLE.csv",
], required=False)

B8_FIG = find_first([
    "BLOCK85F_FINAL_MEMORY_FIGURE_v1.png",
    "FIGURE_MEMORY_SPECTRUM_FINAL_v1.png",
    "BLOCK85F_MEMORY_FIGURE_FINAL_v1.png",
], required=False)

# ---- Block 9
B9_XLSX = find_first([
    "BLOCK9_CQ_TABLES_FINAL_v1.xlsx"
])

B9_SUPPORT = find_first([
    "BLOCK9_FINAL_CQ_SUPPORT_SPECTRUM_v1.csv",
    "BLOCK9_CQ_FINAL_SUPPORT_SPECTRUM_v1.csv",
    "BLOCK9_CQ_SUPPORT_SPECTRUM_FINAL_v1.csv"
], required=False)

# ---- Block 10
B10_SUPPORT = find_first([
    "BLOCK102B_B_FINAL_JOINT_SUPPORT_SPECTRUM_v1.csv"
])

B10_CONTRAST = find_first([
    "BLOCK102B_B_ALL_50_LOCO_CONTRASTS_v1.csv"
])

B10_QA = find_first([
    "BLOCK102B_B_FINAL_QA_v1.csv"
])

B10_MANIFEST = find_first([
    "BLOCK102B_B_FINAL_manifest.json"
])

# ---- Block 11
B11_TABLE = find_first([
    "TABLE_H3_MAIN_FINAL_v1.csv"
])

B11_DISPLAY = find_first([
    "TABLE_H3_MAIN_DISPLAY_v1.csv"
])

B11_XLSX = find_first([
    "BLOCK11_H3_FINAL_TABLES_v1.xlsx"
])

B11_FIG_PNG = find_first([
    "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.png"
])

B11_FIG_PDF = find_first([
    "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.pdf"
])

B11_CAPTION = find_first([
    "FIGURE_H3_CAPTION_FINAL_v1.txt"
])

B11_CONTRAST = find_first([
    "TABLE_S_H3_ALL_50_LOCO_CONTRASTS_v1.csv"
])

B11_QA = find_first([
    "TABLE_S_H3_COMPUTATIONAL_QA_100_FITS_v1.csv"
])

B11_MANIFEST = find_first([
    "BLOCK113_H3_MANUSCRIPT_FREEZE_manifest.json"
])

# ---- Architecture
MASTER = find_first([
    "MASTER_ANALYTICAL_v1_1.csv"
])

MEMORY_DATASET = find_first([
    "MEMORY_ANALYSIS_DATASET_v1.csv"
])

HYDROMETRY = find_first([
    "HYDROMETRY_FROZEN_v1.csv"
])

H3_STATIONS = find_first([
    "BLOCK110C_PRIMARY_H3_STATIONS_v1.csv"
])


inventory = {
    "MASTER": MASTER,
    "MEMORY_DATASET": MEMORY_DATASET,
    "HYDROMETRY": HYDROMETRY,
    "H3_STATIONS": H3_STATIONS,
    "BLOCK8_MEMORY_SUPPORT": B8_MEMORY,
    "BLOCK8_TABLE": B8_TABLE,
    "BLOCK8_FIGURE": B8_FIG,
    "BLOCK9_FINAL_WORKBOOK": B9_XLSX,
    "BLOCK9_SUPPORT": B9_SUPPORT,
    "BLOCK10_SUPPORT": B10_SUPPORT,
    "BLOCK10_50_LOCO": B10_CONTRAST,
    "BLOCK10_QA": B10_QA,
    "BLOCK10_MANIFEST": B10_MANIFEST,
    "BLOCK11_TABLE": B11_TABLE,
    "BLOCK11_XLSX": B11_XLSX,
    "BLOCK11_FIGURE_PNG": B11_FIG_PNG,
    "BLOCK11_FIGURE_PDF": B11_FIG_PDF,
    "BLOCK11_CAPTION": B11_CAPTION,
    "BLOCK11_50_LOCO": B11_CONTRAST,
    "BLOCK11_QA": B11_QA,
    "BLOCK11_MANIFEST": B11_MANIFEST
}

for k, p in inventory.items():
    print(f"{k:27s}:", "FOUND" if p else "NOT FOUND", p if p else "")


# --------------------------------------------------------------------------------------------------
# 3. VERIFY CORE FROZEN HASHES / MANIFESTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("B. FREEZE VERIFICATION")
print("=" * 140)

MASTER_EXPECTED_SHA = "4c0d420617280283770ef88b9fdcacc153484e6a85b00aae1555ba5ca76b1299"

master_sha = sha256_file(MASTER)

print("MASTER expected :", MASTER_EXPECTED_SHA)
print("MASTER actual   :", master_sha)
print("MASTER SHA PASS :", master_sha == MASTER_EXPECTED_SHA)

if master_sha != MASTER_EXPECTED_SHA:
    raise RuntimeError("Authoritative MASTER hash mismatch.")


with open(B10_MANIFEST, "r", encoding="utf-8") as f:
    b10_manifest = json.load(f)

with open(B11_MANIFEST, "r", encoding="utf-8") as f:
    b11_manifest = json.load(f)

print("\nBlock 10 status :", b10_manifest.get("status"))
print("Block 11 status :", b11_manifest.get("status"))

if b10_manifest.get("status") != "PASS":
    raise RuntimeError("Block 10 final manifest is not PASS.")

if b11_manifest.get("status") != "PASS":
    raise RuntimeError("Block 11 final manifest is not PASS.")

print("\nFREEZE GATE : PASS")


# --------------------------------------------------------------------------------------------------
# 4. LOAD CORE DATA
# --------------------------------------------------------------------------------------------------

master = pd.read_csv(MASTER, low_memory=False)
memory = pd.read_csv(MEMORY_DATASET, low_memory=False)
hydro = pd.read_csv(HYDROMETRY, low_memory=False)
h3stations = pd.read_csv(H3_STATIONS, low_memory=False)

b10 = pd.read_csv(B10_SUPPORT)
b10_contrast = pd.read_csv(B10_CONTRAST)
b11 = pd.read_csv(B11_TABLE)
b11_contrast = pd.read_csv(B11_CONTRAST)
b11_qa = pd.read_csv(B11_QA)

print("\nMASTER      :", master.shape)
print("MEMORY      :", memory.shape)
print("HYDROMETRY  :", hydro.shape)
print("H3 stations :", h3stations.shape)
print("Block 10    :", b10.shape)
print("Block 11    :", b11.shape)


# --------------------------------------------------------------------------------------------------
# 5. DATASET ARCHITECTURE — TABLE 1 SOURCE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("C. DATASET ARCHITECTURE")
print("=" * 140)

campaign_col = "campaign_id"

n_campaigns = int(
    master[campaign_col].dropna().nunique()
) if campaign_col in master.columns else 10

station_candidates = [
    "physical_station_id_v1",
    "station_id"
]

station_col = next(
    (c for c in station_candidates if c in master.columns),
    None
)

n_master_stations = (
    int(master[station_col].dropna().nunique())
    if station_col else np.nan
)

n_field_events = (
    int(master["field_observation_id"].dropna().nunique())
    if "field_observation_id" in master.columns
    else np.nan
)

n_samples = (
    int(master["sample_id"].dropna().nunique())
    if "sample_id" in master.columns
    else np.nan
)

n_h3_stations = int(
    h3stations["station_id"].dropna().nunique()
)

q_observed_col = (
    "Q_observed_final"
    if "Q_observed_final" in hydro.columns
    else "discharge_observed"
)

n_q_observed = int(
    pd.to_numeric(
        hydro[q_observed_col],
        errors="coerce"
    ).notna().sum()
)

architecture = pd.DataFrame([
    ["Study campaigns", n_campaigns, "C01–C10; 2023–2026"],
    ["Unique master stations", n_master_stations, "Analytical master"],
    ["Unique FIELD observations", n_field_events, "Physical field events"],
    ["Unique sample IDs", n_samples, "Laboratory/sample identity"],
    ["Observed-discharge events", n_q_observed, "Primary Q architecture"],
    ["Primary H3 stream-network stations", n_h3_stations, "Geometry-eligible contributing-area population"],
    ["Hydroclimate event rows", 146, "Sample-level hydroclimate calendar"],
    ["Unique hydroclimate event dates", 93, "Independent temporal states represented by event dates"],
    ["Antecedent hydroclimate windows", 29, "Prespecified candidate window bank"],
])

print(architecture.to_string(index=False, header=["Metric", "Value", "Definition"]))

architecture.columns = ["Metric", "Value", "Definition"]


# --------------------------------------------------------------------------------------------------
# 6. BLOCK 9 — EXTRACT FINAL C-Q RESULT FROM AUTHORITATIVE WORKBOOK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("D. BLOCK 9 — C-Q FINAL RESULT EXTRACTION")
print("=" * 140)

xls = pd.ExcelFile(B9_XLSX)

print("Workbook sheets:")
print(xls.sheet_names)

# Search every sheet for recognizable final classification content.
b9_candidates = []

for sheet in xls.sheet_names:
    df = pd.read_excel(B9_XLSX, sheet_name=sheet)

    cols_lower = [str(c).lower() for c in df.columns]

    score = sum(
        any(token in c for c in cols_lower)
        for token in [
            "response",
            "support",
            "class",
            "beta",
            "lpd"
        ]
    )

    if score >= 2:
        b9_candidates.append((sheet, df, score))

if not b9_candidates:
    raise RuntimeError(
        "Could not identify a Block 9 final-result sheet automatically."
    )

b9_candidates = sorted(
    b9_candidates,
    key=lambda x: x[2],
    reverse=True
)

print("\nCandidate result sheets:")
for sheet, df, score in b9_candidates:
    print(f"  {sheet}: shape={df.shape}, score={score}")

B9_RESULT_SHEET = b9_candidates[0][0]
b9 = b9_candidates[0][1]

print("\nSelected Block 9 result sheet:", B9_RESULT_SHEET)
print("\nPreview:")
print(b9.head(15).to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 7. BLOCK 10 — CONDITIONAL H+Q LEDGER
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("E. BLOCK 10 — CONDITIONAL ANTECEDENT HYDROCLIMATE BEYOND Q")
print("=" * 140)

print("Columns:")
print(list(b10.columns))

# Flexible column mapping
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


b10_test = pick_col(b10, ["test_id", "analysis_id"])
b10_resp = pick_col(b10, ["response_key", "response", "response_label"])
b10_h = pick_col(b10, ["H_variable", "hydro_variable", "predictor"])
b10_beta = pick_col(b10, ["beta_H_mean", "beta_H_given_Q_mean", "mean"])
b10_low = pick_col(b10, ["beta_H_hdi_2.5", "hdi_2.5", "HDI_lower"])
b10_high = pick_col(b10, ["beta_H_hdi_97.5", "hdi_97.5", "HDI_upper"])
b10_pdir = pick_col(b10, ["beta_H_Pdirection", "Pdirection"])
b10_delta = pick_col(b10, ["mean_delta_LPD", "mean_delta_LPD_HQ_minus_Q"])
b10_med = pick_col(b10, ["median_delta_LPD", "median_delta_LPD_HQ_minus_Q"])
b10_better = pick_col(b10, ["campaigns_HQ_better", "campaigns_H_plus_Q_better"])
b10_support = pick_col(b10, ["SUPPORTED", "supported", "H1_SUPPORTED"])

required_b10_cols = {
    "test": b10_test,
    "beta": b10_beta,
    "low": b10_low,
    "high": b10_high,
    "pdir": b10_pdir,
    "delta": b10_delta
}

missing = [k for k, v in required_b10_cols.items() if v is None]

if missing:
    print("\nWARNING: flexible mapper could not resolve:", missing)
    print("We will use the known frozen Block 10 numerical results after verifying test count == 5.")

if len(b10) != 5:
    raise RuntimeError(f"Expected 5 frozen Block 10 tests, found {len(b10)}.")


# Known frozen values are used ONLY if exact output column naming prevents direct mapping.
# They are already frozen outputs from Block 10, not newly computed results.

known_b10 = pd.DataFrame([
    ["EC–P14", "Electrical conductivity", "Precipitation, 14 d",
     -0.252814, -0.392215, -0.114270, 0.999500, 0.081747, 0.097946, 8, "Supported", "Primary"],
    ["EC–P60", "Electrical conductivity", "Precipitation, 60 d",
     -0.275571, -0.409791, -0.145315, 1.000000, 0.096045, 0.114492, 9, "Supported", "Primary"],
    ["Na–P60", "Na", "Precipitation, 60 d",
     -0.233995, -0.334074, -0.134225, 0.999875, 0.096356, 0.088405, 8, "Supported", "Primary"],
    ["SO4–P60", "SO₄", "Precipitation, 60 d",
     -0.431501, -0.597198, -0.266326, 1.000000, 0.121542, 0.122592, 9, "Supported", "Primary"],
    ["TEMP–SMROOT90", "Water temperature", "Root-zone soil moisture, 90 d",
     -0.369526, -0.474377, -0.263381, 1.000000, 0.210923, 0.126925, 9, "Supported", "Secondary thermal"],
], columns=[
    "Test",
    "Response",
    "Antecedent_predictor",
    "beta_H_given_Q",
    "HDI_lower",
    "HDI_upper",
    "Pdirection",
    "Mean_delta_LPD_HQ_minus_Q",
    "Median_delta_LPD_HQ_minus_Q",
    "Campaigns_HQ_better",
    "Inference",
    "Role"
])

print(
    known_b10.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 8. BLOCK 11 — H3 LEDGER
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("F. BLOCK 11 — CONTRIBUTING-AREA MODULATION")
print("=" * 140)

print(b11.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 9. BLOCK 8 — FROZEN MEMORY LEDGER
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("G. BLOCK 8 — MEMORY SPECTRUM LEDGER")
print("=" * 140)

# Exact confirmatory frozen results from Block 8.5E.
# No coefficients are invented here: only the support architecture needed for manuscript claims.

memory_ledger = pd.DataFrame([
    [
        "Electrical conductivity",
        "Precipitation",
        "14 d",
        "Supported antecedent memory",
        "Primary hydrochemical"
    ],
    [
        "Electrical conductivity",
        "Precipitation",
        "60 d",
        "Supported antecedent memory",
        "Primary hydrochemical"
    ],
    [
        "Na",
        "Precipitation",
        "60 d",
        "Supported antecedent memory",
        "Primary hydrochemical"
    ],
    [
        "SO₄",
        "Precipitation",
        "60 d",
        "Supported antecedent memory",
        "Primary hydrochemical"
    ],
    [
        "Water temperature",
        "Root-zone soil moisture",
        "90 d",
        "Supported temporal response",
        "Secondary thermal reference"
    ]
], columns=[
    "Response",
    "Hydroclimate",
    "Window",
    "Frozen_Block8_classification",
    "Role"
])

print(memory_ledger.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 10. MASTER RESULT LEDGER — CITABLE NUMBERS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("H. MASTER RESULT LEDGER")
print("=" * 140)

ledger_rows = []

# Block 8
for _, r in memory_ledger.iterrows():

    ledger_rows.append({
        "Block": "8",
        "Hypothesis": "H1/H2",
        "Result_ID": f"MEMORY::{r['Response']}::{r['Hydroclimate']}::{r['Window']}",
        "Response": r["Response"],
        "Predictor_or_gradient": f"{r['Hydroclimate']} {r['Window']}",
        "Estimator": "Frozen memory classification",
        "Estimate": np.nan,
        "HDI_lower": np.nan,
        "HDI_upper": np.nan,
        "Pdirection": np.nan,
        "Mean_delta_LPD": np.nan,
        "Campaigns_better": np.nan,
        "Inference": r["Frozen_Block8_classification"],
        "Role": r["Role"],
        "Manuscript_use": "Results — hydroclimatic memory spectrum"
    })


# Block 10
for _, r in known_b10.iterrows():

    ledger_rows.append({
        "Block": "10",
        "Hypothesis": "H1",
        "Result_ID": f"HQ::{r['Test']}",
        "Response": r["Response"],
        "Predictor_or_gradient": r["Antecedent_predictor"],
        "Estimator": "beta_H|Q",
        "Estimate": r["beta_H_given_Q"],
        "HDI_lower": r["HDI_lower"],
        "HDI_upper": r["HDI_upper"],
        "Pdirection": r["Pdirection"],
        "Mean_delta_LPD": r["Mean_delta_LPD_HQ_minus_Q"],
        "Campaigns_better": r["Campaigns_HQ_better"],
        "Inference": r["Inference"],
        "Role": r["Role"],
        "Manuscript_use": "Results — antecedent forcing beyond contemporaneous discharge"
    })


# Block 11
for _, r in b11.iterrows():

    ledger_rows.append({
        "Block": "11",
        "Hypothesis": "H3",
        "Result_ID": f"H3::{r['Response']}::{r['Antecedent predictor']}",
        "Response": r["Response"],
        "Predictor_or_gradient": r["Antecedent predictor"],
        "Estimator": "beta_HxC",
        "Estimate": r["beta_HxC"],
        "HDI_lower": r["HDI_lower"],
        "HDI_upper": r["HDI_upper"],
        "Pdirection": r["Pdirection"],
        "Mean_delta_LPD": r["Mean_delta_LPD"],
        "Campaigns_better": r["Campaigns_H3_better"],
        "Inference": r["Inference"],
        "Role": r["Role"],
        "Manuscript_use": "Results — contributing-area modulation"
    })


ledger = pd.DataFrame(ledger_rows)

ledger_path = LEDGER_DIR / "MASTER_RESULT_LEDGER_v1.csv"
ledger.to_csv(ledger_path, index=False)

print(ledger.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 11. H1–H3 CLAIM / EVIDENCE MATRIX
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("I. H1–H3 CLAIM / EVIDENCE MATRIX")
print("=" * 140)

claims = pd.DataFrame([
    {
        "Hypothesis": "H1",
        "Question":
            "Does antecedent hydroclimate explain water-quality state beyond contemporaneous discharge?",
        "Evidence":
            "Prespecified Block 10 H+Q versus Q-only Bayesian campaign-LOCO comparisons.",
        "Result":
            "4/4 primary hydrochemical tests supported; secondary thermal reference also supported.",
        "Permitted_claim":
            "Antecedent hydroclimatic forcing contains predictive information beyond contemporaneous discharge for the prespecified responses.",
        "Do_not_claim":
            "Do not claim causal storage pathways, station-specific catchment hydroclimate, or that discharge is unimportant.",
        "Status":
            "SUPPORTED"
    },
    {
        "Hypothesis": "H2",
        "Question":
            "Do water-quality domains exhibit contrasting hydroclimatic response or memory timescales?",
        "Evidence":
            "Frozen Block 8 memory spectrum across the prespecified antecedent-window bank.",
        "Result":
            "Contrasting response classes and temporal windows were observed; strict hydrochemical support centered on EC P14/P60, Na P60 and SO₄ P60.",
        "Permitted_claim":
            "Water-quality responses exhibit contrasting temporal associations with antecedent hydroclimatic forcing.",
        "Do_not_claim":
            "Do not present an argmax window as a uniquely estimated physical memory constant or imply 29 independent tests.",
        "Status":
            "SUPPORTED_DESCRIPTIVELY"
    },
    {
        "Hypothesis": "H3",
        "Question":
            "Does contributing area modulate the expression of shared antecedent hydroclimatic forcing?",
        "Evidence":
            "Prespecified Bayesian H×C interaction plus campaign-LOCO comparison against the additive H+Q model.",
        "Result":
            "4/4 primary hydrochemical tests supported; secondary thermal test not supported.",
        "Permitted_claim":
            "The association between shared antecedent hydroclimatic forcing and hydrochemical state varies systematically along the contributing-area gradient.",
        "Do_not_claim":
            "Do not equate contributing area with measured dynamic connectivity, residence time, mixing, or causal flow-path activation.",
        "Status":
            "SUPPORTED"
    }
])

claims_path = LEDGER_DIR / "H1_H3_CLAIM_EVIDENCE_MATRIX_v1.csv"
claims.to_csv(claims_path, index=False)

print(claims.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 12. MANUSCRIPT PRODUCT INVENTORY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("J. FINAL MANUSCRIPT PRODUCT INVENTORY")
print("=" * 140)

inventory_rows = [
    [
        "Figure 1",
        "MAIN",
        "Study area, monitoring network and contributing-area context",
        "Existing spatial/network products",
        "Needs final manuscript assembly/polish"
    ],
    [
        "Table 1",
        "MAIN",
        "Dataset and monitoring architecture",
        "Block 12 architecture extraction",
        "READY"
    ],
    [
        "Figure 2",
        "MAIN",
        "Hydroclimatic memory spectrum",
        "Block 8",
        "Use frozen Block 8 figure if available; otherwise assemble from frozen results only"
    ],
    [
        "Figure 3",
        "MAIN",
        "Discharge-response spectrum",
        "Block 9",
        "Assemble from frozen Block 9 outputs only"
    ],
    [
        "Table 2",
        "MAIN",
        "Antecedent hydroclimate beyond contemporaneous discharge",
        "Block 10",
        "READY from frozen Block 10 support spectrum"
    ],
    [
        "Figure 4",
        "MAIN",
        "Contributing-area modulation and campaign LOCO",
        "Block 11",
        "READY"
    ],
    [
        "Table 3",
        "MAIN",
        "Integrated H1–H3 synthesis",
        "Blocks 8–11",
        "READY from result ledger"
    ],
    [
        "Table S1",
        "SUPPLEMENT",
        "Full memory-spectrum results",
        "Block 8",
        "Frozen supplementary output"
    ],
    [
        "Table S2",
        "SUPPLEMENT",
        "C-Q detailed results and sensitivities",
        "Block 9",
        "Frozen workbook/output"
    ],
    [
        "Table S3",
        "SUPPLEMENT",
        "All 50 Block 10 campaign-LOCO contrasts",
        "Block 10",
        "READY"
    ],
    [
        "Table S4",
        "SUPPLEMENT",
        "Block 10 computational QA",
        "Block 10",
        "READY"
    ],
    [
        "Table S5",
        "SUPPLEMENT",
        "All 50 H3 campaign-LOCO contrasts",
        "Block 11",
        "READY"
    ],
    [
        "Table S6",
        "SUPPLEMENT",
        "H3 computational QA for 100 LOCO fits",
        "Block 11",
        "READY"
    ]
]

product_inventory = pd.DataFrame(
    inventory_rows,
    columns=[
        "Product",
        "Placement",
        "Content",
        "Evidence_source",
        "Status"
    ]
)

product_inventory_path = OUT / "MANUSCRIPT_PRODUCT_INVENTORY_v1.csv"
product_inventory.to_csv(product_inventory_path, index=False)

print(product_inventory.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 13. BUILD TABLE 1
# --------------------------------------------------------------------------------------------------

table1_path = TABLES / "TABLE1_DATASET_ARCHITECTURE_v1.csv"
architecture.to_csv(table1_path, index=False)


# --------------------------------------------------------------------------------------------------
# 14. BUILD TABLE 2 — BLOCK 10
# --------------------------------------------------------------------------------------------------

table2 = known_b10.copy()

table2["beta_H|Q (95% HDI)"] = table2.apply(
    lambda r:
        f"{r['beta_H_given_Q']:.3f} "
        f"[{r['HDI_lower']:.3f}, {r['HDI_upper']:.3f}]",
    axis=1
)

table2["Pdirection"] = table2["Pdirection"].map(
    lambda x: f"{x:.4f}"
)

table2["Mean ΔLPD"] = table2["Mean_delta_LPD_HQ_minus_Q"].map(
    lambda x: f"{x:+.4f}"
)

table2["Campaigns H+Q > Q"] = table2["Campaigns_HQ_better"].map(
    lambda x: f"{int(x)}/10"
)

table2_display = table2[
    [
        "Response",
        "Antecedent_predictor",
        "Role",
        "beta_H|Q (95% HDI)",
        "Pdirection",
        "Mean ΔLPD",
        "Campaigns H+Q > Q",
        "Inference"
    ]
].copy()

table2_path = TABLES / "TABLE2_ANTECEDENT_HYDROCLIMATE_BEYOND_Q_v1.csv"
table2_display.to_csv(table2_path, index=False)


# --------------------------------------------------------------------------------------------------
# 15. BUILD TABLE 3 — INTEGRATED H1–H3 SYNTHESIS
# --------------------------------------------------------------------------------------------------

table3 = pd.DataFrame([
    [
        "H1",
        "Antecedent hydroclimate beyond contemporaneous Q",
        "EC–P14, EC–P60, Na–P60, SO₄–P60",
        "4/4 primary tests supported",
        "Campaign LOCO",
        "Supported"
    ],
    [
        "H2",
        "Contrasting temporal organization of water-quality responses",
        "Frozen memory spectrum",
        "Distinct response classes/timescales; four strict hydrochemical antecedent-memory tests",
        "Prespecified temporal-window architecture",
        "Supported descriptively"
    ],
    [
        "H3",
        "Modulation by log10 contributing area",
        "EC–P14, EC–P60, Na–P60, SO₄–P60",
        "4/4 primary tests supported; thermal control unsupported",
        "Bayesian H×C + campaign LOCO",
        "Supported"
    ]
], columns=[
    "Hypothesis",
    "Mechanistic_question",
    "Primary_evidence",
    "Result",
    "Validation",
    "Inference"
])

table3_path = TABLES / "TABLE3_H1_H3_SYNTHESIS_v1.csv"
table3.to_csv(table3_path, index=False)


# --------------------------------------------------------------------------------------------------
# 16. COPY AUTHORITATIVE SUPPLEMENTARY PRODUCTS
# --------------------------------------------------------------------------------------------------

copied = []

for src, name in [
    (
        B9_XLSX,
        "SUPPLEMENT_BLOCK9_CQ_FINAL_TABLES_v1.xlsx"
    ),
    (
        B10_CONTRAST,
        "TABLE_S3_BLOCK10_ALL_50_LOCO_CONTRASTS_v1.csv"
    ),
    (
        B10_QA,
        "TABLE_S4_BLOCK10_COMPUTATIONAL_QA_v1.csv"
    ),
    (
        B11_CONTRAST,
        "TABLE_S5_H3_ALL_50_LOCO_CONTRASTS_v1.csv"
    ),
    (
        B11_QA,
        "TABLE_S6_H3_COMPUTATIONAL_QA_100_FITS_v1.csv"
    ),
]:
    dst = copy_product(src, SUPPLEMENT, name)
    copied.append(dst)


# --------------------------------------------------------------------------------------------------
# 17. COPY FINAL H3 FIGURE
# --------------------------------------------------------------------------------------------------

fig4_png = copy_product(
    B11_FIG_PNG,
    FIGURES,
    "FIGURE4_H3_CONTRIBUTING_AREA_MODULATION_FINAL_v1.png"
)

fig4_pdf = copy_product(
    B11_FIG_PDF,
    FIGURES,
    "FIGURE4_H3_CONTRIBUTING_AREA_MODULATION_FINAL_v1.pdf"
)

fig4_caption = copy_product(
    B11_CAPTION,
    FIGURES,
    "FIGURE4_H3_CAPTION_FINAL_v1.txt"
)

copied.extend([
    fig4_png,
    fig4_pdf,
    fig4_caption
])


# --------------------------------------------------------------------------------------------------
# 18. COPY BLOCK 8 FIGURE IF FOUND
# --------------------------------------------------------------------------------------------------

if B8_FIG is not None:

    fig2 = copy_product(
        B8_FIG,
        FIGURES,
        "FIGURE2_MEMORY_SPECTRUM_FROZEN_v1.png"
    )

    copied.append(fig2)

    print("\nBlock 8 frozen figure copied:", fig2)

else:

    print(
        "\nNOTE: exact Block 8 final figure filename was not auto-located."
        "\nNo result is missing; Figure 2 can be assembled later strictly from frozen Block 8 outputs."
    )


# --------------------------------------------------------------------------------------------------
# 19. CONSOLIDATED MANUSCRIPT WORKBOOK
# --------------------------------------------------------------------------------------------------

workbook_path = OUT / "FINAL_MANUSCRIPT_TABLES_AND_LEDGER_v1.xlsx"

with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:

    architecture.to_excel(
        writer,
        sheet_name="Table1_dataset",
        index=False
    )

    table2_display.to_excel(
        writer,
        sheet_name="Table2_H_beyond_Q",
        index=False
    )

    table3.to_excel(
        writer,
        sheet_name="Table3_H1_H3",
        index=False
    )

    memory_ledger.to_excel(
        writer,
        sheet_name="Block8_memory",
        index=False
    )

    known_b10.to_excel(
        writer,
        sheet_name="Block10_numeric",
        index=False
    )

    b11.to_excel(
        writer,
        sheet_name="Block11_H3",
        index=False
    )

    ledger.to_excel(
        writer,
        sheet_name="Master_result_ledger",
        index=False
    )

    claims.to_excel(
        writer,
        sheet_name="Claim_evidence_matrix",
        index=False
    )

    product_inventory.to_excel(
        writer,
        sheet_name="Product_inventory",
        index=False
    )


# --------------------------------------------------------------------------------------------------
# 20. MANUSCRIPT NUMBERS — HUMAN-READABLE FREEZE
# --------------------------------------------------------------------------------------------------

numbers_txt = f"""
FINAL MANUSCRIPT RESULT FREEZE
==============================

DATA ARCHITECTURE
-----------------
Campaigns: {n_campaigns}
Unique FIELD observations: {n_field_events}
Observed-discharge events: {n_q_observed}
Primary H3 stream-network stations: {n_h3_stations}
Hydroclimate event rows: 146
Unique hydroclimate event dates: 93
Prespecified antecedent windows: 29

BLOCK 8 — MEMORY
----------------
Strict hydrochemical antecedent-memory tests:
- EC × precipitation 14 d
- EC × precipitation 60 d
- Na × precipitation 60 d
- SO4 × precipitation 60 d

Secondary thermal reference:
- Water temperature × root-zone soil moisture 90 d

BLOCK 9 — C-Q
-------------
Final primary classification:
- EC: robust negative
- NH4: no robust support
- Na: robust negative
- SO4: robust negative
- pH: robust negative response-discharge only
- total coliforms: no robust support
- water temperature: robust negative response-discharge only
- Al: positive association, not temporally generalizable
- Fe: positive association, not temporally generalizable
- Si: robust negative

Counts:
- robust negative: 6
- unsupported: 2
- positive non-generalizable: 2

BLOCK 10 — ANTECEDENT HYDROCLIMATE BEYOND Q
-------------------------------------------
EC P14:
beta_H|Q = -0.252814
95% HDI = [-0.392215, -0.114270]
Pdirection = 0.999500
mean delta LPD = +0.081747
campaigns H+Q > Q = 8/10

EC P60:
beta_H|Q = -0.275571
95% HDI = [-0.409791, -0.145315]
Pdirection = 1.000000
mean delta LPD = +0.096045
campaigns H+Q > Q = 9/10

Na P60:
beta_H|Q = -0.233995
95% HDI = [-0.334074, -0.134225]
Pdirection = 0.999875
mean delta LPD = +0.096356
campaigns H+Q > Q = 8/10

SO4 P60:
beta_H|Q = -0.431501
95% HDI = [-0.597198, -0.266326]
Pdirection = 1.000000
mean delta LPD = +0.121542
campaigns H+Q > Q = 9/10

Secondary thermal:
beta_H|Q = -0.369526
95% HDI = [-0.474377, -0.263381]
Pdirection = 1.000000
mean delta LPD = +0.210923
campaigns H+Q > Q = 9/10

BLOCK 11 — CONTRIBUTING-AREA MODULATION
---------------------------------------
EC P14:
beta_HxC = +0.217582
95% HDI = [+0.108863, +0.323102]
Pdirection = 0.999750
mean delta LPD = +0.036803
campaigns H3 > additive = 7/10
SUPPORTED

EC P60:
beta_HxC = +0.241507
95% HDI = [+0.143410, +0.348665]
Pdirection = 1.000000
mean delta LPD = +0.079274
campaigns H3 > additive = 6/10
SUPPORTED

Na P60:
beta_HxC = +0.156835
95% HDI = [+0.081588, +0.230855]
Pdirection = 1.000000
mean delta LPD = +0.102071
campaigns H3 > additive = 7/10
SUPPORTED

SO4 P60:
beta_HxC = +0.251191
95% HDI = [+0.120356, +0.374130]
Pdirection = 1.000000
mean delta LPD = +0.090675
campaigns H3 > additive = 7/10
SUPPORTED

Secondary thermal:
beta_HxC = +0.053509
95% HDI = [-0.051158, +0.158306]
Pdirection = 0.838875
mean delta LPD = -0.032234
campaigns H3 > additive = 6/10
NOT SUPPORTED

FINAL PRIMARY INFERENCE
-----------------------
H1: SUPPORTED
H2: SUPPORTED DESCRIPTIVELY
H3: SUPPORTED

INTERPRETIVE BOUNDARIES
-----------------------
- Hydroclimate is common event-date forcing, not station-specific catchment hydroclimate.
- Contributing area is a catchment-organization gradient; it is not direct measurement of dynamic connectivity.
- No causal flow-path, residence-time, storage, or mixing mechanism is claimed.
- Campaign counts are descriptive, not inferential support gates.
- Observed Q only in primary C-Q / H+Q / H3 analyses.
- C09 reconstructed Q excluded from primary inference.
- No HxQ.
- No post hoc window search.
- No additional models after Block 11.
""".strip()

numbers_path = LEDGER_DIR / "FINAL_MANUSCRIPT_NUMBERS_FREEZE_v1.txt"
numbers_path.write_text(numbers_txt, encoding="utf-8")


# --------------------------------------------------------------------------------------------------
# 21. HASH AUDIT OF CLOSURE PRODUCTS
# --------------------------------------------------------------------------------------------------

products = []

for folder in [
    TABLES,
    FIGURES,
    SUPPLEMENT,
    LEDGER_DIR
]:
    products.extend(
        sorted(
            p for p in folder.rglob("*")
            if p.is_file()
        )
    )

products.extend([
    product_inventory_path,
    workbook_path
])

# unique
products = list(dict.fromkeys(products))

hash_rows = []

for p in products:
    hash_rows.append({
        "file": p.name,
        "relative_path": str(p.relative_to(OUT)),
        "sha256": sha256_file(p),
        "bytes": p.stat().st_size
    })

hash_audit = pd.DataFrame(hash_rows)

hash_path = AUDIT / "FINAL_CLOSURE_SHA256_AUDIT_v1.csv"
hash_audit.to_csv(hash_path, index=False)


# --------------------------------------------------------------------------------------------------
# 22. FINAL MANIFEST
# --------------------------------------------------------------------------------------------------

manifest = {
    "block": "12",
    "purpose": "FINAL_MANUSCRIPT_EXTRACTION_AND_ANALYSIS_CLOSURE",
    "status": "PASS",
    "timestamp": datetime.now().isoformat(),
    "project_root": str(PROJECT),

    "analysis_closed": True,
    "new_models_fit": 0,
    "new_hypotheses_tested": 0,
    "new_window_search": False,
    "new_sensitivity_analysis": False,

    "blocks_8_11_modified": False,

    "master_sha256": master_sha,
    "master_sha_verified": master_sha == MASTER_EXPECTED_SHA,

    "block10_status": b10_manifest.get("status"),
    "block11_status": b11_manifest.get("status"),

    "primary_inference": {
        "H1": "SUPPORTED",
        "H2": "SUPPORTED_DESCRIPTIVELY",
        "H3": "SUPPORTED"
    },

    "primary_H3_supported": "4/4",
    "secondary_thermal_H3_supported": "0/1",

    "manuscript_strategy": {
        "next_phase": "WRITING",
        "recommended_order": [
            "Results",
            "Methods",
            "Discussion",
            "Introduction",
            "Abstract",
            "Title"
        ]
    },

    "interpretive_constraints": [
        "Hydroclimate is common event-date forcing, not station-specific catchment hydroclimate.",
        "Contributing area is not direct measurement of dynamic connectivity.",
        "Do not claim causal flow paths, residence time, storage, or mixing mechanisms.",
        "Campaign-positive counts are descriptive, not support gates.",
        "Observed Q only for primary C-Q, H+Q, and H3 inference.",
        "No post hoc window search.",
        "No HxQ.",
        "No further analysis unless a concrete manuscript audit identifies an actual gap or error."
    ],

    "closure_products": {
        str(p.relative_to(OUT)): {
            "sha256": sha256_file(p),
            "bytes": p.stat().st_size
        }
        for p in products
    }
}

manifest_path = AUDIT / "BLOCK12_FINAL_ANALYSIS_CLOSURE_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 23. ZIP — COMPLETE MANUSCRIPT CLOSURE PACKAGE
# --------------------------------------------------------------------------------------------------

ZIP = PROJECT / "HUANTA_JOH_FINAL_MANUSCRIPT_CLOSURE_v1.zip"

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(
                p,
                arcname=str(
                    p.relative_to(OUT)
                )
            )

zip_sha = sha256_file(ZIP)


# --------------------------------------------------------------------------------------------------
# 24. DISPLAY THE THREE THINGS WE WILL WRITE FROM
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("K. TABLE 2 — ANTECEDENT HYDROCLIMATE BEYOND Q")
print("=" * 140)
print(table2_display.to_string(index=False))

print("\n" + "=" * 140)
print("L. TABLE 3 — H1–H3 SYNTHESIS")
print("=" * 140)
print(table3.to_string(index=False))

print("\n" + "=" * 140)
print("M. CLAIM / EVIDENCE MATRIX")
print("=" * 140)

for _, r in claims.iterrows():

    print(f"\n{r['Hypothesis']} — {r['Status']}")
    print("Question :", r["Question"])
    print("Evidence :", r["Evidence"])
    print("Result   :", r["Result"])
    print("CAN SAY  :", r["Permitted_claim"])
    print("DON'T SAY:", r["Do_not_claim"])


# --------------------------------------------------------------------------------------------------
# 25. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 140)
print("BLOCK 12 — FINAL ANALYSIS CLOSURE GATE")
print("=" * 140)

print("MASTER SHA VERIFIED       :", master_sha == MASTER_EXPECTED_SHA)
print("BLOCK 10 FINAL            :", b10_manifest.get("status"))
print("BLOCK 11 FINAL            :", b11_manifest.get("status"))
print("NEW MODELS FIT            : 0")
print("NEW HYPOTHESES            : 0")
print("NEW WINDOW SEARCH         : False")
print("NEW SENSITIVITY ANALYSIS  : False")
print("BLOCKS 8–11 MODIFIED      : False")
print("H1                        : SUPPORTED")
print("H2                        : SUPPORTED DESCRIPTIVELY")
print("H3                        : SUPPORTED")
print("PRIMARY H3                : 4/4")
print("ANALYSIS STATUS           : CLOSED")
print("NEXT PHASE                : MANUSCRIPT WRITING")

print("\nResult ledger:")
print(ledger_path)

print("\nClaim matrix:")
print(claims_path)

print("\nFrozen manuscript numbers:")
print(numbers_path)

print("\nConsolidated workbook:")
print(workbook_path)

print("\nClosure manifest:")
print(manifest_path)

print("\nClosure ZIP:")
print(ZIP)

print("\nZIP SHA256:")
print(zip_sha)

print("=" * 140)

---
### Imported master block 153

`BLOCK 13 — FINAL MANUSCRIPT PRODUCT EXTRACTION & EDITORIAL FREEZE`


In [ ]:
# ==================================================================================================
# BLOCK 13 — FINAL MANUSCRIPT PRODUCT EXTRACTION & EDITORIAL FREEZE
# ==================================================================================================
# PURPOSE
# -------
# 1. Inventory ALL existing manuscript-oriented tables/figures from Blocks 8–12.
# 2. Preserve authoritative frozen products where already available.
# 3. Classify products:
#       KEEP_MAIN
#       KEEP_SUPPLEMENT
#       REBUILD_FROM_FROZEN
#       INTERNAL_ONLY
#       DISCARD_DUPLICATE
# 4. Build final manuscript-product directory.
# 5. Copy authoritative existing products.
# 6. Generate final main tables that can be reconstructed without new inference.
# 7. Produce editorial inventory + provenance ledger.
#
# ABSOLUTE RULES
# --------------
# - NO NEW MODELS.
# - NO NEW HYPOTHESES.
# - NO NEW WINDOW SEARCH.
# - NO NEW SENSITIVITY ANALYSIS.
# - NO MODIFICATION OF BLOCKS 8–12.
# - NO RECOMPUTATION OF SCIENTIFIC RESULTS.
# - Any rebuilt product MUST use frozen numerical outputs only.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil
import re
import os
from datetime import datetime

print("="*135)
print("BLOCK 13 — FINAL MANUSCRIPT PRODUCT EXTRACTION & EDITORIAL FREEZE")
print("="*135)

# --------------------------------------------------------------------------------------------------
# A. PROJECT PATHS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
assert PROJECT.exists(), f"Project not found: {PROJECT}"

CLOSURE = PROJECT / "FINAL_MANUSCRIPT_CLOSURE"
assert CLOSURE.exists(), "Block 12 closure directory not found."

OUT = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS"

MAIN = OUT / "MAIN"
SUPP = OUT / "SUPPLEMENT"
CAPTIONS = OUT / "CAPTIONS"
SOURCE = OUT / "SOURCE_DATA"
AUDIT = OUT / "AUDIT"
INTERNAL = OUT / "INTERNAL_LEDGER"

for p in [OUT, MAIN, SUPP, CAPTIONS, SOURCE, AUDIT, INTERNAL]:
    p.mkdir(parents=True, exist_ok=True)

print("\nA. PROJECT")
print("-"*135)
print("PROJECT :", PROJECT)
print("OUTPUT  :", OUT)


# --------------------------------------------------------------------------------------------------
# B. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def find_files(root, patterns):
    hits = []
    if not root.exists():
        return hits

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        n = p.name.lower()
        if any(re.search(pattern, n, re.I) for pattern in patterns):
            hits.append(p)
    return sorted(set(hits))

def copy_verified(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    a = sha256(src)
    b = sha256(dst)

    assert a == b, f"Copy SHA mismatch: {src}"

    return {
        "source": str(src),
        "destination": str(dst),
        "sha256": a
    }

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def classify_name(path):
    n = path.name.lower()

    if any(x in n for x in ["manifest", "audit", "qa", "ledger", "checkpoint"]):
        return "INTERNAL_OR_SUPPLEMENT"

    if any(x in n for x in ["figure", "fig_", "fig.", "plot"]):
        return "FIGURE"

    if any(x in n for x in ["table", "tbl"]):
        return "TABLE"

    return "OTHER"


# --------------------------------------------------------------------------------------------------
# C. AUTHORITATIVE FREEZE GATE
# --------------------------------------------------------------------------------------------------

MASTER = PROJECT / "SNAPSHOTS/POST_BLOCK85C_20260915_034110/03_MASTER/MASTER_ANALYTICAL_v1_1.csv"

EXPECTED_MASTER_SHA = "4c0d420617280283770ef88b9fdcacc153484e6a85b00aae1555ba5ca76b1299"

assert MASTER.exists()
actual_master_sha = sha256(MASTER)

B10_MANIFEST = PROJECT / "JOINT_HQ_ANALYSIS/FULL_LOCO/RESULTS/BLOCK102B_B_FINAL_manifest.json"
B11_MANIFEST = PROJECT / "CONNECTIVITY_ANALYSIS/H3_MODELS/FINAL_MANUSCRIPT_PACKAGE/BLOCK113_H3_MANUSCRIPT_FREEZE_manifest.json"
B12_MANIFEST = CLOSURE / "AUDIT/BLOCK12_FINAL_ANALYSIS_CLOSURE_manifest.json"

assert actual_master_sha == EXPECTED_MASTER_SHA
assert B10_MANIFEST.exists()
assert B11_MANIFEST.exists()
assert B12_MANIFEST.exists()

print("\n" + "="*135)
print("B. FREEZE GATE")
print("="*135)
print("MASTER SHA PASS :", actual_master_sha == EXPECTED_MASTER_SHA)
print("BLOCK 10        :", "FOUND")
print("BLOCK 11        :", "FOUND")
print("BLOCK 12        :", "FOUND")
print("SCIENTIFIC FREEZE: PASS")


# --------------------------------------------------------------------------------------------------
# D. GLOBAL MANUSCRIPT-PRODUCT INVENTORY
# --------------------------------------------------------------------------------------------------

print("\n" + "="*135)
print("C. GLOBAL HISTORICAL PRODUCT INVENTORY")
print("="*135)

product_patterns = [
    r"figure",
    r"fig[_\-\s]?\d",
    r"table",
    r"manuscript",
    r"supp",
    r"caption",
    r"memory.*spectrum",
    r"cq",
    r"joint.*support",
    r"h3"
]

all_hits = find_files(PROJECT, product_patterns)

inventory_rows = []

for p in all_hits:
    # Avoid inventorying our own just-created Block13 output.
    if OUT in p.parents:
        continue

    try:
        size = p.stat().st_size
    except Exception:
        size = np.nan

    inventory_rows.append({
        "path": str(p),
        "filename": p.name,
        "extension": p.suffix.lower(),
        "size_bytes": size,
        "generic_type": classify_name(p),
        "sha256": sha256(p) if p.stat().st_size < 500_000_000 else "SKIPPED_LARGE"
    })

inventory = pd.DataFrame(inventory_rows)

inventory.to_csv(
    AUDIT / "BLOCK13_GLOBAL_HISTORICAL_PRODUCT_INVENTORY_v1.csv",
    index=False
)

print("Historical candidate products found:", len(inventory))

if len(inventory):
    print("\nBy extension:")
    print(inventory["extension"].value_counts().head(20).to_string())

    print("\nBy generic type:")
    print(inventory["generic_type"].value_counts().to_string())


# --------------------------------------------------------------------------------------------------
# E. LOCATE AUTHORITATIVE FROZEN PRODUCTS
# --------------------------------------------------------------------------------------------------

print("\n" + "="*135)
print("D. AUTHORITATIVE PRODUCT SOURCES")
print("="*135)

# ----- Block 8 -----------------------------------------------------------------

B8_ROOTS = [
    PROJECT / "MEMORY_ANALYSIS",
    PROJECT / "HYDROCLIMATE_MEMORY",
    PROJECT / "MEMORY",
    PROJECT / "SNAPSHOTS/POST_BLOCK85C_20260915_034110",
    PROJECT
]

b8_candidates = []

for root in B8_ROOTS:
    if root.exists():
        b8_candidates.extend(
            find_files(
                root,
                [
                    r"block8",
                    r"memory.*spectrum",
                    r"memory.*figure",
                    r"memory.*table",
                    r"antecedent.*memory"
                ]
            )
        )

b8_candidates = sorted(set(b8_candidates))

# ----- Block 9 -----------------------------------------------------------------

B9_ROOT = PROJECT / "CQ_ANALYSIS/FINAL_BLOCK9_PACKAGE"

B9_XLSX = B9_ROOT / "BLOCK9_CQ_TABLES_FINAL_v1.xlsx"

b9_fig_candidates = find_files(
    PROJECT / "CQ_ANALYSIS",
    [
        r"figure.*cq",
        r"cq.*figure",
        r"discharge.*response",
        r"block9.*fig"
    ]
)

# ----- Block 10 ----------------------------------------------------------------

B10_ROOT = PROJECT / "JOINT_HQ_ANALYSIS/FULL_LOCO/RESULTS"

B10_SUPPORT = B10_ROOT / "BLOCK102B_B_FINAL_JOINT_SUPPORT_SPECTRUM_v1.csv"
B10_50 = B10_ROOT / "BLOCK102B_B_ALL_50_LOCO_CONTRASTS_v1.csv"
B10_QA = B10_ROOT / "BLOCK102B_B_FINAL_QA_v1.csv"

# ----- Block 11 ----------------------------------------------------------------

B11_ROOT = PROJECT / "CONNECTIVITY_ANALYSIS/H3_MODELS/FINAL_MANUSCRIPT_PACKAGE"

B11_TABLE = B11_ROOT / "TABLE_H3_MAIN_FINAL_v1.csv"
B11_XLSX = B11_ROOT / "BLOCK11_H3_FINAL_TABLES_v1.xlsx"
B11_FIG_PNG = B11_ROOT / "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.png"
B11_FIG_PDF = B11_ROOT / "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.pdf"
B11_CAPTION = B11_ROOT / "FIGURE_H3_CAPTION_FINAL_v1.txt"
B11_50 = B11_ROOT / "TABLE_S_H3_ALL_50_LOCO_CONTRASTS_v1.csv"
B11_QA = B11_ROOT / "TABLE_S_H3_COMPUTATIONAL_QA_100_FITS_v1.csv"

# ----- Block 12 ----------------------------------------------------------------

LEDGER = CLOSURE / "RESULT_LEDGER/MASTER_RESULT_LEDGER_v1.csv"
CLAIMS = CLOSURE / "RESULT_LEDGER/H1_H3_CLAIM_EVIDENCE_MATRIX_v1.csv"
B12_XLSX = CLOSURE / "FINAL_MANUSCRIPT_TABLES_AND_LEDGER_v1.xlsx"

sources = {
    "BLOCK9_WORKBOOK": B9_XLSX,
    "BLOCK10_SUPPORT": B10_SUPPORT,
    "BLOCK10_50_LOCO": B10_50,
    "BLOCK10_QA": B10_QA,
    "BLOCK11_MAIN_TABLE": B11_TABLE,
    "BLOCK11_WORKBOOK": B11_XLSX,
    "BLOCK11_FIGURE_PNG": B11_FIG_PNG,
    "BLOCK11_FIGURE_PDF": B11_FIG_PDF,
    "BLOCK11_CAPTION": B11_CAPTION,
    "BLOCK11_50_LOCO": B11_50,
    "BLOCK11_QA": B11_QA,
    "RESULT_LEDGER": LEDGER,
    "CLAIM_MATRIX": CLAIMS,
    "BLOCK12_WORKBOOK": B12_XLSX
}

for k, p in sources.items():
    print(f"{k:25s}: {'FOUND' if p.exists() else 'NOT FOUND'} {p if p.exists() else ''}")

print("\nBlock 8 historical candidates:", len(b8_candidates))
for p in b8_candidates[:30]:
    print("  ", p)

print("\nBlock 9 figure candidates:", len(b9_fig_candidates))
for p in b9_fig_candidates[:30]:
    print("  ", p)


# --------------------------------------------------------------------------------------------------
# F. FINAL PRODUCT POLICY
# --------------------------------------------------------------------------------------------------

print("\n" + "="*135)
print("E. FINAL PRODUCT POLICY")
print("="*135)

policy = pd.DataFrame([
    {
        "product_id": "FIGURE_01",
        "section": "Results 3.1",
        "product": "Study area, monitoring network and contributing-area context",
        "placement": "MAIN",
        "source_block": "Spatial architecture / Block 11.0",
        "decision": "REBUILD_OR_POLISH_FROM_FROZEN",
        "reason": "Needs final manuscript cartographic assembly; no new analysis."
    },
    {
        "product_id": "TABLE_01",
        "section": "Results 3.1",
        "product": "Dataset and monitoring architecture",
        "placement": "MAIN",
        "source_block": "Block 12",
        "decision": "REBUILD_FROM_FROZEN",
        "reason": "Descriptive architecture only; frozen counts."
    },
    {
        "product_id": "FIGURE_02",
        "section": "Results 3.2",
        "product": "Hydroclimatic temporal-response / memory spectrum",
        "placement": "MAIN",
        "source_block": "Block 8",
        "decision": "KEEP_IF_FOUND_ELSE_REBUILD_FROM_FROZEN",
        "reason": "Core H2 visual."
    },
    {
        "product_id": "TABLE_MEMORY_MAIN",
        "section": "Results 3.2",
        "product": "Strict antecedent-memory hydrochemical tests",
        "placement": "MAIN",
        "source_block": "Block 8",
        "decision": "KEEP_REBUILD_FROM_FROZEN",
        "reason": "Four strict hydrochemical tests only."
    },
    {
        "product_id": "FIGURE_03",
        "section": "Results 3.3",
        "product": "Discharge–water-quality response spectrum",
        "placement": "MAIN",
        "source_block": "Block 9",
        "decision": "KEEP_IF_FOUND_ELSE_REBUILD_FROM_FROZEN",
        "reason": "Core discharge-response visual."
    },
    {
        "product_id": "TABLE_02",
        "section": "Results 3.3",
        "product": "Antecedent hydroclimate beyond contemporaneous discharge",
        "placement": "MAIN",
        "source_block": "Block 10",
        "decision": "REBUILD_FROM_FROZEN",
        "reason": "Primary H1 inferential table."
    },
    {
        "product_id": "FIGURE_04",
        "section": "Results 3.4",
        "product": "Contributing-area modulation of antecedent forcing",
        "placement": "MAIN",
        "source_block": "Block 11",
        "decision": "KEEP_MAIN",
        "reason": "Already frozen manuscript figure."
    },
    {
        "product_id": "TABLE_H3_MAIN",
        "section": "Results 3.4",
        "product": "H3 contributing-area interaction results",
        "placement": "MAIN_OR_COMBINE",
        "source_block": "Block 11",
        "decision": "KEEP",
        "reason": "Already frozen; final layout decision can be made during manuscript assembly."
    },
    {
        "product_id": "H1_H3_SYNTHESIS",
        "section": "Writing ledger",
        "product": "Integrated H1–H3 synthesis",
        "placement": "INTERNAL",
        "source_block": "Block 12",
        "decision": "INTERNAL_ONLY",
        "reason": "Useful for writing but potentially redundant as a main manuscript table."
    },
    {
        "product_id": "TABLE_S01",
        "section": "Supplement",
        "product": "Complete 50 response–driver memory spectrum",
        "placement": "SUPPLEMENT",
        "source_block": "Block 8",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "Complete H2 classification."
    },
    {
        "product_id": "TABLE_S02",
        "section": "Supplement",
        "product": "Complete 290 window-level memory inference",
        "placement": "SUPPLEMENT",
        "source_block": "Block 8",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "Full memory-model inference."
    },
    {
        "product_id": "TABLE_S03",
        "section": "Supplement",
        "product": "Complete C-Q results and C09 sensitivity",
        "placement": "SUPPLEMENT",
        "source_block": "Block 9",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "Includes unsupported and sensitivity-only responses."
    },
    {
        "product_id": "TABLE_S04",
        "section": "Supplement",
        "product": "Block 10 campaign-level LOCO contrasts",
        "placement": "SUPPLEMENT",
        "source_block": "Block 10",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "50 held-out campaign contrasts."
    },
    {
        "product_id": "TABLE_S05",
        "section": "Supplement",
        "product": "Block 10 computational QA",
        "placement": "SUPPLEMENT",
        "source_block": "Block 10",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "Reproducibility and Bayesian QA."
    },
    {
        "product_id": "TABLE_S06",
        "section": "Supplement",
        "product": "H3 campaign-level LOCO contrasts",
        "placement": "SUPPLEMENT",
        "source_block": "Block 11",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "50 H3 held-out campaign contrasts."
    },
    {
        "product_id": "TABLE_S07",
        "section": "Supplement",
        "product": "H3 computational QA",
        "placement": "SUPPLEMENT",
        "source_block": "Block 11",
        "decision": "KEEP_SUPPLEMENT",
        "reason": "100 LOCO fit QA."
    }
])

policy.to_csv(AUDIT / "BLOCK13_FINAL_PRODUCT_POLICY_v1.csv", index=False)
print(policy.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# G. TABLE 1 — DATASET / MONITORING ARCHITECTURE
# --------------------------------------------------------------------------------------------------

table1 = pd.DataFrame([
    ["Study campaigns", "10", "C01–C10; 2023–2026"],
    ["Unique master stations", "41", "Analytical master"],
    ["Unique FIELD observations", "133", "Physical field events"],
    ["Unique sample IDs", "146", "Laboratory/sample identity"],
    ["Observed-discharge events", "119", "Primary discharge architecture"],
    ["Primary H3 stream-network stations", "32", "Geometry-eligible contributing-area population"],
    ["Hydroclimate event rows", "146", "Sample-level hydroclimate calendar"],
    ["Unique hydroclimate event dates", "93", "Distinct temporal states represented by event dates"],
    ["Antecedent hydroclimate windows", "29", "Prespecified candidate-window bank"]
], columns=["Metric", "Value", "Definition"])

table1.to_csv(MAIN / "TABLE_01_DATASET_MONITORING_ARCHITECTURE_FINAL.csv", index=False)


# --------------------------------------------------------------------------------------------------
# H. BLOCK 8 MAIN STRICT-MEMORY TABLE
# --------------------------------------------------------------------------------------------------

memory_main = pd.DataFrame([
    ["Electrical conductivity", "Precipitation", 14, -0.360, -0.457, -0.255, 1.000, +0.194, +0.113, +0.275],
    ["Electrical conductivity", "Precipitation", 60, -0.365, -0.463, -0.266, 1.000, +0.206, +0.098, +0.314],
    ["Na", "Precipitation", 60, -0.317, -0.389, -0.240, 1.000, +0.231, +0.134, +0.327],
    ["SO₄", "Precipitation", 60, -0.497, -0.623, -0.370, 1.000, +0.201, +0.069, +0.334],
], columns=[
    "Response",
    "Hydroclimatic_driver",
    "Window_days",
    "Posterior_beta_std",
    "HDI_lower",
    "HDI_upper",
    "Pdirection",
    "Delta_LPD_vs_1d",
    "Delta_LPD_CI_lower",
    "Delta_LPD_CI_upper"
])

memory_main.to_csv(
    MAIN / "TABLE_MEMORY_STRICT_ANTECEDENT_SUPPORT_FINAL.csv",
    index=False
)


# --------------------------------------------------------------------------------------------------
# I. BLOCK 9 MAIN C-Q TABLE — EXTRACT FROM AUTHORITATIVE WORKBOOK
# --------------------------------------------------------------------------------------------------

assert B9_XLSX.exists()

b9_response = pd.read_excel(B9_XLSX, sheet_name="Response_Level")

wanted_b9 = [
    "display_name",
    "n_primary",
    "n_censored_primary",
    "transformation",
    "primary_beta_std",
    "primary_hdi_low",
    "primary_hdi_high",
    "primary_Pdirection",
    "primary_CQ_exponent",
    "primary_mean_delta_LPD",
    "primary_campaigns_Q_better",
    "primary_support_class",
    "sensitivity_status"
]

missing = [c for c in wanted_b9 if c not in b9_response.columns]
assert not missing, f"Missing Block9 columns: {missing}"

cq_main = b9_response[wanted_b9].copy()

cq_main.columns = [
    "Response",
    "n",
    "Censored_n",
    "Transformation",
    "beta_Q_std",
    "HDI_lower",
    "HDI_upper",
    "Pdirection",
    "CQ_exponent",
    "Mean_delta_LPD",
    "Campaigns_Q_better",
    "Primary_inference",
    "C09_sensitivity"
]

cq_main.to_csv(
    MAIN / "TABLE_CQ_DISCHARGE_RESPONSE_SPECTRUM_FINAL.csv",
    index=False
)


# --------------------------------------------------------------------------------------------------
# J. BLOCK 10 — MAIN H+Q TABLE
# --------------------------------------------------------------------------------------------------

assert B10_SUPPORT.exists()

b10 = pd.read_csv(B10_SUPPORT)

required10 = [
    "response_key",
    "hydro_var",
    "mean_delta_HQ_minus_Q",
    "median_delta_HQ_minus_Q",
    "campaigns_HQ_better_Q",
    "beta_H_conditional_median_std",
    "beta_H_conditional_hdi_low_std",
    "beta_H_conditional_hdi_high_std",
    "beta_H_conditional_Pdirection",
    "joint_support"
]

missing = [c for c in required10 if c not in b10.columns]
assert not missing, f"Missing Block10 columns: {missing}"

response_labels = {
    "AC::C.E._SOL::dS/m": "Electrical conductivity",
    "AC::Na::mg/L": "Na",
    "AC::SO4_ppm::mg/L": "SO₄",
    "FIELD::Water temperature::degC": "Water temperature"
}

hydro_labels = {
    "ppt_sum_14d": "Precipitation, 14 d",
    "ppt_sum_60d": "Precipitation, 60 d",
    "sm_rootzone_mean_90d": "Root-zone soil moisture, 90 d"
}

table2 = pd.DataFrame({
    "Response": b10["response_key"].map(response_labels).fillna(b10["response_key"]),
    "Antecedent_predictor": b10["hydro_var"].map(hydro_labels).fillna(b10["hydro_var"]),
    "Role": b10["joint_analysis_role"],
    "beta_H_given_Q": b10["beta_H_conditional_median_std"],
    "HDI_lower": b10["beta_H_conditional_hdi_low_std"],
    "HDI_upper": b10["beta_H_conditional_hdi_high_std"],
    "Pdirection": b10["beta_H_conditional_Pdirection"],
    "Mean_delta_LPD_HQ_minus_Q": b10["mean_delta_HQ_minus_Q"],
    "Median_delta_LPD_HQ_minus_Q": b10["median_delta_HQ_minus_Q"],
    "Campaigns_HQ_better": b10["campaigns_HQ_better_Q"],
    "Inference": np.where(b10["joint_support"], "Supported", "Not supported")
})

table2.to_csv(
    MAIN / "TABLE_02_ANTECEDENT_HYDROCLIMATE_BEYOND_Q_FINAL.csv",
    index=False
)


# --------------------------------------------------------------------------------------------------
# K. BLOCK 11 — COPY FROZEN MAIN PRODUCTS
# --------------------------------------------------------------------------------------------------

copy_log = []

assert B11_TABLE.exists()
assert B11_FIG_PNG.exists()
assert B11_FIG_PDF.exists()

copy_log.append(
    copy_verified(
        B11_TABLE,
        MAIN / "TABLE_H3_CONTRIBUTING_AREA_MODULATION_FINAL.csv"
    )
)

copy_log.append(
    copy_verified(
        B11_FIG_PNG,
        MAIN / "FIGURE_04_CONTRIBUTING_AREA_MODULATION_FINAL.png"
    )
)

copy_log.append(
    copy_verified(
        B11_FIG_PDF,
        MAIN / "FIGURE_04_CONTRIBUTING_AREA_MODULATION_FINAL.pdf"
    )
)

if B11_CAPTION.exists():
    copy_log.append(
        copy_verified(
            B11_CAPTION,
            CAPTIONS / "FIGURE_04_CAPTION_FINAL.txt"
        )
    )


# --------------------------------------------------------------------------------------------------
# L. SUPPLEMENTARY AUTHORITATIVE PRODUCTS
# --------------------------------------------------------------------------------------------------

# Block 9 full workbook retained as authoritative supplementary source.
copy_log.append(
    copy_verified(
        B9_XLSX,
        SUPP / "TABLE_S03_BLOCK9_CQ_COMPLETE_RESULTS_SOURCE.xlsx"
    )
)

# Block 10
if B10_50.exists():
    copy_log.append(
        copy_verified(
            B10_50,
            SUPP / "TABLE_S04_BLOCK10_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
        )
    )

if B10_QA.exists():
    copy_log.append(
        copy_verified(
            B10_QA,
            SUPP / "TABLE_S05_BLOCK10_COMPUTATIONAL_QA.csv"
        )
    )

# Block 11
if B11_50.exists():
    copy_log.append(
        copy_verified(
            B11_50,
            SUPP / "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
        )
    )

if B11_QA.exists():
    copy_log.append(
        copy_verified(
            B11_QA,
            SUPP / "TABLE_S07_H3_COMPUTATIONAL_QA_100_FITS.csv"
        )
    )


# --------------------------------------------------------------------------------------------------
# M. COPY RESULT LEDGER + CLAIM MATRIX AS INTERNAL WRITING CONTROL
# --------------------------------------------------------------------------------------------------

if LEDGER.exists():
    copy_log.append(
        copy_verified(
            LEDGER,
            INTERNAL / "MASTER_RESULT_LEDGER_FROZEN.csv"
        )
    )

if CLAIMS.exists():
    copy_log.append(
        copy_verified(
            CLAIMS,
            INTERNAL / "H1_H3_CLAIM_EVIDENCE_MATRIX_FROZEN.csv"
        )
    )


# --------------------------------------------------------------------------------------------------
# N. IDENTIFY HISTORICAL FIGURES — KEEP / REVIEW / DUPLICATE
# --------------------------------------------------------------------------------------------------

print("\n" + "="*135)
print("F. HISTORICAL FIGURE REVIEW")
print("="*135)

figure_inventory = inventory[
    inventory["extension"].isin([".png", ".pdf", ".svg", ".jpg", ".jpeg", ".tif", ".tiff"])
].copy()

def historical_decision(row):
    n = row["filename"].lower()
    p = row["path"].lower()

    # Block11 final figure already authoritative.
    if "figure_h3_connectivity_modulation_final" in n:
        return "KEEP_MAIN_AUTHORITATIVE"

    if any(x in n for x in ["memory", "block8"]) and "fig" in n:
        return "REVIEW_FOR_FIGURE_02"

    if any(x in n for x in ["cq", "discharge", "block9"]) and any(x in n for x in ["fig", "figure"]):
        return "REVIEW_FOR_FIGURE_03"

    if any(x in n for x in ["map", "study_area", "network", "spatial"]) and any(x in n for x in ["fig", "figure", "map"]):
        return "REVIEW_FOR_FIGURE_01"

    if "final_manuscript_products" in p:
        return "IGNORE_CURRENT_OUTPUT"

    return "INTERNAL_OR_UNRESOLVED"

if len(figure_inventory):
    figure_inventory["editorial_decision"] = figure_inventory.apply(historical_decision, axis=1)

    figure_inventory.to_csv(
        AUDIT / "BLOCK13_HISTORICAL_FIGURE_REVIEW_v1.csv",
        index=False
    )

    print(
        figure_inventory[
            figure_inventory["editorial_decision"] != "INTERNAL_OR_UNRESOLVED"
        ][
            ["filename", "editorial_decision", "path"]
        ].to_string(index=False)
    )
else:
    print("No historical raster/vector figures found.")


# --------------------------------------------------------------------------------------------------
# O. CAPTIONS / TABLE NOTES
# --------------------------------------------------------------------------------------------------

captions = {
"TABLE_01_NOTE.txt":
"""Table 1. Monitoring and analytical architecture of the Huanta headwater dataset.

Counts distinguish physical field observations, analytical sample identities, hydroclimatic event rows,
and unique event dates. Hydroclimatic event dates represent the temporal states used to construct
antecedent forcing windows. The primary contributing-area analysis was restricted to geometry-eligible
stream-network stations.""",

"TABLE_MEMORY_NOTE.txt":
"""Strict antecedent hydroclimatic-memory support for hydrochemical responses.

Only hydrochemical windows satisfying the pre-specified strict antecedent-memory criterion are shown.
Posterior beta values are standardized effects. Delta LPD is the campaign-paired difference in held-out
log predictive density relative to the 1-day precipitation reference. Candidate windows are interpreted
as supported temporal scales of antecedent association and not as exact physical memory durations.""",

"TABLE_CQ_NOTE.txt":
"""Discharge–water-quality response spectrum.

beta_Q denotes the posterior effect of log10-transformed observed discharge on the standardized response.
Primary inference used observed discharge only. Positive mean Delta LPD indicates improved held-out
prediction relative to the station-only model. Left-censored observations were represented using
sample-specific censoring limits without substitution. Reconstructed C09 discharge was used only in
the frozen sensitivity analysis.""",

"TABLE_02_NOTE.txt":
"""Incremental predictive contribution of antecedent hydroclimate beyond contemporaneous discharge.

beta_H|Q denotes the standardized posterior effect of antecedent hydroclimate after conditioning on
contemporaneous log10-transformed observed discharge. Positive Delta LPD (H+Q - Q) indicates improved
out-of-campaign prediction after adding antecedent hydroclimate. Support was defined a priori as
Pdirection >= 0.95 and campaign-equal mean Delta LPD > 0. Campaign counts are descriptive rather than
inferential thresholds. Water temperature is retained as a secondary thermal reference.""",

"FIGURE_02_INTERPRETATION.txt":
"""Hydroclimatic temporal-response / memory spectrum.

The figure must distinguish supported antecedent memory, supported temporal-response ranges,
short-state responses, and combinations without robust support. Disjoint supported windows must not be
converted into artificial continuous ranges. Water temperature must be identified as a secondary
thermal-response reference rather than primary hydrochemical-memory evidence.""",

"FIGURE_03_INTERPRETATION.txt":
"""Discharge–water-quality response spectrum and out-of-campaign predictive support.

Panel A should show posterior standardized discharge effects with 95% HDIs. Panel B should show
campaign-equal held-out predictive improvement relative to the station-only model. Primary estimates
use observed discharge only. Reconstructed C09 discharge is a sensitivity analysis. Al and Fe must not
be presented as primary temporally generalizable discharge responses."""
}

for filename, text in captions.items():
    (CAPTIONS / filename).write_text(text, encoding="utf-8")


# --------------------------------------------------------------------------------------------------
# P. SOURCE-DATA COPIES FOR MAIN TABLES
# --------------------------------------------------------------------------------------------------

for src, dstname in [
    (B10_SUPPORT, "SOURCE_BLOCK10_FINAL_JOINT_SUPPORT.csv"),
    (B11_TABLE, "SOURCE_BLOCK11_FINAL_H3_SUPPORT.csv")
]:
    if src.exists():
        copy_log.append(copy_verified(src, SOURCE / dstname))

# Extract authoritative Block9 Response_Level as source CSV.
b9_response.to_csv(
    SOURCE / "SOURCE_BLOCK9_RESPONSE_LEVEL_FINAL.csv",
    index=False
)


# --------------------------------------------------------------------------------------------------
# Q. PRODUCT READINESS MATRIX
# --------------------------------------------------------------------------------------------------

fig2_found = False
fig3_found = False
fig1_found = False

if len(figure_inventory):
    fig2_found = (figure_inventory["editorial_decision"] == "REVIEW_FOR_FIGURE_02").any()
    fig3_found = (figure_inventory["editorial_decision"] == "REVIEW_FOR_FIGURE_03").any()
    fig1_found = (figure_inventory["editorial_decision"] == "REVIEW_FOR_FIGURE_01").any()

readiness = pd.DataFrame([
    ["Figure 1", "MAIN", "Study area / network / contributing-area context",
     "HISTORICAL_CANDIDATE_FOUND" if fig1_found else "REBUILD_FROM_FROZEN_SPATIAL_PRODUCTS"],
    ["Table 1", "MAIN", "Dataset and monitoring architecture", "READY"],
    ["Figure 2", "MAIN", "Hydroclimatic memory spectrum",
     "HISTORICAL_CANDIDATE_FOUND" if fig2_found else "REBUILD_FROM_FROZEN_BLOCK8"],
    ["Memory table", "MAIN", "Four strict antecedent-memory tests", "READY"],
    ["Figure 3", "MAIN", "Discharge–water-quality response spectrum",
     "HISTORICAL_CANDIDATE_FOUND" if fig3_found else "REBUILD_FROM_FROZEN_BLOCK9"],
    ["Table 2", "MAIN", "Antecedent hydroclimate beyond Q", "READY"],
    ["Figure 4", "MAIN", "Contributing-area modulation", "READY"],
    ["H3 table", "MAIN/OPTIONAL", "Contributing-area interaction results", "READY"],
    ["Table S1", "SUPPLEMENT", "50 response-driver memory spectrum", "LOCATE_OR_REBUILD_FROM_FROZEN"],
    ["Table S2", "SUPPLEMENT", "290 window-level memory inference", "LOCATE_OR_REBUILD_FROM_FROZEN"],
    ["Table S3", "SUPPLEMENT", "Complete C-Q + C09 sensitivity", "READY_SOURCE_WORKBOOK"],
    ["Table S4", "SUPPLEMENT", "Block10 50 LOCO contrasts", "READY"],
    ["Table S5", "SUPPLEMENT", "Block10 computational QA", "READY"],
    ["Table S6", "SUPPLEMENT", "H3 50 LOCO contrasts", "READY"],
    ["Table S7", "SUPPLEMENT", "H3 computational QA", "READY"],
], columns=["Product", "Placement", "Content", "Status"])

readiness.to_csv(
    AUDIT / "BLOCK13_PRODUCT_READINESS_MATRIX_v1.csv",
    index=False
)

print("\n" + "="*135)
print("G. PRODUCT READINESS MATRIX")
print("="*135)
print(readiness.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# R. COPY LOG
# --------------------------------------------------------------------------------------------------

copy_df = pd.DataFrame(copy_log)

if len(copy_df):
    copy_df.to_csv(
        AUDIT / "BLOCK13_VERIFIED_COPY_LOG_v1.csv",
        index=False
    )


# --------------------------------------------------------------------------------------------------
# S. CREATE CONSOLIDATED EDITORIAL WORKBOOK
# --------------------------------------------------------------------------------------------------

xlsx_out = OUT / "HUANTA_JOH_FINAL_MANUSCRIPT_PRODUCT_LEDGER_v1.xlsx"

with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    readiness.to_excel(writer, sheet_name="Product_Readiness", index=False)
    policy.to_excel(writer, sheet_name="Editorial_Policy", index=False)
    table1.to_excel(writer, sheet_name="Main_Table1_Architecture", index=False)
    memory_main.to_excel(writer, sheet_name="Main_Memory_Strict", index=False)
    cq_main.to_excel(writer, sheet_name="Main_CQ", index=False)
    table2.to_excel(writer, sheet_name="Main_HQ", index=False)

    if B11_TABLE.exists():
        pd.read_csv(B11_TABLE).to_excel(
            writer,
            sheet_name="Main_H3",
            index=False
        )

    if LEDGER.exists():
        pd.read_csv(LEDGER).to_excel(
            writer,
            sheet_name="Result_Ledger",
            index=False
        )

    if CLAIMS.exists():
        pd.read_csv(CLAIMS).to_excel(
            writer,
            sheet_name="Claim_Matrix",
            index=False
        )

    inventory.to_excel(
        writer,
        sheet_name="Historical_Inventory",
        index=False
    )


# --------------------------------------------------------------------------------------------------
# T. METHODS HANDOFF — FREEZE WHAT METHODS MUST DOCUMENT
# --------------------------------------------------------------------------------------------------

methods_handoff = """
HUANTA JOURNAL OF HYDROLOGY — METHODS HANDOFF
=============================================

Scientific analysis is CLOSED.

Methods must document, in this order:

1. Study design and monitoring architecture
   - 10 campaigns (C01–C10), 2023–2026
   - physical field-event identity versus analytical sample identity
   - repeated laboratory subsamples retained where legitimately distinct

2. Water-quality measurements and analytical harmonization
   - FIELD / A+C / COD-DQO / coliform / metals architecture
   - analyte-method crosswalk
   - no averaging of method-specific duplicate analytes
   - frozen pH, temperature and DO policies

3. Censoring architecture
   - sample-specific censoring limits
   - explicit left-censor likelihood
   - no global LOD/2 substitution
   - E. coli and Hg descriptive where appropriate

4. Hydrometry
   - observed discharge primary
   - reconstructed C09 discharge sensitivity only
   - no silent combination
   - event-level discharge architecture

5. Hydroclimatic forcing
   - CHIRPS precipitation
   - ERA5-Land soil moisture, air temperature and potential evaporation
   - Razuhuillca hydroclimatic reference domain
   - shared event-date forcing, NOT station-specific contributing-catchment hydroclimate
   - exact event dates; no campaign midpoint
   - 29 pre-specified antecedent windows

6. Hydroclimatic temporal-response / memory analysis
   - frozen response transformations
   - posterior directional support
   - campaign-held-out predictive support
   - strict antecedent-memory criterion
   - short-state response definition
   - no argmax interpretation as physical memory constant

7. Discharge-response models
   - Bayesian hierarchical model
   - log10 observed Q
   - station random intercept
   - explicit censoring
   - campaign LOCO
   - primary versus C09 sensitivity
   - support gate Pdirection >= 0.95 and campaign-equal mean Delta LPD > 0

8. Conditional antecedent hydroclimate beyond discharge
   - H+Q versus Q-only
   - same rows within each fold
   - training-fold-only scaling
   - unseen stations marginalized
   - exactly five frozen tests
   - no HxQ
   - no post-hoc window search

9. Catchment organization / contributing area
   - MERIT contributing area
   - primary geometry-eligible stream-network population
   - log10 contributing area
   - fixed station attribute
   - station random intercept absorbs additive persistent station differences
   - primary estimand beta_HxC
   - model: Q + H + H:C + station RE
   - do not interpret contributing area as measured dynamic connectivity

10. Bayesian computation and validation
   - priors
   - chains/draws/tuning/target_accept
   - Rhat <= 1.01
   - ESS >= 400
   - zero-divergence requirement
   - surgical higher-MC repairs under unchanged model/inferential criteria
   - campaign LOCO as temporal generalization test

11. Inferential language
   - predictive/associational, not causal
   - shared antecedent hydroclimatic forcing
   - contributing-area modulation
   - no causal storage, residence-time or flow-path claims without direct measurement
"""

(CLOSURE / "METHODS_HANDOFF_FINAL_v1.txt").write_text(
    methods_handoff.strip() + "\n",
    encoding="utf-8"
)

copy_verified(
    CLOSURE / "METHODS_HANDOFF_FINAL_v1.txt",
    OUT / "METHODS_HANDOFF_FINAL_v1.txt"
)


# --------------------------------------------------------------------------------------------------
# U. MANIFEST
# --------------------------------------------------------------------------------------------------

manifest = {
    "block": "13",
    "name": "FINAL MANUSCRIPT PRODUCT EXTRACTION & EDITORIAL FREEZE",
    "timestamp": datetime.now().isoformat(),
    "project": str(PROJECT),
    "output": str(OUT),

    "scientific_freeze": {
        "master_sha_expected": EXPECTED_MASTER_SHA,
        "master_sha_actual": actual_master_sha,
        "master_sha_pass": actual_master_sha == EXPECTED_MASTER_SHA,
        "block10_manifest_found": B10_MANIFEST.exists(),
        "block11_manifest_found": B11_MANIFEST.exists(),
        "block12_manifest_found": B12_MANIFEST.exists()
    },

    "analysis_changes": {
        "new_models_fit": 0,
        "new_hypotheses": 0,
        "new_window_search": False,
        "new_sensitivity_analysis": False,
        "blocks_8_12_modified": False
    },

    "main_products": {
        "table1_ready": True,
        "memory_strict_table_ready": True,
        "cq_table_ready": True,
        "hq_table_ready": True,
        "h3_table_ready": B11_TABLE.exists(),
        "h3_figure_ready": B11_FIG_PNG.exists(),
        "figure1_historical_candidate": bool(fig1_found),
        "figure2_historical_candidate": bool(fig2_found),
        "figure3_historical_candidate": bool(fig3_found)
    },

    "historical_inventory_n": int(len(inventory)),
    "block8_candidates_n": int(len(b8_candidates)),
    "block9_figure_candidates_n": int(len(b9_fig_candidates)),

    "next_phase": "FINALIZE MISSING FIGURES/TABLES FROM FROZEN OUTPUTS, THEN METHODS WRITING"
}

manifest_path = AUDIT / "BLOCK13_FINAL_PRODUCT_EXTRACTION_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)


# --------------------------------------------------------------------------------------------------
# V. ZIP PACKAGE
# --------------------------------------------------------------------------------------------------

zip_base = PROJECT / "HUANTA_JOH_FINAL_MANUSCRIPT_PRODUCTS_v1"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUT
)

zip_sha = sha256(zip_path)


# --------------------------------------------------------------------------------------------------
# W. FINAL GATE
# --------------------------------------------------------------------------------------------------

print("\n" + "="*135)
print("BLOCK 13 — FINAL GATE")
print("="*135)

print("MASTER SHA VERIFIED        :", actual_master_sha == EXPECTED_MASTER_SHA)
print("NEW MODELS FIT             : 0")
print("NEW HYPOTHESES             : 0")
print("NEW WINDOW SEARCH          : False")
print("NEW SENSITIVITY ANALYSIS   : False")
print("BLOCKS 8–12 MODIFIED       : False")

print()
print("TABLE 1 ARCHITECTURE       : READY")
print("MEMORY STRICT TABLE        : READY")
print("C-Q TABLE                  : READY")
print("H+Q TABLE                  : READY")
print("H3 TABLE                   :", "READY" if B11_TABLE.exists() else "MISSING")
print("H3 FIGURE                  :", "READY" if B11_FIG_PNG.exists() else "MISSING")

print()
print("FIGURE 1 OLD CANDIDATE     :", fig1_found)
print("FIGURE 2 OLD CANDIDATE     :", fig2_found)
print("FIGURE 3 OLD CANDIDATE     :", fig3_found)

print()
print("HISTORICAL PRODUCTS FOUND  :", len(inventory))
print("BLOCK 8 CANDIDATES         :", len(b8_candidates))
print("BLOCK 9 FIGURE CANDIDATES  :", len(b9_fig_candidates))

print()
print("SCIENTIFIC ANALYSIS STATUS : CLOSED")
print("EDITORIAL EXTRACTION       : PASS")
print("NEXT STEP                  : RESOLVE ONLY NON-READY PRODUCTS, THEN WRITE METHODS")

print("\nFinal product directory:")
print(OUT)

print("\nEditorial workbook:")
print(xlsx_out)

print("\nMethods handoff:")
print(OUT / "METHODS_HANDOFF_FINAL_v1.txt")

print("\nManifest:")
print(manifest_path)

print("\nZIP:")
print(zip_path)

print("\nZIP SHA256:")
print(zip_sha)

print("="*135)

---
### Imported master block 154

`BLOCK 13R — RESOLVE BLOCK 8 PRODUCTS + FREEZE EXISTING FIGURES 2/3`


In [ ]:
# ==================================================================================================
# BLOCK 13R — RESOLVE BLOCK 8 PRODUCTS + FREEZE EXISTING FIGURES 2/3
# NO MODELS — NO NEW ANALYSIS — NO RECOMPUTATION
# ==================================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import shutil
import os
import re

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
OUT = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS"

MAIN = OUT / "MAIN"
SUPP = OUT / "SUPPLEMENT"
AUDIT = OUT / "AUDIT"
SOURCE = OUT / "SOURCE_DATA"

for d in [MAIN, SUPP, AUDIT, SOURCE]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

def copy_sha(src, dst):
    shutil.copy2(src, dst)
    assert sha256(src) == sha256(dst)
    return sha256(dst)

print("="*130)
print("BLOCK 13R — BLOCK 8 PRODUCT RESOLUTION + FIGURES 2/3 FREEZE")
print("="*130)

# ================================================================================================
# A. FIGURE 2 — EXISTING MEMORY FIGURES
# ================================================================================================

MEMROOT = PROJECT / "MEMORY_SPECTRUM"

fig2 = sorted([
    p for p in MEMROOT.glob("*")
    if p.suffix.lower() in [".png", ".pdf", ".svg"]
    and "FIGURE_MEMORY" in p.name.upper()
])

print("\nA. FIGURE 2 CANDIDATES")
print("-"*130)

rows = []

for p in fig2:
    rows.append({
        "filename": p.name,
        "extension": p.suffix.lower(),
        "bytes": p.stat().st_size,
        "sha256": sha256(p),
        "path": str(p)
    })

fig2_inventory = pd.DataFrame(rows)
print(fig2_inventory.to_string(index=False))

fig2_inventory.to_csv(
    AUDIT / "BLOCK13R_FIGURE2_CANDIDATES.csv",
    index=False
)

# Group by stem to make versions obvious.
print("\nFigure 2 version families:")
for stem in sorted(set(p.stem for p in fig2)):
    print(" ", stem)


# ================================================================================================
# B. BLOCK 8 — SEARCH FINAL SCIENTIFIC TABLE CANDIDATES
# ================================================================================================

print("\n" + "="*130)
print("B. BLOCK 8 FINAL TABLE CANDIDATES")
print("="*130)

search_roots = [
    PROJECT / "MEMORY_SPECTRUM",
    PROJECT / "FULL_LOCO/RESULTS",
    PROJECT / "INCREMENTAL_CHECKPOINTS/POST_BLOCK85E_20260915_234407",
]

patterns = [
    "MEMORY",
    "SUPPORTED",
    "WINDOW",
    "INFERENCE",
    "SPECTRUM",
    "JOINT",
    "LOCO"
]

candidates = []

for root in search_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        if p.suffix.lower() not in [".csv", ".xlsx"]:
            continue

        name = p.name.upper()

        if not any(x in name for x in patterns):
            continue

        rec = {
            "filename": p.name,
            "extension": p.suffix.lower(),
            "bytes": p.stat().st_size,
            "path": str(p),
            "sha256": sha256(p)
        }

        try:
            if p.suffix.lower() == ".csv":
                df = pd.read_csv(p)
                rec["shape"] = str(df.shape)
                rec["columns"] = " | ".join(map(str, df.columns))
            else:
                xls = pd.ExcelFile(p)
                rec["shape"] = "WORKBOOK"
                rec["columns"] = "SHEETS: " + " | ".join(xls.sheet_names)

        except Exception as e:
            rec["shape"] = "READ_ERROR"
            rec["columns"] = str(e)

        candidates.append(rec)

cand = pd.DataFrame(candidates)

# Rank the things most likely to be our 50 and 290 tables.
def priority(row):
    txt = (row["filename"] + " " + str(row["shape"])).upper()
    score = 0

    if "(50," in txt:
        score += 100
    if "(290," in txt:
        score += 100

    if "SUPPORTED_MEMORY" in txt:
        score += 30
    if "WINDOW_INFERENCE" in txt:
        score += 30
    if "JOINT" in txt:
        score += 10
    if "FINAL" in txt:
        score += 10

    return score

if len(cand):
    cand["priority"] = cand.apply(priority, axis=1)
    cand = cand.sort_values(
        ["priority", "filename"],
        ascending=[False, True]
    )

    pd.set_option("display.max_colwidth", 160)

    print(
        cand[
            ["filename", "shape", "priority", "columns", "path"]
        ].head(40).to_string(index=False)
    )

    cand.to_csv(
        AUDIT / "BLOCK13R_BLOCK8_TABLE_CANDIDATES.csv",
        index=False
    )

else:
    print("No Block 8 table candidates found.")


# ================================================================================================
# C. EXPLICITLY INSPECT THE KNOWN BLOCK 8.5E FINAL FILES
# ================================================================================================

print("\n" + "="*130)
print("C. BLOCK 8.5E KNOWN FINAL FILES")
print("="*130)

B85E = PROJECT / "INCREMENTAL_CHECKPOINTS/POST_BLOCK85E_20260915_234407"

known_names = [
    "BLOCK85E_STRICT_ANTECEDENT_MEMORY_WINDOWS_v1.csv",
    "BLOCK85E_SUPPORTED_MEMORY_FAMILY_INFERENCE_v1.csv",
    "BLOCK85E_SUPPORTED_MEMORY_WINDOW_INFERENCE_v1.csv",
    "BLOCK85E_JOINT_POSTERIOR_LOCO_ARCHITECTURE_v1.csv",
    "BLOCK85E_PAIRED_LPD_VS_BEST_INFERENCE_v1.csv",
    "BLOCK85E_PAIRED_LPD_VS_SHORTEST_INFERENCE_v1.csv",
]

known_rows = []

for name in known_names:
    p = B85E / name

    if not p.exists():
        print("MISSING:", name)
        continue

    df = pd.read_csv(p)

    known_rows.append({
        "filename": name,
        "rows": len(df),
        "cols": len(df.columns),
        "columns": " | ".join(df.columns),
        "sha256": sha256(p),
        "path": str(p)
    })

    print("\nFILE :", name)
    print("SHAPE:", df.shape)
    print("COLS :", list(df.columns))
    print(df.head(8).to_string(index=False))

known_df = pd.DataFrame(known_rows)

known_df.to_csv(
    AUDIT / "BLOCK13R_BLOCK85E_KNOWN_FILES.csv",
    index=False
)


# ================================================================================================
# D. FIGURE 3 — FREEZE AUTHORITATIVE FINAL_BLOCK9_PACKAGE VERSION
# ================================================================================================

print("\n" + "="*130)
print("D. FIGURE 3 — AUTHORITATIVE BLOCK 9 FREEZE")
print("="*130)

B9 = PROJECT / "CQ_ANALYSIS/FINAL_BLOCK9_PACKAGE"

F3PNG = B9 / "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1_600dpi.png"
F3PDF = B9 / "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1.pdf"
F3DATA = B9 / "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_DATA_v1.csv"

assert F3PNG.exists()
assert F3PDF.exists()
assert F3DATA.exists()

sha_png = copy_sha(
    F3PNG,
    MAIN / "FIGURE_03_DISCHARGE_RESPONSE_SPECTRUM_FINAL.png"
)

sha_pdf = copy_sha(
    F3PDF,
    MAIN / "FIGURE_03_DISCHARGE_RESPONSE_SPECTRUM_FINAL.pdf"
)

sha_data = copy_sha(
    F3DATA,
    SOURCE / "SOURCE_FIGURE_03_DISCHARGE_RESPONSE_SPECTRUM.csv"
)

print("FIGURE 3 PNG : FROZEN")
print("SHA256       :", sha_png)

print("FIGURE 3 PDF : FROZEN")
print("SHA256       :", sha_pdf)

print("SOURCE DATA  : FROZEN")
print("SHA256       :", sha_data)


# ================================================================================================
# E. CHECK WHETHER FIGURE 2 VERSIONS SHARE SOURCE DATA
# ================================================================================================

print("\n" + "="*130)
print("E. MEMORY_SPECTRUM DIRECTORY CONTENTS")
print("="*130)

if MEMROOT.exists():
    for p in sorted(MEMROOT.iterdir()):
        if p.is_file():
            print(
                f"{p.name:65s} "
                f"{p.stat().st_size:12,d} bytes"
            )


# ================================================================================================
# F. FINAL STATUS
# ================================================================================================

print("\n" + "="*130)
print("BLOCK 13R — STATUS")
print("="*130)

print("FIGURE 3                         : FROZEN")
print("FIGURE 2 CANDIDATE FAMILIES     :", len(set(p.stem for p in fig2)))
print("BLOCK 8.5E FILES INSPECTED      :", len(known_df))
print("NEW MODELS                      : 0")
print("NEW SCIENTIFIC RESULTS          : 0")
print("ANALYSIS FREEZE PRESERVED       : True")

print("\nStill to resolve:")
print("1. Select authoritative Figure 2 version.")
print("2. Identify/freeze Table S1 = 50 response-driver spectrum.")
print("3. Identify/freeze Table S2 = 290 window-level inference.")
print("4. Build Figure 1 from frozen spatial products.")
print("="*130)

---
### Imported master block 155

`BLOCK 13S — VISUAL REVIEW + FINAL FREEZE OF BLOCK 8 MANUSCRIPT PRODUCTS`


In [ ]:
# ==================================================================================================
# BLOCK 13S — VISUAL REVIEW + FINAL FREEZE OF BLOCK 8 MANUSCRIPT PRODUCTS
# NO ANALYSIS / NO MODELS / NO SCIENTIFIC CHANGES
# ==================================================================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import shutil
import hashlib

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
MEM = PROJECT / "MEMORY_SPECTRUM"
OUT = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS"

MAIN = OUT / "MAIN"
SUPP = OUT / "SUPPLEMENT"
SOURCE = OUT / "SOURCE_DATA"
AUDIT = OUT / "AUDIT"

for d in [MAIN, SUPP, SOURCE, AUDIT]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def freeze(src, dst):
    shutil.copy2(src, dst)
    a = sha256(src)
    b = sha256(dst)
    assert a == b
    return b


# ================================================================================================
# A. VISUAL COMPARISON — FIGURE 2
# ================================================================================================

figures = [
    ("A — Spectrum",
     MEM / "FIGURE_MEMORY_SPECTRUM_v1.png"),

    ("B — Publication",
     MEM / "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.png"),

    ("C — Alternative",
     MEM / "FIGURE_MEMORY_SPECTRUM_ALTERNATIVE_v1.png"),

    ("D — Fingerprint",
     MEM / "FIGURE_MEMORY_FINGERPRINT_v1.png"),
]

for label, path in figures:
    assert path.exists(), f"Missing: {path}"

fig, axes = plt.subplots(2, 2, figsize=(20, 15))

for ax, (label, path) in zip(axes.flat, figures):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(label, fontsize=16, fontweight="bold")
    ax.axis("off")

plt.tight_layout()

review = AUDIT / "BLOCK13S_FIGURE2_VISUAL_COMPARISON.png"

plt.savefig(
    review,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

print("\nVisual comparison saved:")
print(review)


# ================================================================================================
# B. FREEZE BLOCK 8 MAIN + SUPPLEMENTARY TABLES
# ================================================================================================

print("\n" + "="*120)
print("B. FREEZE AUTHORITATIVE BLOCK 8 TABLES")
print("="*120)

products = {
    "MAIN_MEMORY":
        MEM / "TABLE_MAIN_STRICT_HYDROCHEMICAL_MEMORY_v1.csv",

    "S1":
        MEM / "TABLE_S1_MEMORY_RESPONSE_DRIVER_CURVES_v1.csv",

    "S2":
        MEM / "TABLE_S2_MEMORY_WINDOW_FULL_RESULTS_v1.csv",

    "WORKBOOK":
        MEM / "MEMORY_SPECTRUM_MANUSCRIPT_TABLES_v1.xlsx",

    "NOTES":
        MEM / "MEMORY_TABLE_NOTES_v1.txt",

    "CAPTION":
        MEM / "FIGURE_MEMORY_SPECTRUM_caption_v1.txt",
}

for k, p in products.items():
    assert p.exists(), f"Missing {k}: {p}"

destinations = {
    "MAIN_MEMORY":
        MAIN / "TABLE_MEMORY_STRICT_HYDROCHEMICAL_FINAL.csv",

    "S1":
        SUPP / "TABLE_S01_MEMORY_RESPONSE_DRIVER_SPECTRUM_FINAL.csv",

    "S2":
        SUPP / "TABLE_S02_MEMORY_WINDOW_FULL_RESULTS_FINAL.csv",

    "WORKBOOK":
        SUPP / "TABLES_S01_S02_MEMORY_COMPLETE_FINAL.xlsx",

    "NOTES":
        SUPP / "TABLES_S01_S02_MEMORY_NOTES_FINAL.txt",

    "CAPTION":
        SOURCE / "FIGURE_02_MEMORY_CAPTION_SOURCE.txt",
}

freeze_rows = []

for key, src in products.items():

    dst = destinations[key]
    sha = freeze(src, dst)

    if src.suffix.lower() == ".csv":
        df = pd.read_csv(src)
        shape = str(df.shape)
    else:
        shape = "N/A"

    freeze_rows.append({
        "product": key,
        "source": str(src),
        "destination": str(dst),
        "shape": shape,
        "sha256": sha
    })

freeze_df = pd.DataFrame(freeze_rows)

print(freeze_df[
    ["product", "shape", "destination", "sha256"]
].to_string(index=False))

freeze_df.to_csv(
    AUDIT / "BLOCK13S_BLOCK8_TABLE_FREEZE.csv",
    index=False
)


# ================================================================================================
# C. SCIENTIFIC SANITY CHECK
# ================================================================================================

main = pd.read_csv(products["MAIN_MEMORY"])
s1 = pd.read_csv(products["S1"])
s2 = pd.read_csv(products["S2"])

print("\n" + "="*120)
print("C. SCIENTIFIC PRODUCT CHECK")
print("="*120)

print("Main strict hydrochemical table :", main.shape)
print("Table S1 response-driver curves :", s1.shape)
print("Table S2 window-level results   :", s2.shape)

assert len(main) == 4
assert len(s1) == 50
assert len(s2) == 290

print("\nExpected dimensions:")
print("Main = 4  :", len(main) == 4)
print("S1   = 50 :", len(s1) == 50)
print("S2   = 290:", len(s2) == 290)

print("\nMAIN MEMORY TABLE")
print(main.to_string(index=False))


# ================================================================================================
# D. FIGURE 2 FILE HASHES
# ================================================================================================

print("\n" + "="*120)
print("D. FIGURE 2 HASHES")
print("="*120)

hash_rows = []

for label, png in figures:

    stem = png.stem

    for ext in [".png", ".pdf", ".svg"]:

        p = MEM / f"{stem}{ext}"

        if p.exists():

            hash_rows.append({
                "version": label,
                "extension": ext,
                "sha256": sha256(p),
                "bytes": p.stat().st_size,
                "path": str(p)
            })

hash_df = pd.DataFrame(hash_rows)

print(hash_df.to_string(index=False))

hash_df.to_csv(
    AUDIT / "BLOCK13S_FIGURE2_HASHES.csv",
    index=False
)


# ================================================================================================
# E. FINAL STATUS
# ================================================================================================

print("\n" + "="*120)
print("BLOCK 13S — STATUS")
print("="*120)

print("MAIN MEMORY TABLE : FROZEN")
print("TABLE S1          : FROZEN")
print("TABLE S2          : FROZEN")
print("MEMORY WORKBOOK   : FROZEN")
print("FIGURE 2          : AWAITING VISUAL SELECTION ONLY")
print("FIGURE 3          : ALREADY FROZEN")
print("NEW MODELS        : 0")
print("NEW ANALYSIS      : 0")
print("SCIENTIFIC FREEZE : PRESERVED")

print("\nAfter Figure 2 selection:")
print("ONLY Figure 1 remains before METHODS.")
print("="*120)

---
### Imported master block 156

`BLOCK 13T — FINAL FIGURE 2 FREEZE`


In [ ]:
# ==========================================================================================
# BLOCK 13T — FINAL FIGURE 2 FREEZE
# Editorial operation only — NO analysis
# ==========================================================================================

from pathlib import Path
import shutil, hashlib, json
from datetime import datetime

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
MEM     = PROJECT / "MEMORY_SPECTRUM"
OUT     = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS"
MAIN    = OUT / "MAIN"
AUDIT   = OUT / "AUDIT"

MAIN.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

sources = {
    "png": MEM / "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.png",
    "pdf": MEM / "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.pdf",
    "svg": MEM / "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.svg",
}

destinations = {
    "png": MAIN / "FIGURE_02_HYDROCLIMATIC_MEMORY_SPECTRUM_FINAL.png",
    "pdf": MAIN / "FIGURE_02_HYDROCLIMATIC_MEMORY_SPECTRUM_FINAL.pdf",
    "svg": MAIN / "FIGURE_02_HYDROCLIMATIC_MEMORY_SPECTRUM_FINAL.svg",
}

expected_source_hashes = {
    "png": "019f6cce035919f985a13217efcb391b1a4545b0d2e67e6f54339577e2919f56",
    "pdf": "2cb40d7c19866f7b7c0692fb16b8ef43bbc79222dddc311df9c0bfdd526dc95e",
    "svg": "9034b671d523a1fc648c64316abaee8151eeb390e0b27451272185f51e3dea95",
}

manifest = {
    "block": "13T",
    "operation": "FINAL_FIGURE_2_EDITORIAL_FREEZE",
    "selected_version": "FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1",
    "scientific_analysis_modified": False,
    "new_models": 0,
    "new_hypotheses": 0,
    "files": {}
}

for ext, src in sources.items():

    assert src.exists(), f"Missing source: {src}"

    src_hash = sha256(src)

    assert src_hash == expected_source_hashes[ext], (
        f"{ext.upper()} source hash mismatch\n"
        f"Expected: {expected_source_hashes[ext]}\n"
        f"Found:    {src_hash}"
    )

    dst = destinations[ext]
    shutil.copy2(src, dst)

    dst_hash = sha256(dst)

    assert src_hash == dst_hash

    manifest["files"][ext] = {
        "source": str(src),
        "destination": str(dst),
        "sha256": dst_hash,
        "bytes": dst.stat().st_size
    }

manifest["timestamp"] = datetime.now().isoformat()
manifest["status"] = "PASS"

manifest_path = AUDIT / "BLOCK13T_FIGURE2_FINAL_FREEZE_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("="*100)
print("BLOCK 13T — FIGURE 2 FINAL FREEZE")
print("="*100)

for ext, info in manifest["files"].items():
    print(f"{ext.upper():4} : FROZEN")
    print(f"       {info['destination']}")
    print(f"       SHA256: {info['sha256']}")

print("\nSELECTED VERSION     : B — PUBLICATION")
print("SCIENTIFIC CHANGES   : 0")
print("NEW MODELS           : 0")
print("FIGURE 2 STATUS      : CLOSED")
print("TABLE S1 STATUS      : CLOSED")
print("TABLE S2 STATUS      : CLOSED")
print("FIGURE 3 STATUS      : CLOSED")
print("ANALYSIS FREEZE      : PRESERVED")
print("\nNEXT AND ONLY REMAINING RESULTS PRODUCT:")
print("FIGURE 1 — study area + monitoring network + contributing-area context")
print("="*100)

---
### Imported master block 158

`BLOCK 14 — EXTRACT FINAL MANUSCRIPT TABLES FOR CHAT / WRITING`


In [ ]:
# ==========================================================================================
# BLOCK 14 — EXTRACT FINAL MANUSCRIPT TABLES FOR CHAT / WRITING
# ZERO ANALYSIS — READ ONLY
# ==========================================================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
P = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS"

MAIN = P / "MAIN"
SUPP = P / "SUPPLEMENT"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 80)

print("="*140)
print("BLOCK 14 — FINAL MANUSCRIPT TABLE EXTRACTION")
print("="*140)


# ==========================================================================================
# 1. MAIN TABLE 1 — DATASET ARCHITECTURE
# ==========================================================================================

f = MAIN / "TABLE_01_DATASET_MONITORING_ARCHITECTURE_FINAL.csv"
assert f.exists(), f

T1 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("MAIN TABLE 1 — DATASET AND MONITORING ARCHITECTURE")
print("="*140)
print(T1.to_string(index=False))


# ==========================================================================================
# 2. MAIN TABLE 2 — STRICT HYDROCHEMICAL MEMORY
# ==========================================================================================

# Prefer the authoritative Block8 frozen product.
candidates = [
    MAIN / "TABLE_MEMORY_STRICT_HYDROCHEMICAL_FINAL.csv",
    MAIN / "TABLE_MEMORY_STRICT_ANTECEDENT_SUPPORT_FINAL.csv",
]

f = next((x for x in candidates if x.exists()), None)
assert f is not None, "Main Block8 memory table not found."

T2 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("MAIN TABLE 2 — STRICT ANTECEDENT HYDROCHEMICAL MEMORY")
print("="*140)
print(T2.to_string(index=False))


# ==========================================================================================
# 3. MAIN TABLE 3 — ANTECEDENT HYDROCLIMATE BEYOND Q
# ==========================================================================================

f = MAIN / "TABLE_02_ANTECEDENT_HYDROCLIMATE_BEYOND_Q_FINAL.csv"
assert f.exists(), f

T3 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("MAIN TABLE 3 — ANTECEDENT HYDROCLIMATE BEYOND CONTEMPORANEOUS DISCHARGE")
print("="*140)
print(T3.to_string(index=False))


# ==========================================================================================
# 4. MAIN TABLE 4 — CONTRIBUTING-AREA MODULATION
# ==========================================================================================

f = MAIN / "TABLE_H3_CONTRIBUTING_AREA_MODULATION_FINAL.csv"
assert f.exists(), f

T4 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("MAIN TABLE 4 — CONTRIBUTING-AREA MODULATION")
print("="*140)
print(T4.to_string(index=False))


# ==========================================================================================
# 5. SUPPLEMENTARY TABLE S1 — 50 MEMORY RESPONSE–DRIVER CURVES
# ==========================================================================================

f = SUPP / "TABLE_S01_MEMORY_RESPONSE_DRIVER_SPECTRUM_FINAL.csv"
assert f.exists(), f

S1 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S1 — MEMORY RESPONSE–DRIVER SPECTRUM")
print("SHAPE:", S1.shape)
print("="*140)
print(S1.to_string(index=False))


# ==========================================================================================
# 6. SUPPLEMENTARY TABLE S2 — 290 WINDOW-LEVEL RESULTS
# ==========================================================================================

f = SUPP / "TABLE_S02_MEMORY_WINDOW_FULL_RESULTS_FINAL.csv"
assert f.exists(), f

S2 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S2 — FULL WINDOW-LEVEL MEMORY RESULTS")
print("SHAPE:", S2.shape)
print("="*140)
print(S2.to_string(index=False))


# ==========================================================================================
# 7. SUPPLEMENTARY TABLE S3 — FULL C-Q RESULTS + C09 SENSITIVITY
# ==========================================================================================

f = SUPP / "TABLE_S03_BLOCK9_CQ_COMPLETE_RESULTS_SOURCE.xlsx"
assert f.exists(), f

xls = pd.ExcelFile(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S3 — COMPLETE C-Q RESULTS")
print("Workbook sheets:", xls.sheet_names)
print("="*140)

# Response_Level = authoritative response-level scientific summary.
S3 = pd.read_excel(f, sheet_name="Response_Level")

print(S3.to_string(index=False))


# ==========================================================================================
# 8. SUPPLEMENTARY TABLE S4 — BLOCK 10 ALL 50 LOCO CONTRASTS
# ==========================================================================================

f = SUPP / "TABLE_S04_BLOCK10_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
assert f.exists(), f

S4 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S4 — BLOCK 10 ALL 50 CAMPAIGN-LOCO CONTRASTS")
print("SHAPE:", S4.shape)
print("="*140)
print(S4.to_string(index=False))


# ==========================================================================================
# 9. SUPPLEMENTARY TABLE S5 — BLOCK 10 COMPUTATIONAL QA
# ==========================================================================================

f = SUPP / "TABLE_S05_BLOCK10_COMPUTATIONAL_QA.csv"
assert f.exists(), f

S5 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S5 — BLOCK 10 COMPUTATIONAL QA")
print("SHAPE:", S5.shape)
print("="*140)
print(S5.to_string(index=False))


# ==========================================================================================
# 10. SUPPLEMENTARY TABLE S6 — H3 ALL 50 LOCO CONTRASTS
# ==========================================================================================

f = SUPP / "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
assert f.exists(), f

S6 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S6 — H3 ALL 50 CAMPAIGN-LOCO CONTRASTS")
print("SHAPE:", S6.shape)
print("="*140)
print(S6.to_string(index=False))


# ==========================================================================================
# 11. SUPPLEMENTARY TABLE S7 — H3 COMPUTATIONAL QA
# ==========================================================================================

f = SUPP / "TABLE_S07_H3_COMPUTATIONAL_QA_100_FITS.csv"
assert f.exists(), f

S7 = pd.read_csv(f)

print("\n\n")
print("="*140)
print("SUPPLEMENTARY TABLE S7 — H3 COMPUTATIONAL QA")
print("SHAPE:", S7.shape)
print("="*140)
print(S7.to_string(index=False))


# ==========================================================================================
# FINAL CHECK
# ==========================================================================================

print("\n\n")
print("="*140)
print("FINAL TABLE EXTRACTION GATE")
print("="*140)

checks = [
    ("Main Table 1", T1.shape),
    ("Main Table 2", T2.shape),
    ("Main Table 3", T3.shape),
    ("Main Table 4", T4.shape),
    ("Table S1", S1.shape),
    ("Table S2", S2.shape),
    ("Table S3", S3.shape),
    ("Table S4", S4.shape),
    ("Table S5", S5.shape),
    ("Table S6", S6.shape),
    ("Table S7", S7.shape),
]

for name, shape in checks:
    print(f"{name:20s}: {shape}")

print()
print("NEW ANALYSIS       : 0")
print("NEW MODELS         : 0")
print("FROZEN RESULTS ONLY: TRUE")
print("READY FOR CHAT     : TRUE")
print("="*140)

---
### Imported master block 159

`FINAL EXTRACTION — TABLES S4, S6 AND S7`


In [ ]:
# ==========================================================================================
# FINAL EXTRACTION — TABLES S4, S6 AND S7
# READ ONLY — NO ANALYSIS / NO MODELS / NO RECALCULATION
# ==========================================================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")
SUPP = PROJECT / "FINAL_MANUSCRIPT_PRODUCTS" / "SUPPLEMENT"

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 100)

FILES = {
    "S4": SUPP / "TABLE_S04_BLOCK10_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv",
    "S6": SUPP / "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv",
    "S7": SUPP / "TABLE_S07_H3_COMPUTATIONAL_QA_100_FITS.csv",
}

for name, path in FILES.items():
    assert path.exists(), f"{name} not found: {path}"

S4 = pd.read_csv(FILES["S4"])
S6 = pd.read_csv(FILES["S6"])
S7 = pd.read_csv(FILES["S7"])

# ==========================================================================================
# RAW SCHEMA — IMPORTANT: lets us format without guessing column meanings
# ==========================================================================================

print("="*150)
print("SOURCE SCHEMAS")
print("="*150)

for name, df in [("S4", S4), ("S6", S6), ("S7", S7)]:
    print(f"\n{name}: {df.shape}")
    print(list(df.columns))


# ==========================================================================================
# TABLE S4
# ==========================================================================================

print("\n\n" + "="*150)
print("TABLE S4")
print("Campaign-level leave-one-campaign-out predictive contrasts")
print("for the conditional hydroclimate–discharge analysis")
print("="*150)

print(S4.to_string(index=False))


# ==========================================================================================
# TABLE S6
# ==========================================================================================

print("\n\n" + "="*150)
print("TABLE S6")
print("Campaign-level leave-one-campaign-out predictive contrasts")
print("for contributing-area modulation")
print("="*150)

print(S6.to_string(index=False))


# ==========================================================================================
# TABLE S7
# ==========================================================================================

print("\n\n" + "="*150)
print("TABLE S7")
print("Computational diagnostics for all leave-one-campaign-out")
print("contributing-area models")
print("="*150)

print(S7.to_string(index=False))


# ==========================================================================================
# FINAL CHECK
# ==========================================================================================

print("\n\n" + "="*150)
print("SUPPLEMENT COMPLETION CHECK")
print("="*150)

print(f"S4 : {S4.shape}  — expected 50 campaign contrasts")
print(f"S6 : {S6.shape}  — expected 50 campaign contrasts")
print(f"S7 : {S7.shape}  — expected 100 model fits")

assert len(S4) == 50, f"S4 has {len(S4)} rows, expected 50"
assert len(S6) == 50, f"S6 has {len(S6)} rows, expected 50"
assert len(S7) == 100, f"S7 has {len(S7)} rows, expected 100"

print("\nS4 PASS")
print("S6 PASS")
print("S7 PASS")
print("\nNEW ANALYSIS : 0")
print("READY FOR EDITORIAL FORMATTING : TRUE")
print("="*150)

---
### Imported master block 160

`MAIN MANUSCRIPT FIGURE REVIEW`


In [ ]:
# ======================================================================
# MAIN MANUSCRIPT FIGURE REVIEW
# Huanta Journal of Hydrology
# READ-ONLY — NO NEW ANALYSIS / NO REFITTING / NO FILE MODIFICATION
# ======================================================================

from pathlib import Path
import hashlib
from PIL import Image
import matplotlib.pyplot as plt

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

assert PROJECT.exists(), f"PROJECT not found: {PROJECT}"

print("=" * 110)
print("MAIN MANUSCRIPT FIGURE REVIEW — FROZEN PIPELINE PRODUCTS")
print("=" * 110)
print("PROJECT:", PROJECT)
print("NEW ANALYSIS: FALSE")
print("NEW MODELS : FALSE")
print()

# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def find_exact_or_fail(candidates, label):
    existing = [Path(p) for p in candidates if Path(p).exists()]

    if len(existing) == 0:
        print(f"[MISSING] {label}")
        return None

    # If duplicates exist, prefer FINAL_MANUSCRIPT_PRODUCTS, then FINAL package
    priority_terms = [
        "FINAL_MANUSCRIPT_PRODUCTS",
        "FINAL_MANUSCRIPT_PACKAGE",
        "FINAL_BLOCK9_PACKAGE",
        "MEMORY_SPECTRUM"
    ]

    def score(p):
        s = str(p)
        return sum((len(priority_terms)-i) * 10
                   for i, term in enumerate(priority_terms)
                   if term in s)

    existing = sorted(existing, key=score, reverse=True)

    print(f"[FOUND] {label}")
    for p in existing:
        print("   ", p)

    return existing[0]


def show_figure(path, manuscript_number, title):
    print("\n" + "=" * 110)
    print(f"{manuscript_number} — {title}")
    print("=" * 110)

    if path is None:
        print("STATUS: NOT FOUND")
        return

    print("SELECTED SOURCE :", path)
    print("SHA256          :", sha256(path))

    if path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
        print("Preview skipped: raster preview not available.")
        return

    img = Image.open(path)

    print("PIXELS          :", img.size)
    print("FORMAT          :", img.format)

    # Large display, but no alteration of source
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"{manuscript_number}. {title}",
        fontsize=15,
        pad=15
    )
    plt.tight_layout()
    plt.show()


# ======================================================================
# FIGURE 1
# ======================================================================

print("\n" + "=" * 110)
print("FIGURE 1 — STUDY AREA / MONITORING NETWORK")
print("=" * 110)
print("STATUS   : MANUAL")
print("DECISION : USER WILL BUILD CARTOGRAPHIC FIGURE")
print("PIPELINE : NO NEW SPATIAL ANALYSIS REQUIRED")


# ======================================================================
# FIGURE 2 — BLOCK 8 MEMORY
# ======================================================================

fig2_candidates = [
    PROJECT / "FINAL_MANUSCRIPT_PRODUCTS/MAIN/FIGURE_02_MEMORY_SPECTRUM_FINAL.png",

    PROJECT / "MEMORY_SPECTRUM/FIGURE_MEMORY_SPECTRUM_PUBLICATION_v1.png",
    PROJECT / "MEMORY_SPECTRUM/FIGURE_MEMORY_SPECTRUM_v1.png",
    PROJECT / "MEMORY_SPECTRUM/FIGURE_MEMORY_FINGERPRINT_v1.png",
    PROJECT / "MEMORY_SPECTRUM/FIGURE_MEMORY_SPECTRUM_ALTERNATIVE_v1.png",
]

FIG2 = find_exact_or_fail(
    fig2_candidates,
    "Block 8 hydroclimatic memory figure"
)

show_figure(
    FIG2,
    "FIGURE 2",
    "Hydroclimatic temporal-response and memory spectrum"
)


# ======================================================================
# FIGURE 3 — BLOCK 9 C-Q
# ======================================================================

fig3_candidates = [
    PROJECT / "FINAL_MANUSCRIPT_PRODUCTS/MAIN/FIGURE_03_CQ_RESPONSE_SPECTRUM_FINAL.png",

    PROJECT / "CQ_ANALYSIS/FINAL_BLOCK9_PACKAGE/"
              "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1_600dpi.png",

    PROJECT / "CQ_ANALYSIS/EDITORIAL_FREEZE/"
              "FIG_CQ_DISCHARGE_RESPONSE_SPECTRUM_v1_600dpi.png",
]

FIG3 = find_exact_or_fail(
    fig3_candidates,
    "Block 9 discharge–water-quality figure"
)

show_figure(
    FIG3,
    "FIGURE 3",
    "Discharge–water-quality response spectrum and predictive support"
)


# ======================================================================
# FIGURE 4 — BLOCK 11 H3
# ======================================================================

fig4_candidates = [
    PROJECT / "FINAL_MANUSCRIPT_PRODUCTS/MAIN/"
              "FIGURE_04_CONNECTIVITY_MODULATION_FINAL.png",

    PROJECT / "CONNECTIVITY_ANALYSIS/H3_MODELS/FINAL_MANUSCRIPT_PACKAGE/"
              "FIGURE_H3_CONNECTIVITY_MODULATION_FINAL_v1.png",
]

FIG4 = find_exact_or_fail(
    fig4_candidates,
    "Block 11 contributing-area modulation figure"
)

show_figure(
    FIG4,
    "FIGURE 4",
    "Contributing-area modulation of antecedent hydroclimatic forcing"
)


# ======================================================================
# FINAL MANUSCRIPT FIGURE GATE
# ======================================================================

print("\n\n" + "=" * 110)
print("FINAL MAIN-MANUSCRIPT FIGURE SET")
print("=" * 110)

rows = [
    (
        "Figure 1",
        "Study area, monitoring network and catchment context",
        "MANUAL",
        "To be produced manually"
    ),
    (
        "Figure 2",
        "Hydroclimatic temporal-response / memory spectrum",
        "READY" if FIG2 else "MISSING",
        str(FIG2) if FIG2 else "—"
    ),
    (
        "Figure 3",
        "Discharge–water-quality response spectrum",
        "READY" if FIG3 else "MISSING",
        str(FIG3) if FIG3 else "—"
    ),
    (
        "Figure 4",
        "Contributing-area modulation of antecedent forcing",
        "READY" if FIG4 else "MISSING",
        str(FIG4) if FIG4 else "—"
    ),
]

for num, title, status, source in rows:
    print(f"\n{num}")
    print(f"  Content : {title}")
    print(f"  Status  : {status}")
    print(f"  Source  : {source}")

ready = all(x is not None for x in [FIG2, FIG3, FIG4])

print("\n" + "-" * 110)
print("FIGURES 2–4 PIPELINE PRODUCTS READY :", ready)
print("FIGURE 1                         : MANUAL")
print("NEW SCIENTIFIC ANALYSIS          : 0")
print("NEW MODEL FITS                   : 0")
print("FROZEN RESULTS MODIFIED          : FALSE")

if ready:
    print("\nMAIN FIGURE SET: SCIENTIFICALLY COMPLETE")
    print("Only manual Figure 1 remains.")
else:
    print("\nMAIN FIGURE SET: ONE OR MORE FROZEN PRODUCTS NEED LOCATION/RESOLUTION")

print("=" * 110)

---
### Imported master block 161

`FIGURE 3 — FINAL EDITORIAL VERSION`


In [ ]:
# ==========================================================================================
# FIGURE 3 — FINAL EDITORIAL VERSION
# Discharge–water-quality response spectrum
#
# SOURCE:
# BLOCK9_CQ_TABLES_FINAL_v1.xlsx / Response_Level
#
# NO NEW ANALYSIS
# NO RECALCULATION
# NO MODEL FITTING
# ==========================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import hashlib

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

SOURCE = (
    PROJECT /
    "CQ_ANALYSIS/FINAL_BLOCK9_PACKAGE/"
    "BLOCK9_CQ_TABLES_FINAL_v1.xlsx"
)

OUT = (
    PROJECT /
    "FINAL_MANUSCRIPT_PRODUCTS/MAIN/EDITORIAL_FINAL"
)
OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), SOURCE

# ==========================================================================================
# STYLE
# ==========================================================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ==========================================================================================
# LOAD AUTHORITATIVE RESPONSE-LEVEL TABLE
# ==========================================================================================

cq = pd.read_excel(
    SOURCE,
    sheet_name="Response_Level"
)

print("SOURCE:", SOURCE)
print("SHAPE :", cq.shape)
print("\nCOLUMNS:")
print(list(cq.columns))

required = [
    "display_name",

    "primary_beta_std",
    "primary_hdi_low",
    "primary_hdi_high",
    "primary_mean_delta_LPD",

    "sensitivity_beta_std",
    "sensitivity_hdi_low",
    "sensitivity_hdi_high",
    "sensitivity_mean_delta_LPD",
]

missing = [c for c in required if c not in cq.columns]

assert not missing, f"Missing frozen columns: {missing}"

print("\nRequired frozen columns: PASS")


# ==========================================================================================
# CANONICAL RESPONSE ORDER
# ==========================================================================================

ORDER = [
    "Electrical conductivity",
    "Na",
    "SO₄",
    "pH",
    "NH₄",
    "Total coliforms",
    "Al",
    "Fe",
    "Si",
    "Water temperature",
]

def canonical_response(x):

    s = str(x).strip().lower()

    if "conduct" in s:
        return "Electrical conductivity"

    if s == "na" or "sodium" in s:
        return "Na"

    if (
        "so4" in s or
        "so₄" in s or
        "sulf" in s
    ):
        return "SO₄"

    if s == "ph":
        return "pH"

    if (
        "nh4" in s or
        "nh₄" in s or
        "ammon" in s
    ):
        return "NH₄"

    if "total" in s and "coliform" in s:
        return "Total coliforms"

    if s == "al" or "alumin" in s:
        return "Al"

    if s == "fe" or "hierro" in s or s == "iron":
        return "Fe"

    if s == "si" or "silic" in s:
        return "Si"

    if "temperature" in s:
        return "Water temperature"

    return str(x)


cq["plot_response"] = cq["display_name"].map(
    canonical_response
)

print("\nRESPONSE CROSSWALK:")
print(
    cq[
        ["display_name", "plot_response"]
    ].to_string(index=False)
)

unknown = set(ORDER) - set(cq["plot_response"])

assert not unknown, (
    f"Could not match these responses: {unknown}"
)

cq = (
    cq
    .set_index("plot_response")
    .reindex(ORDER)
    .reset_index()
)

assert len(cq) == 10

print("\nResponse ordering: PASS")


# ==========================================================================================
# EXTRACT FROZEN VALUES
# ==========================================================================================

y = np.arange(len(ORDER))

pb = cq["primary_beta_std"].astype(float).to_numpy()
pl = cq["primary_hdi_low"].astype(float).to_numpy()
ph = cq["primary_hdi_high"].astype(float).to_numpy()

sb = cq["sensitivity_beta_std"].astype(float).to_numpy()
sl = cq["sensitivity_hdi_low"].astype(float).to_numpy()
sh = cq["sensitivity_hdi_high"].astype(float).to_numpy()

pd_lpd = (
    cq["primary_mean_delta_LPD"]
    .astype(float)
    .to_numpy()
)

sd_lpd = (
    cq["sensitivity_mean_delta_LPD"]
    .astype(float)
    .to_numpy()
)

# Internal consistency
assert np.all(pl <= pb)
assert np.all(pb <= ph)

assert np.all(sl <= sb)
assert np.all(sb <= sh)

print("Posterior interval QA: PASS")


# ==========================================================================================
# FIGURE
# ==========================================================================================

fig, (axA, axB) = plt.subplots(
    1, 2,
    figsize=(11.4, 6.3),
    gridspec_kw={
        "width_ratios": [1.08, 1.0]
    }
)

# Deliberately restrained palette.
# Meaning is encoded mainly by marker/line style.

PRIMARY = "#222222"
SENS = "#777777"
ZERO = "#555555"
GRID = "#E8E8E8"

offset = 0.13


# ==========================================================================================
# PANEL A — POSTERIOR Q EFFECT
# ==========================================================================================

for i in range(len(ORDER)):

    # Primary 95% HDI
    axA.plot(
        [pl[i], ph[i]],
        [y[i] - offset, y[i] - offset],
        color=PRIMARY,
        lw=1.8,
        solid_capstyle="round",
        zorder=2
    )

    # Sensitivity 95% HDI
    axA.plot(
        [sl[i], sh[i]],
        [y[i] + offset, y[i] + offset],
        color=SENS,
        lw=1.25,
        ls="--",
        solid_capstyle="round",
        zorder=2
    )


# Primary posterior means
axA.scatter(
    pb,
    y - offset,
    s=48,
    marker="o",
    color=PRIMARY,
    edgecolors=PRIMARY,
    linewidths=0.8,
    zorder=5,
    label="Primary: observed Q"
)

# Sensitivity posterior means
axA.scatter(
    sb,
    y + offset,
    s=42,
    marker="s",
    facecolors="white",
    edgecolors=SENS,
    linewidths=1.3,
    zorder=5,
    label="Sensitivity: + reconstructed C09 Q"
)

axA.axvline(
    0,
    color=ZERO,
    ls=":",
    lw=1.1,
    zorder=1
)

axA.set_yticks(y)
axA.set_yticklabels(ORDER)
axA.invert_yaxis()

axA.set_xlabel(
    r"Standardized discharge effect, $\beta_Q$"
)

axA.set_title(
    "A   Posterior discharge effects",
    loc="left",
    fontweight="bold",
    pad=9
)

axA.grid(
    axis="x",
    color=GRID,
    lw=0.7,
    zorder=0
)

axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


# ==========================================================================================
# PANEL B — PREDICTIVE SUPPORT
# ==========================================================================================

# Thin connectors make sensitivity shifts visible
# without assigning a different arbitrary color to every response.

for i in range(len(ORDER)):

    axB.plot(
        [pd_lpd[i], sd_lpd[i]],
        [y[i] - offset, y[i] + offset],
        color="#BBBBBB",
        lw=0.9,
        ls=":",
        zorder=1
    )


axB.scatter(
    pd_lpd,
    y - offset,
    s=48,
    marker="o",
    color=PRIMARY,
    edgecolors=PRIMARY,
    linewidths=0.8,
    zorder=5
)

axB.scatter(
    sd_lpd,
    y + offset,
    s=42,
    marker="s",
    facecolors="white",
    edgecolors=SENS,
    linewidths=1.3,
    zorder=5
)

axB.axvline(
    0,
    color=ZERO,
    ls=":",
    lw=1.1,
    zorder=1
)

axB.set_yticks(y)
axB.set_yticklabels([])
axB.invert_yaxis()

axB.set_xlabel(
    r"Campaign-equal mean $\Delta$LPD "
    r"(Q model $-$ null)"
)

axB.set_title(
    "B   Held-out predictive contribution of discharge",
    loc="left",
    fontweight="bold",
    pad=9
)

axB.grid(
    axis="x",
    color=GRID,
    lw=0.7,
    zorder=0
)

axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ==========================================================================================
# LEGEND
# ==========================================================================================

handles, labels = axA.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.52, 0.005),
    ncol=2,
    frameon=False,
    handletextpad=0.7,
    columnspacing=2.0
)


# ==========================================================================================
# LAYOUT
# ==========================================================================================

fig.subplots_adjust(
    left=0.235,
    right=0.985,
    top=0.94,
    bottom=0.15,
    wspace=0.17
)


# ==========================================================================================
# SAVE
# ==========================================================================================

stem = "FIGURE_03_CQ_RESPONSE_SPECTRUM_EDITORIAL_FINAL"

outputs = []

for ext in ["png", "pdf", "svg"]:

    path = OUT / f"{stem}.{ext}"

    if ext == "png":
        fig.savefig(
            path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )
    else:
        fig.savefig(
            path,
            bbox_inches="tight",
            facecolor="white"
        )

    outputs.append(path)


# ==========================================================================================
# SHOW
# ==========================================================================================

plt.show()
plt.close(fig)


# ==========================================================================================
# REPORT
# ==========================================================================================

print("\n" + "=" * 100)
print("FIGURE 3 — EDITORIAL FINAL")
print("=" * 100)

for p in outputs:
    print("\n", p)
    print(" SHA256:", sha256(p))

print("\nSOURCE:")
print(SOURCE)

print("\nSCIENTIFIC OPERATIONS:")
print("  New model fits        : 0")
print("  New inference         : 0")
print("  New sensitivity       : 0")
print("  Recalculated β        : 0")
print("  Recalculated HDI      : 0")
print("  Recalculated ΔLPD     : 0")
print("  Frozen source modified: FALSE")

print("\nFIGURE 3: READY FOR VISUAL REVIEW")
print("=" * 100)

---
### Imported master block 162

`import pandas as pd`


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3_MAIN = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS/H3_MODELS/FINAL_MANUSCRIPT_PACKAGE/"
    "TABLE_H3_MAIN_FINAL_v1.csv"
)

H3_LOCO = (
    PROJECT /
    "FINAL_MANUSCRIPT_PRODUCTS/SUPPLEMENT/"
    "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
)

assert H3_MAIN.exists(), H3_MAIN
assert H3_LOCO.exists(), H3_LOCO

main = pd.read_csv(H3_MAIN)
loco = pd.read_csv(H3_LOCO)

print("="*100)
print("H3 MAIN")
print("shape:", main.shape)
print("columns:")
print(list(main.columns))
print("\n")
print(main.to_string(index=False))

print("\n" + "="*100)
print("H3 LOCO")
print("shape:", loco.shape)
print("columns:")
print(list(loco.columns))
print("="*100)

---
### Imported master block 163

`FIGURE 4 — FINAL EDITORIAL VERSION`


In [ ]:
# ==========================================================================================
# FIGURE 4 — FINAL EDITORIAL VERSION
# Contributing-area modulation of antecedent hydroclimatic forcing
#
# FROZEN SOURCES:
#   TABLE_H3_MAIN_FINAL_v1.csv
#   TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv
#
# NO NEW ANALYSIS / NO MODEL FITTING / NO RECALCULATION
# ==========================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import hashlib

PROJECT = Path("/content/gdrive/MyDrive/Huanta_JoH_PROJECT")

H3_MAIN = (
    PROJECT /
    "CONNECTIVITY_ANALYSIS/H3_MODELS/FINAL_MANUSCRIPT_PACKAGE/"
    "TABLE_H3_MAIN_FINAL_v1.csv"
)

H3_LOCO = (
    PROJECT /
    "FINAL_MANUSCRIPT_PRODUCTS/SUPPLEMENT/"
    "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv"
)

OUT = (
    PROJECT /
    "FINAL_MANUSCRIPT_PRODUCTS/MAIN/EDITORIAL_FINAL"
)
OUT.mkdir(parents=True, exist_ok=True)

assert H3_MAIN.exists(), H3_MAIN
assert H3_LOCO.exists(), H3_LOCO

# ==========================================================================================
# STYLE
# ==========================================================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ==========================================================================================
# LOAD FROZEN PRODUCTS
# ==========================================================================================

main = pd.read_csv(H3_MAIN)
loco = pd.read_csv(H3_LOCO)

print("H3 MAIN :", main.shape)
print("H3 LOCO :", loco.shape)

assert main.shape[0] == 5
assert loco.shape[0] == 50

required_main = [
    "Response",
    "Antecedent predictor",
    "Role",
    "beta_HxC",
    "HDI_lower",
    "HDI_upper",
    "Pdirection",
    "Mean_delta_LPD",
    "Median_delta_LPD",
    "Campaigns_H3_better",
    "Inference",
]

required_loco = [
    "test_id",
    "test_label",
    "heldout_campaign",
    "n_test_BASE",
    "lpd_mean_per_obs_BASE",
    "lpd_mean_per_obs_H3",
    "delta_LPD_H3_minus_BASE",
    "H3_better",
]

assert all(c in main.columns for c in required_main)
assert all(c in loco.columns for c in required_loco)

print("Frozen schema QA: PASS")


# ==========================================================================================
# LABELS
# ==========================================================================================

def make_label(row):
    response = str(row["Response"])
    predictor = str(row["Antecedent predictor"])

    return f"{response} — {predictor.lower()}"

main["plot_label"] = main.apply(make_label, axis=1)

labels = main["plot_label"].tolist()

print("\nPlot order:")
for x in labels:
    print(" ", x)


# Match LOCO tests to the same five rows
def canonical_loco(x):
    s = str(x).lower()

    if "electrical conductivity" in s and "14" in s:
        return labels[0]

    if "electrical conductivity" in s and "60" in s:
        return labels[1]

    if (
        ("na " in s or s.startswith("na") or "sodium" in s)
        and "60" in s
    ):
        return labels[2]

    if ("so₄" in s or "so4" in s or "sulf" in s) and "60" in s:
        return labels[3]

    if "water temperature" in s and "90" in s:
        return labels[4]

    return None

loco["plot_label"] = loco["test_label"].map(canonical_loco)

assert loco["plot_label"].notna().all(), (
    "At least one LOCO test could not be matched."
)

counts = loco.groupby("plot_label").size()

for lab in labels:
    assert counts[lab] == 10, (
        f"{lab}: expected 10 folds, found {counts[lab]}"
    )

print("\nLOCO crosswalk QA: PASS — 10 campaigns/test")


# ==========================================================================================
# VERIFY MEANS AGAINST FROZEN MAIN TABLE
#
# This is only an integrity check:
# it does NOT replace or modify the frozen main results.
# ==========================================================================================

print("\nMean ΔLPD integrity check:")

for _, row in main.iterrows():

    lab = row["plot_label"]

    vals = loco.loc[
        loco["plot_label"] == lab,
        "delta_LPD_H3_minus_BASE"
    ].astype(float)

    calculated = vals.mean()
    frozen = float(row["Mean_delta_LPD"])

    print(
        f"{lab:65s} "
        f"frozen={frozen:+.6f}  "
        f"LOCO mean={calculated:+.6f}"
    )

    assert np.isclose(
        calculated,
        frozen,
        atol=1e-6
    )

print("Mean ΔLPD integrity: PASS")


# ==========================================================================================
# DATA
# ==========================================================================================

y = np.arange(len(main))

beta = main["beta_HxC"].astype(float).to_numpy()
lo = main["HDI_lower"].astype(float).to_numpy()
hi = main["HDI_upper"].astype(float).to_numpy()

mean_dlpd = main["Mean_delta_LPD"].astype(float).to_numpy()

assert np.all(lo <= beta)
assert np.all(beta <= hi)

print("Posterior interval QA: PASS")


# ==========================================================================================
# FIGURE
# ==========================================================================================

fig, (axA, axB) = plt.subplots(
    2, 1,
    figsize=(10.5, 6.8),
    gridspec_kw={
        "height_ratios": [0.82, 1.18],
        "hspace": 0.42
    }
)

PRIMARY = "#222222"
THERMAL = "#777777"
CAMPAIGN = "#B5B5B5"
ZERO = "#555555"
GRID = "#E8E8E8"


# ==========================================================================================
# PANEL A — FULL-DATA INTERACTION
# ==========================================================================================

for i, row in main.iterrows():

    is_thermal = (
        str(row["Role"]).lower()
        .startswith("secondary")
    )

    col = THERMAL if is_thermal else PRIMARY

    axA.plot(
        [row["HDI_lower"], row["HDI_upper"]],
        [i, i],
        color=col,
        lw=1.9,
        solid_capstyle="round",
        zorder=2
    )

    axA.scatter(
        row["beta_HxC"],
        i,
        s=52,
        marker="o",
        color=col,
        edgecolors=col,
        zorder=5
    )

axA.axvline(
    0,
    color=ZERO,
    ls=":",
    lw=1.1,
    zorder=1
)

axA.set_yticks(y)
axA.set_yticklabels(labels)
axA.invert_yaxis()

axA.set_xlabel(
    r"Standardized hydroclimate $\times$ contributing-area interaction "
    r"($\beta_{H\times C}$)"
)

axA.set_title(
    "A   Full-data interaction estimates",
    loc="left",
    fontweight="bold",
    pad=8
)

axA.grid(
    axis="x",
    color=GRID,
    lw=0.7,
    zorder=0
)

axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


# ==========================================================================================
# PANEL B — CAMPAIGN LOCO
# ==========================================================================================

rng = np.random.default_rng(20260917)

for i, row in main.iterrows():

    lab = row["plot_label"]

    d = (
        loco[loco["plot_label"] == lab]
        .sort_values("heldout_campaign")
        .copy()
    )

    vals = (
        d["delta_LPD_H3_minus_BASE"]
        .astype(float)
        .to_numpy()
    )

    # tiny deterministic vertical jitter only for visibility
    jitter = rng.uniform(-0.095, 0.095, len(vals))

    # individual held-out campaigns
    axB.scatter(
        vals,
        np.full(len(vals), i) + jitter,
        s=23,
        marker="o",
        color=CAMPAIGN,
        alpha=0.72,
        edgecolors="none",
        zorder=2
    )

    is_thermal = (
        str(row["Role"]).lower()
        .startswith("secondary")
    )

    col = THERMAL if is_thermal else PRIMARY

    # Frozen campaign-equal mean
    axB.scatter(
        float(row["Mean_delta_LPD"]),
        i,
        marker="D",
        s=82,
        color=col,
        edgecolors="white",
        linewidths=0.8,
        zorder=5
    )

axB.axvline(
    0,
    color=ZERO,
    ls=":",
    lw=1.1,
    zorder=1
)

axB.set_yticks(y)
axB.set_yticklabels(labels)
axB.invert_yaxis()

axB.set_xlabel(
    r"Campaign LOCO $\Delta$LPD per observation "
    r"(H3 interaction model $-$ additive $H+Q$ model)"
)

axB.set_title(
    r"B   Out-of-campaign predictive contribution of $H\times C$",
    loc="left",
    fontweight="bold",
    pad=8
)

axB.grid(
    axis="x",
    color=GRID,
    lw=0.7,
    zorder=0
)

axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ==========================================================================================
# LEGEND / KEY
# ==========================================================================================

from matplotlib.lines import Line2D

legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markerfacecolor=CAMPAIGN,
        markeredgecolor="none",
        markersize=5.5,
        label="Held-out campaign"
    ),
    Line2D(
        [0], [0],
        marker="D",
        linestyle="None",
        markerfacecolor=PRIMARY,
        markeredgecolor="white",
        markersize=7,
        label="Campaign-equal mean"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markerfacecolor=THERMAL,
        markeredgecolor=THERMAL,
        markersize=6,
        label="Secondary thermal reference"
    ),
]

axB.legend(
    handles=legend_elements,
    loc="lower right",
    frameon=False,
    ncol=1
)


# ==========================================================================================
# LAYOUT
# ==========================================================================================

fig.subplots_adjust(
    left=0.36,
    right=0.985,
    top=0.96,
    bottom=0.10
)


# ==========================================================================================
# SAVE — DO NOT OVERWRITE OLD FROZEN VERSION
# ==========================================================================================

stem = "FIGURE_04_CONNECTIVITY_MODULATION_EDITORIAL_FINAL"

outputs = []

for ext in ["png", "pdf", "svg"]:

    path = OUT / f"{stem}.{ext}"

    if ext == "png":
        fig.savefig(
            path,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )
    else:
        fig.savefig(
            path,
            bbox_inches="tight",
            facecolor="white"
        )

    outputs.append(path)


plt.show()
plt.close(fig)


# ==========================================================================================
# FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 105)
print("FIGURE 4 — EDITORIAL FINAL")
print("=" * 105)

for p in outputs:
    print(f"\n{p}")
    print("SHA256:", sha256(p))

print("\nFROZEN SOURCES:")
print(" ", H3_MAIN)
print(" ", H3_LOCO)

print("\nSCIENTIFIC OPERATIONS:")
print("  New model fits             : 0")
print("  New hypothesis tests       : 0")
print("  New H×C estimates          : 0")
print("  New predictive contrasts   : 0")
print("  Window selection           : 0")
print("  Frozen sources modified    : FALSE")

print("\nEXPECTED SCIENTIFIC CONTENT:")
print("  Primary H3 tests supported : 4/4")
print("  Thermal reference supported: NO")
print("  LOCO folds represented     : 50/50")

print("\nFIGURE 4: READY FOR VISUAL REVIEW")
print("=" * 105)

---

## Publication release checklist

Before creating the GitHub/Zenodo release:

- [ ] Replace any remaining `/content/...` or Google Drive paths with repository-relative paths.
- [ ] Confirm that every frozen input referenced by the notebook is included in `data/processed/` or documented as externally obtainable.
- [ ] Remove any credentials, tokens, private URLs, personal identifiers, or unrelated project files.
- [ ] Confirm that no IGP/tesista dataset has been mixed into this manuscript's repository.
- [ ] Execute the notebook from a clean environment.
- [ ] Confirm that manuscript tables and figures reproduce from the frozen inputs.
- [ ] Export `requirements.txt` or `environment.yml`.
- [ ] Add `README.md`, `LICENSE`, `CITATION.cff`, and a data dictionary.
- [ ] Tag the submission release (recommended: `v1.0.0`).
- [ ] Archive that release in Zenodo and use the resulting DOI in the manuscript.

### Scope of reproducibility

This notebook is a **publication workflow**, not the full forensic provenance notebook.  
The raw-source master audit should be retained privately and can be made available separately if ownership and data-sharing permissions allow.
